# Section 0, cell 1 - MRI classification overview and settings guide

## MAJOR/ESSENTIAL SETTINGS

These are the settings that most runs need to change. They are grouped at the very top of the code
cell labelled `Section 0, cell 3`, under the heading `MAJOR/ESSENTIAL SETTINGS`, in the order shown here.

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `DEVELOPMENT_ROOT =` | Folder, or list of folders, holding the development patients' paired image and mask NRRD files. The folders in a list are read one after the other and all of their patients form the one development set; a patient's image and mask must be in the same folder. A folder path that contains `none` anywhere (any capitalization) is ignored with a warning. At least one real folder is required. Every patient here is used for cross-validation, tuning and the final refit. Used by every section. |
| Section 0, cell 3 | `TEST_ROOT =` | One or two folders holding the holdout test patients' paired image and mask NRRD files. Each folder is its own test set, named after the folder; every test set is scored and reported separately. A folder path that contains `none` anywhere (any capitalization) is ignored entirely, so leaving the second line on such a path runs the notebook with one test set. Test patients are scored once at the end and never used for training or setting selection. |
| Section 0, cell 3 | `OUT_ROOT =` | Main Google Drive folder for all result files. |
| Section 0, cell 3 | `LABELS_CSV =` | The one CSV file of 0/1 labels (the outcome) analysed in this run, used by every section. To analyse another label file, put its path here and run the notebook again; change `OUT_ROOT` as well if the earlier results should be kept. |
| Section 0, cell 3 | `N_SLICES         =` | Number of tumour-bearing slices used per patient. Valid choices: 1 = axial; 2 = axial + sagittal; 3 = axial + sagittal + coronal (central maximum-area slice in each plane). |
| Section 0, cell 3 | `RUN_FROZEN_CNN_E2E    =` | Runs Section 1, the frozen-backbone end-to-end CNN workflow. |
| Section 0, cell 3 | `RUN_CNN_LR            =` | Runs Section 2, the frozen-CNN feature extraction plus machine-learning workflow. |
| Section 0, cell 3 | `RUN_CNN_FINETUNE_E2E  =` | Runs Section 3, the fine-tuned end-to-end CNN workflow. `False` skips fine-tuning. |
| Section 0, cell 3 | `SHARED_BACKBONES =` | CNN architectures considered by the workflows. |
| Section 0, cell 3 | `RUN_AUTOSEG_NNINTERACTIVE =` | Runs the nnInteractive automatic segmentation check in Section 0, cell 5. It starts from the existing mask (one prompt on one slice per tumour) and reports the Dice Similarity Coefficient (DSC) against the existing mask. It never changes a classification result. |
| Section 0, cell 3 | `RUN_AUTOSEG_SAM2          =` | Runs the SAM 2 automatic segmentation check in Section 0, cell 5 (one box on every tumour slice). With both switches on `False` the cell is skipped and no extra package is installed. |
| Section 0, cell 3 | `NNINTERACTIVE_WEIGHTS_PATH =` | Folder holding the nnInteractive v1.0 weights. Download: https://huggingface.co/MIC-DKFZ/nnInteractive/tree/main/nnInteractive_v1.0 (about 410 MB; weights licensed CC BY-NC-SA 4.0, non-commercial). |
| Section 0, cell 3 | `SAM2_WEIGHTS_PATH          =` | Folder holding the SAM 2.1 Hiera Base+ weights. Download: https://huggingface.co/facebook/sam2.1-hiera-base-plus/tree/main (about 325 MB; Apache-2.0). |
| Section 0, cell 3 | `AUTOSEG_DOWNLOAD_MISSING_WEIGHTS =` | `True` lets Section 0, cell 5 download a missing weights folder itself, once, into the path given. `False` stops with a message instead. |

This notebook evaluates MRI classification with one fixed patient split, defined by folder location
(DEVELOPMENT folder = development patients, each TEST folder = one holdout test set), and three possible workflows:
Section 1 trains an end-to-end CNN with the backbone frozen, Section 2 extracts CNN features and then
tests classical machine-learning models and Section 3 trains an end-to-end CNN with fine-tuning.

The visible cell labels in this notebook use the format `Section X, cell Y`. Use those labels together
with the quoted text below to find settings quickly. The quoted text is the start of the code line,
up to the equals sign. Search for exactly this text in the notebook. It stays the same when the
value of a setting is changed.

Plain-language terms used in the notebook:

- `CV` (cross-validation): the development patients are split into folds so each fold can be tested by a model that did not train on those patients.
- `OOF` (out-of-fold): the prediction for each development patient made by a model that did not train on that patient.
- `holdout`: the test patients in the TEST folder(s), kept out of all training and setting selection.
- `test_set`: the holdout test set a result belongs to, named after its `TEST_ROOT` folder. One or two test sets can be given. The model is refitted once on all development patients and scored on each test set separately, and the result tables are given separately for each test set.
- `AUC` is a threshold-free ranking score; higher is better.
- `LR`: learning rate in training settings such as `LR_HEAD`, and logistic regression in model names such as `LOGREG`.
- `optimization arm`: one tested combination of settings, for example one image size, one slice count and one augmentation level.
- `frozen`, `efficient_ML`, `finetune`: the three workflow names used in every result table. `frozen` is Section 1 (end-to-end CNN, backbone frozen), `efficient_ML` is Section 2 (frozen-CNN features plus classical machine learning) and `finetune` is Section 3 (end-to-end CNN, backbone fine-tuned).
- `outcome`: the label file analysed in this run, named after the file (`labels` for `labels.csv`, `labels_35` for `labels-35.csv`). One run analyses one label file. An outcome name is not a workflow and not a cut-off.
- `threshold`: always the probability cut-off that turns a predicted score into a predicted class.

### General settings shared by all three workflows

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `PACKAGE_PINS =` | Package versions installed at the start of the notebook. |
| Section 0, cell 3 | `GLOBAL_SEED =` | Sets the base random seed so splits and training are repeatable. |
| Section 0, cell 3 | `HF_TOKEN =` | Optional Hugging Face access key; leave blank when using public downloads or a Colab Secret. |
| Section 0, cell 3 | `CNN_INPUT_SIZE_SHARED =` | Default image size in pixels for CNN inputs. |
| Section 0, cell 3 | `SHOW_ANALYSED_SLICES =` | Shows visual checks of the slices before long runs. |
| Section 0, cell 3 | `QC_N_PATIENTS        =` | Number of patients shown in the visual check. |
| Section 0, cell 3 | `QC_PATIENTS_PER_FIG  =` | Number of patients placed in each visual-check figure. |

### Optimization comparison settings

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `OPTIMIZATION =` | `False` runs every backbone with the standard settings. `True` compares multiple setting combinations before summarising results. |
| Section 0, cell 3 | `OPTIMIZATION_EVALUATE_HOLDOUT =` | Used only in an optimization comparison. `False` leaves every holdout test set unscored, so settings are chosen from development data alone. `True` also refits every setting combination and scores it on every holdout test set. |
| Section 0, cell 3 | `OPTIMIZATION_INPUT_SIZES =` | Compare multiple image sizes. |
| Section 0, cell 3 | `OPTIMIZATION_N_SLICES    =` | Slice counts to compare; keep the comma for a one-item tuple. |
| Section 0, cell 3 | `OPTIMIZATION_AUG_LEVELS  =` | Training augmentation levels to compare for CNN training workflows. |
| Section 0, cell 3 | `TUMOR_CONTEXT_MODE =` | Chooses what image content is kept: minimal bounding-box crop or tumour-only pixels. |
| Section 0, cell 3 | `ROI_ONLY_BACKGROUND_VALUE =` | Background fill value used only when `TUMOR_CONTEXT_MODE` is `ROI_ONLY`. |
| Section 0, cell 3 | `OPTIMIZATION_INCLUDE_REFERENCE_ARM =` | Adds the current standard setting combination so every comparison has a reference. With `False` the run stops at the start unless the comparison lists already contain that combination. |
| Section 0, cell 3 | `OPTIMIZATION_BACKBONES =` | CNN architectures included in the optimization comparison. |
| Section 0, cell 3 | `OPTIMIZATION_N_FOLDS   =` | Cross-validation folds used during optimization comparisons. |
| Section 0, cell 3 | `OPTIMIZATION_N_REPEATS =` | How many repeated cross-validation runs are used during optimization comparisons. |
| Section 0, cell 3 | `OPTIMIZATION_MIN_DELTA =` | Minimum AUC gain required before a new setting is treated as meaningfully better. |
| Section 0, cell 3 | `OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW =` | Reference augmentation level for each CNN training workflow. |

### Augmentation and image-preparation settings

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `TRAIN_AUGMENTATION_LEVEL =` | Training augmentation level used by Section 3 when optimization comparison is off. |
| Section 0, cell 3 | `FROZEN_E2E_AUGMENTATION_LEVEL =` | Training augmentation level used by Section 1 when optimization comparison is off. |
| Section 0, cell 3 | `AUG_ROTATE_DEG  =` | Maximum random rotation, in degrees, used by `BASIC`, `COMPLEX` and `COMPLEX_JITTER`. |
| Section 0, cell 3 | `AUG_SCALE_RANGE =` | Random zoom range used by `COMPLEX` and `COMPLEX_JITTER`. |
| Section 0, cell 3 | `AUG_TRANSLATE   =` | Random shift as a fraction of image size, used by `COMPLEX` and `COMPLEX_JITTER`. |
| Section 0, cell 3 | `AUG_GAMMA_RANGE =` | Random gamma/intensity curve range used by `COMPLEX_JITTER`. |
| Section 0, cell 3 | `AUG_BRIGHTNESS  =` | Random brightness change used by `COMPLEX_JITTER`. |
| Section 0, cell 3 | `AUG_CONTRAST    =` | Random contrast change used by `COMPLEX_JITTER`. |
| Section 0, cell 3 | `AUG_NOISE_STD   =` | Random image noise strength used by `COMPLEX_JITTER`. |
| Section 0, cell 3 | `CROP_SAFETY_MARGIN_PX =` | Extra native pixels kept around the tumour crop on every side. |
| Section 0, cell 3 | `INTENSITY_Z_SPAN_SD =` | Intensity range kept after z-scoring, before rescaling to 0-1. |
| Section 0, cell 3 | `LOCAL_CACHE_DIR =` | Local Colab cache folder for copied MRI files. Shared by every section, so each file is read from Drive once. |
| Section 0, cell 3 | `SAVE_SLICE_LEVEL_FEATURE_CSVS =` | Whether to write per-slice feature CSVs in addition to patient-level features. |

### Evaluation, feature-selection and output settings

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `TOP_RISK_PERCENT =` | Also reports metrics for the highest-risk 25 percent of patients. |
| Section 0, cell 3 | `AUC_CI_CONFIDENCE =` | Confidence level for AUC interval estimates, in all three sections. |
| Section 0, cell 3 | `AUC_BOOTSTRAP_REPEATS =` | Number of bootstrap repeats used for AUC intervals, in all three sections. |
| Section 0, cell 3 | `MAX_DISPLAY_ROWS =` | Maximum rows shown by notebook table displays. |
| Section 0, cell 3 | `USE_FEATURE_SELECTOR_VOTING =` | Turns on feature-selector voting in the Section 2 machine-learning workflow. |
| Section 0, cell 3 | `RUN_FEATURE_SELECTOR_COMBINATIONS =` | Feature-selection methods used together as a voting group. |
| Section 0, cell 3 | `FEATURE_SELECTOR_TOP_K =` | Number of features nominated by each feature-selection method. |
| Section 0, cell 3 | `FEATURE_SELECTOR_MIN_KEEP =` | Minimum number of features kept after voting. |
| Section 0, cell 3 | `FEATURE_SELECTOR_MIN_VOTES =` | Votes required for a feature to be kept. |
| Section 0, cell 3 | `RUN_CLASSIFIERS =` | Classical machine-learning classifiers run in Section 2. |
| Section 0, cell 3 | `PART1_OUT_DIR         =` | Result folder for Section 1. |
| Section 0, cell 3 | `PART2_OUT_DIR         =` | Result folder for Section 2 feature extraction. |
| Section 0, cell 3 | `SEG2_FEATURE_DIR      =` | Folder containing patient-level feature CSVs. |
| Section 0, cell 3 | `PART3_OUT_DIR         =` | Result folder for Section 2 machine-learning models. |
| Section 0, cell 3 | `FINETUNE_OUT_DIR      =` | Result folder for Section 3 fine-tuning. |
| Section 0, cell 3 | `COMBINED_OVERVIEW_DIR =` | Folder for final combined overview files. |

### Automatic segmentation check settings

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `AUTOSEG_PACKAGE_PINS =` | Package versions installed for the automatic segmentation check. They are installed in the same line as `PACKAGE_PINS`, which keeps `timm` at its pinned version. |
| Section 0, cell 3 | `AUTOSEG_OUT_DIR       =` | Result folder of the automatic segmentation check: tables, overlay figures and the automatic masks. |
| Section 0, cell 5 | `AUTOSEG_PILOT_N_PATIENTS =` | A number runs a pilot on that many DEVELOPMENT patients only. `None` runs the whole cohort, development and test. |
| Section 0, cell 5 | `NNINTERACTIVE_PROMPTS =` | Fixed to `("lasso",)`: only nnInteractive lasso is generated, cached and analysed. |
| Section 0, cell 5 | `SAM2_BOX_MARGIN_PX =` | Pixels added on every side of each SAM 2 box. |
| Section 0, cell 5 | `AUTOSEG_MIN_LESION_VOXELS =` | Separate pieces of a mask smaller than this are treated as stray voxels and get no prompt. |
| Section 0, cell 5 | `AUTOSEG_MAX_LESIONS_PER_PATIENT =` | Largest number of separate tumours per patient that get their own prompt, largest first. |
| Section 0, cell 5 | `AUTOSEG_KEEP_LARGEST_PIECE =` | Keeps only the largest connected piece of what a model returns for one tumour, so that stray voxels cannot stretch the tumour crop. |
| Section 0, cell 5 | `AUTOSEG_SAVE_MASKS =` | Fixed to `True`: successful new masks are always saved as NRRD files in `CAS2_ROOT` or `LASSO_ROOT`. |
| Section 0, cell 3 | `CAS2_ROOT =` | Permanent SAM2 masks: `/content/drive/MyDrive/DATA/CAS2/<patient ID>.nrrd`. |
| Section 0, cell 3 | `LASSO_ROOT =` | Permanent nnInteractive lasso masks: `/content/drive/MyDrive/DATA/nninteraction-lasso/<patient ID>.nrrd`. |
| Section 0, cell 5 | `AUTOSEG_QC_N_PATIENTS =` | Number of patients shown in the overlay figures. |
| Section 0, cell 5 | `AUTOSEG_LOW_DICE =` | A DSC below this value is counted as low in the summary table. |

### Workflow-specific settings

| Location | Search for this text | Plain-language meaning |
|---|---|---|
| Section 0, cell 3 | `WEIGHTS_BASE =` | Folder containing the downloaded RadImageNet checkpoints, used by every section. |
| Section 0, cell 3 | `RADIMAGENET_NORM_BY_BACKBONE =` | Input scaling used for each RadImageNet backbone, in all three sections. |
| Section 0, cell 3 | `RADIMAGENET_RESNET50_USE_OFFICIAL_NORM =` | Chooses whether ResNet50 uses its official RadImageNet input scaling, in all three sections. |
| Section 1, cell 3 | `CHANNEL_POLICY =` | Uses the MRI image as three grey channels and does not pass the mask as a channel. |
| Section 1, cell 3 | `N_FOLDS         =` | Cross-validation folds for Section 1. |
| Section 1, cell 3 | `N_REPEATS       =` | Repeated cross-validation runs for Section 1. |
| Section 1, cell 3 | `EPOCHS_HEAD     =` | Maximum training epochs for the classification head. |
| Section 1, cell 3 | `FROZEN_E2E_EPOCH_SELECTION_FRACTION =` | Fraction of each training fold set aside to choose the number of training epochs. The fold that is scored takes no part in that choice. |
| Section 1, cell 3 | `LR_HEAD         =` | Learning rate for the classification head. |
| Section 1, cell 3 | `BATCH_SIZE      =` | Patients per training batch. |
| Section 1, cell 3 | `DROPOUT         =` | Dropout strength in the classifier head. |
| Section 1, cell 3 | `WEIGHT_DECAY    =` | Regularisation strength for training. |
| Section 1, cell 3 | `EARLY_STOP_PATIENCE =` | Stops training after this many epochs without improvement. |
| Section 1, cell 3 | `USE_POS_WEIGHT   =` | Gives extra weight to the positive class when classes are imbalanced. |
| Section 1, cell 3 | `POS_WEIGHT_SCALE =` | Strength of positive-class weighting. |
| Section 1, cell 3 | `DECISION_THRESHOLD =` | Fallback probability cut-off when top-risk reporting is disabled. |
| Section 2, cell 4 | `BATCH_SIZE_FEATURES =` | Image slices per feature-extraction batch. |
| Section 2, cell 4 | `USE_GITHUB_IDH_PRETRAINED_WEIGHTS =` | Uses the released Choi IDH checkpoint for that backbone. |
| Section 2, cell 4 | `CHANNEL_POLICIES_TO_COMPARE =` | Channel policies tested by the feature-extraction workflow. |
| Section 2, cell 13 | `USE_UNION_FS =` | Uses feature-selector voting from Section 0. |
| Section 2, cell 13 | `USE_POSTHOC_IMPORTANCE_FS =` | Optional final feature-ranking and refit stage. |
| Section 2, cell 13 | `USE_CORR_FILTER =` | Removes highly correlated features before modelling. |
| Section 2, cell 13 | `OUTER_FOLDS =` | Outer folds used to estimate model performance. |
| Section 2, cell 13 | `INNER_FOLDS =` | Inner folds used to tune model settings. |
| Section 2, cell 13 | `OUTER_REPEATS =` | Repeats of the outer cross-validation loop. |
| Section 2, cell 13 | `INNER_REPEATS =` | Repeats of the inner tuning loop. |
| Section 2, cell 13 | `SEARCH_MAX_CANDIDATES =` | Maximum random search candidates when a grid is large. |
| Section 2, cell 13 | `SEARCH_NJOBS =` | CPU workers used for inner model search. |
| Section 2, cell 13 | `SVM_KERNEL_MODE =` | Support-vector-machine kernels to test. |
| Section 2, cell 13 | `BALANCING_METHOD =` | Class-balancing method for machine-learning models. |
| Section 2, cell 13 | `SCALING_METHOD =` | Feature rescaling method before modelling. |
| Section 2, cell 13 | `VARIANCE_THRESHOLD =` | Removes almost-constant features. |
| Section 2, cell 13 | `USE_AUC_FILTER =` | Turns on the single-feature AUC filter. |
| Section 2, cell 13 | `INVERT_LOW_AUC_FEATURES =` | Reverses features whose lower values predict the positive class. |
| Section 2, cell 13 | `MIN_FEATURES_AFTER_AUC  =` | Minimum feature count after the AUC filter. |
| Section 2, cell 13 | `MAX_FEATURES_AFTER_AUC  =` | Maximum feature count after the AUC filter. |
| Section 2, cell 13 | `PCA_VARIANTS =` | Principal-component-analysis options; currently no PCA. |
| Section 2, cell 13 | `PCA_VARIANCE_GRID =` | PCA variance levels available if PCA is enabled. |
| Section 2, cell 13 | `TOP_IMPORTANCE_MODELS =` | Number of top models used in optional post-hoc feature ranking. |
| Section 2, cell 13 | `TOP_FEATURES_PER_MODEL =` | Number of important features nominated by each top model. |
| Section 2, cell 13 | `POSTHOC_MIN_VOTES =` | Votes required during optional post-hoc feature ranking. |
| Section 2, cell 13 | `POSTHOC_MAX_KEEP  =` | Optional maximum number of post-hoc features to keep. |
| Section 2, cell 13 | `FINAL_REFIT_CLASSIFIERS =` | Classifiers used in the optional final refit stage. |
| Section 3, cell 8 | `N_FOLDS         =` | Cross-validation folds for fine-tuning. |
| Section 3, cell 8 | `N_REPEATS       =` | Repeated cross-validation runs for fine-tuning. |
| Section 3, cell 8 | `INNER_VAL_FRACTION =` | Fraction of each training fold reserved for inner validation. |
| Section 3, cell 8 | `EPOCHS_HEAD_MAX       =` | Maximum head-only training epochs before fine-tuning. |
| Section 3, cell 8 | `EPOCHS_FINETUNE_MAX   =` | Maximum fine-tuning epochs. |
| Section 3, cell 8 | `LR_HEAD               =` | Learning rate for the head-only phase. |
| Section 3, cell 8 | `WEIGHT_DECAY_HEAD     =` | Weight decay during the head-only phase. |
| Section 3, cell 8 | `INNER_BACKBONE_LR_CANDIDATES =` | Backbone learning rates tested inside the inner split. |
| Section 3, cell 8 | `LR_FINETUNE_HEAD      =` | Head learning rate during fine-tuning. |
| Section 3, cell 8 | `WEIGHT_DECAY_BACKBONE =` | Weight decay applied to trainable backbone weights. |
| Section 3, cell 8 | `FINETUNE_WARMUP_EPOCHS =` | Warm-up epochs before the full fine-tuning learning rate is used. |
| Section 3, cell 8 | `FINETUNE_MIN_SELECTION_EPOCH =` | Earliest fine-tuning epoch allowed to be selected. |
| Section 3, cell 8 | `FINETUNE_MIN_LR_FACTOR =` | Lowest learning-rate factor reached by cosine decay. |
| Section 3, cell 8 | `L2SP_LAMBDA =` | Strength of the penalty that keeps weights near their pretrained values. |
| Section 3, cell 8 | `EARLY_STOP_PATIENCE_HEAD     =` | Head-only patience for early stopping. |
| Section 3, cell 8 | `EARLY_STOP_PATIENCE_FINETUNE =` | Fine-tuning patience for early stopping. |
| Section 3, cell 8 | `MIN_AUC_IMPROVEMENT          =` | Smallest validation-AUC improvement counted as progress. |
| Section 3, cell 8 | `VAL_LOGLOSS_TIE_TOL          =` | Log-loss tolerance used to break near-ties. |
| Section 3, cell 8 | `GRAD_CLIP_NORM               =` | Gradient clipping threshold. |
| Section 3, cell 8 | `FINETUNE_CLIP_FRACTION_WARN_MAX =` | Warning level for how often gradient clipping is active. |
| Section 3, cell 8 | `USE_EMA   =` | Optional exponential moving average of weights. |
| Section 3, cell 8 | `EMA_DECAY =` | Decay setting used only when EMA is enabled. |
| Section 3, cell 8 | `USE_SWA    =` | Optional stochastic weight averaging. |
| Section 3, cell 8 | `SWA_WINDOW =` | Number of final epochs averaged when SWA is enabled. |
| Section 3, cell 8 | `USE_TTA                     =` | Optional test-time augmentation. |
| Section 3, cell 8 | `USE_TTA_ROTATIONS           =` | Optional rotated test-time copies. |
| Section 3, cell 8 | `TTA_ROTATION_DEGREES        =` | Rotation angles used if rotated test-time augmentation is enabled. |
| Section 3, cell 8 | `USE_TTA_FOR_EPOCH_SELECTION =` | Whether test-time augmentation also affects epoch selection. |
| Section 3, cell 8 | `USE_MIXUP   =` | Optional training on blended patient images and labels. |
| Section 3, cell 8 | `MIXUP_ALPHA =` | Blend strength used only when mixup is enabled. |
| Section 3, cell 8 | `USE_STAGED_UNFREEZE           =` | Optional gradual unfreezing of deeper backbone layers. |
| Section 3, cell 8 | `STAGE1_UNFREEZE_EPOCHS        =` | Epoch at which the next backbone stage can be unfrozen. |
| Section 3, cell 8 | `STAGE2_LR_FACTOR              =` | Learning-rate multiplier for the second unfreezing stage. |
| Section 3, cell 8 | `USE_COMPLEXITY_GATE           =` | Requires more complex choices to outperform the simpler choice by a margin. |
| Section 3, cell 8 | `INNER_COMPLEXITY_MIN_AUC_GAIN =` | Required AUC gain for the complexity gate. |
| Section 3, cell 8 | `USE_MOVEMENT_CAP          =` | Stops fine-tuning when the backbone moves too far from the pretrained weights. |
| Section 3, cell 8 | `FINETUNE_MAX_REL_RMS_MOVE =` | Movement cap: 3 percent relative RMS movement. |
| Section 3, cell 8 | `LABEL_SMOOTHING =` | Soft-label training amount; 0.0 uses hard labels. |
| Section 3, cell 8 | `DROP_PATH_RATE =` | Stochastic-depth rate for supported ResNet-10t backbones. |
| Section 3, cell 8 | `SLICE_POOLING     =` | How multiple slices are combined into one patient prediction. |
| Section 3, cell 8 | `ATTENTION_HIDDEN  =` | Hidden width used only for attention pooling. |
| Section 3, cell 8 | `DENSENET_STAGE1_LAST_DENSE_LAYERS =` | Number of DenseNet denseblock4 layers trainable in stage 1. |
| Section 3, cell 8 | `USE_SMOOTHED_SELECTION =` | Chooses epochs from a smoothed validation curve. |
| Section 3, cell 8 | `SMOOTHING_WINDOW       =` | Window size for smoothed epoch selection. |
| Section 3, cell 8 | `SKIP_FINETUNE_IF_HEAD_WINS   =` | Continues fine-tuning even if the head-only model wins in diagnostics. |
| Section 3, cell 8 | `FORCE_FINAL_REFIT_FINETUNE =` | Forces the final refit to include the fine-tuning phase. |
| Section 3, cell 8 | `FINETUNE_REL_RMS_MOVE_WARN_MIN    =` | Warning threshold for small but visible backbone movement. |
| Section 3, cell 8 | `SKIP_FINETUNE_MIN_POOLED_OOF_DELTA =` | Minimum pooled OOF gain used by the optional skip rule. |
| Section 3, cell 8 | `END2END_MODES =` | Workflow name written to the Section 3 result tables. |
| Section 3, cell 8 | `BATCH_SIZE =` | Patients per training batch in fine-tuning. |
| Section 3, cell 8 | `DROPOUT    =` | Dropout strength in the fine-tuning classifier head. |
| Section 3, cell 8 | `USE_POS_WEIGHT   =` | Gives extra weight to positive cases during fine-tuning. |
| Section 3, cell 8 | `POS_WEIGHT_SCALE =` | Strength of positive-class weighting in fine-tuning. |
| Section 3, cell 8 | `DECISION_THRESHOLD =` | Fallback probability cut-off for fine-tuning. |
| Section 3, cell 8 | `MIN_RADIMAGENET_LOAD_FRACTION =` | Minimum acceptable fraction of pretrained weights loaded. |
| Section 3, cell 8 | `CHECKPOINT_SHA256 =` | Optional checksum records for pretrained checkpoint files. |
| Section 3, cell 8 | `RUN_FINAL_REFIT_AND_HOLDOUT =` | Runs the final development-set refit and holdout evaluation. In an optimization comparison both run only when `OPTIMIZATION_EVALUATE_HOLDOUT` is also `True`. |


### Shared inputs: prepare once, reuse in Sections 1–3

Section 0, cell 3 discovers the folders once, pairs images and masks, loads the label
CSV once, and reports excluded IDs before training. It fixes the development/test
cohort from those paired, labelled patients. The same local-path manifest and label
tables are reused by Sections 1–3; folders and CSV files are not reread there.

Section 0 also prepares every configured CNN input size/slice count. Each patient's
full image and mask are decoded once, then discarded; compact deterministic slices
are saved on local disk with a bounded 128 MiB RAM cache. Later sections reuse these
inputs. Training augmentation, model-specific scaling and feature extraction still
run in their original workflows. Optional automatic segmentation needs its own full
3D inputs, but reuses the shared file manifest and local copies.

Audit files are written under `OUT_ROOT/SHARED_INPUTS_AUDIT`. Missing/ambiguous pairs
and patients without labels are reported as exclusions shared by all workflows.
Unreadable volumes or invalid preprocessing stop in Section 0 before training.
After changing filenames, folders, labels or preprocessing settings, rerun from
Section 0, cell 3 (preferably in a fresh runtime). This rebuilds the shared inputs;
rerunning a later section alone deliberately reuses the prepared cohort.


### Full analyses with generated segmentations

The generation switches `RUN_AUTOSEG_NNINTERACTIVE` and `RUN_AUTOSEG_SAM2` only control
cache-first resegmentation and agreement measurements. nnInteractive generates only
lasso; `NNINTERACTIVE_ANALYSIS_PROMPT_TYPE="lasso"` is fixed. The independent switches
`ANALYZE_NNINTERACTIVE_MASKS` and `ANALYZE_SAM2_MASKS` add full downstream analyses.
The attached notebook's analysis and generation switch values are retained.

After resegmentation, Section 0, cells 6–7 validate and prepare each selected mask
set. Original-mask analyses then run as usual in Sections 1–5. A successful-cell
capture hook records their exact code; Section 6 replays the same enabled workflows
in isolated namespaces for the selected generated masks. This preserves the same
patient cohort, labels, CV split logic and starting random state. Crops, central
slices and ROI normalization are recomputed independently using each mask set.
Run all cells in order, including Section 6. The additional analyses run sequentially.

When generation is disabled but analysis is enabled, saved full-cohort masks are
loaded from `CAS2_ROOT/<patient ID>.nrrd` or `LASSO_ROOT/<patient ID>.nrrd`.
Before any inference, an existing mask is loaded directly. Only a missing mask triggers
model loading, generation and an atomic save to Drive. Matching legacy masks under
`AUTOSEG_OUT_DIR/masks/` (or `masks__pilot/`) are moved into the new cache by copying
once, so prior completed segmentations can be reused. A corrupt, empty or mismatched
cache is reported as a failure and is not overwritten. Existing caches persist across
runtimes; to deliberately regenerate a patient after changing segmentation settings,
remove or rename that patient's cached mask and any matching legacy mask first.
Set `AUTOSEG_PILOT_N_PATIENTS=None` when generating masks for full analysis. Every labelled
original-cohort patient must have a successful, nonempty mask. Incomplete or failed
sets stop before training; original masks are never substituted silently. Masks for
patients outside the original labelled cohort are ignored.

Original results keep their existing folders. Additional results go under
`OUT_ROOT/SEGMENTATION_ANALYSES/nninteractive_lasso/` and/or
`OUT_ROOT/SEGMENTATION_ANALYSES/sam2_box_per_slice/`, with their own caches, feature
exports, trained models and summary tables. After Section 6, the main results file
`COMBINED_OVERVIEW_DIR/combined_pipeline_overview.csv` includes original and all selected
generated-mask results, identified by `segmentation_method` (`original`,
`sam2_box_per_slice` for CAS2/SAM2, or `nninteractive_lasso`). Per-test-set overview
files are refreshed too. The same combined results remain available in
`OUT_ROOT/SEGMENTATION_ANALYSES/combined_segmentation_overview.csv`.
Original images are copied from Drive once, and labels
are read once; each mask method prepares its own ROI-dependent inputs from local files.

These are prompt-assisted resegmentations: their location hints come from the
original masks, as documented in the agreement check.


## Section 0, cell 2 - Settings and Google Drive


In [ ]:
# Section 0, cell 3: Settings, package installation and shared helper functions

# ══════════════════════════════════════════════════════════════════════════
# MAJOR/ESSENTIAL SETTINGS: EDIT THESE, THEN RUN ALL
# ══════════════════════════════════════════════════════════════════════════
# Colab maps "My Drive" to "/content/drive/MyDrive".
#
# DEVELOPMENT_ROOT: one folder or a list, combined into ONE development set.
# Folders are read sequentially. Splitting large datasets may reduce the observed
# first-read RAM spike (>50 GB with approximately 350+ files in one folder).
#
# TEST_ROOT: ONE or TWO independent holdout sets, named after their folders.
# The model is refitted once on all development patients, then each test set is
# scored once, with separate result tables.
#
# Both roots are searched recursively. Each patient's paired image and mask NRRD
# files must share one folder; duplicate patients across folders stop the run.
#
# Any folder path containing "none" (case-insensitive) is ignored with a warning.
# Use this to disable an optional folder. Each root needs at least one real folder.

DEVELOPMENT_ROOT = [
    "/content/drive/MyDrive//DATA/DEVELOPMENT-150",
    "/content/drive/MyDrive/DATA/none",
]
TEST_ROOT = [
    "/content/drive/MyDrive/DATA/TEST-40",
    "/content/drive/MyDrive/none",
]

# ── Results and labels ────────────────────────────────────────────────────
# OUT_ROOT receives all result tables, feature CSVs, model files and overviews.
# LABELS_CSV: column A = patient_id; column B = binary outcome label (0/1).
# Each run analyses one label file. To analyse another, change LABELS_CSV and
# rerun; use a different OUT_ROOT to preserve earlier summary tables.
OUT_ROOT = "/content/drive/MyDrive/DATA/RESULTS/RESULTS-AUTOsegm"
LABELS_CSV = "/content/drive/MyDrive/DATA/labels.csv"

# ── Slices per patient ────────────────────────────────────────────────────
# 1 = axial; 2 = axial + sagittal; 3 = axial + sagittal + coronal.
# Each plane uses its slice with the largest tumour cross-section.
N_SLICES         = 3

# ── Workflow switches: False skips the corresponding section ──────────────
RUN_FROZEN_CNN_E2E    = True   # Section 1, "frozen": frozen-backbone end-to-end CNN
RUN_CNN_LR            = False  # Section 2, "efficient_ML": frozen-CNN features + classical ML
RUN_CNN_FINETUNE_E2E  = True  # Section 3, "finetune": fine-tuned end-to-end CNN

# ── CNN architectures shared by all three sections ────────────────────────
# Sections 1 and 3 report unsupported architectures and skip them.
# Section 2 extracts features with all listed architectures.
SHARED_BACKBONES = [
#   "radimagenet_densenet121",
#    "imagenet_densenet121",
#    "radimagenet_inceptionv3",
#    "imagenet_inceptionv3",
    "radimagenet_resnet10t",
    "imagenet_resnet10t",
#   "radimagenet_resnet18",
#   "imagenet_resnet18",
#    "radimagenet_resnet50",
#    "imagenet_resnet50",
#    "choi_idh",
]

# ── Optional segmentation comparison: Section 0, cell 5 ───────────────────
# Generates segmentations using prompts derived from the existing masks and
# reports Dice Similarity Coefficient (DSC) against those masks.
# Original-mask classification always remains included. The ANALYZE switches below
# independently add full downstream analyses using selected generated masks.
# nnInteractive: 3D, one prompt on one slice per tumour.
# SAM 2.1: 2D, one box on every tumour slice.
# Set both switches to False to skip this cell and its extra package installation.
RUN_AUTOSEG_NNINTERACTIVE = True
RUN_AUTOSEG_SAM2          = False

ANALYZE_NNINTERACTIVE_MASKS = True
ANALYZE_SAM2_MASKS          = False

# nnInteractive is permanently restricted to lasso for generation and analysis.
NNINTERACTIVE_ANALYSIS_PROMPT_TYPE = "lasso"  # fixed; other prompt types are unsupported

# Permanent per-patient NRRD caches on mounted Google Drive.
CAS2_ROOT = "/content/drive/MyDrive/DATA/AUTOMATIC-SEG-FILES/CAS2"
LASSO_ROOT = "/content/drive/MyDrive/DATA/AUTOMATIC-SEG-FILES/nninteraction-lasso"

if type(ANALYZE_NNINTERACTIVE_MASKS) is not bool or type(ANALYZE_SAM2_MASKS) is not bool:
    raise TypeError("ANALYZE_NNINTERACTIVE_MASKS and ANALYZE_SAM2_MASKS must be True or False.")
NNINTERACTIVE_ANALYSIS_PROMPT_TYPE = str(NNINTERACTIVE_ANALYSIS_PROMPT_TYPE).strip().lower()
if NNINTERACTIVE_ANALYSIS_PROMPT_TYPE != "lasso":
    raise ValueError("Only nnInteractive lasso is supported for generation and analysis.")

# Download weights manually from the links below, or enable automatic downloading
# of missing weight folders in Section 0, cell 5. Drive downloads are retained.
#
# nnInteractive v1.0 (~410 MB; CC BY-NC-SA 4.0, non-commercial):
# https://huggingface.co/MIC-DKFZ/nnInteractive/tree/main/nnInteractive_v1.0
# Required layout: plans.json, dataset.json, inference_session_class.json,
#                 fold_0/checkpoint_final.pth
#
# SAM 2.1 Hiera Base+ (~325 MB; Apache-2.0):
# https://huggingface.co/facebook/sam2.1-hiera-base-plus/tree/main
# Required files: config.json, model.safetensors,
#                 preprocessor_config.json, processor_config.json
NNINTERACTIVE_WEIGHTS_PATH = "/content/drive/MyDrive/DATA/weights/nnInteractive_v1.0"
SAM2_WEIGHTS_PATH          = "/content/drive/MyDrive/DATA/weights/sam2.1-hiera-base-plus"
AUTOSEG_DOWNLOAD_MISSING_WEIGHTS = False

# ════════════════════════ END OF MAJOR/ESSENTIAL SETTINGS ═════════════════

# ── Other paths: pretrained weights and shared local cache ─────────────────
# All notebook paths are defined above or here.
# Download RadImageNet checkpoints manually into WEIGHTS_BASE.
# ResNet-10t requires exactly "RadImageNet_ResNet10t.pth" (lowercase t, .pth).
WEIGHTS_BASE = "/content/drive/MyDrive/DATA/weights"
RADIMAGENET_RESNET10T_WEIGHTS_PATH   = f"{WEIGHTS_BASE}/RadImageNet_ResNet10t.pth"
RADIMAGENET_RESNET18_WEIGHTS_PATH    = f"{WEIGHTS_BASE}/RadImageNet_ResNet18.pt"
RADIMAGENET_RESNET50_WEIGHTS_PATH    = f"{WEIGHTS_BASE}/RadImageNet_ResNet50.pt"
RADIMAGENET_DENSENET121_WEIGHTS_PATH = f"{WEIGHTS_BASE}/RadImageNet_DenseNet121.pt"

# Section 2 may also use InceptionV3.
RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH = f"{WEIGHTS_BASE}/RadImageNet_InceptionV3.pt"

# Shared raw-volume cache; per-patient subdirectories prevent basename collisions.
LOCAL_CACHE_DIR = "/content/IMAGE_LOCAL_CACHE_PATIENT_SAFE"

# ── Whole-notebook timer: includes package installation and Drive setup ────
import time as _notebook_time
from datetime import datetime as _notebook_datetime

_NOTEBOOK_RUN_T0_MONOTONIC = _notebook_time.perf_counter()
_NOTEBOOK_RUN_STARTED_AT = _notebook_datetime.now().astimezone()
print(
    "[RUN TIMER] Notebook started:",
    _NOTEBOOK_RUN_STARTED_AT.isoformat(timespec="seconds"),
)

# ── Package versions ──────────────────────────────────────────────────────
# Install required packages once here before their imports.
# Downstream version checks must use this mapping rather than repeat literals.
PACKAGE_PINS = {
    "SimpleITK": "2.5.5",
    "pandas": "2.2.3",
    "scikit-learn": "1.9.0",
    "tqdm": "4.69.0",
    "timm": "1.0.28",
}

# The workflow switches RUN_FROZEN_CNN_E2E, RUN_CNN_LR and RUN_CNN_FINETUNE_E2E are set
# in MAJOR/ESSENTIAL SETTINGS at the top of this cell.

print(
    "[WORKFLOW SWITCHES] "
    f"frozen={RUN_FROZEN_CNN_E2E} | "
    f"efficient_ML={RUN_CNN_LR} | "
    f"finetune={RUN_CNN_FINETUNE_E2E}"
)

# ── Packages of the optional automatic segmentation check ──────────────────
# nnInteractive needs nnU-Net. The newest nnU-Net requires timm below 1.0.23 and would
# replace the pinned timm 1.0.28, so nnU-Net 2.7.0 and its network package 0.4.2 are
# pinned here: they accept timm 1.0.28. Everything is installed in ONE line together
# with the pinned packages above, so the installer checks all pins at once. Nothing is
# added when the two RUN_AUTOSEG switches are False.
AUTOSEG_PACKAGE_PINS = {
    "nnInteractive": "2.6.0",
    "nnunetv2": "2.7.0",
    "dynamic-network-architectures": "0.4.2",
    "transformers": "5.18.0",
}
_AUTOSEG_PIP_PACKAGES = ""
if RUN_AUTOSEG_NNINTERACTIVE:
    _AUTOSEG_PIP_PACKAGES += " nnInteractive==2.6.0 nnunetv2==2.7.0 dynamic-network-architectures==0.4.2"
if RUN_AUTOSEG_SAM2:
    _AUTOSEG_PIP_PACKAGES += " transformers==5.18.0"

!pip -q install SimpleITK==2.5.5 nibabel pandas==2.2.3 scikit-learn==1.9.0 tqdm==4.69.0 timm==1.0.28 {_AUTOSEG_PIP_PACKAGES}
!pip -q install imbalanced-learn xgboost lightgbm catboost shap


import os, sys, random, re, warnings
import numpy as np

# ── Reproducibility: the single base seed for every workflow ───────────────
GLOBAL_SEED = 42

def seed_everything(seed=None):
    """Seed Python, NumPy and PyTorch from one notebook-wide base seed."""
    seed = GLOBAL_SEED if seed is None else int(seed)
    random.seed(seed)
    np.random.seed(seed)
    import torch as _torch
    _torch.manual_seed(seed)
    _torch.cuda.manual_seed_all(seed)


# ── Hugging Face access key (optional) ────────────────────────────────────
# Public pretrained weights download without a key; a token only raises rate limits.
# Prefer a Colab Secret named HF_TOKEN to the literal below.
HF_TOKEN = ""  # Leave blank unless no Colab Secret is set.

def configure_optional_hf_token():
    """Use the first available Hugging Face access key, or download public files without one."""
    token = os.environ.get("HF_TOKEN", "").strip()
    source = "environment variable"

    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get("HF_TOKEN") or "").strip()
            source = "Colab Secret"
        except Exception:
            token = ""

    if not token:
        token = str(globals().get("HF_TOKEN", "") or "").strip()
        source = "notebook HF_TOKEN line"

    if token:
        # Export before any download library is imported.
        os.environ["HF_TOKEN"] = token
        print(f"[Hugging Face] Optional access key found in: {source}.")
        return True

    print("[Hugging Face] No HF_TOKEN found; public pretrained weights will be downloaded without an access key.")
    return False

HF_AUTHENTICATED = configure_optional_hf_token()

# Longer timeouts for slow weight downloads.
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

# Single-threaded BLAS: reproducible cross-validation, no oversubscription of cores.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

def ensure_drive_mounted(mountpoint="/content/drive"):
    """Connect Google Drive only when it is not already connected."""
    global IN_COLAB
    if os.path.exists(os.path.join(mountpoint, "MyDrive")):
        IN_COLAB = True
        return True
    try:
        from google.colab import drive
        drive.mount(mountpoint)            # Mount only when needed.
        IN_COLAB = True
        return True
    except Exception as _e:
        IN_COLAB = False
        print("[Google Drive] Could not connect. The notebook may not be running in Colab:", repr(_e))
        return False

ensure_drive_mounted()

# SHARED_BACKBONES, the CNN architectures shared by all three sections, is set in
# MAJOR/ESSENTIAL SETTINGS at the top of this cell.

# ── Input scaling per backbone: the only definition, shared by all three sections ──
# Every pretrained network expects its input in the value range it was trained on. A
# mismatch degrades the features silently, without raising.
#   "imagenet"        ImageNet mean/std normalisation
#   "unit_interval"   values stay in [0, 1]
#   "minus_one_one"   [0, 1] -> [-1, 1], i.e. (x-127.5)*2/255, per RadImageNet's
#                     official pytorch_example.ipynb
#   "caffe_over_255"  subtract RADIMAGENET_CAFFE_MEAN_BGR from [0, 1]
# radimagenet_densenet121 uses "minus_one_one", the transform in the BMEII-AI reference
# notebook, which gave the best frozen-feature effective rank and OOF head AUC on the
# development set. radimagenet_resnet50 was not measured and keeps its own setting
# behind RADIMAGENET_RESNET50_USE_OFFICIAL_NORM. radimagenet_resnet18 and
# radimagenet_resnet10t are third-party re-trains on the "imagenet" convention. Every
# imagenet_* backbone uses "imagenet" and is not listed here.
RADIMAGENET_RESNET50_USE_OFFICIAL_NORM = False
RADIMAGENET_NORM_BY_BACKBONE = {
    "radimagenet_resnet18":    "imagenet",
    "radimagenet_resnet50":    ("minus_one_one" if RADIMAGENET_RESNET50_USE_OFFICIAL_NORM
                                else "unit_interval"),
    "radimagenet_resnet10t":   "imagenet",
    "radimagenet_densenet121": "minus_one_one",
    "radimagenet_inceptionv3": "unit_interval",
}

_VALID_INPUT_SCALINGS = ("imagenet", "unit_interval", "minus_one_one", "caffe_over_255")
for _bb, _scaling in RADIMAGENET_NORM_BY_BACKBONE.items():
    if str(_scaling).lower() not in _VALID_INPUT_SCALINGS:
        raise ValueError(
            f"RADIMAGENET_NORM_BY_BACKBONE[{_bb!r}] is {_scaling!r}. Valid: {_VALID_INPUT_SCALINGS}"
        )

# Common spelling variants of the RadImageNet model names.
_RADIMAGENET_NAME_ALIASES = {
    "radimagenet": "radimagenet_resnet50",
    "radimagenet50": "radimagenet_resnet50",
    "radimagenet121": "radimagenet_densenet121",
    "radimagenet_densenet": "radimagenet_densenet121",
    "radimagenet_inception": "radimagenet_inceptionv3",
    "radimagenet_inception_v3": "radimagenet_inceptionv3",
    "radimagenet18": "radimagenet_resnet18",
    "radimagenet10t": "radimagenet_resnet10t",
    "radimagenet_resnet10": "radimagenet_resnet10t",
}


def radimagenet_norm_for(backbone_name):
    """Input scaling of one backbone, read from RADIMAGENET_NORM_BY_BACKBONE."""
    name = str(backbone_name).lower()
    name = _RADIMAGENET_NAME_ALIASES.get(name, name)
    if name in RADIMAGENET_NORM_BY_BACKBONE:
        return str(RADIMAGENET_NORM_BY_BACKBONE[name]).lower()
    # imagenet_* backbones must receive ImageNet mean/std normalisation. Feeding them
    # raw [0, 1] collapses the representation to an effective rank near 1.
    if name.startswith("imagenet"):
        return "imagenet"
    return "unit_interval"


# ImageNet mean/std and the RadImageNet "caffe" offsets on the [0, 1] scale. Keras
# preprocess_input(mode='caffe') subtracts [103.939, 116.779, 123.68] in BGR from 0-255
# data and RadImageNet then rescales by 1/255. Dividing the offsets by 255 folds both
# into one subtraction on [0, 1] data. BGR is the order the converted conv0 filters expect.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RADIMAGENET_CAFFE_MEAN_BGR = [103.939 / 255.0, 116.779 / 255.0, 123.68 / 255.0]


def apply_input_scaling(xs, norm):
    """Scale slices from [0, 1] to the value range one backbone expects.

    xs has shape (..., 3, H, W). Section 1, Section 2 and Section 3 all call this one
    function, so a scaling name means the same arithmetic in every workflow. torch is
    resolved at call time, after the active workflow has imported it.
    """
    mode = str(norm).lower()
    if mode in ("unit_interval", "none"):
        return xs
    if mode == "imagenet":
        mean = torch.as_tensor(IMAGENET_MEAN, dtype=xs.dtype, device=xs.device).view(1, 3, 1, 1)
        std = torch.as_tensor(IMAGENET_STD, dtype=xs.dtype, device=xs.device).view(1, 3, 1, 1)
        return (xs - mean) / std
    if mode == "minus_one_one":
        return xs * 2.0 - 1.0
    if mode == "caffe_over_255":
        mean = torch.as_tensor(
            RADIMAGENET_CAFFE_MEAN_BGR, dtype=xs.dtype, device=xs.device
        ).view(1, 3, 1, 1)
        return xs - mean
    raise ValueError(f"Unknown input scaling {norm!r}. Valid: {_VALID_INPUT_SCALINGS}")


# ── RadImageNet checkpoint files: resolved once, shared by all three sections ──────
# A path set at the top of this cell is used as it is when that file exists. When it
# does not, WEIGHTS_BASE and its direct sub-folders are searched for a file whose name
# matches the patterns below, without regard to letter case. Exactly one match is
# accepted. Two or more matches stop the run for a backbone listed in SHARED_BACKBONES,
# because choosing among them silently could load the wrong pretrained network.
def find_weight_file(preferred_path, patterns, search_dirs=None, required=True):
    """Resolve a checkpoint path without silently choosing among ambiguous files."""
    import fnmatch
    from pathlib import Path as _Path

    preferred = _Path(str(preferred_path)).expanduser()
    if preferred.exists():
        return str(preferred)

    search_roots = []
    if preferred.parent and str(preferred.parent) not in ("", "."):
        search_roots.append(preferred.parent)
    if search_dirs:
        search_roots.extend(_Path(str(d)).expanduser() for d in search_dirs)

    _patterns = [str(p).lower() for p in patterns]
    matches = []
    seen_roots = set()
    for root in search_roots:
        if not root.is_dir():
            continue
        root_key = str(root.resolve())
        if root_key in seen_roots:
            continue
        seen_roots.add(root_key)
        # The folder itself plus one level below it. A full recursive scan times out on Drive.
        candidates = []
        for entry in root.iterdir():
            if entry.is_file():
                candidates.append(entry)
            elif entry.is_dir():
                candidates.extend(p for p in entry.iterdir() if p.is_file())
        for candidate in candidates:
            if any(fnmatch.fnmatchcase(candidate.name.lower(), pat) for pat in _patterns):
                matches.append(candidate)

    unique = sorted({str(m.resolve()) for m in matches})
    if len(unique) == 1:
        print(f"[checkpoint resolver] Exact path missing; using sole fallback match: {unique[0]}")
        return unique[0]
    if len(unique) > 1:
        _message = (
            "Ambiguous checkpoint fallback search. Refusing to choose automatically. "
            f"Preferred path: {preferred}. Candidates: {unique}"
        )
        if required:
            raise RuntimeError(_message)
        print("[checkpoint resolver][note] " + _message
              + " This backbone is not listed in SHARED_BACKBONES, so the run continues.")
    return str(preferred)


_LISTED_BACKBONES = {str(b).lower() for b in SHARED_BACKBONES}
RADIMAGENET_RESNET10T_WEIGHTS_PATH = find_weight_file(
    RADIMAGENET_RESNET10T_WEIGHTS_PATH,
    ["*resnet*10*.pt", "*resnet*10*.pth", "*resnet10*"],
    [WEIGHTS_BASE], required="radimagenet_resnet10t" in _LISTED_BACKBONES,
)
RADIMAGENET_RESNET18_WEIGHTS_PATH = find_weight_file(
    RADIMAGENET_RESNET18_WEIGHTS_PATH,
    ["*resnet*18*.pt", "*resnet*18*.pth", "*resnet18*"],
    [WEIGHTS_BASE], required="radimagenet_resnet18" in _LISTED_BACKBONES,
)
RADIMAGENET_RESNET50_WEIGHTS_PATH = find_weight_file(
    RADIMAGENET_RESNET50_WEIGHTS_PATH,
    ["*resnet*50*.pt", "*resnet*50*.pth", "*resnet50*"],
    [WEIGHTS_BASE], required="radimagenet_resnet50" in _LISTED_BACKBONES,
)
RADIMAGENET_DENSENET121_WEIGHTS_PATH = find_weight_file(
    RADIMAGENET_DENSENET121_WEIGHTS_PATH,
    ["*densenet*121*.pt", "*densenet*121*.pth"],
    [WEIGHTS_BASE], required="radimagenet_densenet121" in _LISTED_BACKBONES,
)
RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH = find_weight_file(
    RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH,
    ["*inception*v3*.pt", "*inception*v3*.pth"],
    [WEIGHTS_BASE], required="radimagenet_inceptionv3" in _LISTED_BACKBONES,
)
# Section 2 refers to the ResNet-50 checkpoint by this alias.
RADIMAGENET_WEIGHTS_PATH = RADIMAGENET_RESNET50_WEIGHTS_PATH

# ── Outcome analysed in this run ───────────────────────────────────────────
# Built automatically from LABELS_CSV at the top of this cell; nothing to edit here.
# The outcome is named after the label file (labels.csv -> "labels",
# labels-35.csv -> "labels_35"). That name fills the "outcome" column of the
# section result tables and appears in output file names.
if not isinstance(LABELS_CSV, str) or not LABELS_CSV.strip():
    raise ValueError(
        "LABELS_CSV at the top of this cell must be the path of ONE label file, "
        "for example \"/content/drive/MyDrive/DATA/labels.csv\". "
        f"Found: {LABELS_CSV!r}"
    )
LABELS_CSV = LABELS_CSV.strip()
OUTCOME_NAME = (
    re.sub(r"[^A-Za-z0-9]+", "_", os.path.splitext(os.path.basename(LABELS_CSV))[0])
    .strip("_").lower()
    or "labels"
)
# The sections read the label file through these two names.
LABELS_CONFIGS = [{"tag": OUTCOME_NAME, "path": LABELS_CSV, "enabled": True}]
SHARED_OUTCOMES = [OUTCOME_NAME]


# ── Development / holdout split: the only definition ──────────────────────
# The split is set by DEVELOPMENT_ROOT and TEST_ROOT at the top of this cell. Every
# folder is scanned once further down this cell (scan_split_folders()); all three
# sections read that one record, and get_or_create_patient_split() below stops the
# run if they disagree.
SPLIT_SET_DEVELOPMENT = "development"
SPLIT_SET_TEST = "test"


def split_folder_roots(value, set_name="folder input"):
    """Return configured folder paths in order, ignoring entries whose path contains "none"."""
    if value is None:
        candidates = []
    elif isinstance(value, (list, tuple)):
        candidates = list(value)
    else:
        candidates = [value]

    roots = []
    for slot, candidate in enumerate(candidates, start=1):
        if candidate is None or not str(candidate).strip():
            continue
        path = str(candidate).strip()
        # "none" anywhere in the path (any letter case) marks an unused entry, for example
        # "none", "/content/drive/MyDrive/project/none" or ".../TEST-NONE". The path is
        # not looked up on Drive at all.
        if "none" in path.casefold():
            print(
                f"[PATIENT SPLIT][WARNING] Ignoring {set_name} folder entry {slot}: "
                f"the path {path!r} contains 'none'."
            )
            continue
        roots.append(path)

    if not roots:
        raise ValueError(
            f"[PATIENT SPLIT] {set_name} needs at least one real folder; "
            'entries whose path contains "none" are ignored.'
        )
    if len({os.path.normpath(r) for r in roots}) != len(roots):
        raise ValueError(f"[PATIENT SPLIT] The same folder is listed more than once: {roots}")
    return roots

def split_folder_label(split_set, index, n_roots):
    """'DEVELOPMENT folder' for a single folder, 'DEVELOPMENT folder 1 of 2' for a list."""
    name = f"{str(split_set).upper()} folder"
    return name if int(n_roots) == 1 else f"{name} {int(index)} of {int(n_roots)}"


# Each set maps to a LIST of folders, read in the order given. The DEVELOPMENT folders
# form one set; every TEST folder is a separate holdout test set (see below).
SPLIT_FOLDERS = {
    SPLIT_SET_DEVELOPMENT: split_folder_roots(DEVELOPMENT_ROOT, "DEVELOPMENT_ROOT"),
    SPLIT_SET_TEST: split_folder_roots(TEST_ROOT, "TEST_ROOT"),
}
# One-line text of each set's folder(s), for messages and the patient-split record.
DEVELOPMENT_ROOT_TEXT = " + ".join(SPLIT_FOLDERS[SPLIT_SET_DEVELOPMENT])
TEST_ROOT_TEXT = " + ".join(SPLIT_FOLDERS[SPLIT_SET_TEST])

# ── Holdout test sets: one per TEST_ROOT folder ────────────────────────────
# A test set is named after its folder. The name fills the "test_set" column of the
# result tables and, when there are two test sets, is added to output file names.
TEST_SET_ROOTS = list(SPLIT_FOLDERS[SPLIT_SET_TEST])
TEST_SET_NAMES = [
    (r.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1].strip() or f"TEST-{i}")
    for i, r in enumerate(TEST_SET_ROOTS, start=1)
]


def _test_set_token(test_set_name):
    """A test-set name reduced to letters, digits and underscores, for file names."""
    return re.sub(r"[^A-Za-z0-9]+", "_", str(test_set_name)).strip("_") or "TEST"


if len({_test_set_token(n).lower() for n in TEST_SET_NAMES}) != len(TEST_SET_NAMES):
    raise ValueError(
        "[PATIENT SPLIT] The TEST_ROOT folders must have different folder names, because each "
        f"test set is named after its folder. Found: {TEST_SET_NAMES}"
    )
TEST_SET_ROOT_BY_NAME = dict(zip(TEST_SET_NAMES, TEST_SET_ROOTS))

# ── Image preprocessing: the only definition, shared by all sections ──────
# Central planes, physical-square tumour crop and intensity scaling are identical everywhere.
CNN_INPUT_SIZE_SHARED = 224
# N_SLICES, the number of slices per patient, is set in MAJOR/ESSENTIAL SETTINGS at the
# top of this cell and checked below.

# ── Slice visual-QC controls ──────────────────────────────────────────────
SHOW_ANALYSED_SLICES = True
QC_N_PATIENTS        = 35
QC_PATIENTS_PER_FIG  = 1

if isinstance(N_SLICES, (bool, np.bool_)) or N_SLICES not in (1, 2, 3):
    raise ValueError(f"N_SLICES must be 1, 2 or 3; got {N_SLICES}")
SLICE_PLANES = ("axial", "sagittal", "coronal")[:int(N_SLICES)]


# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION MODE - compare image size, slice count and augmentation level on selected backbones
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION = False -> run every configured backbone with one setting,
#   nested CV, final refit and holdout evaluation.
#
# OPTIMIZATION = True -> enabled workflows use the comparison settings below.
#   Frozen end-to-end CNN and fine-tuned end-to-end CNN run once for each
#   setting combination in OPTIMIZATION_BACKBONES. Frozen-CNN features + downstream ML run once for each unique
#   image representation (input size x slice count) on the same backbones; augmentation
#   is a training-only factor and is not applicable to a frozen feature extractor.
#   In every workflow:
#     * The comparison is decided from development data alone. With
#       OPTIMIZATION_EVALUATE_HOLDOUT set to False (the default) no setting combination is
#       scored on a holdout test set, so every test set stays unseen while settings are
#       chosen. Sections 1 and 3 then also skip the final refit on all development patients.
#     * Each setting combination uses the same seed and therefore the same fold assignment,
#       so settings are compared patient-by-patient rather than as unrelated runs.
#     * The main readout is the pooled development OOF AUC for each setting combination
#       and its paired delta against the reference setting, with a bootstrap CI.
#
# Accept a setting combination only when the paired delta clears OPTIMIZATION_MIN_DELTA and its CI
# excludes zero. On ~150 patients a smaller difference is treated as random variation.
OPTIMIZATION = False

# Holdout use in OPTIMIZATION mode.
#   False  no holdout test set is scored for any setting combination. Sections 1 and 3
#          skip the final refit and save no final-refit model. Choose the settings from
#          the development results, then run once with OPTIMIZATION set to False to
#          score the holdout for the chosen settings only.
#   True   every setting combination is refitted and scored on every holdout test set.
#          This shows holdout results while settings are still being chosen.
# The switch has no effect when OPTIMIZATION is False.
OPTIMIZATION_EVALUATE_HOLDOUT = False


def holdout_evaluation_enabled():
    """True unless an OPTIMIZATION run keeps the holdout test sets unseen."""
    return (not bool(OPTIMIZATION)) or bool(OPTIMIZATION_EVALUATE_HOLDOUT)


class HoldoutNotEvaluated(dict):
    """Holdout metrics of a run that did not score the holdout.

    Any metric that is not set explicitly reads as NaN. The result tables therefore keep
    their columns and their one row per test set, with empty holdout cells.
    """
    def __missing__(self, key):
        return float("nan")


HOLDOUT_NOT_EVALUATED_TEXT = "holdout not evaluated (OPTIMIZATION_EVALUATE_HOLDOUT is False)"


def rows_for_display(df):
    """A result table as it is shown on screen when the holdout was not scored.

    Such a run stores every result once per test set, each time with the same development
    numbers and empty holdout cells. On screen every result is shown once and the
    test_set column is left out. The saved CSV files keep all rows. The table is
    returned unchanged when the run scored the holdout or has one test set.
    """
    if holdout_evaluation_enabled() or "test_set" not in df.columns or len(TEST_SET_NAMES) <= 1:
        return df
    first_test_set = df["test_set"].astype(str) == str(TEST_SET_NAMES[0])
    if not first_test_set.any():
        return df
    return df[first_test_set].drop(columns="test_set")

# ── The three settings compared ──────────────────────────────────────────────────
# Every combination of these lists is tested. To test one setting at a time, as suited
# to a cohort of about 150 patients, collapse the other lists to a
# single level, e.g. OPTIMIZATION_INPUT_SIZES = (224,).
OPTIMIZATION_INPUT_SIZES = (128, 224)      # CNN_INPUT_SIZE_SHARED per arm
OPTIMIZATION_N_SLICES    = (1,)         # fixed at one slice for this optimization run
OPTIMIZATION_AUG_LEVELS  = ("FLIPS", "BASIC", "COMPLEX")

# Stop immediately for the common one-item tuple mistake: (1) is an int, while (1,) is iterable.
if not isinstance(OPTIMIZATION_N_SLICES, (tuple, list)):
    raise TypeError("OPTIMIZATION_N_SLICES must be a tuple/list, e.g. (1,) for one slice.")

# ── Tumour context representation: MANUAL GLOBAL SWITCH ONLY ────────────────
# Choose one mode here before running the notebook; it is not compared automatically.
# The same mode is then used by every setting combination and every enabled image
# workflow (frozen CNN, CNN features/ML and fine-tuning).
#
# "MIN_BBOX": keep MRI signal inside the minimal bounding-box crop.
# "ROI_ONLY": keep the same crop/scale/slices, but replace pixels outside the tumour
#             mask with ROI_ONLY_BACKGROUND_VALUE.
TUMOR_CONTEXT_MODE = "MIN_BBOX"  # "MIN_BBOX" or "ROI_ONLY"
ROI_ONLY_BACKGROUND_VALUE = 0.5
_VALID_TUMOR_CONTEXT_MODES = ("MIN_BBOX", "ROI_ONLY")
if str(TUMOR_CONTEXT_MODE).upper() not in _VALID_TUMOR_CONTEXT_MODES:
    raise ValueError(
        f"Unknown TUMOR_CONTEXT_MODE={TUMOR_CONTEXT_MODE!r}; "
        f"valid: {_VALID_TUMOR_CONTEXT_MODES}"
    )
if not np.isfinite(float(ROI_ONLY_BACKGROUND_VALUE)) or not (0.0 <= float(ROI_ONLY_BACKGROUND_VALUE) <= 1.0):
    raise ValueError(
        f"ROI_ONLY_BACKGROUND_VALUE must be in [0,1]; got {ROI_ONLY_BACKGROUND_VALUE}"
    )

#   Available augmentation levels. These names are used by TRAIN_AUGMENTATION_LEVEL, OPTIMIZATION_AUG_LEVELS and the augmentation functions:
#     "NONE"           no augmentation at all
#     "FLIPS"          random horizontal and vertical flips; current standard setting
#     "BASIC"          flips plus rotation up to AUG_ROTATE_DEG degrees
#     "COMPLEX"        BASIC plus random scaling and shifting
#     "COMPLEX_JITTER" COMPLEX plus random gamma, brightness, contrast and additive noise
#   Current optimization compares FLIPS, BASIC and COMPLEX at each image size listed in OPTIMIZATION_INPUT_SIZES.
#   To test intensity jitter as an additional level as well:
#     OPTIMIZATION_AUG_LEVELS = ("FLIPS", "BASIC", "COMPLEX", "COMPLEX_JITTER")

# The current standard setting is added when needed and measured on the same folds
# as the other settings, allowing paired deltas against it.

OPTIMIZATION_INCLUDE_REFERENCE_ARM = True

# ── Comparison scope and statistics ───────────────────────────────────────────────
# Two structurally different backbones, not one: in the 77c->77d comparison the
# 224->128 change cost the deep networks 0.03-0.05 OOF AUC while BOTH ResNet-10t
# variants gained 0.02-0.03. A single-backbone comparison would have selected the wrong
# resolution for half the model set.
OPTIMIZATION_BACKBONES = ("radimagenet_densenet121", "radimagenet_resnet10t", "radimagenet_resnet18")

# More repeats than the production run improve resolution of small paired
# differences across the three selected optimization backbones.
OPTIMIZATION_N_FOLDS   = 5
OPTIMIZATION_N_REPEATS = 2

# Predefined decision rule; the winner is not chosen by visual judgement.
OPTIMIZATION_MIN_DELTA = 0.02

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING AUGMENTATION: level and parameters
# ══════════════════════════════════════════════════════════════════════════════
# The levels used when OPTIMIZATION = False. In OPTIMIZATION mode each arm sets them.
#   TRAIN_AUGMENTATION_LEVEL        Section 3, end-to-end CNN with fine-tuned backbone
#   FROZEN_E2E_AUGMENTATION_LEVEL   Section 1, end-to-end CNN with frozen backbone
# Section 2 extracts features without augmentation.
TRAIN_AUGMENTATION_LEVEL = "BASIC"
FROZEN_E2E_AUGMENTATION_LEVEL = "FLIPS"

# Parameters owned by the levels above. Geometry is safe because the crop is already
# tumour-centred; the ranges are the ones script 28 used.
AUG_ROTATE_DEG  = 12.0          # BASIC, COMPLEX, COMPLEX_JITTER: rotation +/- degrees
AUG_SCALE_RANGE = (0.92, 1.08)  # COMPLEX, COMPLEX_JITTER: object scale factor
AUG_TRANSLATE   = 0.06          # COMPLEX, COMPLEX_JITTER: shift, fraction of image size
AUG_GAMMA_RANGE = (0.80, 1.25)  # COMPLEX_JITTER only
AUG_BRIGHTNESS  = 0.10          # COMPLEX_JITTER only: additive, on [0,1] data
AUG_CONTRAST    = 0.10          # COMPLEX_JITTER only: multiplicative about the mean
AUG_NOISE_STD   = 0.02          # COMPLEX_JITTER only: gaussian sigma on [0,1] data

_VALID_AUG_LEVELS = ("NONE", "FLIPS", "BASIC", "COMPLEX", "COMPLEX_JITTER")
for _lvl in tuple(OPTIMIZATION_AUG_LEVELS) + (TRAIN_AUGMENTATION_LEVEL, FROZEN_E2E_AUGMENTATION_LEVEL):
    if str(_lvl).upper() not in _VALID_AUG_LEVELS:
        raise ValueError(f"Unknown augmentation level {_lvl!r}; valid: {_VALID_AUG_LEVELS}")

# ── Which arm each workflow is measured against ──────────────────────────────
# The reference arm is the current protocol, which differs between sections:
# Section 1 trains its frozen head with
# FROZEN_E2E_AUGMENTATION_LEVEL while Section 3 fine-tunes with
# TRAIN_AUGMENTATION_LEVEL. A single shared reference level therefore labels one
# section against a configuration it does not use. Each workflow therefore names
# its own reference level, and build_optimization_arms() includes all of them
# in the grid, so both sections get a paired "versus production"
# delta on the same folds. efficient_ML has no training augmentation at all and keys
# its reference on size x slice count only.
OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW = {
    "frozen":   str(FROZEN_E2E_AUGMENTATION_LEVEL).upper(),
    "finetune": str(TRAIN_AUGMENTATION_LEVEL).upper(),
}
for _wf, _wf_lvl in OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW.items():
    if str(_wf_lvl).upper() not in _VALID_AUG_LEVELS:
        raise ValueError(
            f"Reference augmentation for workflow {_wf!r} is {_wf_lvl!r}; "
            f"valid: {_VALID_AUG_LEVELS}"
        )


def optimization_reference_triple(workflow):
    """The (input size, slice count, augmentation) this workflow runs in production."""
    workflow = str(workflow)
    if workflow not in OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW:
        raise KeyError(
            f"Unknown optimization workflow {workflow!r}; "
            f"known: {sorted(OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW)}"
        )
    return (
        int(OPTIMIZATION_REFERENCE_INPUT_SIZE),
        int(OPTIMIZATION_REFERENCE_N_SLICES),
        str(OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW[workflow]).upper(),
    )


def optimization_arm_is_reference(workflow, arm):
    """True when `arm` is the production protocol of `workflow`.

    Reference status belongs to the (arm, workflow) pair rather than to the arm:
    one grid serves both end-to-end sections and they use different current augmentation settings.
    """
    if arm is None:
        return False
    return (
        int(arm["input_size"]),
        int(arm["n_slices"]),
        str(arm["augmentation"]).upper(),
    ) == optimization_reference_triple(workflow)


def tumor_context_tag(mode=None, background=None):
    """Short filesystem-safe tag identifying the tumour-image representation.

    Used in output folder names and file prefixes so two runs that differ only in
    TUMOR_CONTEXT_MODE cannot overwrite each other's results.
    """
    mode = str(TUMOR_CONTEXT_MODE if mode is None else mode).upper()
    background = float(ROI_ONLY_BACKGROUND_VALUE if background is None else background)
    if mode == "ROI_ONLY":
        return "ROIONLY_bg" + f"{background:.3f}".replace(".", "p")
    return "MINBBOX"


def build_optimization_arms():
    """Full factorial of ONLY image size × slice count × augmentation, plus reference.

    TUMOR_CONTEXT_MODE and ROI_ONLY_BACKGROUND_VALUE are manual global settings.
    They are held constant across every arm and are carried only as provenance metadata.
    """
    arms = []
    seen = set()

    def _canon(size, n_slices, aug):
        return (int(size), int(n_slices), str(aug).upper())

    def _add(size, n_slices, aug, is_reference=False):
        if isinstance(n_slices, (bool, np.bool_)) or n_slices not in (1, 2, 3):
            raise ValueError(f"N_SLICES must be 1, 2 or 3; got {n_slices}")
        key = _canon(size, n_slices, aug)
        if key in seen:
            if is_reference:
                for a in arms:
                    if _canon(a["input_size"], a["n_slices"], a["augmentation"]) == key:
                        a["is_reference"] = True
            return

        seen.add(key)
        arms.append({
            "arm_id": (
                f"s{int(size)}_sl{int(n_slices)}_{str(aug).upper()}_"
                f"{tumor_context_tag()}"
            ),
            "input_size": int(size),
            "n_slices": int(n_slices),
            "augmentation": str(aug).upper(),

            # Provenance only. These values are recorded only; they are not compared automatically.
            "context_mode": str(TUMOR_CONTEXT_MODE).upper(),
            "roi_background": float(ROI_ONLY_BACKGROUND_VALUE),
            "is_reference": bool(is_reference),
        })

    if OPTIMIZATION_INCLUDE_REFERENCE_ARM:
        # One reference arm per workflow, de-duplicated when two workflows happen
        # to use the same current augmentation level. The arm-level "is_reference" flag
        # below marks the production protocol of at least one workflow;
        # optimization_arm_is_reference() answers the per-workflow question.
        for _ref_aug in dict.fromkeys(OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW.values()):
            _add(
                CNN_INPUT_SIZE_SHARED,
                N_SLICES,
                _ref_aug,
                is_reference=True,
            )

    for _size in OPTIMIZATION_INPUT_SIZES:
        for _ns in OPTIMIZATION_N_SLICES:
            for _aug in OPTIMIZATION_AUG_LEVELS:
                _add(_size, _ns, _aug)

    # The arm-level flag marks an arm that is the standard combination of at least one
    # workflow, whether it was added above or is part of the grid itself. No other arm
    # is ever relabelled as a reference.
    for a in arms:
        a["is_reference"] = any(
            optimization_arm_is_reference(_wf, a)
            for _wf in OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW
        )

    return arms


OPTIMIZATION_REFERENCE_INPUT_SIZE = int(CNN_INPUT_SIZE_SHARED)
OPTIMIZATION_REFERENCE_N_SLICES = int(N_SLICES)
OPTIMIZATION_ARMS = build_optimization_arms() if OPTIMIZATION else []

def build_ml_optimization_arms():
    """Unique image-representation arms for frozen CNN features + downstream ML.

    Augmentation is deliberately excluded here: the CNN is never trained and the existing
    feature table contains one deterministic row per patient. Applying training augmentation
    would require fold-specific augmented feature extraction and a different data model.
    Size and slice count remain fully optimized and are paired on the same development split.
    """
    if not OPTIMIZATION_ARMS:
        return []
    arms, by_key = [], {}
    for src in OPTIMIZATION_ARMS:
        key = (int(src["input_size"]), int(src["n_slices"]))
        if key not in by_key:
            arm = dict(src)
            arm["arm_id"] = (
                f"s{key[0]}_sl{key[1]}_FROZENFEATURES_{tumor_context_tag()}"
            )
            arm["augmentation"] = "NOT_APPLICABLE"
            arm["source_augmentation_levels"] = []
            # A frozen feature extractor has no training augmentation, so its standard
            # combination is defined by image size and slice count alone.
            arm["is_reference"] = bool(
                key == (int(OPTIMIZATION_REFERENCE_INPUT_SIZE), int(OPTIMIZATION_REFERENCE_N_SLICES))
            )
            by_key[key] = arm
            arms.append(arm)
        arm = by_key[key]
        _src_aug = str(src.get("augmentation", "")).upper()
        if _src_aug and _src_aug not in arm["source_augmentation_levels"]:
            arm["source_augmentation_levels"].append(_src_aug)
    return arms

OPTIMIZATION_ML_ARMS = build_ml_optimization_arms() if OPTIMIZATION else []


def check_optimization_reference_arms():
    """Stop before any training when an enabled workflow has no standard combination to compare against.

    Section 4 reports every setting combination as a paired difference against the
    standard combination of its workflow. Without that arm no comparison is possible,
    so the run stops here and not after all workflows have finished.
    """
    problems = []
    for _wf, _enabled in (("frozen", RUN_FROZEN_CNN_E2E), ("finetune", RUN_CNN_FINETUNE_E2E)):
        if not _enabled:
            continue
        _n_ref = sum(1 for a in OPTIMIZATION_ARMS if optimization_arm_is_reference(_wf, a))
        if _n_ref != 1:
            _size, _ns, _aug = optimization_reference_triple(_wf)
            problems.append(
                f"workflow '{_wf}' needs size={_size}, slices={_ns}, augmentation={_aug}"
            )
    if RUN_CNN_LR:
        _n_ref = sum(1 for a in OPTIMIZATION_ML_ARMS if bool(a.get("is_reference")))
        if _n_ref != 1:
            problems.append(
                f"workflow 'efficient_ML' needs size={int(OPTIMIZATION_REFERENCE_INPUT_SIZE)}, "
                f"slices={int(OPTIMIZATION_REFERENCE_N_SLICES)}"
            )
    if problems:
        raise ValueError(
            "[OPTIMIZATION] The comparison grid does not contain the standard combination of every "
            "enabled workflow: " + " | ".join(problems) + ". "
            "Set OPTIMIZATION_INCLUDE_REFERENCE_ARM to True or add these values to "
            "OPTIMIZATION_INPUT_SIZES, OPTIMIZATION_N_SLICES and OPTIMIZATION_AUG_LEVELS."
        )


if OPTIMIZATION:
    check_optimization_reference_arms()

# Per-workflow pooled development OOF predictions used for paired arm comparisons.
# The original name is retained for fine-tuning compatibility.
OPTIMIZATION_OOF_STORE = {}
OPTIMIZATION_FROZEN_OOF_STORE = {}
OPTIMIZATION_ML_OOF_STORE = {}

def _batch_auc_positives_first(scores, n_positive):
    """AUC of every row of `scores`, whose first `n_positive` columns are the positives.

    Tie-aware Mann-Whitney statistic on average ranks, which is exactly what
    sklearn.metrics.roc_auc_score computes, including its half credit for tied
    scores. Scoring a whole block of bootstrap resamples in one call is what makes
    the paired bootstrap below affordable at full resample counts.
    """
    scores = np.asarray(scores, dtype=float)
    n_batch, n_total = scores.shape
    n_positive = int(n_positive)
    n_negative = n_total - n_positive
    if n_positive < 1 or n_negative < 1:
        return np.full(n_batch, np.nan)

    order = np.argsort(scores, axis=1, kind="stable")
    ordered = np.take_along_axis(scores, order, axis=1)

    # Tied values must share one averaged rank, so mark where each run of equal
    # scores starts and give every member of the run that run's mean rank.
    starts_run = np.empty((n_batch, n_total), dtype=bool)
    starts_run[:, 0] = True
    np.not_equal(ordered[:, 1:], ordered[:, :-1], out=starts_run[:, 1:])
    run_id = np.cumsum(starts_run, axis=1) - 1

    flat_run = (np.arange(n_batch)[:, None] * n_total + run_id).ravel()
    ordinal = np.broadcast_to(np.arange(1.0, n_total + 1.0), (n_batch, n_total))
    run_rank_sum = np.bincount(flat_run, weights=ordinal.ravel(), minlength=n_batch * n_total)
    run_size = np.bincount(flat_run, minlength=n_batch * n_total)
    mean_rank = (run_rank_sum / np.maximum(run_size, 1)).reshape(n_batch, n_total)

    ranks_ordered = np.take_along_axis(mean_rank, run_id, axis=1)
    ranks = np.empty_like(ranks_ordered)
    np.put_along_axis(ranks, order, ranks_ordered, axis=1)

    positive_rank_sum = ranks[:, :n_positive].sum(axis=1)
    return (
        positive_rank_sum - n_positive * (n_positive + 1) / 2.0
    ) / (n_positive * n_negative)


def optimization_paired_bootstrap_auc_delta(
    y_true, reference_score, candidate_score, confidence=0.95,
    n_bootstrap=2000, random_state=GLOBAL_SEED + 137, resample_block=512,
):
    """Fast paired patient-level bootstrap for AUC(candidate) - AUC(reference).

    Positives and negatives are resampled separately in the same order as the older loop.
    The complete bootstrap index matrix is pre-drawn in the SAME per-replicate RNG
    order (positive draw, then negative draw), so changing `resample_block` cannot
    change which bootstrap samples are analysed. Only the AUC scoring is vectorised.
    """
    y_true = np.asarray(y_true, dtype=int).ravel()
    reference_score = np.asarray(reference_score, dtype=float).ravel()
    candidate_score = np.asarray(candidate_score, dtype=float).ravel()
    valid = np.isfinite(y_true) & np.isfinite(reference_score) & np.isfinite(candidate_score)
    y_true, reference_score, candidate_score = (
        y_true[valid], reference_score[valid], candidate_score[valid]
    )
    if y_true.size == 0 or np.unique(y_true).size != 2:
        nan = float("nan")
        return {"reference_auc": nan, "candidate_auc": nan, "delta": nan,
                "ci_lower": nan, "ci_upper": nan, "n_bootstrap": 0,
                "p_candidate_gt_reference": nan}

    pos_idx = np.flatnonzero(y_true == 1)
    neg_idx = np.flatnonzero(y_true == 0)
    n_positive, n_negative = pos_idx.size, neg_idx.size

    # Point estimates use the same tie-aware Mann-Whitney AUC statistic as the
    # vectorised bootstrap scoring (mathematically identical to roc_auc_score).
    _paired = np.stack([
        np.concatenate([reference_score[pos_idx], reference_score[neg_idx]]),
        np.concatenate([candidate_score[pos_idx], candidate_score[neg_idx]]),
    ])
    ref_auc, cand_auc = (float(v) for v in _batch_auc_positives_first(_paired, n_positive))
    observed = cand_auc - ref_auc

    n_bootstrap = int(n_bootstrap)
    if n_positive < 1 or n_negative < 1 or n_bootstrap < 2:
        nan = float("nan")
        return {"reference_auc": ref_auc, "candidate_auc": cand_auc, "delta": observed,
                "ci_lower": nan, "ci_upper": nan, "n_bootstrap": 0,
                "p_candidate_gt_reference": nan}

    # IMPORTANT: keep the old RNG call order exactly. The inexpensive Python loop
    # only draws indices; the former 2 * n_bootstrap sklearn AUC calls are gone.
    rng = np.random.default_rng(int(random_state))
    _take_all = np.empty((n_bootstrap, n_positive + n_negative), dtype=np.intp)
    for _b in range(n_bootstrap):
        _take_all[_b, :n_positive] = rng.choice(
            pos_idx, size=n_positive, replace=True
        )
        _take_all[_b, n_positive:] = rng.choice(
            neg_idx, size=n_negative, replace=True
        )

    # Score in blocks only to cap temporary memory. Because all indices were already
    # drawn above, the result is invariant to `resample_block`.
    _block = max(1, int(resample_block))
    deltas = np.empty(n_bootstrap, dtype=float)
    for _start in range(0, n_bootstrap, _block):
        _stop = min(_start + _block, n_bootstrap)
        _take = _take_all[_start:_stop]
        deltas[_start:_stop] = (
            _batch_auc_positives_first(candidate_score[_take], n_positive)
            - _batch_auc_positives_first(reference_score[_take], n_positive)
        )

    alpha = 1.0 - float(confidence)
    return {
        "reference_auc": ref_auc, "candidate_auc": cand_auc, "delta": float(observed),
        "ci_lower": float(np.quantile(deltas, alpha / 2.0)),
        "ci_upper": float(np.quantile(deltas, 1.0 - alpha / 2.0)),
        "n_bootstrap": int(deltas.size),
        "p_candidate_gt_reference": float(np.mean(deltas > 0.0)),
    }

if OPTIMIZATION:
    print("=" * 78)
    print(f"[OPTIMIZATION] ON - CNN training workflows: {len(OPTIMIZATION_ARMS)} arms x "
          f"{len(OPTIMIZATION_BACKBONES)} backbone(s), "
          f"{OPTIMIZATION_N_FOLDS}x{OPTIMIZATION_N_REPEATS} CV"
          + (" + final refit + holdout test per arm." if holdout_evaluation_enabled()
             else " on development patients only."))
    print(f"[OPTIMIZATION] CNN+ML: {len(OPTIMIZATION_ML_ARMS)} unique size x slice-count "
          "representation arms; augmentation is NOT_APPLICABLE to frozen feature extraction.")
    if holdout_evaluation_enabled():
        print("[OPTIMIZATION] EVERY arm is final-refit on all development patients and then "
              "evaluated on the SAME holdout test set.")
        print("[OPTIMIZATION][TEST-REUSE WARNING] If holdout metrics are used to choose an arm, "
              "this cohort functions as an optimization/validation set rather than an untouched final test.")
    else:
        print("[OPTIMIZATION] Holdout test: NOT scored for any arm "
              "(OPTIMIZATION_EVALUATE_HOLDOUT is False). Settings are chosen from development data only.")
    for _a in OPTIMIZATION_ARMS:
        _bg_txt = (f"bg={_a['roi_background']:.3f}"
                   if _a["context_mode"] == "ROI_ONLY" else "bg=n/a ")
        _ref_for = [
            _wf for _wf in OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW
            if optimization_arm_is_reference(_wf, _a)
        ]
        print(f"    {_a['arm_id']:46s} size={_a['input_size']:3d} | slices={_a['n_slices']} | "
              f"aug={_a['augmentation']:15s} | {_a['context_mode']:8s} | {_bg_txt}"
              + (f"   <- REFERENCE for {', '.join(_ref_for)}" if _ref_for else ""))
    print(f"[OPTIMIZATION] Adoption rule: paired delta >= {OPTIMIZATION_MIN_DELTA:+.3f} "
          "AND bootstrap CI excluding zero.")
    if len(OPTIMIZATION_ARMS) > 8:
        print(f"[OPTIMIZATION][multiplicity warning] {len(OPTIMIZATION_ARMS)} arms is a wide "
              "grid for ~150 patients. Varying ONE factor at a time from the reference "
              "(collapse the other level lists to a single value) resolves a real effect "
              "more reliably than a full factorial does.")
    print("=" * 78)
# Each central plane uses the smallest physical square containing its tumour bounding box after adding this many
# native pixels of safety margin on every side, then it is resized to 224x224.
CROP_SAFETY_MARGIN_PX = 2
INTENSITY_Z_SPAN_SD = 2.5

# ── Tumour context validation ───────────────────────────────────────────────
# TUMOR_CONTEXT_MODE and ROI_ONLY_BACKGROUND_VALUE are defined once in the
# master manual switch above. Do not redefine them here. This block only
# normalises and validates the selected mode before image processing.
_VALID_TUMOR_CONTEXT_MODES = ("MIN_BBOX", "ROI_ONLY")
TUMOR_CONTEXT_MODE = str(TUMOR_CONTEXT_MODE).upper()
if TUMOR_CONTEXT_MODE not in _VALID_TUMOR_CONTEXT_MODES:
    raise ValueError(
        f"TUMOR_CONTEXT_MODE must be one of {_VALID_TUMOR_CONTEXT_MODES}; got {TUMOR_CONTEXT_MODE!r}"
    )
if not np.isfinite(float(ROI_ONLY_BACKGROUND_VALUE)) or not (0.0 <= float(ROI_ONLY_BACKGROUND_VALUE) <= 1.0):
    raise ValueError(
        f"ROI_ONLY_BACKGROUND_VALUE must be finite and within [0,1]; got {ROI_ONLY_BACKGROUND_VALUE}"
    )

# ══════════════════════════════════════════════════════════════════════════════
# COMMON IMAGE PREPROCESSING: defined once for all image workflows
# ══════════════════════════════════════════════════════════════════════════════
# These functions intentionally resolve sitk / torch / F / CHANNEL_MODE at call
# time, after the active workflow has imported its libraries and set its channel
# policy. Geometry, normalization and ROI handling therefore remain identical
# across Sections 1, 2 and 3 without later cells redefining the same function names.

def read_sitk_pair(image_path: str, mask_path: str):
    """Read image and mask. If geometry differs, resample mask to image grid with nearest-neighbor."""
    img = sitk.ReadImage(str(image_path))
    mask = sitk.ReadImage(str(mask_path))
    if (img.GetSize() != mask.GetSize() or
        img.GetSpacing() != mask.GetSpacing() or
        img.GetDirection() != mask.GetDirection() or
        img.GetOrigin() != mask.GetOrigin()):
        mask = sitk.Resample(mask, img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, mask.GetPixelID())
    # Standardize anatomical axes before extracting orthogonal planes.
    img = sitk.DICOMOrient(img, "LPS")
    mask = sitk.DICOMOrient(mask, "LPS")
    img_arr = sitk.GetArrayFromImage(img).astype(np.float32)   # z,y,x
    mask_arr = sitk.GetArrayFromImage(mask)                 # z,y,x; labels kept as-is
    spacing_zyx = np.array(img.GetSpacing()[::-1], dtype=np.float32)
    return img_arr, mask_arr, spacing_zyx

def normalize_volume_for_cnn(img_arr, mask_arr):
    """Clip/z-score/scale similarly to the uploaded notebooks: output mostly in [0, 1]."""
    finite = np.isfinite(img_arr)
    tumor_vals = img_arr[(mask_arr > 0) & finite]
    if tumor_vals.size >= 10:
        vals = tumor_vals
    else:
        vals = img_arr[(img_arr != 0) & finite]
    if vals.size < 10:
        vals = img_arr[finite]
    mean = float(np.mean(vals))
    sd = float(np.std(vals)) + 1e-8
    out = (img_arr.astype(np.float32) - mean) / sd
    # Map +/- INTENSITY_Z_SPAN_SD standard deviations onto [0, 1]. At 2.5 SD the
    # within-ROI pixel SD lands near 0.2-0.3, the contrast range these backbones
    # were pretrained on and the range their frozen BatchNorm statistics expect.
    z_span = float(INTENSITY_Z_SPAN_SD)
    out = np.clip(out, -z_span, z_span)
    out = out / (2.0 * z_span) + 0.5
    out[~np.isfinite(out)] = 0.0
    return out.astype(np.float32)

def selected_tumor_slice_indices(mask_arr, spacing_zyx=None, n_slices=None):
    """Select central (maximum tumour-area) slices in axial, sagittal, coronal order.

    Arrays are z,y,x in LPS orientation. Ties use the first maximum, preserving
    the original axial selection. No peripheral slices or count padding are used.
    """
    count = N_SLICES if n_slices is None else n_slices
    if isinstance(count, (bool, np.bool_)) or count not in (1, 2, 3):
        raise ValueError(f"N_SLICES must be 1, 2 or 3; got {count}")
    binary = np.asarray(mask_arr) > 0
    if binary.ndim != 3 or not binary.any():
        raise ValueError("Expected a nonempty 3D segmentation mask.")
    rows = []
    for plane, axis in (("axial", 0), ("sagittal", 2), ("coronal", 1))[:int(count)]:
        areas = np.moveaxis(binary, axis, 0).reshape(binary.shape[axis], -1).sum(axis=1)
        index = int(np.argmax(areas))
        rows.append({"plane": plane, "slice_axis": axis, "slice_index": index,
                     "tumor_area": float(areas[index]), "reused": False,
                     "n_unique_tumour_slices": int(count),
                     "n_available_tumour_slices": int(np.count_nonzero(areas))})
    return rows


def extract_plane_slice(volume, row):
    """Return axial y,x; sagittal z,y; or coronal z,x pixels."""
    return np.take(volume, int(row["slice_index"]), axis=int(row["slice_axis"]))


def build_central_plane_inputs(img_norm, binary_mask, spacing_zyx, out_size):
    """Crop each plane in its own physical coordinates, then stack CNN inputs."""
    rows = selected_tumor_slice_indices(binary_mask)
    sp = np.asarray(spacing_zyx, dtype=float).ravel()
    if sp.size != 3 or not np.isfinite(sp).all() or np.any(sp <= 1e-6):
        raise ValueError(f"Invalid volume spacing: {spacing_zyx}")
    tensors, metadata = [], []
    for row in rows:
        axis = int(row["slice_axis"])
        in_plane_axes = [a for a in range(3) if a != axis]
        plane_spacing = sp[[axis] + in_plane_axes]
        image2d = extract_plane_slice(img_norm, row)
        mask2d = extract_plane_slice(binary_mask, row)
        # Reuse the existing physical-square crop on a singleton plane volume.
        spec = build_patient_min_square_crop_spec(mask2d[None], [{"z_index": 0}], plane_spacing)
        meta = attach_crop_diagnostics([row], spec, plane_spacing)[0]
        meta.pop("crop_peak_z_index", None)
        meta["crop_plane"] = row["plane"]
        tensors.append(crop_resize_slice(image2d, mask2d, spec, out_size))
        metadata.append(meta)
    return torch.stack(tensors, dim=0), metadata

def build_patient_min_square_crop_spec(mask_arr, slice_rows, spacing_zyx,
                                       safety_margin_px=None):
    """Build one minimal patient-level physical-square crop for all selected slices.

    The union bounding box of the complete tumour across the selected native slices is
    expanded by CROP_SAFETY_MARGIN_PX native pixels on every side. The smallest
    physical square that contains that expanded box is then used for every selected
    slice, preserving a common field of view and scale. If the requested crop extends
    beyond the image, zero-padding preserves the full safety margin before resizing.
    """
    if safety_margin_px is None:
        safety_margin_px = CROP_SAFETY_MARGIN_PX
    safety_margin_px = int(safety_margin_px)
    if safety_margin_px < 0:
        raise ValueError(f"CROP_SAFETY_MARGIN_PX must be >= 0; got {safety_margin_px}")
    if not slice_rows:
        raise ValueError("Cannot build a crop without selected tumour slices.")

    sp = np.asarray(spacing_zyx, dtype=float).ravel()
    if sp.size < 3 or not np.isfinite(sp[1:3]).all() or np.any(sp[1:3] <= 1e-6):
        raise ValueError(f"Invalid in-plane spacing for physical crop: {spacing_zyx}")
    sy, sx = float(sp[1]), float(sp[2])

    binary = (mask_arr > 0)
    areas_px = binary.reshape(binary.shape[0], -1).sum(axis=1)
    z_peak = int(np.argmax(areas_px))
    peak_coords = np.argwhere(binary[z_peak])
    if peak_coords.size == 0:
        raise ValueError("Peak tumour slice unexpectedly has an empty mask.")

    py0, px0 = peak_coords.min(axis=0).astype(int)
    py1, px1 = (peak_coords.max(axis=0) + 1).astype(int)

    # Union bounding box across the selected native slices. This is the smallest
    # rectangle that contains the complete tumour on every slice used for the patient.
    selected_boxes = []
    for row in slice_rows:
        z = int(row["z_index"])
        coords = np.argwhere(binary[z])
        if coords.size == 0:
            raise ValueError(f"Selected slice z={z} unexpectedly has an empty tumour mask.")
        y0, x0 = coords.min(axis=0).astype(int)
        y1, x1 = (coords.max(axis=0) + 1).astype(int)
        selected_boxes.append((y0, y1, x0, x1))

    uy0 = int(min(b[0] for b in selected_boxes))
    uy1 = int(max(b[1] for b in selected_boxes))
    ux0 = int(min(b[2] for b in selected_boxes))
    ux1 = int(max(b[3] for b in selected_boxes))

    # Add the requested native-pixel safety margin on all four sides BEFORE
    # squaring. Bounds are intentionally allowed outside the image; padding below
    # preserves the full margin for edge-touching tumours.
    req_y0 = uy0 - safety_margin_px
    req_y1 = uy1 + safety_margin_px
    req_x0 = ux0 - safety_margin_px
    req_x1 = ux1 + safety_margin_px

    req_h_px = int(req_y1 - req_y0)
    req_w_px = int(req_x1 - req_x0)
    req_h_mm = float(req_h_px * sy)
    req_w_mm = float(req_w_px * sx)

    # Smallest physical square containing the margin-expanded union box.
    requested_side_mm = float(max(req_h_mm, req_w_mm))
    crop_h_px = max(req_h_px, int(np.ceil(requested_side_mm / sy - 1e-12)))
    crop_w_px = max(req_w_px, int(np.ceil(requested_side_mm / sx - 1e-12)))

    # Centre on the union tumour box while guaranteeing that the full
    # margin-expanded box is inside the crop.
    preferred_cy_px = (float(uy0) + float(uy1)) / 2.0
    preferred_cx_px = (float(ux0) + float(ux1)) / 2.0

    def _place_interval(required_lo, required_hi, length, preferred_center):
        start_min = int(required_hi - length)
        start_max = int(required_lo)
        if start_min > start_max:
            raise ValueError("Crop length is smaller than the tumour plus safety margin.")
        preferred_start = float(preferred_center) - float(length) / 2.0
        start = int(np.rint(preferred_start))
        start = min(max(start, start_min), start_max)
        return int(start), int(start + length)

    y0i, y1i = _place_interval(req_y0, req_y1, crop_h_px, preferred_cy_px)
    x0i, x1i = _place_interval(req_x0, req_x1, crop_w_px, preferred_cx_px)

    H, W = binary.shape[1:]
    pad_top = max(0, -y0i)
    pad_left = max(0, -x0i)
    pad_bottom = max(0, y1i - H)
    pad_right = max(0, x1i - W)
    y0c, y1c = max(0, y0i), min(H, y1i)
    x0c, x1c = max(0, x0i), min(W, x1i)

    crop_height_mm = float(crop_h_px * sy)
    crop_width_mm = float(crop_w_px * sx)
    crop_area_mm2 = float(crop_height_mm * crop_width_mm)

    peak_area_px = float(areas_px[z_peak])
    peak_area_mm2 = float(peak_area_px * sy * sx)
    achieved_peak_occupancy = float(peak_area_mm2 / crop_area_mm2)

    peak_bbox_h_mm = float((py1 - py0) * sy)
    peak_bbox_w_mm = float((px1 - px0) * sx)
    peak_min_square_side_mm = float(max(peak_bbox_h_mm, peak_bbox_w_mm))
    union_h_mm = float((uy1 - uy0) * sy)
    union_w_mm = float((ux1 - ux0) * sx)
    union_min_square_side_mm = float(max(union_h_mm, union_w_mm))

    return {
        "y0": int(y0c), "y1": int(y1c), "x0": int(x0c), "x1": int(x1c),
        "pad_top": int(pad_top), "pad_bottom": int(pad_bottom),
        "pad_left": int(pad_left), "pad_right": int(pad_right),
        "crop_center_y_px": float((y0i + y1i) / 2.0),
        "crop_center_x_px": float((x0i + x1i) / 2.0),
        "crop_preferred_center_y_px": float(preferred_cy_px),
        "crop_preferred_center_x_px": float(preferred_cx_px),
        "crop_height_px": int(crop_h_px), "crop_width_px": int(crop_w_px),
        "crop_height_mm": crop_height_mm, "crop_width_mm": crop_width_mm,
        "crop_area_mm2": crop_area_mm2,
        "crop_requested_side_mm": requested_side_mm,
        "crop_safety_margin_px": int(safety_margin_px),
        "crop_union_bbox_height_px": int(uy1 - uy0),
        "crop_union_bbox_width_px": int(ux1 - ux0),
        "crop_margin_box_height_px": int(req_h_px),
        "crop_margin_box_width_px": int(req_w_px),
        "crop_peak_tumor_area_px": peak_area_px,
        "crop_peak_tumor_area_mm2": peak_area_mm2,
        "crop_peak_occupancy": achieved_peak_occupancy,
        "crop_peak_min_side_mm": peak_min_square_side_mm,
        "crop_all_selected_min_side_mm": union_min_square_side_mm,
        "crop_physical_square_error_mm": float(abs(crop_height_mm - crop_width_mm)),
        "crop_limiting_reason": "union_bbox_plus_safety_margin_min_physical_square",
        "crop_peak_z_index": int(z_peak),
    }

def attach_crop_diagnostics(slice_rows, crop_spec, spacing_zyx):
    """Attach common-crop and per-slice occupancy diagnostics to slice metadata."""
    sp = np.asarray(spacing_zyx, dtype=float).ravel()
    sy, sx = float(sp[1]), float(sp[2])
    crop_area_mm2 = float(crop_spec["crop_area_mm2"])
    common = {k: v for k, v in crop_spec.items()
              if k.startswith("crop_") and k not in {"crop_peak_tumor_area_px"}}
    out = []
    for row in slice_rows:
        r = dict(row)
        r.update(common)
        r["slice_tumor_area_px"] = float(r["tumor_area"])
        r["slice_tumor_area_mm2"] = float(r["tumor_area"] * sy * sx)
        r["slice_tumor_occupancy"] = float(r["slice_tumor_area_mm2"] / crop_area_mm2)
        out.append(r)
    return out

def crop_resize_slice(img2d, mask2d, crop_spec, out_size):
    """Apply one precomputed patient-level crop, verify no tumour was clipped, then resize."""
    y0, y1 = int(crop_spec["y0"]), int(crop_spec["y1"])
    x0, x1 = int(crop_spec["x0"]), int(crop_spec["x1"])
    img_crop = img2d[y0:y1, x0:x1]
    mask_crop = (mask2d[y0:y1, x0:x1] > 0).astype(np.float32)

    pads = (
        int(crop_spec["pad_top"]), int(crop_spec["pad_bottom"]),
        int(crop_spec["pad_left"]), int(crop_spec["pad_right"]),
    )
    pad_top, pad_bottom, pad_left, pad_right = pads
    if any(v > 0 for v in pads):
        img_crop = np.pad(img_crop, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="constant")
        mask_crop = np.pad(mask_crop, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="constant")

    full_tumour_px = int((mask2d > 0).sum())
    kept_tumour_px = int(np.rint(mask_crop.sum()))
    if kept_tumour_px != full_tumour_px:
        raise ValueError(
            f"Patient crop clipped tumour pixels: kept {kept_tumour_px}/{full_tumour_px}."
        )

    # Optional tumour-ROI-only representation. Geometry is deliberately unchanged:
    # the same patient crop, slice positions and resize are used, so this switch tests
    # removal of peritumoural/anatomical context rather than a different object scale.
    _context_mode = str(TUMOR_CONTEXT_MODE).upper()
    if _context_mode == "ROI_ONLY":
        _bg = float(ROI_ONLY_BACKGROUND_VALUE)
        img_crop = np.where(mask_crop > 0.5, img_crop, _bg).astype(np.float32, copy=False)
    elif _context_mode != "MIN_BBOX":
        raise ValueError(f"Unknown TUMOR_CONTEXT_MODE={TUMOR_CONTEXT_MODE!r}")

    if CHANNEL_MODE == "t1cc_t1cc_mask":
        ch = np.stack([img_crop, img_crop, mask_crop], axis=0)
    elif CHANNEL_MODE == "t1cc_masked_mask":
        ch = np.stack([img_crop, img_crop * mask_crop, mask_crop], axis=0)
    else:
        raise ValueError(f"Unknown CHANNEL_MODE={CHANNEL_MODE}")

    x = torch.from_numpy(ch[None]).float()
    x = F.interpolate(x, size=(int(out_size), int(out_size)), mode="bilinear", align_corners=False)
    return x.squeeze(0)


# Slice-level CSVs are not exported; slice features are still computed internally
# to form the patient means.
SAVE_SLICE_LEVEL_FEATURE_CSVS = True

# Section 0 establishes one split from the validated, labelled pairs.
# Every later section must present that same canonical patient list.
GLOBAL_PATIENT_SPLIT_KEY = "global_patient_split"
PATIENT_SPLIT_RECORDS = {}
SPLIT_FOLDER_FILE_INVENTORY = {}


def canonical_patient_id(value) -> str:
    """Normalize a complete patient ID; never identify a patient from a digit substring."""
    if value is None:
        return ""
    s = str(value).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return ""

    # Normalize numeric IDs such as 0008456 and CSV numeric values such as 8456.0,
    # but only when the whole value is numeric. A partial match would merge distinct
    # patients (for example, 8456 and 8400 both contain the substring 84).
    numeric_id = re.fullmatch(r"([0-9]+)(?:\.0+)?", s)
    if numeric_id:
        return str(int(numeric_id.group(1)))
    return s


# Regression guards against partial-number matching in folder split checks.
assert canonical_patient_id("8456") == "8456"
assert canonical_patient_id("8400") == "8400"
assert canonical_patient_id("8456") != canonical_patient_id("8400")
assert canonical_patient_id("8456") != canonical_patient_id("84")
assert canonical_patient_id("8456.0") == canonical_patient_id("8456")


def patient_id_sort_key(value):
    """Sort numeric patient IDs by number and place any non-numeric IDs afterwards."""
    cid = canonical_patient_id(value)
    if cid.isdigit():
        return (0, int(cid), cid)
    return (1, 0, cid)


def _canonical_unique_patient_ids(patient_ids, context="patient cohort"):
    """Convert IDs to the shared format and stop if an ID is blank or repeated."""
    ids = [canonical_patient_id(v) for v in list(patient_ids)]
    blank_positions = [i for i, cid in enumerate(ids) if not cid]
    if blank_positions:
        raise ValueError(
            f"{context}: blank/unusable patient IDs at row positions {blank_positions[:20]}"
        )

    counts = {}
    for cid in ids:
        counts[cid] = counts.get(cid, 0) + 1
    duplicates = sorted(
        [cid for cid, count in counts.items() if count > 1],
        key=patient_id_sort_key,
    )
    if duplicates:
        raise ValueError(f"{context}: patient IDs become duplicates after standard formatting: {duplicates}")
    return ids


def load_label_table(path, context="labels"):
    """Read the label CSV: one patient ID column and one 0/1 label column.

    Every section reads the label file through this one function. Column names are
    matched without regard to letter case. The ID column is the first one whose name
    contains "patient" or is id, pid or case, otherwise the first column. The label
    column is the first one named label, outcome, class, target or y, otherwise the last
    column. The two columns that were used are printed, so a wrong choice is visible.
    """
    df = pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    id_col = next((c for c in df.columns if "patient" in c or c in ("id", "pid", "case")), df.columns[0])
    lbl_col = next((c for c in df.columns if c in ("label", "outcome", "class", "target", "y")), df.columns[-1])
    if id_col == lbl_col:
        raise ValueError(f"{path}: the label table needs a patient ID column and a separate label column.")
    print(f"[labels] {context}: patient ID column '{id_col}' | label column '{lbl_col}'")
    df = df[[id_col, lbl_col]].copy()
    df.columns = ["patient_id", "label"]
    df["patient_id"] = df["patient_id"].astype(str)
    df["canonical_id"] = df["patient_id"].map(canonical_patient_id)
    _canonical_unique_patient_ids(df["canonical_id"], context=f"{context}: {path}")

    numeric_labels = pd.to_numeric(df["label"], errors="coerce")
    if numeric_labels.isna().any():
        raise ValueError(f"{path}: labels contain missing or non-numeric values.")
    if not np.allclose(numeric_labels.to_numpy(), numeric_labels.astype(int).to_numpy()):
        raise ValueError(f"{path}: labels must be integer 0/1 values.")
    df["label"] = numeric_labels.astype(int)
    classes = set(df["label"].unique().tolist())
    if classes != {0, 1}:
        raise ValueError(f"{path}: labels must be exactly {{0, 1}}; found {sorted(classes)}.")
    return df[["patient_id", "canonical_id", "label"]]


# ── Folder-defined development / holdout membership ───────────────────────
# Each image or mask file in DEVELOPMENT_ROOT or TEST_ROOT is mapped to its patient ID
# with the same rule the pairing cells use: drop the file extension and one image/mask
# tag such as "_t1c", "_seg" or "_mask", then apply canonical_patient_id().
_SPLIT_VOLUME_EXTENSIONS = (".nii.gz", ".nii", ".nrrd", ".nhdr", ".mha", ".mhd")
_SPLIT_ID_TAG_SUFFIXES = (
    "mask.seg",
    "-t1cc", "_t1cc", "-t1ce", "_t1ce", "-t1c", "_t1c", "-t1", "_t1",
    "-seg", "_seg", "-gtv", "_gtv", "-mask", "_mask", "-label", "_label", "-roi", "_roi",
    ".seg", ".gtv", ".mask", ".label", ".roi",
)


def patient_id_from_volume_filename(filename) -> str:
    """Patient ID of one image or mask file, in the shared canonical format."""
    base = os.path.basename(str(filename))
    low = base.lower()
    for ext in _SPLIT_VOLUME_EXTENSIONS:
        if low.endswith(ext):
            base = base[: -len(ext)]
            break
    low = base.lower()
    for tag in _SPLIT_ID_TAG_SUFFIXES:
        if low.endswith(tag):
            base = base[: -len(tag)]
            break
    return canonical_patient_id(base)


def scan_split_folders(split_folders=None):
    """Record which set - DEVELOPMENT or TEST - and which folder holds each patient's files.

    Each set is one folder or a list of folders, scanned in the order given.
    Returns two records: {canonical patient ID: "development" | "test"} and
    {canonical patient ID: folder path}. Stops the run when a folder is missing or
    holds no image/mask volumes, or when one patient has files in more than one folder.
    """
    from pathlib import Path as _Path
    split_folders = dict(SPLIT_FOLDERS if split_folders is None else split_folders)
    ids_by_set, n_files_by_set, folder_rows = {}, {}, []
    first_folder_of_id, in_several_folders = {}, {}
    for split_set, roots in split_folders.items():
        roots = split_folder_roots(roots)
        ids_by_set[split_set], n_files_by_set[split_set] = set(), 0
        for root_number, root in enumerate(roots, start=1):
            folder_text = split_folder_label(split_set, root_number, len(roots))
            root_path = _Path(str(root))
            if not root_path.is_dir():
                raise FileNotFoundError(
                    f"[PATIENT SPLIT] {folder_text} not found: {root}\n"
                    "Check the DEVELOPMENT_ROOT / TEST_ROOT paths at the top of Section 0, cell 3, "
                    "and that Google Drive is connected."
                )
            files = sorted(
                p for p in root_path.rglob("*")
                if p.is_file() and p.name.lower().endswith(_SPLIT_VOLUME_EXTENSIONS)
            )
            if not files:
                raise RuntimeError(
                    f"[PATIENT SPLIT] No image/mask volumes (.nrrd, .nii, .nii.gz, ...) were found "
                    f"in the {folder_text}: {root}"
                )
            # Keep this one discovery result for the subsequent shared pairing pass.
            SPLIT_FOLDER_FILE_INVENTORY[os.path.normpath(str(root))] = tuple(str(p) for p in files)
            ids = set()
            for p in files:
                cid = patient_id_from_volume_filename(p.name)
                if not cid:
                    raise ValueError(f"[PATIENT SPLIT] Cannot read a patient ID from the file name: {p}")
                ids.add(cid)
            for cid in ids:
                if cid in first_folder_of_id:
                    in_several_folders.setdefault(cid, [first_folder_of_id[cid]]).append(str(root))
                else:
                    first_folder_of_id[cid] = str(root)
            ids_by_set[split_set] |= ids
            n_files_by_set[split_set] += len(files)
            folder_rows.append((folder_text, len(ids), len(files), str(root)))

    membership, in_both = {}, set()
    for split_set, ids in ids_by_set.items():
        for cid in ids:
            if cid in membership and membership[cid] != split_set:
                in_both.add(cid)
            membership[cid] = split_set
    if in_both:
        raise ValueError(
            "[PATIENT SPLIT] These patient IDs have files in BOTH the DEVELOPMENT and the TEST "
            f"folder: {sorted(in_both, key=patient_id_sort_key)}. Each patient must be in exactly "
            "one folder; move or remove the duplicates, then rerun Section 0."
        )
    if in_several_folders:
        split_ids = sorted(in_several_folders, key=patient_id_sort_key)
        examples = "; ".join(
            f"{cid}: " + " and ".join(os.path.basename(f.rstrip("/")) for f in in_several_folders[cid])
            for cid in split_ids[:10]
        )
        raise ValueError(
            f"[PATIENT SPLIT] {len(split_ids)} patient ID(s) have files in more than one folder: "
            f"{split_ids}. A patient's image and mask must sit together in ONE "
            f"folder ({examples}). Move the files, then rerun Section 0."
        )
    label_width = max(len(row[0]) for row in folder_rows)
    for folder_text, n_ids, n_files, root in folder_rows:
        print(
            f"[PATIENT SPLIT] {folder_text + ':':<{label_width + 1}s} {n_ids} patient ID(s) "
            f"from {n_files} image/mask file(s) | {root}"
        )
    for split_set, roots in split_folders.items():
        n_roots = len(split_folder_roots(roots))
        if n_roots > 1 and split_set == SPLIT_SET_TEST:
            print(
                f"[PATIENT SPLIT] TEST: {n_roots} separate test sets, together "
                f"{len(ids_by_set[split_set])} patient ID(s) from "
                f"{n_files_by_set[split_set]} image/mask file(s)"
            )
        elif n_roots > 1:
            print(
                f"[PATIENT SPLIT] {split_set.upper()} set, all {n_roots} folders: "
                f"{len(ids_by_set[split_set])} patient ID(s) from "
                f"{n_files_by_set[split_set]} image/mask file(s)"
            )
    return membership, dict(first_folder_of_id)


def split_set_of_patient(patient_id):
    """Return "development", "test" or None when the patient is in neither folder."""
    return SPLIT_FOLDER_MEMBERSHIP.get(canonical_patient_id(patient_id))


# ── Image and mask pairing: the only definition, shared by all three sections ─────
# A file is a mask when its name contains one of MASK_FILENAME_KEYWORDS and an image
# otherwise. The patient ID of a file is read by patient_id_from_volume_filename(), the
# rule the folder scan above uses. The files are listed by the extensions of that scan
# (_SPLIT_VOLUME_EXTENSIONS). The pairing and the folder scan therefore always agree on
# every patient ID. A patient is used only when exactly one image and exactly one mask
# carry its ID.
MASK_FILENAME_KEYWORDS = ("seg", "gtv", "mask", "label", "roi", "contour",
                          "annotation", "delineation")


def is_mask_file(name):
    """True when the file name contains one of MASK_FILENAME_KEYWORDS."""
    low = str(name).lower()
    return any(keyword in low for keyword in MASK_FILENAME_KEYWORDS)


def collect_generic_image_mask_pairs(root, audit_csv=None):
    """Pair each image with its mask in ONE folder by exact canonical patient ID.

    No substring matching is used, so the wrong image and mask can never be paired by
    accident. Sub-folders are searched too. Every section hands this one function to
    collect_pairs_from_split_folders(), which calls it once for every DEVELOPMENT and
    TEST folder, so every section pairs the same files in the same way.

    An ID with no image, no mask or more than one of either is not used. Those IDs are
    printed with the reason and, when audit_csv is given, written to that file.
    """
    from pathlib import Path as _Path

    inventory = SPLIT_FOLDER_FILE_INVENTORY.get(os.path.normpath(str(root)))
    if inventory is None:
        raise RuntimeError("Folder inventory is missing. Rerun Section 0, cell 3.")
    files = [_Path(path) for path in inventory]

    image_by_cid, mask_by_cid = {}, {}
    for p in files:
        cid = patient_id_from_volume_filename(p.name)
        (mask_by_cid if is_mask_file(p.name) else image_by_cid).setdefault(cid, []).append(p)

    pairs, audit_rows = [], []
    for cid in sorted(set(image_by_cid) | set(mask_by_cid), key=patient_id_sort_key):
        imgs = image_by_cid.get(cid, [])
        masks = mask_by_cid.get(cid, [])
        if len(imgs) == 1 and len(masks) == 1:
            pairs.append({
                "patient_id": cid,
                "canonical_id": cid,
                "image": str(imgs[0]),
                "mask": str(masks[0]),
            })
            continue
        if not imgs:
            status = "mask_without_image"
        elif not masks:
            status = "image_without_mask"
        elif len(imgs) > 1 and len(masks) > 1:
            status = "ambiguous_multiple_images_and_masks"
        elif len(imgs) > 1:
            status = "ambiguous_multiple_images"
        else:
            status = "ambiguous_multiple_masks"
        audit_rows.append({
            "canonical_id": cid,
            "status": status,
            "n_images": len(imgs),
            "n_masks": len(masks),
            "image_names": " | ".join(p.name for p in imgs),
            "mask_names": " | ".join(p.name for p in masks),
            "image_paths": " | ".join(str(p) for p in imgs),
            "mask_paths": " | ".join(str(p) for p in masks),
        })

    n_images = sum(len(v) for v in image_by_cid.values())
    n_masks = sum(len(v) for v in mask_by_cid.values())
    print(f"[image pairing] image files: {n_images} | mask files: {n_masks} | "
          f"paired by exact patient ID: {len(pairs)} | IDs not paired: {len(audit_rows)}")
    if audit_rows:
        _audit = pd.DataFrame(audit_rows)
        print("[image pairing] IDs that are NOT used (first 12):")
        print(_audit[["canonical_id", "status", "image_names", "mask_names"]].head(12).to_string(index=False))
        if audit_csv:
            _audit.to_csv(audit_csv, index=False)
            print(f"[image pairing] Full list saved: {audit_csv}")
    return pairs


def collect_pairs_from_split_folders(collect_fn, context, split_folders=None):
    """Run the image/mask pairing on each split folder and tag every pair.

    collect_fn is collect_generic_image_mask_pairs() above, the one pairing function of
    all three sections. A section may pass a function that calls it for one folder. Each
    returned pair gains "split_set" ("development" or "test"). The run stops if a pair's
    patient ID does not belong to the folder it was found in, or if one patient is paired
    in more than one folder.

    A set made of several folders is paired one folder after the other, and the pairs are
    returned in that folder order. The Drive-to-local copy follows the same order, so one
    Drive folder is read completely before the next one is opened.
    """
    split_folders = dict(SPLIT_FOLDERS if split_folders is None else split_folders)
    all_pairs, seen = [], {}
    for split_set, roots in split_folders.items():
        roots = split_folder_roots(roots)
        for root_number, root in enumerate(roots, start=1):
            folder_text = split_folder_label(split_set, root_number, len(roots))
            print(f"[{context}] Pairing images and masks in the {folder_text}: {root}")
            for item in collect_fn(root):
                cid = canonical_patient_id(item.get("canonical_id", item.get("patient_id")))
                recorded = SPLIT_FOLDER_MEMBERSHIP.get(cid)
                if recorded is None:
                    raise ValueError(
                        f"{context}: patient {cid} was paired in the {folder_text} but was "
                        "not found by the Section 0 folder scan. Rerun Section 0 after any change to "
                        "the DEVELOPMENT/TEST folders."
                    )
                if recorded != split_set:
                    raise ValueError(
                        f"{context}: patient {cid} was paired in the {folder_text}, but the "
                        f"Section 0 folder scan recorded it in the {recorded.upper()} set."
                    )
                if cid in seen:
                    raise ValueError(f"{context}: patient {cid} is paired in more than one folder.")
                seen[cid] = split_set
                all_pairs.append({**item, "canonical_id": cid, "split_set": split_set})
    n_dev_pairs = sum(1 for p in all_pairs if p["split_set"] == SPLIT_SET_DEVELOPMENT)
    _dev_s = "s" if len(split_folder_roots(split_folders[SPLIT_SET_DEVELOPMENT])) > 1 else ""
    _test_s = "s" if len(split_folder_roots(split_folders[SPLIT_SET_TEST])) > 1 else ""
    print(
        f"[{context}] Paired image+mask cases: DEVELOPMENT folder{_dev_s}={n_dev_pairs} | "
        f"TEST folder{_test_s}={len(all_pairs) - n_dev_pairs}"
    )
    return all_pairs


def cache_pairs_locally(pairs, context="local cache", cache_dir=None, max_copy_retries=3):
    """Copy each patient's image and mask from Drive to local disk - once for the whole notebook.

    Every section calls this one function, so all sections share one local layout:
        <LOCAL_CACHE_DIR>/patient_<ID>/image_<file name on Drive>
        <LOCAL_CACHE_DIR>/patient_<ID>/mask_<file name on Drive>
    One folder per patient and the image_/mask_ prefixes mean no file can overwrite
    another patient's file. Nothing is renamed on Drive.

    A file that an earlier section already copied is reused while it still matches the
    Drive file: same size and not older than it. Its content is not read from Drive
    again. A Drive file that was replaced or edited after it was copied is copied again,
    so a corrected image or mask is never ignored. A new file is written under a
    temporary name and renamed only when complete, so an interrupted copy cannot leave a
    half-written file behind.

    Returns new pair records, in the same order, whose "image" and "mask" are the local files.
    """
    import shutil as _shutil
    import time as _time
    from pathlib import Path as _Path
    from tqdm.auto import tqdm as _tqdm

    cache_root = _Path(str(LOCAL_CACHE_DIR if cache_dir is None else cache_dir))
    cached_pairs, n_copied, n_reused, n_refreshed = [], 0, 0, 0
    for item in _tqdm(list(pairs), desc="Caching files locally"):
        cid = canonical_patient_id(item.get("canonical_id", item.get("patient_id")))
        if not cid:
            raise ValueError(f"Cannot cache pair with blank patient ID: {item}")
        patient_dir = cache_root / f"patient_{cid}"
        patient_dir.mkdir(parents=True, exist_ok=True)
        cached_item = dict(item)
        for key in ("image", "mask"):
            src = _Path(item[key])
            if src.parent == patient_dir:
                continue  # This record already points at its local copy.
            dst = patient_dir / f"{key}_{src.name}"
            src_stat = src.stat()
            src_size = src_stat.st_size
            is_stale = False
            if dst.exists():
                dst_stat = dst.stat()
                is_stale = (dst_stat.st_size != src_size) or (src_stat.st_mtime > dst_stat.st_mtime)
            if dst.exists() and not is_stale:
                n_reused += 1
            else:
                if is_stale:
                    n_refreshed += 1
                tmp = dst.with_name(dst.name + ".tmp")
                last_error = None
                for attempt in range(1, int(max_copy_retries) + 1):
                    try:
                        _shutil.copyfile(str(src), str(tmp))
                        tmp_size = tmp.stat().st_size
                        if tmp_size != src_size:
                            raise IOError(f"Size mismatch: source={src_size}, local copy={tmp_size}")
                        tmp.replace(dst)
                        last_error = None
                        break
                    except Exception as copy_error:
                        last_error = copy_error
                        try:
                            if tmp.exists():
                                tmp.unlink()
                        except Exception:
                            pass
                        _time.sleep(0.5 * attempt)
                if last_error is not None:
                    raise RuntimeError(
                        f"[{context}] Could not copy {src} to {dst} after "
                        f"{int(max_copy_retries)} attempt(s): {last_error!r}"
                    ) from last_error
                n_copied += 1
            cached_item[key] = str(dst)
        cached_pairs.append(cached_item)
    print(
        f"[{context}] Local copies: {n_copied} file(s) read from Drive "
        f"({n_refreshed} of them replaced an outdated local copy), "
        f"{n_reused} already on local disk and reused | {cache_root}"
    )
    return cached_pairs


# Scan every folder now, once. Every section reads this one membership record.
SPLIT_FOLDER_MEMBERSHIP, SPLIT_FOLDER_OF_PATIENT = scan_split_folders()
DEV_FOLDER_PATIENT_IDS = sorted(
    [cid for cid, s in SPLIT_FOLDER_MEMBERSHIP.items() if s == SPLIT_SET_DEVELOPMENT],
    key=patient_id_sort_key,
)
TEST_FOLDER_PATIENT_IDS = sorted(
    [cid for cid, s in SPLIT_FOLDER_MEMBERSHIP.items() if s == SPLIT_SET_TEST],
    key=patient_id_sort_key,
)

# Which holdout test set each test patient belongs to: the TEST_ROOT folder holding its files.
_TEST_SET_NAME_BY_ROOT = {
    os.path.normpath(str(root)): name for name, root in zip(TEST_SET_NAMES, TEST_SET_ROOTS)
}
TEST_SET_OF_PATIENT = {
    cid: _TEST_SET_NAME_BY_ROOT[os.path.normpath(str(SPLIT_FOLDER_OF_PATIENT[cid]))]
    for cid in TEST_FOLDER_PATIENT_IDS
}
TEST_SET_PATIENT_IDS = {
    name: [cid for cid in TEST_FOLDER_PATIENT_IDS if TEST_SET_OF_PATIENT[cid] == name]
    for name in TEST_SET_NAMES
}


def test_set_of_patient(patient_id):
    """Name of the holdout test set of one patient, or None for a development patient."""
    return TEST_SET_OF_PATIENT.get(canonical_patient_id(patient_id))


def test_set_file_tag(test_set_name):
    """Text added to the name of a per-test-set output file.

    Empty when the run has one test set, so file names are then exactly as in a
    single-test-set run; "__<test set>" when there are two.
    """
    if len(TEST_SET_NAMES) <= 1:
        return ""
    return "__" + _test_set_token(test_set_name)


def holdout_test_sets(patient_ids, labels=None, context="holdout test"):
    """Divide the holdout patients of one analysis into the separate test sets.

    patient_ids are the holdout patients in the order the analysis uses them; labels are
    their 0/1 outcomes, if given. Returns [(test set name, positions)] in TEST_ROOT order,
    where positions index into patient_ids. Stops the run, before any training, if a
    test set has no patient in this analysis or, when labels are given, holds only one
    outcome class, because an AUC cannot be calculated for such a test set.
    """
    ids = [canonical_patient_id(v) for v in list(patient_ids)]
    not_test = [cid for cid in ids if cid not in TEST_SET_OF_PATIENT]
    if not_test:
        raise ValueError(f"{context}: these patients are in no TEST_ROOT folder: {not_test[:30]}")
    label_list = None if labels is None else [int(v) for v in list(labels)]
    if label_list is not None and len(label_list) != len(ids):
        raise ValueError(f"{context}: {len(ids)} holdout patients but {len(label_list)} labels.")
    result = []
    for name in TEST_SET_NAMES:
        positions = [i for i, cid in enumerate(ids) if TEST_SET_OF_PATIENT[cid] == name]
        if not positions:
            raise ValueError(
                f"{context}: test set '{name}' has no patient in this analysis "
                f"({TEST_SET_ROOT_BY_NAME[name]}). Check that its patients have labels in LABELS_CSV, "
                "or mark the folder as unused by putting 'none' in its TEST_ROOT path."
            )
        if label_list is not None:
            n_pos = sum(label_list[i] for i in positions)
            if n_pos == 0 or n_pos == len(positions):
                raise ValueError(
                    f"{context}: test set '{name}' has {len(positions)} patient(s) but only one "
                    f"outcome class (positives={n_pos}). An AUC needs both classes in every test set."
                )
        result.append((name, positions))
    return result


def rng_state_snapshot():
    """Current state of the Python, NumPy and PyTorch random-number generators."""
    state = {"python": random.getstate(), "numpy": np.random.get_state()}
    _torch = sys.modules.get("torch")
    if _torch is not None:
        state["torch"] = _torch.get_rng_state()
        if _torch.cuda.is_available():
            state["torch_cuda"] = _torch.cuda.get_rng_state_all()
    return state


def rng_state_restore(state):
    """Put the random-number generators back to a state from rng_state_snapshot().

    Used around the scoring of a second holdout test set, so that everything computed
    afterwards is exactly what it would be in a run with one test set. None does nothing.
    """
    if state is None:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    _torch = sys.modules.get("torch")
    if _torch is not None and "torch" in state:
        _torch.set_rng_state(state["torch"])
        if "torch_cuda" in state:
            _torch.cuda.set_rng_state_all(state["torch_cuda"])


def order_by_test_set(df, sort_col=None, ascending=False):
    """Rows of a result table grouped by test set in TEST_ROOT order.

    Within each test set the rows are sorted by sort_col, when given. A table without a
    "test_set" column is only sorted by sort_col.
    """
    out = df
    if sort_col is not None and sort_col in out.columns:
        out = out.sort_values(sort_col, ascending=ascending, na_position="last", kind="mergesort")
    if "test_set" in out.columns:
        rank = {name: i for i, name in enumerate(TEST_SET_NAMES)}
        out = out.sort_values(
            "test_set", key=lambda s: s.map(lambda v: rank.get(v, len(rank))), kind="mergesort"
        )
    return out


def patient_split_record_key(outcome_tag) -> str:
    """Return the ONE split-record key used by every section and outcome definition."""
    tag = str(outcome_tag).strip()
    if not tag:
        raise ValueError("A non-empty outcome tag is required for the patient split.")
    return GLOBAL_PATIENT_SPLIT_KEY


def infer_patient_split_key_from_feature_file(path) -> str:
    """Read the outcome tag from a Section 2 patient-feature filename."""
    name = os.path.basename(str(path))
    match = re.search(r"__label_([^./\\]+)\.csv$", name, flags=re.IGNORECASE)
    if match:
        return patient_split_record_key(match.group(1))
    if len(SHARED_OUTCOMES) == 1:
        return patient_split_record_key(SHARED_OUTCOMES[0])
    raise ValueError(
        f"Cannot infer the outcome tag from feature filename '{name}'. "
        "Expected a name ending in '__label_<tag>.csv'."
    )


def get_or_create_patient_split(split_key, patient_ids, context):
    """Divide patients by folder once, then check that every later section uses exactly the same division.

    Development = patients found in DEVELOPMENT_ROOT; holdout test = patients found in
    the TEST_ROOT folder(s), each folder being its own test set (see scan_split_folders()
    and holdout_test_sets()). Patient-ID order plays no part in the split;
    within each group patients are listed in ascending ID order only so that printed
    lists and exported tables have a fixed row order.
    """
    canonical_ids = _canonical_unique_patient_ids(patient_ids, context=context)
    ordered_ids = tuple(sorted(canonical_ids, key=patient_id_sort_key))

    outside = [cid for cid in ordered_ids if cid not in SPLIT_FOLDER_MEMBERSHIP]
    if outside:
        raise ValueError(
            f"{context}: {len(outside)} patient(s) are in neither the DEVELOPMENT nor the TEST "
            f"folder: {outside[:30]}{' ...' if len(outside) > 30 else ''}. "
            "Rerun Section 0 after any change to the folders."
        )
    dev_ids = tuple(cid for cid in ordered_ids if SPLIT_FOLDER_MEMBERSHIP[cid] == SPLIT_SET_DEVELOPMENT)
    test_ids = tuple(cid for cid in ordered_ids if SPLIT_FOLDER_MEMBERSHIP[cid] == SPLIT_SET_TEST)
    if not dev_ids:
        raise ValueError(
            f"{context}: no patients from the DEVELOPMENT folder are in this cohort ({DEVELOPMENT_ROOT_TEXT})."
        )
    if not test_ids:
        raise ValueError(
            f"{context}: no patients from the TEST folder are in this cohort ({TEST_ROOT_TEXT}); "
            "a holdout test set is required."
        )

    key = GLOBAL_PATIENT_SPLIT_KEY
    current = PATIENT_SPLIT_RECORDS.get(key)
    if current is None:
        current = {
            "split_rule": "folder",
            "development_root": str(DEVELOPMENT_ROOT_TEXT),
            "test_root": str(TEST_ROOT_TEXT),
            "all_ids": ordered_ids,
            "dev_ids": dev_ids,
            "test_ids": test_ids,
            "test_sets": {
                name: tuple(cid for cid in test_ids if TEST_SET_OF_PATIENT[cid] == name)
                for name in TEST_SET_NAMES
            },
            "created_by": str(context),
        }
        PATIENT_SPLIT_RECORDS[key] = current
        print(
            f"[PATIENT SPLIT] Created patient-split record '{key}' from {context}: "
            f"development (DEVELOPMENT folder)={len(dev_ids)}, holdout (TEST folder)={len(test_ids)}."
        )
        if len(TEST_SET_NAMES) > 1:
            for _name, _ids in current["test_sets"].items():
                print(f"[PATIENT SPLIT]   holdout test set '{_name}': {len(_ids)} patient(s)")
    else:
        if (current.get("development_root"), current.get("test_root")) != (str(DEVELOPMENT_ROOT_TEXT), str(TEST_ROOT_TEXT)):
            raise ValueError(
                f"{context}: patient-split record '{key}' was created from different folders "
                f"({current.get('development_root')} | {current.get('test_root')}). "
                "Restart the Colab session and rerun from Section 0."
            )
        expected = tuple(current["all_ids"])
        if ordered_ids != expected:
            expected_set, actual_set = set(expected), set(ordered_ids)
            missing = sorted(expected_set - actual_set, key=patient_id_sort_key)
            unexpected = sorted(actual_set - expected_set, key=patient_id_sort_key)
            raise ValueError(
                f"{context}: patient list does not match patient-split record '{key}'. "
                f"Missing={missing}; unexpected={unexpected}; "
                f"expected_n={len(expected)}, actual_n={len(ordered_ids)}."
            )
        if dev_ids != tuple(current["dev_ids"]) or test_ids != tuple(current["test_ids"]):
            raise ValueError(
                f"{context}: development/test membership differs from patient-split record '{key}'. "
                "Restart the Colab session and rerun from Section 0."
            )

    return list(current["dev_ids"]), list(current["test_ids"])


def print_patient_split(section, dev_ids, test_ids):
    """Print the complete patient lists so the development and holdout groups can be checked."""
    dev_ids = [canonical_patient_id(v) for v in dev_ids]
    test_ids = [canonical_patient_id(v) for v in test_ids]
    overlap = sorted(set(dev_ids) & set(test_ids), key=patient_id_sort_key)
    if overlap:
        raise RuntimeError(f"{section}: development/test patient overlap: {overlap}")
    wrong_dev = [cid for cid in dev_ids if SPLIT_FOLDER_MEMBERSHIP.get(cid) != SPLIT_SET_DEVELOPMENT]
    wrong_test = [cid for cid in test_ids if SPLIT_FOLDER_MEMBERSHIP.get(cid) != SPLIT_SET_TEST]
    if wrong_dev or wrong_test:
        raise RuntimeError(
            f"{section}: patients placed in the wrong group for their folder: "
            f"development but not in the DEVELOPMENT folder={wrong_dev}; "
            f"test but not in the TEST folder={wrong_test}"
        )
    print(f"\n[PATIENT SPLIT CHECK] {section}")
    print(f"  Split rule: folder location | DEVELOPMENT={DEVELOPMENT_ROOT_TEXT} | TEST={TEST_ROOT_TEXT}")
    print(f"  DEVELOPMENT ({len(dev_ids)}): {dev_ids}")
    if len(TEST_SET_NAMES) <= 1:
        print(f"  HOLDOUT TEST ({len(test_ids)}): {test_ids}")
    else:
        print(f"  HOLDOUT TEST, {len(TEST_SET_NAMES)} separate test sets ({len(test_ids)} patients in total):")
        for _name in TEST_SET_NAMES:
            _ids = [cid for cid in test_ids if TEST_SET_OF_PATIENT.get(cid) == _name]
            print(f"    TEST SET '{_name}' ({len(_ids)}): {_ids}")

# ── Visual check of the analysed slices: defined once, used by Sections 1 and 2 ───
# Renders the slices exactly as they reach the CNN: after tumour-centred cropping,
# intensity normalisation and resampling to CNN_INPUT_SIZE. The green outline is the
# segmentation carried in channel 2, so the crop framing can be judged directly.
def preview_analysed_slices(pairs, slice_fn, n_patients=QC_N_PATIENTS,
                            per_fig=QC_PATIENTS_PER_FIG, title=""):
    """Show the CNN input slices for the first n_patients.

    slice_fn(item) must return either a (S, 3, H, W) tensor or a
    (tensor, slice_rows) pair; channel 0 is the image and channel 2 the mask.
    """
    if not SHOW_ANALYSED_SLICES:
        print("[QC] Slice preview skipped (SHOW_ANALYSED_SLICES = False).")
        return
    items = list(pairs)[:int(n_patients)]
    if not items:
        print("[QC] No patients available to preview.")
        return

    import matplotlib.pyplot as plt

    for start in range(0, len(items), int(per_fig)):
        chunk = items[start:start + int(per_fig)]
        rendered = []
        for item in chunk:
            try:
                out = slice_fn(item)
                xs, meta = out if isinstance(out, tuple) else (out, None)
                rendered.append((item, np.asarray(xs.detach().cpu().numpy()), meta))
            except Exception as exc:
                print(f"[QC] {item.get('patient_id')}: could not render ({exc!r})")
        if not rendered:
            continue

        ncols = max(int(x.shape[0]) for _, x, _ in rendered)
        nrows = len(rendered)
        fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.6 * nrows),
                                 squeeze=False)
        for r, (item, xs, meta) in enumerate(rendered):
            for c in range(ncols):
                ax = axes[r][c]
                ax.axis("off")
                if c >= xs.shape[0]:
                    continue
                img = xs[c, 0]
                msk = xs[c, 2] if xs.shape[1] > 2 else None
                ax.imshow(img, cmap="gray", vmin=0.0, vmax=1.0)
                if msk is not None and float(msk.max()) > 0:
                    ax.contour(msk, levels=[0.5], colors="lime", linewidths=0.8)
                    frac = float((msk > 0.5).mean()) * 100.0
                else:
                    frac = float("nan")
                if meta is not None and c < len(meta):
                    plane = meta[c].get("plane", ("axial", "sagittal", "coronal")[c])
                    ax.set_title(f"{plane} | {frac:.0f}%", fontsize=7)
                else:
                    ax.set_title(f"slice {c + 1} | {frac:.0f}%", fontsize=7)
            axes[r][0].text(-0.06, 0.5,
                            f"{item.get('patient_id')}\nlabel={item.get('label')}",
                            transform=axes[r][0].transAxes, fontsize=8,
                            va="center", ha="right")
        n_from = start + 1
        fig.suptitle(f"{title}patients {n_from}-{n_from + len(rendered) - 1} "
                     f"| {ncols} slice(s) per patient | % = tumour area of the crop",
                     fontsize=9)
        fig.tight_layout(rect=(0.04, 0, 1, 0.96))
        plt.show()
        plt.close(fig)

# ── Score-to-class cut-offs, reported side by side ────────────────────────
#   1) Out-of-fold cut-off: Youden's J from development out-of-fold predictions only,
#      then applied unchanged to the holdout. This is the primary rule.
#   2) Top-risk cut-off: the highest-scoring TOP_RISK_PERCENT of the evaluated group.
#      Secondary; it does not replace (1).
TOP_RISK_PERCENT = 25.0

def top_risk_mode_enabled():
    """State whether the additional top-risk result should be calculated and reported."""
    return TOP_RISK_PERCENT is not False and TOP_RISK_PERCENT is not None

def threshold_for_top_risk(scores, percentage=None):
    """
    Return the score that separates the requested highest-risk percentage from the rest.

    Scores equal to or above this value are classified as positive. CNN and machine-learning
    scores are usually continuous, so equal scores at the cut-off are uncommon. When equal scores
    occur, slightly more than the requested percentage may be classified as positive.
    """
    values = np.asarray(scores, dtype=float).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")

    pct = TOP_RISK_PERCENT if percentage is None else percentage
    if pct is False or pct is None:
        raise ValueError("Top-risk evaluation is disabled.")

    pct = float(pct)
    if not 0.0 <= pct <= 100.0:
        raise ValueError("TOP_RISK_PERCENT must be from 0 to 100, or False.")

    if pct == 0.0:
        return float(np.nextafter(np.max(values), np.inf))
    if pct == 100.0:
        return float(np.min(values))

    requested_count = values.size * pct / 100.0
    n_positive = max(1, int(np.floor(requested_count + 0.5)))
    n_positive = min(values.size, n_positive)
    ordered = np.sort(values)[::-1]
    return float(ordered[n_positive - 1])

# ── AUC confidence intervals: defined once, shared by all three sections ───
# Three patient-level estimates: DeLong analytic, delete-one jackknife and a
# stratified nonparametric bootstrap percentile interval. All three read the saved
# labels and scores without retraining. The bootstrap resamples within class
# so every replicate contains both classes.
# Sections 1, 2 and 3 all take calculate_auc_intervals() from here, so each section
# runs on its own, whichever of the other two are switched on or off.
from sklearn.metrics import roc_auc_score

AUC_CI_CONFIDENCE = 0.95
AUC_BOOTSTRAP_REPEATS = 2000


def _prepare_auc_inputs(y_true, y_score):
    """Return valid one-dimensional labels and scores for AUC calculations."""
    y_true = np.asarray(y_true, dtype=int).ravel()
    y_score = np.asarray(y_score, dtype=float).ravel()

    valid = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = y_true[valid]
    y_score = y_score[valid]

    if y_true.size == 0 or y_true.size != y_score.size:
        raise ValueError("AUC requires equal non-empty label and score arrays.")
    if not np.all(np.isin(y_true, [0, 1])):
        raise ValueError("AUC labels must contain only 0 and 1.")
    if np.unique(y_true).size != 2:
        raise ValueError("AUC requires both outcome classes.")

    return y_true, y_score


def format_auc_with_ci(auc, ci_lower, ci_upper) -> str:
    """Format AUC and a two-sided confidence interval."""
    values = np.asarray([auc, ci_lower, ci_upper], dtype=float)
    if not np.all(np.isfinite(values)):
        return "nan"
    return f"{float(auc):.3f} ({float(ci_lower):.3f}–{float(ci_upper):.3f})"


def delong_auc_ci(y_true, y_score, confidence: float = AUC_CI_CONFIDENCE):
    """
    Calculate AUC and a two-sided DeLong confidence interval.

    This is the single-model DeLong variance estimate. It uses pairwise
    comparisons between positive and negative test scores and a normal
    approximation for the confidence limits.
    """
    from statistics import NormalDist

    try:
        y_true, y_score = _prepare_auc_inputs(y_true, y_score)
    except ValueError:
        nan = float("nan")
        return nan, nan, nan

    positive_scores = y_score[y_true == 1]
    negative_scores = y_score[y_true == 0]
    n_positive = positive_scores.size
    n_negative = negative_scores.size

    if n_positive < 2 or n_negative < 2:
        nan = float("nan")
        return nan, nan, nan

    comparisons = (
        (positive_scores[:, None] > negative_scores[None, :]).astype(float)
        + 0.5 * (positive_scores[:, None] == negative_scores[None, :]).astype(float)
    )

    positive_placements = comparisons.mean(axis=1)
    negative_placements = comparisons.mean(axis=0)
    auc = float(positive_placements.mean())

    variance = (
        np.var(positive_placements, ddof=1) / n_positive
        + np.var(negative_placements, ddof=1) / n_negative
    )
    standard_error = float(np.sqrt(max(float(variance), 0.0)))

    alpha = 1.0 - float(confidence)
    z_value = NormalDist().inv_cdf(1.0 - alpha / 2.0)
    lower = max(0.0, auc - z_value * standard_error)
    upper = min(1.0, auc + z_value * standard_error)
    return auc, float(lower), float(upper)


def jackknife_auc_ci(y_true, y_score, confidence: float = AUC_CI_CONFIDENCE):
    """
    Calculate a delete-one jackknife confidence interval for AUC.

    The function removes one test patient at a time. The CI uses the standard
    jackknife standard error with a normal approximation. It also returns the
    lowest and highest leave-one-out AUC values.
    """
    from statistics import NormalDist

    try:
        y_true, y_score = _prepare_auc_inputs(y_true, y_score)
    except ValueError:
        nan = float("nan")
        return nan, nan, nan, nan, nan

    observed_auc = float(roc_auc_score(y_true, y_score))
    n_patients = y_true.size
    leave_one_out_aucs = []

    for patient_index in range(n_patients):
        keep = np.ones(n_patients, dtype=bool)
        keep[patient_index] = False

        # Undefined if deleting a patient leaves one class.
        if np.unique(y_true[keep]).size != 2:
            continue

        leave_one_out_aucs.append(
            float(roc_auc_score(y_true[keep], y_score[keep]))
        )

    leave_one_out_aucs = np.asarray(leave_one_out_aucs, dtype=float)
    if leave_one_out_aucs.size < 2:
        nan = float("nan")
        return observed_auc, nan, nan, nan, nan

    jackknife_mean = float(leave_one_out_aucs.mean())
    jackknife_se = float(
        np.sqrt(
            ((leave_one_out_aucs.size - 1.0) / leave_one_out_aucs.size)
            * np.sum((leave_one_out_aucs - jackknife_mean) ** 2)
        )
    )

    alpha = 1.0 - float(confidence)
    z_value = NormalDist().inv_cdf(1.0 - alpha / 2.0)
    lower = max(0.0, observed_auc - z_value * jackknife_se)
    upper = min(1.0, observed_auc + z_value * jackknife_se)

    return (
        observed_auc,
        float(lower),
        float(upper),
        float(np.min(leave_one_out_aucs)),
        float(np.max(leave_one_out_aucs)),
    )


def bootstrap_auc_ci(
    y_true,
    y_score,
    confidence: float = AUC_CI_CONFIDENCE,
    n_bootstrap: int = AUC_BOOTSTRAP_REPEATS,
    random_state: int = GLOBAL_SEED,
):
    """
    Calculate a stratified nonparametric patient-level bootstrap CI for AUC.

    Positive and negative patients are independently sampled with replacement,
    preserving the original class counts in every replicate. The CI is the
    percentile interval across bootstrap AUC values.
    """
    try:
        y_true, y_score = _prepare_auc_inputs(y_true, y_score)
    except ValueError:
        nan = float("nan")
        return nan, nan, nan, 0

    observed_auc = float(roc_auc_score(y_true, y_score))
    pos_idx = np.flatnonzero(y_true == 1)
    neg_idx = np.flatnonzero(y_true == 0)

    if pos_idx.size < 1 or neg_idx.size < 1 or int(n_bootstrap) < 2:
        nan = float("nan")
        return observed_auc, nan, nan, 0

    rng = np.random.default_rng(int(random_state))
    aucs = np.empty(int(n_bootstrap), dtype=float)

    for b in range(int(n_bootstrap)):
        sample_pos = rng.choice(pos_idx, size=pos_idx.size, replace=True)
        sample_neg = rng.choice(neg_idx, size=neg_idx.size, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        aucs[b] = roc_auc_score(y_true[sample_idx], y_score[sample_idx])

    alpha = 1.0 - float(confidence)
    lower = float(np.quantile(aucs, alpha / 2.0))
    upper = float(np.quantile(aucs, 1.0 - alpha / 2.0))
    return observed_auc, lower, upper, int(aucs.size)


def calculate_auc_intervals(y_true, y_score, confidence: float = AUC_CI_CONFIDENCE):
    """
    Return AUC plus DeLong, delete-one Jackknife and patient-level Bootstrap CIs.
    """
    auc, delong_lower, delong_upper = delong_auc_ci(
        y_true,
        y_score,
        confidence=confidence,
    )
    (
        jackknife_auc,
        jackknife_lower,
        jackknife_upper,
        jackknife_min,
        jackknife_max,
    ) = jackknife_auc_ci(
        y_true,
        y_score,
        confidence=confidence,
    )
    (
        bootstrap_auc,
        bootstrap_lower,
        bootstrap_upper,
        bootstrap_repeats,
    ) = bootstrap_auc_ci(
        y_true,
        y_score,
        confidence=confidence,
    )

    # All three estimate the same observed AUC; take DeLong's, else jackknife,
    # else bootstrap.
    if not np.isfinite(auc):
        auc = jackknife_auc
    if not np.isfinite(auc):
        auc = bootstrap_auc

    return {
        "AUC": float(auc),
        "DeLong_CI_lower": float(delong_lower),
        "DeLong_CI_upper": float(delong_upper),
        "AUC_with_DeLong_95CI": format_auc_with_ci(
            auc, delong_lower, delong_upper
        ),
        "Jackknife_CI_lower": float(jackknife_lower),
        "Jackknife_CI_upper": float(jackknife_upper),
        "AUC_with_Jackknife_95CI": format_auc_with_ci(
            auc, jackknife_lower, jackknife_upper
        ),
        "Jackknife_min_AUC": float(jackknife_min),
        "Jackknife_max_AUC": float(jackknife_max),
        "Bootstrap_CI_lower": float(bootstrap_lower),
        "Bootstrap_CI_upper": float(bootstrap_upper),
        "AUC_with_Bootstrap_95CI": format_auc_with_ci(
            auc, bootstrap_lower, bootstrap_upper
        ),
        "Bootstrap_repeats": int(bootstrap_repeats),
    }

# ── Table display ─────────────────────────────────────────────────────────
# Rows are capped so that an accidental display() of a wide feature table cannot
# flood the notebook. Result tables print through to_string(), which is unaffected.
MAX_DISPLAY_ROWS = 200
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", MAX_DISPLAY_ROWS)

# ── Section 2 feature selection and classifiers ───────────────────────────
# Each entry is one combination of selection methods, applied as a consensus vote:
#   N_MRMR / mRMR  relevance minus redundancy      N_BORUTA  shadow-feature test
#   N_L1           sparse logistic regression      N_ENET    elastic-net logistic
#   RFE            recursive elimination (available, not selected)
USE_FEATURE_SELECTOR_VOTING = True
RUN_FEATURE_SELECTOR_COMBINATIONS = [
    ("N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET"),
]
FEATURE_SELECTOR_TOP_K = 25       # nominations per method
FEATURE_SELECTOR_MIN_KEEP = 4     # floor on features kept
FEATURE_SELECTOR_MIN_VOTES = 3    # votes needed to keep a feature

# Classifiers compared and tuned in Section 2. Valid names:
# SVM, LDA, LOGREG, ADABOOST, RANDOM_FOREST, DECISION_TREE, XGBOOST,
# NAIVE_BAYES, GAUSSIAN_PROCESS, AE, EXTRATREES, HIST_GBDT, SGD_LOGLOSS,
# LIGHTGBM, CATBOOST, BAGGING
RUN_CLASSIFIERS = [
    "NAIVE_BAYES",

    "LOGREG",

]

# Architectures trainable end to end (Sections 1 and 3). Anything in SHARED_BACKBONES
# outside this set is feature-extraction only: InceptionV3 needs 299-pixel input and
# Choi-IDH is used with its released checkpoint unchanged.
_SEG1_SUPPORTED_BACKBONES = {
    "radimagenet_resnet10t", "radimagenet_resnet18",
    "radimagenet_resnet50", "radimagenet_densenet121",
    "imagenet_resnet10t", "imagenet_resnet18",
    "imagenet_resnet50", "imagenet_densenet121",
}

# ── Result tables and folders shared between sections ─────────────────────
RESULTS_REGISTRY = {}  # result table of each section, read by the final overview

# Result sub-folders, derived from OUT_ROOT set at the top of this cell.

PART1_OUT_DIR         = os.path.join(OUT_ROOT, "PART1_frozen_e2e_cnn")
PART2_OUT_DIR         = os.path.join(OUT_ROOT, "PART2_cnn_features")
SEG2_FEATURE_DIR      = os.path.join(PART2_OUT_DIR, "patient_features")
PART3_OUT_DIR         = os.path.join(OUT_ROOT, "PART3_cnn_ml")
FINETUNE_OUT_DIR      = os.path.join(OUT_ROOT, "PART4_finetuning")
COMBINED_OVERVIEW_DIR = os.path.join(OUT_ROOT, "combined_overview")
# Tables, figures and masks of the optional automatic segmentation check (Section 0, cell 5).
AUTOSEG_OUT_DIR       = os.path.join(OUT_ROOT, "AUTO_SEGMENTATION_CHECK")

for _d in (
    OUT_ROOT, PART1_OUT_DIR, PART2_OUT_DIR, SEG2_FEATURE_DIR,
    PART3_OUT_DIR, FINETUNE_OUT_DIR, COMBINED_OVERVIEW_DIR,
):
    os.makedirs(_d, exist_ok=True)

print("[SECTION 0] CNN architectures for Sections 1 to 3:", SHARED_BACKBONES)
print("[SECTION 0] Label file of this run:", LABELS_CSV, "| outcome name:", OUTCOME_NAME)
print(f"[SECTION 0] Central planes per patient: {SLICE_PLANES}")
print("[SECTION 0] Development/holdout split: by folder location")
print(f"[SECTION 0]   DEVELOPMENT folder(s): {len(DEV_FOLDER_PATIENT_IDS)} patient ID(s) | {DEVELOPMENT_ROOT_TEXT}")
for _name in TEST_SET_NAMES:
    print(f"[SECTION 0]   TEST set '{_name}': {len(TEST_SET_PATIENT_IDS[_name])} patient ID(s) | "
          f"{TEST_SET_ROOT_BY_NAME[_name]}")
if len(TEST_SET_NAMES) > 1:
    print(f"[SECTION 0]   {len(TEST_SET_NAMES)} holdout test sets: each is scored and reported separately.")
print("[SECTION 0] Percentage classified as top risk:", TOP_RISK_PERCENT)
print("[SECTION 0] CNN architectures trained end to end (Sections 1 and 3):",
      [b for b in SHARED_BACKBONES if b.lower() in _SEG1_SUPPORTED_BACKBONES])
print("[SECTION 0] Tumour-crop safety margin (all image sections):", f"{CROP_SAFETY_MARGIN_PX} native pixels per side")
print("[SECTION 0] Tumour image mode:", TUMOR_CONTEXT_MODE, "| ROI-only neutral background:", ROI_ONLY_BACKGROUND_VALUE)
print("[SECTION 0] Folder for Section 2 feature files:", SEG2_FEATURE_DIR)
print("[SECTION 0] Output root for every section:", OUT_ROOT)
print("[SECTION 0]   Section 1, frozen:                  ", PART1_OUT_DIR)
print("[SECTION 0]   Section 2, efficient_ML features:   ", PART2_OUT_DIR)
print("[SECTION 0]   Section 2, efficient_ML classifiers:", PART3_OUT_DIR + "/<feature-file sub-folder>")
print("[SECTION 0]   Section 3, finetune:                ", FINETUNE_OUT_DIR)
print("[SECTION 0]   Combined final table:               ", COMBINED_OVERVIEW_DIR)
print(f"[SECTION 0] Automatic segmentation check (Section 0, cell 5): nnInteractive={RUN_AUTOSEG_NNINTERACTIVE} | "
      f"SAM 2={RUN_AUTOSEG_SAM2} | folder: {AUTOSEG_OUT_DIR}")
print("[SECTION 0] Feature-selection combinations for Section 2:", RUN_FEATURE_SELECTOR_COMBINATIONS)
print("[SECTION 0] Classifiers for Section 2:", RUN_CLASSIFIERS)


# ── One patient manifest, one label read, one preprocessing pass ──────────────
_SHARED_PATIENT_DATA = None
_SHARED_INPUT_FILES = {}
_SHARED_TENSOR_RAM_CACHE = None
_SHARED_TENSOR_RAM_BYTES = 0
_SHARED_TENSOR_RAM_LIMIT = 128 * 1024 * 1024


def _shared_input_key(image_path, mask_path, out_size, n_slices):
    """Cache only deterministic inputs; labels and model weights are never cached here."""
    import hashlib, json
    from pathlib import Path as _Path
    paths = []
    for path in (image_path, mask_path):
        p = _Path(path)
        stat = p.stat()
        paths.append((str(p.resolve()), stat.st_size, stat.st_mtime_ns))
    settings = (
        "shared-inputs-v1", paths, int(out_size), int(n_slices),
        int(CROP_SAFETY_MARGIN_PX), float(INTENSITY_Z_SPAN_SD),
        str(TUMOR_CONTEXT_MODE).upper(), float(ROI_ONLY_BACKGROUND_VALUE),
        str(CHANNEL_MODE),
    )
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()


def _write_shared_input(path, tensor, metadata):
    """Write compact slices and JSON crop diagnostics without pickled objects."""
    import json
    def scalar(value):
        if isinstance(value, np.generic):
            return value.item()
        raise TypeError(f"Unsupported crop metadata value: {type(value)}")
    tmp = path.with_suffix(".tmp")
    with tmp.open("wb") as fh:
        np.savez(fh, slices=tensor.detach().cpu().numpy(),
                 metadata=np.array(json.dumps(metadata, default=scalar)))
    tmp.replace(path)


def get_shared_patient_inputs(image_path, mask_path, out_size=None):
    """Reuse Section 0's compact slices, with a bounded CPU-memory cache.

    Returns a fresh tensor and metadata copy, so augmentation or caller changes can
    never modify another fold's deterministic input. New preprocessing settings need
    a rerun of Section 0; later sections never silently reread raw volumes.
    """
    import copy, json
    from collections import OrderedDict
    global _SHARED_TENSOR_RAM_CACHE, _SHARED_TENSOR_RAM_BYTES
    if _SHARED_PATIENT_DATA is None:
        raise RuntimeError("Run Section 0, cell 3 to prepare the shared patient inputs first.")
    size = int(CNN_INPUT_SIZE_SHARED if out_size is None else out_size)
    key = _shared_input_key(image_path, mask_path, size, N_SLICES)
    path = _SHARED_INPUT_FILES.get(key)
    if path is None or not path.is_file():
        raise RuntimeError(
            "Shared CNN input is missing or its files/settings changed. "
            "Rerun from Section 0, cell 3; no section will rebuild a different cohort."
        )
    if _SHARED_TENSOR_RAM_CACHE is None:
        _SHARED_TENSOR_RAM_CACHE = OrderedDict()
    entry = _SHARED_TENSOR_RAM_CACHE.pop(key, None)
    if entry is None:
        with np.load(path, allow_pickle=False) as data:
            tensor = torch.from_numpy(data["slices"].copy())
            metadata = json.loads(str(data["metadata"].item()))
        entry = (tensor, metadata)
        size_bytes = tensor.numel() * tensor.element_size()
        while _SHARED_TENSOR_RAM_CACHE and _SHARED_TENSOR_RAM_BYTES + size_bytes > _SHARED_TENSOR_RAM_LIMIT:
            _, (old_tensor, _) = _SHARED_TENSOR_RAM_CACHE.popitem(last=False)
            _SHARED_TENSOR_RAM_BYTES -= old_tensor.numel() * old_tensor.element_size()
        if size_bytes <= _SHARED_TENSOR_RAM_LIMIT:
            _SHARED_TENSOR_RAM_CACHE[key] = entry
            _SHARED_TENSOR_RAM_BYTES += size_bytes
    else:
        _SHARED_TENSOR_RAM_CACHE[key] = entry
    return entry[0].clone(), copy.deepcopy(entry[1])


def get_shared_patient_data(context="workflow"):
    """Copies of one frozen manifest and label tables; no Drive access or reparsing."""
    if _SHARED_PATIENT_DATA is None:
        raise RuntimeError("Run Section 0, cell 3 to prepare the shared patient cohort first.")
    print(f"[{context}] Reusing Section 0 patient manifest and label tables: "
          f"{len(_SHARED_PATIENT_DATA['pairs'])} paired patients; no folder rescan or file copy.")
    return (
        [dict(item) for item in _SHARED_PATIENT_DATA["pairs"]],
        {tag: df.copy(deep=True) for tag, df in _SHARED_PATIENT_DATA["labels"].items()},
    )


def initialize_shared_patient_data():
    """Validate the cohort and prepare every configured input before model training.

    Full volumes are decoded one patient at a time and discarded. All sizes/slice
    counts needed by the enabled workflows are produced from that single decoding.
    Disk and bounded RAM caches hold compact slices, never full-cohort 3D volumes.
    """
    global _SHARED_PATIENT_DATA, N_SLICES, SLICE_PLANES, CHANNEL_MODE
    global sitk, torch, F
    from pathlib import Path as _Path
    from tqdm.auto import tqdm as _tqdm
    import SimpleITK as sitk
    import torch
    import torch.nn.functional as F

    if _SHARED_PATIENT_DATA is not None:
        return get_shared_patient_data("Section 0")
    configs = [c for c in LABELS_CONFIGS if c.get("enabled") and c.get("path")]
    if not configs:
        raise ValueError("No active label table; check LABELS_CSV in Section 0.")
    labels = {c["tag"]: load_label_table(c["path"], context="Section 0 shared cohort") for c in configs}
    pairs = collect_pairs_from_split_folders(collect_generic_image_mask_pairs, context="Section 0 shared cohort")
    if not pairs:
        raise ValueError("No unambiguous image/mask pairs. Check the filenames before training.")

    audit_dir = _Path(OUT_ROOT) / "SHARED_INPUTS_AUDIT"
    audit_dir.mkdir(parents=True, exist_ok=True)
    pair_ids = {p["canonical_id"] for p in pairs}
    audit_rows = []
    for cid in sorted(SPLIT_FOLDER_MEMBERSHIP, key=patient_id_sort_key):
        row = {"canonical_id": cid, "split_set": SPLIT_FOLDER_MEMBERSHIP[cid],
               "test_set": TEST_SET_OF_PATIENT.get(cid, ""), "paired": cid in pair_ids}
        for tag, df in labels.items():
            label_ids = set(df["canonical_id"])
            row[f"included_{tag}"] = cid in pair_ids and cid in label_ids
            row[f"reason_{tag}"] = ("unpaired_or_ambiguous_files" if cid not in pair_ids else
                                    "missing_label" if cid not in label_ids else "included")
        audit_rows.append(row)
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(audit_dir / "patient_cohort_audit.csv", index=False)
    for tag in labels:
        excluded = audit.loc[~audit[f"included_{tag}"]]
        if not excluded.empty:
            print(f"[SHARED COHORT] {tag}: {len(excluded)} folder patient ID(s) excluded in ALL workflows:")
            print(excluded[["canonical_id", "split_set", f"reason_{tag}"]].to_string(index=False))

    # Freeze the split from exactly the labelled, paired patients, before any training.
    labelled_ids_union = set()
    for tag, df in labels.items():
        label_map = dict(zip(df["canonical_id"], df["label"]))
        ids = [p["canonical_id"] for p in pairs if p["canonical_id"] in label_map]
        if len(ids) < 4:
            raise ValueError(f"Shared cohort '{tag}' has only {len(ids)} labelled pairs.")
        dev_ids, test_ids = get_or_create_patient_split(patient_split_record_key(tag), ids,
                                                       context=f"Section 0 preflight | outcome={tag}")
        if len({int(label_map[cid]) for cid in dev_ids}) != 2:
            raise ValueError(f"Shared development cohort '{tag}' needs both outcome classes.")
        holdout_test_sets(test_ids, [label_map[cid] for cid in test_ids], context=f"Section 0 preflight | {tag}")
        labelled_ids_union.update(ids)

    # Copy once. The immutable local-path manifest is reused by all later sections.
    pairs = cache_pairs_locally(pairs, context="Section 0 shared cohort")
    pd.DataFrame(pairs).to_csv(audit_dir / "shared_patient_manifest.csv", index=False)
    n_decoded, n_reused, n_variants = prepare_shared_cnn_inputs(pairs, labelled_ids_union)
    _SHARED_PATIENT_DATA = {"pairs": tuple(dict(p) for p in pairs), "labels": labels}
    print(f"[SHARED INPUT] Ready: {len(pairs)} pairs; {len(labelled_ids_union)} labelled patients; "
          f"{n_variants} input setting(s). Decoded {n_decoded} image/mask pairs once; "
          f"reused {n_reused} prepared inputs. Later sections reuse this cohort and its slices.")
    return get_shared_patient_data("Section 0")


def prepare_shared_cnn_inputs(pairs, labelled_ids_union, context="shared cohort"):
    """Prepare mask-specific deterministic slices from local images, one patient at a time."""
    global N_SLICES, SLICE_PLANES, CHANNEL_MODE
    from pathlib import Path as _Path
    from tqdm.auto import tqdm as _tqdm
    variants = {(int(CNN_INPUT_SIZE_SHARED), int(N_SLICES))}
    if OPTIMIZATION:
        variants.update((int(a["input_size"]), int(a["n_slices"])) for a in OPTIMIZATION_ARMS)
        variants.update((int(a["input_size"]), int(a["n_slices"])) for a in OPTIMIZATION_ML_ARMS)
    input_dir = _Path(LOCAL_CACHE_DIR) / "prepared_cnn_inputs"
    input_dir.mkdir(parents=True, exist_ok=True)
    saved = (N_SLICES, SLICE_PLANES, globals().get("CHANNEL_MODE"))
    CHANNEL_MODE = "t1cc_t1cc_mask"
    n_decoded, n_reused = 0, 0
    try:
        for item in _tqdm([p for p in pairs if p["canonical_id"] in labelled_ids_union],
                          desc=f"Preparing CNN inputs: {context} (before training)"):
            pending = []
            for size, count in sorted(variants):
                key = _shared_input_key(item["image"], item["mask"], size, count)
                path = input_dir / f"{key}.npz"
                if path.is_file():
                    # Reject interrupted/corrupt caches now, not midway through training.
                    try:
                        import json
                        with np.load(path, allow_pickle=False) as cached:
                            valid = (cached["slices"].shape == (count, 3, size, size) and
                                     np.isfinite(cached["slices"]).all() and
                                     len(json.loads(str(cached["metadata"].item()))) == count)
                    except Exception:
                        valid = False
                    if valid:
                        n_reused += 1
                        _SHARED_INPUT_FILES[key] = path
                        continue
                pending.append((size, count, key, path))
            if not pending:
                continue
            try:
                img, mask, spacing = read_sitk_pair(item["image"], item["mask"])
                binary = (mask > 0).astype(np.uint8)
                normalized = normalize_volume_for_cnn(img, binary)
                n_decoded += 1
                for size, count, key, path in pending:
                    N_SLICES = count
                    SLICE_PLANES = ("axial", "sagittal", "coronal")[:count]
                    tensor, metadata = build_central_plane_inputs(normalized, binary, spacing, size)
                    if not np.isfinite(tensor.detach().cpu().numpy()).all():
                        raise ValueError("Preprocessed slices contain non-finite values.")
                    _write_shared_input(path, tensor, metadata)
                    _SHARED_INPUT_FILES[key] = path
                    del tensor, metadata
                del img, mask, binary, normalized
            except Exception as exc:
                raise RuntimeError(
                    f"[SHARED INPUT][PREFLIGHT] patient {item['canonical_id']} could not be prepared: {exc}. "
                    "Fix its image/mask, then rerun Section 0. No model training has started."
                ) from exc
    finally:
        N_SLICES, SLICE_PLANES = saved[:2]
        if saved[2] is None:
            globals().pop("CHANNEL_MODE", None)
        else:
            CHANNEL_MODE = saved[2]
    return n_decoded, n_reused, len(variants)


SEGMENTATION_PIPELINE_CELL_KEYS = [(1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (2, 13), (2, 15), (3, 2), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (3, 11), (3, 12), (3, 13), (3, 14), (3, 15), (3, 16), (3, 17), (4, 2), (5, 2)]

# ── Additional full analyses with selected resegmentation masks ───────────────
SEGMENTATION_ANALYSIS_METHOD = "original"
SEGMENTATION_RESULTS_REGISTRY = {}
_SEGMENTATION_ANALYSIS_JOBS = []
_SEGMENTATION_PIPELINE_SOURCES = {}
_SEGMENTATION_BASE_GLOBALS = None
_SEGMENTATION_BASE_RNG = None
_SEGMENTATION_ANALYSES_COMPLETED = False


def segmentation_cache_path(method, patient_id):
    """One permanent NRRD per method and canonical patient ID, shared by all sections."""
    from pathlib import Path as _Path
    roots = {"sam2_box_per_slice": CAS2_ROOT, "nninteractive_lasso": LASSO_ROOT}
    if method not in roots:
        raise ValueError(f"Unsupported segmentation cache method: {method!r}. Only SAM2 and lasso are supported.")
    cid = canonical_patient_id(patient_id)
    if not cid or cid in (".", "..") or "/" in cid or "\\" in cid:
        raise ValueError(f"Invalid patient ID for segmentation cache: {patient_id!r}")
    return _Path(roots[method]) / f"{cid}.nrrd"


def find_saved_segmentation(method, patient_id):
    """Use the permanent cache; migrate a matching legacy mask once if necessary."""
    import shutil, uuid
    from pathlib import Path as _Path
    destination = segmentation_cache_path(method, patient_id)
    if destination.exists():
        if not destination.is_file():
            raise ValueError(f"Segmentation cache path is not a file: {destination}")
        return destination
    # Older notebook versions saved these same masks below AUTOSEG_OUT_DIR.
    # Explicit patient paths preserve completed work without scanning input folders.
    for subdir in ("masks", "masks__pilot"):
        legacy = _Path(AUTOSEG_OUT_DIR) / subdir / method / destination.name
        if legacy.is_file():
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_name(f".{destination.stem}.{uuid.uuid4().hex}.tmp.nrrd")
            try:
                shutil.copy2(legacy, temporary)
                if not destination.exists():
                    temporary.replace(destination)
            finally:
                temporary.unlink(missing_ok=True)
            print(f"[AUTO-SEG] Reusing legacy mask in permanent cache: {destination}")
            break
    return destination


def selected_segmentation_analysis_methods():
    """Generation and downstream analysis switches are independent."""
    methods = []
    if ANALYZE_NNINTERACTIVE_MASKS:
        methods.append("nninteractive_lasso")
    if ANALYZE_SAM2_MASKS:
        methods.append("sam2_box_per_slice")
    return methods


def _segmentation_namespace(base, method, pairs=None):
    """Bind shared helpers to an isolated analysis state, not the original globals."""
    import copy, types
    from pathlib import Path as _Path
    module_name = "_segmentation_analysis_" + re.sub(r"\W+", "_", method)
    module = types.ModuleType(module_name)
    env = module.__dict__
    env.update(base)
    env["__name__"] = module_name
    sys.modules[module_name] = module
    # Configuration and registries can contain mutable lists/dictionaries.
    for name, value in base.items():
        if isinstance(value, (dict, list, set)) and not name.startswith("__"):
            try:
                env[name] = copy.deepcopy(value)
            except (TypeError, ValueError):
                pass
    env["SEGMENTATION_ANALYSIS_METHOD"] = method
    env["RESULTS_REGISTRY"] = {}
    for name in ("OPTIMIZATION_OOF_STORE", "OPTIMIZATION_FROZEN_OOF_STORE", "OPTIMIZATION_ML_OOF_STORE"):
        env[name] = {}
    env["_SHARED_TENSOR_RAM_CACHE"] = None
    env["_SHARED_TENSOR_RAM_BYTES"] = 0
    env["_SHARED_INPUT_FILES"] = {}
    root = _Path(base["OUT_ROOT"]) / "SEGMENTATION_ANALYSES" / method
    env["OUT_ROOT"] = str(root)
    paths = {
        "PART1_OUT_DIR": "PART1_frozen_e2e_cnn",
        "PART2_OUT_DIR": "PART2_cnn_features",
        "PART3_OUT_DIR": "PART3_cnn_ml",
        "FINETUNE_OUT_DIR": "PART4_finetuning",
        "COMBINED_OVERVIEW_DIR": "combined_overview",
    }
    for name, relative in paths.items():
        env[name] = str(root / relative)
    env["SEG2_FEATURE_DIR"] = str(_Path(env["PART2_OUT_DIR"]) / "patient_features")
    for name in tuple(paths) + ("SEG2_FEATURE_DIR",):
        _Path(env[name]).mkdir(parents=True, exist_ok=True)
    shared = base["_SHARED_PATIENT_DATA"]
    env["_SHARED_PATIENT_DATA"] = {
        "pairs": tuple(dict(p) for p in (shared["pairs"] if pairs is None else pairs)),
        "labels": {tag: df.copy(deep=True) for tag, df in shared["labels"].items()},
    }
    # Functions defined in Section 0 resolve globals dynamically; rebinding is essential
    # for mask-specific caches, output paths and independent optimization stores.
    for name, function in base.items():
        if isinstance(function, types.FunctionType) and function.__globals__ is globals():
            rebound = types.FunctionType(function.__code__, env, function.__name__,
                                         function.__defaults__, function.__closure__)
            rebound.__kwdefaults__ = function.__kwdefaults__
            rebound.__annotations__ = function.__annotations__
            env[name] = rebound
    return env


def _copy_generated_masks(method, source_pairs, expected_ids):
    """Use explicit patient paths; never rediscover the image folders or label CSV."""
    import shutil
    from pathlib import Path as _Path
    generating_now = (RUN_AUTOSEG_SAM2 if method == "sam2_box_per_slice" else RUN_AUTOSEG_NNINTERACTIVE)
    current_rows = globals().get("_autoseg_rows", {})
    selected = []
    missing, failures = [], []
    for item in source_pairs:
        cid = item["canonical_id"]
        if cid not in expected_ids:
            continue
        source = find_saved_segmentation(method, cid)
        # A failed current generation must not fall back to an older saved mask.
        if generating_now:
            status = current_rows.get((cid, method), {}).get("status", "not generated in this run")
            if status != "ok":
                failures.append(f"{cid}: {status}")
                continue
        if not source.is_file():
            missing.append(cid)
            continue
        selected.append((item, source))
    if missing or failures or len(selected) != len(expected_ids):
        raise RuntimeError(
            f"[SEGMENTATION ANALYSIS][PREFLIGHT] {method} does not have a successful saved mask "
            f"for every labelled original-cohort patient. Missing={missing[:30]}; "
            f"failed/unprocessed={failures[:30]}; expected={len(expected_ids)}, available={len(selected)}. "
            "Generate full-cohort masks with AUTOSEG_PILOT_N_PATIENTS=None into CAS2_ROOT/LASSO_ROOT, "
            "then rerun from Section 0. No original-mask substitution or patient dropping is allowed."
        )
    pairs, audit = [], []
    cache_dir = _Path(LOCAL_CACHE_DIR) / "generated_analysis_masks" / method
    cache_dir.mkdir(parents=True, exist_ok=True)
    for item, source in selected:
        destination = cache_dir / f"{item['canonical_id']}_mask.nrrd"
        source_stat = source.stat()
        if (not destination.is_file() or destination.stat().st_size != source_stat.st_size or
                destination.stat().st_mtime_ns < source_stat.st_mtime_ns):
            temporary = destination.with_suffix(".tmp")
            shutil.copyfile(source, temporary)
            if temporary.stat().st_size != source_stat.st_size:
                raise IOError(f"Incomplete generated-mask copy: {source}")
            temporary.replace(destination)
        pairs.append({**item, "mask": str(destination), "segmentation_method": method,
                      "original_mask": item["mask"], "generated_mask_source": str(source)})
        audit.append({"canonical_id": item["canonical_id"], "split_set": item["split_set"],
                      "test_set": TEST_SET_OF_PATIENT.get(item["canonical_id"], ""),
                      "segmentation_method": method, "image": item["image"],
                      "original_mask": item["mask"], "generated_mask_source": str(source),
                      "analysis_mask": str(destination)})
    return pairs, pd.DataFrame(audit)


def _capture_segmentation_pipeline_cell(result):
    """Capture only successful, visible Sections 1–5 cells for exact later replay."""
    if result.error_before_exec is not None or result.error_in_exec is not None:
        return
    source = result.info.raw_cell
    match = re.match(r"\s*#\s*Section\s+([1-5]),\s*cell\s+(\d+)\s*:", source)
    if match:
        _SEGMENTATION_PIPELINE_SOURCES[(int(match.group(1)), int(match.group(2)))] = source


def prepare_segmentation_analysis_jobs():
    """After resegmentation, preflight all requested masks before any CNN training."""
    global _SEGMENTATION_ANALYSIS_JOBS, _SEGMENTATION_BASE_GLOBALS, _SEGMENTATION_BASE_RNG
    global _SEGMENTATION_PIPELINE_SOURCES, _SEGMENTATION_ANALYSES_COMPLETED
    from IPython import get_ipython
    from pathlib import Path as _Path
    ipython = get_ipython()
    if ipython is None:
        raise RuntimeError("Run this .ipynb in Colab/Jupyter: downstream cell capture requires IPython.")
    old_hook = getattr(ipython, "_segmentation_pipeline_capture_hook", None)
    if old_hook is not None:
        ipython.events.unregister("post_run_cell", old_hook)
    _SEGMENTATION_PIPELINE_SOURCES = {}
    _SEGMENTATION_ANALYSIS_JOBS = []
    _SEGMENTATION_ANALYSES_COMPLETED = False
    _SEGMENTATION_BASE_RNG = rng_state_snapshot()
    # Freeze Section 0 settings before workflow cells redefine names and globals.
    _SEGMENTATION_BASE_GLOBALS = dict(_SEGMENTATION_SECTION0_BASE)
    source_pairs = _SHARED_PATIENT_DATA["pairs"] if _SHARED_PATIENT_DATA else ()
    record = PATIENT_SPLIT_RECORDS.get(GLOBAL_PATIENT_SPLIT_KEY, {})
    expected_ids = set(record.get("all_ids", ()))
    if selected_segmentation_analysis_methods() and not (RUN_FROZEN_CNN_E2E or RUN_CNN_LR or RUN_CNN_FINETUNE_E2E):
        raise ValueError("Enable at least one downstream workflow to analyze generated masks.")
    for method in selected_segmentation_analysis_methods():
        pairs, audit = _copy_generated_masks(method, source_pairs, expected_ids)
        env = _segmentation_namespace(_SEGMENTATION_BASE_GLOBALS, method, pairs)
        audit_dir = _Path(env["OUT_ROOT"]) / "SHARED_INPUTS_AUDIT"
        audit_dir.mkdir(parents=True, exist_ok=True)
        audit.to_csv(audit_dir / "segmentation_analysis_manifest.csv", index=False)
        # Recompute crop, selected slices and ROI normalization using this mask.
        # Each image is already local; no original Drive volumes or labels are reloaded.
        env["prepare_shared_cnn_inputs"](pairs, expected_ids, context=method)
        _SEGMENTATION_ANALYSIS_JOBS.append({"method": method, "environment": env})
    ipython.events.register("post_run_cell", _capture_segmentation_pipeline_cell)
    ipython._segmentation_pipeline_capture_hook = _capture_segmentation_pipeline_cell
    print("[SEGMENTATION ANALYSIS] Original-mask workflows run first. Additional full analyses: "
          + (", ".join(job["method"] for job in _SEGMENTATION_ANALYSIS_JOBS) or "none"))
    print("[SEGMENTATION ANALYSIS] Run all remaining cells, including the final orchestration cell.")


def _release_segmentation_environment(env):
    """Keep importable helper/class definitions for saved ML models, release fitted data."""
    import types
    def small_setting(value):
        if value is None or isinstance(value, (str, bool, int, float, np.generic)):
            return True
        if isinstance(value, (list, tuple, set)):
            return len(value) <= 1000 and all(small_setting(v) for v in value)
        if isinstance(value, dict):
            return len(value) <= 1000 and all(small_setting(k) and small_setting(v) for k, v in value.items())
        return False
    for name, value in list(env.items()):
        if name == "__builtins__" or isinstance(value, (types.ModuleType, types.FunctionType, type)):
            continue
        if not small_setting(value):
            env.pop(name, None)


def run_additional_segmentation_analyses():
    """Run each selected mask cohort and expose incomplete analyses in the final report."""
    import gc
    from IPython import get_ipython
    from pathlib import Path as _Path
    global SEGMENTATION_RESULTS_REGISTRY, SEGMENTATION_COMPARISON_DF, SEGMENTATION_ANALYSIS_STATUS_DF
    global _SEGMENTATION_ANALYSES_COMPLETED

    def save_pipeline_overview(table):
        """Refresh the usual overview CSVs with original AND generated-mask results."""
        global overview_csv, overview_csv_by_test_set
        final_table = order_by_test_set(table.copy(deep=True), "AUC").reset_index(drop=True)
        # Keep a readable header even if no holdout results were produced this run.
        if "segmentation_method" not in final_table.columns:
            final_table["segmentation_method"] = pd.Series(dtype="object")
        main_dir = _Path(COMBINED_OVERVIEW_DIR)
        main_dir.mkdir(parents=True, exist_ok=True)
        overview_csv = str(main_dir / "combined_pipeline_overview.csv")
        overview_csv_by_test_set = {}
        # Match Section 5's existing one-test / multiple-test CSV layout.
        if len(TEST_SET_NAMES) > 1 and "test_set" in final_table.columns:
            for test_set_name in TEST_SET_NAMES:
                part = (final_table[final_table["test_set"] == test_set_name]
                        .drop(columns="test_set").reset_index(drop=True))
                part_path = main_dir / (
                    "combined_pipeline_overview__"
                    + re.sub(r"[^A-Za-z0-9]+", "_", test_set_name).strip("_") + ".csv"
                )
                # Also overwrite empty tables to remove stale rows from an earlier run.
                part.to_csv(part_path, index=False, encoding="utf-8-sig")
                overview_csv_by_test_set[test_set_name] = str(part_path)
        elif len(TEST_SET_NAMES) == 1:
            final_table = final_table.drop(columns="test_set", errors="ignore")
        # Replace the Section 5 original-only export; never append or duplicate rows.
        final_table.to_csv(overview_csv, index=False, encoding="utf-8-sig")
        print("[FINAL OVERVIEW] All available segmentation results saved:", overview_csv)
        if len(final_table):
            counts = final_table.groupby("segmentation_method", sort=False).size()
            print("[FINAL OVERVIEW] Rows by segmentation method:\n" + counts.to_string())
        for test_set_name, part_path in overview_csv_by_test_set.items():
            print(f"[FINAL OVERVIEW] Test set {test_set_name}:", part_path)

    required = SEGMENTATION_PIPELINE_CELL_KEYS
    if _SEGMENTATION_BASE_GLOBALS is None:
        raise RuntimeError("Run Section 0's segmentation-analysis setup cell first.")
    if _SEGMENTATION_ANALYSES_COMPLETED:
        status = globals().get("SEGMENTATION_ANALYSIS_STATUS_DF", pd.DataFrame())
        if len(status) and status["status"].isin(["failed", "incomplete"]).any():
            raise RuntimeError("Previous segmentation analyses are incomplete. Review segmentation_analysis_status.csv "
                               "and the error logs, then rerun from Section 0 after fixing the errors.")
        # Re-export without retraining if Section 5 was run again afterwards.
        save_pipeline_overview(SEGMENTATION_COMPARISON_DF)
        print("[SEGMENTATION ANALYSIS] Already completed. Rerun from Section 0 for a new full analysis.")
        return
    missing = [key for key in required if key not in _SEGMENTATION_PIPELINE_SOURCES]
    if _SEGMENTATION_ANALYSIS_JOBS and missing:
        raise RuntimeError(f"Downstream pipeline cells were not completed/captured: {missing}. Run all cells in order.")
    SEGMENTATION_RESULTS_REGISTRY = {}
    frames, status_rows = [], []

    def collect_method(method, environment, replay_error=""):
        registry = {name: df.copy(deep=True) for name, df in environment["RESULTS_REGISTRY"].items()}
        SEGMENTATION_RESULTS_REGISTRY[method] = registry
        table = environment.get("publication_df")
        result_rows = len(table) if table is not None else 0
        if result_rows:
            table = table.copy(deep=True)
            table["segmentation_method"] = method
            frames.append(table)
        error_rows = sum(len(df) for name, df in registry.items() if name.endswith("_errors"))
        if replay_error:
            state = "failed"
        elif error_rows:
            state = "incomplete" if result_rows else "failed"
        elif result_rows:
            state = "completed"
        elif holdout_evaluation_enabled():
            state = "failed" if (RUN_FROZEN_CNN_E2E or RUN_CNN_LR or RUN_CNN_FINETUNE_E2E) else "disabled"
        else:
            state = "holdout not evaluated"
        status_rows.append({"segmentation_method": method, "status": state,
                            "result_rows": result_rows, "error_rows": error_rows,
                            "replay_error": replay_error})

    collect_method("original", globals())
    saved_rng = rng_state_snapshot()
    try:
        for job in _SEGMENTATION_ANALYSIS_JOBS:
            method, env = job["method"], job["environment"]
            rng_state_restore(_SEGMENTATION_BASE_RNG)
            print("\n" + "=" * 110)
            print(f"FULL DOWNSTREAM ANALYSIS: {method} | {len(env['_SHARED_PATIENT_DATA']['pairs'])} patients")
            print("=" * 110)
            replay_error = ""
            try:
                for section, cell in required:
                    source = _SEGMENTATION_PIPELINE_SOURCES[(section, cell)]
                    print(f"[{method}] Section {section}, cell {cell}")
                    exec(compile(source, f"<{method}: Section {section}, cell {cell}>", "exec"), env, env)
            except Exception as exc:
                replay_error = repr(exc)
                print(f"[SEGMENTATION ANALYSIS][FAILED] {method}: {exc!r}")
            finally:
                collect_method(method, env, replay_error)
                _release_segmentation_environment(env)
                job.pop("environment", None)
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    finally:
        rng_state_restore(saved_rng)
        ipython = get_ipython()
        hook = getattr(ipython, "_segmentation_pipeline_capture_hook", None) if ipython else None
        if hook is not None:
            ipython.events.unregister("post_run_cell", hook)
            del ipython._segmentation_pipeline_capture_hook
    SEGMENTATION_COMPARISON_DF = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    SEGMENTATION_ANALYSIS_STATUS_DF = pd.DataFrame(status_rows)
    out_dir = _Path(OUT_ROOT) / "SEGMENTATION_ANALYSES"
    out_dir.mkdir(parents=True, exist_ok=True)
    status_path = out_dir / "segmentation_analysis_status.csv"
    SEGMENTATION_ANALYSIS_STATUS_DF.to_csv(status_path, index=False, encoding="utf-8-sig")
    print("\n[SEGMENTATION ANALYSIS] Final completion status:")
    print(SEGMENTATION_ANALYSIS_STATUS_DF.to_string(index=False))
    print("[SEGMENTATION ANALYSIS] Status table:", status_path)
    # Write even an empty comparison so an old successful table cannot remain current.
    if "test_set" not in SEGMENTATION_COMPARISON_DF and len(TEST_SET_NAMES) == 1:
        SEGMENTATION_COMPARISON_DF["test_set"] = TEST_SET_NAMES[0]
    output = out_dir / "combined_segmentation_overview.csv"
    SEGMENTATION_COMPARISON_DF.to_csv(output, index=False, encoding="utf-8-sig")
    print("[SEGMENTATION ANALYSIS] Combined results:", output)
    # The main results CSV must include every collected segmentation method too.
    # Save before the failure guard so successfully completed rows remain available.
    save_pipeline_overview(SEGMENTATION_COMPARISON_DF)
    _SEGMENTATION_ANALYSES_COMPLETED = True
    bad = SEGMENTATION_ANALYSIS_STATUS_DF[SEGMENTATION_ANALYSIS_STATUS_DF["status"].isin(["failed", "incomplete"])]
    if len(bad):
        raise RuntimeError("Segmentation analysis incomplete for " + ", ".join(bad["segmentation_method"])
                           + ". Completed results were saved; check segmentation_analysis_status.csv and "
                           "PART3_cnn_ml/segment3_errors.csv. An original-only table is not a completed mask comparison.")


# Initialize before automatic segmentation or any CNN training starts.
if RUN_FROZEN_CNN_E2E or RUN_CNN_LR or RUN_CNN_FINETUNE_E2E or RUN_AUTOSEG_NNINTERACTIVE or RUN_AUTOSEG_SAM2 or ANALYZE_NNINTERACTIVE_MASKS or ANALYZE_SAM2_MASKS:
    initialize_shared_patient_data()


# Freeze settings/helpers now, before resegmentation creates temporary 3D arrays.
_SEGMENTATION_SECTION0_BASE = {name: globals()[name] for name in ['CAS2_ROOT', 'LASSO_ROOT', 'segmentation_cache_path', 'find_saved_segmentation', 'ANALYZE_NNINTERACTIVE_MASKS', 'ANALYZE_SAM2_MASKS', 'AUC_BOOTSTRAP_REPEATS', 'AUC_CI_CONFIDENCE', 'AUG_BRIGHTNESS', 'AUG_CONTRAST', 'AUG_GAMMA_RANGE', 'AUG_NOISE_STD', 'AUG_ROTATE_DEG', 'AUG_SCALE_RANGE', 'AUG_TRANSLATE', 'AUTOSEG_DOWNLOAD_MISSING_WEIGHTS', 'AUTOSEG_OUT_DIR', 'AUTOSEG_PACKAGE_PINS', 'CNN_INPUT_SIZE_SHARED', 'COMBINED_OVERVIEW_DIR', 'CROP_SAFETY_MARGIN_PX', 'DEVELOPMENT_ROOT', 'DEVELOPMENT_ROOT_TEXT', 'DEV_FOLDER_PATIENT_IDS', 'F', 'FEATURE_SELECTOR_MIN_KEEP', 'FEATURE_SELECTOR_MIN_VOTES', 'FEATURE_SELECTOR_TOP_K', 'FINETUNE_OUT_DIR', 'FROZEN_E2E_AUGMENTATION_LEVEL', 'GLOBAL_PATIENT_SPLIT_KEY', 'GLOBAL_SEED', 'HF_AUTHENTICATED', 'HF_TOKEN', 'HOLDOUT_NOT_EVALUATED_TEXT', 'HoldoutNotEvaluated', 'IMAGENET_MEAN', 'IMAGENET_STD', 'INTENSITY_Z_SPAN_SD', 'LABELS_CONFIGS', 'LABELS_CSV', 'LOCAL_CACHE_DIR', 'MASK_FILENAME_KEYWORDS', 'MAX_DISPLAY_ROWS', 'NNINTERACTIVE_ANALYSIS_PROMPT_TYPE', 'NNINTERACTIVE_WEIGHTS_PATH', 'N_SLICES', 'OPTIMIZATION', 'OPTIMIZATION_ARMS', 'OPTIMIZATION_AUG_LEVELS', 'OPTIMIZATION_BACKBONES', 'OPTIMIZATION_EVALUATE_HOLDOUT', 'OPTIMIZATION_FROZEN_OOF_STORE', 'OPTIMIZATION_INCLUDE_REFERENCE_ARM', 'OPTIMIZATION_INPUT_SIZES', 'OPTIMIZATION_MIN_DELTA', 'OPTIMIZATION_ML_ARMS', 'OPTIMIZATION_ML_OOF_STORE', 'OPTIMIZATION_N_FOLDS', 'OPTIMIZATION_N_REPEATS', 'OPTIMIZATION_N_SLICES', 'OPTIMIZATION_OOF_STORE', 'OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW', 'OPTIMIZATION_REFERENCE_INPUT_SIZE', 'OPTIMIZATION_REFERENCE_N_SLICES', 'OUTCOME_NAME', 'OUT_ROOT', 'PACKAGE_PINS', 'PART1_OUT_DIR', 'PART2_OUT_DIR', 'PART3_OUT_DIR', 'PATIENT_SPLIT_RECORDS', 'QC_N_PATIENTS', 'QC_PATIENTS_PER_FIG', 'RADIMAGENET_CAFFE_MEAN_BGR', 'RADIMAGENET_DENSENET121_WEIGHTS_PATH', 'RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH', 'RADIMAGENET_NORM_BY_BACKBONE', 'RADIMAGENET_RESNET10T_WEIGHTS_PATH', 'RADIMAGENET_RESNET18_WEIGHTS_PATH', 'RADIMAGENET_RESNET50_USE_OFFICIAL_NORM', 'RADIMAGENET_RESNET50_WEIGHTS_PATH', 'RADIMAGENET_WEIGHTS_PATH', 'RESULTS_REGISTRY', 'ROI_ONLY_BACKGROUND_VALUE', 'RUN_AUTOSEG_NNINTERACTIVE', 'RUN_AUTOSEG_SAM2', 'RUN_CLASSIFIERS', 'RUN_CNN_FINETUNE_E2E', 'RUN_CNN_LR', 'RUN_FEATURE_SELECTOR_COMBINATIONS', 'RUN_FROZEN_CNN_E2E', 'SAM2_WEIGHTS_PATH', 'SAVE_SLICE_LEVEL_FEATURE_CSVS', 'SEG2_FEATURE_DIR', 'SEGMENTATION_ANALYSIS_METHOD', 'SEGMENTATION_PIPELINE_CELL_KEYS', 'SEGMENTATION_RESULTS_REGISTRY', 'SHARED_BACKBONES', 'SHARED_OUTCOMES', 'SHOW_ANALYSED_SLICES', 'SLICE_PLANES', 'SPLIT_FOLDERS', 'SPLIT_FOLDER_FILE_INVENTORY', 'SPLIT_FOLDER_MEMBERSHIP', 'SPLIT_FOLDER_OF_PATIENT', 'SPLIT_SET_DEVELOPMENT', 'SPLIT_SET_TEST', 'TEST_FOLDER_PATIENT_IDS', 'TEST_ROOT', 'TEST_ROOT_TEXT', 'TEST_SET_NAMES', 'TEST_SET_OF_PATIENT', 'TEST_SET_PATIENT_IDS', 'TEST_SET_ROOTS', 'TEST_SET_ROOT_BY_NAME', 'TOP_RISK_PERCENT', 'TRAIN_AUGMENTATION_LEVEL', 'TUMOR_CONTEXT_MODE', 'USE_FEATURE_SELECTOR_VOTING', 'WEIGHTS_BASE', '_AUTOSEG_PIP_PACKAGES', '_LISTED_BACKBONES', '_NOTEBOOK_RUN_STARTED_AT', '_NOTEBOOK_RUN_T0_MONOTONIC', '_RADIMAGENET_NAME_ALIASES', '_SEG1_SUPPORTED_BACKBONES', '_SHARED_INPUT_FILES', '_SHARED_PATIENT_DATA', '_SHARED_TENSOR_RAM_BYTES', '_SHARED_TENSOR_RAM_CACHE', '_SHARED_TENSOR_RAM_LIMIT', '_SPLIT_ID_TAG_SUFFIXES', '_SPLIT_VOLUME_EXTENSIONS', '_TEST_SET_NAME_BY_ROOT', '_VALID_AUG_LEVELS', '_VALID_INPUT_SCALINGS', '_VALID_TUMOR_CONTEXT_MODES', '__builtins__', '_a', '_batch_auc_positives_first', '_bb', '_bg_txt', '_canonical_unique_patient_ids', '_capture_segmentation_pipeline_cell', '_copy_generated_masks', '_d', '_lvl', '_name', '_notebook_datetime', '_notebook_time', '_prepare_auc_inputs', '_ref_for', '_release_segmentation_environment', '_scaling', '_segmentation_namespace', '_shared_input_key', '_test_set_token', '_wf', '_wf_lvl', '_write_shared_input', 'apply_input_scaling', 'attach_crop_diagnostics', 'b', 'bootstrap_auc_ci', 'build_central_plane_inputs', 'build_ml_optimization_arms', 'build_optimization_arms', 'build_patient_min_square_crop_spec', 'cache_pairs_locally', 'calculate_auc_intervals', 'canonical_patient_id', 'check_optimization_reference_arms', 'cid', 'collect_generic_image_mask_pairs', 'collect_pairs_from_split_folders', 'configure_optional_hf_token', 'crop_resize_slice', 'delong_auc_ci', 'display', 'ensure_drive_mounted', 'extract_plane_slice', 'find_weight_file', 'format_auc_with_ci', 'get_or_create_patient_split', 'get_shared_patient_data', 'get_shared_patient_inputs', 'holdout_evaluation_enabled', 'holdout_test_sets', 'i', 'infer_patient_split_key_from_feature_file', 'initialize_shared_patient_data', 'is_mask_file', 'jackknife_auc_ci', 'load_label_table', 'n', 'name', 'normalize_volume_for_cnn', 'np', 'optimization_arm_is_reference', 'optimization_paired_bootstrap_auc_delta', 'optimization_reference_triple', 'order_by_test_set', 'os', 'patient_id_from_volume_filename', 'patient_id_sort_key', 'patient_split_record_key', 'pd', 'prepare_segmentation_analysis_jobs', 'prepare_shared_cnn_inputs', 'preview_analysed_slices', 'print_patient_split', 'r', 'radimagenet_norm_for', 'random', 're', 'read_sitk_pair', 'rng_state_restore', 'rng_state_snapshot', 'roc_auc_score', 'root', 'rows_for_display', 'run_additional_segmentation_analyses', 's', 'scan_split_folders', 'seed_everything', 'selected_segmentation_analysis_methods', 'selected_tumor_slice_indices', 'sitk', 'split_folder_label', 'split_folder_roots', 'split_set_of_patient', 'sys', 'test_set_file_tag', 'test_set_of_patient', 'threshold_for_top_risk', 'top_risk_mode_enabled', 'torch', 'tumor_context_tag', 'warnings'] if name in globals()}


## Section 0, cell 4 - Cached automatic segmentation (optional)

The next cell runs when `RUN_AUTOSEG_NNINTERACTIVE` or `RUN_AUTOSEG_SAM2` is enabled.
For each patient, it checks the permanent Drive mask before loading a model. Existing
masks are loaded directly; missing masks are generated and saved immediately as NRRD.
nnInteractive is permanently restricted to **lasso only**.

| Method | Prompt from the original mask | Permanent cache |
|---|---|---|
| nnInteractive (3D) | Filled lasso on the largest axial slice of each tumour, widened by one pixel. | `LASSO_ROOT/<patient ID>.nrrd` |
| SAM2 (2D) | One box on every tumour slice. | `CAS2_ROOT/<patient ID>.nrrd` |

`CAS2_ROOT` is `/content/drive/MyDrive/DATA/CAS2`; `LASSO_ROOT` is
`/content/drive/MyDrive/DATA/nninteraction-lasso`. Completed masks from the old
`AUTOSEG_OUT_DIR/masks/<method>/` or `masks__pilot/<method>/` location are copied
once into these permanent caches when no new-cache file exists.

Cached and newly generated masks both receive agreement measurements and QC overlays.
The per-patient table records `cache_source` and `segmentation_file`; the summary counts
reused and generated masks. Invalid cached masks are reported without overwriting them.
To deliberately regenerate a patient, remove or rename its cached mask and any matching
legacy mask first.

Original-mask classification remains included. The independent `ANALYZE_*` switches
add the complete downstream pipeline on the corresponding saved mask set in Section 6.
Use `AUTOSEG_PILOT_N_PATIENTS=None` for full-cohort generation and analysis. A pilot uses
the same permanent patient masks but writes separate agreement tables with `__pilot`.

Because SAM2 receives a box on every tumour slice, part of its DSC is built in.
`dice_box_only_control` compares an ellipse in those same boxes without using the image.
Tables, run records and overlay figures remain under `AUTOSEG_OUT_DIR`.


In [ ]:
# Section 0, cell 5: Automatic segmentation agreement check (nnInteractive and SAM 2)
#
# Optional. Runs only when RUN_AUTOSEG_NNINTERACTIVE or RUN_AUTOSEG_SAM2 is True in
# MAJOR/ESSENTIAL SETTINGS (Section 0, cell 3). It re-segments every tumour automatically,
# STARTING FROM THE EXISTING MASK, and reports the Dice Similarity Coefficient (DSC) between
# each automatic mask and the existing one.
#
# Original-mask analyses remain included. The ANALYZE switches can also run the complete
# pipeline on these saved masks in Section 6. No outcome label is used for segmentation.
# Existing Drive masks are loaded; only missing masks require model inference.
#
# What each method takes from the existing mask:
#   nnInteractive  ONE prompt on ONE axial slice per tumour (the slice where that tumour is
#                  largest). The model draws the whole 3D mask itself.
#   SAM 2          one box on EVERY axial slice of every tumour. The existing mask therefore
#                  fixes the slice range and the frame on each slice, and part of the DSC is
#                  built in. The "box only" control column shows how much: it is the DSC of
#                  an ellipse drawn inside the same boxes without looking at the image.
if globals().get("RUN_AUTOSEG_NNINTERACTIVE", False) or globals().get("RUN_AUTOSEG_SAM2", False):

    # ── Settings of this cell ───────────────────────────────────────────────
    # A "prompt" is a location hint given to the segmentation model, such as a
    # marked point or a rectangle around the tumour. Different hint formats are
    # called "prompt types". Here, hints are generated from the existing mask.
    #
    # Set a small pilot count to process development patients only, or None for the
    # full development/test cohort. Pilot masks share the same permanent patient cache.
    AUTOSEG_PILOT_N_PATIENTS = None

    # Fixed: only the filled lasso outline on each tumour's largest axial slice,
    # widened by one pixel. Box, scribble and point generation are removed.
    NNINTERACTIVE_PROMPTS = ("lasso",)

    # SAM 2: pixels added on every side of each per-slice box. 0 gives the tightest box and
    # the highest built-in agreement; a margin makes the DSC more informative.
    SAM2_BOX_MARGIN_PX = 2

    # A mask may hold several separate tumours; each one gets its own prompt(s). Separate
    # pieces smaller than this many voxels are treated as stray voxels and get no prompt.
    AUTOSEG_MIN_LESION_VOXELS = 10
    AUTOSEG_MAX_LESIONS_PER_PATIENT = 25   # largest first

    # Keep only the largest connected piece of what a model returns for one tumour. Stray
    # voxels would otherwise stretch the tumour crop the notebook builds from a mask.
    AUTOSEG_KEEP_LARGEST_PIECE = True

    AUTOSEG_SAVE_MASKS = True     # fixed: every successful new mask is saved permanently on Drive
    # A full downstream comparison requires saved full-cohort masks.
    _analysis_generating_now = ((RUN_AUTOSEG_NNINTERACTIVE and ANALYZE_NNINTERACTIVE_MASKS)
                               or (RUN_AUTOSEG_SAM2 and ANALYZE_SAM2_MASKS))
    if _analysis_generating_now and AUTOSEG_PILOT_N_PATIENTS is not None:
        raise ValueError("Full generated-mask analysis requires AUTOSEG_PILOT_N_PATIENTS=None.")
    AUTOSEG_QC_N_PATIENTS = 12    # patients shown in the overlay figures
    AUTOSEG_LOW_DICE = 0.70       # DSC below this is counted as low in the summary

    # ── Imports ─────────────────────────────────────────────────────────────
    import contextlib, gc, hashlib, io, time, traceback, uuid
    import importlib.metadata as _autoseg_metadata
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    import SimpleITK as sitk
    from scipy import ndimage as _autoseg_ndi
    from tqdm.auto import tqdm

    # ── Checks on the settings ──────────────────────────────────────────────
    if NNINTERACTIVE_PROMPTS != ("lasso",):
        raise ValueError("Only nnInteractive lasso generation is supported.")
    if int(SAM2_BOX_MARGIN_PX) < 0 or int(AUTOSEG_MIN_LESION_VOXELS) < 1 or int(AUTOSEG_MAX_LESIONS_PER_PATIENT) < 1:
        raise ValueError("SAM2_BOX_MARGIN_PX must be >= 0; the two lesion limits must be >= 1.")

    _AUTOSEG_PILOT = AUTOSEG_PILOT_N_PATIENTS is not None and int(AUTOSEG_PILOT_N_PATIENTS) > 0
    _AUTOSEG_TAG = "__pilot" if _AUTOSEG_PILOT else ""
    _AUTOSEG_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _AUTOSEG_SAM2_METHOD = "sam2_box_per_slice"
    _AUTOSEG_NNI_METHODS = [f"nninteractive_{p}" for p in NNINTERACTIVE_PROMPTS] if RUN_AUTOSEG_NNINTERACTIVE else []
    _AUTOSEG_DEPENDENCE = {m: "one prompt on one slice per tumour" for m in _AUTOSEG_NNI_METHODS}
    _AUTOSEG_DEPENDENCE[_AUTOSEG_SAM2_METHOD] = "one box on every tumour slice"
    os.makedirs(AUTOSEG_OUT_DIR, exist_ok=True)

    # nnInteractive changes two global PyTorch settings (thread count and the cuDNN benchmark
    # mode). They and the random-number state are put back at the end of this cell, so that
    # Sections 1 to 3 run exactly as they do without this cell.
    _autoseg_saved_state = {
        "rng": rng_state_snapshot(),
        "threads": torch.get_num_threads(),
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
    }

    print("=" * 78)
    print("[AUTO-SEG] Automatic segmentation agreement check")
    print(f"[AUTO-SEG] nnInteractive={RUN_AUTOSEG_NNINTERACTIVE} (prompts: {NNINTERACTIVE_PROMPTS}) | "
          f"SAM 2={RUN_AUTOSEG_SAM2} (box margin {int(SAM2_BOX_MARGIN_PX)} px) | device: {_AUTOSEG_DEVICE}")
    if _AUTOSEG_DEVICE.type != "cuda":
        print("[AUTO-SEG] No GPU found; any missing masks will be generated on the CPU.")
    print("=" * 78)

    # ── Model weights: use the folders set in Section 0, cell 3; download once if missing ──
    def _autoseg_missing_files(folder, required):
        folder = Path(str(folder))
        return [name for name in required if not (folder / name).is_file()]

    def _autoseg_ensure_weights(label, folder, required, repo_id, patterns, repo_subfolder, manual_url):
        """Check a weights folder; when files are missing, download them into it once."""
        folder = Path(str(folder))
        missing = _autoseg_missing_files(folder, required)
        if not missing:
            print(f"[AUTO-SEG] {label} weights found: {folder}")
            return
        if not AUTOSEG_DOWNLOAD_MISSING_WEIGHTS:
            raise FileNotFoundError(
                f"{label} weights are incomplete in {folder} (missing: {missing}). Download them from "
                f"{manual_url} into that folder, or set AUTOSEG_DOWNLOAD_MISSING_WEIGHTS = True in Section 0, cell 3."
            )
        from huggingface_hub import snapshot_download
        print(f"[AUTO-SEG] {label} weights not found in {folder}; downloading them once from {manual_url}")
        try:
            if repo_subfolder is None:
                folder.mkdir(parents=True, exist_ok=True)
                snapshot_download(repo_id=repo_id, allow_patterns=list(patterns), local_dir=str(folder))
            else:
                # The repository keeps the model inside a sub-folder of this name.
                folder.parent.mkdir(parents=True, exist_ok=True)
                snapshot_download(repo_id=repo_id, local_dir=str(folder.parent),
                                  allow_patterns=[f"{repo_subfolder}/*", f"{repo_subfolder}/**"])
                downloaded = folder.parent / repo_subfolder
                if downloaded != folder and downloaded.is_dir():
                    if folder.is_dir() and not any(folder.iterdir()):
                        folder.rmdir()
                    downloaded.rename(folder)
        except Exception as exc:
            raise FileNotFoundError(
                f"{label} weights could not be downloaded ({exc!r}). Download them by hand from {manual_url} "
                f"into {folder}"
            ) from exc
        missing = _autoseg_missing_files(folder, required)
        if missing:
            raise FileNotFoundError(
                f"{label} weights are still incomplete in {folder} after the download (missing: {missing}). "
                f"Download them by hand from {manual_url}"
            )
        print(f"[AUTO-SEG] {label} weights saved in: {folder}")

    def _autoseg_sha256(path, block=1 << 22):
        digest = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(block), b""):
                digest.update(chunk)
        return digest.hexdigest()

    # ── Small helpers ───────────────────────────────────────────────────────
    def _autoseg_dice(a, b):
        """Dice Similarity Coefficient of two binary masks; NaN when both are empty."""
        a = np.asarray(a, dtype=bool)
        b = np.asarray(b, dtype=bool)
        total = int(a.sum()) + int(b.sum())
        return float("nan") if total == 0 else 2.0 * int((a & b).sum()) / total

    def _autoseg_pack(mask):
        """Keep a mask as its bounding box only, to save memory."""
        mask = np.asarray(mask, dtype=bool)
        if not mask.any():
            return (mask.shape, None, None)
        box = _autoseg_ndi.find_objects(mask.astype(np.uint8))[0]
        return (mask.shape, box, mask[box].copy())

    def _autoseg_unpack(packed):
        shape, box, crop = packed
        mask = np.zeros(shape, dtype=bool)
        if box is not None:
            mask[box] = crop
        return mask

    def _autoseg_largest_piece(mask):
        """The largest connected piece of one tumour's automatic mask (see AUTOSEG_KEEP_LARGEST_PIECE)."""
        if not AUTOSEG_KEEP_LARGEST_PIECE or not mask.any():
            return mask
        box = _autoseg_ndi.find_objects(mask.astype(np.uint8))[0]
        labelled, n_pieces = _autoseg_ndi.label(mask[box], structure=np.ones((3, 3, 3), dtype=bool))
        if n_pieces <= 1:
            return mask
        sizes = np.bincount(labelled.ravel())[1:]
        cleaned = np.zeros_like(mask)
        cleaned[box] = labelled == (1 + int(np.argmax(sizes)))
        return cleaned

    def _autoseg_box_iou(a, b):
        """Overlap (intersection over union) of two boxes given as (y0, y1, x0, x1)."""
        inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[2], b[2]))
        union = (a[1] - a[0]) * (a[3] - a[2]) + (b[1] - b[0]) * (b[3] - b[2]) - inter
        return float(inter / union) if union > 0 else float("nan")

    def _autoseg_lesions(ref):
        """Separate tumours of one existing mask, largest first.

        Returns (lesions, number of separate pieces in the mask). Each lesion holds its bounding
        box in the volume ("box"), its mask inside that box ("mask"), the axial slice where it is
        largest ("z_prompt") and its size in voxels.
        """
        labelled, n_pieces = _autoseg_ndi.label(ref, structure=np.ones((3, 3, 3), dtype=bool))
        if n_pieces == 0:
            return [], 0
        sizes = np.bincount(labelled.ravel())[1:]
        order = [int(i) for i in np.argsort(-sizes, kind="stable")]
        keep = [i for i in order if sizes[i] >= int(AUTOSEG_MIN_LESION_VOXELS)] or order[:1]
        keep = keep[:int(AUTOSEG_MAX_LESIONS_PER_PATIENT)]
        boxes = _autoseg_ndi.find_objects(labelled)
        lesions = []
        for i in keep:
            box = boxes[i]
            piece = labelled[box] == (i + 1)
            areas = piece.reshape(piece.shape[0], -1).sum(axis=1)
            lesions.append({"box": box, "mask": piece, "voxels": int(sizes[i]),
                            "z_prompt": int(box[0].start + int(np.argmax(areas)))})
        return lesions, int(n_pieces)

    def _autoseg_central_plane_agreement(ref, auto, spacing_zyx):
        """Does the automatic mask lead the notebook to the same slices and crops as the existing one?

        Uses the notebook's own slice-selection and crop functions on the planes it analyses
        (N_SLICES). Returns the mean DSC on the existing mask's central slices, how many central
        slices are the same, the largest slice shift in mm and the smallest crop-box overlap.
        """
        out = {"dice_analysed_slices": float("nan"), "same_central_slices": "",
               "max_central_slice_shift_mm": float("nan"), "min_crop_overlap": float("nan")}
        rows_ref = selected_tumor_slice_indices(ref)
        out["dice_analysed_slices"] = float(np.mean(
            [_autoseg_dice(extract_plane_slice(ref, r), extract_plane_slice(auto, r)) for r in rows_ref]
        ))
        if not auto.any():
            out["same_central_slices"] = f"0/{len(rows_ref)}"
            return out
        rows_auto = selected_tumor_slice_indices(auto)
        sp = np.asarray(spacing_zyx, dtype=float).ravel()
        n_same, shifts, overlaps = 0, [], []
        for row_ref, row_auto in zip(rows_ref, rows_auto):
            axis = int(row_ref["slice_axis"])
            plane_spacing = sp[[axis] + [a for a in range(3) if a != axis]]
            n_same += int(row_ref["slice_index"] == row_auto["slice_index"])
            shifts.append(abs(int(row_ref["slice_index"]) - int(row_auto["slice_index"])) * float(sp[axis]))
            crop_boxes = []
            for volume, row in ((ref, row_ref), (auto, row_auto)):
                spec = build_patient_min_square_crop_spec(
                    extract_plane_slice(volume, row)[None], [{"z_index": 0}], plane_spacing
                )
                half_h, half_w = spec["crop_height_px"] / 2.0, spec["crop_width_px"] / 2.0
                crop_boxes.append((spec["crop_center_y_px"] - half_h, spec["crop_center_y_px"] + half_h,
                                   spec["crop_center_x_px"] - half_w, spec["crop_center_x_px"] + half_w))
            overlaps.append(_autoseg_box_iou(*crop_boxes))
        out["same_central_slices"] = f"{n_same}/{len(rows_ref)}"
        out["max_central_slice_shift_mm"] = float(max(shifts))
        out["min_crop_overlap"] = float(np.nanmin(overlaps))
        return out

    def _autoseg_outside_percent(ref, auto, spacing_zyx, margin_mm=5.0):
        """Percentage of the automatic mask lying more than margin_mm outside the existing mask's box."""
        n_auto = int(auto.sum())
        if n_auto == 0:
            return float("nan")
        box = _autoseg_ndi.find_objects(ref.astype(np.uint8))[0]
        sp = np.asarray(spacing_zyx, dtype=float).ravel()
        grown = tuple(
            slice(max(s.start - int(np.ceil(margin_mm / sp[a])), 0), min(s.stop + int(np.ceil(margin_mm / sp[a])), ref.shape[a]))
            for a, s in enumerate(box)
        )
        return 100.0 * (n_auto - int(auto[grown].sum())) / n_auto

    def _autoseg_image_grid(image_path):
        """Read geometry only and construct the same LPS grid used for preprocessing."""
        reader = sitk.ImageFileReader()
        reader.SetFileName(str(image_path))
        reader.ReadImageInformation()
        grid = sitk.Image([int(v) for v in reader.GetSize()], sitk.sitkUInt8)
        grid.SetSpacing(reader.GetSpacing())
        grid.SetOrigin(reader.GetOrigin())
        grid.SetDirection(reader.GetDirection())
        return sitk.DICOMOrient(grid, "LPS")


    def _autoseg_load_mask(path, image_path, expected_shape):
        """Load and validate a saved mask without invoking either segmentation model."""
        mask_image = sitk.DICOMOrient(sitk.ReadImage(str(path)), "LPS")
        grid = _autoseg_image_grid(image_path)
        if (mask_image.GetSize() != grid.GetSize() or
            any(not np.allclose(getattr(mask_image, attr)(), getattr(grid, attr)(), rtol=0, atol=1e-6)
                for attr in ("GetSpacing", "GetOrigin", "GetDirection"))):
            raise ValueError(f"Cached mask geometry differs from the image grid: {path}")
        array = sitk.GetArrayFromImage(mask_image)
        if array.shape != tuple(expected_shape) or not np.isfinite(array).all():
            raise ValueError(f"Cached mask has an invalid shape or nonfinite values: {path}")
        mask = array > 0
        if not mask.any():
            raise ValueError(f"Cached mask is empty: {path}")
        return mask


    def _autoseg_save_mask(auto, image_path, out_path):
        """Atomically save a new nonempty mask on Drive; never replace an existing cache."""
        out_path = Path(out_path)
        if out_path.exists():
            raise FileExistsError(f"Refusing to overwrite an existing segmentation cache: {out_path}")
        if not np.asarray(auto).any():
            raise ValueError("An empty automatic mask is not saved as a permanent segmentation cache.")
        grid = _autoseg_image_grid(image_path)
        mask_image = sitk.GetImageFromArray(np.asarray(auto, dtype=np.uint8))
        if mask_image.GetSize() != grid.GetSize():
            raise RuntimeError(f"Mask grid {mask_image.GetSize()} differs from image grid {grid.GetSize()}.")
        mask_image.CopyInformation(grid)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = out_path.with_name(f".{out_path.stem}.{uuid.uuid4().hex}.tmp.nrrd")
        try:
            sitk.WriteImage(mask_image, str(temporary), True)
            if out_path.exists():
                raise FileExistsError(f"A segmentation cache appeared while saving: {out_path}")
            temporary.replace(out_path)
        finally:
            temporary.unlink(missing_ok=True)

    # ── Results kept while the two models run one after the other ─────────────
    _autoseg_rows = {}   # (patient ID, method) -> one row of the per-patient table
    _autoseg_kept = {}   # patient ID -> {nnInteractive method: packed mask}, for the comparison with SAM 2
    _autoseg_qc = {}     # patient ID -> image and mask crops for the overlay figures

    def _autoseg_qc_start(cid, img, ref):
        """Remember the existing mask's central axial slice of a patient for the overlay figure."""
        if cid in _autoseg_qc or len(_autoseg_qc) >= int(AUTOSEG_QC_N_PATIENTS):
            return
        z = int(selected_tumor_slice_indices(ref, n_slices=1)[0]["slice_index"])
        ys, xs = np.nonzero(ref[z])
        pad = max(12, int(0.6 * max(int(ys.max() - ys.min()) + 1, int(xs.max() - xs.min()) + 1)))
        window = (slice(max(int(ys.min()) - pad, 0), min(int(ys.max()) + pad + 1, ref.shape[1])),
                  slice(max(int(xs.min()) - pad, 0), min(int(xs.max()) + pad + 1, ref.shape[2])))
        crop = np.nan_to_num(np.asarray(img[z][window], dtype=np.float32))
        low, high = np.percentile(crop, [1.0, 99.0])
        crop = np.clip((crop - low) / max(float(high - low), 1e-6), 0.0, 1.0)
        _autoseg_qc[cid] = {"z": z, "window": window, "image": crop, "ref": ref[z][window].copy(), "auto": {}, "dice": {}}

    def _autoseg_record(case, method, ref, auto, spacing_zyx, n_pieces, lesions, seconds,
                        prompt_slices=None, n_prompts=None, box_only=None, cache_source="generated"):
        """Compute every agreement figure of one patient and one method, and store the row."""
        cid = case["canonical_id"]
        cache_path = segmentation_cache_path(method, cid)
        # Persist expensive inference before agreement metrics or plotting can fail.
        if cache_source == "generated" and auto.any():
            _autoseg_save_mask(auto, case["image"], cache_path)
        voxel_ml = float(np.prod(np.asarray(spacing_zyx, dtype=float))) / 1000.0
        ref_ml, auto_ml = float(ref.sum()) * voxel_ml, float(auto.sum()) * voxel_ml
        row = {
            "patient_id": cid,
            "split_set": case["split_set"],
            "test_set": TEST_SET_OF_PATIENT.get(cid, "") if case["split_set"] == SPLIT_SET_TEST else "",
            "method": method,
            "cache_source": cache_source,
            "segmentation_file": str(cache_path),
            "taken_from_existing_mask": _AUTOSEG_DEPENDENCE[method],
            "status": "ok" if auto.any() else "empty automatic mask",
            "pieces_in_existing_mask": int(n_pieces),
            "tumours_prompted": len(lesions),
            "prompts_given": int(n_prompts if n_prompts is not None else len(lesions)),
            "existing_volume_ml": ref_ml,
            "automatic_volume_ml": auto_ml,
            "volume_ratio": (auto_ml / ref_ml) if ref_ml > 0 else float("nan"),
            "dice": _autoseg_dice(ref, auto),
            "dice_without_prompt_slices": float("nan"),
            "dice_box_only_control": float("nan") if box_only is None else _autoseg_dice(ref, box_only),
            "dice_vs_sam2": float("nan"),
        }
        if prompt_slices:
            other = np.ones(ref.shape[0], dtype=bool)
            other[sorted(set(int(z) for z in prompt_slices))] = False
            row["dice_without_prompt_slices"] = _autoseg_dice(ref[other], auto[other])
        row.update(_autoseg_central_plane_agreement(ref, auto, spacing_zyx))
        row["automatic_outside_existing_region_pct"] = _autoseg_outside_percent(ref, auto, spacing_zyx)
        row["seconds"] = float(seconds)
        _autoseg_rows[(cid, method)] = row
        if cid in _autoseg_qc:
            q = _autoseg_qc[cid]
            q["auto"][method] = auto[q["z"]][q["window"]].copy()
            q["dice"][method] = row["dice"]
        return row

    def _autoseg_record_failure(case, methods, error):
        for method in methods:
            cid = case["canonical_id"]
            _autoseg_rows[(cid, method)] = {
                "patient_id": cid, "split_set": case["split_set"],
                "test_set": TEST_SET_OF_PATIENT.get(cid, "") if case["split_set"] == SPLIT_SET_TEST else "",
                "method": method, "taken_from_existing_mask": _AUTOSEG_DEPENDENCE[method],
                "status": f"failed: {error!r}"[:300],
            }

    def _autoseg_read_case(case):
        """Image, existing binary mask and spacing, read exactly as Sections 1 to 3 read them."""
        img, mask, spacing_zyx = read_sitk_pair(case["image"], case["mask"])
        ref = np.asarray(mask) > 0
        if not ref.any():
            raise ValueError("The existing mask is empty.")
        return img, ref, spacing_zyx

    def _autoseg_release_gpu():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _autoseg_quiet():
        """nnInteractive prints several lines for every prompt; they are kept off the screen."""
        return contextlib.redirect_stdout(io.StringIO())

    def _autoseg_median(values):
        values = pd.to_numeric(values, errors="coerce").dropna()
        return float(values.median()) if len(values) else float("nan")

    # ── nnInteractive: one prompt on one slice per tumour, full 3D mask from the model ──
    def _autoseg_nninteractive_prompt(session, prompt, lesion, shape):
        """Give nnInteractive the fixed lasso prompt for one tumour."""
        if prompt != "lasso":
            raise ValueError("Only nnInteractive lasso prompts are supported.")
        z = int(lesion["z_prompt"])
        y_off, x_off = int(lesion["box"][1].start), int(lesion["box"][2].start)
        piece = lesion["mask"][z - int(lesion["box"][0].start)]
        # Same filled outline and one-pixel widening as the previous lasso implementation.
        y0, y1 = max(y_off - 1, 0), min(y_off + piece.shape[0] + 1, shape[1])
        x0, x1 = max(x_off - 1, 0), min(x_off + piece.shape[1] + 1, shape[2])
        region = np.zeros((y1 - y0, x1 - x0), dtype=bool)
        region[y_off - y0:y_off - y0 + piece.shape[0], x_off - x0:x_off - x0 + piece.shape[1]] = piece
        region = _autoseg_ndi.binary_dilation(_autoseg_ndi.binary_fill_holes(region))
        session.add_lasso_interaction(region[None].astype(np.uint8), include_interaction=True,
                                      interaction_bbox=[[z, z + 1], [y0, y1], [x0, x1]])

    def _autoseg_run_nninteractive(cases):
        session, initialization_error = None, None
        method = "nninteractive_lasso"
        try:
            for case in tqdm(cases, desc="nnInteractive lasso (Drive cache first)"):
                cid = case["canonical_id"]
                try:
                    t0 = time.perf_counter()
                    img, ref, spacing_zyx = _autoseg_read_case(case)
                    _autoseg_qc_start(cid, img, ref)
                    lesions, n_pieces = _autoseg_lesions(ref)
                    path = find_saved_segmentation(method, cid)
                    if path.is_file():
                        auto = _autoseg_load_mask(path, case["image"], ref.shape)
                        cache_source = "drive_cache"
                    else:
                        if initialization_error is not None:
                            raise RuntimeError(f"nnInteractive initialization failed: {initialization_error}")
                        if session is None:
                            try:
                                _autoseg_ensure_weights(
                                    "nnInteractive", NNINTERACTIVE_WEIGHTS_PATH,
                                    ("plans.json", "dataset.json", "inference_session_class.json", "fold_0/checkpoint_final.pth"),
                                    "MIC-DKFZ/nnInteractive", None, "nnInteractive_v1.0",
                                    "https://huggingface.co/MIC-DKFZ/nnInteractive/tree/main/nnInteractive_v1.0",
                                )
                                with _autoseg_quiet():
                                    from nnInteractive.inference.inference_session import nnInteractiveInferenceSession
                                session = nnInteractiveInferenceSession(
                                    device=_AUTOSEG_DEVICE, use_torch_compile=False, verbose=False,
                                    torch_n_threads=os.cpu_count() or 1, do_autozoom=True,
                                )
                                session.initialize_from_trained_model_folder(str(NNINTERACTIVE_WEIGHTS_PATH))
                            except Exception as exc:
                                initialization_error = exc
                                session = None
                                raise
                        # Raw image: nnInteractive performs its own normalization.
                        target = torch.zeros(img.shape, dtype=torch.uint8)
                        with _autoseg_quiet():
                            session.set_image(np.asarray(img, dtype=np.float32)[None])
                            session.set_target_buffer(target)
                        auto = np.zeros(ref.shape, dtype=bool)
                        for lesion in lesions:
                            with _autoseg_quiet():
                                session.reset_interactions()
                                _autoseg_nninteractive_prompt(session, "lasso", lesion, ref.shape)
                            auto |= _autoseg_largest_piece(target.cpu().numpy() > 0)
                        cache_source = "generated"
                    _autoseg_record(case, method, ref, auto, spacing_zyx, n_pieces, lesions,
                                    time.perf_counter() - t0,
                                    prompt_slices=[lesion["z_prompt"] for lesion in lesions],
                                    cache_source=cache_source)
                    _autoseg_kept.setdefault(cid, {})[method] = _autoseg_pack(auto)
                    print(f"[AUTO-SEG][nnInteractive lasso] patient {cid}: {cache_source} -> {path}")
                except Exception as exc:
                    print(f"[AUTO-SEG][nnInteractive lasso] patient {cid}: failed ({exc!r})")
                    _autoseg_record_failure(case, [method], exc)
                    _autoseg_release_gpu()
        finally:
            del session
            _autoseg_release_gpu()

    # ── SAM 2: one box on every axial slice of every tumour ────────────────────
    def _autoseg_run_sam2(cases):
        model, processor, initialization_error = None, None, None
        margin = int(SAM2_BOX_MARGIN_PX)
        try:
            for case in tqdm(cases, desc="SAM 2"):
                cid = case["canonical_id"]
                try:
                    t0 = time.perf_counter()
                    img, ref, spacing_zyx = _autoseg_read_case(case)
                    _autoseg_qc_start(cid, img, ref)
                    lesions, n_pieces = _autoseg_lesions(ref)
                    height, width = ref.shape[1], ref.shape[2]
                    # Boxes per axial slice: (y0, y1, x0, x1) in whole pixels, margin included.
                    boxes_by_slice = {}
                    for index, lesion in enumerate(lesions):
                        z_off, y_off, x_off = (int(s.start) for s in lesion["box"])
                        for k in range(lesion["mask"].shape[0]):
                            ys, xs = np.nonzero(lesion["mask"][k])
                            if ys.size == 0:
                                continue
                            boxes_by_slice.setdefault(z_off + k, []).append((
                                max(y_off + int(ys.min()) - margin, 0), min(y_off + int(ys.max()) + 1 + margin, height),
                                max(x_off + int(xs.min()) - margin, 0), min(x_off + int(xs.max()) + 1 + margin, width),
                                index,
                            ))
                    box_only = np.zeros(ref.shape, dtype=bool)
                    n_boxes = sum(len(boxes) for boxes in boxes_by_slice.values())
                    for z, boxes in boxes_by_slice.items():
                        for y0, y1, x0, x1, _ in boxes:
                            yy, xx = np.mgrid[y0:y1, x0:x1] + 0.5
                            box_only[z, y0:y1, x0:x1] |= (
                                ((yy - (y0 + y1) / 2.0) / ((y1 - y0) / 2.0)) ** 2
                                + ((xx - (x0 + x1) / 2.0) / ((x1 - x0) / 2.0)) ** 2
                            ) <= 1.0
                    path = find_saved_segmentation(_AUTOSEG_SAM2_METHOD, cid)
                    if path.is_file():
                        auto = _autoseg_load_mask(path, case["image"], ref.shape)
                        _autoseg_record(case, _AUTOSEG_SAM2_METHOD, ref, auto, spacing_zyx, n_pieces, lesions,
                                        time.perf_counter() - t0, n_prompts=n_boxes, box_only=box_only,
                                        cache_source="drive_cache")
                        for method, packed in _autoseg_kept.get(cid, {}).items():
                            if (cid, method) in _autoseg_rows:
                                _autoseg_rows[(cid, method)]["dice_vs_sam2"] = _autoseg_dice(_autoseg_unpack(packed), auto)
                        print(f"[AUTO-SEG][SAM 2] patient {cid}: drive_cache -> {path}")
                        continue
                    if initialization_error is not None:
                        raise RuntimeError(f"SAM2 initialization failed: {initialization_error}")
                    if model is None:
                        try:
                            _autoseg_ensure_weights(
                                "SAM 2", SAM2_WEIGHTS_PATH, ("config.json", "preprocessor_config.json", "model.safetensors"),
                                "facebook/sam2.1-hiera-base-plus",
                                ("config.json", "model.safetensors", "preprocessor_config.json", "processor_config.json"), None,
                                "https://huggingface.co/facebook/sam2.1-hiera-base-plus/tree/main",
                            )
                            try:
                                from transformers import Sam2Model, Sam2Processor
                                from transformers.utils import logging as _hf_logging
                            except Exception as exc:
                                raise ImportError(
                                    "This transformers version has no SAM 2 support. Run Section 0, cell 3 again with "
                                    "RUN_AUTOSEG_SAM2 = True: its install line installs the pinned version."
                                ) from exc
                            _verbosity = _hf_logging.get_verbosity()
                            _hf_logging.set_verbosity_error()   # the folder also holds the video part of SAM 2, which is not used here
                            try:
                                processor = Sam2Processor.from_pretrained(str(SAM2_WEIGHTS_PATH), local_files_only=True)
                                model = Sam2Model.from_pretrained(str(SAM2_WEIGHTS_PATH), local_files_only=True).to(_AUTOSEG_DEVICE).eval()
                            finally:
                                _hf_logging.set_verbosity(_verbosity)
                        except Exception as exc:
                            initialization_error = exc
                            model, processor = None, None
                            raise
                    # Display window for the 8-bit pictures SAM 2 expects: 0.5th to 99.5th
                    # percentile of the non-zero voxels of the whole volume.
                    finite = np.isfinite(img)
                    values = img[finite & (img != 0)]
                    if values.size < 10:
                        values = img[finite]
                    low, high = (float(v) for v in np.percentile(values, [0.5, 99.5]))
                    if high <= low:
                        high = low + 1.0
                    # Each tumour's slices are collected in its own small volume (its box plus the
                    # margin), so that its largest connected piece can be kept afterwards.
                    regions = [tuple(slice(max(s.start - margin, 0), min(s.stop + margin, n)) if a else s
                                     for a, (s, n) in enumerate(zip(lesion["box"], ref.shape))) for lesion in lesions]
                    per_tumour = [np.zeros(tuple(s.stop - s.start for s in region), dtype=bool) for region in regions]
                    auto = np.zeros(ref.shape, dtype=bool)
                    for z in sorted(boxes_by_slice):
                        boxes = boxes_by_slice[z]
                        picture = np.clip((np.nan_to_num(img[z], nan=low) - low) / (high - low), 0.0, 1.0)
                        picture = np.repeat((picture * 255.0).astype(np.uint8)[..., None], 3, axis=2)
                        inputs = processor(
                            images=picture, return_tensors="pt",
                            input_boxes=[[[float(x0), float(y0), float(x1), float(y1)] for (y0, y1, x0, x1, _) in boxes]],
                        ).to(_AUTOSEG_DEVICE)
                        with torch.inference_mode():
                            outputs = model(**inputs, multimask_output=False)
                        masks = processor.post_process_masks(
                            outputs.pred_masks.cpu(), inputs["original_sizes"].cpu()
                        )[0].numpy()
                        for (y0, y1, x0, x1, index), mask in zip(boxes, masks):
                            # Only the part inside the box is kept, so the mask cannot land elsewhere.
                            rz, ry, rx = (s.start for s in regions[index])
                            per_tumour[index][z - rz, y0 - ry:y1 - ry, x0 - rx:x1 - rx] |= mask[0, y0:y1, x0:x1].astype(bool)
                    for region, tumour_mask in zip(regions, per_tumour):
                        auto[region] |= _autoseg_largest_piece(tumour_mask)
                    print(f"[AUTO-SEG][SAM 2] patient {cid}: generated -> {path}")
                    row = _autoseg_record(case, _AUTOSEG_SAM2_METHOD, ref, auto, spacing_zyx, n_pieces, lesions,
                                          time.perf_counter() - t0, n_prompts=n_boxes, box_only=box_only)
                    for method, packed in _autoseg_kept.get(cid, {}).items():
                        if (cid, method) in _autoseg_rows:
                            _autoseg_rows[(cid, method)]["dice_vs_sam2"] = _autoseg_dice(_autoseg_unpack(packed), auto)
                except Exception as exc:
                    print(f"[AUTO-SEG][SAM 2] patient {cid}: failed ({exc!r})")
                    _autoseg_record_failure(case, [_AUTOSEG_SAM2_METHOD], exc)
                    _autoseg_release_gpu()
        finally:
            del model, processor
            _autoseg_release_gpu()

    # ── Run ─────────────────────────────────────────────────────────────────
    _autoseg_errors = {}
    _autoseg_cases = []
    try:
        _autoseg_cases, _ = get_shared_patient_data("Auto-segmentation")
        if _AUTOSEG_PILOT:
            _autoseg_cases = [c for c in _autoseg_cases if c["split_set"] == SPLIT_SET_DEVELOPMENT]
            _autoseg_cases = _autoseg_cases[:int(AUTOSEG_PILOT_N_PATIENTS)]
            print(f"[AUTO-SEG] PILOT: the first {len(_autoseg_cases)} DEVELOPMENT patients only. Set "
                  "AUTOSEG_PILOT_N_PATIENTS = None at the top of this cell for the whole cohort.")
        else:
            print(f"[AUTO-SEG] Whole cohort: {len(_autoseg_cases)} patients (development and test).")

        if RUN_AUTOSEG_NNINTERACTIVE:
            try:
                _autoseg_run_nninteractive(_autoseg_cases)
            except Exception as _autoseg_exc:
                _autoseg_errors["nnInteractive"] = repr(_autoseg_exc)
                print("[AUTO-SEG][ERROR] nnInteractive did not run:", _autoseg_exc)
                if not isinstance(_autoseg_exc, (FileNotFoundError, ImportError)):
                    traceback.print_exc()

        if RUN_AUTOSEG_SAM2:
            try:
                _autoseg_run_sam2(_autoseg_cases)
            except Exception as _autoseg_exc:
                _autoseg_errors["SAM 2"] = repr(_autoseg_exc)
                print("[AUTO-SEG][ERROR] SAM 2 did not run:", _autoseg_exc)
                if not isinstance(_autoseg_exc, (FileNotFoundError, ImportError)):
                    traceback.print_exc()
    finally:
        # Put the global PyTorch settings and the random-number state back (see above).
        torch.set_num_threads(_autoseg_saved_state["threads"])
        torch.backends.cudnn.benchmark = _autoseg_saved_state["cudnn_benchmark"]
        rng_state_restore(_autoseg_saved_state["rng"])
        _autoseg_release_gpu()

    # ── Tables ──────────────────────────────────────────────────────────────
    _AUTOSEG_COLUMNS = [
        "patient_id", "split_set", "test_set", "method", "cache_source", "segmentation_file", "taken_from_existing_mask", "status",
        "pieces_in_existing_mask", "tumours_prompted", "prompts_given",
        "existing_volume_ml", "automatic_volume_ml", "volume_ratio",
        "dice", "dice_without_prompt_slices", "dice_box_only_control", "dice_vs_sam2",
        "dice_analysed_slices", "same_central_slices", "max_central_slice_shift_mm", "min_crop_overlap",
        "automatic_outside_existing_region_pct", "seconds",
    ]
    _AUTOSEG_METHOD_ORDER = {m: i for i, m in enumerate(_AUTOSEG_NNI_METHODS + [_AUTOSEG_SAM2_METHOD])}

    def _autoseg_tables(cases):
        """Per-patient table and one summary line per method and patient set; both are saved."""
        results = pd.DataFrame(list(_autoseg_rows.values())).reindex(columns=_AUTOSEG_COLUMNS)
        case_order = {c["canonical_id"]: i for i, c in enumerate(cases)}
        results = results.sort_values(
            by=["method", "patient_id"],
            key=lambda col: col.map(_AUTOSEG_METHOD_ORDER if col.name == "method" else case_order),
        ).reset_index(drop=True)
        per_patient_csv = Path(AUTOSEG_OUT_DIR) / f"auto_segmentation_dice_per_patient{_AUTOSEG_TAG}.csv"
        results.to_csv(per_patient_csv, index=False)

        summary_rows = []
        set_name = results["test_set"].where(results["split_set"] == SPLIT_SET_TEST, "development")
        for (method, patient_set), part in results.groupby(["method", set_name], sort=False):
            done = part[part["dice"].notna()]
            same = done["same_central_slices"].astype(str).str.split("/", expand=True)
            same_pct = float("nan")
            if len(done) and same.shape[1] == 2:
                same_pct = 100.0 * pd.to_numeric(same[0], errors="coerce").sum() / pd.to_numeric(same[1], errors="coerce").sum()
            summary_rows.append({
                "method": method,
                "patient_set": patient_set,
                "taken_from_existing_mask": _AUTOSEG_DEPENDENCE[method],
                "patients": len(part),
                "reused": int((part["cache_source"] == "drive_cache").sum()),
                "generated": int((part["cache_source"] == "generated").sum()),
                "failed": int(part["status"].astype(str).str.startswith("failed").sum()),
                "dice_median": _autoseg_median(done["dice"]),
                "dice_q1": float(done["dice"].quantile(0.25)) if len(done) else float("nan"),
                "dice_q3": float(done["dice"].quantile(0.75)) if len(done) else float("nan"),
                "dice_mean": float(done["dice"].mean()) if len(done) else float("nan"),
                "dice_min": float(done["dice"].min()) if len(done) else float("nan"),
                f"n_dice_below_{float(AUTOSEG_LOW_DICE):.2f}": int((done["dice"] < float(AUTOSEG_LOW_DICE)).sum()),
                "dice_without_prompt_slices_median": _autoseg_median(done["dice_without_prompt_slices"]),
                "dice_box_only_control_median": _autoseg_median(done["dice_box_only_control"]),
                "dice_analysed_slices_median": _autoseg_median(done["dice_analysed_slices"]),
                "same_central_slices_pct": same_pct,
                "min_crop_overlap_median": _autoseg_median(done["min_crop_overlap"]),
                "dice_vs_sam2_median": _autoseg_median(done["dice_vs_sam2"]),
                "seconds_per_patient_median": _autoseg_median(done["seconds"]),
            })
        summary = pd.DataFrame(summary_rows)
        summary_csv = Path(AUTOSEG_OUT_DIR) / f"auto_segmentation_dice_summary{_AUTOSEG_TAG}.csv"
        summary.to_csv(summary_csv, index=False)

        print("\n" + "=" * 78)
        print("[AUTO-SEG] DSC between each automatic mask and the existing mask"
              + (" - PILOT, development patients only" if _AUTOSEG_PILOT else ""))
        print("=" * 78)
        print(summary.drop(columns=["taken_from_existing_mask"]).round(3).to_string(index=False))
        print("\nHow to read it:")
        print("  dice                        whole 3D masks.")
        print("  dice_without_prompt_slices  nnInteractive only: the slices that carried a prompt are left out,")
        print("                              so nothing in this figure is copied from the existing mask.")
        print("  dice_box_only_control       SAM 2 only: an ellipse drawn in the same boxes without looking at")
        print("                              the image. SAM 2 adds information only where its dice is higher.")
        print("  dice_analysed_slices        on the central slice(s) the classifier analyses (N_SLICES).")
        print("  same_central_slices_pct     how often the automatic mask selects the same central slice(s).")
        print("  min_crop_overlap            overlap of the tumour crops the two masks produce (1 = identical).")
        print(f"\n[AUTO-SEG] Per-patient table: {per_patient_csv}")
        print(f"[AUTO-SEG] Summary table:     {summary_csv}")
        if RUN_AUTOSEG_SAM2:
            print(f"[AUTO-SEG] Permanent SAM2 masks:  {CAS2_ROOT}")
        if RUN_AUTOSEG_NNINTERACTIVE:
            print(f"[AUTO-SEG] Permanent lasso masks: {LASSO_ROOT}")
        not_ok = results[results["status"].astype(str) != "ok"]
        if len(not_ok):
            print(f"\n[AUTO-SEG] {len(not_ok)} patient/method result(s) need a look:")
            print(not_ok[["patient_id", "method", "status"]].to_string(index=False))
        return results, summary

    def _autoseg_run_record(cases):
        """Save the settings, package versions and weight-file checksums of this run."""
        info = {
            "pilot": _AUTOSEG_PILOT, "patients": len(cases), "device": str(_AUTOSEG_DEVICE),
            "nninteractive_prompts": ",".join(NNINTERACTIVE_PROMPTS) if RUN_AUTOSEG_NNINTERACTIVE else "",
            "sam2_box_margin_px": int(SAM2_BOX_MARGIN_PX) if RUN_AUTOSEG_SAM2 else "",
            "min_lesion_voxels": int(AUTOSEG_MIN_LESION_VOXELS),
            "keep_largest_piece": bool(AUTOSEG_KEEP_LARGEST_PIECE),
            "max_lesions_per_patient": int(AUTOSEG_MAX_LESIONS_PER_PATIENT),
            "n_slices": int(N_SLICES),
            "sam2_cache_root": CAS2_ROOT,
            "lasso_cache_root": LASSO_ROOT,
            "masks_reused": sum(row.get("cache_source") == "drive_cache" for row in _autoseg_rows.values()),
            "masks_generated": sum(row.get("cache_source") == "generated" for row in _autoseg_rows.values()),
        }
        for package in AUTOSEG_PACKAGE_PINS:
            try:
                info[f"version_{package}"] = _autoseg_metadata.version(package)
            except Exception:
                info[f"version_{package}"] = "not installed"
        info["version_torch"] = torch.__version__
        for label, enabled, weight_file in (
            ("nninteractive", RUN_AUTOSEG_NNINTERACTIVE,
             Path(str(NNINTERACTIVE_WEIGHTS_PATH)) / "fold_0" / "checkpoint_final.pth"),
            ("sam2", RUN_AUTOSEG_SAM2, Path(str(SAM2_WEIGHTS_PATH)) / "model.safetensors"),
        ):
            method = "nninteractive_lasso" if label == "nninteractive" else _AUTOSEG_SAM2_METHOD
            generated_now = any(row.get("method") == method and row.get("cache_source") == "generated"
                                for row in _autoseg_rows.values())
            if enabled and generated_now and weight_file.is_file():
                info[f"weights_{label}"] = str(weight_file)
                info[f"weights_{label}_sha256"] = _autoseg_sha256(weight_file)
        for name, error in _autoseg_errors.items():
            info[f"error_{name}"] = error
        pd.DataFrame([{"item": k, "value": v} for k, v in info.items()]).to_csv(
            Path(AUTOSEG_OUT_DIR) / f"auto_segmentation_run_record{_AUTOSEG_TAG}.csv", index=False
        )
        used = (["nnInteractive", "nnunetv2", "dynamic-network-architectures"] if RUN_AUTOSEG_NNINTERACTIVE else []) \
            + (["transformers"] if RUN_AUTOSEG_SAM2 else [])
        differs = [f"{p}: running {info[f'version_{p}']}, pinned {AUTOSEG_PACKAGE_PINS[p]}"
                   for p in used if p in AUTOSEG_PACKAGE_PINS and info[f"version_{p}"] != AUTOSEG_PACKAGE_PINS[p]]
        if differs:
            print("[AUTO-SEG][versions] Running packages differ from AUTOSEG_PACKAGE_PINS:", "; ".join(differs))

    def _autoseg_overlays():
        """Existing mask and automatic mask on the existing mask's central axial slice."""
        if not _autoseg_qc or not SHOW_ANALYSED_SLICES:
            return
        import matplotlib.pyplot as plt
        from matplotlib.lines import Line2D
        methods = [m for m in _AUTOSEG_METHOD_ORDER if any(m in q["auto"] for q in _autoseg_qc.values())]
        patient_ids = list(_autoseg_qc)
        per_figure = 4
        for start in range(0, len(patient_ids), per_figure):
            chunk = patient_ids[start:start + per_figure]
            fig, axes = plt.subplots(len(chunk), max(len(methods), 1), squeeze=False,
                                     figsize=(2.6 * max(len(methods), 1), 2.8 * len(chunk)))
            for r, cid in enumerate(chunk):
                q = _autoseg_qc[cid]
                for c in range(axes.shape[1]):
                    axes[r][c].axis("off")
                for c, method in enumerate(methods):
                    ax = axes[r][c]
                    ax.imshow(q["image"], cmap="gray", vmin=0.0, vmax=1.0)
                    if q["ref"].any():
                        ax.contour(q["ref"], levels=[0.5], colors="lime", linewidths=1.2)
                    auto = q["auto"].get(method)
                    if auto is not None and auto.any():
                        ax.contour(auto, levels=[0.5], colors="magenta", linewidths=1.2, linestyles="dashed")
                    dice = q["dice"].get(method, float("nan"))
                    ax.set_title(f"{method}\n3D DSC {dice:.2f}", fontsize=7)
                axes[r][0].text(-0.05, 0.5, f"{cid}\nslice {q['z']}", transform=axes[r][0].transAxes,
                                fontsize=8, va="center", ha="right")
            fig.legend(
                handles=[Line2D([0], [0], color="lime", lw=1.6, label="existing mask (solid)"),
                         Line2D([0], [0], color="magenta", lw=1.6, ls="--", label="automatic mask (dashed)")],
                loc="upper center", ncol=2, fontsize=8, frameon=False,
            )
            fig.tight_layout(rect=(0.05, 0, 1, 0.95))
            png = Path(AUTOSEG_OUT_DIR) / f"overlays{_AUTOSEG_TAG}" / (
                f"overlay_patients_{start + 1:03d}-{start + len(chunk):03d}.png")
            png.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(png, dpi=130)
            plt.show()
            plt.close(fig)
        print(f"[AUTO-SEG] Overlay figures: {Path(AUTOSEG_OUT_DIR) / ('overlays' + _AUTOSEG_TAG)}")

    AUTOSEG_RESULTS, AUTOSEG_SUMMARY = pd.DataFrame(columns=_AUTOSEG_COLUMNS), pd.DataFrame()
    if not _autoseg_rows:
        print("[AUTO-SEG] No result was produced; see the messages above.")
    else:
        AUTOSEG_RESULTS, AUTOSEG_SUMMARY = _autoseg_tables(_autoseg_cases)
        _autoseg_run_record(_autoseg_cases)
        try:
            _autoseg_overlays()
        except Exception as _autoseg_exc:
            print("[AUTO-SEG] Overlay figures unavailable:", repr(_autoseg_exc))

    # Free the memory held for the comparison and the figures.
    _autoseg_kept.clear()
    _autoseg_qc.clear()
    _autoseg_release_gpu()
else:
    print("[AUTO-SEG] Skipped: RUN_AUTOSEG_NNINTERACTIVE and RUN_AUTOSEG_SAM2 are both False.")

## Section 0, cell 6 - Prepare selected segmentation analyses

Checks all selected generated masks before training, retains the original patient
cohort, and prepares separate mask-specific CNN inputs. The original analysis runs
first; the additional full analyses run in Section 6. If generation switches are off,
previously saved full-cohort masks can be used.


In [ ]:
# Section 0, cell 7: Prepare generated-mask downstream analyses
prepare_segmentation_analysis_jobs()


## Section 1, cell 1 - End-to-end CNN with frozen backbone

A 2D CNN is trained directly on the tumour images. The pretrained backbone stays frozen and only
the classification head is trained. `EPOCHS_HEAD` is the maximum number of epochs of the
epoch-selection run in each fold. The final refit uses the median of the epoch counts selected in
the folds.

**Input.** Paired NRRD image and tumour mask per patient; the mask is resampled to the image grid
when geometry differs. Each plane has its own physical-square crop around the selected slice tumour
bounding box, which is expanded by `CROP_SAFETY_MARGIN_PX = 2` native pixels on every
side, converted to the smallest enclosing physical square and then resized to 224×224. This retains
the complete tumour in each selected plane without an additional occupancy target.
The ROI is z-scored, clipped at `INTENSITY_Z_SPAN_SD = 2.5` SD and rescaled to [0, 1]. The configured
1, 2 or 3 central slices are taken in axial, sagittal and coronal order, resampled to
`CNN_INPUT_SIZE = 224` and the grey channel is replicated to three (`CHANNEL_POLICY = "gray3"`).
Per-backbone normalisation then matches each network's pretraining. Slice features are mean-pooled
into one patient-level prediction.

**Evaluation.** `N_FOLDS` × `N_REPEATS` cross-validation within the development set. In each fold,
`FROZEN_E2E_EPOCH_SELECTION_FRACTION` of the training patients are set aside and the epoch with the
best AUC on them is selected. The fold model is then trained on all training patients of the fold
for that number of epochs and the validation patients of the fold are scored once, so the patients
that are scored never influence the choice of epoch. The median of the selected epoch counts is then
used to retrain one model on the full development set before a single evaluation on the untouched
holdout. In an optimization comparison this retraining and the holdout evaluation run only when
`OPTIMIZATION_EVALUATE_HOLDOUT` is `True`.

**Output.** `OUT_ROOT/PART1_frozen_e2e_cnn`, plus `RESULTS_REGISTRY['segment1_cnn']` for the final
overview.

In [ ]:
# Section 1, cell 2: Imports
if globals().get("RUN_FROZEN_CNN_E2E", True):
    import sys, subprocess, os, warnings, re, shutil, random, math
    from pathlib import Path
    from copy import deepcopy


    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import torchvision.models as tv_models
    import timm
    import SimpleITK as sitk
    from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, StratifiedShuffleSplit
    from sklearn.metrics import (
        roc_auc_score, balanced_accuracy_score, f1_score, accuracy_score,
        precision_score, recall_score, confusion_matrix, matthews_corrcoef,
        brier_score_loss
    )
    from tqdm.auto import tqdm

    # Warnings stay visible: numerical, API and checkpoint warnings must not be hidden.
    warnings.filterwarnings("default")
    # Silence only the repetitive Jupyter utcnow deprecation notice.
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        module=r"jupyter_client\.session",
    )

    ensure_drive_mounted()   # helper defined in Section 0

    seed_everything(GLOBAL_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
else:
    print('[SKIP] Section 1: frozen end-to-end CNN disabled by RUN_FROZEN_CNN_E2E=False.')


In [ ]:
# Section 1, cell 3: Configuration
if globals().get("RUN_FROZEN_CNN_E2E", True):

    # ── Data paths ────────────────────────────────────────────────────────────────
    # DEVELOPMENT_ROOT, TEST_ROOT, LABELS_CONFIGS, WEIGHTS_BASE, the resolved RadImageNet
    # checkpoint paths and the input scaling of every backbone (RADIMAGENET_NORM_BY_BACKBONE
    # and radimagenet_norm_for()) are defined once in Section 0, cell 3 and are reused here.

    # ── Output ────────────────────────────────────────────────────────────────────
    for _required in ("LOCAL_CACHE_DIR", "CNN_INPUT_SIZE_SHARED", "SLICE_PLANES",
                      "CROP_SAFETY_MARGIN_PX", "INTENSITY_Z_SPAN_SD",
                      "TUMOR_CONTEXT_MODE", "ROI_ONLY_BACKGROUND_VALUE",
                      "FROZEN_E2E_AUGMENTATION_LEVEL", "radimagenet_norm_for", "apply_input_scaling",
                      "collect_generic_image_mask_pairs", "load_label_table",
                      "holdout_evaluation_enabled", "HoldoutNotEvaluated", "rows_for_display",
                      "calculate_auc_intervals", "preview_analysed_slices"):
        if _required not in globals():
            raise RuntimeError(f"{_required} is missing. Run Section 0 before Section 1.")

    # Sub-folder of the Section 0 output root, so results persist after the session.
    OUT_DIR         = PART1_OUT_DIR
    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)

    # ── Backbones trainable in this section ───────────────────────────────────────
    _seg1_backbone_source = (list(OPTIMIZATION_BACKBONES) if OPTIMIZATION else list(SHARED_BACKBONES))
    RUN_BACKBONES = [b for b in _seg1_backbone_source
                     if str(b).lower() in _SEG1_SUPPORTED_BACKBONES]
    _seg1_skipped_backbones = [b for b in _seg1_backbone_source
                               if str(b).lower() not in _SEG1_SUPPORTED_BACKBONES]
    if _seg1_skipped_backbones:
        print(f"[SECTION 1] Not trained end-to-end (feature-extracted in Section 2 only): {_seg1_skipped_backbones}")
    CHANNEL_POLICY = "gray3"   # replicate T1cc into all 3 channels; mask channel dropped
    CNN_INPUT_SIZE = CNN_INPUT_SIZE_SHARED  # shared in Section 0
    # ── Preprocessing constants, shared with Section 2 ────────────────────────────
    CHANNEL_MODE   = "t1cc_t1cc_mask"  # intermediate slice assembly before gray3 is applied

    # ── Training ──────────────────────────────────────────────────────────────────
    # The development/holdout split (DEVELOPMENT_ROOT / TEST_ROOT folders) comes from Section 0.
    N_FOLDS         = int(OPTIMIZATION_N_FOLDS) if OPTIMIZATION else 5
    N_REPEATS       = int(OPTIMIZATION_N_REPEATS) if OPTIMIZATION else 2
    EPOCHS_HEAD     = 15     # Maximum epochs of the epoch-selection run in each fold.
    LR_HEAD         = 1e-3
    # Epoch selection never sees the fold that is scored. Each outer-training fold is split
    # once more: this share of its patients forms the inner validation set on which the
    # best epoch is found. The fold model is then trained on all outer-training patients
    # for that number of epochs and the outer fold is scored once. The final refit uses
    # the median of the epochs selected in the folds.
    FROZEN_E2E_EPOCH_SELECTION_FRACTION = 0.20

    # One execution mode; backbone parameters are never unfrozen here.
    END2END_MODES = ['frozen']
    # FROZEN_E2E_AUGMENTATION_LEVEL, the training augmentation of this section, is set in
    # Section 0, cell 3. In optimization mode each arm replaces it and
    # apply_part1_optimization_arm(None) restores the Section 0 value.
    print(f"[SECTION 1] modes={END2END_MODES} | backbone remains frozen | "
          f"training augmentation={str(FROZEN_E2E_AUGMENTATION_LEVEL).upper()}")

    def apply_part1_optimization_arm(arm):
        """Apply the shared optimization arm to frozen end-to-end CNN training."""
        global CNN_INPUT_SIZE, N_SLICES, SLICE_PLANES
        global FROZEN_E2E_AUGMENTATION_LEVEL
        if arm is None:
            CNN_INPUT_SIZE = int(CNN_INPUT_SIZE_SHARED)
            N_SLICES = int(globals().get("OPTIMIZATION_REFERENCE_N_SLICES", N_SLICES))
            FROZEN_E2E_AUGMENTATION_LEVEL = str(OPTIMIZATION_REFERENCE_AUG_BY_WORKFLOW["frozen"]).upper()
        else:
            CNN_INPUT_SIZE = int(arm["input_size"])
            N_SLICES = int(arm["n_slices"])
            FROZEN_E2E_AUGMENTATION_LEVEL = str(arm["augmentation"]).upper()
        if N_SLICES not in (1, 2, 3):
            raise ValueError(f"N_SLICES must be 1, 2 or 3; got {N_SLICES}")
        SLICE_PLANES = ("axial", "sagittal", "coronal")[:N_SLICES]
        try:
            _SLICE_CACHE.clear()
        except Exception:
            pass
        if arm is not None:
            print("\n" + "=" * 78)
            print(f"[SECTION 1 OPTIMIZATION ARM] {arm['arm_id']}"
                  + ("   (REFERENCE)"
                     if optimization_arm_is_reference("frozen", arm) else ""))
            print(f"    input size={CNN_INPUT_SIZE} | slices={N_SLICES} "
                  f"planes={SLICE_PLANES} | augmentation={FROZEN_E2E_AUGMENTATION_LEVEL}")
            print("=" * 78)

    PART1_OPTIMIZATION_ARM_MODE_PAIRS = (
        [(_arm, _mode) for _arm in OPTIMIZATION_ARMS for _mode in END2END_MODES]
        if OPTIMIZATION else [(None, _mode) for _mode in END2END_MODES]
    )

    BATCH_SIZE      = 8
    DROPOUT         = 0.5
    WEIGHT_DECAY    = 1e-4
    EARLY_STOP_PATIENCE = 10  # Stop after this many epochs without an AUC improvement.

    # ── Class-imbalance / decision threshold ──────────────────────────────────────
    USE_POS_WEIGHT   = True   # if True, BCEWithLogitsLoss uses pos_weight to counter imbalance
    POS_WEIGHT_SCALE = 1.0    # multiplier on the automatic n_neg/n_pos ratio:
                              # 1.0 = fully balanced (default), <1.0 = milder, >1.0 = stronger.
                              # Ignored when USE_POS_WEIGHT is False.
    DECISION_THRESHOLD = 0.5  # fallback threshold,
                              # used only when TOP_RISK_PERCENT is False.

    # IMAGENET_MEAN and IMAGENET_STD are defined once in Section 0, cell 3.

    ACTIVE_LABEL_CONFIGS = [c for c in LABELS_CONFIGS
                            if c.get("enabled")]
    print(f"Outcome of this run: {[c['tag'] for c in ACTIVE_LABEL_CONFIGS]}")
    print(f"Backbones: {RUN_BACKBONES}")
    print(f"Channel policy: {CHANNEL_POLICY}")

    def _print_weight_status(label, path):
        p = Path(str(path))
        status = "FOUND" if p.exists() else "MISSING"
        size = f" ({p.stat().st_size/1024**2:.1f} MB)" if p.exists() else ""
        print(f"  {label}: {status}{size} | {p}")

    print("RadImageNet weights:")
    _print_weight_status("ResNet-10t", RADIMAGENET_RESNET10T_WEIGHTS_PATH)
    _print_weight_status("ResNet-18", RADIMAGENET_RESNET18_WEIGHTS_PATH)
    _print_weight_status("ResNet-50", RADIMAGENET_RESNET50_WEIGHTS_PATH)
    _print_weight_status("DenseNet-121", RADIMAGENET_DENSENET121_WEIGHTS_PATH)
    print("Input scaling per backbone:",
          {b: radimagenet_norm_for(b) for b in RUN_BACKBONES})


In [ ]:
# Section 1, cell 4: Shared patient manifest
if globals().get("RUN_FROZEN_CNN_E2E", True):
    # Discovery, pairing, labels and local copying were completed once in Section 0.
    # The next cell requests independent copies of that fixed shared manifest.
    pass


In [ ]:
# Section 1, cell 5: Label loading
if globals().get("RUN_FROZEN_CNN_E2E", True):

    # Reuse the label tables read and validated once in Section 0, cell 3.

    def attach_labels(pairs: list, label_df: pd.DataFrame) -> list:
        """Return the unique image/mask pairs that have an exact canonical-ID label."""
        _canonical_unique_patient_ids(
            [item.get("canonical_id", item.get("patient_id")) for item in pairs],
            context="Section 1 image/mask pairs",
        )
        _canonical_unique_patient_ids(label_df["canonical_id"], context="Section 1 label table")
        lmap = dict(zip(label_df["canonical_id"].astype(str), label_df["label"].astype(int)))
        out = []
        for item in pairs:
            cid = canonical_patient_id(item.get("canonical_id", item.get("patient_id")))
            if cid in lmap:
                out.append({**item, "canonical_id": cid, "label": int(lmap[cid])})
        print(f"  Labelled pairs: {len(out)} / {len(pairs)}")
        return out

    # Reuse Section 0's fixed cohort and labels; no Drive scan, CSV read or copying.
    raw_pairs, all_labels_dfs = get_shared_patient_data("Section 1")
    if set(all_labels_dfs) != {cfg["tag"] for cfg in ACTIVE_LABEL_CONFIGS}:
        raise RuntimeError("Active label settings changed. Rerun Section 0, cell 3.")


In [ ]:
# Section 1, cell 6: Shared volume preprocessing
if globals().get("RUN_FROZEN_CNN_E2E", True):
    # Arm-switching cells may clear this compatibility dictionary. The shared disk
    # cache and bounded CPU cache persist across arms and workflow sections.
    _SLICE_CACHE = {}

    def make_frozen_e2e_patient_slices(image_path: str, mask_path: str,
                            out_size: int = None, use_cache: bool = True) -> torch.Tensor:
        """Return a private copy of Section 0's deterministic (slices, 3, H, W) input."""
        if out_size is None:
            out_size = CNN_INPUT_SIZE
        tensor, _ = get_shared_patient_inputs(image_path, mask_path, out_size)
        return tensor


In [ ]:
# Section 1, cell 7: Dataset and augmentation
if globals().get("RUN_FROZEN_CNN_E2E", True):

    def _augment_slices_frozen_e2e(xs):
        """Training-only augmentation for Section 1, controlled per optimization arm."""
        level = str(globals().get("FROZEN_E2E_AUGMENTATION_LEVEL", "FLIPS")).upper()
        if level == "NONE":
            return xs
        if level not in ("FLIPS", "BASIC", "COMPLEX", "COMPLEX_JITTER"):
            raise ValueError(f"Unknown Section 1 augmentation level: {level!r}")

        _roi_only = str(globals().get("TUMOR_CONTEXT_MODE", "MIN_BBOX")).upper() == "ROI_ONLY"
        _bg_value = float(globals().get("ROI_ONLY_BACKGROUND_VALUE", 0.5))
        _bg_mask = ((xs - _bg_value).abs() < 1e-4).to(xs.dtype) if _roi_only else None

        if random.random() > 0.5:
            xs = torch.flip(xs, dims=[-1])
        if random.random() > 0.5:
            xs = torch.flip(xs, dims=[-2])
        if level == "FLIPS":
            return xs

        angle = math.radians(random.uniform(-float(AUG_ROTATE_DEG), float(AUG_ROTATE_DEG)))
        if level == "BASIC":
            scale, tx, ty = 1.0, 0.0, 0.0
        else:
            scale = random.uniform(float(AUG_SCALE_RANGE[0]), float(AUG_SCALE_RANGE[1]))
            tx = random.uniform(-float(AUG_TRANSLATE), float(AUG_TRANSLATE)) * 2.0
            ty = random.uniform(-float(AUG_TRANSLATE), float(AUG_TRANSLATE)) * 2.0
        cos_a, sin_a = math.cos(angle), math.sin(angle)
        inv = 1.0 / float(scale)
        theta = torch.tensor(
            [[cos_a * inv, -sin_a * inv, tx],
             [sin_a * inv,  cos_a * inv, ty]],
            dtype=xs.dtype, device=xs.device,
        ).unsqueeze(0).expand(xs.shape[0], -1, -1)
        grid = F.affine_grid(theta, list(xs.shape), align_corners=False)
        xs = F.grid_sample(xs, grid, mode="bilinear", padding_mode="border", align_corners=False)
        if _bg_mask is not None:
            _bg_mask = F.grid_sample(_bg_mask, grid, mode="nearest", padding_mode="border", align_corners=False)
        if level in ("BASIC", "COMPLEX"):
            return xs

        xs = xs.clamp(0.0, 1.0)
        gamma = random.uniform(float(AUG_GAMMA_RANGE[0]), float(AUG_GAMMA_RANGE[1]))
        xs = xs.clamp_min(1e-6).pow(gamma)
        contrast = 1.0 + random.uniform(-float(AUG_CONTRAST), float(AUG_CONTRAST))
        _mean = xs.mean()
        xs = (xs - _mean) * contrast + _mean
        xs = xs + random.uniform(-float(AUG_BRIGHTNESS), float(AUG_BRIGHTNESS))
        if float(AUG_NOISE_STD) > 0.0:
            xs = xs + torch.randn_like(xs) * float(AUG_NOISE_STD)
        xs = xs.clamp(0.0, 1.0)
        if _bg_mask is not None:
            xs = torch.where(_bg_mask > 0.5, torch.full_like(xs, _bg_value), xs)
        return xs

    class SliceDataset(Dataset):
        """
        Returns (slices, label) per patient, slices as (S, 3, input_size, input_size) with
        gray3 applied and normalised per `norm`, label as a float32 0/1 scalar.

        norm: one of the input scalings defined in Section 0, cell 3 ("imagenet",
        "unit_interval", "minus_one_one" or "caffe_over_255"), applied by apply_input_scaling().
        """
        def __init__(self, pairs: list, input_size: int = 224,
                     augment: bool = False, norm: str = "imagenet"):
            self.pairs      = pairs
            self.input_size = input_size
            self.augment    = augment
            self.norm       = str(norm).lower()

        def __len__(self):
            return len(self.pairs)

        def __getitem__(self, idx):
            item = self.pairs[idx]
            # Slices are built directly at the backbone's input size; upsampling from a
            # smaller crop would blur the tumour margin without adding detail.
            xs = make_frozen_e2e_patient_slices(item["image"], item["mask"], out_size=self.input_size)

            # gray3: replicate the T1cc channel three times, drop the mask channel
            xs = xs[:, 0:1, :, :].repeat(1, 3, 1, 1)  # (S, 3, input_size, input_size)

            # Training-only augmentation; the optimization arm controls its level.
            # One patient-level transform is applied coherently to all selected slices.
            if self.augment:
                xs = _augment_slices_frozen_e2e(xs)

            # Per-backbone input scaling, the same function in all three sections.
            xs = apply_input_scaling(xs, self.norm)  # (S, 3, H, W)

            label = torch.tensor(float(item["label"]), dtype=torch.float32)
            return xs, label


In [ ]:
# Section 1, cell 8: Visual check of analysed slices
if globals().get("RUN_FROZEN_CNN_E2E", True):
    # Renders the slices exactly as they reach the CNN: after tumour-centred cropping,
    # intensity normalisation and resampling to CNN_INPUT_SIZE. The green outline is the
    # segmentation carried in channel 2, so the crop framing can be judged directly.
    # Run this before committing to a long training run.
    # preview_analysed_slices() is defined in Section 0, cell 3 and shared with Section 2.

    # Run it on the first patients of the first active outcome definition.
    try:
        _qc_tag = ACTIVE_LABEL_CONFIGS[0]["tag"]
        _qc_pairs = attach_labels(raw_pairs, all_labels_dfs[_qc_tag])
        print(f"[QC] Section 1 input slices | outcome '{_qc_tag}' | "
              f"{int(N_SLICES)} central slice(s): {SLICE_PLANES}")
        preview_analysed_slices(
            _qc_pairs,
            lambda it: make_frozen_e2e_patient_slices(it["image"], it["mask"], out_size=CNN_INPUT_SIZE),
            title="Section 1: ",
        )
    except Exception as _e:
        print("[QC] Slice preview unavailable:", repr(_e))


In [ ]:
# Section 1, cell 9: Backbone builders
if globals().get("RUN_FROZEN_CNN_E2E", True):

    def _make_timm_resnet10t_backbone() -> tuple:
        """Return (feature_net, feature_dim) for ResNet-10t, without head."""
        model = timm.create_model("resnet10t", pretrained=False, num_classes=0, global_pool="avg")
        dim = model.num_features
        return model, int(dim)

    def _make_torchvision_densenet121_backbone() -> tuple:
        """Return (feature_net, feature_dim) for DenseNet-121, without classifier."""
        net = tv_models.densenet121(weights=None)
        dim = net.classifier.in_features  # 1024
        net.classifier = nn.Identity()
        return net, int(dim)

    def _make_torchvision_resnet18_backbone() -> tuple:
        """Return (feature_net, feature_dim) for ResNet-18, without classifier."""
        net = tv_models.resnet18(weights=None)
        dim = net.fc.in_features  # 512
        net.fc = nn.Identity()  # keep global avg-pool, drop the 1000-way head
        return net, int(dim)

    def _make_torchvision_resnet50_backbone() -> tuple:
        """Return (feature_net, feature_dim) for ResNet-50, without classifier."""
        net = tv_models.resnet50(weights=None)
        dim = net.fc.in_features  # 2048
        net.fc = nn.Identity()  # keep global avg-pool, drop the 1000-way head
        return net, int(dim)


    def load_radimagenet_weights(net, weights_path, architecture_name="RadImageNet backbone"):
        p = Path(weights_path) if weights_path else None
        if (p is None) or (not p.exists()):
            parent = p.parent if p is not None else Path(".")
            nearby = []
            if parent.exists():
                nearby = sorted([str(x) for x in parent.rglob("*.pt")][:20] + [str(x) for x in parent.rglob("*.pth")][:20])
            raise FileNotFoundError(
                f"{architecture_name} weights not found at the resolved path. "
                "If the file is visible in Drive, check that Cell 2 runs again after Drive mounts, "
                "and that the filename/folder matches exactly. "
                f"Looked at: {weights_path}. "
                f"Nearby .pt/.pth files under {parent}: {nearby[:20]}"
            )

        try:
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
        except TypeError:
            raw = torch.load(str(p), map_location="cpu")

        def _unwrap_state_dict(obj):
            """Find the actual tensor state_dict inside common checkpoint wrappers."""
            if hasattr(obj, "state_dict"):
                obj = obj.state_dict()
            if not isinstance(obj, dict):
                return obj, "<non-dict checkpoint>"

            # Already a state_dict.
            tensor_like_values = [v for v in obj.values() if torch.is_tensor(v)]
            if len(tensor_like_values) >= 10:
                return obj, "<top-level>"

            for key in (
                "state_dict", "model_state_dict", "model", "net", "network",
                "backbone", "encoder", "weights", "params", "module",
                "model_weights", "weight", "pretrained_dict",
            ):
                if key in obj:
                    nested, where = _unwrap_state_dict(obj[key])
                    if isinstance(nested, dict):
                        return nested, key if where == "<top-level>" else f"{key}.{where}"
            return obj, "<dict without tensor state_dict>"

        sd, sd_location = _unwrap_state_dict(raw)
        if not isinstance(sd, dict):
            raise RuntimeError(
                f"{architecture_name} checkpoint at {p} did not contain a PyTorch state_dict. "
                f"Loaded object type: {type(sd)}"
            )

        current = net.state_dict()
        loadable = {}

        import re as _re

        def _densenet_legacy(key):
            """Convert older torchvision DenseNet keys (norm.N/conv.N) to modern (normN/convN).

            torchvision below ~0.12 serialised dense-layer sub-modules as Sequential
            children, e.g. features.denseblock1.denselayer1.norm.1.weight against the
            modern ...norm1.weight. Pass 1 finds no matches on such a checkpoint.
            """
            key = _re.sub(r'\.norm\.(\d+)', r'.norm\1', key)
            key = _re.sub(r'\.conv\.(\d+)', r'.conv\1', key)
            return key

        def candidate_keys(key):
            # Try the key as-is, then strip known wrapper prefixes only. The
            # torchvision DenseNet "features." prefix is genuine and is kept.
            candidates = []
            seen = set()
            queue = [key]
            prefixes = (
                "module.", "backbone.", "model.", "encoder.", "resnet.",
                "base_model.", "feature_extractor.", "network.", "net.",
                "densenet.", "inception.", "inception_v3.",
                "0.", "1.", "2.", "3.",   # nn.Sequential / numeric-index wrappers
                "_orig_mod.",             # torch.compile saves with this prefix (e.g. ResNet-18)
            )
            while queue:
                cur = queue.pop(0)
                if cur not in seen:
                    seen.add(cur)
                    candidates.append(cur)
                    for pre in prefixes:
                        if cur.startswith(pre):
                            stripped = cur[len(pre):]
                            if stripped not in seen:
                                queue.append(stripped)
            return candidates

        _HEAD_PREFIXES = ("fc.", "classifier.", "AuxLogits.fc.")
        _pass3_rescued = 0  # initialised here; may be updated in Pass 3 below

        tensor_items = []
        non_tensor_keys = []
        for k, v in sd.items():
            if not torch.is_tensor(v):
                non_tensor_keys.append(str(k))
                continue
            tensor_items.append((str(k), v))
            for nk in candidate_keys(str(k)):
                if nk.startswith(_HEAD_PREFIXES):
                    continue
                if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                    loadable[nk] = v
                    break

        # Pass 2, older DenseNet key names: "norm.1"/"conv.1" -> "norm1"/"conv1".
        # Runs for DenseNet, or whenever pass 1 matched fewer than five tensors.
        _arch_lower = architecture_name.lower()
        if "densenet" in _arch_lower or len(loadable) < 5:
            _matched_so_far = set(loadable.keys())
            for k, v in tensor_items:
                modern_k = _densenet_legacy(k)
                if modern_k == k:
                    continue  # key unchanged; already tried in pass 1
                for nk in candidate_keys(modern_k):
                    if nk.startswith(_HEAD_PREFIXES) or nk in _matched_so_far:
                        continue
                    if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                        loadable[nk] = v
                        _matched_so_far.add(nk)
                        break

        # Pass 3: sequential wrappers. RadImageNet checkpoints saved through an
        # nn.Sequential('backbone') use backbone.N.subkey instead of named keys, e.g.
        # ResNet-50 backbone.0.weight -> conv1.weight, DenseNet-121 backbone.0.conv0.weight
        # -> features.conv0.weight. Keys are remapped, then matching is retried.
        _inception_had_aux = False  # default; set precisely in Pass 3 if needed
        _pass3_rescued = 0
        if len(loadable) < 5:
            _matched_so_far = set(loadable.keys())
            _ckpt_keys = [k for k, _ in tensor_items]
            _uses_backbone_prefix = any(k.startswith("backbone.") for k in _ckpt_keys)

            # InceptionV3 variant is identified by the largest backbone.N index:
            #   max_idx <= 17  no AuxLogits, backbone.15/16/17 = Mixed_7a/7b/7c
            #   max_idx >= 18  AuxLogits present, 15 = AuxLogits, 16/17/18 = Mixed_7a/7b/7c
            # Each variant needs its own map. Using the wrong one either leaves 54 tensors
            # unmatched or, because Mixed_7b and Mixed_7c share shapes, loads them into the
            # wrong block; the load-fraction check below catches both.
            _inception_had_aux = False
            if _uses_backbone_prefix and "inception" in _arch_lower:
                _bb_idxs = {
                    int(_m.group(1)) for _k in _ckpt_keys
                    for _m in [_re.match(r"backbone\.(\d+)\.", _k)] if _m
                }
                _inception_had_aux = max(_bb_idxs, default=0) >= 18

            def _normalize_key(k):
                if not _uses_backbone_prefix:
                    return k
                if "densenet" in _arch_lower:
                    if k.startswith("backbone.0."):
                        return "features." + k[len("backbone.0."):]
                elif "inception" in _arch_lower:
                    _imap = {
                        "backbone.0.":  "Conv2d_1a_3x3.", "backbone.1.":  "Conv2d_2a_3x3.",
                        "backbone.2.":  "Conv2d_2b_3x3.",
                        "backbone.4.":  "Conv2d_3b_1x1.", "backbone.5.":  "Conv2d_4a_3x3.",
                        "backbone.7.":  "Mixed_5b.",       "backbone.8.":  "Mixed_5c.",
                        "backbone.9.":  "Mixed_5d.",       "backbone.10.": "Mixed_6a.",
                        "backbone.11.": "Mixed_6b.",       "backbone.12.": "Mixed_6c.",
                        "backbone.13.": "Mixed_6d.",       "backbone.14.": "Mixed_6e.",
                    }
                    if _inception_had_aux:
                        # AuxLogits present (max index >= 18).
                        _imap.update({
                            "backbone.15.": "AuxLogits.",
                            "backbone.16.": "Mixed_7a.",
                            "backbone.17.": "Mixed_7b.",
                            "backbone.18.": "Mixed_7c.",
                        })
                    else:
                        # No AuxLogits (max index <= 17).
                        _imap.update({
                            "backbone.15.": "Mixed_7a.",
                            "backbone.16.": "Mixed_7b.",
                            "backbone.17.": "Mixed_7c.",
                        })
                    for old, dst in _imap.items():
                        if k.startswith(old):
                            return dst + k[len(old):]
                else:  # resnet-50 / resnet-18 / resnet-10t
                    # backbone.N indexes the top-level ResNet children: 0=conv1, 1=bn1,
                    # 4..7=layer1..4. Activations and pooling carry no parameters and are
                    # absent from the checkpoint. The ResNet-10t stem sits inside child 0 as
                    # a conv1 sequence, so the same map applies with the inner index kept.
                    _rmap = {
                        "backbone.0.": "conv1.", "backbone.1.": "bn1.",
                        "backbone.4.": "layer1.", "backbone.5.": "layer2.",
                        "backbone.6.": "layer3.", "backbone.7.": "layer4.",
                    }
                    for old, dst in _rmap.items():
                        if k.startswith(old):
                            return dst + k[len(old):]
                return k

            for orig_k, v in tensor_items:
                norm_k = _normalize_key(orig_k)
                for nk in candidate_keys(norm_k):
                    if nk.startswith(_HEAD_PREFIXES) or nk in _matched_so_far:
                        continue
                    if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                        loadable[nk] = v
                        _matched_so_far.add(nk)
                        _pass3_rescued += 1
                        break

            if _pass3_rescued:
                print(f"[RadImageNet] Pass 3 (sequential-backbone remapping) rescued {_pass3_rescued} tensors")

        sample_ckpt = [(k, tuple(v.shape)) for k, v in tensor_items[:12]]
        sample_expected = [(k, tuple(v.shape)) for k, v in list(current.items())[:12]]

        print(f"[RadImageNet] {architecture_name}: checkpoint={p.name} | state_dict_location={sd_location}")
        print(f"[RadImageNet] tensor keys in checkpoint: {len(tensor_items)} | non-tensor keys: {len(non_tensor_keys)}")
        print(f"[RadImageNet] sample checkpoint keys: {sample_ckpt}")
        print(f"[RadImageNet] sample expected torchvision keys: {sample_expected}")

        if len(loadable) == 0:
            raise RuntimeError(
                f"{architecture_name} loaded 0 compatible tensors from {p}. "
                "This usually means the checkpoint key names do not match the torchvision "
                "architecture expected for this backbone.\n"
                "Common causes:\n"
                "  • Wrong .pt file (e.g. ResNet50 file used for DenseNet121).\n"
                "  • DenseNet saved with old torchvision (<0.12): norm.1/conv.1 vs norm1/conv1 "
                "(pass 2 should have fixed this automatically. If you still see this, the file "
                "may not be a torchvision DenseNet at all).\n"
                "  • InceptionV3 checkpoint from a custom/non-torchvision implementation.\n"
                "  • Checkpoint is a full Keras/TF model, not a PyTorch state_dict.\n"
                "Inspect the sample checkpoint keys printed above vs. the expected torchvision keys "
                "and add an architecture-specific key-conversion pass if needed."
            )

        missing, unexpected = net.load_state_dict(loadable, strict=False)
        _used_legacy = ("densenet" in architecture_name.lower() or len(loadable) < 5) and any(
            _densenet_legacy(k) != k for k, _ in tensor_items
        )
        _strategy = ("densenet_legacy_keys" if _used_legacy
                     else "sequential_backbone_remapping" if _pass3_rescued
                     else "standard")
        print(f"[RadImageNet] {architecture_name}: loaded {len(loadable)}/{len(current)} compatible "
              f"backbone tensors from {p.name} (strategy={_strategy})")

        # ── Load completeness ──────────────────────────────────────────────────────
        # Counting the target model's non-head tensors gives an exact load fraction for
        # every architecture, so the incomplete-load warning stays meaningful.
        _n_expected = len([k for k in current if not k.startswith(_HEAD_PREFIXES)])
        _load_frac = len(loadable) / max(1, _n_expected)
        if _load_frac < 0.80:
            print(
                f"[RadImageNet][warning] {architecture_name}: loaded {len(loadable)}/{_n_expected} "
                f"expected tensors ({_load_frac:.0%}). A healthy load is typically >90%. "
                "Inspect the sample keys above for a naming mismatch."
            )

        # ── Strict InceptionV3 check ──────────────────────────────────────────────
        # Key order alone cannot confirm the pass-3 index map, and equal shapes can hide a
        # wrong assignment. A wrong map leaves target slots empty, so the load fraction
        # falls; a correct load fills all 94 convolution blocks and ~98% of non-head slots,
        # the remainder being the auxiliary branch. Anything below 0.95 raises.
        if "inception" in _arch_lower:
            _bb_indices = sorted({
                int(_m.group(1)) for _k, _ in tensor_items
                for _m in [_re.match(r"backbone\.(\d+)\.", _k)] if _m
            })
            print(f"[RadImageNet] {architecture_name}: checkpoint backbone.N block "
                  f"indices present: {_bb_indices} | inferred aux_logits_when_saved={_inception_had_aux}")
            if _load_frac < 0.95:
                raise RuntimeError(
                    f"[RadImageNet][FATAL] {architecture_name}: loaded only "
                    f"{len(loadable)}/{_n_expected} non-head tensors ({_load_frac:.0%}); "
                    "a correct InceptionV3 load is ~98%. A partial load means the "
                    "backbone.N -> torchvision index map (Pass 3) does not match this "
                    "checkpoint's Sequential order, which risks silently loading weights "
                    "into the WRONG Inception block. "
                    f"Observed backbone.N indices: {_bb_indices}. "
                    "Validate them against the torchvision inception_v3 child order "
                    "(Conv2d_1a/2a/2b, [maxpool], Conv2d_3b/4a, [maxpool], "
                    "Mixed_5b/5c/5d/6a/6b/6c/6d/6e, [AuxLogits], Mixed_7a/7b/7c) and "
                    "correct the _imap in Pass 3 before trusting InceptionV3 features."
                )

        # ── BatchNorm statistics ──────────────────────────────────────────────────
        # A missing running_mean/running_var makes eval() fall back to mean 0, variance 1
        # for that layer, which silently corrupts the features.
        _bn_expected_keys = [k for k in current if k.endswith((".running_mean", ".running_var"))]
        _bn_missing_keys  = [k for k in missing  if k.endswith((".running_mean", ".running_var"))]
        if _bn_missing_keys:
            _bn_frac = len(_bn_missing_keys) / max(1, len(_bn_expected_keys))
            if _bn_frac > 0.5:
                raise RuntimeError(
                    f"[RadImageNet][FATAL] {architecture_name}: "
                    f"{len(_bn_missing_keys)}/{len(_bn_expected_keys)} BatchNorm "
                    "running_mean/running_var buffers MISSING from checkpoint. "
                    "In eval() mode every BN layer normalises with mean=0/var=1, "
                    "producing silently invalid features. The checkpoint likely stores "
                    "only learnable parameters (conv weights + BN scale/bias) without "
                    "running statistics. "
                    f"Sample missing BN keys: {_bn_missing_keys[:4]}"
                )
            print(
                f"[RadImageNet][warning] {architecture_name}: "
                f"{len(_bn_missing_keys)}/{len(_bn_expected_keys)} BN running buffers "
                "missing. Those layers use default statistics in eval() mode."
            )
        else:
            print(
                f"[RadImageNet] {architecture_name}: all "
                f"{len(_bn_expected_keys)} BN running buffers loaded OK."
            )
        return net


    # CPU cache of the original pretrained weights. Every fold builds a fresh model from
    # these same starting weights, so no learned weights, BatchNorm statistics or optimiser
    # state from one fold could influence another fold. Each checkpoint is read and remapped only once.
    _BACKBONE_SD_CACHE = {}
    _BACKBONE_META_CACHE = {}


    def _clone_state_dict_to_cpu(module: nn.Module):
        """Create a separate CPU copy of the model weights."""
        return {
            key: value.detach().cpu().clone()
            for key, value in module.state_dict().items()
        }


    def _build_imagenet_architecture(backbone_name: str, pretrained: bool):
        """Build an ImageNet model and optionally load pretrained weights."""
        name = str(backbone_name).lower()

        if name == "imagenet_resnet10t":
            net = timm.create_model(
                "resnet10t",
                pretrained=bool(pretrained),
                num_classes=0,
                global_pool="avg",
            )
            return net, int(net.num_features)

        def _torchvision_model(factory):
            if pretrained:
                try:
                    return factory(weights="DEFAULT")
                except TypeError:
                    return factory(pretrained=True)
            try:
                return factory(weights=None)
            except TypeError:
                return factory(pretrained=False)

        if name == "imagenet_resnet18":
            net = _torchvision_model(tv_models.resnet18)
            dim = int(net.fc.in_features)
            net.fc = nn.Identity()
        elif name == "imagenet_resnet50":
            net = _torchvision_model(tv_models.resnet50)
            dim = int(net.fc.in_features)
            net.fc = nn.Identity()
        elif name == "imagenet_densenet121":
            net = _torchvision_model(tv_models.densenet121)
            dim = int(net.classifier.in_features)
            net.classifier = nn.Identity()
        else:
            raise ValueError(f"Unknown ImageNet backbone: {backbone_name!r}")

        return net, dim


    def build_feature_backbone(backbone_name: str):
        """
        Return a fresh (backbone_nn_module, feature_dim, norm_mode).

        Every call returns a newly constructed model object. The model starts from
        the same pristine pretrained weights held in _BACKBONE_SD_CACHE, so folds
        remain independent without repeating remote downloads or Drive checkpoint
        remapping.
        """
        name = str(backbone_name).lower()

        imagenet_names = {
            "imagenet_resnet10t",
            "imagenet_resnet18",
            "imagenet_resnet50",
            "imagenet_densenet121",
        }

        if name in imagenet_names:
            if name not in _BACKBONE_SD_CACHE:
                # Only the first call touches the network; later ones hit the disk cache.
                net, dim = _build_imagenet_architecture(name, pretrained=True)
                _BACKBONE_SD_CACHE[name] = _clone_state_dict_to_cpu(net)
                _BACKBONE_META_CACHE[name] = {
                    "feature_dim": int(dim),
                    "norm": "imagenet",
                    "source": "ImageNet pretrained",
                }
                print(
                    f"[ImageNet cache primed] {name}: pretrained weights loaded once "
                    f"| feature_dim={dim}"
                )
            else:
                # Architecture only: no download, no network access.
                net, dim = _build_imagenet_architecture(name, pretrained=False)
                net.load_state_dict(_BACKBONE_SD_CACHE[name], strict=True)

            meta = _BACKBONE_META_CACHE[name]
            return net, int(meta["feature_dim"]), str(meta["norm"])

        if name == "radimagenet_resnet10t":
            net, dim = _make_timm_resnet10t_backbone()
            weights_path, label = RADIMAGENET_RESNET10T_WEIGHTS_PATH, "RadImageNet ResNet-10t"
        elif name == "radimagenet_resnet18":
            net, dim = _make_torchvision_resnet18_backbone()
            weights_path, label = RADIMAGENET_RESNET18_WEIGHTS_PATH, "RadImageNet ResNet-18"
        elif name == "radimagenet_resnet50":
            net, dim = _make_torchvision_resnet50_backbone()
            weights_path, label = RADIMAGENET_RESNET50_WEIGHTS_PATH, "RadImageNet ResNet-50"
        elif name == "radimagenet_densenet121":
            net, dim = _make_torchvision_densenet121_backbone()
            weights_path, label = RADIMAGENET_DENSENET121_WEIGHTS_PATH, "RadImageNet DenseNet-121"
        else:
            raise ValueError(f"Unknown backbone: {backbone_name!r}")

        if name not in _BACKBONE_SD_CACHE:
            net = load_radimagenet_weights(net, weights_path, label)
            _BACKBONE_SD_CACHE[name] = _clone_state_dict_to_cpu(net)
            _BACKBONE_META_CACHE[name] = {
                "feature_dim": int(dim),
                "norm": radimagenet_norm_for(name),
                "source": label,
            }
            print(
                f"[RadImageNet cache primed] {name}: checkpoint read/remapped once "
                f"| feature_dim={dim}"
            )
        else:
            net.load_state_dict(_BACKBONE_SD_CACHE[name], strict=True)

        meta = _BACKBONE_META_CACHE[name]
        return net, int(meta["feature_dim"]), str(meta["norm"])


    def prime_backbone_initial_state(backbone_name: str):
        """
        Load/download one backbone before CV begins, then discard the temporary model.

        This makes any network/download problem occur once at a clearly labelled
        preflight step instead of appearing to freeze in the middle of fold 1.
        """
        name = str(backbone_name).lower()
        if name in _BACKBONE_SD_CACHE:
            return

        print(f"\n[backbone setup] Preparing original starting weights for: {name}")
        temporary_model, feature_dim, norm_mode = build_feature_backbone(name)
        del temporary_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(
            f"[backbone setup complete] {name} | feature_dim={feature_dim} "
            f"| norm={norm_mode} | all folds use the CPU weight cache"
        )

    def freeze_backbone(net: nn.Module):
        for p in net.parameters():
            p.requires_grad = False


In [ ]:
# Section 1, cell 10: Multi-slice classifier
if globals().get("RUN_FROZEN_CNN_E2E", True):

    class MultiSliceCNNClassifier(nn.Module):
        """
        Wraps a CNN backbone with a binary classification head.
        Input: (B, S, 3, H, W)  = S slices per patient
        Output: (B,)             = raw logits (use BCEWithLogitsLoss)

        All S slices are pushed through the shared backbone;
        features are mean-pooled across slices before the head.
        """
        def __init__(self, backbone: nn.Module, feature_dim: int, dropout: float = 0.5):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.BatchNorm1d(feature_dim),
                nn.Dropout(dropout),
                nn.Linear(feature_dim, 1),
            )

        def forward(self, x: torch.Tensor, slice_mask: torch.Tensor = None) -> torch.Tensor:
            # x: (B, S, 3, H, W). slice_mask: (B, S) True for real slices, False for padding.
            B, S, C, H, W = x.shape
            # Flatten batch × slice for efficient backbone pass
            feats = self.backbone(x.view(B * S, C, H, W))  # (B*S, D)
            # Reduce spatial feature maps (e.g. DenseNet) to vectors.
            if feats.dim() > 2:
                feats = F.adaptive_avg_pool2d(feats, 1).flatten(1)
            feats = feats.view(B, S, -1)
            # Mean over real slices only, so a patient with three tumour slices weights
            # each of them 1/3 rather than diluting them with padded slots.
            if slice_mask is None:
                feats = feats.mean(dim=1)                     # (B, D)
            else:
                m = slice_mask.to(feats.dtype).unsqueeze(-1)  # (B, S, 1)
                feats = (feats * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
            return self.head(feats).squeeze(-1)              # (B,)


    def build_classifier(backbone_name: str, dropout: float = DROPOUT):
        backbone, feat_dim, norm = build_feature_backbone(backbone_name)
        model = MultiSliceCNNClassifier(backbone, feat_dim, dropout=dropout)
        freeze_backbone(model.backbone)
        print(f"Built classifier | backbone={backbone_name} | feature_dim={feat_dim} | norm={norm}")
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable parameters (head only): {n_trainable:,}")
        return model, feat_dim, norm


In [ ]:
# Section 1, cell 11: Training and evaluation utilities
if globals().get("RUN_FROZEN_CNN_E2E", True):

    def compute_pos_weight(labels: list):
        """BCEWithLogitsLoss pos_weight for class imbalance, controlled by the Cell-2 config.

        Returns None when USE_POS_WEIGHT is False (unweighted loss). Otherwise returns
        (n_neg / n_pos) * POS_WEIGHT_SCALE so the weighting strength can be tuned.
        """
        if not USE_POS_WEIGHT:
            return None
        arr = np.array(labels)
        n_pos = arr.sum()
        n_neg = len(arr) - n_pos
        pw = (float(n_neg) / max(float(n_pos), 1.0)) * float(POS_WEIGHT_SCALE)
        return torch.tensor([pw], device=device)


    def pad_slices_collate(batch):
        """Collate patients whose slice counts differ.

        Each patient contributes (S_i, 3, H, W) with S_i = its number of tumour-bearing
        slices. The batch is padded to max(S_i) by repeating that patient's last real
        slice, and a boolean mask marks which slots are real. Padding with a real slice
        rather than zeros keeps the backbone's BatchNorm inputs in-distribution; the mask
        then excludes those slots from pooling, so they cannot affect the patient feature.
        """
        xs_list, ys = zip(*batch)
        s_max = max(int(x.shape[0]) for x in xs_list)
        B = len(xs_list)
        C_, H_, W_ = xs_list[0].shape[1:]
        out = xs_list[0].new_zeros((B, s_max, C_, H_, W_))
        mask = torch.zeros((B, s_max), dtype=torch.bool)
        for i, x in enumerate(xs_list):
            s = int(x.shape[0])
            out[i, :s] = x
            mask[i, :s] = True
            if s < s_max:
                out[i, s:] = x[-1:].expand(s_max - s, -1, -1, -1)
        return out, mask, torch.stack(ys)


    def make_dataloader(pairs: list, input_size: int, augment: bool, batch_size: int,
                        norm: str = "imagenet") -> DataLoader:
        ds = SliceDataset(pairs, input_size=input_size, augment=augment, norm=norm)
        # drop_last avoids a final training batch of size 1, which the head BatchNorm1d
        # cannot handle; never applied to val/test so every case is scored.
        drop_last = augment and (len(ds) > batch_size)
        # num_workers=0 keeps the slice cache in-process; the dataset is fully cached
        # after epoch 1, so workers would add overhead.
        return DataLoader(
            ds, batch_size=batch_size, shuffle=augment,
            num_workers=0, pin_memory=torch.cuda.is_available(), drop_last=drop_last,
            collate_fn=pad_slices_collate,
        )


    def train_one_epoch(model, loader, optimiser, criterion):
        model.train()
        # Backbone BatchNorm stays in eval mode so the pretrained running statistics are
        # used rather than overwritten by small-batch estimates. The head, including its
        # BatchNorm1d, stays in train mode.
        model.backbone.eval()
        total_loss, all_logits, all_labels = 0.0, [], []
        for xs, slice_mask, ys in loader:
            xs, slice_mask, ys = xs.to(device), slice_mask.to(device), ys.to(device)
            optimiser.zero_grad()
            logits = model(xs, slice_mask)
            loss   = criterion(logits, ys)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            total_loss += loss.item() * len(ys)
            all_logits.extend(logits.detach().cpu().tolist())
            all_labels.extend(ys.cpu().tolist())
        mean_loss = total_loss / max(len(all_labels), 1)
        try:
            auc = roc_auc_score(all_labels, all_logits)
        except Exception:
            auc = float("nan")
        return mean_loss, auc


    # AUC confidence intervals (DeLong, jackknife, bootstrap): calculate_auc_intervals()
    # and its settings AUC_CI_CONFIDENCE and AUC_BOOTSTRAP_REPEATS are defined once in
    # Section 0, cell 3 and shared by all three sections.


    def optimal_threshold_youden(labels, probs):
        """
        Estimate a classification threshold by maximizing Youden's J.

        This helper is used on pooled development OOF predictions. The selected
        threshold is then frozen and applied unchanged to the untouched holdout test.
        """
        labels = np.asarray(labels, dtype=int).ravel()
        probs = np.asarray(probs, dtype=float).ravel()
        valid = np.isfinite(labels) & np.isfinite(probs)
        labels = labels[valid]
        probs = probs[valid]

        if labels.size == 0 or np.unique(labels).size != 2:
            return float(DECISION_THRESHOLD)

        scores = np.unique(probs)
        if scores.size > 1001:
            scores = np.unique(np.quantile(probs, np.linspace(0.0, 1.0, 1001)))

        if scores.size > 1:
            thresholds = (scores[:-1] + scores[1:]) / 2.0
            thresholds = np.r_[scores[0] - 1e-12, thresholds, scores[-1] + 1e-12]
        else:
            thresholds = np.array([scores[0]], dtype=float)

        best_t = float(thresholds[0])
        best_j = -np.inf
        for threshold in thresholds:
            preds = (probs >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
            specificity = tn / (tn + fp) if (tn + fp) else 0.0
            youden = sensitivity + specificity - 1.0
            if youden > best_j:
                best_j = youden
                best_t = float(threshold)

        return best_t


    def compute_binary_metrics(
        labels,
        probs,
        threshold: float = DECISION_THRESHOLD,
        threshold_source: str = "explicit classification threshold",
    ):
        labels = np.array(labels, dtype=int)
        probs = np.array(probs, dtype=float)
        threshold = float(threshold)
        preds = (probs >= threshold).astype(int)

        out = {
            "threshold": float(threshold),
            "threshold_source": str(threshold_source),
        }
        try:
            out["auc"] = roc_auc_score(labels, probs)
        except Exception:
            out["auc"] = float("nan")

        try:
            out["accuracy"] = accuracy_score(labels, preds)
            out["balacc"] = balanced_accuracy_score(labels, preds)
            out["f1"] = f1_score(labels, preds, zero_division=0)
            out["precision"] = precision_score(labels, preds, zero_division=0)
            out["sensitivity"] = recall_score(labels, preds, zero_division=0)
            out["mcc"] = matthews_corrcoef(labels, preds)
            out["brier"] = brier_score_loss(labels, probs)

            cm = confusion_matrix(labels, preds, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            out["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
            out["tn"], out["fp"], out["fn"], out["tp"] = int(tn), int(fp), int(fn), int(tp)
        except Exception:
            for k in [
                "accuracy", "balacc", "f1", "precision", "sensitivity",
                "specificity", "mcc", "brier",
            ]:
                out[k] = float("nan")
            out["tn"] = out["fp"] = out["fn"] = out["tp"] = 0

        return out


    def compute_top_risk_binary_metrics(labels, probs, percentage=TOP_RISK_PERCENT):
        """Calculate classification metrics when the highest-risk percentage is positive."""
        threshold = threshold_for_top_risk(probs, percentage=percentage)
        return compute_binary_metrics(
            labels,
            probs,
            threshold=threshold,
            threshold_source=f"top {float(percentage):g}% risk scores in evaluated set",
        )


    @torch.no_grad()
    def evaluate(model, loader, threshold: float = DECISION_THRESHOLD):
        model.eval()
        all_logits, all_labels = [], []
        for xs, slice_mask, ys in loader:
            xs, slice_mask = xs.to(device), slice_mask.to(device)
            logits = model(xs, slice_mask).cpu()
            all_logits.extend(logits.tolist())
            all_labels.extend(ys.tolist())
        probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
        labels = np.array(all_labels, dtype=int)
        metrics = compute_binary_metrics(
            labels,
            probs,
            threshold=threshold,
            threshold_source=f"fixed CNN threshold {float(threshold):g}",
        )
        metrics.update({"probs": probs, "labels": labels})
        return metrics


    def split_inner_train_val(outer_train_pairs: list, fold_seed: int):
        """Split one outer-training fold into inner training and inner validation patients.

        The split is stratified by label and uses outer-training patients only, so the
        outer validation fold cannot influence the epoch that is selected. Returns
        (outer_train_pairs, []) when the fold is too small to be split.
        """
        y = np.asarray([p["label"] for p in outer_train_pairs], dtype=int)
        if min(int(y.sum()), int((1 - y).sum())) < 2:
            return list(outer_train_pairs), []
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=float(FROZEN_E2E_EPOCH_SELECTION_FRACTION),
            random_state=int(fold_seed),
        )
        try:
            inner_train_idx, inner_val_idx = next(splitter.split(np.arange(len(outer_train_pairs)), y))
        except ValueError:
            return list(outer_train_pairs), []
        inner_train = [outer_train_pairs[i] for i in inner_train_idx]
        inner_val = [outer_train_pairs[i] for i in inner_val_idx]
        return inner_train, inner_val


    def train_classifier(
        model: nn.Module,
        train_pairs: list,
        val_pairs: list,
        backbone_name: str,
        input_size: int = CNN_INPUT_SIZE,
        feat_dim: int = 512,
        norm: str = "imagenet",
    ):
        """
        Epoch-selection run of the frozen-backbone workflow: train the classification head only.

        val_pairs must be an inner validation split of the training patients. It must
        never be the outer fold that is scored afterwards, because the epoch returned
        here is the one with the best AUC on val_pairs. Returns the best validation AUC,
        the best model weights and that epoch.
        """
        train_loader = make_dataloader(train_pairs, input_size, augment=True,  batch_size=BATCH_SIZE, norm=norm)
        val_loader   = make_dataloader(val_pairs,   input_size, augment=False, batch_size=BATCH_SIZE, norm=norm)

        pos_weight = compute_pos_weight([p["label"] for p in train_pairs])
        criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        # -inf, not 0.0: a fold whose validation AUC is nan or exactly 0.0 must still be
        # able to set best_state, otherwise the model silently keeps its last-epoch weights.
        # np.isfinite() additionally prevents nan from winning.
        best_auc, patience_ctr = -np.inf, 0
        best_state = None
        best_epoch = None

        # ── Frozen backbone: head only ──────────────────────────────────────────
        opt_p1 = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=WEIGHT_DECAY
        )
        sch_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_p1, T_max=EPOCHS_HEAD)

        print(f"  Epoch selection on the inner split: frozen backbone, head only "
              f"(maximum {EPOCHS_HEAD} epochs)")
        for epoch in range(1, EPOCHS_HEAD + 1):
            tr_loss, tr_auc = train_one_epoch(model, train_loader, opt_p1, criterion)
            val_metrics     = evaluate(model, val_loader)
            sch_p1.step()

            val_auc = val_metrics["auc"]
            if np.isfinite(val_auc) and val_auc > best_auc:
                best_auc   = val_auc
                best_state = deepcopy(model.state_dict())
                best_epoch = int(epoch)
                patience_ctr = 0
            else:
                patience_ctr += 1

            print(f"    ep{epoch:02d}/{EPOCHS_HEAD} | loss={tr_loss:.4f}  tr_auc={tr_auc:.3f}  "
                  f"inner_val_auc={val_auc:.3f}  inner_val_balacc={val_metrics['balacc']:.3f}")

            if patience_ctr >= EARLY_STOP_PATIENCE:
                print(f"    Early stop (phase 1) at epoch {epoch}")
                break

        # Load the best weights.
        if best_state is not None:
            model.load_state_dict(best_state)
        else:
            # No epoch produced a usable validation AUC; report that rather than
            # returning the sentinel as a score.
            print("    [warning] no finite validation AUC in any epoch; "
                  "keeping last-epoch weights and reporting AUC=nan")
            best_auc = float("nan")
        return best_auc, best_state, best_epoch


    def train_classifier_fixed_budget(
        model: nn.Module,
        train_pairs: list,
        backbone_name: str,
        input_size: int = CNN_INPUT_SIZE,
        norm: str = "imagenet",
        head_epochs: int = EPOCHS_HEAD,
        run_label: str = "Final retrain: full development set",
    ):
        """
        Train for a fixed number of epochs. No validation data are used here.

        The same function trains two kinds of model. Each outer-fold model is trained
        on all patients of its outer-training fold for the number of epochs selected on
        that fold's inner split. The final refit is trained on the full development set
        for the median of those numbers, before untouched-holdout evaluation.
        """
        train_loader = make_dataloader(train_pairs, input_size, augment=True, batch_size=BATCH_SIZE, norm=norm)
        pos_weight = compute_pos_weight([p["label"] for p in train_pairs])
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        history = []

        # Frozen backbone, head only. The epoch budget is passed in and never chosen here.
        head_epochs = int(np.clip(int(head_epochs), 1, EPOCHS_HEAD))
        opt_p1 = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=WEIGHT_DECAY
        )
        sch_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_p1, T_max=head_epochs)

        print(
            f"  {run_label}, frozen backbone / head only "
            f"(epochs fixed beforehand: {head_epochs})"
        )
        for epoch in range(1, head_epochs + 1):
            tr_loss, tr_auc = train_one_epoch(model, train_loader, opt_p1, criterion)
            sch_p1.step()
            history.append({
                "phase": "head_only",
                "epoch": epoch,
                "train_loss": tr_loss,
                "train_auc": tr_auc,
                "cv_selected_refit_epochs": head_epochs,
            })
            print(f"    ep{epoch:02d}/{head_epochs} | loss={tr_loss:.4f}  tr_auc={tr_auc:.3f}")

        return history


In [ ]:
# Section 1, cell 12: Cross-validation and holdout test
if globals().get("RUN_FROZEN_CNN_E2E", True):

    def safe_token(s: str) -> str:
        return re.sub(r"[^\w]+", "_", str(s)).strip("_")


    def split_dev_test_by_patient_id(labelled_pairs: list, split_key, context):
        """Apply the shared folder-based patient division (DEVELOPMENT / TEST folders)."""
        pair_by_id = {}
        for item in labelled_pairs:
            cid = canonical_patient_id(item.get("canonical_id", item.get("patient_id")))
            if not cid:
                raise ValueError(f"{context}: blank patient ID in labelled pairs.")
            if cid in pair_by_id:
                raise ValueError(f"{context}: duplicate patient ID in labelled pairs: {cid}")
            pair_by_id[cid] = {**item, "canonical_id": cid}

        dev_ids, test_ids = get_or_create_patient_split(
            split_key=split_key,
            patient_ids=pair_by_id.keys(),
            context=context,
        )
        return [pair_by_id[cid] for cid in dev_ids], [pair_by_id[cid] for cid in test_ids]


    def mean_std(values):
        values = np.array(values, dtype=float)
        return float(np.nanmean(values)), float(np.nanstd(values))


    def fmt_metric(x):
        return "nan" if pd.isna(x) else f"{x:.3f}"


    def print_metrics(prefix: str, metrics: dict):
        print(
            f"{prefix}: "
            f"AUC={fmt_metric(metrics.get('auc'))} | "
            f"Acc={fmt_metric(metrics.get('accuracy'))} | "
            f"BalAcc={fmt_metric(metrics.get('balacc'))} | "
            f"F1={fmt_metric(metrics.get('f1'))} | "
            f"Sens={fmt_metric(metrics.get('sensitivity'))} | "
            f"Spec={fmt_metric(metrics.get('specificity'))} | "
            f"MCC={fmt_metric(metrics.get('mcc'))} | "
            f"Brier={fmt_metric(metrics.get('brier'))}"
        )


    def print_cv_threshold_line(cv_summary: dict, prefix: str, cutoff_label: str):
        """Print one line of threshold-dependent CV fold metrics, naming its cut-off.

        Every threshold-dependent CV number must say which cut-off produced it, so
        the fixed-0.5 fold metrics can never again be read as if they came from the
        tuned out-of-fold threshold.
        """
        def g(metric):
            return cv_summary.get(f"{prefix}_mean_{metric}", float("nan"))

        def s(metric):
            return cv_summary.get(f"{prefix}_std_{metric}", float("nan"))

        print(
            f"  Mean over folds @ {cutoff_label}: "
            f"Acc={fmt_metric(g('accuracy'))} ± {fmt_metric(s('accuracy'))} | "
            f"BalAcc={fmt_metric(g('balacc'))} ± {fmt_metric(s('balacc'))} | "
            f"F1={fmt_metric(g('f1'))} ± {fmt_metric(s('f1'))} | "
            f"Sens={fmt_metric(g('sensitivity'))} ± {fmt_metric(s('sensitivity'))} | "
            f"Spec={fmt_metric(g('specificity'))} ± {fmt_metric(s('specificity'))} | "
            f"MCC={fmt_metric(g('mcc'))} ± {fmt_metric(s('mcc'))}"
        )


    def print_final_retrain_summary(rows: list, title: str):
        """Tabulate, per backbone x outcome: the development-CV AUC (mean ± std across
        all n_folds x n_repeats folds) next to the final-retrain holdout-test metrics."""
        if not rows:
            print(f"\n{title}: (no completed runs)")
            return
        df = order_by_test_set(pd.DataFrame(rows)).reset_index(drop=True)
        # An OPTIMIZATION run that keeps the holdout unseen has development numbers only.
        # Every run is then shown once and not once per test set.
        holdout_scored = holdout_evaluation_enabled()
        if not holdout_scored:
            title = f"{title} | holdout not scored in this run, development CV only"
            df = rows_for_display(df).reset_index(drop=True)

        # CV AUC as mean ± SD per backbone and outcome.
        df["CV AUC (mean±sd)"] = [
            ("nan" if pd.isna(m) else f"{m:.3f} ± {(0.0 if pd.isna(s) else s):.3f}")
            for m, s in zip(df["cv_mean_auc"], df["cv_std_auc"])
        ]

        cols = (["backbone"]
                + (["e2e_mode"] if "e2e_mode" in df.columns else [])
                + ["outcome"]
                + (["test_set"] if "test_set" in df.columns else [])
                + ["CV AUC (mean±sd)"]
                + (["final_test_auc", "final_test_accuracy", "final_test_balacc",
                    "final_test_f1", "final_test_sensitivity", "final_test_specificity",
                    "final_test_mcc", "final_test_brier"] if holdout_scored else []))
        disp = df[cols].rename(columns={
            "final_test_auc": "Test_AUC", "final_test_accuracy": "Acc",
            "final_test_balacc": "BalAcc", "final_test_f1": "F1",
            "final_test_sensitivity": "Sens", "final_test_specificity": "Spec",
            "final_test_mcc": "MCC", "final_test_brier": "Brier",
        })
        print("\n" + "=" * 100)
        print(title)
        print("-" * 100)
        print(disp.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        # CV AUC mean is a formatted string column; holdout means stay numeric.
        # Means are taken within one test set; test sets are never averaged together.
        _sets = list(dict.fromkeys(disp["test_set"])) if "test_set" in disp.columns else [None]
        for _set in _sets:
            _rows = disp if _set is None else disp[disp["test_set"] == _set]
            _num = _rows.select_dtypes(include="number")
            if len(_rows) > 1 and not _num.empty:
                means = _num.mean()
                cv_mean_over_runs = float(np.nanmean(df.loc[_rows.index, "cv_mean_auc"].to_numpy(dtype=float)))
                print("-" * 100)
                print(
                    "Mean over runs"
                    + (f" [test set {_set}]" if len(_sets) > 1 else "")
                    + f": CV_AUC={cv_mean_over_runs:.3f} | "
                    + " | ".join(f"{c}={means[c]:.3f}" for c in _num.columns)
                )
        print("=" * 100)


    all_results = []   # one row per arm x backbone x outcome when OPTIMIZATION is on

    print("\n" + "#" * 78)
    print("# SECTION 1: FROZEN-BACKBONE END-TO-END CNN")
    print("#" * 78)

    for _opt_arm, e2e_mode in PART1_OPTIMIZATION_ARM_MODE_PAIRS:
        apply_part1_optimization_arm(_opt_arm)
        print("\n" + "#" * 78)
        print(f"# END-TO-END MODE: {e2e_mode}")
        print("#" * 78)
        for backbone_name in RUN_BACKBONES:
            # Backbone loaded once; each fold still gets its own model instance.
            try:
                prime_backbone_initial_state(backbone_name)
            except Exception as _preflight_error:
                print(
                    f"\n[backbone skipped after setup error] {backbone_name}: "
                    f"{repr(_preflight_error)}"
                )
                continue

            for lcfg in ACTIVE_LABEL_CONFIGS:
                ltag = lcfg["tag"]
                label_df = all_labels_dfs.get(ltag)
                if label_df is None:
                    print(f"[skip] no labels for {ltag}")
                    continue

                print(f"\n{'='*78}")
                print(f"Backbone: {backbone_name} | Mode: {e2e_mode} | Outcome: {ltag} | Policy: {CHANNEL_POLICY}")
                print("Split: DEVELOPMENT folder = development/CV; each TEST folder = one holdout test set")

                labelled_pairs = attach_labels(raw_pairs, label_df)
                if len(labelled_pairs) < 4:
                    print(f"  [skip] too few labelled cases: {len(labelled_pairs)}")
                    continue

                try:
                    dev_pairs, all_test_pairs = split_dev_test_by_patient_id(
                        labelled_pairs,
                        split_key=patient_split_record_key(ltag),
                        context=f"Section 1 | outcome={ltag}",
                    )
                    # One entry per holdout test set (one per TEST_ROOT folder). Checked
                    # here, before any training, that every test set can be scored.
                    _holdout_sets = holdout_test_sets(
                        [p["canonical_id"] for p in all_test_pairs],
                        labels=[p["label"] for p in all_test_pairs],
                        context=f"Section 1 | outcome={ltag}",
                    )
                except Exception as e:
                    print(f"  [skip] could not create development/test split: {e}")
                    continue

                print(
                    f"  Total labelled={len(labelled_pairs)} | development={len(dev_pairs)} "
                    f"(pos={sum(p['label'] for p in dev_pairs)}) | holdout test={len(all_test_pairs)} "
                    f"(pos={sum(p['label'] for p in all_test_pairs)})"
                )
                if len(_holdout_sets) > 1:
                    for _test_set_name, _test_positions in _holdout_sets:
                        print(
                            f"    holdout test set '{_test_set_name}': {len(_test_positions)} "
                            f"(pos={sum(all_test_pairs[_i]['label'] for _i in _test_positions)})"
                        )
                print_patient_split(
                    section=f"Section 1 | backbone={backbone_name} | mode={e2e_mode} | outcome={ltag}",
                    dev_ids=[p["canonical_id"] for p in dev_pairs],
                    test_ids=[p["canonical_id"] for p in all_test_pairs],
                )

                # ── Cross-validation inside development set only ────────────────────
                indices = list(range(len(dev_pairs)))
                y_dev = np.array([p["label"] for p in dev_pairs])
                n_folds = min(N_FOLDS, int(y_dev.sum()), int((1 - y_dev).sum()))
                if n_folds < 2:
                    print(f"  [skip] not enough minority samples for {n_folds}-fold CV inside development set")
                    continue

                rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=N_REPEATS, random_state=GLOBAL_SEED)

                fold_metrics = []
                fold_best_epochs = []
                # Average the repeats per patient: each patient is validated N_REPEATS
                # times, so pooling raw predictions would double-count them.
                oof_prob_accum, oof_label_by_pid = {}, {}

                for fold_idx, (train_idx, val_idx) in enumerate(rskf.split(indices, y_dev)):
                    print(f"\n  CV fold {fold_idx + 1}/{n_folds * N_REPEATS}")
                    train_pairs_fold = [dev_pairs[i] for i in train_idx]
                    val_pairs_fold   = [dev_pairs[i] for i in val_idx]
                    print(f"    train={len(train_pairs_fold)} (pos={sum(p['label'] for p in train_pairs_fold)})  "
                          f"val={len(val_pairs_fold)} (pos={sum(p['label'] for p in val_pairs_fold)})")

                    fold_seed = GLOBAL_SEED + fold_idx

                    # ── Step 1: choose the epoch count inside the outer-training fold ──
                    # The outer validation fold takes no part in this choice. A stratified
                    # inner split of the outer-training patients supplies the validation
                    # patients on which the best epoch is found.
                    inner_train_pairs, inner_val_pairs = split_inner_train_val(train_pairs_fold, fold_seed)
                    best_epoch = None
                    if inner_val_pairs:
                        print(f"    inner split for epoch selection: "
                              f"train={len(inner_train_pairs)} (pos={sum(p['label'] for p in inner_train_pairs)})  "
                              f"val={len(inner_val_pairs)} (pos={sum(p['label'] for p in inner_val_pairs)})")
                        seed_everything(fold_seed)
                        selection_model, feat_dim, norm_mode = build_classifier(backbone_name, dropout=DROPOUT)
                        selection_model = selection_model.to(device)
                        best_inner_auc, _, best_epoch = train_classifier(
                            selection_model, inner_train_pairs, inner_val_pairs,
                            backbone_name=backbone_name,
                            input_size=CNN_INPUT_SIZE,
                            feat_dim=feat_dim,
                            norm=norm_mode,
                        )
                        del selection_model
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                    if best_epoch is not None:
                        fold_epochs = int(best_epoch)
                        fold_best_epochs.append(fold_epochs)
                    else:
                        fold_epochs = int(EPOCHS_HEAD)
                        print("    [warning] no epoch could be selected on an inner split of this fold. "
                              f"It is trained for EPOCHS_HEAD={EPOCHS_HEAD} epochs and "
                              "left out of the final-refit epoch median.")

                    # ── Step 2: train the fold model on all outer-training patients ──
                    # Fresh starting weights and the epoch count from step 1. No validation
                    # data are used, exactly as in the final refit.
                    seed_everything(fold_seed + 50_000)
                    model, feat_dim, norm_mode = build_classifier(backbone_name, dropout=DROPOUT)
                    model = model.to(device)
                    train_classifier_fixed_budget(
                        model,
                        train_pairs_fold,
                        backbone_name=backbone_name,
                        input_size=CNN_INPUT_SIZE,
                        norm=norm_mode,
                        head_epochs=fold_epochs,
                        run_label="Outer-fold model: all outer-training patients",
                    )

                    # ── Step 3: score the outer validation fold, once ──
                    val_loader = make_dataloader(val_pairs_fold, CNN_INPUT_SIZE, augment=False, batch_size=BATCH_SIZE, norm=norm_mode)
                    val_metrics = evaluate(model, val_loader)
                    fold_metrics.append(val_metrics)
                    # val_loader preserves patient order.
                    for _p, _prob in zip(val_pairs_fold, val_metrics["probs"].tolist()):
                        _pid = str(_p["canonical_id"])
                        oof_prob_accum.setdefault(_pid, []).append(float(_prob))
                        oof_label_by_pid[_pid] = int(_p["label"])

                    print_metrics("    CV validation (never used for selection)", val_metrics)

                    del model
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                # ── CV-selected epoch budget for the final development-set refit ──
                # Each fold contributes the epoch that was best on its inner validation split.
                # The median is robust to unusually early/late folds. With an even number
                # of folds, .5 medians are rounded half-up, matching the fine-tuning section.
                if fold_best_epochs:
                    cv_best_epoch_median_raw = float(np.median(np.asarray(fold_best_epochs, dtype=float)))
                    selected_refit_epochs = int(np.clip(
                        np.floor(cv_best_epoch_median_raw + 0.5), 1, EPOCHS_HEAD
                    ))
                else:
                    # Defensive fallback only. Normally every fold selects an epoch on its inner split.
                    cv_best_epoch_median_raw = float(EPOCHS_HEAD)
                    selected_refit_epochs = int(EPOCHS_HEAD)
                    print(
                        "  [warning] no fold selected an epoch on its inner split. "
                        f"Falling back to EPOCHS_HEAD={EPOCHS_HEAD}."
                    )

                print(
                    f"  Epochs selected on the inner splits: {fold_best_epochs} | "
                    f"median={cv_best_epoch_median_raw:.2f} -> "
                    f"final refit={selected_refit_epochs} epochs"
                )

                _oof_pids       = list(oof_label_by_pid.keys())
                fold_labels_all = [oof_label_by_pid[_pid] for _pid in _oof_pids]
                fold_probs_all  = [float(np.mean(oof_prob_accum[_pid])) for _pid in _oof_pids]

                # Youden's J from pooled development OOF predictions only, then frozen for
                # the holdout. Computed before the fold summary, which re-scores at it.
                oof_youden_threshold = optimal_threshold_youden(fold_labels_all, fold_probs_all)
                oof_metrics = compute_binary_metrics(
                    fold_labels_all,
                    fold_probs_all,
                    threshold=oof_youden_threshold,
                    threshold_source="development pooled OOF Youden threshold",
                )

                # Secondary top-risk view.
                oof_toprisk_metrics = None
                if top_risk_mode_enabled():
                    oof_toprisk_metrics = compute_top_risk_binary_metrics(
                        fold_labels_all,
                        fold_probs_all,
                        percentage=TOP_RISK_PERCENT,
                    )

                # ── Per-fold CV summary; every column names its cut-off ────────────
                #   cv_mean_* / cv_std_*   AUC and Brier, no cut-off involved
                #   cv_fixed05_*           the fixed 0.5 cut-off used by evaluate()
                #   cv_oofthr_*            the frozen development OOF-Youden cut-off,
                #                          the same one applied to the holdout
                #   cv_toprisk_*           each fold's own top-TOP_RISK_PERCENT cut-off
                # Re-scoring is free: evaluate() already returned per-fold probabilities.
                _CV_THRESHOLD_FREE_KEYS = ["auc", "brier"]
                _CV_THRESHOLD_DEPENDENT_KEYS = [
                    "accuracy", "balacc", "f1", "precision",
                    "sensitivity", "specificity", "mcc",
                ]

                cv_summary = {}
                for key in _CV_THRESHOLD_FREE_KEYS:
                    cv_summary[f"cv_mean_{key}"], cv_summary[f"cv_std_{key}"] = mean_std(
                        [m[key] for m in fold_metrics]
                    )

                _fold_at_oofthr = [
                    compute_binary_metrics(
                        m["labels"], m["probs"],
                        threshold=oof_youden_threshold,
                        threshold_source="development pooled OOF Youden threshold",
                    )
                    for m in fold_metrics
                ]
                _fold_at_toprisk = None
                if top_risk_mode_enabled():
                    _fold_at_toprisk = [
                        compute_top_risk_binary_metrics(
                            m["labels"], m["probs"], percentage=TOP_RISK_PERCENT
                        )
                        for m in fold_metrics
                    ]

                for key in _CV_THRESHOLD_DEPENDENT_KEYS:
                    cv_summary[f"cv_fixed05_mean_{key}"], cv_summary[f"cv_fixed05_std_{key}"] = (
                        mean_std([m[key] for m in fold_metrics])
                    )
                    cv_summary[f"cv_oofthr_mean_{key}"], cv_summary[f"cv_oofthr_std_{key}"] = (
                        mean_std([m[key] for m in _fold_at_oofthr])
                    )
                    if _fold_at_toprisk is not None:
                        cv_summary[f"cv_toprisk_mean_{key}"], cv_summary[f"cv_toprisk_std_{key}"] = (
                            mean_std([m[key] for m in _fold_at_toprisk])
                        )
                    else:
                        cv_summary[f"cv_toprisk_mean_{key}"] = np.nan
                        cv_summary[f"cv_toprisk_std_{key}"] = np.nan

                # Provenance columns for the exported CSV.
                cv_summary["cv_threshold_free_metrics"] = "auc, brier"
                cv_summary["cv_fixed05_threshold"] = float(DECISION_THRESHOLD)
                cv_summary["cv_fixed05_threshold_source"] = (
                    f"fixed CNN threshold {float(DECISION_THRESHOLD):g} (never tuned)"
                )
                cv_summary["cv_oofthr_threshold"] = float(oof_youden_threshold)
                cv_summary["cv_oofthr_threshold_source"] = (
                    "development pooled OOF Youden threshold "
                    "(the same cut-off frozen and applied to the holdout)"
                )
                cv_summary["cv_toprisk_percent"] = (
                    float(TOP_RISK_PERCENT) if _fold_at_toprisk is not None else np.nan
                )
                cv_summary["cv_toprisk_threshold_source"] = (
                    f"top {float(TOP_RISK_PERCENT):g}% risk scores within each fold"
                    if _fold_at_toprisk is not None else ""
                )

                # Epoch-selection provenance for the final refit.
                cv_summary["cv_fold_best_epochs"] = ",".join(str(int(x)) for x in fold_best_epochs)
                cv_summary["cv_best_epoch_median_raw"] = float(cv_best_epoch_median_raw)
                cv_summary["final_refit_head_epochs"] = int(selected_refit_epochs)
                cv_summary["final_refit_epoch_source"] = (
                    "median of the epochs selected on the inner validation splits of the development CV folds"
                )
                cv_summary["cv_epoch_selection"] = (
                    "epoch selected on an inner validation split of each outer-training fold, outer fold scored once"
                )
                cv_summary["cv_inner_val_fraction"] = float(FROZEN_E2E_EPOCH_SELECTION_FRACTION)

                print(f"\n  ── CV summary inside development set [{backbone_name} | {ltag}] ──")
                print(
                    f"  Mean over folds, cut-off independent: "
                    f"AUC={fmt_metric(cv_summary['cv_mean_auc'])} ± {fmt_metric(cv_summary['cv_std_auc'])} | "
                    f"Brier={fmt_metric(cv_summary['cv_mean_brier'])} ± {fmt_metric(cv_summary['cv_std_brier'])}"
                )
                print_cv_threshold_line(
                    cv_summary, "cv_oofthr",
                    f"development OOF-Youden cut-off {float(oof_youden_threshold):.4f}",
                )
                print_cv_threshold_line(
                    cv_summary, "cv_fixed05",
                    f"fixed {float(DECISION_THRESHOLD):g} cut-off (untuned)",
                )
                if _fold_at_toprisk is not None:
                    print_cv_threshold_line(
                        cv_summary, "cv_toprisk",
                        f"top {float(TOP_RISK_PERCENT):g}% risk within each fold",
                    )
                print_metrics("  Pooled OOF @ OOF-Youden threshold", oof_metrics)
                if oof_toprisk_metrics is not None:
                    print_metrics(
                        f"  Pooled OOF @ top {float(TOP_RISK_PERCENT):g}% risk",
                        oof_toprisk_metrics,
                    )

                # An OPTIMIZATION run with OPTIMIZATION_EVALUATE_HOLDOUT set to False keeps every
                # test set unseen: no final refit, no holdout scoring and no saved model.
                _holdout_scored = holdout_evaluation_enabled()
                if _holdout_scored:
                    # ── Final retrain on all development patients, evaluate holdout test ─
                    print(f"\n  ── Final retrain on all development patients, then untouched holdout test [{backbone_name} | {ltag}] ──")
                    seed_everything(GLOBAL_SEED + 10_000)
                    final_model, feat_dim, norm_mode = build_classifier(backbone_name, dropout=DROPOUT)
                    final_model = final_model.to(device)

                    final_history = train_classifier_fixed_budget(
                        final_model,
                        dev_pairs,
                        backbone_name=backbone_name,
                        input_size=CNN_INPUT_SIZE,
                        norm=norm_mode,
                        head_epochs=selected_refit_epochs,
                    )
                else:
                    print("\n  Final refit and holdout test: not run for this arm "
                          "(OPTIMIZATION_EVALUATE_HOLDOUT is False).")
                    final_model = None

                # Write per-backbone results as they complete.
                _arm_prefix = (safe_token(_opt_arm["arm_id"]) + "__") if _opt_arm is not None else ""
                out_prefix = (f"{_arm_prefix}{safe_token(backbone_name)}_{safe_token(ltag)}_"
                              f"{CHANNEL_POLICY}_{safe_token(e2e_mode)}")
                if _holdout_scored:
                    pd.DataFrame(final_history).to_csv(Path(OUT_DIR) / f"{out_prefix}_final_retrain_history.csv", index=False)
                    torch.save(final_model.state_dict(), Path(OUT_DIR) / f"{out_prefix}_final_retrain_model.pt")

                # The model refitted above is scored on each holdout test set separately:
                # one pass, one prediction file and one result row per test set. With a
                # single test set this loop runs once. When the holdout is not scored, the
                # loop writes the same rows with empty holdout values.
                _test_results_by_set = {}
                for _test_set_name, _test_positions in _holdout_sets:
                    test_pairs = [all_test_pairs[_i] for _i in _test_positions]
                    # Scoring an additional test set must not move the random-number generators,
                    # so later results are exactly those of a run with one test set.
                    _rng_state = rng_state_snapshot() if _test_set_name != _holdout_sets[0][0] else None
                    if _holdout_scored:
                        if len(_holdout_sets) > 1:
                            print(f"\n  ── Holdout test set '{_test_set_name}' ({len(test_pairs)} patients) ──")
                        test_loader = make_dataloader(test_pairs, CNN_INPUT_SIZE, augment=False, batch_size=BATCH_SIZE, norm=norm_mode)
                        test_scored = evaluate(final_model, test_loader)

                        # Threshold comes from pooled development OOF predictions only; holdout
                        # labels never inform it.
                        test_metrics = compute_binary_metrics(
                            test_scored["labels"],
                            test_scored["probs"],
                            threshold=oof_youden_threshold,
                            threshold_source="development pooled OOF Youden threshold applied unchanged to holdout",
                        )
                        test_metrics.update({
                            "labels": test_scored["labels"],
                            "probs": test_scored["probs"],
                        })

                        # Secondary top-risk view on the same scores.
                        test_toprisk_metrics = None
                        if top_risk_mode_enabled():
                            test_toprisk_metrics = compute_top_risk_binary_metrics(
                                test_scored["labels"],
                                test_scored["probs"],
                                percentage=TOP_RISK_PERCENT,
                            )

                        test_auc_intervals = calculate_auc_intervals(
                            test_metrics["labels"],
                            test_metrics["probs"],
                        )
                        test_metrics.update(test_auc_intervals)

                        print_metrics("  HOLDOUT TEST @ development OOF-Youden threshold", test_metrics)
                        if test_toprisk_metrics is not None:
                            print_metrics(
                                f"  HOLDOUT TEST @ top {float(TOP_RISK_PERCENT):g}% risk",
                                test_toprisk_metrics,
                            )
                        print(f"  DeLong AUC with 95% CI: {test_metrics['AUC_with_DeLong_95CI']}")
                        print(f"  Jackknife AUC with 95% CI: {test_metrics['AUC_with_Jackknife_95CI']}")
                        print(f"  Bootstrap AUC with 95% CI: {test_metrics['AUC_with_Bootstrap_95CI']}")
                        print(
                            f"  Jackknife leave-one-out AUC range: "
                            f"{test_metrics['Jackknife_min_AUC']:.3f}–"
                            f"{test_metrics['Jackknife_max_AUC']:.3f}"
                        )

                        _test_pred_export = {
                            "true_label": test_metrics["labels"],
                            "probability": test_metrics["probs"],
                            "predicted_label_oof_youden": (
                                test_metrics["probs"] >= test_metrics["threshold"]
                            ).astype(int),
                            "classification_threshold_oof_youden": test_metrics["threshold"],
                            "threshold_source_oof_youden": test_metrics["threshold_source"],
                        }
                        if test_toprisk_metrics is not None:
                            _test_pred_export.update({
                                "predicted_label_top_risk": (
                                    test_metrics["probs"] >= test_toprisk_metrics["threshold"]
                                ).astype(int),
                                "classification_threshold_top_risk": test_toprisk_metrics["threshold"],
                                "threshold_source_top_risk": test_toprisk_metrics["threshold_source"],
                            })
                        pd.DataFrame(_test_pred_export).to_csv(
                            Path(OUT_DIR) / f"{out_prefix}_holdout_test_predictions{test_set_file_tag(_test_set_name)}.csv",
                            index=False,
                        )
                    else:
                        # The row of this test set is still written. Its holdout values stay empty.
                        test_metrics = HoldoutNotEvaluated(threshold_source=HOLDOUT_NOT_EVALUATED_TEXT)
                        test_toprisk_metrics = None

                    row = {
                        "backbone": backbone_name,
                        "e2e_mode": e2e_mode,
                        "outcome": ltag,
                        "test_set": _test_set_name,
                        "channel_policy": CHANNEL_POLICY,
                        "optimization": bool(OPTIMIZATION),
                        "opt_arm_id": (_opt_arm["arm_id"] if _opt_arm is not None else ""),
                        "opt_is_reference": bool(
                            optimization_arm_is_reference("frozen", _opt_arm)
                        ),
                        "opt_input_size": int(CNN_INPUT_SIZE),
                        "opt_n_slices": int(N_SLICES),
                        "opt_augmentation": str(FROZEN_E2E_AUGMENTATION_LEVEL).upper(),
                        "tumor_context_mode": str(TUMOR_CONTEXT_MODE).upper(),
                        "roi_only_background_value": (
                            float(ROI_ONLY_BACKGROUND_VALUE)
                            if str(TUMOR_CONTEXT_MODE).upper() == "ROI_ONLY" else np.nan
                        ),
                        "n_total_cases": len(labelled_pairs),
                        "n_dev_cases": len(dev_pairs),
                        "n_test_cases": len(test_pairs),
                        "n_folds": n_folds,
                        "n_repeats": N_REPEATS,
                        **cv_summary,

                        # Development OOF metrics at the pooled OOF Youden cut-off.
                        "oof_auc": oof_metrics["auc"],
                        "oof_accuracy": oof_metrics["accuracy"],
                        "oof_balacc": oof_metrics["balacc"],
                        "oof_f1": oof_metrics["f1"],
                        "oof_sensitivity": oof_metrics["sensitivity"],
                        "oof_specificity": oof_metrics["specificity"],
                        "oof_mcc": oof_metrics["mcc"],
                        "oof_brier": oof_metrics["brier"],
                        "oof_youden_threshold": float(oof_youden_threshold),

                        # Development OOF, top-risk cut-off.
                        "oof_toprisk_percent": float(TOP_RISK_PERCENT) if oof_toprisk_metrics is not None else np.nan,
                        "oof_toprisk_threshold": (
                            oof_toprisk_metrics["threshold"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_accuracy": (
                            oof_toprisk_metrics["accuracy"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_balacc": (
                            oof_toprisk_metrics["balacc"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_f1": (
                            oof_toprisk_metrics["f1"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_sensitivity": (
                            oof_toprisk_metrics["sensitivity"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_specificity": (
                            oof_toprisk_metrics["specificity"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_mcc": (
                            oof_toprisk_metrics["mcc"] if oof_toprisk_metrics is not None else np.nan
                        ),

                        # Holdout AUC and intervals (threshold-free).
                        "final_test_auc": test_metrics["auc"],
                        "final_test_DeLong_CI_lower": test_metrics["DeLong_CI_lower"],
                        "final_test_DeLong_CI_upper": test_metrics["DeLong_CI_upper"],
                        "final_test_AUC_with_DeLong_95CI": test_metrics["AUC_with_DeLong_95CI"],
                        "final_test_Jackknife_CI_lower": test_metrics["Jackknife_CI_lower"],
                        "final_test_Jackknife_CI_upper": test_metrics["Jackknife_CI_upper"],
                        "final_test_AUC_with_Jackknife_95CI": test_metrics["AUC_with_Jackknife_95CI"],
                        "final_test_Jackknife_min_AUC": test_metrics["Jackknife_min_AUC"],
                        "final_test_Jackknife_max_AUC": test_metrics["Jackknife_max_AUC"],
                        "final_test_Bootstrap_CI_lower": test_metrics["Bootstrap_CI_lower"],
                        "final_test_Bootstrap_CI_upper": test_metrics["Bootstrap_CI_upper"],
                        "final_test_AUC_with_Bootstrap_95CI": test_metrics["AUC_with_Bootstrap_95CI"],
                        "final_test_Bootstrap_repeats": test_metrics["Bootstrap_repeats"],

                        # Holdout metrics at the OOF-derived Youden cut-off.
                        "final_test_accuracy": test_metrics["accuracy"],
                        "final_test_balacc": test_metrics["balacc"],
                        "final_test_f1": test_metrics["f1"],
                        "final_test_precision": test_metrics["precision"],
                        "final_test_sensitivity": test_metrics["sensitivity"],
                        "final_test_specificity": test_metrics["specificity"],
                        "final_test_mcc": test_metrics["mcc"],
                        "final_test_brier": test_metrics["brier"],
                        "final_test_threshold": test_metrics["threshold"],
                        "final_test_threshold_source": test_metrics["threshold_source"],

                        # Holdout metrics, top-risk cut-off.
                        "final_test_toprisk_percent": float(TOP_RISK_PERCENT) if test_toprisk_metrics is not None else np.nan,
                        "final_test_toprisk_threshold": (
                            test_toprisk_metrics["threshold"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_accuracy": (
                            test_toprisk_metrics["accuracy"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_balacc": (
                            test_toprisk_metrics["balacc"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_f1": (
                            test_toprisk_metrics["f1"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_precision": (
                            test_toprisk_metrics["precision"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_sensitivity": (
                            test_toprisk_metrics["sensitivity"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_specificity": (
                            test_toprisk_metrics["specificity"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_mcc": (
                            test_toprisk_metrics["mcc"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_brier": (
                            test_toprisk_metrics["brier"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_threshold_source": (
                            test_toprisk_metrics["threshold_source"] if test_toprisk_metrics is not None else ""
                        ),
                    }

                    _test_results_by_set[_test_set_name] = {
                        _k: _v for _k, _v in row.items() if str(_k).startswith("final_test_")
                    }

                    if OPTIMIZATION and _opt_arm is not None:
                        OPTIMIZATION_FROZEN_OOF_STORE[(_opt_arm["arm_id"], backbone_name, ltag)] = {
                            "probs": {str(_pid): float(_p) for _pid, _p in zip(_oof_pids, fold_probs_all)},
                            "labels": {str(_pid): int(_l) for _pid, _l in zip(_oof_pids, fold_labels_all)},
                            "oof_auc": float(oof_metrics["auc"]),
                            "is_reference": bool(
                                optimization_arm_is_reference("frozen", _opt_arm)
                            ),
                            "input_size": int(_opt_arm["input_size"]),
                            "n_slices": int(_opt_arm["n_slices"]),
                            "augmentation": str(_opt_arm["augmentation"]),
                            "context_mode": str(_opt_arm["context_mode"]),
                            "roi_background": float(_opt_arm["roi_background"]),
                            # Holdout results of the first test set, and of every test set by name.
                            "test_results": dict(next(iter(_test_results_by_set.values()))),
                            "test_results_by_test_set": {_n: dict(_r) for _n, _r in _test_results_by_set.items()},
                        }

                    all_results.append(row)

                    if _holdout_scored:
                        print(
                            f"\n  ── Completed [{backbone_name} | {ltag}"
                            + (f" | test set {_test_set_name}" if len(_holdout_sets) > 1 else "")
                            + "] ──"
                        )
                        print(
                            f"  CV mean AUC={row['cv_mean_auc']:.3f} ± {row['cv_std_auc']:.3f} | "
                            f"Pooled OOF AUC={row['oof_auc']:.3f} | "
                            f"Final holdout-test AUC={row['final_test_auc']:.3f}"
                        )
                    elif _test_set_name == _holdout_sets[0][0]:
                        # Without holdout scoring the run is reported once, not once per test set.
                        print(f"\n  ── Completed [{backbone_name} | {ltag}] ──")
                        print(
                            f"  CV mean AUC={row['cv_mean_auc']:.3f} ± {row['cv_std_auc']:.3f} | "
                            f"Pooled OOF AUC={row['oof_auc']:.3f} | "
                            "Final holdout-test AUC=not evaluated"
                        )
                    rng_state_restore(_rng_state)

                del final_model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # ── Per-backbone summary across outcome definitions ───────────────────
            _bb_rows = [
                r for r in all_results
                if r["backbone"] == backbone_name
                and r.get("e2e_mode") == e2e_mode
                and (not OPTIMIZATION or r.get("opt_arm_id") == (_opt_arm["arm_id"] if _opt_arm else ""))
            ]
            print_final_retrain_summary(
                _bb_rows,
                f"FINAL-RETRAIN SUMMARY (holdout test): backbone={backbone_name}, mode={e2e_mode}, all outcomes",
            )

    # ── Overall summary: every backbone × outcome ──
    print_final_retrain_summary(
        all_results,
        "OVERALL FINAL-RETRAIN SUMMARY (holdout test): all CNNs × all outcomes",
    )

    if OPTIMIZATION:
        # Do not carry the final setting combination's representation into the next workflow's setup or visual check.
        apply_part1_optimization_arm(None)

    print("\n" + "="*78)
    print("All backbone × outcome runs complete.")


In [ ]:
# Section 1, cell 13: Results export
if globals().get("RUN_FROZEN_CNN_E2E", True):

    results_df = pd.DataFrame(all_results)
    results_df["segmentation_method"] = SEGMENTATION_ANALYSIS_METHOD

    # CV AUC as mean ± SD.
    if len(results_df) > 0:
        results_df["cv_auc_mean_sd"] = [
            ("nan" if pd.isna(m) else f"{m:.3f} ± {(0.0 if pd.isna(s) else s):.3f}")
            for m, s in zip(results_df["cv_mean_auc"], results_df["cv_std_auc"])
        ]

    display_cols = [
        "backbone", "outcome", "test_set", "channel_policy",
        "optimization", "opt_arm_id", "opt_is_reference",
        "opt_input_size", "opt_n_slices", "opt_augmentation",
        "tumor_context_mode", "roi_only_background_value",
        "n_total_cases", "n_dev_cases", "n_test_cases", "n_folds", "n_repeats",
        "cv_auc_mean_sd",
        # Cut-off independent.
        "cv_mean_auc", "cv_std_auc", "cv_mean_brier", "cv_std_brier",
        # Threshold-dependent metrics; the cut-off is named in the column.
        "cv_oofthr_threshold", "cv_oofthr_threshold_source",
        "cv_oofthr_mean_accuracy", "cv_oofthr_mean_balacc", "cv_oofthr_mean_f1",
        "cv_oofthr_mean_sensitivity", "cv_oofthr_mean_specificity",
        "cv_oofthr_mean_mcc",
        "cv_fixed05_threshold",
        "cv_fixed05_mean_accuracy", "cv_fixed05_mean_balacc", "cv_fixed05_mean_f1",
        "cv_fixed05_mean_sensitivity", "cv_fixed05_mean_specificity",
        "cv_fixed05_mean_mcc",
        "oof_auc",
        "final_test_auc",
        "final_test_DeLong_CI_lower", "final_test_DeLong_CI_upper",
        "final_test_AUC_with_DeLong_95CI",
        "final_test_Jackknife_CI_lower", "final_test_Jackknife_CI_upper",
        "final_test_AUC_with_Jackknife_95CI",
        "final_test_Jackknife_min_AUC", "final_test_Jackknife_max_AUC",
        "final_test_Bootstrap_CI_lower", "final_test_Bootstrap_CI_upper",
        "final_test_AUC_with_Bootstrap_95CI", "final_test_Bootstrap_repeats",
        "final_test_accuracy", "final_test_balacc", "final_test_f1",
        "final_test_sensitivity", "final_test_specificity", "final_test_mcc", "final_test_brier",
        "final_test_threshold", "final_test_threshold_source",
        "final_test_toprisk_percent", "final_test_toprisk_threshold",
        "final_test_toprisk_accuracy", "final_test_toprisk_balacc",
        "final_test_toprisk_sensitivity", "final_test_toprisk_specificity",
        "final_test_toprisk_f1", "final_test_toprisk_mcc",
        "final_test_toprisk_threshold_source",
    ]

    if "e2e_mode" in results_df.columns and "e2e_mode" not in display_cols:
        display_cols.insert(1, "e2e_mode")

    print("\n── Final Results ──")
    if len(results_df) == 0:
        print("No completed runs.")
    else:
        # Print only the columns this run produced.
        _show_cols = [c for c in display_cols if c in results_df.columns]
        _missing = [c for c in display_cols if c not in results_df.columns]
        if _missing:
            print(f"[note] columns not present in this run: {_missing}")
        if holdout_evaluation_enabled():
            # One block of rows per test set, best holdout AUC first within each.
            print(order_by_test_set(results_df, "final_test_auc")[_show_cols].to_string(index=False))
        else:
            # OPTIMIZATION run without holdout scoring: every final_test_* value is empty.
            # The table shows the development columns and every run once. It is sorted by
            # pooled out-of-fold AUC. The CSV below keeps one row per test set.
            print("[note] Holdout test not scored in this run (OPTIMIZATION_EVALUATE_HOLDOUT is False). "
                  "Each run is shown once, sorted by development out-of-fold AUC.")
            _dev_table = rows_for_display(order_by_test_set(results_df, "oof_auc"))
            _dev_cols = [
                c for c in _show_cols
                if c in _dev_table.columns and not c.startswith("final_test_") and c != "n_test_cases"
            ]
            print(_dev_table[_dev_cols].to_string(index=False))

    # Mode tag and development/test-set sizes are part of the filename.
    _modes_tag = "_".join(safe_token(m) for m in END2END_MODES) or "none"
    _opt_tag = "_OPTIMIZATION_ALL_ARMS" if OPTIMIZATION else ""
    _split_rec = PATIENT_SPLIT_RECORDS.get(GLOBAL_PATIENT_SPLIT_KEY) or {}
    _split_tag = (f"_dev{len(_split_rec.get('dev_ids', ()))}"
                  f"_test{len(_split_rec.get('test_ids', ()))}")
    out_csv = (Path(OUT_DIR) /
               f"cnn_classification_results_{CHANNEL_POLICY}_all_supported_backbones"
               f"{_split_tag}_mode_{_modes_tag}{_opt_tag}_holdout.csv")
    results_df.to_csv(out_csv, index=False)
    print(f"\nResults CSV: {out_csv}")

    # OUT_DIR is already under OUT_ROOT; no second copy is made.

    # Hand the result table to the final overview.
    try:
        RESULTS_REGISTRY['segment1_cnn'] = results_df.copy()
        print('[SECTION 1] final overview contains', len(results_df), 'backbone x outcome x test-set rows.')
    except Exception as _e:
        print('[SECTION 1] result table is unavailable:', repr(_e))


## Section 2, cell 1 - Frozen-CNN feature extraction

No training. Each of the `SHARED_BACKBONES` networks runs over the same configured 1, 2 or 3 central orthogonal slices per patient
with frozen weights, and its embeddings are mean-pooled into one row of numbers per patient.

Backbones come from three pretraining sources: RadImageNet (radiology), ImageNet (natural images)
and the Choi IDH brain-tumour network. One CSV is written per backbone × channel policy × outcome
definition to `SEG2_FEATURE_DIR`, which Section 2's machine-learning cells then read.

In [ ]:
# Section 2, cell 2: Reset Section 2-only settings
if globals().get("RUN_CNN_LR", True):
    # Reset only workflow-local settings. The RadImageNet paths and the input scaling map
    # RADIMAGENET_NORM_BY_BACKBONE are shared Section 0 configuration. They are not
    # deleted, because all three sections read them.
    for _name in [
        "RUN_BACKBONES", "IMAGENET_CHANNEL_POLICY", "CHANNEL_POLICIES_TO_COMPARE",
    ]:
        globals().pop(_name, None)

    # In optimization mode, use the same backbone scope as both end-to-end workflows.
    # Otherwise Section 2 keeps its original full shared list.
    RUN_BACKBONES = (list(OPTIMIZATION_BACKBONES) if OPTIMIZATION else list(SHARED_BACKBONES))
    print("[SECTION 2] feature extraction backbones:", RUN_BACKBONES)
else:
    print('[SKIP] Section 2: efficient_ML workflow disabled by RUN_CNN_LR=False.')


In [ ]:
# Section 2, cell 3: Imports
if globals().get("RUN_CNN_LR", True):
    import sys, subprocess, os, warnings

    # Packages were installed in Section 0.

    # Warnings stay visible: numerical, API and checkpoint warnings must not be hidden.
    warnings.filterwarnings("default")
    # Silence only the repetitive Jupyter utcnow deprecation notice.
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        module=r"jupyter_client\.session",
    )
    print("Dependencies installed.")


In [ ]:
# Section 2, cell 4: Paths and configuration
if globals().get("RUN_CNN_LR", True):
    from pathlib import Path
    import os, re, time, shutil, hashlib, random, math
    import numpy as np
    import pandas as pd
    from tqdm.auto import tqdm

    import SimpleITK as sitk

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader

    ensure_drive_mounted()   # helper defined in Section 0

    # ── File locations ────────────────────────────────────────────────────────
    # DEVELOPMENT_ROOT, TEST_ROOT and LABELS_CONFIGS come from Section 0, cell 3.
    # LABELS_CONFIGS is built there from LABELS_CSV and holds the one label file of this run.
    ACTIVE_LABEL_CONFIGS = [c for c in LABELS_CONFIGS
                            if c.get("enabled") and c.get("path")]
    assert len(ACTIVE_LABEL_CONFIGS) >= 1, "LABELS_CSV in Section 0, cell 3 must be the path of the label file."
    # Section 0 copies and prepares inputs once for every workflow.

    # OUT_DIR holds this section's tables and summaries. Patient-level feature CSVs go
    # straight to SEG2_FEATURE_DIR: unnamed ID column, "labels", then cnn_f0000...
    OUT_DIR = PART2_OUT_DIR
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    Path(SEG2_FEATURE_DIR).mkdir(parents=True, exist_ok=True)

    # ── Feature-extraction settings ───────────────────────────────────────────
    CNN_INPUT_SIZE = CNN_INPUT_SIZE_SHARED  # shared in Section 0
    # ── Preprocessing constants, shared with Section 1 ────────────────────────
    BATCH_SIZE_FEATURES = 32

    # T1cc is a single modality but the CNNs expect three channels. Default assembly is
    # [T1cc, T1cc, mask]; "t1cc_masked_mask" uses [T1cc, T1cc inside the tumour, mask].
    CHANNEL_MODE = "t1cc_t1cc_mask"

    # Load the published Choi-IDH weights. False gives a randomly initialised network,
    # valid only as a wiring test and never for reported results.
    USE_GITHUB_IDH_PRETRAINED_WEIGHTS = True
    GITHUB_MODEL2_URL = "https://github.com/yoonchoi-neuro/automated_hybrid_IDH/releases/download/final/MODEL2_CNNclassifier.pth"
    MODEL2_CKPT_PATH = str(Path(OUT_DIR) / "MODEL2_CNNclassifier.pth")

    # ── Models ────────────────────────────────────────────────────────────────
    # All models see the same configured tumour-centred slices, so their outputs are
    # directly comparable; the model name is written into each filename. Entries read
    # "<pretraining>_<architecture>"; embedding width per scan:
    #   choi_idh 512 | resnet18 512 | resnet10t 512 | densenet121 1024
    #   resnet50 2048 | inceptionv3 2048

    # Parameter counts: resnet10t ~5M < densenet121 ~8M < resnet18 ~11.7M
    # < inceptionv3 ~24M < resnet50 ~25.6M

    # RUN_BACKBONES is set once in Section 2 Cell 1 from the Section 0 scope.
    # Do not reassign it here; feature extraction uses that already-reset list.

    PART2_OPTIMIZATION_ARMS = (list(OPTIMIZATION_ML_ARMS) if OPTIMIZATION else [None])

    def apply_part2_optimization_arm(arm):
        """Apply size/slice representation settings to frozen feature extraction."""
        global CNN_INPUT_SIZE, N_SLICES, SLICE_PLANES
        if arm is None:
            CNN_INPUT_SIZE = int(CNN_INPUT_SIZE_SHARED)
            N_SLICES = int(globals().get("OPTIMIZATION_REFERENCE_N_SLICES", N_SLICES))
        else:
            CNN_INPUT_SIZE = int(arm["input_size"])
            N_SLICES = int(arm["n_slices"])
        if isinstance(N_SLICES, (bool, np.bool_)) or N_SLICES not in (1, 2, 3):
            raise ValueError(f"N_SLICES must be 1, 2 or 3; got {N_SLICES}")
        SLICE_PLANES = ("axial", "sagittal", "coronal")[:int(N_SLICES)]
        if arm is not None:
            print("\n" + "=" * 78)
            print(f"[SECTION 2 OPTIMIZATION ARM] {arm['arm_id']}"
                  + ("   (REFERENCE)" if arm.get("is_reference") else ""))
            print(f"    preprocessing size={CNN_INPUT_SIZE} | slices={N_SLICES} | "
                  f"planes={SLICE_PLANES}")
            print("    augmentation=NOT_APPLICABLE (CNN is frozen; one deterministic feature row per patient)")
            print("=" * 78)

    # Current model name, used by the helpers and by one-off runs.
    BACKBONE = str(RUN_BACKBONES[0])

    # Channel policy, one CSV per entry in CHANNEL_POLICIES_TO_COMPARE:
    #   "gray3"  T1cc replicated three times; the mask only defines the crop. Closest to
    #            how these networks were pretrained.
    #   "keep"   [T1cc, T1cc, mask]; supplies tumour location but gives the third channel
    #            a distribution unlike anything seen in pretraining.
    IMAGENET_CHANNEL_POLICY = globals().get("IMAGENET_CHANNEL_POLICY", "gray3")
    # options: "gray3", "keep"
    CHANNEL_POLICIES_TO_COMPARE = globals().get("CHANNEL_POLICIES_TO_COMPARE", ["gray3"])

    # The RadImageNet checkpoint paths are resolved once in Section 0, cell 3. The input
    # scaling of every backbone (RADIMAGENET_NORM_BY_BACKBONE, radimagenet_norm_for() and
    # apply_input_scaling()) is defined there as well and is shared by all three sections.

    def print_weight_status(label, path):
        p = Path(str(path))
        status = "FOUND" if p.exists() else "MISSING"
        size_mb = f" ({p.stat().st_size / 1024**2:.1f} MB)" if p.exists() else ""
        print(f"  {label}: {status}{size_mb} | {p}")

    print("CNN backbones scheduled to run:", RUN_BACKBONES)
    print("RadImageNet weights:")
    print_weight_status("ResNet-50", RADIMAGENET_WEIGHTS_PATH)
    print_weight_status("DenseNet-121", RADIMAGENET_DENSENET121_WEIGHTS_PATH)
    print_weight_status("InceptionV3", RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH)
    print_weight_status("ResNet-18",  RADIMAGENET_RESNET18_WEIGHTS_PATH)
    print_weight_status("ResNet-10t", RADIMAGENET_RESNET10T_WEIGHTS_PATH)
    print("Input scaling (per RadImageNet backbone):")
    for _bk in ("radimagenet_resnet50", "radimagenet_densenet121", "radimagenet_inceptionv3",
                "radimagenet_resnet18", "radimagenet_resnet10t"):
        print(f"  {_bk}: {radimagenet_norm_for(_bk)}")

    # Fixed seed for reproducibility.
    seed_everything(GLOBAL_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


In [ ]:
# Section 2, cell 5: Image/mask pairing and reading
if globals().get("RUN_CNN_LR", True):

    # Pairing and validation were completed once in Section 0.

    def match_pairs_to_labels(pairs, labels_df):
        """Attach each patient's outcome label to their image+mask pair (exact ID match only)."""
        label_map = {str(r.canonical_id): (str(r.patient_id), int(r.label)) for r in labels_df.itertuples(index=False)}
        keep, missing = [], []
        for item in pairs:
            cid = canonical_patient_id(item["patient_id"])
            hit = label_map.get(cid)
            if hit is None:
                missing.append({
                    "pair_patient_id": item.get("patient_id"),
                    "canonical_id": cid,
                    "image": item.get("image"),
                    "mask": item.get("mask"),
                })
                continue
            label_patient_id, label = hit
            item2 = dict(item)
            item2["patient_id"] = label_patient_id
            item2["canonical_id"] = cid
            item2["label"] = label
            keep.append(item2)
        print(f"Pairs matched to labels: {len(keep)} / {len(pairs)}")
        if missing:
            out = Path(OUT_DIR) / "pairs_without_label_match.csv"
            pd.DataFrame(missing).to_csv(out, index=False)
            print(f"Saved unmatched pair list: {out}")
            print(pd.DataFrame(missing).head(10).to_string(index=False))
        return keep


    def print_image_mask_matches(pairs, title="[image] image-mask matches", csv_name="image_mask_matches.csv"):
        rows = []
        for i, item in enumerate(pairs, start=1):
            rows.append({
                "n": i,
                "patient_id": item.get("patient_id"),
                "canonical_id": item.get("canonical_id", canonical_patient_id(item.get("patient_id", ""))),
                "label": item.get("label", ""),
                "split_set": item.get("split_set", ""),
                "image_file": Path(str(item.get("image", ""))).name,
                "mask_file": Path(str(item.get("mask", ""))).name,
                "image_path": item.get("image"),
                "mask_path": item.get("mask"),
            })
        print("\n" + "=" * 120)
        print(f"{title}: {len(rows)} pair(s)")
        print("=" * 120)
        for row in rows[:200]:
            label_text = f" | label={row['label']}" if row.get("label") != "" else ""
            set_text = f" | folder={str(row['split_set']).upper()}" if row.get("split_set") else ""
            print(f"{row['n']:03d} | patient_id={row['patient_id']} | canonical_id={row['canonical_id']}{label_text}{set_text}")
            print(f"      image: {row['image_file']}")
            print(f"      mask : {row['mask_file']}")
        if len(rows) > 200:
            print(f"... printed first 200 of {len(rows)} pairs")
        out_path = Path(OUT_DIR) / csv_name
        pd.DataFrame(rows).to_csv(out_path, index=False)
        print(f"Saved match list to: {out_path}")




In [ ]:
# Section 2, cell 6: Load pairs and labels
if globals().get("RUN_CNN_LR", True):
    # Features are extracted once and reused for every outcome definition, so a scan is
    # kept if its patient appears in any active label file. Cell 9 then writes one
    # feature CSV per definition, naming the definition in each filename.
    image_pairs_all, all_labels_dfs = get_shared_patient_data("Section 2")
    if set(all_labels_dfs) != {cfg["tag"] for cfg in ACTIVE_LABEL_CONFIGS}:
        raise RuntimeError("Active label settings changed. Rerun Section 0, cell 3.")

    # Labels are merged only to decide which scans to keep; the label attached here is
    # a placeholder that Cell 9 replaces per outcome definition.
    _labels_for_extraction_parts = []
    for _ltag, _ldf in all_labels_dfs.items():
        _tmp = _ldf.copy()
        _tmp["outcome_used_for_matching"] = _ltag
        _labels_for_extraction_parts.append(_tmp)

    labels_for_extraction_df = (
        pd.concat(_labels_for_extraction_parts, ignore_index=True)
          .sort_values(["canonical_id", "outcome_used_for_matching"])
          .drop_duplicates("canonical_id", keep="first")
          [["patient_id", "canonical_id", "label"]]
          .reset_index(drop=True)
    )

    labels_df = labels_for_extraction_df
    print("Union label set for feature extraction:", labels_df.shape)
    print(labels_df.head().to_string(index=False))
    print(
        "NOTE: labels shown here are temporary matching labels only; "
        "Cell 9 writes one correctly relabeled feature CSV per outcome definition."
    )

    # Only row ordering differs here; patient identity and folder membership are fixed.
    image_pairs_all = sorted(
        image_pairs_all, key=lambda p: (len(str(p["canonical_id"])), str(p["canonical_id"]))
    )
    print_image_mask_matches(
        image_pairs_all,
        title="[image load finished] Exact image-mask matches after collection/existence check",
        csv_name="image_mask_matches_after_load.csv",
    )

    image_pairs = match_pairs_to_labels(image_pairs_all, labels_df)
    assert len(image_pairs) > 0, "No image+mask pairs matched to any active outcome definition."
    print_image_mask_matches(
        image_pairs,
        title="[image final cohort] Image-mask matches entering CNN feature extraction "
              "(matched to union of active outcome definitions)",
        csv_name="image_mask_matches_final_labeled.csv",
    )


In [ ]:
# Section 2, cell 7: Slice extraction
if globals().get("RUN_CNN_LR", True):
    import warnings

    def make_patient_2d_cnn_inputs(image_path, mask_path):
        """Build one patient's central orthogonal CNN slices with a physical-square crop per plane."""
        return get_shared_patient_inputs(image_path, mask_path, CNN_INPUT_SIZE)


    # Basic input check on the first patient before the full run.
    first = image_pairs[0]
    x5, meta5 = make_patient_2d_cnn_inputs(first["image"], first["mask"])
    print("First patient:", first["patient_id"], "label=", first["label"])
    print("CNN input tensor:", tuple(x5.shape), "min/max:", float(x5.min()), float(x5.max()))
    print(pd.DataFrame(meta5).to_string(index=False))


In [ ]:
# Section 2, cell 8: Visual check of analysed slices
if globals().get("RUN_CNN_LR", True):
    # Same panel as in Section 1, run on this section's own extractor so the frozen
    # feature inputs can be inspected. preview_analysed_slices() is defined in Section 0,
    # cell 3, so this check does not need Section 1.
    print(f"[QC] Section 2 central planes | {SLICE_PLANES}")
    try:
        preview_analysed_slices(
            image_pairs,
            lambda it: make_patient_2d_cnn_inputs(it["image"], it["mask"]),
            title="Section 2: ",
        )
    except Exception as _e:
        print("[QC] Slice preview unavailable:", repr(_e))


In [ ]:
# Section 2, cell 9: Feature-extractor construction
if globals().get("RUN_CNN_LR", True):

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        return nn.Conv2d(
            in_planes, out_planes, kernel_size=3, stride=stride,
            padding=dilation, groups=groups, bias=False, dilation=dilation,
        )


    def conv1x1(in_planes, out_planes, stride=1):
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None):
            super().__init__()
            self.conv1 = conv3x3(inplanes, planes, stride)
            self.bn1 = nn.BatchNorm2d(planes)
            self.relu1 = nn.ReLU(inplace=True)
            self.conv2 = conv3x3(planes, planes)
            self.bn2 = nn.BatchNorm2d(planes)
            self.relu2 = nn.ReLU(inplace=True)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            residual = x
            out = self.conv1(x)
            out = self.bn1(out)
            out = self.relu1(out)
            out = self.conv2(out)
            out = self.bn2(out)
            if self.downsample is not None:
                residual = self.downsample(x)
            out += residual
            out = self.relu2(out)
            return out


    class ResNetFeatureExtractor(nn.Module):
        """The image half of the published Choi IDH network (a ResNet-34-style CNN); gives 512 numbers per image."""
        def __init__(self, n_seqs=3, block=BasicBlock, layers=(3, 4, 6, 3)):
            super().__init__()
            self.inplanes = 64
            self.conv1 = nn.Conv2d(n_seqs, 64, kernel_size=7, stride=2, padding=3, bias=False)
            self.bn1 = nn.BatchNorm2d(64)
            self.relu = nn.ReLU(inplace=True)
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
            self.avgpool = nn.AvgPool2d(8, stride=1)
            self.out_dim = 512 * block.expansion

        def _make_layer(self, block, planes, blocks, stride=1):
            downsample = None
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                    nn.BatchNorm2d(planes * block.expansion),
                )
            layers = [block(self.inplanes, planes, stride, downsample)]
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes))
            return nn.Sequential(*layers)

        def forward(self, x):
            x = self.conv1(x)     # 128 -> 64
            x = self.bn1(x)
            x = self.relu(x)
            x = self.layer1(x)    # 64
            x = self.layer2(x)    # 32
            x = self.layer3(x)    # 16
            x = self.layer4(x)    # 8
            x = self.avgpool(x)   # 1
            x = x.view(x.size(0), -1)
            return x


    def download_github_model2_checkpoint(url=GITHUB_MODEL2_URL, dst=MODEL2_CKPT_PATH):
        dst = Path(dst)
        if dst.exists() and dst.stat().st_size > 1024:
            print(f"Checkpoint already exists: {dst}")
            return str(dst)
        from torch.hub import download_url_to_file
        print(f"Downloading GitHub Model 2 checkpoint to: {dst}")
        download_url_to_file(url, str(dst), progress=True)
        return str(dst)


    def load_github_model2_image_backbone_weights(feature_model, ckpt_path):
        """Copy in only the image-processing weights from the saved Choi IDH file.

        In that file the image layers are usually tagged with a 'model_pre.' name
        prefix. We take those and skip the final decision layers and the separate
        numeric-input layers, which aren't used here.
        """
        # This checkpoint carries a few non-tensor entries, so weights_only is relaxed;
        # the file comes from a trusted release.
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        except TypeError:
            # Older PyTorch has no weights_only argument.
            ckpt = torch.load(ckpt_path, map_location="cpu")
        sd = ckpt.get("model_state_dict", ckpt)
        current = feature_model.state_dict()
        loadable = {}

        for k, v in sd.items():
            nk = None
            for prefix in ("model_pre.", "module.model_pre."):
                if k.startswith(prefix):
                    nk = k[len(prefix):]
                    break
            # Also accept a plain ResNet checkpoint without the 'model_pre.' prefix.
            if nk is None and (k.startswith("conv1.") or k.startswith("bn1.") or k.startswith("layer")):
                nk = k
            if nk is None:
                continue
            if nk.startswith("fc.") or nk.startswith("fc"):
                continue
            if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                loadable[nk] = v

        missing, unexpected = feature_model.load_state_dict(loadable, strict=False)
        print(f"Loaded {len(loadable)} compatible image-backbone tensors from: {ckpt_path}")
        print(f"Model tensors total: {len(current)}")
        if len(loadable) == 0:
            raise RuntimeError("The checkpoint contains no compatible CNN backbone weights.")
        return feature_model


    # ── Shared input wrapper, plus one builder per model ──────────────────────
    # Every model receives slices as (batch, 3, H, W) with channel 0 the scan and
    # channel 2 the mask, and returns a fixed-width embedding. The wrapper applies each
    # model's own resizing, channel policy and normalisation, so all models see the
    # same crops and their outputs stay comparable.
    import torchvision

    # The scaling arithmetic (apply_input_scaling(), IMAGENET_MEAN and IMAGENET_STD) is
    # defined once in Section 0, cell 3.

    class BackboneFeatureExtractor(nn.Module):
        def __init__(self, backbone, out_dim, input_size=128,
                     channel_policy="keep", norm="none"):
            super().__init__()
            self.backbone = backbone
            self.out_dim = int(out_dim)
            self.input_size = int(input_size)
            self.channel_policy = channel_policy   # "keep" retains the mask channel, "gray3" drops it
            # "none" or one of the input scalings defined in Section 0, cell 3.
            self.norm = str(norm).lower()

        def _prep(self, x):
            # x: (batch, 3, H, W); channel 0 is the scan, channel 2 the mask.
            if self.channel_policy == "gray3":
                x = x[:, 0:1, :, :].repeat(1, 3, 1, 1)
            if x.shape[-1] != self.input_size or x.shape[-2] != self.input_size:
                x = F.interpolate(x, size=(self.input_size, self.input_size),
                                  mode="bilinear", align_corners=False)
            # Per-backbone input scaling, the same function in all three sections.
            # "unit_interval" and "none" need nothing further: preprocessing already
            # delivered roughly [0, 1].
            return apply_input_scaling(x, self.norm)

        def forward(self, x):
            h = self.backbone(self._prep(x))
            if h.dim() > 2:
                h = F.adaptive_avg_pool2d(h, 1).flatten(1)
            return h


    def _make_torchvision_resnet(arch, pretrained):
        try:
            net = getattr(torchvision.models, arch)(weights=("DEFAULT" if pretrained else None))
        except TypeError:
            net = getattr(torchvision.models, arch)(pretrained=bool(pretrained))
        dim = net.fc.in_features
        net.fc = nn.Identity()   # forward returns the pooled feature with shape (B, dim)
        return net, dim


    def _make_torchvision_densenet121(pretrained=False):
        try:
            net = torchvision.models.densenet121(weights=("DEFAULT" if pretrained else None))
        except TypeError:
            net = torchvision.models.densenet121(pretrained=bool(pretrained))
        dim = net.classifier.in_features
        net.classifier = nn.Identity()
        return net, dim


    def _make_torchvision_inception_v3(pretrained=False):
        try:
            net = torchvision.models.inception_v3(
                weights=("DEFAULT" if pretrained else None), aux_logits=True, init_weights=False
            )
        except TypeError:
            net = torchvision.models.inception_v3(
                pretrained=bool(pretrained), aux_logits=True, init_weights=False
            )
        dim = net.fc.in_features
        net.fc = nn.Identity()
        return net, dim


    def _make_timm_resnet10t():
        """Build the small ResNet-10t model as a feature extractor (512 numbers per image)."""
        import timm
        # num_classes=0 drops the classifier and returns the pooled embedding.
        net = timm.create_model("resnet10t", pretrained=False, num_classes=0)
        dim = net.num_features  # 512
        return net, dim


    def load_radimagenet_weights(net, weights_path, architecture_name="RadImageNet backbone"):
        p = Path(weights_path) if weights_path else None
        if (p is None) or (not p.exists()):
            parent = p.parent if p is not None else Path(".")
            nearby = []
            if parent.exists():
                nearby = sorted([str(x) for x in parent.rglob("*.pt")][:20] + [str(x) for x in parent.rglob("*.pth")][:20])
            raise FileNotFoundError(
                f"{architecture_name} weights not found at the resolved path. "
                "If the file is visible in Drive, check that Cell 2 runs again after Drive mounts, "
                "and that the filename/folder matches exactly. "
                f"Looked at: {weights_path}. "
                f"Nearby .pt/.pth files under {parent}: {nearby[:20]}"
            )

        try:
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
        except TypeError:
            raw = torch.load(str(p), map_location="cpu")

        def _unwrap_state_dict(obj):
            """Dig out the real table of weights, which saved files often bury inside wrapper layers."""
            if hasattr(obj, "state_dict"):
                obj = obj.state_dict()
            if not isinstance(obj, dict):
                return obj, "<non-dict checkpoint>"

            # Already a state_dict.
            tensor_like_values = [v for v in obj.values() if torch.is_tensor(v)]
            if len(tensor_like_values) >= 10:
                return obj, "<top-level>"

            for key in (
                "state_dict", "model_state_dict", "model", "net", "network",
                "backbone", "encoder", "weights", "params", "module",
                "model_weights", "weight", "pretrained_dict",
            ):
                if key in obj:
                    nested, where = _unwrap_state_dict(obj[key])
                    if isinstance(nested, dict):
                        return nested, key if where == "<top-level>" else f"{key}.{where}"
            return obj, "<dict without tensor state_dict>"

        sd, sd_location = _unwrap_state_dict(raw)
        if not isinstance(sd, dict):
            raise RuntimeError(
                f"{architecture_name} checkpoint at {p} did not contain a PyTorch state_dict. "
                f"Loaded object type: {type(sd)}"
            )

        current = net.state_dict()
        loadable = {}

        import re as _re

        def _densenet_legacy(key):
            """Rename older DenseNet keys (...norm.1.weight) to the modern form (...norm1.weight).

            Pass 1 finds no matches on a checkpoint using the old spelling.
            """
            key = _re.sub(r'\.norm\.(\d+)', r'.norm\1', key)
            key = _re.sub(r'\.conv\.(\d+)', r'.conv\1', key)
            return key

        def candidate_keys(key):
            # Try each key as-is, then strip known wrapper prefixes only. DenseNet's own
            # "features." prefix is genuine and is kept.
            candidates = []
            seen = set()
            queue = [key]
            prefixes = (
                "module.", "backbone.", "model.", "encoder.", "resnet.",
                "base_model.", "feature_extractor.", "network.", "net.",
                "densenet.", "inception.", "inception_v3.",
                "0.", "1.", "2.", "3.",   # nn.Sequential / numeric-index wrappers
                "_orig_mod.",             # torch.compile saves with this prefix (e.g. ResNet-18)
            )
            while queue:
                cur = queue.pop(0)
                if cur not in seen:
                    seen.add(cur)
                    candidates.append(cur)
                    for pre in prefixes:
                        if cur.startswith(pre):
                            stripped = cur[len(pre):]
                            if stripped not in seen:
                                queue.append(stripped)
            return candidates

        _HEAD_PREFIXES = ("fc.", "classifier.", "AuxLogits.fc.")
        _pass3_rescued = 0  # initialised here; may be updated in Pass 3 below

        tensor_items = []
        non_tensor_keys = []
        for k, v in sd.items():
            if not torch.is_tensor(v):
                non_tensor_keys.append(str(k))
                continue
            tensor_items.append((str(k), v))
            for nk in candidate_keys(str(k)):
                if nk.startswith(_HEAD_PREFIXES):
                    continue
                if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                    loadable[nk] = v
                    break

        # Pass 2, older DenseNet key names: "norm.1"/"conv.1" -> "norm1"/"conv1". Runs for
        # DenseNet, or whenever pass 1 matched fewer than five tensors.
        _arch_lower = architecture_name.lower()
        if "densenet" in _arch_lower or len(loadable) < 5:
            _matched_so_far = set(loadable.keys())
            for k, v in tensor_items:
                modern_k = _densenet_legacy(k)
                if modern_k == k:
                    continue  # key unchanged; already tried in pass 1
                for nk in candidate_keys(modern_k):
                    if nk.startswith(_HEAD_PREFIXES) or nk in _matched_so_far:
                        continue
                    if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                        loadable[nk] = v
                        _matched_so_far.add(nk)
                        break

        # Pass 3: positional keys. Some RadImageNet checkpoints index by position
        # ("backbone.0", "backbone.1", ...) rather than by name, e.g.
        #   ResNet-50     backbone.0.weight       -> conv1.weight
        #   DenseNet-121  backbone.0.conv0.weight -> features.conv0.weight
        #   InceptionV3   backbone.0.conv.weight  -> Conv2d_1a_3x3.conv.weight
        _inception_had_aux = False  # set in pass 3
        _pass3_rescued = 0
        if len(loadable) < 5:
            _matched_so_far = set(loadable.keys())
            _ckpt_keys = [k for k, _ in tensor_items]
            _uses_backbone_prefix = any(k.startswith("backbone.") for k in _ckpt_keys)

            # InceptionV3's training-only AuxLogits branch may or may not be present, and
            # its presence shifts every positional index by one. The highest index decides:
            #   <= 17  no AuxLogits, positions 15/16/17 are the final blocks
            #   >= 18  AuxLogits at 15, final blocks at 16/17/18
            # The wrong map loads weights into a neighbouring block without raising, since
            # the shapes match; the completeness check below catches it.
            _inception_had_aux = False
            if _uses_backbone_prefix and "inception" in _arch_lower:
                _bb_idxs = {
                    int(_m.group(1)) for _k in _ckpt_keys
                    for _m in [_re.match(r"backbone\.(\d+)\.", _k)] if _m
                }
                _inception_had_aux = max(_bb_idxs, default=0) >= 18

            def _normalize_key(k):
                if not _uses_backbone_prefix:
                    return k
                if "densenet" in _arch_lower:
                    if k.startswith("backbone.0."):
                        return "features." + k[len("backbone.0."):]
                elif "inception" in _arch_lower:
                    _imap = {
                        "backbone.0.":  "Conv2d_1a_3x3.", "backbone.1.":  "Conv2d_2a_3x3.",
                        "backbone.2.":  "Conv2d_2b_3x3.",
                        "backbone.4.":  "Conv2d_3b_1x1.", "backbone.5.":  "Conv2d_4a_3x3.",
                        "backbone.7.":  "Mixed_5b.",       "backbone.8.":  "Mixed_5c.",
                        "backbone.9.":  "Mixed_5d.",       "backbone.10.": "Mixed_6a.",
                        "backbone.11.": "Mixed_6b.",       "backbone.12.": "Mixed_6c.",
                        "backbone.13.": "Mixed_6d.",       "backbone.14.": "Mixed_6e.",
                    }
                    if _inception_had_aux:
                        # AuxLogits present (max index >= 18).
                        _imap.update({
                            "backbone.15.": "AuxLogits.",
                            "backbone.16.": "Mixed_7a.",
                            "backbone.17.": "Mixed_7b.",
                            "backbone.18.": "Mixed_7c.",
                        })
                    else:
                        # No AuxLogits (max index <= 17).
                        _imap.update({
                            "backbone.15.": "Mixed_7a.",
                            "backbone.16.": "Mixed_7b.",
                            "backbone.17.": "Mixed_7c.",
                        })
                    for old, dst in _imap.items():
                        if k.startswith(old):
                            return dst + k[len(old):]
                else:  # resnet-50 / resnet-18 / resnet-10t
                    # ResNet positions: 0 = conv1, 1 = bn1, 4..7 = layer1..4. Parameterless
                    # layers are absent from the checkpoint. The ResNet-10t stem follows the
                    # same layout, so one map covers both.
                    _rmap = {
                        "backbone.0.": "conv1.", "backbone.1.": "bn1.",
                        "backbone.4.": "layer1.", "backbone.5.": "layer2.",
                        "backbone.6.": "layer3.", "backbone.7.": "layer4.",
                    }
                    for old, dst in _rmap.items():
                        if k.startswith(old):
                            return dst + k[len(old):]
                return k

            for orig_k, v in tensor_items:
                norm_k = _normalize_key(orig_k)
                for nk in candidate_keys(norm_k):
                    if nk.startswith(_HEAD_PREFIXES) or nk in _matched_so_far:
                        continue
                    if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                        loadable[nk] = v
                        _matched_so_far.add(nk)
                        _pass3_rescued += 1
                        break

            if _pass3_rescued:
                print(f"[RadImageNet] Pass 3 (sequential-backbone remapping) rescued {_pass3_rescued} tensors")

        sample_ckpt = [(k, tuple(v.shape)) for k, v in tensor_items[:12]]
        sample_expected = [(k, tuple(v.shape)) for k, v in list(current.items())[:12]]

        print(f"[RadImageNet] {architecture_name}: checkpoint={p.name} | state_dict_location={sd_location}")
        print(f"[RadImageNet] tensor keys in checkpoint: {len(tensor_items)} | non-tensor keys: {len(non_tensor_keys)}")
        print(f"[RadImageNet] sample checkpoint keys: {sample_ckpt}")
        print(f"[RadImageNet] sample expected torchvision keys: {sample_expected}")

        if len(loadable) == 0:
            raise RuntimeError(
                f"{architecture_name} loaded 0 compatible tensors from {p}. "
                "This usually means the checkpoint key names do not match the torchvision "
                "architecture expected for this backbone.\n"
                "Common causes:\n"
                "  • Wrong .pt file (e.g. ResNet50 file used for DenseNet121).\n"
                "  • DenseNet saved with old torchvision (<0.12): norm.1/conv.1 vs norm1/conv1 "
                "(pass 2 should have fixed this automatically. If you still see this, the file "
                "may not be a torchvision DenseNet at all).\n"
                "  • InceptionV3 checkpoint from a custom/non-torchvision implementation.\n"
                "  • Checkpoint is a full Keras/TF model, not a PyTorch state_dict.\n"
                "Inspect the sample checkpoint keys printed above vs. the expected torchvision keys "
                "and add an architecture-specific key-conversion pass if needed."
            )

        missing, unexpected = net.load_state_dict(loadable, strict=False)
        _used_legacy = ("densenet" in architecture_name.lower() or len(loadable) < 5) and any(
            _densenet_legacy(k) != k for k, _ in tensor_items
        )
        _strategy = ("densenet_legacy_keys" if _used_legacy
                     else "sequential_backbone_remapping" if _pass3_rescued
                     else "standard")
        print(f"[RadImageNet] {architecture_name}: loaded {len(loadable)}/{len(current)} compatible "
              f"backbone tensors from {p.name} (strategy={_strategy})")

        # ── Load completeness ─────────────────────────────────────────────────
        # Counting the model's non-head tensors gives an exact load fraction, keeping the
        # incomplete-load warning reliable for every architecture.
        _n_expected = len([k for k in current if not k.startswith(_HEAD_PREFIXES)])
        _load_frac = len(loadable) / max(1, _n_expected)
        if _load_frac < 0.80:
            print(
                f"[RadImageNet][warning] {architecture_name}: loaded {len(loadable)}/{_n_expected} "
                f"expected tensors ({_load_frac:.0%}). A healthy load is typically >90%. "
                "Inspect the sample keys above for a naming mismatch."
            )

        # ── Strict InceptionV3 check ──────────────────────────────────────────
        # The positional map cannot be verified from the checkpoint alone, and equal shapes
        # hide a wrong assignment. A wrong map leaves slots empty, so the load fraction
        # falls; a correct load fills all 94 convolution blocks, ~98% of non-head tensors,
        # the remainder being AuxLogits. Below 0.95 raises.
        if "inception" in _arch_lower:
            _bb_indices = sorted({
                int(_m.group(1)) for _k, _ in tensor_items
                for _m in [_re.match(r"backbone\.(\d+)\.", _k)] if _m
            })
            print(f"[RadImageNet] {architecture_name}: checkpoint backbone.N block "
                  f"indices present: {_bb_indices} | inferred aux_logits_when_saved={_inception_had_aux}")
            if _load_frac < 0.95:
                raise RuntimeError(
                    f"[RadImageNet][FATAL] {architecture_name}: loaded only "
                    f"{len(loadable)}/{_n_expected} non-head tensors ({_load_frac:.0%}); "
                    "a correct InceptionV3 load is ~98%. A partial load means the "
                    "backbone.N -> torchvision index map (Pass 3) does not match this "
                    "checkpoint's Sequential order, which risks silently loading weights "
                    "into the WRONG Inception block. "
                    f"Observed backbone.N indices: {_bb_indices}. "
                    "Validate them against the torchvision inception_v3 child order "
                    "(Conv2d_1a/2a/2b, [maxpool], Conv2d_3b/4a, [maxpool], "
                    "Mixed_5b/5c/5d/6a/6b/6c/6d/6e, [AuxLogits], Mixed_7a/7b/7c) and "
                    "correct the _imap in Pass 3 before trusting InceptionV3 features."
                )

        # ── BatchNorm statistics ──────────────────────────────────────────────
        # Missing running_mean/running_var makes eval() fall back to mean 0, variance 1
        # and silently corrupts the features.
        _bn_expected_keys = [k for k in current if k.endswith((".running_mean", ".running_var"))]
        _bn_missing_keys  = [k for k in missing  if k.endswith((".running_mean", ".running_var"))]
        if _bn_missing_keys:
            _bn_frac = len(_bn_missing_keys) / max(1, len(_bn_expected_keys))
            if _bn_frac > 0.5:
                raise RuntimeError(
                    f"[RadImageNet][FATAL] {architecture_name}: "
                    f"{len(_bn_missing_keys)}/{len(_bn_expected_keys)} BatchNorm "
                    "running_mean/running_var buffers MISSING from checkpoint. "
                    "In eval() mode every BN layer normalises with mean=0/var=1, "
                    "producing silently invalid features. The checkpoint likely stores "
                    "only learnable parameters (conv weights + BN scale/bias) without "
                    "running statistics. "
                    f"Sample missing BN keys: {_bn_missing_keys[:4]}"
                )
            print(
                f"[RadImageNet][warning] {architecture_name}: "
                f"{len(_bn_missing_keys)}/{len(_bn_expected_keys)} BN running buffers "
                "missing. Those layers use default statistics in eval() mode."
            )
        else:
            print(
                f"[RadImageNet] {architecture_name}: all "
                f"{len(_bn_expected_keys)} BN running buffers loaded OK."
            )
        return net


    def _build_backbone_extractor_native(backbone_name, channel_policy=None):
        name = str(backbone_name).lower()
        _cp = channel_policy if channel_policy is not None else IMAGENET_CHANNEL_POLICY
        if name in ("choi_idh", "choi", "idh"):
            fe = ResNetFeatureExtractor(n_seqs=3, block=BasicBlock, layers=(3, 4, 6, 3))
            if USE_GITHUB_IDH_PRETRAINED_WEIGHTS:
                ckpt_path = download_github_model2_checkpoint()
                fe = load_github_model2_image_backbone_weights(fe, ckpt_path)
            else:
                print("WARNING: random Choi-architecture features (debug only, not meaningful).")
            return BackboneFeatureExtractor(fe, out_dim=fe.out_dim, input_size=128,
                                            channel_policy="keep", norm="none")
        if name in ("imagenet_resnet10t", "imagenet10t", "imagenet_resnet10"):
            import timm as _timm
            net = _timm.create_model("resnet10t", pretrained=True, num_classes=0)
            dim = net.num_features
            print("[ImageNet] resnet10t: timm ImageNet-pretrained (c3_in1k) weights")
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm="imagenet")
        if name in ("imagenet_resnet50", "imagenet50", "imagenet"):
            net, dim = _make_torchvision_resnet("resnet50", pretrained=True)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm="imagenet")
        if name in ("imagenet_resnet18", "imagenet18"):
            net, dim = _make_torchvision_resnet("resnet18", pretrained=True)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm="imagenet")
        if name in ("imagenet_densenet121", "imagenet_densenet", "imagenet121"):
            net, dim = _make_torchvision_densenet121(pretrained=True)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm="imagenet")
        if name in ("imagenet_inceptionv3", "imagenet_inception_v3", "imagenet_inception"):
            net, dim = _make_torchvision_inception_v3(pretrained=True)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=299,
                                            channel_policy=_cp, norm="imagenet")
        if name in ("radimagenet_resnet50", "radimagenet", "radimagenet50"):
            net, dim = _make_torchvision_resnet("resnet50", pretrained=False)
            net = load_radimagenet_weights(net, RADIMAGENET_WEIGHTS_PATH, "RadImageNet ResNet-50")
            rin_norm = radimagenet_norm_for(name)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm=rin_norm)
        if name in ("radimagenet_densenet121", "radimagenet_densenet", "radimagenet121"):
            net, dim = _make_torchvision_densenet121(pretrained=False)
            net = load_radimagenet_weights(net, RADIMAGENET_DENSENET121_WEIGHTS_PATH, "RadImageNet DenseNet-121")
            rin_norm = radimagenet_norm_for(name)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm=rin_norm)
        if name in ("radimagenet_inceptionv3", "radimagenet_inception_v3", "radimagenet_inception"):
            net, dim = _make_torchvision_inception_v3(pretrained=False)
            net = load_radimagenet_weights(net, RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH, "RadImageNet InceptionV3")
            rin_norm = radimagenet_norm_for(name)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=299,
                                            channel_policy=_cp, norm=rin_norm)
        if name in ("radimagenet_resnet18", "radimagenet18"):
            net, dim = _make_torchvision_resnet("resnet18", pretrained=False)
            net = load_radimagenet_weights(net, RADIMAGENET_RESNET18_WEIGHTS_PATH, "RadImageNet ResNet-18")
            rin_norm = radimagenet_norm_for(name)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm=rin_norm)
        if name in ("radimagenet_resnet10t", "radimagenet10t", "radimagenet_resnet10"):
            net, dim = _make_timm_resnet10t()
            net = load_radimagenet_weights(net, RADIMAGENET_RESNET10T_WEIGHTS_PATH, "RadImageNet ResNet-10t")
            rin_norm = radimagenet_norm_for(name)
            return BackboneFeatureExtractor(net, out_dim=dim, input_size=224,
                                            channel_policy=_cp, norm=rin_norm)
        raise ValueError(
            f"Unknown BACKBONE={backbone_name!r}. Use choi_idh, imagenet_resnet50, "
            "imagenet_resnet18, imagenet_densenet121, imagenet_inceptionv3, "
            "radimagenet_resnet50, radimagenet_densenet121, radimagenet_inceptionv3, "
            "radimagenet_resnet18 or radimagenet_resnet10t."
        )


    def cnn_weight_source(backbone_name):
        name = str(backbone_name).lower()
        if name in ("choi_idh", "choi", "idh"):
            if USE_GITHUB_IDH_PRETRAINED_WEIGHTS:
                return f"github_idh_model2:{MODEL2_CKPT_PATH}"
            return "random_init_debug_only"
        if name in (
            "imagenet_resnet50", "imagenet50", "imagenet",
            "imagenet_resnet18", "imagenet18",
            "imagenet_densenet121", "imagenet_densenet", "imagenet121",
            "imagenet_inceptionv3", "imagenet_inception_v3", "imagenet_inception",
        ):
            return "torchvision_default_imagenet"
        if name in ("radimagenet_resnet50", "radimagenet", "radimagenet50"):
            return f"radimagenet_file:{RADIMAGENET_WEIGHTS_PATH}"
        if name in ("radimagenet_densenet121", "radimagenet_densenet", "radimagenet121"):
            return f"radimagenet_file:{RADIMAGENET_DENSENET121_WEIGHTS_PATH}"
        if name in ("radimagenet_inceptionv3", "radimagenet_inception_v3", "radimagenet_inception"):
            return f"radimagenet_file:{RADIMAGENET_INCEPTIONV3_WEIGHTS_PATH}"
        if name in ("radimagenet_resnet18", "radimagenet18"):
            return f"radimagenet_file:{RADIMAGENET_RESNET18_WEIGHTS_PATH}"
        if name in ("radimagenet_resnet10t", "radimagenet10t", "radimagenet_resnet10"):
            return f"radimagenet_file:{RADIMAGENET_RESNET10T_WEIGHTS_PATH}"
        return "unknown"


    def cnn_weight_status(backbone_name):
        name = str(backbone_name).lower()
        if name in ("choi_idh", "choi", "idh"):
            return "loaded_github_idh_model2" if USE_GITHUB_IDH_PRETRAINED_WEIGHTS else "random_init_debug_only"
        if name.startswith("imagenet") or name in ("imagenet50", "imagenet18", "imagenet", "imagenet121"):
            return "loaded_torchvision_imagenet"
        if name.startswith("radimagenet") or name in ("radimagenet", "radimagenet50", "radimagenet121", "radimagenet18"):
            return "loaded_radimagenet_compatible_tensors"
        return "unknown"


    def cnn_feature_metadata(feature_model=None, backbone_name=None):
        feature_model = feature_model if feature_model is not None else globals().get("feature_extractor")
        backbone_name = str(backbone_name if backbone_name is not None else globals().get("BACKBONE", "unknown"))
        return {
            "cnn_backbone": backbone_name,
            "cnn_weight_source": cnn_weight_source(backbone_name),
            "cnn_weight_status": cnn_weight_status(backbone_name),
            "cnn_feature_dim": int(getattr(feature_model, "out_dim", -1)),
            "cnn_input_size": int(getattr(feature_model, "input_size", -1)),
            # What the weights were pretrained at, which is what the network consumes
            # unless an optimization arm asked for a different size.
            "cnn_native_input_size": int(getattr(
                feature_model, "native_input_size",
                getattr(feature_model, "input_size", -1),
            )),
            "cnn_channel_policy": str(getattr(feature_model, "channel_policy", "unknown")),
            "cnn_norm": str(getattr(feature_model, "norm", "unknown")),
            "cnn_workflow": "frozen_feature_extractor_only",
            "cnn_training_epochs": "none",
            "cnn_learning_rate": "none",
            "cnn_optimizer": "none",
            "cnn_gradient_updates": "disabled_torch_no_grad",
            "cnn_mode": "eval",
            "slice_strategy": f"{len(SLICE_PLANES)} central orthogonal slices",
            "slice_planes": str(tuple(SLICE_PLANES)),
            "patient_feature_aggregation": "mean_of_selected_slice_embeddings",
            "downstream_training": "classical_ml_cv_only_in_cell_10",
        }


    def print_cnn_feature_info(feature_info):
        print("CNN feature source:")
        for k, v in feature_info.items():
            print(f"  {k}: {v}")


    def print_feature_extraction_protocol(feature_info, n_cases=None):
        print("CNN extraction protocol:")
        print("  task: frozen CNN embedding extraction, not CNN training/fine-tuning")
        print(f"  backbone: {feature_info.get('cnn_backbone')}")
        print(f"  weight_source: {feature_info.get('cnn_weight_source')}")
        print(f"  weight_status: {feature_info.get('cnn_weight_status')}")
        print(f"  model_mode: {feature_info.get('cnn_mode')} | gradients: {feature_info.get('cnn_gradient_updates')}")
        print(f"  epochs: {feature_info.get('cnn_training_epochs')} | learning_rate: {feature_info.get('cnn_learning_rate')} | optimizer: {feature_info.get('cnn_optimizer')}")
        print(f"  input_channels: [T1cc, T1cc, tumor_mask] | channel_policy: {feature_info.get('cnn_channel_policy')}")
        print(f"  selected_slices: {feature_info.get('slice_strategy')} | planes: {feature_info.get('slice_planes')}")
        print(f"  patient_embedding: {feature_info.get('patient_feature_aggregation')}")
        if n_cases is not None:
            print(f"  cases_to_process: {n_cases}")


    # Smallest spatial input each architecture family can consume; below
    # this the forward pass either raises or collapses to a 1x1 feature map.
    _BACKBONE_MIN_INPUT_SIZE = (("inception", 75),)
    _BACKBONE_DEFAULT_MIN_INPUT_SIZE = 32

    def _backbone_min_input_size(backbone_name):
        name = str(backbone_name).lower()
        for _needle, _floor in _BACKBONE_MIN_INPUT_SIZE:
            if _needle in name:
                return int(_floor)
        return int(_BACKBONE_DEFAULT_MIN_INPUT_SIZE)

    def build_backbone_extractor(backbone_name, channel_policy=None, input_size=None):
        """Frozen extractor, optionally consuming a size other than its native one.

        Every factory branch pins the resolution its weights were pretrained at, and
        BackboneFeatureExtractor._prep resizes whatever it receives to that value.
        That is right for a fixed protocol, but it silently cancels an optimization
        arm that compares image size: a 128 px arm would be resampled straight back
        up to 224 and would measure a blur round-trip rather than the 128 px forward
        pass that Sections 1 and 3 measure for the same arm ID. Passing input_size
        makes the network consume the arm's size for real; leaving it None keeps the
        native pretraining size, so the non-optimization protocol is unchanged.
        """
        extractor = _build_backbone_extractor_native(
            backbone_name, channel_policy=channel_policy
        )
        extractor.native_input_size = int(extractor.input_size)
        if input_size is not None:
            _requested = int(input_size)
            _floor = _backbone_min_input_size(backbone_name)
            if _requested < _floor:
                raise ValueError(
                    f"{backbone_name}: requested input_size={_requested} is below the "
                    f"{_floor} px minimum this architecture can consume."
                )
            extractor.input_size = _requested
        return extractor

    def prepare_backbone(backbone_name, channel_policy=None, input_size=None):
        global BACKBONE, feature_extractor, CNN_FEATURE_INFO
        BACKBONE = str(backbone_name)
        _cp = channel_policy if channel_policy is not None else IMAGENET_CHANNEL_POLICY
        print("\n" + "=" * 72)
        print(f"Preparing CNN backbone: {BACKBONE} | channel_policy={_cp}")
        feature_extractor = build_backbone_extractor(
            BACKBONE, channel_policy=_cp, input_size=input_size
        ).to(device)
        feature_extractor.eval()
        CNN_FEATURE_INFO = cnn_feature_metadata(feature_extractor, BACKBONE)
        _native = int(getattr(feature_extractor, "native_input_size", feature_extractor.input_size))
        _size_txt = (
            f"{feature_extractor.input_size} (native {_native})"
            if int(feature_extractor.input_size) != _native
            else f"{feature_extractor.input_size}"
        )
        print(f"Backbone='{BACKBONE}' ready | feature_dim={feature_extractor.out_dim} "
              f"| input_size={_size_txt} "
              f"| channel_policy={feature_extractor.channel_policy} | norm={feature_extractor.norm}")
        print_cnn_feature_info(CNN_FEATURE_INFO)
        return feature_extractor, CNN_FEATURE_INFO


    print("Backbone factory ready. Scheduled backbones:", RUN_BACKBONES)


In [ ]:
# Section 2, cell 10: Feature extraction to CSV
if globals().get("RUN_CNN_LR", True):

    def feature_columns(prefix="cnn_f", n=512):
        return [f"{prefix}{i:04d}" for i in range(n)]


    def safe_filename_token(x):
        """Clean up a tag so it is safe to drop into a CSV filename."""
        return re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_").lower() or "unknown"


    def extract_features_for_pairs(pairs, feature_model, batch_size=BATCH_SIZE_FEATURES):
        patient_rows_all, slice_rows_all = [], []
        n_failures = 0
        first_error_repr = None
        fcols = feature_columns(n=feature_model.out_dim)
        feature_info = cnn_feature_metadata(feature_model, globals().get("BACKBONE", "unknown"))
        print_feature_extraction_protocol(feature_info, n_cases=len(pairs))

        feature_model.eval()
        with torch.no_grad():
            for item in tqdm(pairs, desc="Extracting CNN features"):
                try:
                    xs, meta = make_patient_2d_cnn_inputs(item["image"], item["mask"])
                    feats = []
                    for start in range(0, len(xs), batch_size):
                        # make_patient_2d_cnn_inputs returns the slices already stacked
                        # as (slices, 3, H, W); batch and move to device without restacking.
                        xb = xs[start:start + batch_size].to(device)
                        fb = feature_model(xb).detach().cpu().numpy()
                        feats.append(fb)
                    feats = np.vstack(feats)
                    mean_feat = feats.mean(axis=0)

                    base = {
                        "patient_id": item["patient_id"],
                        "canonical_id": item["canonical_id"],
                        "label": int(item["label"]),
                        "image": item["image"],
                        "mask": item["mask"],
                        "n_slices": len(meta),
                        "selected_plane_indices": ",".join(f"{m['plane']}:{m['slice_index']}" for m in meta),
                        "selected_planes": ",".join(m["plane"] for m in meta),
                    }
                    # Patient summary records axial crop diagnostics; slice-level rows
                    # contain the separate diagnostics for each plane.
                    if meta:
                        for _k in [
                            "crop_safety_margin_px", "crop_peak_occupancy",
                            "crop_height_px", "crop_width_px", "crop_height_mm", "crop_width_mm",
                            "crop_area_mm2", "crop_requested_side_mm", "crop_peak_tumor_area_mm2",
                            "crop_union_bbox_height_px", "crop_union_bbox_width_px",
                            "crop_margin_box_height_px", "crop_margin_box_width_px",
                            "crop_peak_min_side_mm", "crop_all_selected_min_side_mm",
                            "crop_physical_square_error_mm", "crop_limiting_reason",
                            "crop_peak_z_index", "n_available_tumour_slices",
                        ]:
                            base[_k] = meta[0].get(_k)
                    base.update(feature_info)
                    base.update({c: float(v) for c, v in zip(fcols, mean_feat)})
                    patient_rows_all.append(base)

                    for m, vec in zip(meta, feats):
                        out = {
                            "patient_id": item["patient_id"],
                            "canonical_id": item["canonical_id"],
                            "label": int(item["label"]),
                            "image": item["image"],
                            "mask": item["mask"],
                        }
                        out.update(m)
                        out.update(feature_info)
                        out.update({c: float(v) for c, v in zip(fcols, vec)})
                        slice_rows_all.append(out)

                except Exception as e:
                    n_failures += 1
                    if first_error_repr is None:
                        first_error_repr = repr(e)
                    print(f"[feature extraction warning] skipped patient={item.get('patient_id')} | {repr(e)}")
                    fail_path = Path(OUT_DIR) / "feature_extraction_failures.csv"
                    pd.DataFrame([{
                        "patient_id": item.get("patient_id"),
                        "image": item.get("image"),
                        "mask": item.get("mask"),
                        "error": repr(e),
                    }]).to_csv(fail_path, mode="a", index=False, header=not fail_path.exists())

        if not patient_rows_all:
            raise RuntimeError(
                f"Feature extraction produced 0 usable patient rows: all {n_failures} case(s) failed. "
                f"First error: {first_error_repr}. "
                "See feature_extraction_failures.csv in OUT_DIR for the full list."
            )
        return pd.DataFrame(patient_rows_all), pd.DataFrame(slice_rows_all)


    all_feature_output_rows = []

    for _opt_arm in PART2_OPTIMIZATION_ARMS:
        apply_part2_optimization_arm(_opt_arm)
        _opt_row_meta = {
            "optimization": bool(OPTIMIZATION),
            "opt_arm_id": (_opt_arm["arm_id"] if _opt_arm is not None else ""),
            "opt_is_reference": bool(_opt_arm.get("is_reference")) if _opt_arm is not None else False,
            "opt_input_size": int(CNN_INPUT_SIZE),
            "opt_n_slices": int(N_SLICES),
            "opt_augmentation": "NOT_APPLICABLE",
            "tumor_context_mode": str(TUMOR_CONTEXT_MODE).upper(),
            "roi_only_background_value": (
                float(ROI_ONLY_BACKGROUND_VALUE)
                if str(TUMOR_CONTEXT_MODE).upper() == "ROI_ONLY" else np.nan
            ),
        }
        for backbone_name in RUN_BACKBONES:
            # Channel policies apply only to the ImageNet/RadImageNet models; Choi-IDH always
            # takes its fixed [T1cc, T1cc, mask] input.
            _is_choi = str(backbone_name).lower() in ("choi_idh", "choi", "idh")
            _policies = ["keep"] if _is_choi else CHANNEL_POLICIES_TO_COMPARE

            for _cp in _policies:
                _base_bb_tag = (
                    safe_filename_token(backbone_name)
                    + f"__{safe_filename_token(_cp)}"
                )
                _bb_tag = ((safe_filename_token(_opt_arm["arm_id"]) + "__")
                           if _opt_arm is not None else "") + _base_bb_tag
                try:
                    feature_extractor, CNN_FEATURE_INFO = prepare_backbone(
                        backbone_name,
                        channel_policy=_cp,
                        # The arm's size has to reach the network, or the extractor's
                        # own resize undoes the compared setting. When optimization comparison is off, this stays
                        # None so every backbone keeps its native resolution.
                        input_size=(
                            int(CNN_INPUT_SIZE)
                            if (OPTIMIZATION and _opt_arm is not None) else None
                        ),
                    )
                except Exception as e:
                    err = repr(e)
                    print(f"[backbone skipped] {backbone_name} ({_cp}): {err}")
                    for _lcfg in ACTIVE_LABEL_CONFIGS:
                        all_feature_output_rows.append({
                            **_opt_row_meta,
                            "cnn_backbone": str(backbone_name),
                            "cnn_channel_policy": _cp,
                            "outcome": _lcfg["tag"],
                            "label_csv": _lcfg["path"],
                            "cnn_weight_source": cnn_weight_source(backbone_name),
                            "cnn_weight_status": "skipped_weight_load_failed",
                            "cnn_workflow": "frozen_feature_extractor_only",
                            "cnn_training_epochs": "none",
                            "cnn_learning_rate": "none",
                            "cnn_optimizer": "none",
                            "cnn_gradient_updates": "disabled_torch_no_grad",
                            "downstream_training": "not_run_for_skipped_backbone",
                            "status": "skipped",
                            "skip_reason": err,
                            "patient_csv": "",
                            "slice_csv": "",
                            "n_patients": 0,
                            "n_slice_rows": 0,
                        })
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue

                try:
                    # Features do not depend on the outcome definition, so each model runs once
                    # per channel policy and only the label column is swapped.
                    patient_features_base_df, slice_features_base_df = extract_features_for_pairs(
                        image_pairs, feature_extractor
                    )

                    # Preprocessing diagnostics are independent of the CNN backbone. This file
                    # is overwritten with the same patient-level geometry on each completed
                    # backbone so the latest run documents the crop used.
                    _diag_cols = [
                        "patient_id", "canonical_id", "n_slices", "n_available_tumour_slices",
                        "selected_plane_indices", "selected_planes",
                        "crop_safety_margin_px", "crop_peak_occupancy",
                        "crop_peak_tumor_area_mm2", "crop_area_mm2",
                        "crop_width_mm", "crop_height_mm", "crop_width_px", "crop_height_px",
                        "crop_requested_side_mm", "crop_union_bbox_height_px", "crop_union_bbox_width_px",
                        "crop_margin_box_height_px", "crop_margin_box_width_px",
                        "crop_peak_min_side_mm", "crop_all_selected_min_side_mm",
                        "crop_physical_square_error_mm", "crop_limiting_reason", "crop_peak_z_index",
                    ]
                    _diag_cols = [c for c in _diag_cols if c in patient_features_base_df.columns]
                    _diag_suffix = ("__" + safe_filename_token(_opt_arm["arm_id"])) if _opt_arm is not None else ""
                    preprocessing_diag_csv = Path(OUT_DIR) / f"min_square_crop_preprocessing_diagnostics{_diag_suffix}.csv"
                    (patient_features_base_df[_diag_cols]
                        .drop_duplicates(subset=["canonical_id"])
                        .sort_values("canonical_id", key=lambda x: x.map(patient_id_sort_key))
                        .to_csv(preprocessing_diag_csv, index=False))
                    print("[SECTION 2] Saved minimal-square crop diagnostics:", preprocessing_diag_csv)
                except Exception as e:
                    err = repr(e)
                    print(f"[feature extraction skipped] {backbone_name} ({_cp}): {err}")
                    for _lcfg in ACTIVE_LABEL_CONFIGS:
                        all_feature_output_rows.append({
                            **_opt_row_meta,
                            **CNN_FEATURE_INFO,
                            "outcome": _lcfg["tag"],
                            "label_csv": _lcfg["path"],
                            "status": "skipped",
                            "skip_reason": err,
                            "patient_csv": "",
                            "slice_csv": "",
                            "n_patients": 0,
                            "n_slice_rows": 0,
                        })
                    del feature_extractor
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue

                saved_outcome_files = []

                for _lcfg in ACTIVE_LABEL_CONFIGS:
                    _raw_ltag = str(_lcfg["tag"])
                    _ltag = safe_filename_token(_raw_ltag)
                    _ldf = all_labels_dfs.get(_raw_ltag)

                    if _ldf is None:
                        err = f"Label file for outcome={_raw_ltag} is unavailable."
                        print(f"[outcome skipped] {backbone_name} ({_cp}) | {_raw_ltag}: {err}")
                        all_feature_output_rows.append({
                            **_opt_row_meta,
                            **CNN_FEATURE_INFO,
                            "outcome": _raw_ltag,
                            "label_csv": _lcfg.get("path", ""),
                            "status": "skipped",
                            "skip_reason": err,
                            "patient_csv": "",
                            "slice_csv": "",
                            "n_patients": 0,
                            "n_slice_rows": 0,
                        })
                        continue

                    _label_df = _ldf[["canonical_id", "label"]].copy()
                    _label_df["canonical_id"] = _label_df["canonical_id"].astype(str)
                    _label_df["label"] = _label_df["label"].astype(int)

                    # Replace the placeholder label with this definition's label.
                    patient_features_df = (
                        patient_features_base_df
                        .drop(columns=["label", "outcome", "label_csv"], errors="ignore")
                        .assign(canonical_id=lambda d: d["canonical_id"].astype(str))
                        .merge(_label_df, on="canonical_id", how="inner")
                    )
                    slice_features_df = (
                        slice_features_base_df
                        .drop(columns=["label", "outcome", "label_csv"], errors="ignore")
                        .assign(canonical_id=lambda d: d["canonical_id"].astype(str))
                        .merge(_label_df, on="canonical_id", how="inner")
                    )

                    if len(patient_features_df) == 0:
                        err = f"No extracted feature rows matched outcome={_raw_ltag}."
                        print(f"[outcome skipped] {backbone_name} ({_cp}) | {_raw_ltag}: {err}")
                        all_feature_output_rows.append({
                            **_opt_row_meta,
                            **CNN_FEATURE_INFO,
                            "outcome": _raw_ltag,
                            "label_csv": _lcfg.get("path", ""),
                            "status": "skipped",
                            "skip_reason": err,
                            "patient_csv": "",
                            "slice_csv": "",
                            "n_patients": 0,
                            "n_slice_rows": 0,
                        })
                        continue

                    try:
                        dev_ids, test_ids = get_or_create_patient_split(
                            split_key=patient_split_record_key(_raw_ltag),
                            patient_ids=patient_features_df["canonical_id"],
                            context=(
                                f"Section 2 | backbone={backbone_name} | "
                                f"channel={_cp} | outcome={_raw_ltag}"
                            ),
                        )
                    except Exception as e:
                        err = f"Patient split/cohort validation failed: {e}"
                        print(f"[outcome skipped] {backbone_name} ({_cp}) | {_raw_ltag}: {err}")
                        all_feature_output_rows.append({
                            **_opt_row_meta,
                            **CNN_FEATURE_INFO,
                            "outcome": _raw_ltag,
                            "label_csv": _lcfg.get("path", ""),
                            "status": "skipped",
                            "skip_reason": err,
                            "patient_csv": "",
                            "slice_csv": "",
                            "n_patients": 0,
                            "n_slice_rows": 0,
                        })
                        continue

                    print_patient_split(
                        section=(
                            f"Section 2 | backbone={backbone_name} | "
                            f"channel={_cp} | outcome={_raw_ltag}"
                        ),
                        dev_ids=dev_ids,
                        test_ids=test_ids,
                    )

                    # Fixed patient order across every exported table.
                    split_order = {cid: i for i, cid in enumerate(dev_ids + test_ids)}
                    patient_features_df["_shared_split_order"] = patient_features_df["canonical_id"].map(split_order)
                    if patient_features_df["_shared_split_order"].isna().any():
                        raise RuntimeError("Section 2 patient table contains IDs that are not present in the shared patient-split record.")
                    patient_features_df = (
                        patient_features_df.sort_values("_shared_split_order", kind="mergesort")
                                           .drop(columns="_shared_split_order")
                                           .reset_index(drop=True)
                    )
                    slice_features_df["_shared_split_order"] = slice_features_df["canonical_id"].map(split_order)
                    if slice_features_df["_shared_split_order"].isna().any():
                        raise RuntimeError("Section 2 slice table contains IDs that are not present in the shared patient-split record.")
                    slice_features_df = (
                        slice_features_df.sort_values(["_shared_split_order", "canonical_id"], kind="mergesort")
                                         .drop(columns="_shared_split_order")
                                         .reset_index(drop=True)
                    )

                    patient_features_df.insert(0, "outcome", _raw_ltag)
                    patient_features_df.insert(1, "label_csv", _lcfg["path"])
                    slice_features_df.insert(0, "outcome", _raw_ltag)
                    slice_features_df.insert(1, "label_csv", _lcfg["path"])

                    # Patient-level CSVs are the input to the machine-learning cell and are
                    # written straight into the shared feature folder, named per definition.
                    patient_csv = Path(SEG2_FEATURE_DIR) / f"cnn_features_patient_mean__{_bb_tag}__label_{_ltag}.csv"
                    slice_csv = (
                        Path(OUT_DIR) / f"cnn_features_5slices__{_bb_tag}__label_{_ltag}.csv"
                        if SAVE_SLICE_LEVEL_FEATURE_CSVS else None
                    )
                    # CSV layout: unnamed patient-ID column, "labels", then cnn_f0000...
                    _feat_cols = [c for c in patient_features_df.columns
                                  if re.fullmatch(r"cnn_f\d+", str(c))]
                    patient_export_df = (
                        patient_features_df[["patient_id", "label"] + _feat_cols]
                        .rename(columns={"label": "labels"})
                        .set_index("patient_id")
                    )
                    patient_export_df.index.name = None      # blank header on the ID column
                    patient_export_df.to_csv(patient_csv)
                    if slice_csv is not None:
                        slice_features_df.to_csv(slice_csv, index=False)

                    print(f"Saved patient-level CNN feature table [{_raw_ltag}]:", patient_csv, patient_features_df.shape)
                    if slice_csv is not None:
                        print(f"Saved slice-level CNN feature table   [{_raw_ltag}]:", slice_csv, slice_features_df.shape)
                    else:
                        print("[SECTION 2] Slice-level CSV saving disabled; patient means are retained for the machine-learning step.")
                    print_cnn_feature_info(CNN_FEATURE_INFO)
                    print(patient_features_df[
                        ["outcome", "patient_id", "canonical_id", "label", "n_slices",
                         "cnn_backbone", "cnn_weight_source"]
                    ].head().to_string(index=False))

                    saved_outcome_files.append(patient_csv)
                    if slice_csv is not None:
                        saved_outcome_files.append(slice_csv)


                    all_feature_output_rows.append({
                        **_opt_row_meta,
                        **CNN_FEATURE_INFO,
                        "outcome": _raw_ltag,
                        "label_csv": _lcfg["path"],
                        "status": "completed",
                        "skip_reason": "",
                        "patient_csv": str(patient_csv),
                        "slice_csv": (str(slice_csv) if slice_csv is not None else ""),
                        "n_patients": int(patient_features_df.shape[0]),
                        "n_slice_rows": int(slice_features_df.shape[0]),
                    })

                # The feature folder holds feature CSVs only; diagnostic tables stay in OUT_DIR.

                del feature_extractor
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    if OPTIMIZATION:
        apply_part2_optimization_arm(None)

    feature_output_summary_df = pd.DataFrame(all_feature_output_rows)
    feature_output_summary_df["segmentation_method"] = SEGMENTATION_ANALYSIS_METHOD
    feature_output_summary_csv = Path(OUT_DIR) / "cnn_feature_output_summary__all_backbones_all_labels.csv"
    feature_output_summary_df.to_csv(feature_output_summary_csv, index=False)
    print("\nFeature extraction complete for all scheduled backbones and active outcome definitions.")
    print(feature_output_summary_df.to_string(index=False))
    skipped_feature_df = feature_output_summary_df[
        feature_output_summary_df.get("status", pd.Series(dtype=str)).eq("skipped")
    ]
    if len(skipped_feature_df) > 0:
        print("\nSKIPPED BACKBONES/OUTCOMES DURING FEATURE EXTRACTION:")
        print(skipped_feature_df[
            ["cnn_backbone", "cnn_channel_policy", "outcome", "cnn_weight_status", "skip_reason"]
        ].to_string(index=False))
    else:
        print("\nAll requested backbones and outcome definitions completed feature extraction.")
    print("Saved feature output summary:", feature_output_summary_csv)

    print(f"[SECTION 2] Persistent patient-feature folder: {SEG2_FEATURE_DIR}")


In [ ]:
# Section 2, cell 11: Feature-file handover
if globals().get("RUN_CNN_LR", True):
    import os
    from pathlib import Path

    _seg2_feature_csvs = []
    try:
        _summary = feature_output_summary_df.copy()
        if {"status", "patient_csv"}.issubset(set(_summary.columns)):
            _seg2_feature_csvs = (
                _summary.loc[
                    _summary["status"].astype(str).str.lower().eq("completed")
                    & _summary["patient_csv"].notna()
                    & _summary["patient_csv"].astype(str).str.len().gt(0),
                    "patient_csv",
                ]
                .astype(str)
                .tolist()
            )
    except Exception as _e:
        raise RuntimeError(f"[SECTION 2] cannot read feature_output_summary_df: {_e!r}") from _e

    # De-duplicate while preserving run order.
    _seen = set()
    _seg2_feature_csvs = [
        _f for _f in _seg2_feature_csvs
        if not (_f in _seen or _seen.add(_f))
    ]
    _missing = [_f for _f in _seg2_feature_csvs if not os.path.isfile(_f)]
    if _missing:
        raise RuntimeError(
            "[SECTION 2] Current-run patient feature file(s) are missing: " + repr(_missing[:20])
        )

    _wrong_root = [
        _f for _f in _seg2_feature_csvs
        if Path(_f).resolve().parent != Path(SEG2_FEATURE_DIR).resolve()
    ]
    if _wrong_root:
        raise RuntimeError(
            "[SECTION 2] Feature files escaped the single configured feature directory: "
            + repr(_wrong_root[:20])
        )

    print(f"[SECTION 2] {len(_seg2_feature_csvs)} current patient-level feature CSV(s) "
          f"ready for the machine-learning step in {SEG2_FEATURE_DIR}")

    # Exact current-run provenance for each feature file, including optimization arm.
    SEG2_FEATURE_METADATA_BY_PATH = {}
    try:
        _completed = _summary.loc[
            _summary["status"].astype(str).str.lower().eq("completed")
            & _summary["patient_csv"].notna()
        ].copy()
        for _, _r in _completed.iterrows():
            _p = os.path.abspath(str(_r["patient_csv"]))
            SEG2_FEATURE_METADATA_BY_PATH[_p] = _r.to_dict()
    except Exception as _e:
        raise RuntimeError(f"[SECTION 2] cannot build feature metadata map: {_e!r}") from _e

    RESULTS_REGISTRY['segment2_features'] = feature_output_summary_df.copy()


### Section 2, cell 12 - Classical machine learning on the extracted features

Nested cross-validation on each feature CSV: an outer loop scores the model on held-out development
patients while an inner loop tunes hyper-parameters on the remaining ones only, so no model is
tuned on the patients used to score it.

Feature selection (variance, univariate AUC, correlation and consensus voting) runs inside the
pipeline, and therefore inside each fold. Every feature file produced by the current run is
processed. A file that fails its format checks is recorded and skipped without stopping the run.
Feature files that earlier runs left in `FEATURE_INPUT_DIR` are not used.

Generated-mask analyses use thread-based parallel searches because their custom
ML components live in the notebook's separate analysis namespace. Completed models
are checkpointed after each configuration. A later model failure is recorded in
`classifier_errors.csv` and `segment3_errors.csv` without discarding completed rows.


In [ ]:
# Section 2, cell 13: Nested cross-validation and model comparison
if globals().get("RUN_CNN_LR", True):
    # Outer split scores each model, inner split tunes its hyper-parameters on different
    # patients, so settings are never tuned on the patients used to judge the model.
    # Feature selection sits inside the pipeline and is therefore refitted per fold.
    #
    # Reported per model: pooled out-of-fold AUC over all development patients, the
    # fold-mean AUC with its SD and the holdout-test metrics.


    import os
    import json
    import math
    import shutil
    import itertools
    import re
    import numpy as np
    import pandas as pd

    from dataclasses import dataclass
    from typing import Dict, Any
    from collections import Counter

    # Colab packages
    import shap

    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"

    from sklearn.base import BaseEstimator, TransformerMixin

    from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV, GridSearchCV, ParameterGrid

    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    from sklearn.feature_selection import VarianceThreshold, RFE
    from sklearn.decomposition import PCA
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.inspection import permutation_importance
    from sklearn.utils import check_random_state

    from xgboost import XGBClassifier


    from sklearn.metrics import (
        roc_auc_score, accuracy_score, balanced_accuracy_score,
        confusion_matrix, matthews_corrcoef, precision_score, recall_score, f1_score,
        brier_score_loss
    )

    from sklearn.svm import SVC
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
    from sklearn.gaussian_process import GaussianProcessClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.naive_bayes import GaussianNB
    from sklearn.neural_network import MLPClassifier
    from sklearn.ensemble import ExtraTreesClassifier, BaggingClassifier
    from sklearn.linear_model import SGDClassifier
    from sklearn.ensemble import HistGradientBoostingClassifier

    from lightgbm import LGBMClassifier
    from catboost import CatBoostClassifier


    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import RandomOverSampler, SMOTE


    from scipy.stats import kruskal
    from scipy.stats import loguniform, randint
    from IPython.display import display

    import lightgbm as lgb

    import warnings
    import time

    import joblib
    import traceback
    from contextlib import nullcontext
    from threadpoolctl import threadpool_limits
    from pathlib import Path

    warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

    # LogisticRegression below chooses its penalty with l1_ratio, which scikit-learn reads
    # in this way from release 1.8 on. An older release would ignore l1_ratio and fit an
    # L2 model without stopping, so such a release stops the run here.
    import sklearn as _sklearn
    _sklearn_release = tuple(int(_part) for _part in re.findall(r"\d+", _sklearn.__version__)[:2])
    if _sklearn_release < (1, 8):
        raise RuntimeError(
            f"scikit-learn {_sklearn.__version__} is running, but Section 2 needs release 1.8 or "
            f"newer (Section 0 installs {PACKAGE_PINS['scikit-learn']}). Run the install line in "
            "Section 0, cell 3 and restart the runtime."
        )

    # ── Master switches ─────────────────────────────────────────────────────────

    USE_UNION_FS = bool(USE_FEATURE_SELECTOR_VOTING)  # Keep features supported by several methods; set in Section 0.
    USE_POSTHOC_IMPORTANCE_FS = False   # Optional final stage: rank features, retain the top ones and retrain.
    USE_CORR_FILTER = True

    # ── 1 · User settings ───────────────────────────────────────────────────────


    OUTER_FOLDS = int(OPTIMIZATION_N_FOLDS) if OPTIMIZATION else 3
    INNER_FOLDS = 3

    OUTER_REPEATS = int(OPTIMIZATION_N_REPEATS) if OPTIMIZATION else 1
    INNER_REPEATS = 1


    # Hyper-parameter grids shorter than this are searched exhaustively; longer ones are
    # sampled with RandomizedSearchCV at this cap, under a fixed seed.
    SEARCH_MAX_CANDIDATES = 30

    # One parallel level for the inner search; estimators stay n_jobs=1 to avoid nested
    # oversubscription. One core is left free so the VM stays responsive during long
    # searches. Set -1 to use every core.
    SEARCH_NJOBS = max(1, (os.cpu_count() or 2) - 1)
    # Generated-mask classes live in an in-memory module. Process workers cannot
    # import it; threads share it and keep the same estimators, folds and grids.
    SEARCH_BACKEND = ("loky" if SEGMENTATION_ANALYSIS_METHOD == "original" else "threading")
    print(f"[SECTION 2 ML] method={SEGMENTATION_ANALYSIS_METHOD} | backend={SEARCH_BACKEND} | "
          f"SEARCH_NJOBS={SEARCH_NJOBS} of {os.cpu_count()} detected CPUs.")

    # SVM (support vector machine) kernel options:
    SVM_KERNEL_MODE = "linear+rbf"  # "linear" or "linear+rbf"

    # Class imbalance: "upsample" (duplicate minority cases), "smote" (synthesise), none.
    BALANCING_METHOD = "upsample"

    # Rescale features by: "zscore" (shift to mean 0, spread 1) or "minmax" (squash to 0-1)
    SCALING_METHOD = "zscore"

    VARIANCE_THRESHOLD = 1e-6

    # Univariate AUC filter, refitted inside each fold so held-out patients cannot influence the filter.
    USE_AUC_FILTER = True      # Main switch for the single-feature AUC filter.


    # ── Preprocessing values offered to the inner search ────────────────────────
    GLOBAL_TUNE_GRID = {}

    if USE_AUC_FILTER:
        GLOBAL_TUNE_GRID.update({
            "aucf__high": [0.58],
            "aucf__low":  [0.42],
        })

    if USE_CORR_FILTER:
        GLOBAL_TUNE_GRID.update({
            "corr__threshold": [0.98],
        })

    GLOBAL_COMBOS = list(ParameterGrid(GLOBAL_TUNE_GRID)) if GLOBAL_TUNE_GRID else [dict()]

    print(
        f"[GLOBAL_GRID] combos={len(GLOBAL_COMBOS)} | "
        f"AUC_FILTER={USE_AUC_FILTER} | CORR_FILTER={USE_CORR_FILTER} | "
        f"keys={list(GLOBAL_TUNE_GRID.keys())}"
    )


    # ── Feature-selector voting ─────────────────────────────────────────────────
    # Each enabled method nominates features; a feature is kept only with enough votes.
    # Runs after the AUC and correlation filters. Methods and vote count come from Section 0.

    UNION_FS_K = int(FEATURE_SELECTOR_TOP_K)
    UNION_FS_MIN_KEEP = int(FEATURE_SELECTOR_MIN_KEEP)

    # Internal names of the available selection methods.
    FS_METHODS_ALL = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE"]

    # Read the feature-selection combinations chosen in Section 0.
    FS_COMBOS = [tuple(combo) for combo in RUN_FEATURE_SELECTOR_COMBINATIONS]
    # Keep a feature when at least this many selected methods choose it.
    UNION_FS_MIN_VOTES = int(FEATURE_SELECTOR_MIN_VOTES)

    def _validate_fs_combo(combo):
        combo = tuple(combo)
        bad = [m for m in combo if m not in FS_METHODS_ALL]
        if bad:
            raise ValueError(f"Unknown FS methods in combo {combo}: {bad}")
        if len(combo) < 1:
            raise ValueError("FS combo cannot be empty.")
        if UNION_FS_MIN_VOTES < 1 or UNION_FS_MIN_VOTES > len(combo):
            raise ValueError(
                f"UNION_FS_MIN_VOTES must be 1..{len(combo)} for combo {combo} "
                f"(got {UNION_FS_MIN_VOTES})"
            )
        return combo

    FS_COMBOS = [_validate_fs_combo(c) for c in FS_COMBOS]
    print(f"[UNION_FS] combos={len(FS_COMBOS)} | min_votes={UNION_FS_MIN_VOTES}")

    # ========================================================================

    # AUC
    print(f"[UNION_FS] selectors voting={len(FS_METHODS_ALL)} | min_votes={UNION_FS_MIN_VOTES}")
    INVERT_LOW_AUC_FEATURES = True   # reverse low-AUC features so higher values consistently support the positive class
    MIN_FEATURES_AFTER_AUC  = 50    # keep at least this many features even if the AUC filter is strict

    MAX_FEATURES_AFTER_AUC  = 3000     # keep at most this many of the strongest features (None = no cap)


    # PCA (principal component analysis) options. Current setting: no PCA.
    PCA_VARIANTS = ["no_pca"]
    PCA_VARIANCE_GRID = [0.90, 0.95, 0.99]


    # Classifiers, read from RUN_CLASSIFIERS in Section 0.
    CLASSIFIERS = [c.upper() for c in RUN_CLASSIFIERS]
    print("Configured CLASSIFIERS:", CLASSIFIERS)

    # ── Optional final refit: rank features, keep the top ones, retrain ─────────
    TOP_IMPORTANCE_MODELS = 5        # Number of top models used for feature ranking.
    TOP_FEATURES_PER_MODEL = 35      # Number of features nominated by each model.

    POSTHOC_MIN_VOTES = 4   # keep a feature only if at least this many of those models nominate it
    POSTHOC_MAX_KEEP  = None  # optional cap on how many features to keep (None = no cap)


    # "LOGREG", "ADABOOST", "EXTRATREES", "RANDOM_FOREST"
    FINAL_REFIT_CLASSIFIERS = ["LOGREG"]  # Classifiers to retrain on the retained features.


    # ── Run settings ────────────────────────────────────────────────────────────

    _valid_classifiers = {
        "SVM", "LDA", "LOGREG", "ADABOOST", "RANDOM_FOREST", "DECISION_TREE",
        "XGBOOST", "NAIVE_BAYES", "GAUSSIAN_PROCESS", "AE",
        "EXTRATREES", "HIST_GBDT", "SGD_LOGLOSS", "LIGHTGBM", "CATBOOST", "BAGGING"
    }

    bad_c = [c for c in CLASSIFIERS if c not in _valid_classifiers]
    if bad_c:
        raise ValueError(f"Unknown classifier(s) in RUN_CLASSIFIERS: {bad_c}")



    # ── 2 · Google Drive ────────────────────────────────────────────────────────
    ensure_drive_mounted()   # connect to Google Drive once (set up in Section 0)

    # Paths
    # Folder of the feature CSVs. Only the files that the feature-extraction cells of the
    # current run produced are read from it.
    FEATURE_INPUT_DIR = SEG2_FEATURE_DIR
    # PART3_OUT_DIR and the shared evaluation helpers are defined once in Section 0.
    # Checked here, so that a missing helper stops the run at once and is not
    # reported separately for every feature file.
    for _required in ("PART3_OUT_DIR", "calculate_auc_intervals", "holdout_test_sets",
                      "holdout_evaluation_enabled", "HoldoutNotEvaluated", "rows_for_display"):
        if _required not in globals():
            raise RuntimeError(f"{_required} is missing. Run Section 0 before Section 2.")
    os.makedirs(PART3_OUT_DIR, exist_ok=True)


    def _safe_run_name(name: str, max_len: int = 120) -> str:
        name = re.sub(r"[^0-9a-zA-Z._-]+", "_", str(name)).strip("._-")
        return (name[:max_len] or "feature_file")


    # Only the feature files produced by the current feature-extraction run are modelled
    # (the list built in Section 2, cell 11). The feature folder is persistent and can
    # still hold files of earlier runs, so the folder itself is never listed here. This
    # matters most in optimization mode, where several arm-specific CSVs share the folder.
    FEATURE_INPUT_FILES = list(globals().get("_seg2_feature_csvs", []))
    if not FEATURE_INPUT_FILES:
        print("[FEATURE INPUT] The current run produced no patient-level feature CSV.")
        print("[FEATURE INPUT] The machine-learning step is skipped. Files of earlier runs "
              "in the feature folder are not used.")
    print(f"[FEATURE INPUT] Folder: {FEATURE_INPUT_DIR}")
    print(f"[FEATURE INPUT] Files queued: {len(FEATURE_INPUT_FILES)}")
    for _fp in FEATURE_INPUT_FILES:
        print(" -", os.path.basename(_fp))


    def _load_feature_table(path: str) -> pd.DataFrame:
        ext = os.path.splitext(path)[1].lower()
        if ext in (".xlsx", ".xls"):
            return pd.read_excel(path)
        if ext == ".tsv":
            return pd.read_csv(path, sep="\t")
        return pd.read_csv(path)


    class FeatureStructureError(Exception):
        """Raised when a feature CSV doesn't have the expected layout.

        The batch driver catches this, records it and moves on to the next file, so
        one malformed feature CSV never stops the whole run."""
        pass


    def _detect_label_column(df):
        """Find the label column, accepting 'labels' or 'label' (case-insensitive)."""
        lower = {str(c).strip().lower(): c for c in df.columns}
        for cand in ("labels", "label"):
            if cand in lower:
                return lower[cand]
        return None


    def validate_feature_table_structure(full_df, feature_file):
        """Validate one patient-level feature table before any modelling."""
        if full_df is None or full_df.shape[1] < 3:
            raise FeatureStructureError(
                f"{feature_file}: expected at least 3 columns (id, labels, >=1 feature); "
                f"got shape {None if full_df is None else full_df.shape}.")

        id_col = full_df.columns[0]
        label_col = _detect_label_column(full_df)
        if label_col is None:
            raise FeatureStructureError(
                f"{feature_file}: no 'labels'/'label' column found. "
                f"Columns seen: {list(map(str, full_df.columns[:10]))}.")

        try:
            canonical_ids = _canonical_unique_patient_ids(
                full_df[id_col], context=f"Section 2 ML feature table {feature_file}"
            )
        except ValueError as e:
            raise FeatureStructureError(str(e)) from e

        lab = pd.to_numeric(full_df[label_col], errors="coerce")
        if lab.isna().any():
            raise FeatureStructureError(
                f"{feature_file}: label column '{label_col}' has missing/non-numeric values.")
        if not np.allclose(lab.to_numpy(), lab.astype(int).to_numpy()):
            raise FeatureStructureError(
                f"{feature_file}: label column '{label_col}' must contain integer 0/1 values.")
        classes = set(lab.astype(int).unique().tolist())
        if classes != {0, 1}:
            raise FeatureStructureError(
                f"{feature_file}: labels must be exactly {{0, 1}}; found {sorted(classes)}.")

        feature_cols = [c for c in full_df.columns if c not in (id_col, label_col)]
        if len(feature_cols) < 1:
            raise FeatureStructureError(f"{feature_file}: no feature columns besides id and label.")
        x_numeric = full_df[feature_cols].apply(pd.to_numeric, errors="coerce")
        if int(x_numeric.notna().to_numpy().sum()) == 0:
            raise FeatureStructureError(f"{feature_file}: feature columns are not numeric.")
        _split_sets = [SPLIT_FOLDER_MEMBERSHIP.get(cid) for cid in canonical_ids]
        if SPLIT_SET_DEVELOPMENT not in _split_sets or SPLIT_SET_TEST not in _split_sets:
            raise FeatureStructureError(
                f"{feature_file}: needs patients from both the DEVELOPMENT and the TEST folder; "
                f"found development={_split_sets.count(SPLIT_SET_DEVELOPMENT)}, "
                f"test={_split_sets.count(SPLIT_SET_TEST)}.")
        return id_col, label_col, feature_cols, canonical_ids


    class CorrelationFilter(BaseEstimator, TransformerMixin):
        def __init__(self, threshold=0.95):
            self.threshold = threshold
            self.keep_idx_ = None

        def fit(self, X, y=None):
            X = np.asarray(X)
            n_features = X.shape[1]
            if n_features <= 1:
                self.keep_idx_ = np.arange(n_features)
                return self

            variances = np.nanvar(X, axis=0)
            corr = np.corrcoef(X, rowvar=False)
            corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
            abs_corr = np.abs(corr)

            keep = np.ones(n_features, dtype=bool)

            def current_mean_abs_corr(mask):
                sub = abs_corr[np.ix_(mask, mask)]
                if sub.shape[0] <= 1:
                    return np.zeros(sub.shape[0])
                return (sub.sum(axis=0) - 1.0) / (sub.shape[0] - 1)

            while True:
                idx = np.where(keep)[0]
                if len(idx) <= 1:
                    break

                sub = abs_corr[np.ix_(keep, keep)].copy()
                np.fill_diagonal(sub, 0.0)

                max_val = sub.max()
                if max_val <= self.threshold:
                    break

                a, b = np.argwhere(sub == max_val)[0]
                feat_a = idx[a]
                feat_b = idx[b]

                mean_corr = current_mean_abs_corr(keep)
                mean_a = mean_corr[a]
                mean_b = mean_corr[b]

                if mean_a > mean_b:
                    drop = feat_a
                elif mean_b > mean_a:
                    drop = feat_b
                else:
                    drop = feat_a if variances[feat_a] < variances[feat_b] else feat_b

                keep[drop] = False

            self.keep_idx_ = np.where(keep)[0]
            return self

        def transform(self, X):
            X = np.asarray(X, dtype=float)
            # Imputation happened upstream; this step only drops columns and learns
            # nothing from X, so held-out feature values cannot influence it.
            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
            return X[:, self.keep_idx_]


    class UnivariateAUCFilterProper(BaseEstimator, TransformerMixin):
        def __init__(self, high=0.51, low=0.01, invert_low=True, min_keep=50, max_keep=None):
            self.high = float(high)
            self.low = float(low)
            self.invert_low = bool(invert_low)
            self.min_keep = int(min_keep)
            self.max_keep = None if max_keep is None else int(max_keep)

            self.keep_idx_ = None
            self.invert_mask_kept_ = None
            self.auc_ = None
            self.median_ = None

        def fit(self, X, y):
            X = np.asarray(X, dtype=float)
            y = np.asarray(y).astype(int)
            n, p = X.shape

            # Medians are learned on training patients only and reused downstream.
            Xc = X.copy()
            Xc[~np.isfinite(Xc)] = np.nan
            med = np.nanmedian(Xc, axis=0)
            med = np.where(np.isfinite(med), med, 0.0)
            Xc = np.where(np.isfinite(Xc), Xc, med)
            self.median_ = med

            aucs = np.full(p, 0.5, dtype=float)

            # Score each feature on its own by AUC, counting tied values correctly.
            for j in range(p):
                col = Xc[:, j]
                if np.nanstd(col) < 1e-12:
                    continue
                try:
                    aucs[j] = roc_auc_score(y, col)
                except Exception:
                    aucs[j] = 0.5

            keep = (aucs >= self.high) | (aucs <= self.low)

            # Never drop below min_keep features (a safety floor).
            if keep.sum() < min(self.min_keep, p):
                k = min(self.min_keep, p)
                order = np.argsort(np.abs(aucs - 0.5))[::-1]
                keep = np.zeros(p, dtype=bool)
                keep[order[:k]] = True

            # Never keep more than max_keep of the strongest features (a cap).
            if self.max_keep is not None and keep.sum() > self.max_keep:
                kept_idx = np.where(keep)[0]
                strength = np.abs(aucs[kept_idx] - 0.5)   # bigger = more predictive
                order = kept_idx[np.argsort(strength)[::-1]]
                keep = np.zeros(p, dtype=bool)
                keep[order[: self.max_keep]] = True

            self.auc_ = aucs
            self.keep_idx_ = np.where(keep)[0]
            self.invert_mask_kept_ = (aucs[self.keep_idx_] <= self.low) if self.invert_low else np.zeros(len(self.keep_idx_), bool)
            return self


        def transform(self, X):
            X = np.asarray(X, dtype=float)
            Xc = X.copy()
            Xc[~np.isfinite(Xc)] = np.nan
            Xc = np.where(np.isfinite(Xc), Xc, self.median_)

            Xk = Xc[:, self.keep_idx_]
            if self.invert_low and self.invert_mask_kept_ is not None and self.invert_mask_kept_.any():
                Xk = Xk.copy()
                Xk[:, self.invert_mask_kept_] *= -1.0
            return Xk

    def _safe_finite(X):
        X = np.asarray(X, dtype=float)
        X = np.where(np.isfinite(X), X, np.nan)
        # Fill each column with its median.
        med = np.nanmedian(X, axis=0)
        med = np.where(np.isfinite(med), med, 0.0)
        X = np.where(np.isfinite(X), X, med)
        return X

    def _topk_idx(scores, k):
        scores = np.asarray(scores, dtype=float)
        scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.max(scores[np.isfinite(scores)]) if np.isfinite(scores).any() else 0.0, neginf=-np.inf)
        order = np.argsort(scores)[::-1]
        return order[:k]

    def _rfe_topk(X, y, k, random_state=GLOBAL_SEED):
        X = _safe_finite(X)
        y = np.asarray(y).astype(int)

        base = LogisticRegression(
            solver="liblinear",   # L2 penalty, the scikit-learn default (l1_ratio=0)
            C=1.0,
            max_iter=500,
            random_state=random_state,
        )
        rfe = RFE(estimator=base, n_features_to_select=min(k, X.shape[1]), step=0.2)
        rfe.fit(X, y)
        idx = np.where(rfe.get_support())[0]
        return idx[:k]

    def _lasso_l1_topk(X, y, k, random_state=GLOBAL_SEED):
        X = _safe_finite(X)
        y = np.asarray(y).astype(int)

        clf = LogisticRegression(
            solver="liblinear",
            l1_ratio=1.0,         # pure L1 penalty
            C=0.5,
            max_iter=1000,
            random_state=random_state,
        )
        clf.fit(X, y)
        coef = np.abs(clf.coef_).ravel()
        if coef.size == 0 or np.all(coef == 0):
            # If L1 kept too few features, fall back to ranking the rest by variance.
            coef = np.var(X, axis=0)
        return _topk_idx(coef, min(k, X.shape[1]))

    def _enet_topk(X, y, k, random_state=GLOBAL_SEED):
        # Elastic-net logistic regression via "saga"; steadier coefficients than pure L1.
        X = _safe_finite(X)
        y = np.asarray(y).astype(int)

        clf = LogisticRegression(
            solver="saga",
            l1_ratio=0.5,         # elastic net: half L1, half L2
            C=1.0,
            max_iter=3000,
            random_state=random_state,
        )
        clf.fit(X, y)
        coef = np.abs(clf.coef_).ravel()
        if coef.size == 0 or np.all(coef == 0):
            coef = np.var(X, axis=0)
        return _topk_idx(coef, min(k, X.shape[1]))

    def _mrmr_select(X, y, k, mode="diff", random_state=GLOBAL_SEED):
        """Minimum-redundancy maximum-relevance selection, run inside a fold.

        Prefers features with high mutual information about the outcome and low mean
        absolute correlation with those already chosen. mode:
          "diff"   MI - mean(|corr|)
          "ratio"  MI / (1 + mean(|corr|))
        """
        X = _safe_finite(X)
        y = np.asarray(y).astype(int)

        # Standardise within the fold so correlation and mutual information are comparable.
        Xs = StandardScaler().fit_transform(X)

        mi = mutual_info_classif(Xs, y, random_state=random_state)
        mi = np.nan_to_num(mi, nan=0.0, posinf=0.0, neginf=0.0)

        # Calculate absolute correlations safely.
        C = np.corrcoef(Xs, rowvar=False)
        C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
        absC = np.abs(C)
        np.fill_diagonal(absC, 0.0)

        p = Xs.shape[1]
        k = min(int(k), p)

        selected = []
        remaining = set(range(p))

        # Seed the selection with the single most informative feature.
        first = int(np.argmax(mi)) if p > 0 else 0
        selected.append(first)
        remaining.discard(first)

        while len(selected) < k and len(remaining) > 0:
            sel_arr = np.array(selected, dtype=int)

            # Redundancy: mean correlation with the already-selected features, penalised.
            red = absC[:, sel_arr].mean(axis=1) if sel_arr.size > 0 else np.zeros(p)

            if mode == "ratio":
                score = mi / (1.0 + red)
            else:
                score = mi - red

            # Do not select the same feature twice.
            score[sel_arr] = -np.inf
            nxt = int(np.argmax(score))
            if not np.isfinite(score[nxt]):
                # Fallback: if the score ties or fails, take the most informative leftover.
                rem = np.array(list(remaining), dtype=int)
                nxt = int(rem[np.argmax(mi[rem])])
            selected.append(nxt)
            remaining.discard(nxt)

        return np.array(selected[:k], dtype=int)

    def _boruta_lite_topk(X, y, k, random_state=GLOBAL_SEED, n_estimators=150):
        """Lightweight Boruta selection.

        Shuffled "shadow" copies of each column set the noise floor; a random forest then
        keeps only features whose importance beats the strongest shadow. If too few clear
        the bar, the list is topped up by raw importance.
        """
        X = _safe_finite(X)
        y = np.asarray(y).astype(int)

        n, p = X.shape
        k = min(int(k), p)

        rng = np.random.RandomState(random_state)
        X_shadow = X.copy()
        for j in range(p):
            rng.shuffle(X_shadow[:, j])

        Xb = np.hstack([X, X_shadow])

        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=random_state,
            n_jobs=1,
            class_weight="balanced_subsample",
        )
        rf.fit(Xb, y)
        imp = np.asarray(rf.feature_importances_, dtype=float)

        imp_real = imp[:p]
        imp_shadow = imp[p:]
        thr = float(np.max(imp_shadow)) if imp_shadow.size else 0.0

        keep = np.where(imp_real > thr)[0]
        if keep.size >= k:
            # Choose the strongest feature from this group.
            keep = keep[np.argsort(imp_real[keep])[::-1][:k]]
            return keep.astype(int)

        # If still short of k features, add the next most important ones.
        order = np.argsort(imp_real)[::-1]
        keep2 = np.unique(np.concatenate([keep, order[:k]]))
        # Keep exactly k features in a reproducible order.
        keep2 = keep2[np.argsort(imp_real[keep2])[::-1]]
        return keep2[:k].astype(int)


    class UnionFeatureSelector50(BaseEstimator, TransformerMixin):
        def __init__(self, k=50, min_keep=50, min_votes=2, random_state=GLOBAL_SEED, methods=("N_MRMR","mRMR","N_BORUTA","N_L1","N_ENET","RFE")):
            self.k = int(k)
            self.min_keep = int(min_keep)
            self.min_votes = int(min_votes)
            self.random_state = int(random_state)
            self.methods = tuple(methods)  # enabled selector methods
            self.keep_idx_ = None
            self.details_ = None

        def fit(self, X, y):
            X = _safe_finite(X)
            y = np.asarray(y).astype(int)

            p = X.shape[1]
            k = min(self.k, p)

            methods = tuple(self.methods)
            if len(methods) < 1:
                raise ValueError("UnionFeatureSelector50.methods cannot be empty.")
            if not (1 <= self.min_votes <= len(methods)):
                raise ValueError(f"min_votes must be 1..{len(methods)} (got {self.min_votes})")

            # Run only the selection methods switched on for this combination.
            selector_sets = {}

            if "RFE" in methods:
                idx_rfe = _rfe_topk(X, y, k, random_state=self.random_state)
                selector_sets["RFE"] = set(map(int, idx_rfe))
            else:
                idx_rfe = np.array([], dtype=int)

            if "N_MRMR" in methods:
                idx_nmrmr = _mrmr_select(X, y, k, mode="ratio", random_state=self.random_state)
                selector_sets["N_MRMR"] = set(map(int, idx_nmrmr))
            else:
                idx_nmrmr = np.array([], dtype=int)

            if "mRMR" in methods:
                idx_mrmr = _mrmr_select(X, y, k, mode="diff", random_state=self.random_state)
                selector_sets["mRMR"] = set(map(int, idx_mrmr))
            else:
                idx_mrmr = np.array([], dtype=int)

            if "N_BORUTA" in methods:
                idx_bor = _boruta_lite_topk(X, y, k, random_state=self.random_state)
                selector_sets["N_BORUTA"] = set(map(int, idx_bor))
            else:
                idx_bor = np.array([], dtype=int)

            if "N_L1" in methods:
                idx_l1 = _lasso_l1_topk(X, y, k, random_state=self.random_state)
                selector_sets["N_L1"] = set(map(int, idx_l1))
            else:
                idx_l1 = np.array([], dtype=int)

            if "N_ENET" in methods:
                idx_enet = _enet_topk(X, y, k, random_state=self.random_state)
                selector_sets["N_ENET"] = set(map(int, idx_enet))
            else:
                idx_enet = np.array([], dtype=int)

            # ── Consensus: keep a feature only with enough selector votes ───────
            vote_counter = Counter()
            for s in selector_sets.values():
                vote_counter.update(s)

            mv = int(self.min_votes)
            consensus = np.array(sorted([i for i, c in vote_counter.items() if c >= mv]), dtype=int)

            # If the agreed-on set is smaller than min_keep, top it up to that floor.
            if consensus.size < min(self.min_keep, p):
                ranked_by_votes = [i for i, c in vote_counter.most_common()]
                kept = list(consensus.tolist())

                for i in ranked_by_votes:
                    if i not in kept:
                        kept.append(int(i))
                    if len(kept) >= min(self.min_keep, p):
                        break

                if len(kept) < min(self.min_keep, p):
                    Xs = StandardScaler().fit_transform(X)
                    mi = mutual_info_classif(Xs, y, random_state=self.random_state)
                    topup = _topk_idx(mi, min(self.min_keep, p))
                    for i in topup.tolist():
                        if int(i) not in kept:
                            kept.append(int(i))
                        if len(kept) >= min(self.min_keep, p):
                            break

                consensus = np.array(sorted(set(kept)), dtype=int)

            self.keep_idx_ = consensus

            vote_dict = {int(i): int(vote_counter[i]) for i in consensus.tolist()}
            # The saved details cover only the selectors that ran.
            self.details_ = {
                "METHODS": list(methods),
                "RFE": idx_rfe.tolist() if idx_rfe.size else [],
                "N_MRMR": idx_nmrmr.tolist() if idx_nmrmr.size else [],
                "mRMR": idx_mrmr.tolist() if idx_mrmr.size else [],
                "N_BORUTA": idx_bor.tolist() if idx_bor.size else [],
                "N_L1": idx_l1.tolist() if idx_l1.size else [],
                "N_ENET": idx_enet.tolist() if idx_enet.size else [],
                "CONSENSUS_MIN_VOTES": int(self.min_votes),
                "CONSENSUS_N": int(len(consensus)),
                "CONSENSUS_VOTES": vote_dict,
            }
            return self


        def transform(self, X):
            X = np.asarray(X, dtype=float)
            # SimpleImputer ran upstream on training data only; no held-out information is used here.
            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
            return X[:, self.keep_idx_]


    @dataclass
    class ModelRunResult:
        model_id: str
        pca_variant: str
        selector: str
        classifier: str
        best_params: Dict[str, Any]
        train_metrics: Dict[str, Any]
        val_metrics: Dict[str, Any]
        test_metrics: Dict[str, Any]
        threshold: float
        paths: Dict[str, str]


    def _save_completed_ml_rows(rows, output_dir, feature_path):
        """Keep current-run completed rows in memory and atomically checkpoint them on disk."""
        table = pd.DataFrame(rows)
        table["segmentation_method"] = SEGMENTATION_ANALYSIS_METHOD
        key = os.path.abspath(str(feature_path))
        _SEGMENT3_COMPLETED_TABLES[key] = table.copy(deep=True)
        path = Path(output_dir) / "all_models_summary_metrics.csv"
        temporary = path.with_suffix(".csv.tmp")
        try:
            table.to_csv(temporary, index=False)
            os.replace(temporary, path)
        finally:
            temporary.unlink(missing_ok=True)
        return table


    def run_pipeline_for_feature_file(feature_path: str):
        feature_path = str(feature_path)
        feature_file = os.path.basename(feature_path)
        _feature_meta = dict(globals().get("SEG2_FEATURE_METADATA_BY_PATH", {}).get(
            os.path.abspath(feature_path), {}
        ))
        feature_run_name = _safe_run_name(os.path.splitext(feature_file)[0])
        OUT_DIR = os.path.join(PART3_OUT_DIR, feature_run_name)
        os.makedirs(OUT_DIR, exist_ok=True)

        print("\n" + "#" * 100)
        print(f"[FEATURE INPUT] Processing file: {feature_file}")
        print(f"[FEATURE INPUT] Output folder: {OUT_DIR}")
        print("#" * 100)

        # ── 3 · Load data and split ─────────────────────────────────────────────
        # Patients in the DEVELOPMENT folder are development; patients in the TEST folder
        # are the untouched holdout, scored once.
        full_df = _load_feature_table(feature_path)

        # A malformed file raises FeatureStructureError; the driver records it and continues.
        id_col, label_col, feature_cols, canonical_ids = validate_feature_table_structure(
            full_df, feature_file
        )
        full_df = full_df.copy()
        full_df["_canonical_id"] = canonical_ids

        split_key = infer_patient_split_key_from_feature_file(feature_file)
        train_ids, test_ids = get_or_create_patient_split(
            split_key=split_key,
            patient_ids=full_df["_canonical_id"],
            context=f"Section 2 ML | feature_file={feature_file}",
        )
        print_patient_split(
            section=f"Section 2 ML | feature_file={feature_file}",
            dev_ids=train_ids,
            test_ids=test_ids,
        )

        # Split on patient IDs, not row position, so row order cannot change the split.
        indexed = full_df.set_index("_canonical_id", verify_integrity=True)
        dev_df = indexed.loc[train_ids].copy()
        test_df = indexed.loc[test_ids].copy()

        y_train = dev_df[label_col].astype(int).to_numpy()
        y_test = test_df[label_col].astype(int).to_numpy()
        X_train = dev_df[feature_cols].apply(pd.to_numeric, errors="coerce").copy()
        X_test = test_df[feature_cols].apply(pd.to_numeric, errors="coerce").copy()
        train_ids = np.asarray(train_ids, dtype=str)
        test_ids = np.asarray(test_ids, dtype=str)
        # One entry per holdout test set (one per TEST_ROOT folder): (name, row positions
        # in X_test / y_test / test_ids). Every model is scored on each test set separately.
        _holdout_sets = [
            (_name, np.asarray(_positions, dtype=int))
            for _name, _positions in holdout_test_sets(
                test_ids, labels=y_test, context=f"Section 2 ML | feature_file={feature_file}"
            )
        ]

        feature_names = X_train.columns.tolist()

        print(f"DEVELOPMENT (DEVELOPMENT folder, N={len(train_ids)}) shape:", X_train.shape, " Pos%:", y_train.mean())
        print(f"HOLDOUT TEST (TEST folder, N={len(test_ids)}) shape:", X_test.shape, " Pos%:", y_test.mean())
        if len(_holdout_sets) > 1:
            for _name, _positions in _holdout_sets:
                print(f"  HOLDOUT TEST SET '{_name}': N={len(_positions)}  Pos%: {y_test[_positions].mean()}")
        # ── 4 · Transformers fitted on training data only ───────────────────────────────────────
        # ── 5 · Metrics and decision cut-off ────────────────────────────────────
        def optimal_threshold_youden(y_true, y_score):
            y_true = np.asarray(y_true).astype(int)
            y_score = np.asarray(y_score, dtype=float).ravel()

            # Take the distinct, valid score values.
            scores = np.unique(y_score[np.isfinite(y_score)])
            if scores.size == 0:
                return 0.0

            # Thin to 1000 quantiles when there are many distinct scores.
            if scores.size > 1000:
                scores = np.quantile(y_score, np.linspace(0, 1, 1001))
                scores = np.sort(np.unique(scores))

            # Midpoints between neighbouring scores; keep the best Youden J.
            if scores.size >= 2:
                thresholds = (scores[:-1] + scores[1:]) / 2.0
                thresholds = np.r_[scores[0] - 1e-12, thresholds, scores[-1] + 1e-12]
            else:
                thresholds = np.array([scores[0]])

            best_t = float(thresholds[0])
            best_j = -1e9

            for t in thresholds:
                y_pred = (y_score >= t).astype(int)
                tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

                sens = tp / (tp + fn) if (tp + fn) else 0.0
                spec = tn / (tn + fp) if (tn + fp) else 0.0
                j = sens + spec - 1.0

                if j > best_j:
                    best_j = j
                    best_t = float(t)

            return best_t


        def compute_metrics(
            y_true,
            y_score,
            threshold=None,
            threshold_source=None,
            use_top_risk=False,
            top_risk_percentage=None,
        ):
            y_true = np.asarray(y_true)
            y_score = np.asarray(y_score)

            auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan

            if use_top_risk:
                pct = TOP_RISK_PERCENT if top_risk_percentage is None else top_risk_percentage
                threshold = threshold_for_top_risk(y_score, percentage=pct)
                threshold_source = f"top {float(pct):g}% risk scores in evaluated set"
            else:
                if threshold is None:
                    threshold = optimal_threshold_youden(y_true, y_score)
                    if threshold_source is None:
                        threshold_source = "Youden threshold estimated from supplied scores"
                if threshold_source is None:
                    threshold_source = "explicit frozen classification threshold"

            threshold = float(threshold)
            y_pred = (y_score >= threshold).astype(int)

            acc = accuracy_score(y_true, y_pred)
            bacc = balanced_accuracy_score(y_true, y_pred)
            mcc = matthews_corrcoef(y_true, y_pred)

            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
            sens = tp / (tp + fn) if (tp + fn) else 0.0
            spec = tn / (tn + fp) if (tn + fp) else 0.0
            youden = sens + spec - 1.0

            prec = precision_score(y_true, y_pred, zero_division=0)
            rec  = recall_score(y_true, y_pred, zero_division=0)
            f1v  = f1_score(y_true, y_pred, zero_division=0)

            # Brier needs calibrated [0, 1] probabilities from predict_proba.
            if np.all(np.isfinite(y_score)) and np.all((y_score >= 0.0) & (y_score <= 1.0)):
                brier = brier_score_loss(y_true, y_score)
            else:
                brier = np.nan

            return {
                "AUC": float(auc),
                "ACC": float(acc),
                "BACC": float(bacc),
                "MCC": float(mcc),
                "SENS": float(sens),
                "SPEC": float(spec),
                "YOUDEN": float(youden),
                "PREC": float(prec),
                "RECALL": float(rec),
                "F1": float(f1v),
                "BRIER": float(brier),
                "THRESH": float(threshold),
                "THRESH_SOURCE": str(threshold_source),
                "TN": int(tn),
                "FP": int(fp),
                "FN": int(fn),
                "TP": int(tp),
            }


        def predict_score(estimator, X):
            if hasattr(estimator, "predict_proba"):
                return estimator.predict_proba(X)[:, 1]
            if hasattr(estimator, "decision_function"):
                s = estimator.decision_function(X)
                return np.asarray(s).ravel()
            return estimator.predict(X).astype(float)

        def safe_auc(y_true, y_score):
            y_true = np.asarray(y_true) if y_true is not None else None
            y_score = np.asarray(y_score)
            if y_true is None:
                return np.nan
            if len(np.unique(y_true)) != 2:
                return np.nan
            try:
                return float(roc_auc_score(y_true.astype(int), y_score.astype(float)))
            except Exception:
                return np.nan


        # ── 6 · Build components ────────────────────────────────────────────────
        def make_sampler(y_for_fold=None):
            m = (BALANCING_METHOD or "none").lower().strip()
            if m == "upsample":
                return RandomOverSampler(random_state=GLOBAL_SEED)
            if m == "smote":
                # Shrink SMOTE's k to the minority count available in this fold.
                k = 5
                if y_for_fold is not None:
                    y = np.asarray(y_for_fold).astype(int)
                    vals, counts = np.unique(y, return_counts=True)
                    if len(counts) == 2:
                        minority = int(np.min(counts))
                        k = max(1, min(5, minority - 1))
                return SMOTE(random_state=GLOBAL_SEED, k_neighbors=k)
            if m in ("none", "off", "no", ""):
                return "passthrough"
            raise ValueError("BALANCING_METHOD must be 'upsample', 'smote' or 'none'")


        def make_scaler():
            if SCALING_METHOD.lower() == "zscore":
                return StandardScaler()

            elif SCALING_METHOD.lower() == "minmax":
                return MinMaxScaler()

            else:
                raise ValueError("SCALING_METHOD must be 'zscore' or 'minmax'")


        # Build only the classifiers and selectors named above.

        from sklearn.svm import SVC
        from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
        from sklearn.linear_model import LogisticRegression
        from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
        from sklearn.gaussian_process import GaussianProcessClassifier
        from sklearn.tree import DecisionTreeClassifier
        from sklearn.neural_network import MLPClassifier

        # mRMR is provided by the selector functions above; no separate class is used.


        def classifier_and_grid(clf_name: str):
            """Return one classifier plus the exact set of settings to try for it.

            The grids are deliberately STRICT (only the listed settings are tried).
            A label of "EXACT" on a classifier below means its short list of settings
            is searched in full; classifiers with larger grids get sampled instead.
            """
            clf_name = clf_name.lower()

            # 1) AdaBoost - EXACT
            if clf_name == "adaboost":
                clf = AdaBoostClassifier(random_state=GLOBAL_SEED)
                dist = {"clf__n_estimators": [10, 50, 100]}
                return clf, dist

            # 2) AE (MLP) - EXACT
            if clf_name == "ae":
                clf = MLPClassifier(solver="adam", random_state=GLOBAL_SEED)
                dist = {
                    "clf__hidden_layer_sizes": [(30,), (100,)],
                    "clf__alpha": [0.0001, 0.001],
                    "clf__learning_rate_init": [0.001, 0.01],
                }
                return clf, dist

            # 3) Decision Tree - EXACT
            if clf_name == "decision_tree":
                clf = DecisionTreeClassifier(random_state=GLOBAL_SEED)
                dist = {
                    "clf__criterion": ["gini", "entropy"],
                    "clf__max_features": ["log2", "sqrt"],
                }
                return clf, dist

            # 4) Gaussian Process - EXACT (no tuning)
            if clf_name == "gaussian_process":
                clf = GaussianProcessClassifier()
                dist = {}
                return clf, dist


            # 5) LDA - EXACT (3 configs)
            if clf_name == "lda":
                clf = LinearDiscriminantAnalysis()
                dist = [
                    {"clf__solver": ["svd"]},
                    {"clf__solver": ["lsqr"], "clf__shrinkage": ["auto"]},
                    {"clf__solver": ["eigen"], "clf__shrinkage": ["auto"]},
                ]
                return clf, dist

            # 6) Logistic Regression - EXACT
            if clf_name == "logreg":
                clf = LogisticRegression(solver="liblinear", random_state=GLOBAL_SEED)
                dist = {
                    "clf__tol": [0.00003, 0.0003, 0.003],
                    "clf__C": [0.01, 0.1, 1.0, 5],
                    "clf__max_iter": [100, 200],
                }
                return clf, dist

            # 7) LASSO classifier (LogReg L1) - EXACT
            if clf_name == "lasso":
                clf = LogisticRegression(
                    solver="liblinear", l1_ratio=1.0, random_state=GLOBAL_SEED  # pure L1 penalty
                )
                dist = {
                    "clf__C": [0.1, 1.0],
                    "clf__tol": [0.00003, 0.0003],
                    "clf__max_iter": [100],
                }
                return clf, dist


            # 8) Random Forest - EXACT
            if clf_name == "random_forest":
                clf = RandomForestClassifier(random_state=GLOBAL_SEED, n_jobs=1)
                dist = {
                    "clf__n_estimators": [10, 50, 100],
                    "clf__criterion": ["gini", "entropy"],
                }
                return clf, dist


            # 9) SVM - kernels controlled by SVM_KERNEL_MODE
            if clf_name == "svm":
                clf = SVC(probability=False, random_state=GLOBAL_SEED)

                mode = (SVM_KERNEL_MODE or "linear").lower().strip()

                if mode == "linear":
                    dist = [{"clf__kernel": ["linear"], "clf__C": [0.1, 0.3, 1.0, 3.0]}]

                elif mode in ("linear+rbf", "rbf+linear"):
                    dist = [
                        {"clf__kernel": ["linear"], "clf__C": [0.1, 0.3, 1.0, 3.0]},
                        {"clf__kernel": ["rbf"], "clf__C": [0.1, 0.3, 1.0, 3.0], "clf__gamma": [0.1, 0.3, 1.0, 3.0]},
                    ]

                elif mode == "rbf":
                    dist = [{"clf__kernel": ["rbf"], "clf__C": [0.1, 0.3, 1.0, 3.0], "clf__gamma": [0.1, 0.3, 1.0, 3.0]}]

                elif mode == "poly":
                    dist = [{"clf__kernel": ["poly"], "clf__C": [0.1, 0.3, 1.0, 3.0], "clf__degree": [2, 3, 4]}]

                else:
                    raise ValueError(
                        f"Unknown SVM_KERNEL_MODE='{SVM_KERNEL_MODE}'. Use 'linear', 'rbf', 'poly' or 'linear+rbf'."
                    )

                return clf, dist


            # 10) XGBoost - small parameter grid
            if clf_name == "xgboost":
                clf = XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=GLOBAL_SEED,
                    n_jobs=1,
                    tree_method="hist",
                )
                dist = {
                    "clf__n_estimators": [50, 150],
                    "clf__max_depth": [2, 3, 4],
                    "clf__learning_rate": [0.03, 0.1],
                    "clf__subsample": [0.8, 1.0],
                    "clf__colsample_bytree": [0.6, 0.8, 1.0],
                    "clf__min_child_weight": [1, 5],
                    "clf__reg_lambda": [1.0, 5.0],
                }
                return clf, dist

            # 11) Naive Bayes (GaussianNB) - no parameter tuning

            if clf_name == "naive_bayes":
                clf = GaussianNB()
                dist = {}
                return clf, dist


            # 12) ExtraTrees
            if clf_name == "extratrees":
                clf = ExtraTreesClassifier(
                    random_state=GLOBAL_SEED,
                    n_jobs=1,
                )
                dist = {
                    "clf__n_estimators": [200, 600],
                    "clf__max_depth": [None, 3, 5],
                    "clf__min_samples_leaf": [1, 2, 5],
                    "clf__max_features": ["sqrt", "log2", 1.0],
                }
                return clf, dist

            # 13) HistGradientBoosting (scikit-learn)
            if clf_name in ("hist_gbdt", "histgradientboosting", "histgradientboostingclassifier"):
                clf = HistGradientBoostingClassifier(random_state=GLOBAL_SEED)
                dist = {
                    "clf__learning_rate": [0.03, 0.1],
                    "clf__max_depth": [2, 3, None],
                    "clf__max_leaf_nodes": [15, 31, 63],
                    "clf__min_samples_leaf": [5, 10, 20],
                    "clf__l2_regularization": [0.0, 0.1, 1.0],
                }
                return clf, dist

            # 14) SGDClassifier with log loss
            if clf_name in ("sgd_logloss", "sgd"):
                clf = SGDClassifier(
                    loss="log_loss",
                    penalty="elasticnet",
                    random_state=GLOBAL_SEED,
                    max_iter=3000,
                    tol=1e-3,
                )
                dist = {
                    "clf__alpha": [1e-5, 1e-4, 1e-3],
                    "clf__l1_ratio": [0.0, 0.15, 0.5, 0.85],
                    "clf__class_weight": [None, "balanced"],
                }
                return clf, dist

            # 15) LightGBM
            if clf_name in ("lightgbm", "lgbm"):
                clf = LGBMClassifier(
                    objective="binary",
                    random_state=GLOBAL_SEED,
                    n_jobs=1,
                    verbose=-1,
                    verbosity=-1,
                )
                dist = {
                    "clf__n_estimators": [200, 600],
                    "clf__learning_rate": [0.03, 0.1],
                    "clf__num_leaves": [7, 15, 31],
                    "clf__min_child_samples": [5, 10, 20],
                    "clf__subsample": [0.8, 1.0],
                    "clf__colsample_bytree": [0.8, 1.0],
                    "clf__reg_lambda": [0.0, 1.0, 5.0],
                }
                return clf, dist

            # 16) CatBoost with quiet Colab output
            if clf_name == "catboost":
                clf = CatBoostClassifier(
                    loss_function="Logloss",
                    random_seed=GLOBAL_SEED,
                    verbose=0,
                    allow_writing_files=False,
                    thread_count=1,
                )
                dist = {
                    "clf__iterations": [300, 800],
                    "clf__depth": [2, 3, 4, 5],
                    "clf__learning_rate": [0.03, 0.1],
                    "clf__l2_leaf_reg": [1.0, 3.0, 10.0],
                }
                return clf, dist

            # 17) Bagging for more stable small-data predictions
            if clf_name == "bagging":
                base = DecisionTreeClassifier(random_state=GLOBAL_SEED)
                try:
                    clf = BaggingClassifier(
                        estimator=base,
                        random_state=GLOBAL_SEED,
                        n_jobs=1,
                    )
                    est_key = "clf__estimator__"
                except TypeError:
                    clf = BaggingClassifier(
                        base_estimator=base,
                        random_state=GLOBAL_SEED,
                        n_jobs=1,
                    )
                    est_key = "clf__base_estimator__"

                dist = {
                    "clf__n_estimators": [50, 200, 500],
                    "clf__max_samples": [0.7, 1.0],
                    "clf__max_features": [0.7, 1.0],
                    "clf__bootstrap": [True, False],
                    f"{est_key}max_depth": [None, 2, 3, 4],
                    f"{est_key}min_samples_leaf": [1, 2, 5],
                }
                return clf, dist

            raise ValueError(f"Unknown classifier '{clf_name}' (strict mode).")


        # ── 7 · Pipeline ────────────────────────────────────────────────────────
        def build_pipeline(pca_variant: str, selector_name: str, clf_name: str, fs_methods=None):

            imputer = SimpleImputer(strategy="median")
            var = VarianceThreshold(threshold=VARIANCE_THRESHOLD)

            aucf = (

                UnivariateAUCFilterProper(
                    high=0.5,   # Placeholder; overridden by aucf__high.
                    low=0.5,    # Placeholder; overridden by aucf__low.
                    invert_low=INVERT_LOW_AUC_FEATURES,
                    min_keep=MIN_FEATURES_AFTER_AUC,
                    max_keep=MAX_FEATURES_AFTER_AUC,
                )
                if USE_AUC_FILTER
                else "passthrough"
            )

            scaler = make_scaler()

            # Correlation filter; its cut-off is searchable.
            corr = CorrelationFilter(threshold=0.95) if USE_CORR_FILTER else "passthrough"

            if fs_methods is None:
                fs_methods = tuple(FS_METHODS_ALL)

            fsu = (
                UnionFeatureSelector50(
                    k=UNION_FS_K,
                    min_keep=UNION_FS_MIN_KEEP,
                    min_votes=min(UNION_FS_MIN_VOTES, len(fs_methods)),  # safety
                    random_state=GLOBAL_SEED,
                    methods=tuple(fs_methods),
                )
                if USE_UNION_FS
                else "passthrough"
            )

            if pca_variant == "pca":
                pca = PCA(svd_solver="full")
            else:
                pca = "passthrough"

            clf, clf_dist = classifier_and_grid(clf_name)
            sampler = make_sampler()

            pipe = ImbPipeline(steps=[
                ("impute", imputer),
                ("var", var),
                ("aucf", aucf),
                ("scale", scaler),
                ("corr", corr),
                ("fsu", fsu),
                ("sample", sampler),
                ("pca", pca),
                ("clf", clf),
            ])


            # ---- Settings to search over for this pipeline ----
            if pca_variant == "pca":
                if isinstance(clf_dist, list):
                    dist = []
                    for d in clf_dist:
                        merged = dict(d)
                        merged.update({"pca__n_components": PCA_VARIANCE_GRID})
                        dist.append(merged)
                else:
                    dist = dict(clf_dist)
                    dist.update({"pca__n_components": PCA_VARIANCE_GRID})
                return pipe, dist

            # no_pca
            if isinstance(clf_dist, list):
                dist = []
                for d in clf_dist:
                    merged = dict(d)
                    dist.append(merged)
                return pipe, dist

            dist = dict(clf_dist) if clf_dist else {}
            return pipe, dist


        def make_search(estimator, param_grid, inner_cv):
            """Try every setting when the list is short; sample a fixed-size subset when
            it is long (so long searches stay fast and repeatable)."""
            if not param_grid:
                return None

            try:
                total_candidates = len(ParameterGrid(param_grid))
            except Exception:
                total_candidates = None

            common = dict(
                estimator=estimator,
                error_score="raise",
                scoring="roc_auc",
                cv=inner_cv,
                n_jobs=SEARCH_NJOBS,
                refit=True,
                verbose=0,
            )

            if total_candidates is not None and total_candidates <= SEARCH_MAX_CANDIDATES:
                print(f"    [SEARCH] GridSearchCV: all {total_candidates} candidate settings")
                return GridSearchCV(param_grid=param_grid, **common)

            n_iter = int(SEARCH_MAX_CANDIDATES)
            if total_candidates is not None:
                n_iter = min(n_iter, total_candidates)
                detail = f"{n_iter} of {total_candidates} candidate settings"
            else:
                detail = f"{n_iter} sampled candidate settings"
            print(f"    [SEARCH] RandomizedSearchCV: {detail}")
            return RandomizedSearchCV(
                param_distributions=param_grid,
                n_iter=n_iter,
                random_state=GLOBAL_SEED,
                **common,
            )


        def _step_support_indices(step, n_features):
            if hasattr(step, "get_support"):
                mask = step.get_support()
                return np.where(mask)[0]

            for attr in ["top_idx_", "selected_idx_", "support_", "keep_idx_"]:
                if hasattr(step, attr) and getattr(step, attr) is not None:
                    idx = np.asarray(getattr(step, attr))
                    if idx.dtype == bool:
                        return np.where(idx)[0]
                    return idx.astype(int)

            return np.arange(n_features)

        def extract_selected_feature_names(fitted_pipe, feature_names):
            names = list(feature_names)

            if hasattr(fitted_pipe, "named_steps"):
                steps = list(fitted_pipe.named_steps.items())
            elif hasattr(fitted_pipe, "steps"):
                steps = list(fitted_pipe.steps)
            else:
                return names

            for name, step in steps:
                if step is None:
                    continue
                if isinstance(step, str) and step == "passthrough":
                    continue

                if name.lower() in ["sample","sampler","balance","scaler","normalize","norm","scale","impute","clf"]:
                    continue

                if "pca" in name.lower():
                    n_comp = getattr(step, "n_components_", None)
                    if n_comp is None:
                        n_comp = getattr(step, "n_components", None)
                    if n_comp is None:
                        continue
                    try:
                        n_comp_int = int(n_comp)
                    except Exception:
                        n_comp_int = int(getattr(step, "n_components_", len(names)))
                    names = [f"PC{i+1}" for i in range(n_comp_int)]
                    continue

                idx = _step_support_indices(step, len(names))
                idx = idx[(idx >= 0) & (idx < len(names))]
                names = [names[i] for i in idx]

            return names

        def feature_names_before_step(fitted_pipe, feature_names, stop_step="fsu"):
            """Return the feature names as they are *entering* a given step (e.g. "fsu",
            the voting selector). This matters because the selector records its picks by
            position within its own input, which is already after the var/aucf/corr
            filters have run - so we need the name list at exactly that point.
            """
            names = list(feature_names)

            for name, step in fitted_pipe.steps:
                lname = name.lower()

                if lname == stop_step.lower():
                    break

                if step is None or (isinstance(step, str) and step == "passthrough"):
                    continue

                if lname in ["sample", "sampler"]:
                    break
                if lname == stop_step.lower():
                    break

                # Skip steps that neither change the feature count nor rename columns.
                if lname in ["impute", "scale", "scaler", "normalize", "norm"]:
                    continue

                if "pca" in lname:
                    n_comp = getattr(step, "n_components_", getattr(step, "n_components", None))
                    if n_comp is None:
                        continue
                    try:
                        n_comp = int(n_comp)
                    except Exception:
                        n_comp = len(names)
                    names = [f"PC{i+1}" for i in range(n_comp)]
                    continue

                idx = _step_support_indices(step, len(names))
                idx = np.asarray(idx, dtype=int)
                idx = idx[(idx >= 0) & (idx < len(names))]
                names = [names[i] for i in idx]

            return names


        def fsu_details_to_fold_df(best_est, orig_feature_names, fold_id, model_id):
            """Build a table (one per fold) of every feature that ANY voting selector
            picked, along with how many selectors voted for each.
            """
            fsu = best_est.named_steps.get("fsu", None)
            if fsu is None or (isinstance(fsu, str) and fsu == "passthrough"):
                return pd.DataFrame()

            det = getattr(fsu, "details_", None)
            if not isinstance(det, dict):
                return pd.DataFrame()

            in_names = feature_names_before_step(best_est, orig_feature_names, stop_step="fsu")

            selector_keys = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE"]
            picked_by = {}

            for sk in selector_keys:
                idx_list = det.get(sk, [])
                if not idx_list:
                    continue
                for i in idx_list:
                    i = int(i)
                    if 0 <= i < len(in_names):
                        fn = in_names[i]
                        picked_by.setdefault(fn, set()).add("RFE" if sk == "RFE" else sk)

            kept_idx = getattr(fsu, "keep_idx_", None)
            kept_feats = set()
            if kept_idx is not None:
                for i in np.asarray(kept_idx, dtype=int).tolist():
                    if 0 <= i < len(in_names):
                        kept_feats.add(in_names[i])

            rows = []
            for feat, sels in picked_by.items():
                rows.append({
                    "model_id": model_id,
                    "fold": int(fold_id),
                    "feature": feat,
                    "selector_votes": int(len(sels)),
                    "selectors": "|".join(sorted(sels)),
                    "kept_by_consensus": bool(feat in kept_feats),
                    "coef_or_importance": np.nan,
                })

            df = pd.DataFrame(rows)
            if not df.empty:
                df = df.sort_values(["kept_by_consensus", "selector_votes", "feature"],
                                    ascending=[False, False, True]).reset_index(drop=True)
            return df


        def _supports_feature_importance(clf) -> bool:
            if hasattr(clf, "feature_importances_"):
                return True
            if hasattr(clf, "coef_"):
                return True
            if clf.__class__.__name__.lower().startswith("catboost"):
                return True
            return False


        def _get_importance_scores(clf):
            """Return one importance score per feature (how much the trained classifier
            relied on it), in the same order as the features fed into the classifier.
            """
            # 1) Tree / boosting models expose importances directly (feature_importances_).
            if hasattr(clf, "feature_importances_"):
                imp = np.asarray(clf.feature_importances_, dtype=float).ravel()
                return imp

            # 2) Linear models: use the size of each coefficient as its importance.
            if hasattr(clf, "coef_"):
                coef = np.asarray(clf.coef_, dtype=float)
                if coef.ndim == 2:
                    coef = coef[0]
                return np.abs(coef).ravel()

            # 3) CatBoost
            if clf.__class__.__name__.lower().startswith("catboost"):
                try:
                    imp = np.asarray(clf.get_feature_importance(), dtype=float).ravel()
                    return imp
                except Exception:
                    pass

            return None


        def _transform_until_clf(fitted_pipe, X_df):
            """Push the data through every pipeline step up to (but not including) the
            balancer and classifier - i.e. into the exact form the classifier sees.
            Returns:
              Xmat: the transformed numbers (a NumPy array)
              feat_names: the matching feature names for those columns (best effort)
            """
            # Keep the feature names in step as the data passes through each stage.
            feat_names = feature_names_before_step(fitted_pipe, X_df.columns.tolist(), stop_step="clf")

            Xcur = X_df
            for name, step in fitted_pipe.steps:
                lname = name.lower()

                # Stop just before the balancer/classifier - we only want the inputs.
                if lname in ("sample", "sampler", "clf"):
                    break

                if step is None or (isinstance(step, str) and step == "passthrough"):
                    continue

                # After the first transform the data may be a NumPy array; handle both.
                Xcur = step.transform(Xcur)

            Xmat = np.asarray(Xcur)
            return Xmat, feat_names


        def top_features_from_model(best_est, X_train_df, k=30):
            """The model is already trained (best_est). Pull out the k features it leaned
            on most, by classifier importance.
            """
            clf = best_est.named_steps.get("clf", None)
            if clf is None or not _supports_feature_importance(clf):
                return []

            # Under PCA, importances describe components rather than original features.
            pca_step = best_est.named_steps.get("pca", None)
            if pca_step is not None and not (isinstance(pca_step, str) and pca_step == "passthrough"):
                # So skip PCA models when the goal is original-feature importance.
                return []

            # Transform the training data into the classifier's input form.
            Xmat, feat_names = _transform_until_clf(best_est, X_train_df)

            # Read the importance score for each feature from the trained classifier.
            imp = _get_importance_scores(clf)
            if imp is None:
                return []

            # Match array lengths.
            m = min(len(imp), len(feat_names))
            imp = imp[:m]
            feat_names = feat_names[:m]

            order = np.argsort(imp)[::-1]
            topk = [feat_names[i] for i in order[: min(k, len(order))]]
            return topk


        def build_final_refit_pipeline(final_clf_name: str, y_for_fit=None):
            """A stripped-down pipeline used to retrain ("refit") on an already-chosen
            set of features. Adjusts SMOTE so it still works when the smaller class is very small.
            """
            imputer = SimpleImputer(strategy="median")
            scaler = make_scaler()

            # Build a valid sampler for final refitting.
            if (BALANCING_METHOD or "").lower().strip() == "smote":
                k = 5
                if y_for_fit is not None:
                    ytmp = np.asarray(y_for_fit).astype(int)
                    vals, counts = np.unique(ytmp, return_counts=True)
                    if len(counts) == 2:
                        minority = int(np.min(counts))
                        k = max(1, min(5, minority - 1))
                sampler = SMOTE(random_state=GLOBAL_SEED, k_neighbors=k)
            else:
                sampler = make_sampler()

            clf, dist = classifier_and_grid(final_clf_name.lower())

            pipe = ImbPipeline(steps=[
                ("impute", imputer),
                ("scale", scaler),
                ("sample", sampler),
                ("clf", clf),
            ])
            return pipe, dist

        def jaccard_mean(list_of_sets):
            if len(list_of_sets) < 2:
                return 1.0
            pairs = list(itertools.combinations(list_of_sets, 2))
            vals = []
            for a, b in pairs:
                if len(a) == 0 and len(b) == 0:
                    vals.append(1.0)
                else:
                    vals.append(len(a & b) / max(1, len(a | b)))
            return float(np.mean(vals))

        def save_stability_csv(model_tag, selected_sets, out_dir):
            counts = Counter([f for s in selected_sets for f in s])
            rows = []
            k = len(selected_sets)
            for feat, c in counts.most_common():
                rows.append([feat, c, c / max(1, k)])
            df = pd.DataFrame(rows, columns=["feature", "count", "freq"])
            df.to_csv(os.path.join(out_dir, f"{model_tag}__feature_stability.csv"), index=False)

        def _print_end_summary(title: str, lines: list):
            print("\n" + "=" * 90)
            print(title)
            print("=" * 90)
            for ln in lines:
                print(ln)
            print("=" * 90 + "\n")

        def print_overall_sanity_from_results(results_df: pd.DataFrame,
                                              final_refit_classifiers: list,
                                              final_feats_count: int = None,
                                              posthoc_union_count: int = None,
                                              posthoc_consensus_count: int = None):
            """End-of-run summary: mean features surviving each step (var, aucf, corr, fsu, pca),
            mean features returned per selection method, features kept by the optional final
            refit (nominated / agreed / final) and the count the final models used.
            """
            lines = []

            # 1) Preprocessing counts for each fold
            if "sanity_counts_json" in results_df.columns and results_df["sanity_counts_json"].notna().any():
                all_rows = []
                for s in results_df["sanity_counts_json"].dropna().tolist():
                    try:
                        lst = json.loads(s) if isinstance(s, str) else s
                        if isinstance(lst, list):
                            all_rows.extend(lst)
                    except Exception:
                        pass

                if all_rows:
                    df = pd.DataFrame(all_rows)
                    for k in ["var", "aucf", "corr", "fsu", "pca"]:
                        if k in df.columns:
                            lines.append(f"[SANITY-OVERALL] mean n_features after {k}: {np.nanmean(df[k].astype(float)):.2f}")

            # 2) Feature-selector outputs for each fold
            if "fsu_counts_json" in results_df.columns and results_df["fsu_counts_json"].notna().any():
                all_rows = []
                for s in results_df["fsu_counts_json"].dropna().tolist():
                    try:
                        lst = json.loads(s) if isinstance(s, str) else s
                        if isinstance(lst, list):
                            all_rows.extend(lst)
                    except Exception:
                        pass

                if all_rows:
                    df = pd.DataFrame(all_rows)
                    keys = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE", "CONSENSUS_KEPT"]
                    present = [k for k in keys if k in df.columns]
                    if present:
                        parts = [f"{k}={np.nanmean(df[k].astype(float)):.2f}" for k in present]
                        lines.append("[FSU-OVERALL] mean features selected per fold: " + " | ".join(parts))

            # 3) Features kept by the optional end stage (rank -> vote -> keep).
            if posthoc_union_count is not None:
                lines.append(f"[POSTHOC] classifier-selected UNION features (unique across top models): {int(posthoc_union_count)}")
            if posthoc_consensus_count is not None:
                lines.append(f"[POSTHOC] classifier-selected CONSENSUS features (vote rule): {int(posthoc_consensus_count)}")
            if final_feats_count is not None:
                lines.append(f"[POSTHOC] features PASSED to final refit classifiers: {int(final_feats_count)}")

            # 4) The models retrained on those kept features.
            if final_refit_classifiers:
                lines.append(f"[FINAL REFIT] classifiers: {', '.join(final_refit_classifiers)}")

            _print_end_summary("OVERALL SANITY CHECK", lines)

        def top_features_with_scores_from_model(best_est, X_train_df, k=50):
            """Return a table of the top-k features and their importance scores.
            - Linear models: importance = the coefficient (with its sign), ranked by size.
            - Tree/boosting models: importance = feature_importances_ (or CatBoost's),
              ranked by value.
            NOTE: with PCA on, returns nothing (scores would refer to components, not
            original features).
            """
            clf = best_est.named_steps.get("clf", None)
            if clf is None or not _supports_feature_importance(clf):
                return pd.DataFrame(columns=["feature", "importance", "abs_importance"])

            # Skip PCA models here, since we need original-feature names.
            pca_step = best_est.named_steps.get("pca", None)
            if pca_step is not None and not (isinstance(pca_step, str) and pca_step == "passthrough"):
                return pd.DataFrame(columns=["feature", "importance", "abs_importance"])

            # Put the training data (and its feature names) into the classifier's input form.
            Xmat, feat_names = _transform_until_clf(best_est, X_train_df)

            # Read out the per-feature importance scores.
            imp = None

            # Tree and boosting models
            if hasattr(clf, "feature_importances_"):
                imp = np.asarray(clf.feature_importances_, dtype=float).ravel()
                raw = imp.copy()
                abs_imp = np.abs(raw)

                # Rank features by their importance value.
                order = np.argsort(raw)[::-1]

            # Linear models
            elif hasattr(clf, "coef_"):
                coef = np.asarray(clf.coef_, dtype=float)
                if coef.ndim == 2:
                    coef = coef[0]
                raw = coef.ravel()
                abs_imp = np.abs(raw)

                # Rank by coefficient size (ignoring sign).
                order = np.argsort(abs_imp)[::-1]

            # CatBoost alternative
            elif clf.__class__.__name__.lower().startswith("catboost"):
                try:
                    raw = np.asarray(clf.get_feature_importance(), dtype=float).ravel()
                    abs_imp = np.abs(raw)
                    order = np.argsort(raw)[::-1]
                except Exception:
                    return pd.DataFrame(columns=["feature", "importance", "abs_importance"])
            else:
                return pd.DataFrame(columns=["feature", "importance", "abs_importance"])

            # Match array lengths.
            m = min(len(raw), len(feat_names))
            raw = raw[:m]
            abs_imp = abs_imp[:m]
            feat_names = feat_names[:m]
            order = [i for i in order if i < m]

            top_idx = order[: min(int(k), len(order))]
            df = pd.DataFrame({
                "feature": [feat_names[i] for i in top_idx],
                "importance": [float(raw[i]) for i in top_idx],
                "abs_importance": [float(abs_imp[i]) for i in top_idx],
            })
            return df

        def compute_fold_mean_abs_shap(best_est, X_tr_df, y_tr, max_samples=200, perm_repeats=5):
            """Per-feature influence on predictions, measured on the training fold only.

            Prefers SHAP (TreeExplainer/LinearExplainer) and falls back to permutation
            importance under the same column name. Uses the raw input features on a
            subsample of the fold; no validation or test data is touched.
            """
            clf = best_est.named_steps.get("clf", None)
            if clf is None:
                return []

            # Work on a subsample of the training fold to keep this fast.
            Xuse = X_tr_df
            yuse = np.asarray(y_tr).astype(int)

            n = len(yuse)
            if n == 0:
                return []

            if n > max_samples:
                rng = check_random_state(GLOBAL_SEED)
                idx = rng.choice(n, size=max_samples, replace=False)
                Xuse = X_tr_df.iloc[idx].copy()
                yuse = yuse[idx]

            # Prefer SHAP contributions for tree and linear models.
            try:
                # SHAP is computed on the features as the classifier sees them.
                Xmat, feat_names = _transform_until_clf(best_est, Xuse)

                model_name = clf.__class__.__name__.lower()
                is_tree = hasattr(clf, "feature_importances_") or any(k in model_name for k in
                            ["forest", "tree", "xgb", "lgbm", "catboost", "histgradient"])
                is_linear = hasattr(clf, "coef_") and any(k in model_name for k in ["logistic", "sgd"])

                if is_tree:
                    explainer = shap.TreeExplainer(clf)
                    sv = explainer.shap_values(Xmat, check_additivity=False)
                elif is_linear:
                    explainer = shap.LinearExplainer(clf, Xmat)
                    sv = explainer.shap_values(Xmat)
                else:
                    sv = None

                if sv is not None:
                    # Take the positive-class array when SHAP returns one per class.
                    if isinstance(sv, list) and len(sv) >= 2:
                        sv = sv[1]
                    sv = np.asarray(sv)
                    if sv.ndim == 2:
                        mean_abs = np.mean(np.abs(sv), axis=0)
                        m = min(len(mean_abs), len(feat_names))
                        mean_abs = mean_abs[:m]
                        feat_names = feat_names[:m]
                        return [{"feature": str(feat_names[i]), "mean_abs_shap": float(mean_abs[i])} for i in range(m)]
            except Exception:
                pass

            # Otherwise fall back to permutation importance on the raw training fold.
            try:
                # Columns of Xuse are shuffled one at a time against the already-fitted model.
                r = permutation_importance(
                    best_est,
                    Xuse,
                    yuse,
                    scoring="roc_auc",
                    n_repeats=int(perm_repeats),
                    random_state=GLOBAL_SEED,
                    n_jobs=SEARCH_NJOBS   # safe: runs after search.fit returns, so not nested with
                                          # the CV search; uses the same one-core-free budget
                )

                imp = np.asarray(r.importances_mean, dtype=float)
                feat_names = list(Xuse.columns)

                # Replace any invalid (NaN/inf) scores with 0.
                m = min(len(imp), len(feat_names))
                imp = imp[:m]
                feat_names = feat_names[:m]

                # Save it under the same name as the SHAP result so both feed one column.
                return [{"feature": str(feat_names[i]), "mean_abs_shap": float(imp[i])} for i in range(m)]
            except Exception:
                return []
        # Feature count after each preprocessing step.
        def feature_counts_by_step(fitted_pipe, X_df, steps=("var","aucf","corr","fsu","pca")):
            """Return how many features remain after each named preprocessing step:
            {step_name: feature_count_after_that_step}.
            """
            Xcur = X_df
            out = {}

            for name, step in fitted_pipe.steps:
                lname = name.lower()

                if lname in ("sample", "sampler", "clf"):
                    break

                if step is None or (isinstance(step, str) and step == "passthrough"):
                    continue

                Xcur = step.transform(Xcur)

                if lname in steps:
                    out[lname] = int(np.asarray(Xcur).shape[1])

            for k in steps:
                out.setdefault(k, None)

            return out


        def pca_n_components_from_est(estimator):
            pca = estimator.named_steps.get("pca", None)
            if pca is None or isinstance(pca, str) or pca == "passthrough":
                return None
            return getattr(pca, "n_components_", getattr(pca, "n_components", None))


        # ── 8 · Nested training and evaluation ──────────────────────────────────
        # Inner loop tunes hyper-parameters; outer loop scores the tuned model on patients
        # excluded from that tuning.
        def nested_oof_predictions_for_config(X, y, pca_variant, selector_name, clf_name, model_id, feature_names, gparams):

            outer_cv = RepeatedStratifiedKFold(
                n_splits=OUTER_FOLDS,
                n_repeats=OUTER_REPEATS,
                random_state=GLOBAL_SEED
            )

            inner_cv = RepeatedStratifiedKFold(
                n_splits=INNER_FOLDS,
                n_repeats=INNER_REPEATS,
                random_state=GLOBAL_SEED
            )

            # Each patient is validated once per repeat, so predictions and thresholds are
            # averaged across repeats rather than overwritten.
            oof_score_sum = np.zeros(len(y), dtype=float)
            oof_thr_sum = np.zeros(len(y), dtype=float)
            oof_count = np.zeros(len(y), dtype=int)

            fold_info = []
            selected_sets = []

            fold_val_aucs = []

            # Gather the per-fold feature-count check results.
            fold_sanity_counts = []      # var/aucf/corr/fsu/pca per fold
            fold_fsu_counts = []         # selector list sizes + consensus kept per fold

            # Store detailed training, validation and test metrics for each fold.
            fold_metrics_rows = []

            # Selector output per fold, long form: one row per (fold, method, feature).
            fold_selector_feature_rows = []
            # Long-form rows: one row per (fold, consensus_feature) with vote count
            fold_consensus_vote_rows = []
            fold_shap_rows = []

            # Track PCA_n and selector_k for the stability summary.
            fold_pca_n = []

            # Helper: get feature names before a selected step such as "fsu".
            def _feature_names_before_step(fitted_pipe, original_feature_names, stop_step: str):
                names = list(original_feature_names)

                if not hasattr(fitted_pipe, "steps"):
                    return names

                for name, step in fitted_pipe.steps:
                    lname = name.lower()

                    # Stop before this step.
                    if lname == str(stop_step).lower():
                        break

                    if step is None or (isinstance(step, str) and step == "passthrough"):
                        continue

                    # Skip steps that do not change the feature set.
                    if lname in ("sample", "sampler", "balance", "scaler", "normalize", "norm", "scale", "impute", "clf"):
                        continue

                    # Under PCA, substitute component names; original names no longer apply.
                    if "pca" in lname:
                        n_comp = getattr(step, "n_components_", None)
                        if n_comp is None:
                            n_comp = getattr(step, "n_components", None)
                        if n_comp is None:
                            continue
                        try:
                            n_comp_int = int(n_comp)
                        except Exception:
                            n_comp_int = int(getattr(step, "n_components_", len(names)))
                        names = [f"PC{i+1}" for i in range(n_comp_int)]
                        continue

                    idx = _step_support_indices(step, len(names))
                    idx = np.asarray(idx, dtype=int)
                    idx = idx[(idx >= 0) & (idx < len(names))]
                    names = [names[i] for i in idx]

                return names

            for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X, y), start=1):
                print(f"  Outer fold {fold}/{OUTER_FOLDS}...")

                X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
                X_va, y_va = X.iloc[va_idx], y[va_idx]

                # selector_name encodes which selection methods this run combines; unpack it.
                fs_methods = tuple(selector_name.replace("FSU__", "").split("+")) if selector_name.startswith("FSU__") else tuple(FS_METHODS_ALL)

                pipe, dist = build_pipeline(pca_variant, selector_name, clf_name, fs_methods=fs_methods)
                pipe.set_params(**gparams)

                # Cap SMOTE's neighbour count to the smaller class's size in this fold.
                if (BALANCING_METHOD or "").lower().strip() == "smote":
                    pipe.set_params(sample=make_sampler(y_for_fold=y_tr))

                search = make_search(pipe, dist, inner_cv)

                if search is None:
                    best_est = pipe.fit(X_tr, y_tr)
                    best_params = {}
                else:
                    search.fit(X_tr, y_tr)
                    best_est = search.best_estimator_
                    best_params = search.best_params_

                # Record how many PCA components were used, even if no search was run.
                fold_pca_n.append(pca_n_components_from_est(best_est))


                selected_names = extract_selected_feature_names(best_est, feature_names)
                selected_sets.append(set(selected_names))

                # SHAP on the training fold only, in the classifier's input space.
                sh_rows = compute_fold_mean_abs_shap(best_est, X_tr, y_tr, max_samples=200, perm_repeats=5)
                print(f"  [SHAP] fold {fold}: clf={clf_name} | n_rows={len(sh_rows)}")


                # When PCA is off, keep just the voted-in (consensus) features.
                pca_step = best_est.named_steps.get("pca", None)
                pca_active = (pca_step is not None) and not (isinstance(pca_step, str) and pca_step == "passthrough")

                if (not pca_active) and selected_names:
                    sel_set = set(map(str, selected_names))
                    sh_rows = [r for r in sh_rows if str(r.get("feature", "")) in sel_set]

                for r in sh_rows:
                    r.update({
                        "model_id": model_id,
                        "pca": pca_variant,
                        "selector": selector_name,
                        "classifier": clf_name,
                        "fold": int(fold),
                    })
                fold_shap_rows.extend(sh_rows)


                # Check how many features remain after each preprocessing step.
                cnt = feature_counts_by_step(best_est, X_tr, steps=("var","aucf","corr","fsu","pca"))
                print(
                    "  [SANITY] n_features after: "
                    f"var={cnt['var']} | "
                    f"aucf={cnt['aucf']} | "
                    f"corr={cnt['corr']} | "
                    f"fsu={cnt['fsu']} | "
                    f"pca={cnt['pca']}"
                )


                fold_sanity_counts.append({
                    "model_id": model_id,
                    "pca_variant": pca_variant,
                    "selector": selector_name,
                    "classifier": clf_name,
                    "fold": int(fold),
                    "var": cnt.get("var", None),
                    "aucf": cnt.get("aucf", None),
                    "corr": cnt.get("corr", None),
                    "fsu": cnt.get("fsu", None),
                    "pca_n": cnt.get("pca", None),
                })

                # Compare what the correlation filter kept against what the vote kept.
                names_before_fsu = feature_names_before_step(best_est, feature_names, stop_step="fsu")

                # Features coming OUT of the voting selector, on their way to the balancer
                names_after_fsu = feature_names_before_step(best_est, feature_names, stop_step="sample")

                print("  [SANITY-ID] fsu changed features:", set(names_before_fsu) != set(names_after_fsu))
                print("  [SANITY] features before FSU:", len(names_before_fsu),
                      " after FSU:", len(names_after_fsu))

                print("  [SANITY-ID] fsu-dropped features (first 20):",
                      sorted(set(names_before_fsu) - set(names_after_fsu))[:20])

                # Per fold: each selector's picks and the consensus votes.
                fsu_step = None
                try:
                    fsu_step = best_est.named_steps.get("fsu", None)
                except Exception:
                    fsu_step = None

                if fsu_step is not None and hasattr(fsu_step, "details_") and isinstance(fsu_step.details_, dict):
                    det = fsu_step.details_
                    names_before_fsu = _feature_names_before_step(best_est, feature_names, stop_step="fsu")

                    selector_keys = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE"]
                    for ksel in selector_keys:
                        idx_list = det.get(ksel, [])
                        if not isinstance(idx_list, (list, tuple, np.ndarray)):
                            idx_list = []
                        selector_method = "RFE" if ksel == "RFE" else ksel

                        for ii in idx_list:
                            try:
                                ii = int(ii)
                            except Exception:
                                continue
                            if 0 <= ii < len(names_before_fsu):
                                fold_selector_feature_rows.append({
                                    "model_id": model_id,
                                    "pca": pca_variant,
                                    "selector": selector_name,
                                    "classifier": clf_name,
                                    "fold": int(fold),
                                    "selector_method": str(selector_method),
                                    "feature": str(names_before_fsu[ii]),
                                    "rank_within_selector": None,
                                })

                    votes_dict = det.get("CONSENSUS_VOTES", {})
                    if isinstance(votes_dict, dict):
                        for ii, vv in votes_dict.items():
                            try:
                                ii = int(ii); vv = int(vv)
                            except Exception:
                                continue
                            if 0 <= ii < len(names_before_fsu):
                                fold_consensus_vote_rows.append({
                                    "model_id": model_id,
                                    "pca": pca_variant,
                                    "selector": selector_name,
                                    "classifier": clf_name,
                                    "fold": int(fold),
                                    "feature": str(names_before_fsu[ii]),
                                    "votes": int(vv),
                                })

                    # Emit one summary row per fold (after the inner loops finish).
                    fold_fsu_counts.append({
                        "model_id": model_id,
                        "pca": pca_variant,
                        "selector": selector_name,
                        "classifier": clf_name,
                        "fold": int(fold),
                        "N_MRMR": int(len(det.get("N_MRMR", []) or [])),
                        "mRMR": int(len(det.get("mRMR", []) or [])),
                        "N_BORUTA": int(len(det.get("N_BORUTA", []) or [])),
                        "N_L1": int(len(det.get("N_L1", []) or [])),
                        "N_ENET": int(len(det.get("N_ENET", []) or [])),
                        "RFE": int(len(det.get("RFE", []) or [])),
                        "CONSENSUS_KEPT": int(len(selected_names)),
                    })
                # The agreed-on features and their vote counts, per fold.
                print(f"  [FOLD {fold}] Consensus features kept: {len(selected_names)}")
                if len(selected_names) > 0:
                    print("  [FOLD {}] Feature names: {}".format(fold, ", ".join(map(str, selected_names))))

                    # Show each consensus feature's own single-feature AUC on the training fold.
                    if any(str(n).startswith("PC") for n in selected_names):
                        print(f"  [FOLD {fold}] PCA active → skipping per-feature univariate AUC print.")
                    else:
                        print(f"  [FOLD {fold}] Univariate AUC per selected feature (TRAIN fold):")
                        for fn in selected_names:
                            try:
                                col = pd.to_numeric(X_tr[fn], errors="coerce").values.astype(float)
                                col2 = col.copy()
                                col2[~np.isfinite(col2)] = np.nan
                                med = np.nanmedian(col2)
                                if not np.isfinite(med):
                                    med = 0.0
                                col2 = np.where(np.isfinite(col2), col2, med)

                                if np.nanstd(col2) < 1e-12 or len(np.unique(y_tr)) != 2:
                                    aucv = np.nan
                                else:
                                    aucv = roc_auc_score(y_tr, col2)

                                print(f"    {fn}: {aucv:.4f}" if np.isfinite(aucv) else f"    {fn}: nan")
                            except Exception:
                                print(f"    {fn}: nan")

                # The model's predicted scores for this fold.
                tr_score = predict_score(best_est, X_tr)
                thr = optimal_threshold_youden(y_tr, tr_score)

                va_score = predict_score(best_est, X_va)

                oof_score_sum[va_idx] += va_score
                oof_count[va_idx] += 1

                # This fold's validation AUC, for the fold-mean and spread later.
                try:
                    fold_auc = roc_auc_score(y_va, va_score) if len(np.unique(y_va)) == 2 else np.nan
                except Exception:
                    fold_auc = np.nan
                fold_val_aucs.append(float(fold_auc))

                fold_info.append({"fold": fold, "best_params": best_params, "thr": thr, "val_auc": float(fold_auc)})

                # Cut-off chosen on the outer training fold, then applied to its validation fold.
                tr_m = compute_metrics(
                    y_tr, tr_score, threshold=thr,
                    threshold_source="outer-train Youden threshold",
                )
                va_m = compute_metrics(
                    y_va, va_score, threshold=thr,
                    threshold_source="outer-train Youden threshold applied to outer-validation",
                )
                thr = tr_m["THRESH"]
                fold_info[-1]["thr"] = float(thr)
                oof_thr_sum[va_idx] += float(va_m["THRESH"])

                def _pick(m, prefix):
                    return {f"{prefix}_{k}": v for k, v in m.items()}

                # Build the rows saved to fold_metrics.csv.
                sanity_counts_fold = fold_sanity_counts[-1] if len(fold_sanity_counts) else {}
                fsu_counts_fold = fold_fsu_counts[-1] if len(fold_fsu_counts) else {}

                row = {
                    "model_id": model_id,
                    "pca": pca_variant,
                    "selector": selector_name,
                    "classifier": clf_name,
                    "fold": fold,
                    "fold_thr_from_train": float(thr),
                    "fold_val_auc": float(fold_auc),

                    # Save this fold's feature counts as JSON text.
                    "sanity_counts_json": json.dumps([sanity_counts_fold], default=str),
                    "fsu_counts_json": json.dumps([fsu_counts_fold], default=str),
                }

                row.update(_pick(tr_m, "train"))
                row.update(_pick(va_m, "val"))
                fold_metrics_rows.append(row)


            expected_oof_count = int(OUTER_REPEATS)
            if np.any(oof_count != expected_oof_count):
                bad = np.where(oof_count != expected_oof_count)[0].tolist()
                raise RuntimeError(
                    f"{model_id}: invalid repeated-OOF coverage; expected {expected_oof_count} "
                    f"prediction(s) per patient, bad indices={bad[:20]}, "
                    f"observed counts={sorted(set(oof_count.tolist()))}."
                )
            oof_score = oof_score_sum / oof_count
            oof_thr = oof_thr_sum / oof_count

            # How steady the PCA component count was across the outer folds.
            pca_arr = np.asarray([np.nan if v is None else float(v) for v in fold_pca_n], dtype=float)
            pca_mean = float(np.nanmean(pca_arr)) if np.isfinite(pca_arr).any() else float("nan")
            pca_std  = float(np.nanstd(pca_arr, ddof=1)) if np.sum(np.isfinite(pca_arr)) >= 2 else float("nan")
            print(f"  [STABILITY] {model_id} | PCA_n: {pca_mean:.2f}±{pca_std:.2f}")

            # Overall out-of-fold (OOF) validation scores, each fold using its own cut-off.
            val_auc = roc_auc_score(y, oof_score) if len(np.unique(y)) == 2 else float("nan")
            y_pred = (oof_score >= oof_thr).astype(int)

            tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
            sens = tp / (tp + fn) if (tp + fn) else 0.0
            spec = tn / (tn + fp) if (tn + fp) else 0.0
            youden = sens + spec - 1.0

            fold_val_aucs_arr = np.asarray(fold_val_aucs, dtype=float)
            val_auc_mean = float(np.nanmean(fold_val_aucs_arr))
            val_auc_std  = float(np.nanstd(fold_val_aucs_arr, ddof=1)) if np.sum(~np.isnan(fold_val_aucs_arr)) >= 2 else float("nan")

            val_metrics = {
                "AUC_OOF": float(val_auc),
                "AUC_FOLD_MEAN": float(val_auc_mean),
                "AUC_FOLD_STD": float(val_auc_std),

                "PCA_N_MEAN": float(pca_mean),
                "PCA_N_STD": float(pca_std),

                "ACC_OOF": float(accuracy_score(y, y_pred)),
                "BACC_OOF": float(balanced_accuracy_score(y, y_pred)),
                "MCC_OOF": float(matthews_corrcoef(y, y_pred)),
                "YOUDEN_OOF": float(youden),

                "THRESH_NOTE": "fold-specific thresholds from outer-train",
                "TN_OOF": int(tn),
                "FP_OOF": int(fp),
                "FN_OOF": int(fn),
                "TP_OOF": int(tp),
            }

            stability = {
                "selected_features_per_fold": selected_sets,
                "stability_jaccard_mean": jaccard_mean(selected_sets),
                "stability_top": Counter([f for s in selected_sets for f in s]).most_common(30)
            }

            # Average the training and validation scores across the folds.
            fold_df = pd.DataFrame(fold_metrics_rows)
            avg_cols = [c for c in fold_df.columns if c.startswith("train_") or c.startswith("val_")]
            fold_metrics_avg = fold_df[avg_cols].mean(numeric_only=True).to_dict()

            # Hand back the per-fold selector picks for the per-model CSV files.
            return (
                oof_score,
                oof_thr,
                val_metrics,
                fold_info,
                stability,
                fold_metrics_rows,
                fold_metrics_avg,
                fold_selector_feature_rows,   # NEW
                fold_consensus_vote_rows,
                fold_shap_rows,
                fold_sanity_counts,   # per-fold feature-count checks
                fold_fsu_counts,      # NEW
            )


        def fit_final_and_test_for_config(X_train, y_train, X_test, y_test, pca_variant, selector_name, clf_name, gparams):
            inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=GLOBAL_SEED)

            fs_methods = tuple(selector_name.replace("FSU__", "").split("+")) if selector_name.startswith("FSU__") else tuple(FS_METHODS_ALL)
            pipe, dist = build_pipeline(pca_variant, selector_name, clf_name, fs_methods=fs_methods)
            pipe.set_params(**gparams)

            # SMOTE's k comes from the full training labels; per-fold labels are not available here.
            if (BALANCING_METHOD or "").lower().strip() == "smote":
                vals, counts = np.unique(np.asarray(y_train).astype(int), return_counts=True)
                if len(counts) == 2:
                    minority = int(np.min(counts))
                    k = max(1, min(5, minority - 1))
                else:
                    k = 1
                pipe.set_params(sample=SMOTE(random_state=GLOBAL_SEED, k_neighbors=k))

            search = make_search(pipe, dist, inner_cv)

            if search is None:
                best_est = pipe.fit(X_train, y_train)
                best_params = {}
            else:
                search.fit(X_train, y_train)
                best_est = search.best_estimator_
                best_params = search.best_params_

            train_score = predict_score(best_est, X_train)
            # An OPTIMIZATION run that keeps the holdout unseen makes no holdout prediction.
            test_score = predict_score(best_est, X_test) if holdout_evaluation_enabled() else None

            # In-sample cut-off, for training-side reports only.
            thr_train = optimal_threshold_youden(y_train, train_score)

            train_metrics = compute_metrics(
                y_train,
                train_score,
                threshold=thr_train,
                threshold_source="full-development in-sample Youden threshold (training report only)",
            )
            thr_train = train_metrics["THRESH"]

            # Holdout at the in-sample cut-off, for comparison only; the headline results
            # below use the out-of-fold cut-off.
            if test_score is None:
                test_metrics = HoldoutNotEvaluated(
                    THRESH=float(thr_train), THRESH_SOURCE=HOLDOUT_NOT_EVALUATED_TEXT
                )
            else:
                test_metrics = compute_metrics(
                    y_test,
                    test_score,
                    threshold=thr_train,
                    threshold_source="full-development in-sample Youden threshold applied to holdout",
                )

            return best_est, best_params, thr_train, train_score, test_score, train_metrics, test_metrics


        def _fmt_metrics(m: dict) -> str:
            # Show missing/NaN values cleanly.
            def g(k, default=np.nan):
                v = m.get(k, default)
                try:
                    return float(v)
                except Exception:
                    return np.nan

            return (
                f"AUC={g('AUC'):.3f} "
                f"BACC={g('BACC'):.3f} "
                f"MCC={g('MCC'):.3f} "
                f"YOUDEN={g('YOUDEN'):.3f}"
            )

        def _fmt_line(m: dict) -> str:
            # Lay out every metric returned by compute_metrics().
            keys = ["AUC","ACC","BACC","MCC","SENS","SPEC","YOUDEN","PREC","RECALL","F1","THRESH","TN","FP","FN","TP"]
            parts = []
            for k in keys:
                v = m.get(k, None)
                if v is None:
                    continue
                if isinstance(v, float):
                    parts.append(f"{k}={v:.3f}")
                else:
                    parts.append(f"{k}={v}")
            return " | ".join(parts)

        def show_fold_report(fold_rows, model_id: str, max_rows=50):
            df = pd.DataFrame(fold_rows).copy()
            if df.empty:
                print(f"[{model_id}] No fold rows to display.")
                return

            # Preserve the original fold order.
            df = df.sort_values("fold").reset_index(drop=True)

            # Use a compact layout.
            view_cols = [
                "fold", "fold_thr_from_train", "fold_val_auc",
                "train_AUC","train_ACC","train_BACC","train_MCC","train_SENS","train_SPEC","train_YOUDEN",
                "val_AUC","val_ACC","val_BACC","val_MCC","val_SENS","val_SPEC","val_YOUDEN",
                "train_TN","train_FP","train_FN","train_TP",
                "val_TN","val_FP","val_FN","val_TP",
            ]
            view_cols = [c for c in view_cols if c in df.columns]

            print(f"\n[{model_id}] === PER-FOLD METRICS (active threshold rule) ===")
            display(df[view_cols].head(max_rows))

            # Per-metric averages and spreads (SD) across folds.
            num_cols = [
                c for c in df.columns
                if (
                    c.startswith("train_")
                    or c.startswith("val_")
                    or c == "fold_val_auc"
                )
                and pd.api.types.is_numeric_dtype(df[c])
            ]
            agg = df[num_cols].agg(["mean", "std"]).T.reset_index().rename(columns={"index":"metric"})
            print(f"\n[{model_id}] === FOLD SUMMARY (mean±std) ===")
            display(agg)

        def show_model_report(model_id: str,
                              val_metrics: dict,
                              val_metrics_at_thr: dict,
                              train_metrics: dict,
                              test_metrics_oofthr: dict,
                              test_metrics_insamplethr: dict,
                              thr_oof: float):
            print(f"\n[{model_id}] === VALIDATION (OOF, fold-specific thresholds) ===")
            # Print the out-of-fold (OOF) validation metrics with clear labels.
            for k in ["AUC_OOF","ACC_OOF","BACC_OOF","MCC_OOF","YOUDEN_OOF","TN_OOF","FP_OOF","FN_OOF","TP_OOF",
                      "AUC_FOLD_MEAN","AUC_FOLD_STD","PCA_N_MEAN","PCA_N_STD"]:
                if k in val_metrics:
                    v = val_metrics[k]
                    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

            print(f"\n[{model_id}] === VALIDATION (OOF, pooled OOF-derived Youden threshold) ===")
            print(f"  thr_oof={thr_oof:.6f}")
            print("  " + _fmt_line(val_metrics_at_thr))

            print(f"\n[{model_id}] === TRAIN (final refit, in-sample-derived Youden threshold) ===")
            print("  " + _fmt_line(train_metrics))

            if not holdout_evaluation_enabled():
                print(f"\n[{model_id}] === TEST (holdout): not scored in this run "
                      "(OPTIMIZATION_EVALUATE_HOLDOUT is False) ===")
                return

            print(f"\n[{model_id}] === TEST (holdout, OOF-derived Youden threshold) ===")
            print("  " + _fmt_line(test_metrics_oofthr))

            print(f"\n[{model_id}] === TEST (holdout, final-refit in-sample-derived Youden threshold) ===")
            print("  " + _fmt_line(test_metrics_insamplethr))

        def make_fsu_union_summary_csv(sel_rows, vote_rows, out_csv_path, shap_rows=None):
            """Per-feature summary pooled across the outer folds.

            In:  sel_rows (fold, method, feature), vote_rows (fold, feature, votes),
                 shap_rows (fold, feature, mean_abs_shap, optional).
            Out: one row per feature with the number of folds in which any method picked it
                 and in which it survived the vote, the mean and maximum consensus votes, the
                 methods that voted for it, the per-fold breakdown and its mean importance.
            """
            df_sel = pd.DataFrame(sel_rows) if sel_rows else pd.DataFrame(columns=["fold","selector_method","feature"])
            df_vot = pd.DataFrame(vote_rows) if vote_rows else pd.DataFrame(columns=["fold","feature","votes"])

            if df_sel.empty and df_vot.empty:
                # If there is nothing to summarise, write an empty file and stop.
                pd.DataFrame(columns=[
                    "feature",
                    "n_folds_selected_any_selector",
                    "n_folds_in_consensus",
                    "mean_selector_votes_in_consensus",
                    "max_selector_votes_in_consensus",
                    "n_selectors_voted",
                    "selectors_voted",
                    "selectors_by_fold",
                    "mean_abs_shap",
                ]).to_csv(out_csv_path, index=False)
                return

            # For each feature: in which folds any selector picked it.
            if not df_sel.empty:
                df_sel["fold"] = df_sel["fold"].astype(int)
                df_sel["feature"] = df_sel["feature"].astype(str)
                df_sel["selector_method"] = df_sel["selector_method"].astype(str)

                folds_any = (
                    df_sel.groupby("feature")["fold"]
                    .nunique()
                    .rename("n_folds_selected_any_selector")
                    .reset_index()
                )

                # Combine, across folds, every selector that ever picked the feature.
                selectors_union = (
                    df_sel.groupby("feature")["selector_method"]
                    .apply(lambda s: "|".join(sorted(set(s))))
                    .rename("selectors_voted")
                    .reset_index()
                )
                n_selectors_union = (
                    df_sel.groupby("feature")["selector_method"]
                    .nunique()
                    .rename("n_selectors_voted")
                    .reset_index()
                )

                # Per-fold selector list, e.g. F1:N_MRMR|RFE (fold 1 picked by N_MRMR and RFE).
                by_fold = (
                    df_sel.groupby(["feature","fold"])["selector_method"]
                    .apply(lambda s: "|".join(sorted(set(s))))
                    .reset_index()
                )
                selectors_by_fold = (
                    by_fold.groupby("feature")[["fold", "selector_method"]]
                    .apply(lambda d: ";".join([f"F{int(r.fold)}:{r.selector_method}" for r in d.itertuples(index=False)]))
                    .rename("selectors_by_fold")
                    .reset_index()
                )
            else:
                folds_any = pd.DataFrame(columns=["feature","n_folds_selected_any_selector"])
                selectors_union = pd.DataFrame(columns=["feature","selectors_voted"])
                n_selectors_union = pd.DataFrame(columns=["feature","n_selectors_voted"])
                selectors_by_fold = pd.DataFrame(columns=["feature","selectors_by_fold"])

            # For each feature: how often it survived the vote, and its vote counts.
            if not df_vot.empty:
                df_vot["fold"] = df_vot["fold"].astype(int)
                df_vot["feature"] = df_vot["feature"].astype(str)
                df_vot["votes"] = df_vot["votes"].astype(int)

                folds_cons = (
                    df_vot.groupby("feature")["fold"]
                    .nunique()
                    .rename("n_folds_in_consensus")
                    .reset_index()
                )

                vote_mean = (
                    df_vot.groupby("feature")["votes"]
                    .mean()
                    .rename("mean_selector_votes_in_consensus")
                    .reset_index()
                )
                vote_max = (
                    df_vot.groupby("feature")["votes"]
                    .max()
                    .rename("max_selector_votes_in_consensus")
                    .reset_index()
                )
            else:
                folds_cons = pd.DataFrame(columns=["feature","n_folds_in_consensus"])
                vote_mean = pd.DataFrame(columns=["feature","mean_selector_votes_in_consensus"])
                vote_max  = pd.DataFrame(columns=["feature","max_selector_votes_in_consensus"])

            # Join the separate tables into one.
            # Begin from the full list of features seen anywhere.
            all_feats = pd.Series(
                pd.concat([
                    df_sel["feature"] if "feature" in df_sel.columns else pd.Series(dtype=str),
                    df_vot["feature"] if "feature" in df_vot.columns else pd.Series(dtype=str),
                ], ignore_index=True).unique(),
                name="feature"
            ).to_frame()

            out = (all_feats
                   .merge(folds_any, on="feature", how="left")
                   .merge(folds_cons, on="feature", how="left")
                   .merge(vote_mean, on="feature", how="left")
                   .merge(vote_max,  on="feature", how="left")
                   .merge(n_selectors_union, on="feature", how="left")
                   .merge(selectors_union,  on="feature", how="left")
                   .merge(selectors_by_fold, on="feature", how="left")
                  )

            # Fill any gaps with sensible defaults.
            out["n_folds_selected_any_selector"] = out["n_folds_selected_any_selector"].fillna(0).astype(int)
            out["n_folds_in_consensus"] = out["n_folds_in_consensus"].fillna(0).astype(int)
            out["mean_selector_votes_in_consensus"] = out["mean_selector_votes_in_consensus"].fillna(np.nan)
            out["max_selector_votes_in_consensus"]  = out["max_selector_votes_in_consensus"].fillna(np.nan)
            out["n_selectors_voted"] = out["n_selectors_voted"].fillna(0).astype(int)
            out["selectors_voted"] = out["selectors_voted"].fillna("")
            out["selectors_by_fold"] = out["selectors_by_fold"].fillna("")


            # Add the optional importance (SHAP) column if that data was supplied.
            if shap_rows is not None:
                df_sh = pd.DataFrame(shap_rows)
                if not df_sh.empty:
                    df_sh["feature"] = df_sh["feature"].astype(str)
                    sh_agg = (df_sh.groupby("feature")["mean_abs_shap"]
                              .mean()
                              .rename("mean_abs_shap")
                              .reset_index())
                    out = out.merge(sh_agg, on="feature", how="left")
                else:
                    out["mean_abs_shap"] = np.nan

            # Rank by vote survival, number of methods and number of folds.
            sort_cols = ["n_folds_in_consensus", "max_selector_votes_in_consensus",
                         "n_selectors_voted", "n_folds_selected_any_selector", "feature"]
            asc = [False, False, False, False, True]
            out = out.sort_values(sort_cols, ascending=asc).reset_index(drop=True)

            out.to_csv(out_csv_path, index=False)


        # Save one model per classifier, fitted on all development patients.
        TARGET_FROZEN_CLASSIFIERS = [
            "NAIVE_BAYES",
            "GAUSSIAN_PROCESS",
            "EXTRATREES",
            "LOGREG",
            "SGD_LOGLOSS",
        ]

        def save_frozen_single_model(
            save_path,
            fitted_pipe,
            classifier_name,
            model_id,
            feature_columns,
            threshold,
            best_params,
            global_params,
            val_auc_mean=None,
            val_auc_std=None,
            val_mcc=None,
        ):
            pkg = {
                "classifier_name": str(classifier_name),
                "model_id": str(model_id),
                "pipeline": fitted_pipe,                    # FULL fitted pipeline
                "feature_columns": list(feature_columns),   # raw input columns expected at inference
                "threshold": float(threshold),
                "best_params": best_params if isinstance(best_params, dict) else {},
                "global_params": global_params if isinstance(global_params, dict) else {},
                "val_auc_mean": None if val_auc_mean is None else float(val_auc_mean),
                "val_auc_std": None if val_auc_std is None else float(val_auc_std),
                "val_mcc": None if val_mcc is None else float(val_mcc),
            }
            joblib.dump(pkg, save_path)
            print(f"[FROZEN MODEL] Saved {classifier_name} -> {save_path}")


        def coerce_best_params_field(x):
            if isinstance(x, dict):
                return {k: _coerce_param_value(v) for k, v in x.items()}
            if isinstance(x, str):
                try:
                    d = json.loads(x)
                    if isinstance(d, dict):
                        return {k: _coerce_param_value(v) for k, v in d.items()}
                except Exception:
                    pass
            return {}


        def _validation_rank_key(row):
            """Replicate the validation-only winner ordering without looking at test metrics."""
            def _finite_or(v, fallback):
                try:
                    v = float(v)
                    return v if np.isfinite(v) else fallback
                except Exception:
                    return fallback
            # Existing ordering: AUC mean high, AUC SD low, MCC high, pooled OOF AUC high.
            return (
                _finite_or(row.get("val_AUC_mean"), -np.inf),
                -_finite_or(row.get("val_AUC_std"), np.inf),
                _finite_or(row.get("val_MCC"), -np.inf),
                _finite_or(row.get("val_oof_AUC"), -np.inf),
            )


        def save_best_already_fitted_model_per_classifier(
            best_fitted_models,
            out_dir,
            target_classifiers,
        ):
            """Save the winning full-development fit from the main loop WITHOUT retraining it."""
            frozen_dir = os.path.join(out_dir, "FROZEN_SINGLE_MODELS")
            os.makedirs(frozen_dir, exist_ok=True)
            frozen_summary_rows = []

            for clf_name in target_classifiers:
                candidate = best_fitted_models.get(clf_name)
                if candidate is None:
                    print(f"[FROZEN MODEL] No already-fitted candidate found for classifier: {clf_name}")
                    continue

                fitted_pipe = candidate["estimator"]
                best_row = candidate["row"]
                best_params = candidate["best_params"]
                global_params = candidate["global_params"]
                thr_train = float(candidate["train_threshold"])
                model_id = best_row["model_id"]

                save_path = os.path.join(frozen_dir, f"{clf_name}__frozen_model.joblib")
                save_frozen_single_model(
                    save_path=save_path,
                    fitted_pipe=fitted_pipe,
                    classifier_name=clf_name,
                    model_id=model_id,
                    feature_columns=feature_names,
                    threshold=thr_train,
                    best_params=best_params,
                    global_params=global_params,
                    val_auc_mean=best_row.get("val_AUC_mean", np.nan),
                    val_auc_std=best_row.get("val_AUC_std", np.nan),
                    val_mcc=best_row.get("val_MCC", np.nan),
                )

                frozen_summary_rows.append({
                    "feature_file": feature_file,
                    "feature_path": feature_path,
                    "classifier": clf_name,
                    "model_id": model_id,
                    "save_path": save_path,
                    "threshold": thr_train,
                    "val_AUC_mean": best_row.get("val_AUC_mean", np.nan),
                    "val_AUC_std": best_row.get("val_AUC_std", np.nan),
                    "val_MCC": best_row.get("val_MCC", np.nan),
                    "best_params": json.dumps(best_params, default=str),
                    "global_params": json.dumps(global_params, default=str),
                    "source": "already-fitted full-development model from main evaluation loop; no second refit",
                })

            frozen_summary_df = pd.DataFrame(frozen_summary_rows)
            frozen_summary_path = os.path.join(frozen_dir, "frozen_models_summary.csv")
            frozen_summary_df.to_csv(frozen_summary_path, index=False)
            print(f"[FROZEN MODEL] Summary saved without redundant refitting: {frozen_summary_path}")
            return frozen_summary_df, frozen_dir

        # ── 9 · Run every selector set x classifier x setting ───────────────────
        def n_features_before_clf(fitted_pipe, X_sample_df):
            """Count how many features are left after preprocessing and selection,
            just before the balancer/classifier.
            """
            Xcur = X_sample_df
            for name, step in fitted_pipe.steps:
                if name.lower() in ("sample", "sampler", "clf"):
                    break
                if step is None or (isinstance(step, str) and step == "passthrough"):
                    continue
                Xcur = step.transform(Xcur)
            return int(np.asarray(Xcur).shape[1])


        Xtr = X_train.copy()
        Xte = X_test.copy()
        summary_rows = []
        _model_errors = []
        _save_completed_ml_rows(summary_rows, OUT_DIR, feature_path)
        _best_fitted_models = {}  # keep only the validation-best already-fitted model per classifier

        TOTAL_MODELS = len(CLASSIFIERS) * len(PCA_VARIANTS) * len(GLOBAL_COMBOS) * (len(FS_COMBOS) if USE_UNION_FS else 1)
        model_counter = 0

        for pca_variant in PCA_VARIANTS:
            for clf_name in CLASSIFIERS:
                for gi, gparams in enumerate(GLOBAL_COMBOS, start=1):
                    # When the voting selector is off, use a single "do nothing" setting.
                    combos = FS_COMBOS if USE_UNION_FS else [()]

                    for ci, combo in enumerate(combos, start=1):
                        selector_name = "FSU__" + "+".join(combo) if combo else "NONE"
                        model_id = f"{pca_variant}__{clf_name}__G{gi:02d}__S{ci:02d}"
                        model_counter += 1
                        print(f"\n=== Running ({model_counter}/{TOTAL_MODELS}): {model_id} ===")
                        print("GLOBAL:", gparams)
                        print("FS_COMBO:", combo)

                        _rows_before_model = len(summary_rows)
                        _model_rng_before = rng_state_snapshot()
                        try:
                            oof_score, oof_thr, val_metrics, fold_info, stability, fold_rows, fold_avg, sel_rows, vote_rows, shap_rows, sanity_counts, fsu_counts = (
                                nested_oof_predictions_for_config(
                                    Xtr, y_train, pca_variant, selector_name, clf_name,
                                    model_id, feature_names, gparams,
                                )
                            )

                            best_est, best_params, thr, train_score, test_score, train_metrics, _pooled_test_metrics = (
                                fit_final_and_test_for_config(
                                    Xtr, y_train, Xte, y_test, pca_variant, selector_name, clf_name, gparams
                                )
                            )
                            # False in an OPTIMIZATION run that keeps the holdout unseen.
                            _holdout_scored = test_score is not None
                            if _holdout_scored and len(_holdout_sets) > 1:
                                # Score each holdout test set on its own. A test set's scores then do
                                # not depend, even in the last decimal place, on which other test
                                # patients are scored in the same call, and are exactly those of a
                                # run in which it is the only test set.
                                test_score = np.asarray(test_score, dtype=float).copy()
                                for _test_set_name, _test_positions in _holdout_sets:
                                    test_score[_test_positions] = predict_score(best_est, Xte.iloc[_test_positions])

                            print(f"[{model_id}] PCA_n={pca_n_components_from_est(best_est)}")

                            # Primary: Youden cut-off from development OOF predictions only,
                            # applied unchanged to the holdout.
                            thr_oof = optimal_threshold_youden(y_train, oof_score)
                            val_metrics_at_thr = compute_metrics(
                                y_train,
                                oof_score,
                                threshold=thr_oof,
                                threshold_source="development pooled OOF Youden threshold",
                            )
                            thr_oof = val_metrics_at_thr["THRESH"]

                            thr_dev_oof = thr_oof

                            # Secondary top-risk view of the development OOF predictions.
                            val_metrics_toprisk = None
                            if top_risk_mode_enabled():
                                val_metrics_toprisk = compute_metrics(
                                    y_train,
                                    oof_score,
                                    use_top_risk=True,
                                    top_risk_percentage=TOP_RISK_PERCENT,
                                )

                            # Print the detailed in-notebook report for this model.
                            show_fold_report(fold_rows, model_id=model_id)

                            print(
                                f"[{model_id}] VAL@OOF-YOUDEN(thr={thr_oof:.4f}) "
                                f"AUC={val_metrics_at_thr['AUC']:.3f} "
                                f"BACC={val_metrics_at_thr['BACC']:.3f} "
                                f"MCC={val_metrics_at_thr['MCC']:.3f} "
                                f"YOUDEN={val_metrics_at_thr['YOUDEN']:.3f}"
                            )

                            # Write this model's output files.
                            model_dir = os.path.join(OUT_DIR, model_id)
                            os.makedirs(model_dir, exist_ok=True)

                            # 1) The two detailed per-fold CSVs.
                            if sel_rows:
                                pd.DataFrame(sel_rows).to_csv(
                                    os.path.join(model_dir, "FSU_selector_outputs_long.csv"),
                                    index=False
                                )

                            if vote_rows:
                                pd.DataFrame(vote_rows).to_csv(
                                    os.path.join(model_dir, "FSU_consensus_votes_long.csv"),
                                    index=False
                                )

                            # Third CSV when voting-selector data exists, with SHAP importances.
                            if shap_rows and len(shap_rows) > 0:
                                pd.DataFrame(shap_rows).to_csv(
                                    os.path.join(model_dir, "FSU_shap_meanabs_long.csv"),
                                    index=False
                                )

                            union_summary_path = os.path.join(model_dir, "FSU_union_features_summary.csv")
                            make_fsu_union_summary_csv(
                                sel_rows=sel_rows,
                                vote_rows=vote_rows,
                                out_csv_path=union_summary_path,
                                shap_rows=shap_rows,
                            )
                            print("Saved:", union_summary_path)

                            # 3) The remaining outputs (everything not tied to the vote records).
                            pd.DataFrame(fold_rows).to_csv(os.path.join(model_dir, "fold_metrics.csv"), index=False)
                            pd.DataFrame([{"feature_file": feature_file, "model_id": model_id, **fold_avg}]).to_csv(
                                os.path.join(model_dir, "fold_metrics_avg.csv"), index=False
                            )

                            pd.DataFrame({"feature_file": feature_file, "patient_id": train_ids, "label": y_train, "score": train_score}).to_csv(
                                os.path.join(model_dir, "train_predictions.csv"), index=False
                            )
                            pd.DataFrame(
                                {"feature_file": feature_file, "patient_id": train_ids, "label": y_train, "score": oof_score, "fold_threshold": oof_thr}
                            ).to_csv(os.path.join(model_dir, "val_oof_predictions.csv"), index=False)

                            # The model fitted above on all development patients is scored on each holdout
                            # test set separately: one report, one prediction file and one result row per
                            # test set. With a single test set this loop runs once. When the holdout is
                            # not scored, the loop writes the same rows with empty holdout values.
                            _test_results_by_set = {}
                            for _test_set_name, _test_positions in _holdout_sets:
                                y_test_set = y_test[_test_positions]
                                test_score_set = np.asarray(test_score)[_test_positions] if _holdout_scored else None
                                test_ids_set = test_ids[_test_positions]
                                # Scoring an additional test set must not move the random-number generators,
                                # so later results are exactly those of a run with one test set.
                                _rng_state = rng_state_snapshot() if _test_set_name != _holdout_sets[0][0] else None
                                # Without holdout scoring every pass would print the same development
                                # report, so only the first pass prints it.
                                _report_this_pass = _holdout_scored or _test_set_name == _holdout_sets[0][0]
                                if _holdout_scored and len(_holdout_sets) > 1:
                                    print(f"\n[{model_id}] ── Holdout test set '{_test_set_name}' ({len(y_test_set)} patients) ──")

                                # Holdout at the in-sample cut-off, for comparison only; the headline results
                                # below use the out-of-fold cut-off.
                                if _holdout_scored:
                                    test_metrics = compute_metrics(
                                        y_test_set,
                                        test_score_set,
                                        threshold=thr,
                                        threshold_source="full-development in-sample Youden threshold applied to holdout",
                                    )
                                else:
                                    test_metrics = HoldoutNotEvaluated(
                                        THRESH=float(thr), THRESH_SOURCE=HOLDOUT_NOT_EVALUATED_TEXT
                                    )

                                if _report_this_pass:
                                    print(
                                        f"[{model_id}] AUCs: "
                                        f"TRAIN={train_metrics['AUC']:.3f} | "
                                        + (f"TEST={test_metrics['AUC']:.3f}" if _holdout_scored
                                           else "TEST=not scored in this run")
                                    )

                                if _holdout_scored:
                                    test_metrics_oofthr = compute_metrics(
                                        y_test_set,
                                        test_score_set,
                                        threshold=thr_dev_oof,
                                        threshold_source="development pooled OOF Youden threshold applied unchanged to holdout",
                                    )
                                else:
                                    test_metrics_oofthr = HoldoutNotEvaluated(
                                        THRESH=float(thr_dev_oof), THRESH_SOURCE=HOLDOUT_NOT_EVALUATED_TEXT
                                    )
                                active_test_threshold = test_metrics_oofthr["THRESH"]

                                # Secondary top-risk view, reported alongside the primary analysis.
                                test_metrics_toprisk = None
                                if top_risk_mode_enabled() and _holdout_scored:
                                    test_metrics_toprisk = compute_metrics(
                                        y_test_set,
                                        test_score_set,
                                        use_top_risk=True,
                                        top_risk_percentage=TOP_RISK_PERCENT,
                                    )

                                # AUC intervals from the holdout scores directly, without retraining:
                                # DeLong, delete-one jackknife, patient-level bootstrap.
                                if _holdout_scored:
                                    test_auc_intervals = calculate_auc_intervals(
                                        y_test_set,
                                        test_score_set,
                                    )
                                else:
                                    test_auc_intervals = HoldoutNotEvaluated()


                                if _report_this_pass:
                                    show_model_report(
                                        model_id=model_id,
                                        val_metrics=val_metrics,
                                        val_metrics_at_thr=val_metrics_at_thr,
                                        train_metrics=train_metrics,
                                        test_metrics_oofthr=test_metrics_oofthr,
                                        test_metrics_insamplethr=test_metrics,
                                        thr_oof=thr_oof,
                                    )

                                if _holdout_scored:
                                    print(
                                        f"[{model_id}] TEST@OOF-YOUDEN(thr={active_test_threshold:.4f}) "
                                        f"AUC={test_metrics_oofthr['AUC']:.3f} "
                                        f"ACC={test_metrics_oofthr['ACC']:.3f} "
                                        f"BACC={test_metrics_oofthr['BACC']:.3f} "
                                        f"MCC={test_metrics_oofthr['MCC']:.3f} "
                                        f"YOUDEN={test_metrics_oofthr['YOUDEN']:.3f}"
                                    )
                                    print(
                                        f"[{model_id}] TEST@IN-SAMPLE-YOUDEN(thr={test_metrics['THRESH']:.4f}) "
                                        f"AUC={test_metrics['AUC']:.3f} "
                                        f"ACC={test_metrics['ACC']:.3f} "
                                        f"BACC={test_metrics['BACC']:.3f} "
                                        f"MCC={test_metrics['MCC']:.3f} "
                                        f"YOUDEN={test_metrics['YOUDEN']:.3f}"
                                    )
                                    if test_metrics_toprisk is not None:
                                        print(
                                            f"[{model_id}] TEST@TOP-{float(TOP_RISK_PERCENT):g}%-RISK"
                                            f"(thr={test_metrics_toprisk['THRESH']:.4f}) "
                                            f"AUC={test_metrics_toprisk['AUC']:.3f} "
                                            f"ACC={test_metrics_toprisk['ACC']:.3f} "
                                            f"BACC={test_metrics_toprisk['BACC']:.3f} "
                                            f"MCC={test_metrics_toprisk['MCC']:.3f} "
                                            f"YOUDEN={test_metrics_toprisk['YOUDEN']:.3f}"
                                        )
                                    print(
                                        f"[{model_id}] HOLDOUT AUC CIs | "
                                        f"DeLong={test_auc_intervals['AUC_with_DeLong_95CI']} | "
                                        f"Jackknife={test_auc_intervals['AUC_with_Jackknife_95CI']} | "
                                        f"Bootstrap={test_auc_intervals['AUC_with_Bootstrap_95CI']}"
                                    )

                                # Holdout predictions exist only when the holdout was scored.
                                if _holdout_scored:
                                    _test_predictions_export = {
                                        "feature_file": feature_file,
                                        "test_set": _test_set_name,
                                        "patient_id": test_ids_set,
                                        "label": y_test_set,
                                        "score": test_score_set,

                                        # Headline cut-off, from the pooled development OOF predictions.
                                        "predicted_label_oof_youden": (
                                            np.asarray(test_score_set) >= active_test_threshold
                                        ).astype(int),
                                        "classification_threshold_oof_youden": float(active_test_threshold),
                                        "threshold_source_oof_youden": test_metrics_oofthr["THRESH_SOURCE"],

                                        # Comparison-only cut-off, from the refit model's in-sample predictions.
                                        "predicted_label_insample_youden": (
                                            np.asarray(test_score_set) >= test_metrics["THRESH"]
                                        ).astype(int),
                                        "classification_threshold_insample_youden": float(test_metrics["THRESH"]),
                                        "threshold_source_insample_youden": test_metrics["THRESH_SOURCE"],
                                    }
                                    if test_metrics_toprisk is not None:
                                        _test_predictions_export.update({
                                            "predicted_label_top_risk": (
                                                np.asarray(test_score_set) >= test_metrics_toprisk["THRESH"]
                                            ).astype(int),
                                            "classification_threshold_top_risk": float(test_metrics_toprisk["THRESH"]),
                                            "threshold_source_top_risk": test_metrics_toprisk["THRESH_SOURCE"],
                                        })
                                    pd.DataFrame(_test_predictions_export).to_csv(
                                        os.path.join(model_dir, f"test_predictions{test_set_file_tag(_test_set_name)}.csv"), index=False
                                    )

                                with open(os.path.join(model_dir, "meta.json"), "w") as f:
                                    json.dump(
                                        {
                                            "feature_file": feature_file,
                                            "feature_path": feature_path,
                                            "model_id": model_id,
                                            "pca_variant": pca_variant,
                                            "selector": selector_name,
                                            "classifier": clf_name,
                                            "best_params": best_params,
                                            "outer_fold_info": fold_info,
                                            "threshold_from_full_train": thr,
                                            "threshold_oof_youden": float(active_test_threshold),
                                            "threshold_oof_youden_source": test_metrics_oofthr["THRESH_SOURCE"],
                                            "threshold_insample_youden": float(test_metrics["THRESH"]),
                                            "threshold_insample_youden_source": test_metrics["THRESH_SOURCE"],
                                            "feature_stability": {
                                            "jaccard_mean": stability["stability_jaccard_mean"],
                                            "top": stability["stability_top"],
                                            },
                                        },
                                        f,
                                        indent=2,
                                    )

                                row = {
                                    "feature_file": feature_file,
                                    "feature_path": feature_path,
                                    "model_id": model_id,
                                    "pca": pca_variant,
                                    "selector": selector_name,
                                    "classifier": clf_name,
                                    "best_params": json.dumps(best_params, default=str),
                                    "global_params": json.dumps(gparams, default=str),
                                    "global_combo_index": int(gi),

                                    # Image-representation optimization provenance from feature extraction.
                                    "optimization": bool(OPTIMIZATION),
                                    "opt_arm_id": str(_feature_meta.get("opt_arm_id", "")),
                                    "opt_is_reference": bool(_feature_meta.get("opt_is_reference", False)),
                                    "opt_input_size": _feature_meta.get("opt_input_size", np.nan),
                                    "opt_n_slices": _feature_meta.get("opt_n_slices", np.nan),
                                    "opt_augmentation": str(_feature_meta.get("opt_augmentation", "NOT_APPLICABLE")),
                                    "tumor_context_mode": str(_feature_meta.get("tumor_context_mode", "")),
                                    "roi_only_background_value": _feature_meta.get("roi_only_background_value", np.nan),
                                    "cnn_backbone": str(_feature_meta.get("cnn_backbone", "")),
                                    "cnn_channel_policy": str(_feature_meta.get("cnn_channel_policy", "")),
                                    "outcome": str(_feature_meta.get("outcome", "")),
                                    "test_set": _test_set_name,

                                    "thr_train": train_metrics["THRESH"],

                                    "train_full_AUC": train_metrics["AUC"],
                                    "train_ACC": train_metrics["ACC"],
                                    "train_BACC": train_metrics["BACC"],
                                    "train_MCC": train_metrics["MCC"],
                                    "train_YOUDEN": train_metrics["YOUDEN"],

                                    "val_oof_AUC": val_metrics["AUC_OOF"],
                                    "val_AUC_mean": val_metrics["AUC_FOLD_MEAN"],
                                    "val_AUC_std": val_metrics["AUC_FOLD_STD"],

                                    "pca_n_mean": val_metrics.get("PCA_N_MEAN", np.nan),
                                    "pca_n_std":  val_metrics.get("PCA_N_STD",  np.nan),

                                    "val_ACC": val_metrics["ACC_OOF"],
                                    "val_BACC": val_metrics["BACC_OOF"],
                                    "val_MCC": val_metrics["MCC_OOF"],
                                    "val_YOUDEN": val_metrics["YOUDEN_OOF"],

                                    # Holdout scores at that comparison (in-sample) cut-off.
                                    "test_AUC": test_metrics["AUC"],
                                    "test_ACC": test_metrics["ACC"],
                                    "test_BACC": test_metrics["BACC"],
                                    "test_MCC": test_metrics["MCC"],
                                    "test_YOUDEN": test_metrics["YOUDEN"],

                                    "test_TN": test_metrics["TN"],
                                    "test_FP": test_metrics["FP"],
                                    "test_FN": test_metrics["FN"],
                                    "test_TP": test_metrics["TP"],

                                    # Holdout AUC and its uncertainty need no cut-off.
                                    "test_DeLong_CI_lower": test_auc_intervals["DeLong_CI_lower"],
                                    "test_DeLong_CI_upper": test_auc_intervals["DeLong_CI_upper"],
                                    "test_AUC_with_DeLong_95CI": test_auc_intervals["AUC_with_DeLong_95CI"],
                                    "test_Jackknife_CI_lower": test_auc_intervals["Jackknife_CI_lower"],
                                    "test_Jackknife_CI_upper": test_auc_intervals["Jackknife_CI_upper"],
                                    "test_AUC_with_Jackknife_95CI": test_auc_intervals["AUC_with_Jackknife_95CI"],
                                    "test_Jackknife_min_AUC": test_auc_intervals["Jackknife_min_AUC"],
                                    "test_Jackknife_max_AUC": test_auc_intervals["Jackknife_max_AUC"],
                                    "test_Bootstrap_CI_lower": test_auc_intervals["Bootstrap_CI_lower"],
                                    "test_Bootstrap_CI_upper": test_auc_intervals["Bootstrap_CI_upper"],
                                    "test_AUC_with_Bootstrap_95CI": test_auc_intervals["AUC_with_Bootstrap_95CI"],
                                    "test_Bootstrap_repeats": test_auc_intervals["Bootstrap_repeats"],

                                    # Brier is threshold-free and reported once.
                                    "test_Brier": test_metrics_oofthr["BRIER"],
                                    "n_test_patients": int(len(y_test_set)),

                                    # Holdout metrics at the main cut-off (the development OOF Youden cut-off).
                                    "test_oof_threshold": float(active_test_threshold),
                                    "test_oof_threshold_source": test_metrics_oofthr["THRESH_SOURCE"],
                                    "test_oof_accuracy": test_metrics_oofthr["ACC"],
                                    "test_oof_balanced_accuracy": test_metrics_oofthr["BACC"],
                                    "test_oof_sensitivity": test_metrics_oofthr["SENS"],
                                    "test_oof_specificity": test_metrics_oofthr["SPEC"],
                                    "test_oof_F1": test_metrics_oofthr["F1"],
                                    "test_oof_MCC": test_metrics_oofthr["MCC"],
                                    "test_oof_YOUDEN": test_metrics_oofthr["YOUDEN"],

                                    # Holdout metrics at the in-sample comparison cut-off.
                                    "test_insample_threshold": float(test_metrics["THRESH"]),
                                    "test_insample_threshold_source": test_metrics["THRESH_SOURCE"],
                                    "test_insample_accuracy": test_metrics["ACC"],
                                    "test_insample_balanced_accuracy": test_metrics["BACC"],
                                    "test_insample_sensitivity": test_metrics["SENS"],
                                    "test_insample_specificity": test_metrics["SPEC"],
                                    "test_insample_F1": test_metrics["F1"],
                                    "test_insample_MCC": test_metrics["MCC"],
                                    "test_insample_YOUDEN": test_metrics["YOUDEN"],

                                    # Mirror the primary metrics into the generic column names.
                                    "test_accuracy": test_metrics_oofthr["ACC"],
                                    "test_balanced_accuracy": test_metrics_oofthr["BACC"],
                                    "test_sensitivity": test_metrics_oofthr["SENS"],
                                    "test_specificity": test_metrics_oofthr["SPEC"],
                                    "test_F1": test_metrics_oofthr["F1"],
                                    "test_MCC_publication": test_metrics_oofthr["MCC"],
                                    "classification_threshold": float(active_test_threshold),
                                    "threshold_source": test_metrics_oofthr["THRESH_SOURCE"],

                                    # The extra top-risk view (flag the highest-scoring share).
                                    "toprisk_percent": float(TOP_RISK_PERCENT) if test_metrics_toprisk is not None else np.nan,
                                    "toprisk_threshold": (
                                        test_metrics_toprisk["THRESH"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_accuracy": (
                                        test_metrics_toprisk["ACC"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_balanced_accuracy": (
                                        test_metrics_toprisk["BACC"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_sensitivity": (
                                        test_metrics_toprisk["SENS"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_specificity": (
                                        test_metrics_toprisk["SPEC"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_F1": (
                                        test_metrics_toprisk["F1"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_MCC": (
                                        test_metrics_toprisk["MCC"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_Brier": (
                                        test_metrics_toprisk["BRIER"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "test_toprisk_YOUDEN": (
                                        test_metrics_toprisk["YOUDEN"] if test_metrics_toprisk is not None else np.nan
                                    ),
                                    "toprisk_threshold_source": (
                                        test_metrics_toprisk["THRESH_SOURCE"] if test_metrics_toprisk is not None else ""
                                    ),

                                    "val_toprisk_threshold": (
                                        val_metrics_toprisk["THRESH"] if val_metrics_toprisk is not None else np.nan
                                    ),
                                    "val_toprisk_accuracy": (
                                        val_metrics_toprisk["ACC"] if val_metrics_toprisk is not None else np.nan
                                    ),
                                    "val_toprisk_balanced_accuracy": (
                                        val_metrics_toprisk["BACC"] if val_metrics_toprisk is not None else np.nan
                                    ),
                                    "val_toprisk_MCC": (
                                        val_metrics_toprisk["MCC"] if val_metrics_toprisk is not None else np.nan
                                    ),

                                    "fold_metrics_csv": os.path.join(model_dir, "fold_metrics.csv"),
                                    "fold_metrics_avg_csv": os.path.join(model_dir, "fold_metrics_avg.csv"),
                                    "stability_jaccard_mean": stability["stability_jaccard_mean"],
                                    "paths_dir": model_dir,
                                    "thr_dev_oof": float(thr_dev_oof),
                                    "active_test_threshold": float(active_test_threshold),

                                    "test_ACC_oofthr": test_metrics_oofthr["ACC"],
                                    "test_BACC_oofthr": test_metrics_oofthr["BACC"],
                                    "test_SENS_oofthr": test_metrics_oofthr["SENS"],
                                    "test_SPEC_oofthr": test_metrics_oofthr["SPEC"],
                                    "test_F1_oofthr": test_metrics_oofthr["F1"],
                                    "test_MCC_oofthr": test_metrics_oofthr["MCC"],
                                    "test_BRIER_oofthr": test_metrics_oofthr["BRIER"],
                                    "test_YOUDEN_oofthr": test_metrics_oofthr["YOUDEN"],
                                                    }
                                _test_results_by_set[_test_set_name] = {
                                    _k: _v for _k, _v in row.items() if str(_k).startswith("test_")
                                }
                                if OPTIMIZATION and row.get("opt_arm_id"):
                                    _ml_key = (
                                        str(row["opt_arm_id"]),
                                        str(row.get("cnn_backbone", "")),
                                        str(row.get("outcome", "")),
                                        str(row.get("cnn_channel_policy", "")),
                                        str(model_id),
                                    )
                                    OPTIMIZATION_ML_OOF_STORE[_ml_key] = {
                                        "probs": {str(_pid): float(_p) for _pid, _p in zip(train_ids, oof_score)},
                                        "labels": {str(_pid): int(_l) for _pid, _l in zip(train_ids, y_train)},
                                        "oof_auc": float(val_metrics["AUC_OOF"]),
                                        "is_reference": bool(row.get("opt_is_reference", False)),
                                        "input_size": int(row.get("opt_input_size")),
                                        "n_slices": int(row.get("opt_n_slices")),
                                        "augmentation": "NOT_APPLICABLE",
                                        "context_mode": str(row.get("tumor_context_mode", "")),
                                        "roi_background": row.get("roi_only_background_value", np.nan),
                                        "cnn_backbone": str(row.get("cnn_backbone", "")),
                                        "channel_policy": str(row.get("cnn_channel_policy", "")),
                                        "outcome": str(row.get("outcome", "")),
                                        "model_id": str(model_id),
                                        "classifier": str(clf_name),
                                        # Holdout results of the first test set, and of every test set by name.
                                        "test_results": dict(next(iter(_test_results_by_set.values()))),
                                        "test_results_by_test_set": {_n: dict(_r) for _n, _r in _test_results_by_set.items()},
                                    }

                                summary_rows.append(row)
                                rng_state_restore(_rng_state)

                            # Keep the validation-best full-development fit; this exact estimator
                            # is the one saved, never refitted.
                            _candidate = {
                                "rank_key": _validation_rank_key(row),
                                "estimator": best_est,
                                "row": row.copy(),
                                "best_params": dict(best_params),
                                "global_params": dict(gparams),
                                "train_threshold": float(thr),
                            }
                            _current = _best_fitted_models.get(clf_name)
                            if _current is None or _candidate["rank_key"] > _current["rank_key"]:
                                _best_fitted_models[clf_name] = _candidate
                        except Exception as _model_exc:
                            # Never publish a partly evaluated model/test-set group.
                            del summary_rows[_rows_before_model:]
                            rng_state_restore(_model_rng_before)
                            _error_row = {
                                "feature_file": feature_file,
                                "feature_path": feature_path,
                                "segmentation_method": SEGMENTATION_ANALYSIS_METHOD,
                                "model_id": model_id,
                                "classifier": clf_name,
                                "status": "failed model",
                                "error": repr(_model_exc),
                                "traceback": traceback.format_exc(),
                            }
                            _model_errors.append(_error_row)
                            pd.DataFrame(_model_errors).to_csv(
                                os.path.join(OUT_DIR, "classifier_errors.csv"), index=False
                            )
                            print(f"[SECTION 2 ML][{SEGMENTATION_ANALYSIS_METHOD}] Model failed: "
                                  f"{model_id}: {_model_exc!r}. Completed models are retained.")
                        # Save after each model, including failures, so prior rows survive.
                        _save_completed_ml_rows(summary_rows, OUT_DIR, feature_path)


        def _coerce_param_value(v):
            if isinstance(v, str):
                s = v.strip()
                if s.lower() == "true": return True
                if s.lower() == "false": return False
                if s.lower() in ("none", "null"): return None
                if s.isdigit() or (s.startswith("-") and s[1:].isdigit()):
                    try: return int(s)
                    except Exception: pass
                try: return float(s)
                except Exception: return v
            return v

        # ── After all models ────────────────────────────────────────────────────
        results_df = _save_completed_ml_rows(summary_rows, OUT_DIR, feature_path)
        results_df.attrs["model_errors"] = list(_model_errors)

        results_path = os.path.join(OUT_DIR, "all_models_summary_metrics.csv")
        print("\nSaved model-level metrics summary:", results_path)

        # Save the validation-best estimator per classifier; already fitted above, so no
        # second refit.
        frozen_summary_df, frozen_models_dir = save_best_already_fitted_model_per_classifier(
            best_fitted_models=_best_fitted_models,
            out_dir=OUT_DIR,
            target_classifiers=TARGET_FROZEN_CLASSIFIERS,
        )

        print("\n[FROZEN MODEL] Final frozen single-classifier models:")
        display(frozen_summary_df)

        # ── Main results table for this feature file ────────────────────────────
        # Written before the optional final refit, so a failure there cannot lose it.
        cols = [
            "feature_file",
            "test_set",
            "model_id",
            "classifier",
            "pca",
            "global_combo_index",
            "global_params",
            "best_params",
            "val_AUC_mean",
            "val_AUC_std",
            "val_ACC",
            "val_MCC",
            "val_YOUDEN",
            "test_AUC",
            "test_DeLong_CI_lower",
            "test_DeLong_CI_upper",
            "test_AUC_with_DeLong_95CI",
            "test_Jackknife_CI_lower",
            "test_Jackknife_CI_upper",
            "test_AUC_with_Jackknife_95CI",
            "test_Jackknife_min_AUC",
            "test_Jackknife_max_AUC",
            "test_Brier",
            "test_oof_threshold",
            "test_oof_accuracy",
            "test_oof_balanced_accuracy",
            "test_oof_sensitivity",
            "test_oof_specificity",
            "test_oof_F1",
            "test_oof_MCC",
            "test_oof_YOUDEN",
            "test_oof_threshold_source",
            "test_insample_threshold",
            "test_insample_accuracy",
            "test_insample_balanced_accuracy",
            "test_insample_sensitivity",
            "test_insample_specificity",
            "test_insample_F1",
            "test_insample_MCC",
            "test_insample_YOUDEN",
            "test_insample_threshold_source",
            "n_test_patients",
        ]

        cols = [c for c in cols if c in results_df.columns]
        end_table = results_df[cols].copy()

        # Rank models best-first: by mean validation AUC, then steadiness, then MCC.
        sort_cols = [c for c in ["val_AUC_mean", "val_AUC_std", "val_MCC"] if c in end_table.columns]
        if sort_cols:
            ascending_map = {
                "val_AUC_mean": False,
                "val_AUC_std": True,
                "val_MCC": False,
            }

            end_table = end_table.sort_values(
                sort_cols,
                ascending=[ascending_map[c] for c in sort_cols],
                kind="mergesort",
            )
            # One block of rows per test set, in the same model order.
            end_table = order_by_test_set(end_table)


        print("\n" + "=" * 110)
        print("END-OF-RUN: Metrics for EACH tested parameter combination (GLOBAL params + best_params)")
        print("=" * 110)

        # Show the results table.
        try:
            display(rows_for_display(end_table).reset_index(drop=True))
        except Exception:
            print(rows_for_display(end_table).reset_index(drop=True).to_string(index=False))

        # Also save it as a standalone CSV for review.
        end_table_path = os.path.join(OUT_DIR, "all_param_combos_metrics.csv")
        end_table.to_csv(end_table_path, index=False)
        print("\nSaved per-parameter-combo table:", end_table_path)

        # ── Optional final refit ────────────────────────────────────────────────
        # Ranks features across the best models, keeps those several models agree on and
        # retrains on that subset. Guarded, so a failure cannot affect the main results.
        try:
            if not USE_POSTHOC_IMPORTANCE_FS:
                print("\n[POST-HOC FS] Skipped (USE_POSTHOC_IMPORTANCE_FS=False).")

            else:
                per_model_feats = {}
                all_top_feats = []
                per_model_impdf = {}
                top_models = pd.DataFrame()

                IMPORTANCE_CAPABLE = {
                    "LOGREG", "SGD_LOGLOSS", "LDA",
                    "RANDOM_FOREST", "DECISION_TREE", "EXTRATREES",
                    "ADABOOST", "HIST_GBDT",
                    "XGBOOST", "LIGHTGBM", "CATBOOST",
                }

                # results_df has one row per model and test set; a model is ranked once.
                cand = results_df.drop_duplicates(subset="model_id").copy()
                cand = cand[cand["classifier"].isin(IMPORTANCE_CAPABLE)].copy()

                if cand.empty:
                    print("\n[POST-HOC FS] No importance-capable models found in results_df. Skipping post-hoc stage.")

                else:
                    # Under PCA, restrict to no-PCA models so importances refer to
                    # original features.

                    cand = cand.sort_values(["val_AUC_mean", "val_AUC_std", "val_MCC"],
                                            ascending=[False, True, False])
                    top_models = cand.head(TOP_IMPORTANCE_MODELS)

                    print("\n" + "=" * 90)
                    print("[POST-HOC FS] Top importance-capable models (by val_AUC_mean):")
                    display(top_models[["model_id", "classifier", "val_AUC_mean", "val_AUC_std", "val_MCC"]])

                    TOPK_PRINT = 50

                    # 1) Refit each chosen model on all development patients and read its
                    #    most important features.
                    for _, r in top_models.iterrows():
                        mid = r["model_id"]
                        clf_name = r["classifier"]
                        pca_variant = r.get("pca", "no_pca")

                        bp = r.get("best_params", "{}")
                        if isinstance(bp, str):
                            try:
                                bp = json.loads(bp)
                            except Exception:
                                bp = {}
                        if not isinstance(bp, dict):
                            bp = {}

                        gp = r.get("global_params", "{}")
                        if isinstance(gp, str):
                            try:
                                gp = json.loads(gp)
                            except Exception:
                                gp = {}
                        if not isinstance(gp, dict):
                            gp = {}

                        # Convert each value to the expected type.
                        bp = {k: _coerce_param_value(v) for k, v in bp.items()}
                        gp = {k: _coerce_param_value(v) for k, v in gp.items()}

                        pipe, _ = build_pipeline(
                            pca_variant=pca_variant,
                            selector_name="NONE",
                            clf_name=clf_name,
                            fs_methods=None,
                        )
                        pipe.set_params(fsu="passthrough")

                        if gp:
                            pipe.set_params(**gp)
                        if bp:
                            pipe.set_params(**bp)

                        pipe.fit(X_train, y_train)

                        imp_df = top_features_with_scores_from_model(pipe, X_train, k=TOPK_PRINT)
                        per_model_impdf[mid] = imp_df.copy()

                        print("\n" + "-" * 90)
                        print(f"[POST-HOC FS] {mid} ({clf_name}) TOP-{TOPK_PRINT} features + importance")
                        print("-" * 90)

                        if imp_df.empty:
                            print("  (No importances available OR PCA active -> PCs, skipped.)")
                            feats = []
                        else:
                            display(imp_df)
                            # keep exactly TOP_FEATURES_PER_MODEL from this model for the vote
                            feats = imp_df["feature"].tolist()[:TOP_FEATURES_PER_MODEL]

                        per_model_feats[mid] = feats
                        all_top_feats.extend(feats)

                        print(f"[POST-HOC FS] {mid} kept_for_votes={len(feats)} (printed={min(TOPK_PRINT, len(imp_df))})")

                    # 2) Write the combined wide CSV.
                    FINAL_OUT = os.path.join(OUT_DIR, "POSTHOC_IMPORTANCE_FS")
                    os.makedirs(FINAL_OUT, exist_ok=True)

                    WIDE_K = TOPK_PRINT
                    wide_df = pd.DataFrame({"rank": list(range(1, WIDE_K + 1))})

                    for mid, df in per_model_impdf.items():
                        df2 = df.copy()
                        if not df2.empty:
                            df2 = df2[["feature", "importance"]].reset_index(drop=True)
                        else:
                            df2 = pd.DataFrame({"feature": [], "importance": []})

                        if df2.shape[0] < WIDE_K:
                            pad = pd.DataFrame({
                                "feature": ["" for _ in range(WIDE_K - df2.shape[0])],
                                "importance": [np.nan for _ in range(WIDE_K - df2.shape[0])],
                            })
                            df2 = pd.concat([df2, pad], ignore_index=True)
                        else:
                            df2 = df2.iloc[:WIDE_K].reset_index(drop=True)

                        wide_df[f"{mid}__feature"] = df2["feature"].astype(str).values
                        wide_df[f"{mid}__importance"] = df2["importance"].values

                    wide_path = os.path.join(FINAL_OUT, f"top{WIDE_K}_CLASSIFIER_features_importance_wide.csv")
                    wide_df.to_csv(wide_path, index=False)
                    print("\n[POST-HOC FS] Saved wide top-features table:", wide_path)

                    # 3) Keep features that enough of the top models agree on.
                    feat_votes = Counter()
                    for mid, feats in per_model_feats.items():
                        feat_votes.update(set(feats))

                    min_votes = int(POSTHOC_MIN_VOTES)
                    consensus_feats = [f for f, v in feat_votes.items() if v >= min_votes]
                    consensus_feats = sorted([f for f in consensus_feats if f in X_train.columns])

                    union_feats = sorted(set(all_top_feats))
                    union_feats = [f for f in union_feats if f in X_train.columns]

                    if POSTHOC_MAX_KEEP is not None and len(consensus_feats) > int(POSTHOC_MAX_KEEP):
                        consensus_feats = sorted(consensus_feats, key=lambda f: (-feat_votes[f], f))[: int(POSTHOC_MAX_KEEP)]
                        consensus_feats = sorted(consensus_feats)

                    print(f"\n[POST-HOC FS] Models used: {len(per_model_feats)}")
                    print(f"[POST-HOC FS] Consensus rule: votes >= {min_votes}")
                    print(f"[POST-HOC FS] Consensus features: {len(consensus_feats)}")

                    final_feats = consensus_feats if len(consensus_feats) else union_feats
                    if len(consensus_feats) == 0:
                        print(f"[POST-HOC FS] WARNING: consensus empty → fallback to UNION ({len(union_feats)})")

                    Xtr_fs = X_train[final_feats].copy()
                    Xte_fs = X_test[final_feats].copy()

                    print(f"[POST-HOC FS] Reduced DEV shape:  {Xtr_fs.shape}")
                    print(f"[POST-HOC FS] Reduced TEST shape: {Xte_fs.shape}")

                    # Final feature-count check from the selector and classifier.
                    print_overall_sanity_from_results(
                        results_df=results_df.drop_duplicates(subset="model_id"),
                        final_refit_classifiers=FINAL_REFIT_CLASSIFIERS,
                        final_feats_count=len(final_feats),
                        posthoc_union_count=len(union_feats),
                        posthoc_consensus_count=len(consensus_feats),
                    )

                    # 4) Refit the final models on that reduced feature set.
                    for final_clf in FINAL_REFIT_CLASSIFIERS:
                        final_id = f"POSTHOC__TOP{TOP_IMPORTANCE_MODELS}x{TOP_FEATURES_PER_MODEL}__{final_clf}"
                        print("\n" + "=" * 90)
                        print(f"[FINAL REFIT] {final_id}")
                        print("=" * 90)

                        inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=GLOBAL_SEED)
                        pipe, dist = build_final_refit_pipeline(final_clf, y_for_fit=y_train)

                        search = make_search(pipe, dist, inner_cv)
                        if search is None:
                            best_est = pipe.fit(Xtr_fs, y_train)
                            best_params = {}
                        else:
                            search.fit(Xtr_fs, y_train)
                            best_est = search.best_estimator_
                            best_params = search.best_params_

                        tr_score = predict_score(best_est, Xtr_fs)
                        # The holdout is scored only when holdout evaluation is enabled.
                        te_score = predict_score(best_est, Xte_fs) if holdout_evaluation_enabled() else None
                        if te_score is not None and len(_holdout_sets) > 1:
                            # As in the main loop: each holdout test set is scored on its own.
                            te_score = np.asarray(te_score, dtype=float).copy()
                            for _name, _positions in _holdout_sets:
                                te_score[_positions] = predict_score(best_est, Xte_fs.iloc[_positions])

                        thr = optimal_threshold_youden(y_train, tr_score)
                        tr_m = compute_metrics(y_train, tr_score, threshold=thr)
                        # Holdout metrics for each test set separately. Empty when the holdout
                        # was not scored.
                        te_m_by_set = {}
                        if te_score is not None:
                            te_m_by_set = {
                                _name: compute_metrics(
                                    y_test[_positions], np.asarray(te_score)[_positions], threshold=thr
                                )
                                for _name, _positions in _holdout_sets
                            }
                        te_m = next(iter(te_m_by_set.values()), {})

                        print("[FINAL REFIT] TRAIN:", _fmt_line(tr_m))
                        for _name, _m in te_m_by_set.items():
                            print(
                                "[FINAL REFIT] TEST"
                                + (f" [{_name}]" if len(te_m_by_set) > 1 else "")
                                + ": ", _fmt_line(_m)
                            )
                        if te_score is None:
                            print("[FINAL REFIT] TEST:  not scored in this run "
                                  "(OPTIMIZATION_EVALUATE_HOLDOUT is False)")

                        ddir = os.path.join(FINAL_OUT, final_id)
                        os.makedirs(ddir, exist_ok=True)

                        feat_vote_df = pd.DataFrame({
                            "feature": final_feats,
                            "votes": [int(feat_votes.get(f, 0)) for f in final_feats],
                        })
                        feat_vote_df.to_csv(os.path.join(ddir, "CLASSIFIER_selected_features_and_votes.csv"), index=False)

                        pd.DataFrame({"feature_file": feature_file, "patient_id": train_ids, "label": y_train, "score": tr_score}).to_csv(
                            os.path.join(ddir, "train_predictions.csv"), index=False
                        )
                        if te_score is not None:
                            pd.DataFrame({"feature_file": feature_file, "patient_id": test_ids,
                                          "test_set": [TEST_SET_OF_PATIENT[_cid] for _cid in test_ids],
                                          "label": y_test, "score": te_score}).to_csv(
                                os.path.join(ddir, "test_predictions.csv"), index=False
                            )

                        with open(os.path.join(ddir, "meta.json"), "w") as f:
                            json.dump(
                                {
                                    "feature_file": feature_file,
                                    "feature_path": feature_path,
                                    "final_id": final_id,
                                    "top_importance_models": int(TOP_IMPORTANCE_MODELS),
                                    "top_features_per_model": int(TOP_FEATURES_PER_MODEL),
                                    "selected_models": top_models[["feature_file", "model_id", "classifier", "val_AUC_mean"]].to_dict(orient="records"),
                                    "per_model_features": per_model_feats,
                                    "union_feature_count": int(len(union_feats)),
                                    "consensus_feature_count": int(len(consensus_feats)),
                                    "best_params": best_params,
                                    "thr_from_train": float(thr),
                                    "train_metrics": tr_m,
                                    "test_metrics": te_m,
                                    "test_metrics_by_test_set": te_m_by_set,
                                },
                                f,
                                indent=2,
                            )

                    print("\n[POST-HOC FS] Done. Saved under:", FINAL_OUT)
        except Exception as _posthoc_err:
            print("\n[POST-HOC FS] FAILED (non-fatal):", repr(_posthoc_err))
            print("[POST-HOC FS] Main result tables above are already printed and "
                  "saved; continuing without post-hoc outputs.")

        return results_df


    # ── Batch driver ────────────────────────────────────────────────────────────
    # Calls run_pipeline_for_feature_file() once per feature CSV, collects the result
    # tables and records any file that fails without stopping the rest.
    _segment3_tables = []
    _segment3_errors = []
    _SEGMENT3_COMPLETED_TABLES = {}  # current execution only; never recover stale files

    if not FEATURE_INPUT_FILES:
        print("[SECTION 2 ML] No current feature files queued; skipping the efficient_ML workflow.")
    else:
        for _feature_path in FEATURE_INPUT_FILES:
            _feature_file = os.path.basename(str(_feature_path))
            try:
                print("\n" + "=" * 110)
                print("[SECTION 2 ML] RUNNING:", _feature_file)
                print("=" * 110)
                # Covers inner-CV searches and permutation importance. Original-mask
                # runs retain their process backend; generated-mask runs use threads.
                # Keep one native BLAS/OpenMP thread per concurrent search task.
                _native_thread_limit = (threadpool_limits(limits=1)
                                        if SEARCH_BACKEND == "threading" else nullcontext())
                with joblib.parallel_backend(SEARCH_BACKEND, n_jobs=SEARCH_NJOBS), _native_thread_limit:
                    _df = run_pipeline_for_feature_file(_feature_path)
                if _df is not None:
                    _segment3_errors.extend(_df.attrs.get("model_errors", []))
                if _df is not None and len(_df):
                    _segment3_tables.append(_df.copy())
                else:
                    print("[SECTION 2 ML] completed with no result rows:", _feature_file)
            except Exception as _e:
                # A failure during later reporting/model export must not discard fits.
                _recovered = _SEGMENT3_COMPLETED_TABLES.get(os.path.abspath(str(_feature_path)))
                _recovered_count = len(_recovered) if _recovered is not None else 0
                if _recovered_count:
                    _segment3_tables.append(_recovered.copy(deep=True))
                    print(f"[SECTION 2 ML] Retained {_recovered_count} completed rows after a later failure.")
                _segment3_errors.append({
                    "feature_file": _feature_file,
                    "feature_path": str(_feature_path),
                    "status": "partial" if _recovered_count else "failed",
                    "recovered_completed_rows": _recovered_count,
                    "error": repr(_e),
                    "traceback": traceback.format_exc(),
                })
                print("[SECTION 2 ML] FAILED:", _feature_file, repr(_e))
                print("[SECTION 2 ML] Continuing with the next feature file.")

    if _segment3_tables:
        segment3_results_df = pd.concat(_segment3_tables, ignore_index=True)
    else:
        segment3_results_df = pd.DataFrame()

    segment3_results_df["segmentation_method"] = SEGMENTATION_ANALYSIS_METHOD
    RESULTS_REGISTRY["segment3_ml"] = segment3_results_df

    if _segment3_errors:
        segment3_errors_df = pd.DataFrame(_segment3_errors)
        RESULTS_REGISTRY["segment3_ml_errors"] = segment3_errors_df
        try:
            _err_csv = os.path.join(PART3_OUT_DIR, "segment3_errors.csv")
            segment3_errors_df.to_csv(_err_csv, index=False)
            print("[SECTION 2 ML] Error log path:", _err_csv)
        except Exception as _save_err:
            print("[SECTION 2 ML] cannot save the error log:", repr(_save_err))
    else:
        segment3_errors_df = pd.DataFrame()

    print("[SECTION 2 ML] final overview contains", len(segment3_results_df), "result row(s).")
    if len(segment3_errors_df):
        print("[SECTION 2 ML] feature file(s) with errors:", len(segment3_errors_df))


### Section 2, cell 14 - CNN plus ML final overview - all backbones and tested models

The table below shows **every completed CNN+ML model/parameter result**, including all available holdout-test measures. Within each backbone × outcome × optimization arm/channel-policy group, `validation_rank` and `validation_selected` show which row would be preferred using development validation only (mean validation AUC, then lower validation-AUC SD, then validation MCC, then pooled OOF AUC). Holdout results are fully visible for every row; they are not hidden or removed. In an optimization comparison the holdout columns are filled only when `OPTIMIZATION_EVALUATE_HOLDOUT` is `True`.


In [ ]:
# Section 2, cell 15: Display all CNN plus ML results
if globals().get("RUN_CNN_LR", True):
    # Show every completed model row, including all available holdout-test metrics.
    # validation_rank / validation_selected preserve the development-only ranking as provenance;
    # test metrics are visible for research review but are not used to compute that rank.

    print("\n" + "=" * 118)
    print("CNN+ML FINAL OVERVIEW: ALL BACKBONES, ALL TESTED MODELS, ALL AVAILABLE TEST MEASURES")
    print("=" * 118)
    if not holdout_evaluation_enabled():
        print("[CNN+ML overview] Holdout test not scored in this run "
              "(OPTIMIZATION_EVALUATE_HOLDOUT is False): the holdout columns are empty. "
              "Each model is shown once. The saved table keeps one row per test set.")

    if "segment3_results_df" not in globals() or segment3_results_df is None or segment3_results_df.empty:
        print("[CNN+ML overview] No completed CNN+ML result rows are available.")
        cnn_ml_final_overview_df = pd.DataFrame()
    else:
        _cnnml = segment3_results_df.copy()

        # Ensure grouping columns exist in both optimization and non-optimization runs.
        _group_cols = []
        for _c, _default in [
            ("cnn_backbone", "unknown"),
            ("cnn_channel_policy", ""),
            ("outcome", ""),
            ("test_set", ""),
            ("opt_arm_id", "production"),
        ]:
            if _c not in _cnnml.columns:
                _cnnml[_c] = _default
            _cnnml[_c] = _cnnml[_c].fillna(_default).astype(str)
            _group_cols.append(_c)

        # Numeric helpers for the validation-only rank marker.
        for _c in ["val_AUC_mean", "val_AUC_std", "val_MCC", "val_oof_AUC"]:
            if _c not in _cnnml.columns:
                _cnnml[_c] = np.nan
            _cnnml[_c] = pd.to_numeric(_cnnml[_c], errors="coerce")

        _cnnml["_rank_val_auc"] = _cnnml["val_AUC_mean"].fillna(-np.inf)
        _cnnml["_rank_val_sd"]  = _cnnml["val_AUC_std"].fillna(np.inf)
        _cnnml["_rank_val_mcc"] = _cnnml["val_MCC"].fillna(-np.inf)
        _cnnml["_rank_oof_auc"] = _cnnml["val_oof_AUC"].fillna(-np.inf)

        _cnnml = _cnnml.sort_values(
            _group_cols + ["_rank_val_auc", "_rank_val_sd", "_rank_val_mcc", "_rank_oof_auc"],
            ascending=[True] * len(_group_cols) + [False, True, False, False],
            kind="mergesort",
        ).copy()

        # The rank is shown for transparency; no rows are discarded.
        _cnnml["validation_rank"] = (
            _cnnml.groupby(_group_cols, sort=False, dropna=False).cumcount() + 1
        ).astype(int)
        _cnnml["validation_selected"] = _cnnml["validation_rank"].eq(1)

        # Put model, provenance and validation columns first, then append every available
        # holdout column exactly as produced by the workflow.
        _front_cols = [
            "cnn_backbone", "cnn_channel_policy", "outcome", "test_set",
            "optimization", "opt_arm_id", "opt_is_reference",
            "opt_input_size", "opt_n_slices", "opt_augmentation",
            "classifier", "model_id", "pca", "selector",
            "validation_rank", "validation_selected",
            "val_oof_AUC", "val_AUC_mean", "val_AUC_std",
            "val_ACC", "val_BACC", "val_MCC", "val_YOUDEN",
        ]
        _front_cols = [c for c in _front_cols if c in _cnnml.columns]

        # Preserve the original workflow column order for test measures.
        _all_test_cols = [
            c for c in _cnnml.columns
            if (str(c).startswith("test_") or str(c).startswith("final_test_"))
            and c not in _front_cols
        ]
        if "n_test_patients" in _cnnml.columns and "n_test_patients" not in _all_test_cols:
            _all_test_cols.append("n_test_patients")
        if "n_test_cases" in _cnnml.columns and "n_test_cases" not in _all_test_cols:
            _all_test_cols.append("n_test_cases")

        # Keep additional useful non-test provenance after the primary columns.
        _extra_cols = [
            c for c in [
                "feature_file", "global_combo_index", "global_params", "best_params",
                "pca_n_mean", "pca_n_std", "stability_jaccard_mean",
                "n_features_selected_mean", "n_features_selected_std",
            ]
            if c in _cnnml.columns and c not in _front_cols and c not in _all_test_cols
        ]

        cnn_ml_final_overview_df = order_by_test_set(
            _cnnml[_front_cols + _all_test_cols + _extra_cols]
        ).reset_index(drop=True)

        # One table per holdout test set (a single table when there is one test set).
        # Without holdout scoring the tables would be the same, so each model is shown once.
        _overview_shown = rows_for_display(cnn_ml_final_overview_df)
        _overview_test_sets = (
            list(dict.fromkeys(_overview_shown["test_set"]))
            if "test_set" in _overview_shown.columns else [None]
        )
        for _test_set_name in _overview_test_sets:
            _part = (
                _overview_shown if _test_set_name is None
                else _overview_shown[_overview_shown["test_set"] == _test_set_name]
            ).reset_index(drop=True)
            if _test_set_name is not None and len(TEST_SET_NAMES) > 1:
                print("\n" + "-" * 118)
                print(f"HOLDOUT TEST SET: {_test_set_name}")
                print("-" * 118)
            try:
                display(_part)
            except Exception:
                print(_part.to_string(index=False))

        # Save the same complete table shown in the notebook.
        try:
            _cnnml_overview_path = os.path.join(PART3_OUT_DIR, "CNN_ML_FINAL_OVERVIEW_ALL_BACKBONES_ALL_MODELS.csv")
            cnn_ml_final_overview_df.to_csv(_cnnml_overview_path, index=False)
            print("\n[CNN+ML overview] Saved combined table:", _cnnml_overview_path)
        except Exception as _e:
            print("\n[CNN+ML overview] Could not save combined overview:", repr(_e))

        RESULTS_REGISTRY["segment3_ml_final_overview"] = cnn_ml_final_overview_df.copy()
        print("[CNN+ML overview] Rows shown (no model rows hidden):", len(_overview_shown))
        print("[CNN+ML overview] Holdout/test columns shown:", len(_all_test_cols))
else:
    print('[SKIP] CNN+ML final overview: efficient_ML workflow disabled by RUN_CNN_LR=False.')


## Section 3, cell 1 - End-to-end CNN with fine-tuned backbone

The backbone is unfrozen after a head warm-up phase, giving the fine-tuned counterpart of
Section 1.

Shared settings come from Section 0: the patient split, minimal square tumour crop with a 2-pixel safety
margin, central planes, intensity scaling, architecture list, outcome definitions, output root and package pins. `PATIENT_SPLIT_RECORDS`
is reused rather than reset, so the run stops if this section disagrees with Sections 1–2 about the
split. Architectures that cannot be fine-tuned end to end (InceptionV3, Choi-IDH) are reported as
skipped.

Hyper-parameters specific to this protocol (`DROPOUT = 0.30`, phase-2 learning rates, EMA, SWA and
L2-SP) differ from Section 1 by design.

Training augmentation is intentionally restricted to the script-26 protocol: random horizontal and
vertical flips only. DenseNet121 phase-2 exposure is intermediate between scripts 74 and 26:
`transition3.conv` plus the last six `denseblock4` layers and `norm5` affine parameters
(~20.8% of the DenseNet feature backbone).


In [ ]:
# Section 3, cell 2: Provenance and section notes
# Fine-tuning code exported from FINE_TUNING_NESTED_CV_REVISED.ipynb.
# The DEVELOPMENT/TEST folder split and OUT_ROOT come from Section 0, so this section shares one
# patient split and one output tree with Sections 1 and 2.
# Single-model policy: the deployed model is one network evaluated in one
# forward pass - USE_EMA=False, USE_SWA=False, USE_TTA=False and no fold or repeat prediction
# ensemble. The development estimate is separate: N_REPEATS=3 repeated outer CV
# gives a stable pooled-OOF AUC and a stable OOF Youden threshold without changing what is
# used for final reporting. Every optional component is switched in one block in the configuration cell below.


### Section 3, cell 3 - Optimization notes

**Optimisation.** Phase-2 backbone learning rate is selected on the inner development split from
`INNER_BACKBONE_LR_CANDIDATES`, never assumed; every candidate shares one cached phase-1 run and the
same warm-up → cosine-decay schedule. Weight averaging (EMA, SWA) is off in this revision, so the
selected phase-2 state is the raw checkpoint. Gradient clipping at `GRAD_CLIP_NORM` is an
exploding-gradient guard only, with a warning when it binds on more than
`FINETUNE_CLIP_FRACTION_WARN_MAX` of phase-2 batches; validation log-loss ties are resolved within
0.002, and an exact residual tie is broken toward the *lower* learning rate.

**Phase-2 movement cap.** `USE_MOVEMENT_CAP` bounds how far phase 2 may move the backbone from its
pretrained anchor, at `FINETUNE_MAX_REL_RMS_MOVE` relative RMS displacement, measured cumulatively
from the start of phase 2. The epoch that crosses the cap remains eligible for selection; only
further training stops. This replaces a longer epoch budget as the way of limiting drift, and it is
enforced in the fixed-budget refit as well as in the inner selection, because refitting on more
patients means more optimiser steps per epoch than the inner split that chose the budget.

### Section 3, cell 4 - Clean-training diagnostic

**Clean-training diagnostic.** A fixed-checkpoint training AUC is measured after phase 1 and after
the selected phase-2 state with `model.eval()`, no augmentation, no TTA and no optimiser updates.
It is diagnostic only and never influences selection; the augmented online figure is reported
separately as `tr_auc_aug_online`. Fold CSVs and summaries export the clean head and fine-tune AUCs
together with the train-to-holdout generalisation gaps.

### Section 3, cell 5 - Preprocessing notes specific to this section

**`radimagenet_densenet121` uses `minus_one_one` input scaling.** Measured on the development set
with the same nested-CV machinery, paired pooled-OOF head AUC was 0.670 for `minus_one_one` against
0.647 (`caffe_over_255`), 0.616 (`unit_interval`) and 0.557 (`imagenet`), with frozen-feature
effective rank ordering the arms identically. `radimagenet_resnet50` shares the checkpoint release
but was not measured, so it keeps its own setting behind an explicit flag. Both settings are defined
in Section 0, cell 3 (`RADIMAGENET_NORM_BY_BACKBONE`) and apply to Sections 1, 2 and 3 alike.

**`GRAD_CLIP_NORM` is set high enough not to bind.** At a low clip value every phase-2 batch of the
DenseNet121 cross-validation clipped at gradient norms of roughly 20–25, which made the clip rather
than the learning rate set the step size.

In [ ]:
# Section 3, cell 6: Settings
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    # Packages are installed in Section 0, before any import. This cell only reports
    # whether the running versions match the pins.
    import sys, subprocess, importlib.metadata as _importlib_metadata

    _pin_mismatches = []
    for _pkg, _want in PACKAGE_PINS.items():
        try:
            _have = _importlib_metadata.version(_pkg)
        except Exception:
            _have = "not installed"
        if _have != _want:
            _pin_mismatches.append(f"{_pkg}: running {_have}, protocol pin {_want}")
    if _pin_mismatches:
        print("[versions] The running packages differ from the Section 0 protocol pins:")
        for _m in _pin_mismatches:
            print("  -", _m)
        print("  Edit the single pinned install line in Section 0 and restart the runtime "
              "if the exact versions matter for this run.")
    else:
        print("[versions] All Section 0 protocol package pins match the running environment.")

    import os, random, re, math
    import numpy as np
    import pandas as pd

    # Optional Hugging Face token for public ImageNet weight downloads.
    HF_TOKEN = ""  # Prefer a Colab Secret named HF_TOKEN; do not paste private keys into shared notebooks.

    def configure_optional_hf_token():
        token = os.environ.get("HF_TOKEN", "").strip()
        source = "environment variable"
        if not token:
            try:
                from google.colab import userdata
                token = (userdata.get("HF_TOKEN") or "").strip()
                source = "Colab Secret"
            except Exception:
                token = ""
        if not token:
            token = str(globals().get("HF_TOKEN", "") or "").strip()
            source = "notebook HF_TOKEN line"
        if token:
            os.environ["HF_TOKEN"] = token
            print(f"[Hugging Face] Optional access key found in: {source}.")
            return True
        print("[Hugging Face] No HF_TOKEN found; public pretrained weights will download anonymously.")
        return False

    HF_AUTHENTICATED = configure_optional_hf_token()
    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

    def ensure_drive_mounted(mountpoint="/content/drive"):
        """Mount Google Drive only when it is not already available."""
        global IN_COLAB
        if os.path.exists(os.path.join(mountpoint, "MyDrive")):
            IN_COLAB = True
            return True
        try:
            from google.colab import drive
            drive.mount(mountpoint)
            IN_COLAB = True
            return True
        except Exception as exc:
            IN_COLAB = False
            print("[Google Drive] Could not mount Drive:", repr(exc))
            return False

    ensure_drive_mounted()

    # ── Architectures, outcome definitions and patient split come from Section 0 ──
    for _required in ("SHARED_BACKBONES", "SHARED_OUTCOMES", "SPLIT_FOLDER_MEMBERSHIP", "TEST_SET_NAMES",
                      "PATIENT_SPLIT_RECORDS", "_SEG1_SUPPORTED_BACKBONES"):
        if _required not in globals():
            raise RuntimeError(
                f"{_required} is missing. Run Section 0 (Settings and Google Drive mounting) "
                "before the fine-tuning section - it defines the architecture list, the outcome "
                "definitions and the single patient split used by the whole notebook."
            )

    # End-to-end-trainable architectures are listed once in Section 0 as
    # _SEG1_SUPPORTED_BACKBONES. Anything else in SHARED_BACKBONES is reported as skipped.

    # The split is created once by the first section to run. PATIENT_SPLIT_RECORDS is
    # deliberately not reset here, so get_or_create_patient_split() below compares this
    # section's patient list against the one already in use and raises on any difference.

    # ID normalisation, the split helper, top-risk reporting and the AUC confidence
    # intervals come from Section 0.
    for _required in (
        "canonical_patient_id", "patient_id_sort_key", "_canonical_unique_patient_ids",
        "patient_split_record_key", "get_or_create_patient_split", "print_patient_split",
        "collect_pairs_from_split_folders", "cache_pairs_locally",
        "holdout_test_sets", "test_set_file_tag", "order_by_test_set",
        "top_risk_mode_enabled", "threshold_for_top_risk", "TOP_RISK_PERCENT",
        "calculate_auc_intervals", "AUC_CI_CONFIDENCE", "AUC_BOOTSTRAP_REPEATS",
    ):
        if _required not in globals():
            raise RuntimeError(
                f"{_required} is missing. Run Section 0 before the fine-tuning section."
            )

    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", None)
    pd.set_option("display.max_columns", None)
    # Same capped row display as Section 0; to_string() output is unaffected.
    pd.set_option("display.max_rows", globals().get("MAX_DISPLAY_ROWS", 200))

    print("[SETUP] Architecture list from Section 0:", SHARED_BACKBONES)
    print("[SETUP] Of those, fine-tunable end-to-end:",
          [b for b in SHARED_BACKBONES if str(b).lower() in _SEG1_SUPPORTED_BACKBONES])
    print("[SETUP] Outcome of this run:", SHARED_OUTCOMES)
    print(f"[SETUP] Split by folder: DEVELOPMENT={len(DEV_FOLDER_PATIENT_IDS)} patient ID(s) "
          f"| TEST={len(TEST_FOLDER_PATIENT_IDS)} patient ID(s) in {len(TEST_SET_NAMES)} test set(s): "
          f"{ {n: len(v) for n, v in TEST_SET_PATIENT_IDS.items()} }")
    print("[SETUP] Top-risk percentage:", TOP_RISK_PERCENT)
else:
    print('[SKIP] Section 3: fine-tuned end-to-end CNN disabled by RUN_CNN_FINETUNE_E2E=False.')


In [ ]:
# Section 3, cell 7: Imports
if globals().get("RUN_CNN_FINETUNE_E2E", True):
    import sys, subprocess, os, warnings, re, shutil, random, hashlib, importlib.metadata
    from pathlib import Path
    from copy import deepcopy


    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import torchvision.models as tv_models
    import timm
    import SimpleITK as sitk
    from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, StratifiedShuffleSplit
    from sklearn.metrics import (
        roc_auc_score, balanced_accuracy_score, f1_score, accuracy_score,
        precision_score, recall_score, confusion_matrix, matthews_corrcoef,
        brier_score_loss, log_loss
    )
    from tqdm.auto import tqdm

    # Warnings stay visible: numerical, API and checkpoint warnings must not be hidden.
    warnings.filterwarnings("default")
    # Silence only the repetitive Jupyter utcnow deprecation notice.
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        module=r"jupyter_client\.session",
    )

    ensure_drive_mounted()   # Defined in Section 0.

    seed_everything(GLOBAL_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")


In [ ]:
# Section 3, cell 8: Configuration
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    # ── Data paths ────────────────────────────────────────────────────────────────
    # DEVELOPMENT_ROOT, TEST_ROOT, LABELS_CONFIGS, WEIGHTS_BASE, the resolved RadImageNet
    # checkpoint paths and the input scaling of every backbone (RADIMAGENET_NORM_BY_BACKBONE,
    # RADIMAGENET_RESNET50_USE_OFFICIAL_NORM and radimagenet_norm_for()) are defined once in
    # Section 0, cell 3 and are reused here.


    # ROI z-scores are rescaled so +/- this many SD fill [0, 1]. At 2.5 the within-ROI
    # pixel SD lands near 0.20, inside the 0.2-0.3 regime RadImageNet was pretrained on.


    # ── Cache and output ──────────────────────────────────────────────────────────
    # Cache and preprocessing constants come from Section 0, so every image workflow
    # uses the same minimal-square tumour crop and the same physical slice and intensity settings.
    for _required in ("LOCAL_CACHE_DIR", "CNN_INPUT_SIZE_SHARED", "SLICE_PLANES",
                      "CROP_SAFETY_MARGIN_PX", "INTENSITY_Z_SPAN_SD",
                      "TUMOR_CONTEXT_MODE", "ROI_ONLY_BACKGROUND_VALUE",
                      "radimagenet_norm_for", "apply_input_scaling",
                      "collect_generic_image_mask_pairs", "load_label_table",
                      "holdout_evaluation_enabled", "HoldoutNotEvaluated", "rows_for_display"):
        if _required not in globals():
            raise RuntimeError(f"{_required} is missing. Run Section 0 before fine-tuning.")

    # OUT_ROOT comes from Section 0; this section writes to its own sub-folder of it.
    for _required in ("OUT_ROOT", "FINETUNE_OUT_DIR"):
        if _required not in globals():
            raise RuntimeError(
                f"{_required} is missing. Run Section 0 before the fine-tuning section - "
                "it defines the single output root shared by every section."
            )
    # Two runs that differ only in TUMOR_CONTEXT_MODE would otherwise write the same
    # filenames into the same folder and silently overwrite each other - every CSV, every
    # checkpoint and the summary table. The representation therefore names the folder.
    # In OPTIMIZATION mode the representation varies per arm, so the folder is a single
    # shared optimization directory, and out_prefix separates each setting combination.
    if OPTIMIZATION:
        # Context is a manual global setting, not one of the automatically compared settings.
        _out_dir_tag = "SWEEP_" + tumor_context_tag()
    else:
        _out_dir_tag = tumor_context_tag()
    OUT_DIR = os.path.join(FINETUNE_OUT_DIR, _out_dir_tag)
    print(f"[Output separation] Fine-tuning results folder tag: {_out_dir_tag}")

    if not os.path.exists("/content/drive/MyDrive"):
        raise RuntimeError(
            "Google Drive is not mounted. Run the settings/mounting cell before this cell; "
            "the notebook will not silently fall back to temporary /content output."
        )

    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"[Persistent output] OUT_ROOT: {OUT_ROOT}")
    print(f"[Persistent output] OUT_DIR : {OUT_DIR}")

    # ── Backbones supported by Section 3 ─────────────────────────────────────────────
    RUN_BACKBONES = [b for b in SHARED_BACKBONES
                     if str(b).lower() in _SEG1_SUPPORTED_BACKBONES]
    if OPTIMIZATION:
        # Compare only the declared subset. Names not present in SHARED_BACKBONES are a
        # configuration error, not something to silently ignore.
        _opt_wanted = [str(b).lower() for b in OPTIMIZATION_BACKBONES]
        _opt_missing = [b for b in _opt_wanted if b not in [str(x).lower() for x in RUN_BACKBONES]]
        if _opt_missing:
            raise RuntimeError(
                f"[OPTIMIZATION][FATAL] OPTIMIZATION_BACKBONES entries not available: {_opt_missing}. "
                "They must be uncommented in SHARED_BACKBONES and fine-tunable end to end."
            )
        RUN_BACKBONES = [b for b in RUN_BACKBONES if str(b).lower() in _opt_wanted]
        print(f"[OPTIMIZATION] Sweeping on {len(RUN_BACKBONES)} backbone(s): {RUN_BACKBONES}")
    _seg1_skipped_backbones = [b for b in SHARED_BACKBONES
                               if str(b).lower() not in _SEG1_SUPPORTED_BACKBONES]
    if _seg1_skipped_backbones:
        print("[SECTION 3] Not fine-tuned end-to-end (feature-extracted in Section 2 only): "
              f"{_seg1_skipped_backbones}")
    if not RUN_BACKBONES:
        raise RuntimeError(
            "[SECTION 3][FATAL] None of the architectures configured in Section 0 can be "
            f"fine-tuned end-to-end; SHARED_BACKBONES={SHARED_BACKBONES}"
        )
    CHANNEL_POLICY = "gray3"   # replicate T1cc into all 3 channels; mask channel dropped
    # 224 matches the pretraining resolution; smaller inputs trade feature quality for compute.
    CNN_INPUT_SIZE = CNN_INPUT_SIZE_SHARED  # shared in Section 0
    # Central planes selected by maximum tumour area in each orientation.
    CHANNEL_MODE   = "t1cc_t1cc_mask"  # intermediate slice assembly before gray3 is applied

    # ── Fine-tuning protocol ─────────────────────────────────────────────────────
    N_FOLDS         = 5
    if OPTIMIZATION:
        # Same GLOBAL_SEED as production, so the fold assignment is identical in every arm and
        # arm-to-arm comparisons are paired patient-by-patient.
        N_FOLDS = int(OPTIMIZATION_N_FOLDS)
    # Repeated outer CV estimates performance; final reporting still uses a single
    # network. The single-model policy applies to the final refit: one network, one forward
    # pass and no EMA, SWA, TTA or fold ensemble. With N_REPEATS=1, each patient contributes
    # one OOF prediction, making the pooled OOF AUC most variable; the "median of repeat-wise
    # Youden thresholds" below degenerates to a single threshold carried to the holdout.
    # Three repeats restore a stable primary estimate and a stable threshold while the
    # deployed artefact is unchanged.
    N_REPEATS       = 3
    if OPTIMIZATION:
        N_REPEATS = int(OPTIMIZATION_N_REPEATS)

    # Inner selection split, taken inside each outer-training fold. Outer validation
    # patients never influence checkpoint, epoch, pooling, unfreezing or EMA choices.
    INNER_VAL_FRACTION = 0.20
    # What the inner selection searches over is configured in the OPTIONAL COMPONENT
    # SWITCHES block further down: INNER_POOLING_CANDIDATES, INNER_STAGED_UNFREEZE_CANDIDATES
    # and INNER_COMPLEXITY_MIN_AUC_GAIN are defined there, next to the switch that activates
    # them. The backbone learning-rate grid below is always searched.

    # Maximum CV budgets. Early stopping selects the best epoch in each fold.
    # Phase 2 stays at 15: in the 77c/77d runs the backbones that trained longest at 5e-5
    # moved 4-7% relative RMS and their paired OOF AUC fell below their own head-only score
    # (RadImageNet-ResNet18: head 0.632 -> FT 0.609). The overall paired FT-head delta was
    # +0.005 with a CI straddling zero, so there is no evidence that a longer phase-2 budget
    # adds useful information. Movement is bounded by FINETUNE_MAX_REL_RMS_MOVE below instead of by a
    # larger training allowance.
    EPOCHS_HEAD_MAX       = 20
    EPOCHS_FINETUNE_MAX   = 15

    # Phase 1: classification head only.
    LR_HEAD               = 1e-3
    WEIGHT_DECAY_HEAD     = 1e-4

    # Select the phase-2 backbone learning rate inside the
    # inner-development split; every candidate follows the same warm-up -> cosine-decay
    # schedule, and the untouched holdout never participates in the choice.
    #
    # A fixed LR was not supported by the previous runs. Across 77c/77d, inner CV picked
    # 1e-5 for ResNet-50 in 3 of 4 runs and 1e-5/5e-5 for the DenseNets; using a fixed
    # 3e-5 / 1e-4 overrides those choices. Giving one architecture a different multiplier
    # also confounds the DenseNet-vs-ResNet comparison in the results table with an LR
    # difference.
    #
    # The grid is low-cost: phase1_cache (see select_protocol_on_inner_split) shares one
    # phase-1 run across every LR candidate, so k learning rates cost 1 phase-1 pass plus k
    # short phase-2 passes.
    #
    # Grids are recentred on the values selected by inner CV in 77c/77d.
    INNER_BACKBONE_LR_CANDIDATES = {
        "resnet10t":   (1e-5, 3e-5, 5e-5),
        "resnet18":    (1e-5, 3e-5, 5e-5),
        "resnet50":    (5e-6, 1e-5, 3e-5),   # data chose 1e-5 in 3 of 4 previous runs
        "densenet121": (5e-6, 1e-5, 5e-5),   # 1e-4 dropped: never selected on RadImageNet
    }

    def inner_backbone_lr_candidates(backbone_name: str):
        name = str(backbone_name).lower()
        for key, values in INNER_BACKBONE_LR_CANDIDATES.items():
            if key in name:
                vals = tuple(float(v) for v in values)
                if not vals or any((not np.isfinite(v) or v <= 0.0) for v in vals):
                    raise ValueError(f"Invalid backbone LR candidates for {backbone_name}: {values}")
                return vals
        raise KeyError(f"No inner-CV backbone LR grid configured for {backbone_name!r}")

    LR_FINETUNE_HEAD      = 1e-4
    WEIGHT_DECAY_BACKBONE = 1e-5
    # LR WARM-UP
    # Start phase 2 below the intended learning rate and increase it over the first few
    #   epochs. The freshly unfrozen block meets a classification head
    #   trained against frozen features, so the first gradients are large and misdirected;
    #   warming up stops those first steps from damaging the pretrained weights. After the
    #   ramp the LR follows a cosine decay down to FINETUNE_MIN_LR_FACTOR of its peak.
    # Here, three epochs use 20% -> 60% -> 100% of the selected backbone LR.
    FINETUNE_WARMUP_EPOCHS = 3
    # Warm-up controls LR only; it must not exclude a genuinely best early checkpoint.
    # Script 26 repeatedly selected fine-tuning epoch 1, so epoch 1 is eligible here.
    FINETUNE_MIN_SELECTION_EPOCH = 1
    FINETUNE_MIN_LR_FACTOR = 0.05

    # L2-SP  (L2 penalty toward the Starting Point)
    # L2-SP is a soft version of the movement cap. Ordinary weight decay pulls weights
    #   toward zero, which erases pretraining; L2-SP instead pulls them toward the
    #   pretrained values, so the network pays a penalty for drifting away from what it
    #   already knew and only does so where the data insists. Where switch 6 is a hard stop,
    #   L2-SP applies a continuous force at every step.
    # Here, the penalty is the sum of squared displacements from the anchor, not the mean,
    #   so a small lambda already acts strongly. 0.0 disables it.
    L2SP_LAMBDA = 1e-4

    EARLY_STOP_PATIENCE_HEAD     = 6
    # Kept at 5 for the same reason as EPOCHS_FINETUNE_MAX: a larger training allowance on phase 2 was
    # not supported by the paired OOF evidence.
    EARLY_STOP_PATIENCE_FINETUNE = 5
    MIN_AUC_IMPROVEMENT          = 1e-4
    # AUC is heavily quantised on ~25 patients, so log-loss breaks ties, and only a
    # meaningful gain counts. Otherwise patience resets on numerical variation.
    VAL_LOGLOSS_TIE_TOL          = 0.002
    # Gradient clipping is an exploding-gradient guard, not a step-size policy, so it is set
    # high enough not to bind. A low value clips every batch, letting the clip rather than
    # the learning rate set the step length.
    GRAD_CLIP_NORM               = 1000.0
    # Warn if clipping binds on most batches. Reported per checkpoint; never alters selection.
    FINETUNE_CLIP_FRACTION_WARN_MAX = 0.50

    # ══════════════════════════════════════════════════════════════════════════════
    # OPTIONAL COMPONENT SWITCHES
    # ══════════════════════════════════════════════════════════════════════════════
    # One place to turn every optional part of the fine-tuning workflow on or off.
    #
    # Each numbered switch controls the parameters listed under it. When the switch is False,
    # those parameters are inactive, and the run summary records the
    # component as "off". Flip one switch to True to activate the component together with
    # its parameters; no edit anywhere else in the notebook is required.
    #
    # Each switch describes the method and how the notebook uses it.
    #
    # The current settings are described below.
    # The current run reports a strict single model: one network, one forward pass, no weight
    # averaging, no test-time augmentation, no fold or repeat ensemble. Switches 1-5 are
    # therefore all False. Switch 6 controls the phase-2 movement cap and is ON, because it
    # is a safety bound rather than an extra component.
    # ══════════════════════════════════════════════════════════════════════════════

    # ── 1. EMA: Exponential Moving Average of the weights ────────────────────────
    # EMA keeps a running weighted average of the weights during training. The average lies
    #   nearer the centre of the region explored by the optimizer, which may improve
    #   generalisation.
    # Here, an EMA "shadow" copy is updated after every optimizer step in
    #   phase 2 and offered to the inner selection as a separate weight strategy competing
    #   with the raw weights. EMA_DECAY sets how long the memory is: 0.97 keeps ~97% of the
    #   old average per step, so the shadow trails the live weights by roughly 30 steps.
    #   Too high a decay on a short phase 2 leaves the shadow stuck at its starting point,
    #   which is what made script 28's "fine-tuned" model effectively head-only.
    USE_EMA   = False
    EMA_DECAY = 0.97                 # Used only by USE_EMA; decay per optimizer step.

    # ── 2. SWA: Stochastic Weight Averaging ──────────────────────────────────────
    # SWA is similar to EMA but uses an unweighted average over
    #   the last few epochs, once the learning rate has decayed and training is circling a
    #   minimum rather than travelling towards one.
    # Here, the weights from the final SWA_WINDOW phase-2 epochs are averaged
    #   into one checkpoint, which is offered to the inner selection as a third weight
    #   strategy next to raw and EMA.
    USE_SWA    = False
    SWA_WINDOW = 5                   # Used only by USE_SWA; number of final epochs averaged.

    # ── 3. TTA: Test-Time Augmentation ───────────────────────────────────────────
    # At prediction time, TTA gives the model several transformed copies of the same
    #   image and averages the predictions. Training is unchanged; this reduces the
    #   variance of the prediction, the way a small ensemble would.
    # USE_TTA averages predictions from deterministic horizontal and vertical flips.
    #   USE_TTA_ROTATIONS additionally averages over the angles in TTA_ROTATION_DEGREES.
    #   USE_TTA_FOR_EPOCH_SELECTION also applies TTA while scoring the inner validation
    #   split, so the epoch chosen is the best TTA epoch rather than the best single-pass
    #   epoch. All three are off here because the reported model must be one forward pass.
    USE_TTA                     = False
    USE_TTA_ROTATIONS           = False
    TTA_ROTATION_DEGREES        = (90.0, 180.0, 270.0)   # Used only by USE_TTA_ROTATIONS.
    USE_TTA_FOR_EPOCH_SELECTION = False                  # Used only by USE_TTA.

    # ── 4. Mixup ─────────────────────────────────────────────────────────────────
    # Mixup trains on blends of two cases. Patients are combined with a random
    #   weight L, and the label is blended by that weight. For example, 70% of patient A
    #   and 30% of patient B carries the label 0.7*y_A + 0.3*y_B. It
    #   discourages over-confident predictions and enlarges the effective training set.
    # Here, blending is applied per patient inside each training batch in both
    #   phases. MIXUP_ALPHA is the shape of the Beta(alpha, alpha) distribution L is drawn
    #   from: small values (0.2) mostly give near-unmixed cases with occasional strong
    #   blends, larger values push towards 50/50 mixes.
    USE_MIXUP   = False
    MIXUP_ALPHA = 0.2                # Used only by USE_MIXUP; Beta(alpha, alpha) mixing weight.

    # ── 5. Extra capacity: staged unfreezing, attention pooling, complexity gate ─
    #
    # STAGED UNFREEZING
    # Staged unfreezing trains the pretrained network gradually. The
    #   last block trains first; a deeper block joins later, at a smaller learning rate,
    #   once the layers above it have settled. It lets more of the network adapt without
    #   destabilising it early, at the cost of more trainable parameters on a small cohort.
    # After STAGE1_UNFREEZE_EPOCHS of phase 2, one further adjacent
    #   backbone segment is unfrozen and given backbone LR * STAGE2_LR_FACTOR.
    #
    # ATTENTION POOLING
    # Attention pooling replaces a simple average with a small learned network that
    #   scores how informative each slice is and takes a weighted average
    #   (gated attention MIL, Ilse et al. 2018), so an uninformative slice can be
    #   down-weighted instead of diluting the patient-level prediction.
    # It is offered as a pooling candidate to the inner selection. It adds
    #   parameters, and with N_SLICES = 1 there is nothing to weight, so it matters only in
    #   a multi-slice configuration.
    #
    # COMPLEXITY GATE
    # The complexity gate requires extra model capacity to earn its place. A more complex
    #   candidate replaces the simple one only if it wins by more than a set margin, not by
    #   the small margin that random variation alone can produce on a ~26-patient inner split.
    # USE_COMPLEXITY_GATE offers the extra-capacity candidates
    #   (attention pooling, and the staged tail when USE_STAGED_UNFREEZE is also True) to
    #   the inner selection, where they must beat the shallow mean-pooling baseline by
    #   INNER_COMPLEXITY_MIN_AUC_GAIN to win. With the gate off the candidate lists collapse
    #   to the single conservative protocol and the inner selection searches learning rate
    #   only.
    USE_STAGED_UNFREEZE           = False
    STAGE1_UNFREEZE_EPOCHS        = 5      # Used only by USE_STAGED_UNFREEZE.
    STAGE2_LR_FACTOR              = 0.5    # Used only by USE_STAGED_UNFREEZE.
    USE_COMPLEXITY_GATE           = False
    INNER_COMPLEXITY_MIN_AUC_GAIN = 0.04   # Used only by USE_COMPLEXITY_GATE.

    # Derived from the two switches above; read by select_protocol_on_inner_split().
    INNER_POOLING_CANDIDATES = ("mean", "attention") if USE_COMPLEXITY_GATE else ("mean",)
    INNER_STAGED_UNFREEZE_CANDIDATES = (
        (False, True) if (USE_COMPLEXITY_GATE and USE_STAGED_UNFREEZE) else (False,)
    )

    # ── 6. Phase-2 backbone movement cap (ON) ────────────────────────────────────
    # The movement cap limits how far fine-tuning can move the pretrained
    #   weights. Transfer learning is only worth doing while the network still resembles
    #   what it learned from millions of images; move it too far on ~160 patients and the
    #   pretraining is destroyed (catastrophic forgetting) with nothing to replace it.
    # Distance is measured as relative RMS displacement from the phase-2
    #   starting weights, RMS(delta) / RMS(anchor), already computed every epoch by
    #   backbone_movement_stats(). Phase 2 stops once the cap is exceeded; the epoch that
    #   crossed it stays eligible for selection, so nothing already reached is discarded.
    #   This replaces "more epochs with early stopping as the only limit": in 77d,
    # RadImageNet-ResNet18 moved 6.65% and its paired OOF AUC ended below its own head-only
    # score. Set USE_MOVEMENT_CAP=False to restore the unbounded behaviour.
    USE_MOVEMENT_CAP          = True
    FINETUNE_MAX_REL_RMS_MOVE = 0.03   # Used only by USE_MOVEMENT_CAP; 3% relative RMS.

    # ══════════════════════════════════════════════════════════════════════════════
    # Always-on settings (not optional components; no switch)
    # ══════════════════════════════════════════════════════════════════════════════

    # LABEL SMOOTHING
    # Label smoothing uses soft targets (e.g. 0.05 / 0.95) instead of hard 0/1, so
    #   the network is never rewarded for being infinitely certain. It improves calibration
    #   and curbs over-confidence, at the cost of slightly blurring the classes.
    # Here, 0.0 disables it, so the loss uses hard labels.
    LABEL_SMOOTHING = 0.0

    # STOCHASTIC DEPTH (drop-path)
    # Stochastic depth randomly skips whole residual blocks during training, so the
    #   network behaves like an ensemble of shallower networks. It regularises without
    #   adding any parameters. Only the timm ResNet-10t backbones support it here.
    # Here, 0.05 drops each eligible block on about 5% of training passes.
    DROP_PATH_RATE = 0.05

    # TRAINING AUGMENTATION
    # Training augmentation applies random label-preserving transformations, so the
    #   network sees a slightly different picture of the same patient each epoch and cannot
    #   memorise one exact image. Kept deliberately minimal here: the crop is already
    #   tumour-centred, and heavier jitter was removed in script 76.
    # Here, only independent random horizontal and vertical flips are applied per patient.
    # TRAIN_AUGMENTATION_LEVEL is set in Section 0 and separately for each arm in
    #   OPTIMIZATION mode. This string is a human-readable echo written to the results.
    TRAIN_AUGMENTATION = str(TRAIN_AUGMENTATION_LEVEL).upper()

    # MULTI-SLICE POOLING
    # Multi-slice pooling combines the slice outputs into one prediction per patient.
    #   "mean" averages them, treating every slice as equally informative.
    # Here, mean pooling is used. ATTENTION_HIDDEN sets the learned scorer's hidden width and is
    #   read only if "attention" pooling is selected (see switch 5).
    SLICE_POOLING     = "mean"
    ATTENTION_HIDDEN  = 128          # Read only when "attention" pooling is selected.

    # DenseNet121 stage-1 capacity:
    # transition3.conv + last 6 denseblock4 layers + norm5 affine = ~20.8% of the
    # torchvision DenseNet121 feature backbone (BN running stats remain frozen).
    DENSENET_STAGE1_LAST_DENSE_LAYERS = 6

    # SMOOTHED EPOCH SELECTION
    # Smoothed selection chooses the best epoch from a rolling average of the validation AUC curve
    #   rather than from its single highest point. On ~26 validation patients the AUC curve
    #   is jagged and its peak is often a one-epoch fluctuation; a smoothed peak identifies a
    #   region where the model was genuinely good, and the best raw epoch inside that region
    #   is retained as the checkpoint.
    # Here, it is enabled with a three-epoch window.
    USE_SMOOTHED_SELECTION = True
    SMOOTHING_WINDOW       = 3

    # ── Fine-tuning-arm policy ──────────────────────────────────────────────────
    # This section represents the fine-tuned CNN arm. Do not replace it with a head-only
    # model when fine-tuning underperforms in an inner fold or in pooled OOF diagnostics;
    # the separate frozen-CNN workflow already provides the non-fine-tuned comparison.
    SKIP_FINETUNE_IF_HEAD_WINS   = False
    SKIP_FINETUNE_MIN_MEDIAN_DELTA = 0.0

    # The final refit must always execute phase 2 so the reported fine-tuning arm is
    # genuinely fine-tuned.
    FORCE_FINAL_REFIT_FINETUNE = True

    # Movement diagnostics. L2-SP/BCE is a scale diagnostic only, since it scales with
    # L2SP_LAMBDA; relative-RMS backbone movement is the warning signal.
    FINETUNE_REL_RMS_MOVE_WARN_MIN    = 0.005   # 0.5% relative RMS movement

    # Judged from the paired pooled outer-OOF AUC delta; the inner-fold median is secondary.
    SKIP_FINETUNE_MIN_POOLED_OOF_DELTA = 0.0

    # Workflow name written to the result tables and to output file names.
    END2END_MODES = ["finetune"]

    BATCH_SIZE = 8
    DROPOUT    = 0.30

    # ── Class-imbalance / decision threshold ──────────────────────────────────────
    # CLASS WEIGHTING (pos_weight)
    # Because progression is the minority outcome, an unweighted loss is minimised
    #   by leaning toward "no progression". pos_weight multiplies the loss contribution of
    #   the positive cases by n_neg/n_pos, making a missed positive as expensive as a false
    #   alarm. It changes the operating point, not the ranking, so AUC is affected only
    #   indirectly through what the network chooses to learn.
    # Here, weighting is enabled at full strength (1.0 = balanced; <1 milder, >1 stronger).
    USE_POS_WEIGHT   = True
    POS_WEIGHT_SCALE = 1.0
    DECISION_THRESHOLD = 0.5

    # ── Checkpoint integrity / reproducibility ──────────────────────────────────
    MIN_RADIMAGENET_LOAD_FRACTION = 0.95
    CHECKPOINT_SHA256 = {}


    def file_sha256(path, chunk_size=1024 * 1024):
        p = Path(str(path))
        if not p.exists():
            return ""
        h = hashlib.sha256()
        with p.open("rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                h.update(chunk)
        return h.hexdigest()

    # IMAGENET_MEAN, IMAGENET_STD and RADIMAGENET_CAFFE_MEAN_BGR are defined once in
    # Section 0, cell 3, next to apply_input_scaling().

    ACTIVE_LABEL_CONFIGS = [c for c in LABELS_CONFIGS
                            if c.get("enabled")]
    print(f"Outcome of this run: {[c['tag'] for c in ACTIVE_LABEL_CONFIGS]}")
    print(f"Backbones: {RUN_BACKBONES}")
    print(f"Channel policy: {CHANNEL_POLICY}")
    print(f"Input size: {CNN_INPUT_SIZE}")
    print(f"Tumour image mode: {TUMOR_CONTEXT_MODE} | ROI-only background={ROI_ONLY_BACKGROUND_VALUE:g}")
    print(f"Slice pooling: {SLICE_POOLING}")
    print(f"Outer CV: {N_FOLDS} folds x {N_REPEATS} repeats "
          f"(estimate only; the final refit remains a single model)")
    print(f"Smoothed epoch selection: {USE_SMOOTHED_SELECTION} (window={SMOOTHING_WINDOW})")

    # ── Optional component switches, reported exactly as configured ──────────────
    # Each row: state, name, one-sentence description, owned parameters.
    _switch_report = [
        ("1. EMA", USE_EMA,
         "running weighted average of the weights over phase-2 steps; the average usually generalises better than the last step",
         f"decay={EMA_DECAY}"),
        ("2. SWA", USE_SWA,
         "plain average of the weights over the final phase-2 epochs, once the LR has decayed and training circles a minimum",
         f"window={SWA_WINDOW}"),
        ("3. TTA", USE_TTA,
         "average the prediction over several transformed copies of the same image at inference; training unchanged",
         f"rotations={USE_TTA_ROTATIONS}"
         f"{'' if not USE_TTA_ROTATIONS else ' ' + str(TTA_ROTATION_DEGREES)}"
         f", used for epoch selection={USE_TTA_FOR_EPOCH_SELECTION}"),
        ("4. Mixup", USE_MIXUP,
         "train on random blends of two patients with correspondingly blended labels, to curb over-confidence",
         f"alpha={MIXUP_ALPHA}"),
        ("5a. Staged unfreezing", USE_STAGED_UNFREEZE,
         "unfreeze a deeper backbone block partway through phase 2, at a reduced learning rate",
         f"after {STAGE1_UNFREEZE_EPOCHS} ep, LR factor={STAGE2_LR_FACTOR}"),
        ("5b. Capacity search + gate", USE_COMPLEXITY_GATE,
         "offer attention pooling / staged tail to the inner selection, where extra capacity must win by a set margin",
         f"min inner AUC gain={INNER_COMPLEXITY_MIN_AUC_GAIN}"),
        ("6. Movement cap", USE_MOVEMENT_CAP,
         "stop phase 2 once the backbone has drifted too far from its pretrained weights (catastrophic-forgetting guard)",
         f"max {100*FINETUNE_MAX_REL_RMS_MOVE:.1f}% relative RMS"),
    ]
    print("Optional component switches:")
    for _label, _on, _what, _params in _switch_report:
        print(f"  [{'ON ' if _on else 'off'}] {_label}")
        print(f"        what it is : {_what}")
        print(f"        parameters : {_params}" + ("" if _on else "   (inert while off)"))
    print(f"Skip fine-tune if head wins: {SKIP_FINETUNE_IF_HEAD_WINS} (inner delta threshold={SKIP_FINETUNE_MIN_MEDIAN_DELTA})")
    print(f"Paired pooled OOF phase-2 support threshold: {SKIP_FINETUNE_MIN_POOLED_OOF_DELTA:+.4f}")
    print(f"Force phase 2 in FINAL refit: {FORCE_FINAL_REFIT_FINETUNE}")
    print(f"Fine-tune diagnostics: relative-RMS warning below {100*FINETUNE_REL_RMS_MOVE_WARN_MIN:.1f}% | L2-SP/BCE reported as scale only")
    print(f"Fine-tune min selection epoch: {FINETUNE_MIN_SELECTION_EPOCH} (warm-up epochs={FINETUNE_WARMUP_EPOCHS})")
    print(f"Gradient clipping: {GRAD_CLIP_NORM:g} (guard only; warn if it binds on >{100*FINETUNE_CLIP_FRACTION_WARN_MAX:.0f}% of phase-2 batches)")
    print(f"Nested selection: inner val={INNER_VAL_FRACTION:.0%} | pooling={INNER_POOLING_CANDIDATES} | staged={INNER_STAGED_UNFREEZE_CANDIDATES}")
    print(
        "Inner-development backbone LR grids: "
        + " | ".join(
            f"{k}=" + ",".join(f"{v:.0e}" for v in vals)
            for k, vals in INNER_BACKBONE_LR_CANDIDATES.items()
        )
    )

    # ── Pre-run compute estimate ─────────────────────────────────────────────────
    # Report the expected cost before training starts.
    # Per backbone and per outer fold: the nested inner selection runs ONE phase 1
    # (shared through phase1_cache) plus one phase 2 per LR candidate; the outer fold
    # then trains its own phase 1 + phase 2 at the selected budget. One final refit
    # per backbone closes the run.
    _outer_folds = int(N_FOLDS) * int(N_REPEATS)
    _n_arms = max(1, len(OPTIMIZATION_ARMS)) if OPTIMIZATION else 1
    _phase1_total = 0
    _phase2_total = 0
    for _bb in RUN_BACKBONES:
        _k = len(inner_backbone_lr_candidates(_bb))
        # Every arm closes with one final refit on the complete development cohort,
        # followed by evaluation on the holdout test cohort.
        _per_arm_p1 = _outer_folds * 2 + 1
        _per_arm_p2 = _outer_folds * (_k + 1) + 1
        _phase1_total += _n_arms * _per_arm_p1
        _phase2_total += _n_arms * _per_arm_p2
    print(
        f"[COMPUTE ESTIMATE] {len(RUN_BACKBONES)} backbone(s) x {_n_arms} arm(s) x "
        f"{_outer_folds} outer folds ({N_FOLDS}x{N_REPEATS})\n"
        f"                   phase-1 (head) runs : {_phase1_total:4d}  (max {EPOCHS_HEAD_MAX} epochs each)\n"
        f"                   phase-2 (FT)   runs : {_phase2_total:4d}  (max {EPOCHS_FINETUNE_MAX} epochs each)\n"
        f"                   Two dials control this cost:\n"
        f"                     N_REPEATS={N_REPEATS} scales BOTH counts linearly "
        f"(set 2, or 1, to cut it; the OOF estimate gets noisier)\n"
        f"                     LR grid size scales the phase-2 count only "
        f"(phase 1 is shared through phase1_cache)\n"
        f"                   Verify the session has budget for this BEFORE starting: a run that "
        f"stops half way produces no usable result."
    )

    def _print_weight_status(label, path):
        p = Path(str(path))
        status = "FOUND" if p.exists() else "MISSING"
        size = f" ({p.stat().st_size/1024**2:.1f} MB)" if p.exists() else ""
        digest = file_sha256(p) if p.exists() else ""
        if digest:
            CHECKPOINT_SHA256[str(label)] = digest
        hash_text = f" | sha256={digest}" if digest else ""
        print(f"  {label}: {status}{size} | {p}{hash_text}")

    # ══════════════════════════════════════════════════════════════════════════
    # Final refit/holdout evaluation + OPTIMIZATION MODE per-arm settings
    # ══════════════════════════════════════════════════════════════════════════
    # True runs the final development-set refit and scores each holdout test set once. An
    # OPTIMIZATION run also needs OPTIMIZATION_EVALUATE_HOLDOUT set to True in Section 0,
    # cell 3: without it the refit and the holdout scoring are skipped for every arm, so
    # the holdout stays unseen while settings are compared. Training always uses
    # development patients only.
    RUN_FINAL_REFIT_AND_HOLDOUT = True
    # Read by the training loop: both conditions above must hold.
    FINAL_REFIT_AND_HOLDOUT_ACTIVE = bool(RUN_FINAL_REFIT_AND_HOLDOUT) and holdout_evaluation_enabled()
    if OPTIMIZATION:
        print("[OPTIMIZATION] Final refit + holdout evaluation: "
              + ("ENABLED for every arm." if FINAL_REFIT_AND_HOLDOUT_ACTIVE
                 else "SKIPPED for every arm, development results only."))

    _FINETUNE_ARM_BASE_SETTINGS = {
        "CNN_INPUT_SIZE": int(CNN_INPUT_SIZE),
        "N_SLICES": int(N_SLICES),
        "SLICE_PLANES": tuple(SLICE_PLANES),
        "TRAIN_AUGMENTATION_LEVEL": str(TRAIN_AUGMENTATION_LEVEL).upper(),
        "TRAIN_AUGMENTATION": str(TRAIN_AUGMENTATION).upper(),
    }

    def restore_finetune_optimization_arm_settings():
        """Restore the pre-comparison fine-tuning settings without touching shared Section 0 values."""
        global CNN_INPUT_SIZE, N_SLICES, SLICE_PLANES
        global TRAIN_AUGMENTATION_LEVEL, TRAIN_AUGMENTATION, _SLICE_CACHE

        CNN_INPUT_SIZE = int(_FINETUNE_ARM_BASE_SETTINGS["CNN_INPUT_SIZE"])
        N_SLICES = int(_FINETUNE_ARM_BASE_SETTINGS["N_SLICES"])
        SLICE_PLANES = tuple(_FINETUNE_ARM_BASE_SETTINGS["SLICE_PLANES"])
        TRAIN_AUGMENTATION_LEVEL = str(_FINETUNE_ARM_BASE_SETTINGS["TRAIN_AUGMENTATION_LEVEL"])
        TRAIN_AUGMENTATION = str(_FINETUNE_ARM_BASE_SETTINGS["TRAIN_AUGMENTATION"])
        try:
            _SLICE_CACHE.clear()
        except Exception:
            pass

    def apply_optimization_arm(arm):
        """Apply ONLY the three optimization factors: size, slice count and augmentation.

        TUMOR_CONTEXT_MODE and ROI_ONLY_BACKGROUND_VALUE are manual global settings
        and are deliberately never changed by an optimization arm.
        """
        global CNN_INPUT_SIZE, N_SLICES, SLICE_PLANES
        global TRAIN_AUGMENTATION_LEVEL, TRAIN_AUGMENTATION, _SLICE_CACHE

        if arm is None:
            return

        CNN_INPUT_SIZE        = int(arm["input_size"])
        N_SLICES              = int(arm["n_slices"])

        if isinstance(N_SLICES, (bool, np.bool_)) or N_SLICES not in (1, 2, 3):
            raise ValueError(f"N_SLICES must be 1, 2 or 3; got {N_SLICES}")

        SLICE_PLANES = ("axial", "sagittal", "coronal")[:int(N_SLICES)]

        TRAIN_AUGMENTATION_LEVEL = str(arm["augmentation"]).upper()
        TRAIN_AUGMENTATION       = TRAIN_AUGMENTATION_LEVEL

        # Cache keys contain output size, slice planes, context mode and ROI background.
        # Clearing between arms additionally prevents memory accumulation.
        try:
            _SLICE_CACHE.clear()
        except Exception:
            pass

        print("\n" + "=" * 78)
        print(
            f"[OPTIMIZATION ARM] {arm['arm_id']}"
            + ("   (REFERENCE)"
               if optimization_arm_is_reference("finetune", arm) else "")
        )
        print(
            f"    input size = {CNN_INPUT_SIZE} px | slices = {N_SLICES} "
            f"in planes {SLICE_PLANES} | augmentation = {TRAIN_AUGMENTATION_LEVEL}"
        )
        print(
            f"    tumour image = {TUMOR_CONTEXT_MODE}  [MANUAL, NOT OPTIMIZED]"
            + (
                f" | background = {ROI_ONLY_BACKGROUND_VALUE:g}"
                if str(TUMOR_CONTEXT_MODE).upper() == "ROI_ONLY"
                else " (background value not used)"
            )
        )
        print("=" * 78)

    # One pass per (arm, mode). With OPTIMIZATION off this is exactly the old loop.
    OPTIMIZATION_ARM_MODE_PAIRS = (
        [(_arm, _mode) for _arm in OPTIMIZATION_ARMS for _mode in END2END_MODES]
        if OPTIMIZATION else
        [(None, _mode) for _mode in END2END_MODES]
    )

    print("RadImageNet weights:")
    _print_weight_status("ResNet-10t", RADIMAGENET_RESNET10T_WEIGHTS_PATH)
    _print_weight_status("ResNet-18", RADIMAGENET_RESNET18_WEIGHTS_PATH)
    _print_weight_status("ResNet-50", RADIMAGENET_RESNET50_WEIGHTS_PATH)
    _print_weight_status("DenseNet-121", RADIMAGENET_DENSENET121_WEIGHTS_PATH)
    print("RadImageNet norm per backbone:",
          {b: radimagenet_norm_for(b) for b in RUN_BACKBONES})

    # Record exact runtime package versions and checkpoint hashes used by this run.
    _runtime_packages = ["numpy", "pandas", "scikit-learn", "SimpleITK", "tqdm", "timm", "torch", "torchvision"]
    _runtime_versions = {}
    for _pkg in _runtime_packages:
        try:
            _runtime_versions[_pkg] = importlib.metadata.version(_pkg)
        except Exception:
            _runtime_versions[_pkg] = "unknown"
    pd.DataFrame([{"package": k, "version": v} for k, v in _runtime_versions.items()]).to_csv(
        Path(OUT_DIR) / "runtime_package_versions.csv", index=False
    )
    pd.DataFrame([
        {"checkpoint": k, "sha256": v} for k, v in CHECKPOINT_SHA256.items()
    ]).to_csv(Path(OUT_DIR) / "checkpoint_sha256.csv", index=False)
    _pretraining_manifest = [
        {"backbone": "radimagenet_resnet10t", "checkpoint_path": RADIMAGENET_RESNET10T_WEIGHTS_PATH, "sha256": file_sha256(RADIMAGENET_RESNET10T_WEIGHTS_PATH), "normalization": radimagenet_norm_for("radimagenet_resnet10t"), "provenance_status": "local checkpoint - verify/document source"},
        {"backbone": "radimagenet_resnet18", "checkpoint_path": RADIMAGENET_RESNET18_WEIGHTS_PATH, "sha256": file_sha256(RADIMAGENET_RESNET18_WEIGHTS_PATH), "normalization": radimagenet_norm_for("radimagenet_resnet18"), "provenance_status": "local checkpoint - verify/document source"},
        {"backbone": "radimagenet_resnet50", "checkpoint_path": RADIMAGENET_RESNET50_WEIGHTS_PATH, "sha256": file_sha256(RADIMAGENET_RESNET50_WEIGHTS_PATH), "normalization": radimagenet_norm_for("radimagenet_resnet50"), "provenance_status": "local checkpoint - verify/document source"},
        {"backbone": "radimagenet_densenet121", "checkpoint_path": RADIMAGENET_DENSENET121_WEIGHTS_PATH, "sha256": file_sha256(RADIMAGENET_DENSENET121_WEIGHTS_PATH), "normalization": radimagenet_norm_for("radimagenet_densenet121"), "provenance_status": "local checkpoint - verify/document source"},
    ]
    pd.DataFrame(_pretraining_manifest).to_csv(Path(OUT_DIR) / "pretraining_checkpoint_manifest.csv", index=False)
    print("Runtime versions:", _runtime_versions)


In [ ]:
# Section 3, cell 9: Shared patient manifest
if globals().get("RUN_CNN_FINETUNE_E2E", True):
    # Discovery, pairing, labels and local copying were completed once in Section 0.
    # The next cell requests independent copies of that fixed shared manifest.
    pass


In [ ]:
# Section 3, cell 10: Label loading
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    # Reuse the label tables read and validated once in Section 0, cell 3.

    def attach_labels(pairs: list, label_df: pd.DataFrame) -> list:
        """Return the unique image/mask pairs that have an exact canonical-ID label."""
        _canonical_unique_patient_ids(
            [item.get("canonical_id", item.get("patient_id")) for item in pairs],
            context="Section 3 image/mask pairs",
        )
        _canonical_unique_patient_ids(label_df["canonical_id"], context="Section 3 label table")
        lmap = dict(zip(label_df["canonical_id"].astype(str), label_df["label"].astype(int)))
        out = []
        for item in pairs:
            cid = canonical_patient_id(item.get("canonical_id", item.get("patient_id")))
            if cid in lmap:
                out.append({**item, "canonical_id": cid, "label": int(lmap[cid])})
        print(f"  Labelled pairs: {len(out)} / {len(pairs)}")
        return out

    # Reuse Section 0's fixed cohort and labels; no Drive scan, CSV read or copying.
    raw_pairs, all_labels_dfs = get_shared_patient_data("Section 3")
    if set(all_labels_dfs) != {cfg["tag"] for cfg in ACTIVE_LABEL_CONFIGS}:
        raise RuntimeError("Active label settings changed. Rerun Section 0, cell 3.")


In [ ]:
# Section 3, cell 11: Shared volume preprocessing
if globals().get("RUN_CNN_FINETUNE_E2E", True):
    # Arm-switching cells may clear this compatibility dictionary. The shared disk
    # cache and bounded CPU cache persist across arms and workflow sections.
    _SLICE_CACHE = {}

    def make_finetune_patient_slices(image_path: str, mask_path: str,
                            out_size: int = None, use_cache: bool = True) -> torch.Tensor:
        """Return a private copy of Section 0's deterministic (slices, 3, H, W) input."""
        if out_size is None:
            out_size = CNN_INPUT_SIZE
        tensor, _ = get_shared_patient_inputs(image_path, mask_path, out_size)
        return tensor


In [ ]:
# Section 3, cell 12: Dataset and augmentation
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    def _augment_slices(xs):
        """Training augmentation at the level named by TRAIN_AUGMENTATION_LEVEL.

        ONE random draw per patient is applied identically to all of that patient's
        slices, so their relative anatomy stays consistent. Input and output are [0, 1]
        float tensors of shape (S, 3, H, W); augmentation runs BEFORE the per-backbone
        normalisation, which is why intensity jitter can assume a [0, 1] range.

        Levels, cumulative:
          NONE            unchanged
          FLIPS           random horizontal + vertical flip          (current protocol)
          BASIC           FLIPS + rotation +/- AUG_ROTATE_DEG
          COMPLEX         BASIC + scale AUG_SCALE_RANGE + translate AUG_TRANSLATE
          COMPLEX_JITTER  COMPLEX + gamma / contrast / brightness / additive noise
        """
        level = str(globals().get("TRAIN_AUGMENTATION_LEVEL", "FLIPS")).upper()
        if level == "NONE":
            return xs
        if level not in ("FLIPS", "BASIC", "COMPLEX", "COMPLEX_JITTER"):
            raise ValueError(
                f"Unknown TRAIN_AUGMENTATION_LEVEL={level!r}; expected one of "
                "NONE, FLIPS, BASIC, COMPLEX, COMPLEX_JITTER."
            )

        # ROI_ONLY: remember which pixels are the constant fill BEFORE anything moves
        # them, so the intensity jitter below can be undone there. Without this, gamma /
        # contrast / brightness / noise modulate the "neutral" background too: measured
        # over 30 draws the fill stayed near 0.5 on average but varied with a
        # within-image sd of 0.020 and an across-draw sd of 0.070, i.e. a random DC
        # level per sample instead of a constant.
        # A tumour pixel lying within 1e-4 of the fill value is frozen too; that is a
        # negligible number of pixels and harmless.
        _roi_only = str(globals().get("TUMOR_CONTEXT_MODE", "MIN_BBOX")).upper() == "ROI_ONLY"
        _bg_value = float(globals().get("ROI_ONLY_BACKGROUND_VALUE", 0.5))
        _bg_mask = ((xs - _bg_value).abs() < 1e-4).to(xs.dtype) if _roi_only else None

        # ── 1. Flips (every level) ──────────────────────────────────────────────
        if random.random() > 0.5:
            xs = torch.flip(xs, dims=[-1])
        if random.random() > 0.5:
            xs = torch.flip(xs, dims=[-2])
        if level == "FLIPS":
            return xs

        # ── 2. Geometry: rotation, and for COMPLEX* also scale and translation ──
        # One affine transform for the whole patient, resampled with border padding so
        # rotation does not paste artificial black wedges into the field of view.
        angle = math.radians(random.uniform(-float(AUG_ROTATE_DEG), float(AUG_ROTATE_DEG)))
        if level == "BASIC":
            scale, tx, ty = 1.0, 0.0, 0.0
        else:
            scale = random.uniform(float(AUG_SCALE_RANGE[0]), float(AUG_SCALE_RANGE[1]))
            # affine_grid works in [-1, 1] coordinates, so the full image width is 2.0:
            # a shift of AUG_TRANSLATE of the image size is 2 * AUG_TRANSLATE here.
            tx = random.uniform(-float(AUG_TRANSLATE), float(AUG_TRANSLATE)) * 2.0
            ty = random.uniform(-float(AUG_TRANSLATE), float(AUG_TRANSLATE)) * 2.0
        cos_a, sin_a = math.cos(angle), math.sin(angle)
        # theta maps OUTPUT coordinates to INPUT sampling coordinates, so the matrix
        # carries 1/scale: sampling a smaller region makes the tumour appear larger.
        inv = 1.0 / float(scale)
        theta = torch.tensor(
            [[cos_a * inv, -sin_a * inv, tx],
             [sin_a * inv,  cos_a * inv, ty]],
            dtype=xs.dtype, device=xs.device,
        ).unsqueeze(0).expand(xs.shape[0], -1, -1)
        grid = F.affine_grid(theta, list(xs.shape), align_corners=False)
        xs = F.grid_sample(xs, grid, mode="bilinear", padding_mode="border",
                           align_corners=False)
        if _bg_mask is not None:
            # Nearest sampling keeps the mask binary; the same grid keeps it aligned to
            # the background after rotation, scaling and translation.
            _bg_mask = F.grid_sample(_bg_mask, grid, mode="nearest",
                                     padding_mode="border", align_corners=False)
        if level in ("BASIC", "COMPLEX"):
            return xs

        # ── 3. Intensity jitter (COMPLEX_JITTER only), on [0, 1] data ───────────
        # Gamma, contrast and brightness are one draw for the whole patient; the additive
        # noise is independent per pixel and per slice, as scanner noise is.
        xs = xs.clamp(0.0, 1.0)
        gamma = random.uniform(float(AUG_GAMMA_RANGE[0]), float(AUG_GAMMA_RANGE[1]))
        xs = xs.clamp_min(1e-6).pow(gamma)
        contrast = 1.0 + random.uniform(-float(AUG_CONTRAST), float(AUG_CONTRAST))
        _mean = xs.mean()
        xs = (xs - _mean) * contrast + _mean
        xs = xs + random.uniform(-float(AUG_BRIGHTNESS), float(AUG_BRIGHTNESS))
        if float(AUG_NOISE_STD) > 0.0:
            xs = xs + torch.randn_like(xs) * float(AUG_NOISE_STD)
        xs = xs.clamp(0.0, 1.0)
        if _bg_mask is not None:
            # Put the neutral fill back exactly, so ROI_ONLY + COMPLEX_JITTER jitters the
            # tumour only and the background stays the constant it was chosen to be.
            xs = torch.where(_bg_mask > 0.5, torch.full_like(xs, _bg_value), xs)
        return xs

    class SliceDataset(Dataset):
        """
        Returns (slices, label) per patient, slices as (S, 3, input_size, input_size) with
        gray3 applied and normalised per `norm`, label as a float32 0/1 scalar.

        norm: one of the input scalings defined in Section 0, cell 3 ("imagenet",
        "unit_interval", "minus_one_one" or "caffe_over_255"), applied by apply_input_scaling().
        """
        def __init__(self, pairs: list, input_size: int = 224,
                     augment: bool = False, norm: str = "imagenet"):
            self.pairs      = pairs
            self.input_size = input_size
            self.augment    = augment
            self.norm       = str(norm).lower()

        def __len__(self):
            return len(self.pairs)

        def __getitem__(self, idx):
            item = self.pairs[idx]
            # Slices arrive already cropped at the backbone's input size, not upsampled.
            xs = make_finetune_patient_slices(item["image"], item["mask"], out_size=self.input_size)

            # gray3: replicate T1cc channel into all 3 channels, drop mask
            xs = xs[:, 0:1, :, :].repeat(1, 3, 1, 1)  # (S, 3, input_size, input_size)

            # Training-only augmentation: geometry, intensity and flips, applied on [0, 1]
            # data before per-backbone normalisation.
            if self.augment:
                xs = _augment_slices(xs)

            # Per-backbone input scaling, the same function in all three sections.
            xs = apply_input_scaling(xs, self.norm)  # (S, 3, H, W)

            label = torch.tensor(float(item["label"]), dtype=torch.float32)
            return xs, label


In [ ]:
# Section 3, cell 13: Backbone builders
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    def _make_timm_resnet10t_backbone() -> tuple:
        """Return (feature_net, feature_dim) for ResNet-10t, without head."""
        model = timm.create_model("resnet10t", pretrained=False, num_classes=0,
                                  global_pool="avg", drop_path_rate=float(DROP_PATH_RATE))
        dim = model.num_features
        return model, int(dim)

    def _make_torchvision_densenet121_backbone() -> tuple:
        """Return (feature_net, feature_dim) for DenseNet-121, without classifier."""
        net = tv_models.densenet121(weights=None)
        dim = net.classifier.in_features  # 1024
        net.classifier = nn.Identity()
        return net, int(dim)

    def _make_torchvision_resnet18_backbone() -> tuple:
        """Return (feature_net, feature_dim) for ResNet-18, without classifier."""
        net = tv_models.resnet18(weights=None)
        dim = net.fc.in_features  # 512
        net.fc = nn.Identity()  # keep global avg-pool, drop the 1000-way head
        return net, int(dim)

    def _make_torchvision_resnet50_backbone() -> tuple:
        """Return (feature_net, feature_dim) for ResNet-50, without classifier."""
        net = tv_models.resnet50(weights=None)
        dim = net.fc.in_features  # 2048
        net.fc = nn.Identity()  # keep global avg-pool, drop the 1000-way head
        return net, int(dim)


    def load_radimagenet_weights(net, weights_path, architecture_name="RadImageNet backbone"):
        p = Path(weights_path) if weights_path else None
        if (p is None) or (not p.exists()):
            parent = p.parent if p is not None else Path(".")
            nearby = []
            if parent.exists():
                nearby = sorted([str(x) for x in parent.rglob("*.pt")][:20] + [str(x) for x in parent.rglob("*.pth")][:20])
            raise FileNotFoundError(
                f"{architecture_name} weights not found at the resolved path. "
                "If the file is visible in Drive, check that Cell 2 runs again after Drive mounts, "
                "and that the filename/folder matches exactly. "
                f"Looked at: {weights_path}. "
                f"Nearby .pt/.pth files under {parent}: {nearby[:20]}"
            )

        try:
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
        except TypeError:
            raw = torch.load(str(p), map_location="cpu")

        def _unwrap_state_dict(obj):
            """Find the actual tensor state_dict inside common checkpoint wrappers."""
            if hasattr(obj, "state_dict"):
                obj = obj.state_dict()
            if not isinstance(obj, dict):
                return obj, "<non-dict checkpoint>"

            # Already a state_dict.
            tensor_like_values = [v for v in obj.values() if torch.is_tensor(v)]
            if len(tensor_like_values) >= 10:
                return obj, "<top-level>"

            for key in (
                "state_dict", "model_state_dict", "model", "net", "network",
                "backbone", "encoder", "weights", "params", "module",
                "model_weights", "weight", "pretrained_dict",
            ):
                if key in obj:
                    nested, where = _unwrap_state_dict(obj[key])
                    if isinstance(nested, dict):
                        return nested, key if where == "<top-level>" else f"{key}.{where}"
            return obj, "<dict without tensor state_dict>"

        sd, sd_location = _unwrap_state_dict(raw)
        if not isinstance(sd, dict):
            raise RuntimeError(
                f"{architecture_name} checkpoint at {p} did not contain a PyTorch state_dict. "
                f"Loaded object type: {type(sd)}"
            )

        current = net.state_dict()
        loadable = {}

        import re as _re

        def _densenet_legacy(key):
            """Convert older torchvision DenseNet keys (norm.N/conv.N) to modern (normN/convN).

            torchvision below ~0.12 serialised dense-layer sub-modules as Sequential
            children, e.g. features.denseblock1.denselayer1.norm.1.weight against the
            modern ...norm1.weight. Pass 1 finds no matches on such a checkpoint.
            """
            key = _re.sub(r'\.norm\.(\d+)', r'.norm\1', key)
            key = _re.sub(r'\.conv\.(\d+)', r'.conv\1', key)
            return key

        def candidate_keys(key):
            # Try the key as-is, then strip known wrapper prefixes only. The
            # torchvision DenseNet "features." prefix is genuine and is kept.
            candidates = []
            seen = set()
            queue = [key]
            prefixes = (
                "module.", "backbone.", "model.", "encoder.", "resnet.",
                "base_model.", "feature_extractor.", "network.", "net.",
                "densenet.", "inception.", "inception_v3.",
                "0.", "1.", "2.", "3.",   # nn.Sequential / numeric-index wrappers
                "_orig_mod.",             # torch.compile saves with this prefix (e.g. ResNet-18)
            )
            while queue:
                cur = queue.pop(0)
                if cur not in seen:
                    seen.add(cur)
                    candidates.append(cur)
                    for pre in prefixes:
                        if cur.startswith(pre):
                            stripped = cur[len(pre):]
                            if stripped not in seen:
                                queue.append(stripped)
            return candidates

        _HEAD_PREFIXES = ("fc.", "classifier.", "AuxLogits.fc.")
        _pass3_rescued = 0  # initialised here; may be updated in Pass 3 below

        tensor_items = []
        non_tensor_keys = []
        for k, v in sd.items():
            if not torch.is_tensor(v):
                non_tensor_keys.append(str(k))
                continue
            tensor_items.append((str(k), v))
            for nk in candidate_keys(str(k)):
                if nk.startswith(_HEAD_PREFIXES):
                    continue
                if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                    loadable[nk] = v
                    break

        # Pass 2, older DenseNet key names: "norm.1"/"conv.1" -> "norm1"/"conv1".
        # Runs for DenseNet, or whenever pass 1 matched fewer than five tensors.
        _arch_lower = architecture_name.lower()
        if "densenet" in _arch_lower or len(loadable) < 5:
            _matched_so_far = set(loadable.keys())
            for k, v in tensor_items:
                modern_k = _densenet_legacy(k)
                if modern_k == k:
                    continue  # key unchanged; already tried in pass 1
                for nk in candidate_keys(modern_k):
                    if nk.startswith(_HEAD_PREFIXES) or nk in _matched_so_far:
                        continue
                    if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                        loadable[nk] = v
                        _matched_so_far.add(nk)
                        break

        # Pass 3: sequential wrappers. RadImageNet checkpoints saved through an
        # nn.Sequential('backbone') use backbone.N.subkey instead of named keys, e.g.
        # ResNet-50 backbone.0.weight -> conv1.weight, DenseNet-121 backbone.0.conv0.weight
        # -> features.conv0.weight. Keys are remapped, then matching is retried.
        _inception_had_aux = False  # default; set precisely in Pass 3 if needed
        _pass3_rescued = 0
        if len(loadable) < 5:
            _matched_so_far = set(loadable.keys())
            _ckpt_keys = [k for k, _ in tensor_items]
            _uses_backbone_prefix = any(k.startswith("backbone.") for k in _ckpt_keys)

            # InceptionV3 variant is identified by the largest backbone.N index:
            #   max_idx <= 17  no AuxLogits, backbone.15/16/17 = Mixed_7a/7b/7c
            #   max_idx >= 18  AuxLogits present, 15 = AuxLogits, 16/17/18 = Mixed_7a/7b/7c
            # Each variant needs its own map. Using the wrong one either leaves 54 tensors
            # unmatched or, because Mixed_7b and Mixed_7c share shapes, loads them into the
            # wrong block; the load-fraction check below catches both.
            _inception_had_aux = False
            if _uses_backbone_prefix and "inception" in _arch_lower:
                _bb_idxs = {
                    int(_m.group(1)) for _k in _ckpt_keys
                    for _m in [_re.match(r"backbone\.(\d+)\.", _k)] if _m
                }
                _inception_had_aux = max(_bb_idxs, default=0) >= 18

            def _normalize_key(k):
                if not _uses_backbone_prefix:
                    return k
                if "densenet" in _arch_lower:
                    if k.startswith("backbone.0."):
                        return "features." + k[len("backbone.0."):]
                elif "inception" in _arch_lower:
                    _imap = {
                        "backbone.0.":  "Conv2d_1a_3x3.", "backbone.1.":  "Conv2d_2a_3x3.",
                        "backbone.2.":  "Conv2d_2b_3x3.",
                        "backbone.4.":  "Conv2d_3b_1x1.", "backbone.5.":  "Conv2d_4a_3x3.",
                        "backbone.7.":  "Mixed_5b.",       "backbone.8.":  "Mixed_5c.",
                        "backbone.9.":  "Mixed_5d.",       "backbone.10.": "Mixed_6a.",
                        "backbone.11.": "Mixed_6b.",       "backbone.12.": "Mixed_6c.",
                        "backbone.13.": "Mixed_6d.",       "backbone.14.": "Mixed_6e.",
                    }
                    if _inception_had_aux:
                        # AuxLogits present (max index >= 18).
                        _imap.update({
                            "backbone.15.": "AuxLogits.",
                            "backbone.16.": "Mixed_7a.",
                            "backbone.17.": "Mixed_7b.",
                            "backbone.18.": "Mixed_7c.",
                        })
                    else:
                        # No AuxLogits (max index <= 17).
                        _imap.update({
                            "backbone.15.": "Mixed_7a.",
                            "backbone.16.": "Mixed_7b.",
                            "backbone.17.": "Mixed_7c.",
                        })
                    for old, dst in _imap.items():
                        if k.startswith(old):
                            return dst + k[len(old):]
                else:  # resnet-50 / resnet-18 / resnet-10t
                    # backbone.N indexes the top-level ResNet children: 0=conv1, 1=bn1,
                    # 4..7=layer1..4. Activations and pooling carry no parameters and are
                    # absent from the checkpoint. The ResNet-10t stem sits inside child 0 as
                    # a conv1 sequence, so the same map applies with the inner index kept.
                    _rmap = {
                        "backbone.0.": "conv1.", "backbone.1.": "bn1.",
                        "backbone.4.": "layer1.", "backbone.5.": "layer2.",
                        "backbone.6.": "layer3.", "backbone.7.": "layer4.",
                    }
                    for old, dst in _rmap.items():
                        if k.startswith(old):
                            return dst + k[len(old):]
                return k

            for orig_k, v in tensor_items:
                norm_k = _normalize_key(orig_k)
                for nk in candidate_keys(norm_k):
                    if nk.startswith(_HEAD_PREFIXES) or nk in _matched_so_far:
                        continue
                    if nk in current and tuple(current[nk].shape) == tuple(v.shape):
                        loadable[nk] = v
                        _matched_so_far.add(nk)
                        _pass3_rescued += 1
                        break

            if _pass3_rescued:
                print(f"[RadImageNet] Pass 3 (sequential-backbone remapping) rescued {_pass3_rescued} tensors")

        sample_ckpt = [(k, tuple(v.shape)) for k, v in tensor_items[:12]]
        sample_expected = [(k, tuple(v.shape)) for k, v in list(current.items())[:12]]

        print(f"[RadImageNet] {architecture_name}: checkpoint={p.name} | state_dict_location={sd_location}")
        print(f"[RadImageNet] tensor keys in checkpoint: {len(tensor_items)} | non-tensor keys: {len(non_tensor_keys)}")
        print(f"[RadImageNet] sample checkpoint keys: {sample_ckpt}")
        print(f"[RadImageNet] sample expected torchvision keys: {sample_expected}")

        if len(loadable) == 0:
            raise RuntimeError(
                f"{architecture_name} loaded 0 compatible tensors from {p}. "
                "This usually means the checkpoint key names do not match the torchvision "
                "architecture expected for this backbone.\n"
                "Common causes:\n"
                "  • Wrong .pt file (e.g. ResNet50 file used for DenseNet121).\n"
                "  • DenseNet saved with old torchvision (<0.12): norm.1/conv.1 vs norm1/conv1 "
                "(pass 2 should have fixed this automatically. If you still see this, the file "
                "may not be a torchvision DenseNet at all).\n"
                "  • InceptionV3 checkpoint from a custom/non-torchvision implementation.\n"
                "  • Checkpoint is a full Keras/TF model, not a PyTorch state_dict.\n"
                "Inspect the sample checkpoint keys printed above vs. the expected torchvision keys "
                "and add an architecture-specific key-conversion pass if needed."
            )

        missing, unexpected = net.load_state_dict(loadable, strict=False)
        _used_legacy = ("densenet" in architecture_name.lower() or len(loadable) < 5) and any(
            _densenet_legacy(k) != k for k, _ in tensor_items
        )
        _strategy = ("densenet_legacy_keys" if _used_legacy
                     else "sequential_backbone_remapping" if _pass3_rescued
                     else "standard")
        print(f"[RadImageNet] {architecture_name}: loaded {len(loadable)}/{len(current)} compatible "
              f"backbone tensors from {p.name} (strategy={_strategy})")

        # ── Load completeness ──────────────────────────────────────────────────────
        # Counting the target model's non-head tensors gives an exact load fraction for
        # every architecture, so the incomplete-load warning stays meaningful.
        _n_expected = len([k for k in current if not k.startswith(_HEAD_PREFIXES)])
        _load_frac = len(loadable) / max(1, _n_expected)
        if _load_frac < float(MIN_RADIMAGENET_LOAD_FRACTION):
            raise RuntimeError(
                f"[RadImageNet][FATAL] {architecture_name}: loaded only "
                f"{len(loadable)}/{_n_expected} expected non-head tensors ({_load_frac:.1%}); "
                f"required minimum={float(MIN_RADIMAGENET_LOAD_FRACTION):.1%}. "
                "A partial checkpoint is not accepted for publication runs. Inspect the "
                "checkpoint provenance, architecture and key mapping before continuing."
            )

        # ── Strict InceptionV3 check ──────────────────────────────────────────────
        # Key order alone cannot confirm the pass-3 index map, and equal shapes can hide a
        # wrong assignment. A wrong map leaves target slots empty, so the load fraction
        # falls; a correct load fills all 94 convolution blocks and ~98% of non-head slots,
        # the remainder being the auxiliary branch. Anything below 0.95 raises.
        if "inception" in _arch_lower:
            _bb_indices = sorted({
                int(_m.group(1)) for _k, _ in tensor_items
                for _m in [_re.match(r"backbone\.(\d+)\.", _k)] if _m
            })
            print(f"[RadImageNet] {architecture_name}: checkpoint backbone.N block "
                  f"indices present: {_bb_indices} | inferred aux_logits_when_saved={_inception_had_aux}")
            if _load_frac < 0.95:
                raise RuntimeError(
                    f"[RadImageNet][FATAL] {architecture_name}: loaded only "
                    f"{len(loadable)}/{_n_expected} non-head tensors ({_load_frac:.0%}); "
                    "a correct InceptionV3 load is ~98%. A partial load means the "
                    "backbone.N -> torchvision index map (Pass 3) does not match this "
                    "checkpoint's Sequential order, which risks silently loading weights "
                    "into the WRONG Inception block. "
                    f"Observed backbone.N indices: {_bb_indices}. "
                    "Validate them against the torchvision inception_v3 child order "
                    "(Conv2d_1a/2a/2b, [maxpool], Conv2d_3b/4a, [maxpool], "
                    "Mixed_5b/5c/5d/6a/6b/6c/6d/6e, [AuxLogits], Mixed_7a/7b/7c) and "
                    "correct the _imap in Pass 3 before trusting InceptionV3 features."
                )

        # ── BatchNorm statistics ──────────────────────────────────────────────────
        # A missing running_mean/running_var makes eval() fall back to mean 0, variance 1
        # for that layer, which silently corrupts the features.
        _bn_expected_keys = [k for k in current if k.endswith((".running_mean", ".running_var"))]
        _bn_missing_keys  = [k for k in missing  if k.endswith((".running_mean", ".running_var"))]
        if _bn_missing_keys:
            _bn_frac = len(_bn_missing_keys) / max(1, len(_bn_expected_keys))
            if _bn_frac > 0.5:
                raise RuntimeError(
                    f"[RadImageNet][FATAL] {architecture_name}: "
                    f"{len(_bn_missing_keys)}/{len(_bn_expected_keys)} BatchNorm "
                    "running_mean/running_var buffers MISSING from checkpoint. "
                    "In eval() mode every BN layer normalises with mean=0/var=1, "
                    "producing silently invalid features. The checkpoint likely stores "
                    "only learnable parameters (conv weights + BN scale/bias) without "
                    "running statistics. "
                    f"Sample missing BN keys: {_bn_missing_keys[:4]}"
                )
            print(
                f"[RadImageNet][warning] {architecture_name}: "
                f"{len(_bn_missing_keys)}/{len(_bn_expected_keys)} BN running buffers "
                "missing. Those layers use default statistics in eval() mode."
            )
        else:
            print(
                f"[RadImageNet] {architecture_name}: all "
                f"{len(_bn_expected_keys)} BN running buffers loaded OK."
            )
        return net


    # CPU cache of the original pretrained weights. Every fold builds a fresh model from
    # these same starting weights, so no learned weights, BatchNorm statistics or optimiser
    # state from one fold could influence another fold. Each checkpoint is read and remapped only once.
    _BACKBONE_SD_CACHE = {}
    _BACKBONE_META_CACHE = {}



    def _clone_state_dict_to_cpu(module: nn.Module):
        """Create a separate CPU copy of the model weights."""
        return {
            key: value.detach().cpu().clone()
            for key, value in module.state_dict().items()
        }


    def _build_imagenet_architecture(backbone_name: str, pretrained: bool):
        """Build an ImageNet model and optionally load pretrained weights."""
        name = str(backbone_name).lower()

        if name == "imagenet_resnet10t":
            net = timm.create_model(
                "resnet10t",
                pretrained=bool(pretrained),
                num_classes=0,
                global_pool="avg",
                drop_path_rate=float(DROP_PATH_RATE),
            )
            return net, int(net.num_features)

        def _torchvision_model(factory):
            if pretrained:
                try:
                    return factory(weights="DEFAULT")
                except TypeError:
                    return factory(pretrained=True)
            try:
                return factory(weights=None)
            except TypeError:
                return factory(pretrained=False)

        if name == "imagenet_resnet18":
            net = _torchvision_model(tv_models.resnet18)
            dim = int(net.fc.in_features)
            net.fc = nn.Identity()
        elif name == "imagenet_resnet50":
            net = _torchvision_model(tv_models.resnet50)
            dim = int(net.fc.in_features)
            net.fc = nn.Identity()
        elif name == "imagenet_densenet121":
            net = _torchvision_model(tv_models.densenet121)
            dim = int(net.classifier.in_features)
            net.classifier = nn.Identity()
        else:
            raise ValueError(f"Unknown ImageNet backbone: {backbone_name!r}")

        return net, dim


    def build_feature_backbone(backbone_name: str):
        """
        Return a fresh (backbone_nn_module, feature_dim, norm_mode).

        Every call returns a newly constructed model object. The model starts from
        the same pristine pretrained weights held in _BACKBONE_SD_CACHE, so folds
        remain independent without repeating remote downloads or Drive checkpoint
        remapping.
        """
        name = str(backbone_name).lower()

        imagenet_names = {
            "imagenet_resnet10t",
            "imagenet_resnet18",
            "imagenet_resnet50",
            "imagenet_densenet121",
        }

        if name in imagenet_names:
            if name not in _BACKBONE_SD_CACHE:
                # Only the first call touches the network; later ones hit the disk cache.
                net, dim = _build_imagenet_architecture(name, pretrained=True)
                _BACKBONE_SD_CACHE[name] = _clone_state_dict_to_cpu(net)
                _BACKBONE_META_CACHE[name] = {
                    "feature_dim": int(dim),
                    "norm": "imagenet",
                    "source": "ImageNet pretrained",
                }
                print(
                    f"[ImageNet cache primed] {name}: pretrained weights loaded once "
                    f"| feature_dim={dim}"
                )
            else:
                # Architecture only: no download, no network access.
                net, dim = _build_imagenet_architecture(name, pretrained=False)
                net.load_state_dict(_BACKBONE_SD_CACHE[name], strict=True)

            meta = _BACKBONE_META_CACHE[name]
            return net, int(meta["feature_dim"]), str(meta["norm"]).lower()

        if name == "radimagenet_resnet10t":
            net, dim = _make_timm_resnet10t_backbone()
            weights_path, label = RADIMAGENET_RESNET10T_WEIGHTS_PATH, "RadImageNet ResNet-10t"
        elif name == "radimagenet_resnet18":
            net, dim = _make_torchvision_resnet18_backbone()
            weights_path, label = RADIMAGENET_RESNET18_WEIGHTS_PATH, "RadImageNet ResNet-18"
        elif name == "radimagenet_resnet50":
            net, dim = _make_torchvision_resnet50_backbone()
            weights_path, label = RADIMAGENET_RESNET50_WEIGHTS_PATH, "RadImageNet ResNet-50"
        elif name == "radimagenet_densenet121":
            net, dim = _make_torchvision_densenet121_backbone()
            weights_path, label = RADIMAGENET_DENSENET121_WEIGHTS_PATH, "RadImageNet DenseNet-121"
        else:
            raise ValueError(f"Unknown backbone: {backbone_name!r}")

        if name not in _BACKBONE_SD_CACHE:
            net = load_radimagenet_weights(net, weights_path, label)
            _BACKBONE_SD_CACHE[name] = _clone_state_dict_to_cpu(net)
            _BACKBONE_META_CACHE[name] = {
                "feature_dim": int(dim),
                "norm": radimagenet_norm_for(name),
                "source": label,
                "checkpoint_path": str(weights_path),
                "checkpoint_sha256": file_sha256(weights_path),
            }
            print(
                f"[RadImageNet cache primed] {name}: checkpoint read/remapped once "
                f"| feature_dim={dim}"
            )
        else:
            net.load_state_dict(_BACKBONE_SD_CACHE[name], strict=True)

        meta = _BACKBONE_META_CACHE[name]
        return net, int(meta["feature_dim"]), str(meta["norm"]).lower()


    def prime_backbone_initial_state(backbone_name: str):
        """
        Load/download one backbone before CV begins, then discard the temporary model.

        This makes any network/download problem occur once at a clearly labelled
        preflight step instead of appearing to freeze in the middle of fold 1.
        """
        name = str(backbone_name).lower()
        if name in _BACKBONE_SD_CACHE:
            return

        print(f"\n[backbone setup] Preparing original starting weights for: {name}")
        temporary_model, feature_dim, norm_mode = build_feature_backbone(name)
        del temporary_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(
            f"[backbone setup complete] {name} | feature_dim={feature_dim} "
            f"| norm={norm_mode} | all folds use the CPU weight cache"
        )

    def freeze_backbone(net: nn.Module):
        for p in net.parameters():
            p.requires_grad = False


    def _bn_affine_exempt_ids(module: nn.Module):
        """ids() of BatchNorm modules allowed to train their affine gamma/beta despite
        the global BN freeze. Running statistics are never exempted."""
        return getattr(module, "_bn_affine_exempt_ids", frozenset())


    def _register_bn_affine_exempt(net: nn.Module, modules):
        """Record BN modules as affine-trainable for the lifetime of this backbone.

        The exemption is stored on the backbone object because
        _freeze_batchnorm_affine_and_stats() is re-applied at the start of EVERY training
        epoch; a one-shot requires_grad flip would be silently undone on the next epoch.
        """
        current = set(getattr(net, "_bn_affine_exempt_ids", set()))
        for m in (modules or []):
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                current.add(id(m))
        net._bn_affine_exempt_ids = current
        return current


    def _freeze_batchnorm_affine_and_stats(module: nn.Module):
        """Freeze BatchNorm running statistics everywhere, and affine gamma/beta everywhere
        except modules registered via _register_bn_affine_exempt().

        Running statistics stay in eval mode unconditionally, since small cohorts adapt them
        unstably. The affine exemption is architecture-specific: DenseNet concatenates rather
        than adds, so its final BN (features.norm5) is the only place the concatenated 1024-d
        classifier input can be rescaled, with no downstream trainable convolution to absorb
        it. A ResNet bottleneck needs no exemption, since its trainable conv3 sits directly
        before bn3.

        Called after model.train() and after every unfreeze.
        """
        exempt = _bn_affine_exempt_ids(module)
        for m in module.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                m.eval()
                _trainable = id(m) in exempt
                for p in m.parameters(recurse=False):
                    p.requires_grad = _trainable


    def _unfreeze_non_bn(module: nn.Module, bn_exempt_ids=frozenset()):
        """Unfreeze trainable weights inside a module. BN affine stays frozen unless the
        BN module is listed in bn_exempt_ids; BN running statistics always stay frozen."""
        for m in module.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                m.eval()
                _trainable = id(m) in bn_exempt_ids
                for p in m.parameters(recurse=False):
                    p.requires_grad = _trainable
            else:
                for p in m.parameters(recurse=False):
                    p.requires_grad = True


    def _module_children(module):
        return list(module.children()) if module is not None else []


    def small_data_finetune_profile(backbone_name: str):
        """Architecture-aware complexity profile.

        Trainable-tail size remains architecture-specific, but the phase-2 backbone LR
        is no longer guessed from architecture: it is selected on inner development data.
        """
        name = str(backbone_name).lower()
        if "resnet10t" in name:
            return {"complexity_min_gain": 0.03, "profile": "small_resnet_balanced"}
        if "resnet18" in name:
            return {"complexity_min_gain": 0.04, "profile": "medium_resnet_balanced"}
        if "resnet50" in name:
            return {"complexity_min_gain": 0.05, "profile": "large_resnet_conservative"}
        if "densenet121" in name:
            return {"complexity_min_gain": 0.05, "profile": "large_densenet"}
        return {"complexity_min_gain": INNER_COMPLEXITY_MIN_AUC_GAIN, "profile": "conservative_default"}


    def _tail_modules_for_stage(net: nn.Module, backbone_name: str, stage: int):
        """Return (description, modules, bn_affine_exempt) for architecture-aware
        small-data unfreezing.

        The third element lists BatchNorm modules whose affine parameters should train
        despite the global BN freeze. It is empty for every ResNet (see
        _freeze_batchnorm_affine_and_stats for why only DenseNet needs it).
        """
        name = str(backbone_name).lower()
        stage = int(stage)

        if "resnet" in name:
            layer4 = getattr(net, "layer4", None)
            layer3 = getattr(net, "layer3", None)
            l4 = _module_children(layer4)
            l3 = _module_children(layer3)
            if "resnet10t" in name:
                # ResNet10t is very small: unfreezing all of layer4 exposed ~3.67M params,
                # roughly three quarters of the feature backbone. To keep adaptation
                # comparable with the ~20% tails used by the other CNNs, stage 1 exposes
                # only conv1 of its final BasicBlock (~1.18M params, about one quarter).
                final_block = l4[-1] if l4 else None
                if final_block is None:
                    return "ResNet10t final layer4 block unavailable", [], []
                if stage == 1:
                    conv1 = getattr(final_block, "conv1", None)
                    return "ResNet10t final layer4 block conv1 only (~25%)", ([conv1] if conv1 is not None else []), []
                # Future staged ablation: add the second convolution of the same residual block.
                conv2 = getattr(final_block, "conv2", None)
                return "ResNet10t final layer4 block conv2 (stage 2)", ([conv2] if conv2 is not None else []), []

            # ResNet18: with staged unfreezing disabled, stage 1 exposes only conv2 of the
            # last layer4 BasicBlock (~2.36M params, about 21% of the feature backbone).
            # Stage 2, if explicitly enabled in a future ablation, adds conv1 of that block.
            # BatchNorm remains frozen.
            if "resnet18" in name:
                final_block = l4[-1] if l4 else None
                if final_block is None:
                    return f"{backbone_name} final layer4 block unavailable", [], []
                if stage == 1:
                    conv2 = getattr(final_block, "conv2", None)
                    return f"{backbone_name} final layer4 block conv2 only", ([conv2] if conv2 is not None else []), []
                conv1 = getattr(final_block, "conv1", None)
                return f"{backbone_name} final layer4 block conv1 (stage 2)", ([conv1] if conv1 is not None else []), []

            # ResNet50: start at the final bottleneck unit and, if justified, add only the
            # unit immediately before it.
            if stage == 1:
                return f"{backbone_name} final layer4 residual unit", ([l4[-1]] if l4 else []), []
            if len(l4) >= 2:
                return f"{backbone_name} preceding layer4 residual unit", [l4[-2]], []
            return f"{backbone_name} final layer3 residual unit", ([l3[-1]] if l3 else []), []

        if "densenet121" in name:
            features = net.features if hasattr(net, "features") else net
            block4 = getattr(features, "denseblock4", None)
            transition3 = getattr(features, "transition3", None)
            norm5 = getattr(features, "norm5", None)
            layers = _module_children(block4)

            # DenseNet needs a broader but still conservative trainable region.
            # Script 74 previously exposed transition3.conv + norm5 only (~7.6% of the
            # backbone), whereas script 26 exposed the full denseblock4 (~31%).
            #
            # Stage 1 now keeps transition3's global 512-channel re-basing AND exposes the
            # last 6 denseblock4 layers. With BN affine frozen inside those layers and norm5
            # affine exempted, this is ~1.448M trainable backbone parameters, ~20.8% of the
            # torchvision DenseNet121 feature backbone.
            if stage == 1:
                if not layers:
                    return "DenseNet121 denseblock4 tail unavailable", [], []
                n_tail = min(int(DENSENET_STAGE1_LAST_DENSE_LAYERS), len(layers))
                chosen_tail = layers[-n_tail:]
                modules = ([transition3] if transition3 is not None else []) + chosen_tail
                exempt = [m for m in (norm5,) if m is not None]
                if not modules and not exempt:
                    return "DenseNet121 transition3/denseblock4-tail/norm5 unavailable", [], []
                return (
                    f"DenseNet121 transition3 conv + last {n_tail} denseblock4 layers + norm5 affine (~20.8%)",
                    modules,
                    exempt,
                )

            # If staged unfreezing is enabled in a future ablation, add the four dense
            # layers immediately preceding the stage-1 tail; do not re-add already-trainable layers.
            if not layers:
                return "DenseNet121 denseblock4 tail unavailable", [], []
            n_tail = min(int(DENSENET_STAGE1_LAST_DENSE_LAYERS), len(layers))
            remaining = layers[:-n_tail]
            chosen = remaining[-4:] if remaining else []
            return "DenseNet121 preceding 4 denseblock4 layers", chosen, []

        return f"{backbone_name} unidentified tail", [], []


    def unfreeze_last_block(net: nn.Module, backbone_name: str):
        """Unfreeze an architecture-specific, small-data-safe tail segment."""
        description, modules, bn_exempt = _tail_modules_for_stage(net, backbone_name, stage=1)
        before = {id(p) for p in net.parameters() if p.requires_grad}
        exempt_ids = _register_bn_affine_exempt(net, bn_exempt)
        for module in modules:
            _unfreeze_non_bn(module, exempt_ids)
        _freeze_batchnorm_affine_and_stats(net)
        newly = [p for p in net.parameters() if p.requires_grad and id(p) not in before]
        if newly:
            print(f"  Unfrozen small-data tail: {description} (+{sum(p.numel() for p in newly):,} trainable params)")
        else:
            print(f"  [warning] could not unfreeze small-data tail: {description}")
        return newly


    def unfreeze_deeper_block(net: nn.Module, backbone_name: str):
        """Add only the immediately adjacent architecture-specific tail segment."""
        description, modules, bn_exempt = _tail_modules_for_stage(net, backbone_name, stage=2)
        before = {id(p) for p in net.parameters() if p.requires_grad}
        exempt_ids = _register_bn_affine_exempt(net, bn_exempt)
        for module in modules:
            _unfreeze_non_bn(module, exempt_ids)
        _freeze_batchnorm_affine_and_stats(net)
        newly = [p for p in net.parameters() if p.requires_grad and id(p) not in before]
        if newly:
            print(f"  Staged unfreeze: {description} (+{sum(p.numel() for p in newly):,} new trainable params)")
        else:
            print(f"  [staged unfreeze] no additional parameters available for: {description}")
        return newly


In [ ]:
# Section 3, cell 14: Multi-slice classifier
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    class GatedAttentionPool(nn.Module):
        """
        Gated attention pooling over the S slice embeddings (Ilse et al. 2018).
        Weights the S per-slice features by a learned, patient-local softmax that
        lets the slice most predictive of the outcome dominate the pooled feature.
        Params: approximately 2 * feat_dim * hidden + hidden (plus linear biases).
        """
        def __init__(self, feature_dim: int, hidden_dim: int = 128):
            super().__init__()
            self.attn_u = nn.Linear(feature_dim, hidden_dim)
            self.attn_v = nn.Linear(feature_dim, hidden_dim)
            self.attn_w = nn.Linear(hidden_dim, 1)

        def forward(self, feats: torch.Tensor, slice_mask: torch.Tensor = None) -> torch.Tensor:
            # feats: (B, S, D). Returns (B, D).
            u = torch.tanh(self.attn_u(feats))
            v = torch.sigmoid(self.attn_v(feats))
            scores = self.attn_w(u * v).squeeze(-1)          # (B, S)
            if slice_mask is not None:
                # Padded slots must receive exactly zero attention weight.
                scores = scores.masked_fill(~slice_mask, float("-inf"))
            weights = torch.softmax(scores, dim=1)           # (B, S)
            pooled = (feats * weights.unsqueeze(-1)).sum(1)  # (B, D)
            return pooled


    class MultiSliceCNNClassifier(nn.Module):
        """
        Wraps a CNN backbone with a binary classification head.
        Input: (B, S, 3, H, W)  = S slices per patient
        Output: (B,)             = raw logits (use BCEWithLogitsLoss)

        All S slices are pushed through the shared backbone;
        features are pooled across slices before the head. Pooling mode is
        controlled by SLICE_POOLING ("mean" or "attention").
        """
        def __init__(self, backbone: nn.Module, feature_dim: int, dropout: float = 0.5,
                     slice_pool: str = "mean", attn_hidden: int = 128):
            super().__init__()
            self.backbone = backbone
            self.slice_pool = str(slice_pool).lower()
            if self.slice_pool == "attention":
                self.pool = GatedAttentionPool(feature_dim, attn_hidden)
            else:
                self.pool = None
            # LayerNorm is stable at small batch sizes, so no incomplete training batch
            # has to be discarded.
            self.head = nn.Sequential(
                nn.LayerNorm(feature_dim),
                nn.Dropout(dropout),
                nn.Linear(feature_dim, 1),
            )

        def forward(self, x: torch.Tensor, slice_mask: torch.Tensor = None) -> torch.Tensor:
            # x: (B, S, 3, H, W). slice_mask: (B, S) True for real slices, False for padding.
            B, S, C, H, W = x.shape
            feats = self.backbone(x.view(B * S, C, H, W))  # (B*S, D)
            # Reduce spatial feature maps (e.g. DenseNet) to vectors.
            if feats.dim() > 2:
                feats = F.adaptive_avg_pool2d(feats, 1).flatten(1)
            feats = feats.view(B, S, -1)
            # Pool over real slices only; padded slots are excluded from both pooling modes.
            if self.pool is not None:
                pooled = self.pool(feats, slice_mask)
            elif slice_mask is None:
                pooled = feats.mean(dim=1)
            else:
                m = slice_mask.to(feats.dtype).unsqueeze(-1)
                pooled = (feats * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
            return self.head(pooled).squeeze(-1)


    def build_classifier(backbone_name: str, dropout: float = DROPOUT, slice_pool: str = None):
        backbone, feat_dim, norm = build_feature_backbone(backbone_name)
        pool_mode = SLICE_POOLING if slice_pool is None else str(slice_pool).lower()
        model = MultiSliceCNNClassifier(
            backbone, feat_dim,
            dropout=dropout,
            slice_pool=pool_mode,
            attn_hidden=ATTENTION_HIDDEN,
        )
        freeze_backbone(model.backbone)
        print(f"Built classifier | backbone={backbone_name} | feature_dim={feat_dim} "
              f"| norm={norm} | slice_pool={pool_mode}")
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable parameters (head + pooling): {n_trainable:,}")
        return model, feat_dim, norm


In [ ]:
# Section 3, cell 15: Training and evaluation utilities
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    def compute_pos_weight(labels: list):
        """Return BCE positive-class weight derived only from the current training fold."""
        if not USE_POS_WEIGHT:
            return None
        arr = np.asarray(labels, dtype=float)
        n_pos = float(arr.sum())
        n_neg = float(len(arr) - n_pos)
        value = (n_neg / max(n_pos, 1.0)) * float(POS_WEIGHT_SCALE)
        return torch.tensor([value], dtype=torch.float32, device=device)


    class ModelEMA:
        """Exponential moving average of model weights, updated once per optimizer
        step. Evaluating/returning the averaged weights biases toward flatter minima
        and usually generalises better. Non-float buffers are copied as-is; the frozen
        backbone BatchNorm and LayerNorm head have no changing running statistics."""
        def __init__(self, model, decay):
            self.decay = float(decay)
            self.shadow = deepcopy(model.state_dict())

        @torch.no_grad()
        def update(self, model):
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
                else:
                    self.shadow[k].copy_(v)

        def copy_to(self, model):
            model.load_state_dict(self.shadow, strict=True)


    class SWATailAverager:
        """
        Averages state_dict tensors across the last N phase-2 checkpoints.
        Float tensors are averaged; non-float buffers keep the most recent value.
        Backbone BatchNorm layers are held in eval mode while the rest of the
        backbone remains in train mode. Running statistics therefore stay fixed;
        the head uses LayerNorm, so no BN re-estimation pass is needed.
        """
        def __init__(self, window: int):
            self.window = int(window)
            self.buffer = []  # list of cpu state_dicts

        @torch.no_grad()
        def add(self, model):
            snapshot = {
                k: (v.detach().to("cpu").clone() if v.dtype.is_floating_point else v.detach().to("cpu").clone())
                for k, v in model.state_dict().items()
            }
            self.buffer.append(snapshot)
            if len(self.buffer) > self.window:
                self.buffer.pop(0)

        def has_any(self) -> bool:
            return len(self.buffer) > 0

        @torch.no_grad()
        def averaged_state_dict(self):
            if not self.buffer:
                raise RuntimeError("SWATailAverager has no snapshots to average.")
            keys = list(self.buffer[0].keys())
            out = {}
            n = float(len(self.buffer))
            for k in keys:
                first = self.buffer[0][k]
                if first.dtype.is_floating_point:
                    acc = torch.zeros_like(first, dtype=torch.float32)
                    for snap in self.buffer:
                        acc.add_(snap[k].to(torch.float32))
                    out[k] = (acc / n).to(first.dtype)
                else:
                    out[k] = self.buffer[-1][k].clone()
            return out


    def pad_slices_collate(batch):
        """Collate patients whose slice counts differ.

        Each patient contributes (S_i, 3, H, W) with S_i = its number of tumour-bearing
        slices. The batch is padded to max(S_i) by repeating that patient's last real
        slice, and a boolean mask marks which slots are real. Padding with a real slice
        rather than zeros keeps the backbone's BatchNorm inputs in-distribution; the mask
        then excludes those slots from pooling, so they cannot affect the patient feature.
        """
        xs_list, ys = zip(*batch)
        s_max = max(int(x.shape[0]) for x in xs_list)
        B = len(xs_list)
        C_, H_, W_ = xs_list[0].shape[1:]
        out = xs_list[0].new_zeros((B, s_max, C_, H_, W_))
        mask = torch.zeros((B, s_max), dtype=torch.bool)
        for i, x in enumerate(xs_list):
            s = int(x.shape[0])
            out[i, :s] = x
            mask[i, :s] = True
            if s < s_max:
                out[i, s:] = x[-1:].expand(s_max - s, -1, -1, -1)
        return out, mask, torch.stack(ys)


    def make_dataloader(
        pairs: list,
        input_size: int,
        augment: bool,
        batch_size: int,
        norm: str = "imagenet",
    ) -> DataLoader:
        ds = SliceDataset(pairs, input_size=input_size, augment=augment, norm=norm)
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=augment,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
            collate_fn=pad_slices_collate,
        )


    def capture_l2sp_anchor(backbone: nn.Module):
        """Copy the currently unfrozen pretrained parameters before fine-tuning starts."""
        return {
            name: param.detach().clone()
            for name, param in backbone.named_parameters()
            if param.requires_grad
        }


    def l2sp_penalty(backbone: nn.Module, anchor: dict):
        """L2-SP sum of squared displacement from the pretrained starting weights.

        The previous mean-of-per-tensor-means could make a very small tensor count as much as a
        multi-million-parameter convolution and made the regularizer nearly invisible.
        """
        penalty = torch.zeros((), device=device)
        found = False
        for name, param in backbone.named_parameters():
            if param.requires_grad and name in anchor:
                penalty = penalty + (param - anchor[name]).pow(2).sum()
                found = True
        return penalty if found else torch.zeros((), device=device)


    @torch.no_grad()
    def backbone_movement_stats(backbone: nn.Module, anchor: dict):
        """Quantify how far the fine-tuned backbone moved from its L2-SP anchor.

        Relative RMS movement is scale-normalised: RMS(delta) / RMS(anchor). It is a
        direct diagnostic of whether phase 2 is actually updating pretrained weights,
        unlike validation AUC on a small inner split where small changes quantise to ties.
        """
        if not anchor:
            return {
                "backbone_rms_delta": float("nan"),
                "backbone_rms_anchor": float("nan"),
                "backbone_rel_rms_move": float("nan"),
                "backbone_max_abs_delta": float("nan"),
                "backbone_anchor_params": 0,
            }

        delta_sq = 0.0
        anchor_sq = 0.0
        max_abs_delta = 0.0
        n_params = 0
        for name, param in backbone.named_parameters():
            if name not in anchor:
                continue
            ref = anchor[name].to(device=param.device, dtype=param.dtype)
            diff = param.detach() - ref
            delta_sq += float(diff.float().pow(2).sum().cpu())
            anchor_sq += float(ref.detach().float().pow(2).sum().cpu())
            if diff.numel():
                max_abs_delta = max(max_abs_delta, float(diff.detach().abs().max().cpu()))
            n_params += int(param.numel())

        if n_params <= 0:
            return {
                "backbone_rms_delta": float("nan"),
                "backbone_rms_anchor": float("nan"),
                "backbone_rel_rms_move": float("nan"),
                "backbone_max_abs_delta": float("nan"),
                "backbone_anchor_params": 0,
            }

        rms_delta = math.sqrt(delta_sq / n_params)
        rms_anchor = math.sqrt(anchor_sq / n_params)
        rel_rms = rms_delta / max(rms_anchor, 1e-12)
        return {
            "backbone_rms_delta": float(rms_delta),
            "backbone_rms_anchor": float(rms_anchor),
            "backbone_rel_rms_move": float(rel_rms),
            "backbone_max_abs_delta": float(max_abs_delta),
            "backbone_anchor_params": int(n_params),
        }


    def _movement_cap_exceeded(stats: dict, epoch: int, context: str = "") -> bool:
        """Phase-2 safety stop: has the backbone drifted past FINETUNE_MAX_REL_RMS_MOVE?

        Returns False whenever USE_MOVEMENT_CAP is off or the movement statistic is not
        finite, so the cap can never stop a run for a missing diagnostic. Relative RMS
        movement is cumulative from the phase-2 starting weights (the L2-SP anchor).
        """
        if not USE_MOVEMENT_CAP:
            return False
        rel = float(stats.get("backbone_rel_rms_move", float("nan")))
        if not np.isfinite(rel) or rel <= float(FINETUNE_MAX_REL_RMS_MOVE):
            return False
        where = f" [{context}]" if context else ""
        print(
            f"    [movement cap]{where} stopping phase 2 at epoch {epoch}: backbone moved "
            f"{100.0 * rel:.3f}% relative RMS, above the {100.0 * float(FINETUNE_MAX_REL_RMS_MOVE):.1f}% cap. "
            "The checkpoint from this epoch remains selectable."
        )
        return True


    def print_finetune_diagnostic(stats: dict, context: str):
        """Print non-binding movement/gradient diagnostics for phase 2.

        L2-SP/BCE is displayed only to show regularisation scale; it is deliberately
        not thresholded because its magnitude depends directly on L2SP_LAMBDA.
        """
        frac = float(stats.get("l2sp_fraction_of_bce", float("nan")))
        rel = float(stats.get("backbone_rel_rms_move", float("nan")))
        grad_bb = float(stats.get("grad_backbone_norm_mean", float("nan")))
        grad_head = float(stats.get("grad_head_norm_mean", float("nan")))
        grad_total = float(stats.get("grad_total_norm_mean", float("nan")))
        clip_frac = float(stats.get("grad_clip_fraction", float("nan")))
        bits = []
        if np.isfinite(rel):
            bits.append(f"relative RMS backbone move={100.0 * rel:.3f}%")
        if np.isfinite(frac):
            bits.append(f"L2-SP contribution={100.0 * frac:.4f}% BCE (scale only)")
        if np.isfinite(grad_bb):
            bits.append(f"grad_bb={grad_bb:.3g}")
        if np.isfinite(grad_head):
            bits.append(f"grad_head={grad_head:.3g}")
        if np.isfinite(grad_total):
            bits.append(f"grad_total={grad_total:.3g}")
        if np.isfinite(clip_frac):
            bits.append(f"clipped_batches={100.0 * clip_frac:.1f}%")
        if bits:
            print(f"  [fine-tune diagnostic] {context}: " + " | ".join(bits))

        if np.isfinite(rel) and rel < float(FINETUNE_REL_RMS_MOVE_WARN_MIN):
            print(
                f"  [diagnostic warning] Backbone movement is below "
                f"{100.0 * float(FINETUNE_REL_RMS_MOVE_WARN_MIN):.2f}% relative RMS. "
                "Phase 2 may still be too conservative; this warning does not alter selection."
            )

        if np.isfinite(clip_frac) and clip_frac > float(FINETUNE_CLIP_FRACTION_WARN_MAX):
            print(
                f"  [diagnostic warning] Gradient clipping bound on "
                f"{100.0 * clip_frac:.1f}% of phase-2 batches at GRAD_CLIP_NORM="
                f"{float(GRAD_CLIP_NORM):g}. Above "
                f"{100.0 * float(FINETUNE_CLIP_FRACTION_WARN_MAX):.0f}% the clip, not the learning "
                "rate, is setting the step size. Raise GRAD_CLIP_NORM; this warning does not "
                "alter selection."
            )


    def _mixup_batch(xs: torch.Tensor, slice_mask: torch.Tensor, ys: torch.Tensor, alpha: float):
        """Patient-level Mixup. xs: (B, S, C, H, W); slice_mask: (B, S); ys: (B,).

        Returns the mixed images, the slice mask valid for the mixed batch and the mixed
        soft target. One lambda per batch (standard Mixup)."""
        if alpha <= 0.0 or xs.size(0) < 2:
            return xs, slice_mask, ys
        lam = float(np.random.beta(alpha, alpha))
        lam = max(lam, 1.0 - lam)  # bias toward the primary sample (common practice)
        perm = torch.randperm(xs.size(0), device=xs.device)
        xs_mixed = lam * xs + (1.0 - lam) * xs[perm]
        ys_mixed = lam * ys + (1.0 - lam) * ys[perm]
        # A mixed slot is usable only where BOTH source patients supplied a real slice;
        # elsewhere the slot is part image, part padding. If two patients share no valid
        # slot, keep the dominant patient's own mask (lam >= 0.5 by construction).
        mask_mixed = slice_mask & slice_mask[perm]
        empty = ~mask_mixed.any(dim=1)
        if bool(empty.any()):
            mask_mixed = torch.where(empty.unsqueeze(1), slice_mask, mask_mixed)
        return xs_mixed, mask_mixed, ys_mixed


    def _set_batchnorm_eval(module: nn.Module):
        """Compatibility alias: fully freeze BatchNorm stats and affine parameters."""
        _freeze_batchnorm_affine_and_stats(module)


    def _grad_l2_norm(parameters):
        """L2 norm of currently populated gradients, measured BEFORE clipping."""
        total_sq = 0.0
        found = False
        for p in parameters:
            if p.grad is None:
                continue
            g = p.grad.detach()
            total_sq += float(torch.sum(g * g).detach().cpu())
            found = True
        return float(np.sqrt(total_sq)) if found else 0.0


    def train_one_epoch(
        model,
        loader,
        optimiser,
        criterion,
        l2sp_anchor=None,
        l2sp_lambda: float = 0.0,
        ema=None,
        use_mixup: bool = False,
        mixup_alpha: float = 0.0,
    ):
        model.train()
        backbone_is_trainable = any(p.requires_grad for p in model.backbone.parameters())
        if backbone_is_trainable:
            # While fine-tuning, non-BN regularisers such as ResNet10t DropPath stay active
            # while BatchNorm statistics and affine parameters are frozen.
            model.backbone.train()
            _set_batchnorm_eval(model.backbone)
        else:
            # Head-only uses a deterministic feature extractor, so DropPath is also disabled.
            model.backbone.eval()

        total_objective = 0.0
        total_bce = 0.0
        total_l2sp = 0.0
        total_l2sp_contribution = 0.0
        all_logits, all_labels = [], []
        n_seen = 0

        # Parameter groups captured once; gradient norms are measured after backward() and
        # before clipping, to show whether clipping is suppressing adaptation.
        _bb_grad_params = [p for p in model.backbone.parameters() if p.requires_grad]
        _head_grad_params = [p for p in model.head.parameters() if p.requires_grad]
        if getattr(model, "pool", None) is not None:
            _head_grad_params += [p for p in model.pool.parameters() if p.requires_grad]
        grad_bb_sum = 0.0
        grad_head_sum = 0.0
        grad_total_sum = 0.0
        grad_total_max = 0.0
        grad_clip_count = 0
        grad_batches = 0

        for xs, slice_mask, ys in loader:
            xs = xs.to(device, non_blocking=True)
            slice_mask = slice_mask.to(device, non_blocking=True)
            ys = ys.to(device, non_blocking=True)
            batch_had_mixup = bool(use_mixup and float(mixup_alpha) > 0.0 and xs.size(0) >= 2)

            if batch_had_mixup:
                xs, slice_mask, ys_target = _mixup_batch(xs, slice_mask, ys, float(mixup_alpha))
            else:
                ys_target = ys

            optimiser.zero_grad(set_to_none=True)
            logits = model(xs, slice_mask)

            target = ys_target
            if float(LABEL_SMOOTHING) > 0.0:
                target = ys_target * (1.0 - float(LABEL_SMOOTHING)) + 0.5 * float(LABEL_SMOOTHING)
            bce = criterion(logits, target)

            reg = torch.zeros((), device=device)
            if l2sp_anchor and float(l2sp_lambda) > 0:
                reg = l2sp_penalty(model.backbone, l2sp_anchor)

            objective = bce + float(l2sp_lambda) * reg
            objective.backward()

            grad_bb = _grad_l2_norm(_bb_grad_params)
            grad_head = _grad_l2_norm(_head_grad_params)
            _clip_norm = float(GRAD_CLIP_NORM)
            raw_total_grad = float(nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=_clip_norm
            ))
            grad_bb_sum += grad_bb
            grad_head_sum += grad_head
            grad_total_sum += raw_total_grad
            grad_total_max = max(grad_total_max, raw_total_grad)
            grad_batches += 1
            if np.isfinite(raw_total_grad) and raw_total_grad > _clip_norm + 1e-12:
                grad_clip_count += 1

            optimiser.step()
            if ema is not None:
                ema.update(model)

            n = len(ys)
            n_seen += n
            total_objective += float(objective.detach().cpu()) * n
            total_bce += float(bce.detach().cpu()) * n
            total_l2sp += float(reg.detach().cpu()) * n
            total_l2sp_contribution += float((float(l2sp_lambda) * reg).detach().cpu()) * n

            # Mixed images have no single hard label, so this diagnostic runs on unmixed data.
            if not batch_had_mixup:
                all_logits.extend(logits.detach().cpu().tolist())
                all_labels.extend(ys.detach().cpu().tolist())

        if len(set(all_labels)) >= 2:
            try:
                auc = roc_auc_score(all_labels, all_logits)
            except Exception:
                auc = float("nan")
        else:
            auc = float("nan")

        denominator = max(n_seen, 1)
        _mean_bce = total_bce / denominator
        _mean_l2sp_contribution = total_l2sp_contribution / denominator
        _movement = backbone_movement_stats(model.backbone, l2sp_anchor) if l2sp_anchor else {
            "backbone_rms_delta": float("nan"),
            "backbone_rms_anchor": float("nan"),
            "backbone_rel_rms_move": float("nan"),
            "backbone_max_abs_delta": float("nan"),
            "backbone_anchor_params": 0,
        }
        return {
            "objective": total_objective / denominator,
            "bce": _mean_bce,
            "l2sp": total_l2sp / denominator,
            "l2sp_contribution": _mean_l2sp_contribution,
            "l2sp_fraction_of_bce": (_mean_l2sp_contribution / max(abs(_mean_bce), 1e-12)),
            "auc": float(auc),
            "grad_backbone_norm_mean": grad_bb_sum / max(grad_batches, 1),
            "grad_head_norm_mean": grad_head_sum / max(grad_batches, 1),
            "grad_total_norm_mean": grad_total_sum / max(grad_batches, 1),
            "grad_total_norm_max": float(grad_total_max),
            "grad_clip_fraction": grad_clip_count / max(grad_batches, 1),
            "grad_clip_count": int(grad_clip_count),
            "grad_batches": int(grad_batches),
            **_movement,
        }


    def make_finetune_scheduler(optimizer, total_epochs: int, stage2_start_epoch: int | None = None):
        """LR warm-up + cosine decay. Groups named ``backbone_deeper`` get a
        local warm-up starting at ``stage2_start_epoch`` so a newly unfrozen
        tail does not skip warm-up when it is added mid-training."""
        total_epochs = max(1, int(total_epochs))
        warmup_epochs = min(max(0, int(FINETUNE_WARMUP_EPOCHS)), max(0, total_epochs - 1))
        minimum = float(FINETUNE_MIN_LR_FACTOR)

        def _factor(local_epoch: int):
            local_epoch = max(0, int(local_epoch))
            if warmup_epochs > 0 and local_epoch < warmup_epochs:
                # Three-step warm-up: make early fine-tuning cautious but meaningful.
                # With FINETUNE_WARMUP_EPOCHS=3:
                #   epoch 1 -> 20%, epoch 2 -> 60%, epoch 3 -> 100% of nominal LR.
                # Epoch 1 remains selectable.
                warmup_factors_3 = (0.20, 0.60, 1.00)
                if warmup_epochs == 3:
                    return warmup_factors_3[local_epoch]
                # Safe fallback if warm-up length is changed later.
                if warmup_epochs == 1:
                    return 1.0
                return 0.20 + 0.80 * (local_epoch / float(warmup_epochs - 1))
            cosine_epochs = max(1, total_epochs - warmup_epochs)
            cosine_index = max(0, local_epoch - warmup_epochs)
            progress = min(1.0, cosine_index / float(max(1, cosine_epochs - 1)))
            cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
            return minimum + (1.0 - minimum) * cosine

        def _group_lambda(offset: int):
            return lambda epoch_index: _factor(int(epoch_index) - int(offset))

        lambdas = []
        for group in optimizer.param_groups:
            if stage2_start_epoch is not None and group.get("group_name") == "backbone_deeper":
                lambdas.append(_group_lambda(int(stage2_start_epoch)))
            else:
                lambdas.append(_group_lambda(0))

        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambdas)


    def _finite_auc(value):
        return np.isfinite(value)


    def _validation_logloss(labels, probs):
        """Continuous secondary selection metric for AUC ties on small validation sets."""
        labels = np.asarray(labels, dtype=int).ravel()
        probs = np.asarray(probs, dtype=float).ravel()
        if labels.size == 0 or labels.size != probs.size:
            return float("inf")
        probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
        try:
            return float(log_loss(labels, probs, labels=[0, 1]))
        except Exception:
            return float("inf")


    def _auc_logloss_better(new_auc, new_logloss, best_auc, best_logloss):
        """Primary: higher AUC. Secondary on AUC ties: lower validation log-loss."""
        if not _finite_auc(new_auc):
            return False
        if not _finite_auc(best_auc):
            return True
        if float(new_auc) > float(best_auc) + float(MIN_AUC_IMPROVEMENT):
            return True
        if abs(float(new_auc) - float(best_auc)) <= float(MIN_AUC_IMPROVEMENT):
            return float(new_logloss) < float(best_logloss) - float(VAL_LOGLOSS_TIE_TOL)
        return False


    # AUC confidence intervals: calculate_auc_intervals() and its settings
    # AUC_CI_CONFIDENCE and AUC_BOOTSTRAP_REPEATS are defined once in Section 0, cell 3
    # and shared by all three sections. Only the paired fine-tune-versus-head bootstrap
    # below is specific to this section.
    def paired_bootstrap_auc_delta(
        y_true,
        head_score,
        finetune_score,
        confidence: float = AUC_CI_CONFIDENCE,
        n_bootstrap: int = AUC_BOOTSTRAP_REPEATS,
        random_state: int = GLOBAL_SEED + 137,
    ):
        """Paired bootstrap for AUC(fine-tune) - AUC(head-only), using the shared fast engine.

        This preserves the older stratified patient-level resampling order but replaces
        the former 2 * n_bootstrap calls to sklearn.roc_auc_score with vectorised,
        tie-aware Mann-Whitney AUC scoring.
        """
        _fast = optimization_paired_bootstrap_auc_delta(
            y_true=y_true,
            reference_score=head_score,
            candidate_score=finetune_score,
            confidence=confidence,
            n_bootstrap=n_bootstrap,
            random_state=random_state,
        )
        return {
            "head_auc": float(_fast["reference_auc"]),
            "finetune_auc": float(_fast["candidate_auc"]),
            "delta": float(_fast["delta"]),
            "ci_lower": float(_fast["ci_lower"]),
            "ci_upper": float(_fast["ci_upper"]),
            "n_bootstrap": int(_fast["n_bootstrap"]),
            "p_finetune_gt_head": float(_fast["p_candidate_gt_reference"]),
        }


    def optimal_threshold_youden(labels, probs):
        labels = np.asarray(labels, dtype=int).ravel()
        probs = np.asarray(probs, dtype=float).ravel()
        valid = np.isfinite(labels) & np.isfinite(probs)
        labels = labels[valid]
        probs = probs[valid]

        if labels.size == 0 or np.unique(labels).size != 2:
            return float(DECISION_THRESHOLD)

        scores = np.unique(probs)
        if scores.size > 1001:
            scores = np.unique(np.quantile(probs, np.linspace(0.0, 1.0, 1001)))

        if scores.size > 1:
            thresholds = (scores[:-1] + scores[1:]) / 2.0
            thresholds = np.r_[scores[0] - 1e-12, thresholds, scores[-1] + 1e-12]
        else:
            thresholds = np.array([scores[0]], dtype=float)

        best_t = float(thresholds[0])
        best_j = -np.inf
        for threshold in thresholds:
            preds = (probs >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
            specificity = tn / (tn + fp) if (tn + fp) else 0.0
            youden = sensitivity + specificity - 1.0
            if youden > best_j:
                best_j = youden
                best_t = float(threshold)

        return best_t


    def compute_binary_metrics(
        labels,
        probs,
        threshold: float = DECISION_THRESHOLD,
        threshold_source: str = "explicit classification threshold",
    ):
        labels = np.array(labels, dtype=int)
        probs = np.array(probs, dtype=float)
        threshold = float(threshold)
        preds = (probs >= threshold).astype(int)

        out = {
            "threshold": float(threshold),
            "threshold_source": str(threshold_source),
        }
        try:
            out["auc"] = roc_auc_score(labels, probs)
        except Exception:
            out["auc"] = float("nan")

        try:
            out["accuracy"] = accuracy_score(labels, preds)
            out["balacc"] = balanced_accuracy_score(labels, preds)
            out["f1"] = f1_score(labels, preds, zero_division=0)
            out["precision"] = precision_score(labels, preds, zero_division=0)
            out["sensitivity"] = recall_score(labels, preds, zero_division=0)
            out["mcc"] = matthews_corrcoef(labels, preds)
            out["brier"] = brier_score_loss(labels, probs)

            cm = confusion_matrix(labels, preds, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            out["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
            out["tn"], out["fp"], out["fn"], out["tp"] = int(tn), int(fp), int(fn), int(tp)
        except Exception:
            for k in [
                "accuracy", "balacc", "f1", "precision", "sensitivity",
                "specificity", "mcc", "brier",
            ]:
                out[k] = float("nan")
            out["tn"] = out["fp"] = out["fn"] = out["tp"] = 0

        return out


    def compute_top_risk_binary_metrics(labels, probs, percentage=TOP_RISK_PERCENT):
        threshold = threshold_for_top_risk(probs, percentage=percentage)
        return compute_binary_metrics(
            labels,
            probs,
            threshold=threshold,
            threshold_source=f"top {float(percentage):g}% risk scores in evaluated set",
        )


    def _rotate_batch(xs: torch.Tensor, degrees: float) -> torch.Tensor:
        """Deterministic rigid rotation by `degrees` (no scaling/translation).
        xs is expected shape (B, S, C, H, W) OR (N, C, H, W); works via affine_grid
        on the last four dims."""
        orig_shape = xs.shape
        if xs.dim() == 5:
            B, S, C, H, W = orig_shape
            xs_flat = xs.view(B * S, C, H, W)
        else:
            xs_flat = xs
        N, C, H, W = xs_flat.shape
        ang = float(np.radians(float(degrees)))
        cos, sin = float(np.cos(ang)), float(np.sin(ang))
        theta = torch.tensor([[cos, -sin, 0.0], [sin, cos, 0.0]],
                             dtype=xs_flat.dtype, device=xs_flat.device).unsqueeze(0).repeat(N, 1, 1)
        grid = F.affine_grid(theta, size=(N, C, H, W), align_corners=False)
        out = F.grid_sample(xs_flat, grid, mode="bilinear",
                            padding_mode="zeros", align_corners=False)
        return out.view(orig_shape)


    def _rotate_batch_before_normalization(xs: torch.Tensor, degrees: float, norm_mode: str) -> torch.Tensor:
        """Rotate on the native ~[0,1] intensity scale, then restore normalization.

        With ImageNet normalization, rotating the already-normalized tensor would make
        grid_sample's zero padding correspond to the ImageNet mean rather than true zero
        background. Flips are unchanged because normalization commutes with flipping.
        """
        norm_mode = str(norm_mode).lower()
        if norm_mode == "unit_interval":
            return _rotate_batch(xs, degrees)
        if xs.dim() != 5:
            raise ValueError("Expected TTA batch shape (B,S,C,H,W).")
        if norm_mode == "minus_one_one":
            raw = (xs + 1.0) * 0.5
            rotated_raw = _rotate_batch(raw, degrees)
            return rotated_raw.mul(2.0).sub(1.0)
        if norm_mode == "caffe_over_255":
            mean = torch.tensor(
                RADIMAGENET_CAFFE_MEAN_BGR, dtype=xs.dtype, device=xs.device
            ).view(1, 1, 3, 1, 1)
            raw = xs + mean
            rotated_raw = _rotate_batch(raw, degrees)
            return rotated_raw - mean
        if norm_mode == "imagenet":
            mean = torch.tensor(IMAGENET_MEAN, dtype=xs.dtype, device=xs.device).view(1, 1, 3, 1, 1)
            std = torch.tensor(IMAGENET_STD, dtype=xs.dtype, device=xs.device).view(1, 1, 3, 1, 1)
            raw = xs * std + mean
            rotated_raw = _rotate_batch(raw, degrees)
            return (rotated_raw - mean) / std
        raise ValueError(f"Unknown normalization mode for TTA rotation: {norm_mode!r}")


    @torch.no_grad()
    def evaluate(model, loader, threshold: float = DECISION_THRESHOLD, tta=None):
        """Score a loader. With TTA on, probabilities are averaged over deterministic
        flips (identity, LR, UD, LR+UD) and, when USE_TTA_ROTATIONS is on, over the
        rotation angles listed in TTA_ROTATION_DEGREES. TTA off = single pass."""
        use_tta = USE_TTA if tta is None else bool(tta)
        model.eval()
        all_probs, all_labels = [], []
        norm_mode = str(getattr(loader.dataset, "norm", "unit_interval")).lower()
        for xs, slice_mask, ys in loader:
            xs = xs.to(device)
            slice_mask = slice_mask.to(device)
            variants = [xs]
            if use_tta:
                variants += [torch.flip(xs, dims=[-1]),
                             torch.flip(xs, dims=[-2]),
                             torch.flip(xs, dims=[-1, -2])]
                if USE_TTA_ROTATIONS:
                    for deg in TTA_ROTATION_DEGREES:
                        variants.append(_rotate_batch_before_normalization(xs, float(deg), norm_mode))
            probs_b = torch.stack([torch.sigmoid(model(v, slice_mask)) for v in variants], dim=0).mean(dim=0)
            all_probs.extend(probs_b.cpu().tolist())
            all_labels.extend(ys.tolist())
        probs = np.asarray(all_probs, dtype=float)
        labels = np.array(all_labels, dtype=int)
        metrics = compute_binary_metrics(
            labels,
            probs,
            threshold=threshold,
            threshold_source=f"fixed CNN threshold {float(threshold):g}",
        )
        metrics.update({"probs": probs, "labels": labels})
        return metrics


    @torch.no_grad()
    def evaluate_return_logits(model, loader):
        """Same forward pass as evaluate() (with TTA if enabled) but returns the
        per-patient MEAN LOGIT alongside the per-patient MEAN PROB for diagnostics
        and threshold calculations."""
        use_tta = bool(USE_TTA)
        model.eval()
        all_probs, all_logits, all_labels = [], [], []
        norm_mode = str(getattr(loader.dataset, "norm", "unit_interval")).lower()
        for xs, slice_mask, ys in loader:
            xs = xs.to(device)
            slice_mask = slice_mask.to(device)
            variants = [xs]
            if use_tta:
                variants += [torch.flip(xs, dims=[-1]),
                             torch.flip(xs, dims=[-2]),
                             torch.flip(xs, dims=[-1, -2])]
                if USE_TTA_ROTATIONS:
                    for deg in TTA_ROTATION_DEGREES:
                        variants.append(_rotate_batch_before_normalization(xs, float(deg), norm_mode))
            logits_stack = torch.stack([model(v, slice_mask) for v in variants], dim=0)  # (V, B)
            mean_logits = logits_stack.mean(dim=0)                               # (B,)
            mean_probs = torch.sigmoid(logits_stack).mean(dim=0)                 # (B,), matches evaluate()
            all_logits.extend(mean_logits.cpu().tolist())
            all_probs.extend(mean_probs.cpu().tolist())
            all_labels.extend(ys.tolist())
        return {
            "probs": np.asarray(all_probs, dtype=float),
            "logits": np.asarray(all_logits, dtype=float),
            "labels": np.array(all_labels, dtype=int),
        }


    @torch.no_grad()
    def evaluate_clean_training_auc(model, pairs, input_size: int, norm: str):
        """AUC on unaugmented training patients at one fixed checkpoint: model.eval(), no
        augmentation, no TTA, no optimiser updates.

        Distinct from the online training AUC accumulated in train_one_epoch(), and never
        used for model, epoch or protocol selection.
        """
        clean_loader = make_dataloader(
            pairs, input_size, augment=False, batch_size=BATCH_SIZE, norm=norm
        )
        scored = evaluate(model, clean_loader, tta=False)
        return float(scored["auc"])


    def _smoothed_scores(values, window: int):
        """Return the trailing-window mean of a numeric series (arithmetic mean,
        ignoring NaN). window=1 or empty series → returns values unchanged."""
        arr = np.asarray(values, dtype=float)
        if arr.size == 0 or int(window) <= 1:
            return arr
        w = int(window)
        out = np.full_like(arr, np.nan)
        for i in range(arr.size):
            lo = max(0, i - w + 1)
            chunk = arr[lo:i + 1]
            chunk = chunk[np.isfinite(chunk)]
            out[i] = float(np.mean(chunk)) if chunk.size else np.nan
        return out


    def _select_raw_checkpoint_from_window(records, window: int):
        """Choose highest RAW AUC in the window; break AUC ties by lower log-loss."""
        if not records:
            return None
        w = max(1, int(window))
        recent = records[-w:]
        finite = [r for r in recent if _finite_auc(r["raw_auc"])]
        if not finite:
            return recent[-1]
        best = finite[0]
        for rec in finite[1:]:
            if _auc_logloss_better(
                rec["raw_auc"], rec.get("val_logloss", float("inf")),
                best["raw_auc"], best.get("val_logloss", float("inf")),
            ):
                best = rec
        return best


    def train_classifier(
        model: nn.Module,
        train_pairs: list,
        val_pairs: list,
        backbone_name: str,
        input_size: int = CNN_INPUT_SIZE,
        feat_dim: int = 512,
        norm: str = "imagenet",
        backbone_lr: float = None,
        use_staged_unfreeze: bool = False,
        phase1_cache: dict = None,
        phase1_cache_key=None,
    ):
        """Inner-development model selection, run only on the inner validation split so the
        outer fold is never seen.

        Smoothed AUC governs patience and selection stability; the stored checkpoint is the
        raw-best epoch inside the winning smoothing window. Raw AUC and selection score are
        tracked separately.
        """
        if backbone_lr is None or not np.isfinite(float(backbone_lr)) or float(backbone_lr) <= 0.0:
            raise ValueError(f"{backbone_name}: train_classifier requires a positive inner-selected backbone_lr.")
        backbone_lr = float(backbone_lr)

        train_loader = make_dataloader(
            train_pairs, input_size, augment=True, batch_size=BATCH_SIZE, norm=norm
        )
        val_loader = make_dataloader(
            val_pairs, input_size, augment=False, batch_size=BATCH_SIZE, norm=norm
        )
        pos_weight = compute_pos_weight([p["label"] for p in train_pairs])
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        # Phase 1: head-only.
        head_best_selection_score = -np.inf
        head_best_auc = -np.inf
        head_best_logloss = float("inf")
        head_best_epoch = 0
        head_best_state = None
        head_val_auc_history = []
        head_records = []
        patience = 0

        opt_head = torch.optim.AdamW(model.head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY_HEAD)
        if getattr(model, "pool", None) is not None:
            opt_head.add_param_group({
                "params": list(model.pool.parameters()), "lr": LR_HEAD,
                "weight_decay": WEIGHT_DECAY_HEAD,
            })
        sch_head = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt_head, T_max=max(1, EPOCHS_HEAD_MAX), eta_min=LR_HEAD * 0.05
        )

        # Staged unfreezing only takes effect from phase-2 epoch STAGE1_UNFREEZE_EPOCHS + 1,
        # so phase 1 is identical across staged candidates and is cached rather than repeated.
        _p1_cached = (
            phase1_cache.get(phase1_cache_key)
            if (phase1_cache is not None and phase1_cache_key is not None)
            else None
        )
        if _p1_cached is not None:
            print(
                f"  Phase 1: reusing cached head-only result "
                f"(AUC={float(_p1_cached['auc']):.3f} @ ep{int(_p1_cached['epoch'])})"
            )
        else:
            print(f"  Phase 1: head warm-up (maximum {EPOCHS_HEAD_MAX} epochs)")
        for epoch in range(1, (0 if _p1_cached is not None else EPOCHS_HEAD_MAX) + 1):
            train_stats = train_one_epoch(
                model, train_loader, opt_head, criterion,
                use_mixup=USE_MIXUP, mixup_alpha=MIXUP_ALPHA,
            )
            val_metrics = evaluate(model, val_loader, tta=USE_TTA_FOR_EPOCH_SELECTION)
            sch_head.step()
            val_auc = float(val_metrics["auc"])
            val_logloss = _validation_logloss(val_metrics["labels"], val_metrics["probs"])
            head_val_auc_history.append(val_auc)
            head_records.append({
                "epoch": epoch,
                "raw_auc": val_auc,
                "val_logloss": val_logloss,
                "state": deepcopy(model.state_dict()),
            })
            # Keep only the states inside the smoothing window; all of them would cost GBs.
            _head_keep = max(1, int(SMOOTHING_WINDOW) if USE_SMOOTHED_SELECTION else 1)
            if len(head_records) > _head_keep:
                head_records = head_records[-_head_keep:]
            if USE_SMOOTHED_SELECTION and SMOOTHING_WINDOW > 1:
                reference_auc = float(_smoothed_scores(head_val_auc_history, SMOOTHING_WINDOW)[-1])
                candidate = _select_raw_checkpoint_from_window(head_records, SMOOTHING_WINDOW)
            else:
                reference_auc = val_auc
                candidate = head_records[-1]

            candidate_logloss = float(candidate.get("val_logloss", float("inf")))
            improved = _auc_logloss_better(
                reference_auc, candidate_logloss,
                head_best_selection_score, head_best_logloss,
            )
            if improved:
                head_best_selection_score = reference_auc
                head_best_auc = float(candidate["raw_auc"])
                head_best_logloss = candidate_logloss
                head_best_epoch = int(candidate["epoch"])
                head_best_state = deepcopy(candidate["state"])
                patience = 0
            else:
                patience += 1

            print(
                f"    ep{epoch:02d}/{EPOCHS_HEAD_MAX} | bce={train_stats['bce']:.4f} "
                f"tr_auc_aug_online={train_stats['auc']:.3f} | val_auc={val_auc:.3f}"
                + (f" (smooth={reference_auc:.3f})" if USE_SMOOTHED_SELECTION and SMOOTHING_WINDOW > 1 else "")
                + f" val_logloss={val_logloss:.4f} val_balacc@0.5={val_metrics['balacc']:.3f}"
            )
            if patience >= EARLY_STOP_PATIENCE_HEAD:
                print(f"    Early stop (head phase) at epoch {epoch}")
                break

        if _p1_cached is not None:
            head_best_state = deepcopy(_p1_cached["state"])
            head_best_epoch = int(_p1_cached["epoch"])
            head_best_auc = float(_p1_cached["auc"])
            head_best_selection_score = float(_p1_cached["selection_score"])
            head_best_logloss = float(_p1_cached.get("logloss", float("inf")))
        elif head_best_state is None:
            candidate = head_records[0]
            for rec in head_records[1:]:
                if _auc_logloss_better(
                    rec["raw_auc"], rec.get("val_logloss", float("inf")),
                    candidate["raw_auc"], candidate.get("val_logloss", float("inf")),
                ):
                    candidate = rec
            head_best_state = deepcopy(candidate["state"])
            head_best_epoch = int(candidate["epoch"])
            head_best_auc = float(candidate["raw_auc"])
            head_best_logloss = float(candidate.get("val_logloss", float("inf")))
            head_best_selection_score = head_best_auc
        model.load_state_dict(head_best_state)

        # The threshold is fitted on inner validation predictions from the selected head-only
        # checkpoint and frozen before the outer fold is scored. TTA stays on so threshold and
        # scores share one scale.
        if _p1_cached is not None:
            head_youden_threshold = float(_p1_cached["youden_threshold"])
        else:
            head_threshold_scored = evaluate_return_logits(model, val_loader)
            head_youden_threshold = optimal_threshold_youden(
                head_threshold_scored["labels"], head_threshold_scored["probs"]
            )
            if phase1_cache is not None and phase1_cache_key is not None:
                phase1_cache[phase1_cache_key] = {
                    "state": deepcopy(head_best_state),
                    "epoch": int(head_best_epoch),
                    "auc": float(head_best_auc),
                    "selection_score": float(head_best_selection_score),
                    "logloss": float(head_best_logloss),
                    "youden_threshold": float(head_youden_threshold),
                }

        # Phase 2 starts from an RNG state independent of whether phase 1 was cached and of
        # use_staged_unfreeze, so staged and unstaged candidates share one random stream up to
        # the stage-2 unfreeze and differ only in that contrast.
        if phase1_cache_key is not None:
            _p2_seed = int(
                hashlib.sha256(f"phase2|{phase1_cache_key}".encode("utf-8")).hexdigest()[:8], 16
            )
            seed_everything(_p2_seed)

        # Phase 2: architecture-specific shallow tail, optionally one adjacent tail segment.
        unfreeze_last_block(model.backbone, backbone_name)
        backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        pool_params = list(model.pool.parameters()) if getattr(model, "pool", None) is not None else []
        if not backbone_params:
            raise RuntimeError(f"{backbone_name}: no backbone parameters were unfrozen; fine-tuning cannot run.")

        l2sp_anchor = capture_l2sp_anchor(model.backbone)
        _ft_profile = small_data_finetune_profile(backbone_name)
        _backbone_lr = float(backbone_lr)
        opt_ft = torch.optim.AdamW([
            {"params": backbone_params, "lr": _backbone_lr,
             "weight_decay": WEIGHT_DECAY_BACKBONE, "group_name": "backbone_last"},
            {"params": head_params + pool_params, "lr": LR_FINETUNE_HEAD,
             "weight_decay": WEIGHT_DECAY_HEAD, "group_name": "head_and_pool"},
        ])
        sch_ft = make_finetune_scheduler(opt_ft, EPOCHS_FINETUNE_MAX)
        ema = ModelEMA(model, EMA_DECAY) if USE_EMA else None
        swa = SWATailAverager(SWA_WINDOW) if USE_SWA else None

        source_best = {
            "raw": {"selection_score": -np.inf, "auc": -np.inf, "logloss": float("inf"), "epoch": 0, "state": None, "diag": None, "history": [], "records": []},
        }
        min_selection_epoch = min(max(1, int(FINETUNE_MIN_SELECTION_EPOCH)), int(EPOCHS_FINETUNE_MAX))
        if ema is not None:
            source_best["ema"] = {"selection_score": -np.inf, "auc": -np.inf, "logloss": float("inf"), "epoch": 0, "state": None, "diag": None, "history": [], "records": []}
        patience = 0
        global_best_selection = -np.inf
        staged_done = False
        phase2_train_diagnostics = []

        n_backbone_trainable = sum(p.numel() for p in backbone_params)
        n_backbone_total = sum(p.numel() for p in model.backbone.parameters())
        n_backbone_pct = 100.0 * n_backbone_trainable / max(1, n_backbone_total)
        n_head_trainable = sum(p.numel() for p in head_params + pool_params)
        print(
            f"  Phase 2: architecture-aware tail fine-tuning (maximum {EPOCHS_FINETUNE_MAX} epochs)\n"
            f"  Trainable backbone={n_backbone_trainable:,}/{n_backbone_total:,} ({n_backbone_pct:.1f}%) | "
            f"head+pool={n_head_trainable:,} | "
            f"LR backbone={_backbone_lr:g} ({_ft_profile['profile']}) | LR head={LR_FINETUNE_HEAD:g} | "
            f"L2-SP={L2SP_LAMBDA:g} | staged_unfreeze={bool(use_staged_unfreeze)} | "
            f"mixup={USE_MIXUP} | ema={USE_EMA} | swa={USE_SWA}"
        )

        for epoch in range(1, EPOCHS_FINETUNE_MAX + 1):
            if (bool(use_staged_unfreeze) and not staged_done
                    and epoch == max(1, int(STAGE1_UNFREEZE_EPOCHS)) + 1):
                newly = unfreeze_deeper_block(model.backbone, backbone_name)
                if newly:
                    deeper_lr = float(_backbone_lr) * float(STAGE2_LR_FACTOR)
                    opt_ft.add_param_group({
                        "params": newly, "lr": deeper_lr,
                        "weight_decay": WEIGHT_DECAY_BACKBONE, "group_name": "backbone_deeper",
                    })
                    for name, p in model.backbone.named_parameters():
                        if p.requires_grad and name not in l2sp_anchor:
                            l2sp_anchor[name] = p.detach().clone()
                    sch_ft = make_finetune_scheduler(opt_ft, EPOCHS_FINETUNE_MAX, stage2_start_epoch=epoch - 1)
                    for _ in range(epoch - 1):
                        sch_ft.step()
                    print(f"    [stage 2 unfreeze @ ep{epoch:02d}] deeper block LR={deeper_lr:g}")
                staged_done = True

            train_stats = train_one_epoch(
                model, train_loader, opt_ft, criterion,
                l2sp_anchor=l2sp_anchor, l2sp_lambda=L2SP_LAMBDA,
                ema=ema, use_mixup=USE_MIXUP, mixup_alpha=MIXUP_ALPHA,
            )
            phase2_train_diagnostics.append(dict(train_stats))

            # Evaluate raw and EMA on the SAME inner validation set. This lets inner
            # selection explicitly decide the final weight strategy.
            candidate_states = {"raw": deepcopy(model.state_dict())}
            if ema is not None:
                candidate_states["ema"] = deepcopy(ema.shadow)
            epoch_aucs = {}
            epoch_logloss = {}
            raw_state_for_restore = candidate_states["raw"]
            for source, state in candidate_states.items():
                model.load_state_dict(state, strict=True)
                _val_metrics = evaluate(model, val_loader, tta=USE_TTA_FOR_EPOCH_SELECTION)
                epoch_aucs[source] = float(_val_metrics["auc"])
                epoch_logloss[source] = _validation_logloss(_val_metrics["labels"], _val_metrics["probs"])
            model.load_state_dict(raw_state_for_restore, strict=True)

            improved_any = False
            for source, val_auc in epoch_aucs.items():
                sb = source_best[source]
                if epoch < min_selection_epoch:
                    continue
                sb["history"].append(val_auc)
                sb["records"].append({
                    "epoch": epoch, "raw_auc": val_auc,
                    "val_logloss": float(epoch_logloss[source]),
                    "state": candidate_states[source],
                    # Diagnostics are bound to this optimiser epoch, so the selected
                    # checkpoint reports its own BCE and gradient statistics.
                    "train_diag": dict(train_stats),
                })
                _source_keep = max(1, int(SMOOTHING_WINDOW) if USE_SMOOTHED_SELECTION else 1)
                if len(sb["records"]) > _source_keep:
                    sb["records"] = sb["records"][-_source_keep:]
                if USE_SMOOTHED_SELECTION and SMOOTHING_WINDOW > 1:
                    reference_auc = float(_smoothed_scores(sb["history"], SMOOTHING_WINDOW)[-1])
                    candidate = _select_raw_checkpoint_from_window(sb["records"], SMOOTHING_WINDOW)
                else:
                    reference_auc = val_auc
                    candidate = sb["records"][-1]
                candidate_logloss = float(candidate.get("val_logloss", float("inf")))
                if _auc_logloss_better(
                    reference_auc, candidate_logloss,
                    sb["selection_score"], sb.get("logloss", float("inf")),
                ):
                    sb["selection_score"] = reference_auc
                    sb["auc"] = float(candidate["raw_auc"])
                    sb["logloss"] = candidate_logloss
                    sb["epoch"] = int(candidate["epoch"])
                    sb["state"] = deepcopy(candidate["state"])
                    sb["diag"] = deepcopy(candidate.get("train_diag"))
                    improved_any = True

            current_global = max(
                [v["selection_score"] for v in source_best.values() if _finite_auc(v["selection_score"])],
                default=-np.inf,
            )
            if current_global > global_best_selection + float(MIN_AUC_IMPROVEMENT):
                global_best_selection = current_global
                patience = 0
            elif improved_any:
                # On ~25 patients AUC is quantised, so a lower log-loss at equal AUC is a
                # real improvement.
                patience = 0
            else:
                patience += 1

            if swa is not None:
                swa.add(model)

            _selection_tag = " [selection locked until warm-up ends]" if epoch < min_selection_epoch else ""
            print(
                f"    ep{epoch:02d}/{EPOCHS_FINETUNE_MAX} | bce={train_stats['bce']:.4f} "
                f"l2sp_sum={train_stats['l2sp']:.4g} l2sp_loss={train_stats['l2sp_contribution']:.4g} "
                f"({100*train_stats['l2sp_fraction_of_bce']:.4f}% BCE; scale only) "
                f"bb_move={100*train_stats['backbone_rel_rms_move']:.3f}% tr_auc_aug_online={train_stats['auc']:.3f} | "
                + " | ".join(
                    f"{src}_val_auc={auc:.3f}/{epoch_logloss[src]:.4f}LL"
                    for src, auc in epoch_aucs.items()
                )
                + f" | grad_bb={train_stats['grad_backbone_norm_mean']:.3g} "
                f"grad_head={train_stats['grad_head_norm_mean']:.3g} "
                f"grad_total={train_stats['grad_total_norm_mean']:.3g} "
                f"clip={100*train_stats['grad_clip_fraction']:.1f}%"
                + f" | lr_bb={opt_ft.param_groups[0]['lr']:.2e} lr_head={opt_ft.param_groups[1]['lr']:.2e}"
                + _selection_tag
            )
            sch_ft.step()

            # Backbone movement cap (switch 6). The epoch that crossed the cap has already
            # been recorded above and stays eligible for selection; only further training is
            # stopped. Movement is measured against the L2-SP anchor, i.e. cumulatively from
            # the start of phase 2, so this bounds total drift rather than a per-epoch step.
            if _movement_cap_exceeded(train_stats, epoch, context="inner selection"):
                break

            if patience >= EARLY_STOP_PATIENCE_FINETUNE:
                print(f"    Early stop (fine-tuning phase) at epoch {epoch}")
                break

        # Optional SWA candidate; compare by raw validation AUC against other raw AUCs.
        swa_val_auc = float("nan")
        if swa is not None and swa.has_any():
            swa_state = swa.averaged_state_dict()
            pre = deepcopy(model.state_dict())
            model.load_state_dict(swa_state, strict=True)
            _swa_metrics = evaluate(model, val_loader, tta=USE_TTA_FOR_EPOCH_SELECTION)
            swa_val_auc = float(_swa_metrics["auc"])
            swa_val_logloss = _validation_logloss(_swa_metrics["labels"], _swa_metrics["probs"])
            model.load_state_dict(pre, strict=True)
            source_best["swa"] = {
                "selection_score": swa_val_auc, "auc": swa_val_auc,
                "logloss": float(swa_val_logloss),
                "epoch": int(epoch), "state": swa_state,
                "diag": deepcopy(phase2_train_diagnostics[-1]) if phase2_train_diagnostics else None,
            }

        # Explicit strategy winner on inner validation, using comparable raw AUCs.
        available = {k: v for k, v in source_best.items() if v.get("state") is not None and _finite_auc(v.get("auc", np.nan))}
        if not available:
            raise RuntimeError(
                f"No valid fine-tuning checkpoint was selected on the inner validation split. "
                f"Checkpoint selection starts at epoch {min_selection_epoch}; ensure EPOCHS_FINETUNE_MAX >= that value."
            )
        finetune_best_source, best = next(iter(available.items()))
        for _source, _candidate in list(available.items())[1:]:
            if _auc_logloss_better(
                _candidate["auc"], _candidate.get("logloss", float("inf")),
                best["auc"], best.get("logloss", float("inf")),
            ):
                finetune_best_source, best = _source, _candidate
        finetune_best_auc = float(best["auc"])
        finetune_best_logloss = float(best.get("logloss", float("inf")))
        finetune_best_epoch = int(best["epoch"])
        model.load_state_dict(best["state"], strict=True)

        # All diagnostics describe the selected state: BCE and gradient norms come from the
        # epoch that produced it, while movement and L2-SP are recomputed after loading the
        # selected raw/EMA/SWA weights, so EMA cannot inherit raw-state values.
        _best_diag_for_print = dict(best.get("diag") or {})
        _best_movement = backbone_movement_stats(model.backbone, l2sp_anchor)
        _selected_l2sp = float(l2sp_penalty(model.backbone, l2sp_anchor).detach().cpu())
        _selected_l2sp_contrib = float(L2SP_LAMBDA) * _selected_l2sp
        _selected_bce = float(_best_diag_for_print.get("bce", float("nan")))
        _best_diag_for_print.update(_best_movement)
        _best_diag_for_print["l2sp"] = _selected_l2sp
        _best_diag_for_print["l2sp_contribution"] = _selected_l2sp_contrib
        _best_diag_for_print["l2sp_fraction_of_bce"] = (
            _selected_l2sp_contrib / max(abs(_selected_bce), 1e-12)
            if np.isfinite(_selected_bce) else float("nan")
        )
        print_finetune_diagnostic(
            _best_diag_for_print,
            f"inner-selected {finetune_best_source} checkpoint @ ft-ep{finetune_best_epoch}",
        )

        # Fit the phase-2 threshold on the INNER validation predictions from the
        # selected raw/EMA/SWA checkpoint. It is used only if phase 2 wins.
        finetune_threshold_scored = evaluate_return_logits(model, val_loader)
        finetune_youden_threshold = optimal_threshold_youden(
            finetune_threshold_scored["labels"], finetune_threshold_scored["probs"]
        )

        # Diagnostics from the last phase-2 epoch, exported separately from the
        # selected-checkpoint diagnostics above.
        _last_phase2_diag = dict(phase2_train_diagnostics[-1]) if phase2_train_diagnostics else {}

        info = {
            "head_best_auc": float(head_best_auc),
            "head_best_selection_score": float(head_best_selection_score),
            "head_best_logloss": float(head_best_logloss),
            "head_best_epoch": int(head_best_epoch),
            "head_youden_threshold": float(head_youden_threshold),
            "finetune_best_auc": finetune_best_auc,
            "finetune_best_selection_score": float(best.get("selection_score", finetune_best_auc)),
            "finetune_best_logloss": float(finetune_best_logloss),
            "finetune_best_epoch": finetune_best_epoch,
            "finetune_youden_threshold": float(finetune_youden_threshold),
            "finetune_best_source": str(finetune_best_source),
            "finetune_delta_vs_head": float(finetune_best_auc - head_best_auc),
            "raw_best_auc": float(source_best["raw"]["auc"]),
            "raw_best_epoch": int(source_best["raw"]["epoch"]),
            "ema_best_auc": float(source_best.get("ema", {}).get("auc", np.nan)),
            "ema_best_epoch": int(source_best.get("ema", {}).get("epoch", 0)),
            "swa_val_auc": float(swa_val_auc),
            "finetune_best_backbone_rms_delta": float(_best_movement["backbone_rms_delta"]),
            "finetune_best_backbone_rel_rms_move": float(_best_movement["backbone_rel_rms_move"]),
            "finetune_best_backbone_max_abs_delta": float(_best_movement["backbone_max_abs_delta"]),
            "finetune_anchor_params": int(_best_movement["backbone_anchor_params"]),
            "finetune_last_l2sp_fraction_of_bce": float(_last_phase2_diag.get("l2sp_fraction_of_bce", np.nan)),
            "finetune_max_l2sp_fraction_of_bce": float(max(
                [d.get("l2sp_fraction_of_bce", float("nan")) for d in phase2_train_diagnostics]
                or [float("nan")]
            )),
            "finetune_mean_grad_backbone_norm": float(np.nanmean([
                d.get("grad_backbone_norm_mean", np.nan) for d in phase2_train_diagnostics
            ])) if phase2_train_diagnostics else float("nan"),
            "finetune_mean_grad_head_norm": float(np.nanmean([
                d.get("grad_head_norm_mean", np.nan) for d in phase2_train_diagnostics
            ])) if phase2_train_diagnostics else float("nan"),
            "finetune_mean_grad_total_norm": float(np.nanmean([
                d.get("grad_total_norm_mean", np.nan) for d in phase2_train_diagnostics
            ])) if phase2_train_diagnostics else float("nan"),
            "finetune_mean_grad_clip_fraction": float(np.nanmean([
                d.get("grad_clip_fraction", np.nan) for d in phase2_train_diagnostics
            ])) if phase2_train_diagnostics else float("nan"),
            "n_train_fold": int(len(train_pairs)),
            "backbone_lr": float(_backbone_lr),
            "use_staged_unfreeze": bool(use_staged_unfreeze),
            "slice_pool": str(getattr(model, "slice_pool", "mean")),
        }
        print(
            f"  Inner selection: head AUC={info['head_best_auc']:.3f} @ ep{info['head_best_epoch']} | "
            f"fine-tuned AUC={info['finetune_best_auc']:.3f} @ ft-ep{info['finetune_best_epoch']} "
            f"({info['finetune_best_source']}) | backbone LR={info['backbone_lr']:.2e} | "
            f"delta={info['finetune_delta_vs_head']:+.3f} | "
            f"Youden(head={info['head_youden_threshold']:.4f}, ft={info['finetune_youden_threshold']:.4f})"
        )
        return info


    def train_classifier_fixed_budget(
        model: nn.Module,
        train_pairs: list,
        backbone_name: str,
        input_size: int = CNN_INPUT_SIZE,
        norm: str = "imagenet",
        backbone_lr: float = None,
        head_epochs: int = 1,
        finetune_epochs: int = 1,
        run_finetune_phase: bool = True,
        use_staged_unfreeze: bool = False,
        final_weight_strategy: str = "raw",
        phase1_eval_loader=None,
    ):
        """Retrain with a protocol selected without using the evaluation fold/test set."""
        head_epochs = max(1, int(head_epochs))
        finetune_epochs = max(1, int(finetune_epochs))
        final_weight_strategy = str(final_weight_strategy).lower()
        if run_finetune_phase:
            if backbone_lr is None or not np.isfinite(float(backbone_lr)) or float(backbone_lr) <= 0.0:
                raise ValueError(f"{backbone_name}: fixed-budget fine-tuning requires the CV-selected backbone_lr.")
            backbone_lr = float(backbone_lr)

        train_loader = make_dataloader(train_pairs, input_size, augment=True, batch_size=BATCH_SIZE, norm=norm)
        pos_weight = compute_pos_weight([p["label"] for p in train_pairs])
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        history = []

        opt_head = torch.optim.AdamW(model.head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY_HEAD)
        if getattr(model, "pool", None) is not None:
            opt_head.add_param_group({"params": list(model.pool.parameters()), "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY_HEAD})
        sch_head = torch.optim.lr_scheduler.CosineAnnealingLR(opt_head, T_max=max(1, EPOCHS_HEAD_MAX), eta_min=LR_HEAD * 0.05)

        print(f"  Fixed-budget phase 1: head only ({head_epochs} selected epochs)")
        for epoch in range(1, head_epochs + 1):
            stats = train_one_epoch(model, train_loader, opt_head, criterion,
                                    use_mixup=USE_MIXUP, mixup_alpha=MIXUP_ALPHA)
            sch_head.step()
            history.append({
                "phase": "head_only", "epoch": epoch,
                "train_objective": stats["objective"], "train_bce": stats["bce"],
                "train_l2sp": stats["l2sp"], "train_l2sp_contribution": stats["l2sp_contribution"],
                "train_l2sp_fraction_of_bce": stats["l2sp_fraction_of_bce"], "train_auc": stats["auc"],
                "backbone_rms_delta": stats["backbone_rms_delta"],
                "backbone_rel_rms_move": stats["backbone_rel_rms_move"],
                "backbone_max_abs_delta": stats["backbone_max_abs_delta"],
                "grad_backbone_norm_mean": stats["grad_backbone_norm_mean"],
                "grad_head_norm_mean": stats["grad_head_norm_mean"],
                "grad_total_norm_mean": stats["grad_total_norm_mean"],
                "grad_total_norm_max": stats["grad_total_norm_max"],
                "grad_clip_fraction": stats["grad_clip_fraction"],
                "lr_backbone": 0.0, "lr_head": opt_head.param_groups[0]["lr"],
            })
            print(f"    ep{epoch:02d}/{head_epochs} | bce={stats['bce']:.4f} tr_auc_aug_online={stats['auc']:.3f}")

        phase1_state = deepcopy(model.state_dict())
        clean_train_auc_head = evaluate_clean_training_auc(
            model, train_pairs, input_size=input_size, norm=norm
        )
        print(f"  [clean training] head-only fixed checkpoint AUC={clean_train_auc_head:.3f}")
        if history:
            history[-1]["clean_train_auc_fixed_checkpoint"] = float(clean_train_auc_head)
            history[-1]["clean_train_auc_role"] = "head"

        phase1_eval_scored = None
        if phase1_eval_loader is not None:
            phase1_eval_scored = evaluate_return_logits(model, phase1_eval_loader)
            if len(np.unique(phase1_eval_scored["labels"])) >= 2:
                _phase1_outer_auc = roc_auc_score(
                    phase1_eval_scored["labels"], phase1_eval_scored["probs"]
                )
                print(f"  [paired OOF capture] head-only outer AUC={_phase1_outer_auc:.3f}")

        if not run_finetune_phase:
            print("  [head-only] Inner selection did not support phase 2; keeping head-only model.")
            clean_train_diag = {
                "head_auc": float(clean_train_auc_head),
                "finetune_auc": float("nan"),
                "final_auc": float(clean_train_auc_head),
                "finetune_minus_head": float("nan"),
            }
            return history, phase1_eval_scored, phase1_state, clean_train_diag

        unfreeze_last_block(model.backbone, backbone_name)
        backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        pool_params = list(model.pool.parameters()) if getattr(model, "pool", None) is not None else []
        if not backbone_params:
            raise RuntimeError(f"{backbone_name}: no backbone parameters were unfrozen during fixed-budget retraining.")

        l2sp_anchor = capture_l2sp_anchor(model.backbone)
        _ft_profile = small_data_finetune_profile(backbone_name)
        _backbone_lr = float(backbone_lr)
        opt_ft = torch.optim.AdamW([
            {"params": backbone_params, "lr": _backbone_lr, "weight_decay": WEIGHT_DECAY_BACKBONE},
            {"params": head_params + pool_params, "lr": LR_FINETUNE_HEAD, "weight_decay": WEIGHT_DECAY_HEAD},
        ])
        sch_ft = make_finetune_scheduler(opt_ft, EPOCHS_FINETUNE_MAX)
        ema = ModelEMA(model, EMA_DECAY) if USE_EMA else None
        swa = SWATailAverager(SWA_WINDOW) if USE_SWA else None

        print(
            f"  Fixed-budget phase 2: architecture-aware tail + head ({finetune_epochs} selected epochs) | "
            f"backbone LR={_backbone_lr:.2e} | staged={bool(use_staged_unfreeze)} | "
            f"final weights={final_weight_strategy}"
        )
        staged_done = False
        phase2_train_diagnostics = []
        for epoch in range(1, finetune_epochs + 1):
            if (bool(use_staged_unfreeze) and not staged_done
                    and epoch == max(1, int(STAGE1_UNFREEZE_EPOCHS)) + 1):
                newly = unfreeze_deeper_block(model.backbone, backbone_name)
                if newly:
                    deeper_lr = float(_backbone_lr) * float(STAGE2_LR_FACTOR)
                    opt_ft.add_param_group({"params": newly, "lr": deeper_lr, "weight_decay": WEIGHT_DECAY_BACKBONE, "group_name": "backbone_deeper"})
                    for name, p in model.backbone.named_parameters():
                        if p.requires_grad and name not in l2sp_anchor:
                            l2sp_anchor[name] = p.detach().clone()
                    sch_ft = make_finetune_scheduler(opt_ft, EPOCHS_FINETUNE_MAX, stage2_start_epoch=epoch - 1)
                    for _ in range(epoch - 1):
                        sch_ft.step()
                    print(f"    [stage 2 unfreeze @ ep{epoch:02d}] deeper block LR={deeper_lr:g}")
                staged_done = True

            stats = train_one_epoch(
                model, train_loader, opt_ft, criterion,
                l2sp_anchor=l2sp_anchor, l2sp_lambda=L2SP_LAMBDA,
                ema=ema, use_mixup=USE_MIXUP, mixup_alpha=MIXUP_ALPHA,
            )
            phase2_train_diagnostics.append(dict(stats))
            history.append({
                "phase": "last_block_finetune", "epoch": epoch,
                "train_objective": stats["objective"], "train_bce": stats["bce"],
                "train_l2sp": stats["l2sp"], "train_l2sp_contribution": stats["l2sp_contribution"],
                "train_l2sp_fraction_of_bce": stats["l2sp_fraction_of_bce"], "train_auc": stats["auc"],
                "backbone_rms_delta": stats["backbone_rms_delta"],
                "backbone_rel_rms_move": stats["backbone_rel_rms_move"],
                "backbone_max_abs_delta": stats["backbone_max_abs_delta"],
                "grad_backbone_norm_mean": stats["grad_backbone_norm_mean"],
                "grad_head_norm_mean": stats["grad_head_norm_mean"],
                "grad_total_norm_mean": stats["grad_total_norm_mean"],
                "grad_total_norm_max": stats["grad_total_norm_max"],
                "grad_clip_fraction": stats["grad_clip_fraction"],
                "lr_backbone": opt_ft.param_groups[0]["lr"], "lr_head": opt_ft.param_groups[1]["lr"],
            })
            print(
                f"    ep{epoch:02d}/{finetune_epochs} | bce={stats['bce']:.4f} "
                f"l2sp_sum={stats['l2sp']:.4g} l2sp_loss={stats['l2sp_contribution']:.4g} "
                f"({100*stats['l2sp_fraction_of_bce']:.4f}% BCE; scale only) "
                f"bb_move={100*stats['backbone_rel_rms_move']:.3f}% tr_auc_aug_online={stats['auc']:.3f} | "
                f"grad_bb={stats['grad_backbone_norm_mean']:.3g} "
                f"grad_head={stats['grad_head_norm_mean']:.3g} "
                f"grad_total={stats['grad_total_norm_mean']:.3g} "
                f"clip={100*stats['grad_clip_fraction']:.1f}% | "
                f"lr_bb={opt_ft.param_groups[0]['lr']:.2e} lr_head={opt_ft.param_groups[1]['lr']:.2e}"
            )
            sch_ft.step()
            if swa is not None:
                swa.add(model)

            # Backbone movement cap (switch 6). A fixed-budget refit trains on more patients
            # than the inner split that chose the budget, so the same epoch count produces
            # more optimizer steps and more drift. The cap is therefore enforced here too;
            # the shortened budget is recorded in the history for the run summary.
            if _movement_cap_exceeded(stats, epoch, context="fixed budget"):
                history[-1]["movement_cap_stopped_at_epoch"] = int(epoch)
                history[-1]["movement_cap_budget_epochs"] = int(finetune_epochs)
                break

        if final_weight_strategy == "ema":
            if ema is None:
                raise RuntimeError("Inner CV selected EMA but USE_EMA=False.")
            ema.copy_to(model)
            print("  [Fixed-budget weights] EMA shadow selected by inner validation.")
        elif final_weight_strategy == "swa":
            if swa is None or not swa.has_any():
                raise RuntimeError("Inner CV selected SWA but no SWA state is available.")
            model.load_state_dict(swa.averaged_state_dict(), strict=True)
            print(f"  [Fixed-budget weights] SWA tail average over {len(swa.buffer)} epochs selected by inner validation.")
        elif final_weight_strategy == "raw":
            print("  [Fixed-budget weights] Raw final epoch selected by inner validation strategy vote.")
        else:
            raise ValueError(f"Unknown final_weight_strategy={final_weight_strategy!r}")

        # A fixed-budget refit ends at the selected epoch count, so BCE and gradient norms
        # come from the last epoch; movement and L2-SP are recomputed on the selected weights.
        _final_move = backbone_movement_stats(model.backbone, l2sp_anchor)
        _final_diag = dict(phase2_train_diagnostics[-1] if phase2_train_diagnostics else {})
        _final_l2sp = float(l2sp_penalty(model.backbone, l2sp_anchor).detach().cpu())
        _final_l2sp_contrib = float(L2SP_LAMBDA) * _final_l2sp
        _final_bce = float(_final_diag.get("bce", float("nan")))
        _final_diag.update(_final_move)
        _final_diag["l2sp"] = _final_l2sp
        _final_diag["l2sp_contribution"] = _final_l2sp_contrib
        _final_diag["l2sp_fraction_of_bce"] = (
            _final_l2sp_contrib / max(abs(_final_bce), 1e-12)
            if np.isfinite(_final_bce) else float("nan")
        )
        print_finetune_diagnostic(
            _final_diag,
            f"fixed-budget final {final_weight_strategy} weights after {finetune_epochs} fine-tune epochs",
        )

        clean_train_auc_finetune = evaluate_clean_training_auc(
            model, train_pairs, input_size=input_size, norm=norm
        )
        clean_train_diag = {
            "head_auc": float(clean_train_auc_head),
            "finetune_auc": float(clean_train_auc_finetune),
            "final_auc": float(clean_train_auc_finetune),
            "finetune_minus_head": float(clean_train_auc_finetune - clean_train_auc_head),
        }
        print(
            f"  [clean training] head AUC={clean_train_auc_head:.3f} | "
            f"FT AUC={clean_train_auc_finetune:.3f} | "
            f"delta={clean_train_auc_finetune - clean_train_auc_head:+.3f} "
            f"(augment=False, TTA=False, fixed checkpoints)"
        )

        if history:
            history[-1]["clean_train_auc_fixed_checkpoint"] = float(clean_train_auc_finetune)
            history[-1]["clean_train_auc_role"] = "finetune"
            history[-1]["clean_train_auc_head_reference"] = float(clean_train_auc_head)
            history[-1]["clean_train_auc_delta_vs_head"] = float(clean_train_auc_finetune - clean_train_auc_head)
            history[-1]["final_selected_weight_strategy"] = str(final_weight_strategy)
            history[-1]["final_selected_backbone_rms_delta"] = float(_final_move["backbone_rms_delta"])
            history[-1]["final_selected_backbone_rel_rms_move"] = float(_final_move["backbone_rel_rms_move"])
            history[-1]["final_selected_backbone_max_abs_delta"] = float(_final_move["backbone_max_abs_delta"])

        return history, phase1_eval_scored, phase1_state, clean_train_diag


In [ ]:
# Section 3, cell 16: Cross-validation and holdout test
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    def safe_token(s: str) -> str:
        return re.sub(r"[^\w]+", "_", str(s)).strip("_")


    def split_dev_test_by_patient_id(labelled_pairs: list, split_key, context):
        """Apply the shared folder-based patient division (DEVELOPMENT / TEST folders)."""
        pair_by_id = {}
        for item in labelled_pairs:
            cid = canonical_patient_id(item.get("canonical_id", item.get("patient_id")))
            if not cid:
                raise ValueError(f"{context}: blank patient ID in labelled pairs.")
            if cid in pair_by_id:
                raise ValueError(f"{context}: duplicate patient ID in labelled pairs: {cid}")
            pair_by_id[cid] = {**item, "canonical_id": cid}

        dev_ids, test_ids = get_or_create_patient_split(
            split_key=split_key,
            patient_ids=pair_by_id.keys(),
            context=context,
        )
        return [pair_by_id[cid] for cid in dev_ids], [pair_by_id[cid] for cid in test_ids]


    def mean_std(values):
        values = np.array(values, dtype=float)
        return float(np.nanmean(values)), float(np.nanstd(values))


    def fmt_metric(x):
        return "nan" if pd.isna(x) else f"{x:.3f}"


    def print_metrics(prefix: str, metrics: dict):
        print(
            f"{prefix}: "
            f"AUC={fmt_metric(metrics.get('auc'))} | "
            f"Acc={fmt_metric(metrics.get('accuracy'))} | "
            f"BalAcc={fmt_metric(metrics.get('balacc'))} | "
            f"F1={fmt_metric(metrics.get('f1'))} | "
            f"Sens={fmt_metric(metrics.get('sensitivity'))} | "
            f"Spec={fmt_metric(metrics.get('specificity'))} | "
            f"MCC={fmt_metric(metrics.get('mcc'))} | "
            f"Brier={fmt_metric(metrics.get('brier'))} | "
            f"Thr={fmt_metric(metrics.get('threshold'))}"
        )


    def print_final_retrain_summary(rows: list, title: str):
        """Tabulate completed runs. In OPTIMIZATION mode, every row is self-identifying:
        arm ID, reference status, input size, slice count, augmentation and tumour-context
        provenance are printed beside the development-CV and holdout-test metrics."""
        if not rows:
            print(f"\n{title}: (no completed runs)")
            return
        df = order_by_test_set(pd.DataFrame(rows)).reset_index(drop=True)
        # Without a final refit there are development numbers only. An OPTIMIZATION run that
        # keeps the holdout unseen then shows every run once and not once per test set.
        holdout_scored = bool(FINAL_REFIT_AND_HOLDOUT_ACTIVE)
        if not holdout_scored:
            title = f"{title} | final refit and holdout test not run, development CV only"
            df = rows_for_display(df).reset_index(drop=True)

        # CV AUC as mean ± SD per completed run.
        df["CV AUC (mean±sd)"] = [
            ("nan" if pd.isna(m) else f"{m:.3f} ± {(0.0 if pd.isna(s) else s):.3f}")
            for m, s in zip(df["cv_mean_auc"], df["cv_std_auc"])
        ]

        # In optimization mode the arm definition must travel with the result row.
        # This makes the table interpretable even when Colab has truncated earlier output.
        optimization_cols = []
        if "optimization" in df.columns and bool(df["optimization"].fillna(False).any()):
            if "opt_is_reference" in df.columns:
                df["Arm role"] = np.where(df["opt_is_reference"].astype(bool), "REFERENCE", "candidate")
            optimization_cols = [
                c for c in [
                    "opt_arm_id", "Arm role", "opt_input_size", "opt_n_slices",
                    "opt_augmentation", "tumor_context_mode", "roi_only_background_value"
                ]
                if c in df.columns
            ]

        cols = (optimization_cols
                + ["backbone"]
                + (["e2e_mode"] if "e2e_mode" in df.columns else [])
                + ["outcome"]
                + (["test_set"] if "test_set" in df.columns else [])
                + ["CV AUC (mean±sd)",
                "cv_mean_head_best_auc", "cv_mean_finetune_best_auc",
                "cv_mean_finetune_delta_vs_head",
                "paired_oof_head_auc", "paired_oof_finetune_auc", "paired_oof_finetune_delta",
                "selected_head_epochs", "selected_finetune_epochs"]
                + (["final_refit_clean_train_auc_head", "final_refit_clean_train_auc_finetune",
                    "final_refit_clean_train_auc_final", "final_refit_clean_train_minus_test_auc_gap",
                    "final_test_auc", "final_test_accuracy", "final_test_balacc",
                    "final_test_f1", "final_test_sensitivity", "final_test_specificity",
                    "final_test_mcc", "final_test_brier"] if holdout_scored else []))
        disp = df[cols].rename(columns={
            "opt_arm_id": "Arm",
            "opt_input_size": "Input_px",
            "opt_n_slices": "Slices",
            "opt_augmentation": "Augmentation",
            "tumor_context_mode": "Context",
            "roi_only_background_value": "ROI_bg",
            "final_refit_clean_train_auc_head": "CleanTrain_Head",
            "final_refit_clean_train_auc_finetune": "CleanTrain_FT",
            "final_refit_clean_train_auc_final": "CleanTrain_Final",
            "final_refit_clean_train_minus_test_auc_gap": "TrainTestGap",
            "final_test_auc": "Test_AUC", "final_test_accuracy": "Acc",
            "final_test_balacc": "BalAcc", "final_test_f1": "F1",
            "final_test_sensitivity": "Sens", "final_test_specificity": "Spec",
            "final_test_mcc": "MCC", "final_test_brier": "Brier",
        })
        print("\n" + "=" * 140)
        print(title)
        print("-" * 140)
        print(disp.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

        # Do not average arm-definition fields such as input size or slice count.
        num = disp.select_dtypes(include="number")
        _non_metric_numeric = {"Input_px", "Slices", "ROI_bg"}
        metric_num = num[[c for c in num.columns if c not in _non_metric_numeric]]
        # Means are taken within one test set; test sets are never averaged together.
        _sets = list(dict.fromkeys(disp["test_set"])) if "test_set" in disp.columns else [None]
        for _set in _sets:
            _rows = metric_num if _set is None else metric_num[disp["test_set"] == _set]
            if len(_rows) > 1 and not _rows.empty:
                means = _rows.mean()
                cv_mean_over_runs = float(np.nanmean(df.loc[_rows.index, "cv_mean_auc"].to_numpy(dtype=float)))
                print("-" * 140)
                print(
                    "Mean over runs"
                    + (f" [test set {_set}]" if len(_sets) > 1 else "")
                    + f": CV_AUC={cv_mean_over_runs:.3f} | "
                    + " | ".join(f"{c}={means[c]:.3f}" for c in metric_num.columns)
                )
        print("=" * 140)


    def _majority_choice(values, default=None, prefer_simple=None):
        vals = [v for v in values if v is not None]
        if not vals:
            return default
        counts = {v: vals.count(v) for v in set(vals)}
        best_n = max(counts.values())
        tied = [v for v, n in counts.items() if n == best_n]
        if prefer_simple in tied:
            return prefer_simple
        return sorted(tied, key=lambda x: str(x))[0]


    def select_protocol_on_inner_split(outer_train_pairs, backbone_name, fold_seed):
        """Choose backbone LR, pooling, staged unfreezing, epochs and weight strategy without outer-val data."""
        y = np.asarray([p["label"] for p in outer_train_pairs], dtype=int)
        if min(int(y.sum()), int((1 - y).sum())) < 2:
            raise ValueError("Outer training fold has too few minority cases for nested selection.")

        splitter = StratifiedShuffleSplit(n_splits=1, test_size=float(INNER_VAL_FRACTION), random_state=int(fold_seed))
        inner_train_idx, inner_val_idx = next(splitter.split(np.arange(len(outer_train_pairs)), y))
        inner_train = [outer_train_pairs[i] for i in inner_train_idx]
        inner_val = [outer_train_pairs[i] for i in inner_val_idx]

        pool_candidates = tuple(dict.fromkeys(str(x).lower() for x in INNER_POOLING_CANDIDATES))
        staged_candidates = tuple(bool(x) for x in INNER_STAGED_UNFREEZE_CANDIDATES)
        lr_candidates = inner_backbone_lr_candidates(backbone_name)
        candidates = []
        # Shared across staged candidates within this fold, keyed on everything phase 1
        # depends on, so a hit reproduces identical numbers.
        phase1_cache = {}
        print(
            f"    Nested inner selection: train={len(inner_train)} | val={len(inner_val)} | "
            f"LRs={','.join(f'{x:.0e}' for x in lr_candidates)} | "
            f"candidates={len(pool_candidates) * len(staged_candidates) * len(lr_candidates)}"
        )

        for pool_mode in pool_candidates:
            for staged in staged_candidates:
                if staged and not USE_STAGED_UNFREEZE:
                    continue
                for backbone_lr in lr_candidates:
                    # Every LR candidate gets the same starting weights and phase-2 RNG.
                    # Phase 1 is cached because it does not depend on backbone LR.
                    seed_everything(int(fold_seed) + 1000)
                    candidate_model, feat_dim, norm_mode = build_classifier(
                        backbone_name, dropout=DROPOUT, slice_pool=pool_mode
                    )
                    candidate_model = candidate_model.to(device)
                    info = train_classifier(
                        candidate_model, inner_train, inner_val,
                        backbone_name=backbone_name,
                        input_size=CNN_INPUT_SIZE,
                        feat_dim=feat_dim,
                        norm=norm_mode,
                        backbone_lr=float(backbone_lr),
                        use_staged_unfreeze=staged,
                        phase1_cache=phase1_cache,
                        phase1_cache_key=f"{backbone_name}|{int(fold_seed)}|{pool_mode}|{norm_mode}",
                    )
                    # Fine-tuning is never replaced by head-only in this branch; this logic
                    # remains generic, but SKIP_FINETUNE_IF_HEAD_WINS is configured False.
                    run_ft = not (
                        SKIP_FINETUNE_IF_HEAD_WINS
                        and info["finetune_delta_vs_head"] < float(SKIP_FINETUNE_MIN_MEDIAN_DELTA)
                    )
                    selected_auc = info["finetune_best_auc"] if run_ft else info["head_best_auc"]
                    selected_logloss = (
                        info["finetune_best_logloss"] if run_ft else info["head_best_logloss"]
                    )
                    selected_epoch = info["finetune_best_epoch"] if run_ft else 0
                    selected_weight = info["finetune_best_source"] if run_ft else "raw"
                    selected_inner_threshold = (
                        info["finetune_youden_threshold"] if run_ft else info["head_youden_threshold"]
                    )
                    candidates.append({
                        "slice_pool": pool_mode,
                        "use_staged_unfreeze": bool(staged),
                        "backbone_lr": float(backbone_lr),
                        "run_finetune_phase": bool(run_ft),
                        "selected_auc": float(selected_auc),
                        "selected_logloss": float(selected_logloss),
                        "finetune_auc": float(info["finetune_best_auc"]),
                        "finetune_logloss": float(info["finetune_best_logloss"]),
                        "inner_youden_threshold": float(selected_inner_threshold),
                        "head_epochs": int(info["head_best_epoch"]),
                        "finetune_epochs": int(max(1, selected_epoch if run_ft else info["finetune_best_epoch"])),
                        "final_weight_strategy": str(selected_weight),
                        "finetune_delta_vs_head": float(info["finetune_delta_vs_head"]),
                        "inner_info": info,
                        "norm": norm_mode,
                    })
                    del candidate_model
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

        if not candidates:
            raise RuntimeError("No nested protocol candidates were evaluated.")

        # Compare LR/protocol candidates on inner development data only.
        # Primary criterion: higher AUC. AUC ties: lower log-loss. Exact residual ties:
        # lower backbone LR, the more conservative transfer-learning choice.
        def _best_inner_candidate(items, auc_key, logloss_key):
            if not items:
                return None
            best = items[0]
            for cand in items[1:]:
                cand_auc = float(cand[auc_key])
                best_auc = float(best[auc_key])
                cand_ll = float(cand[logloss_key])
                best_ll = float(best[logloss_key])
                if _auc_logloss_better(cand_auc, cand_ll, best_auc, best_ll):
                    best = cand
                elif (
                    np.isfinite(cand_auc) and np.isfinite(best_auc)
                    and abs(cand_auc - best_auc) <= float(MIN_AUC_IMPROVEMENT)
                    and np.isfinite(cand_ll) and np.isfinite(best_ll)
                    and abs(cand_ll - best_ll) <= float(VAL_LOGLOSS_TIE_TOL)
                    and float(cand["backbone_lr"]) < float(best["backbone_lr"])
                ):
                    best = cand
            return best

        simple_candidates = [
            x for x in candidates
            if x["slice_pool"] == "mean" and not x["use_staged_unfreeze"]
        ]
        baseline = _best_inner_candidate(simple_candidates, "selected_auc", "selected_logloss")
        winner = _best_inner_candidate(candidates, "selected_auc", "selected_logloss")
        complexity_min_gain = float(small_data_finetune_profile(backbone_name)["complexity_min_gain"])
        # Both gates below are live only when USE_COMPLEXITY_GATE is on (switch 5). With the
        # switch off, INNER_POOLING_CANDIDATES and INNER_STAGED_UNFREEZE_CANDIDATES hold a
        # single conservative entry, so no extra-capacity candidate exists to gate and the
        # inner selection searches the backbone learning rate only.
        if USE_COMPLEXITY_GATE and baseline is not None and (
            winner["slice_pool"] != "mean" or winner["use_staged_unfreeze"]
        ):
            gain = winner["selected_auc"] - baseline["selected_auc"]
            if not np.isfinite(gain) or gain < complexity_min_gain:
                print(
                    f"    [complexity gate] best complex candidate gain={gain:+.3f} < "
                    f"{complexity_min_gain:.3f} for {small_data_finetune_profile(backbone_name)['profile']}; "
                    "using the best-LR mean-pooling shallow-tail candidate."
                )
                winner = baseline

        # Retain the best genuine fine-tuning protocol independently of any generic head gate.
        ft_winner = _best_inner_candidate(candidates, "finetune_auc", "finetune_logloss")
        ft_simple = _best_inner_candidate(simple_candidates, "finetune_auc", "finetune_logloss")
        if USE_COMPLEXITY_GATE and ft_simple is not None and (
            ft_winner["slice_pool"] != "mean" or ft_winner["use_staged_unfreeze"]
        ):
            ft_gain = float(ft_winner["finetune_auc"]) - float(ft_simple["finetune_auc"])
            if not np.isfinite(ft_gain) or ft_gain < complexity_min_gain:
                print(
                    f"    [fine-tune complexity gate] best complex FT candidate gain={ft_gain:+.3f} < "
                    f"{complexity_min_gain:.3f}; using the best-LR shallow mean-pooling FT candidate."
                )
                ft_winner = ft_simple

        winner["finetune_protocol"] = {
            "slice_pool": str(ft_winner["slice_pool"]),
            "use_staged_unfreeze": bool(ft_winner["use_staged_unfreeze"]),
            "backbone_lr": float(ft_winner["backbone_lr"]),
            "head_epochs": int(ft_winner["inner_info"]["head_best_epoch"]),
            "finetune_epochs": int(max(1, ft_winner["inner_info"]["finetune_best_epoch"])),
            "final_weight_strategy": str(ft_winner["inner_info"]["finetune_best_source"]),
            "finetune_auc": float(ft_winner["inner_info"]["finetune_best_auc"]),
            "finetune_youden_threshold": float(ft_winner["inner_info"]["finetune_youden_threshold"]),
            "backbone_rel_rms_move": float(ft_winner["inner_info"].get("finetune_best_backbone_rel_rms_move", np.nan)),
            "last_l2sp_fraction_of_bce": float(ft_winner["inner_info"].get("finetune_last_l2sp_fraction_of_bce", np.nan)),
            "max_l2sp_fraction_of_bce": float(ft_winner["inner_info"].get("finetune_max_l2sp_fraction_of_bce", np.nan)),
            "delta_vs_head": float(ft_winner["inner_info"].get("finetune_delta_vs_head", np.nan)),
        }

        print(
            f"    Selected nested protocol: pool={winner['slice_pool']} | staged={winner['use_staged_unfreeze']} | "
            f"backbone LR={winner['backbone_lr']:.2e} | head_ep={winner['head_epochs']} | "
            f"ft_ep={winner['finetune_epochs']} | "
            f"phase2={winner['run_finetune_phase']} | weights={winner['final_weight_strategy']} | "
            f"inner AUC={winner['selected_auc']:.3f} | inner Youden={winner['inner_youden_threshold']:.4f}"
        )
        _ftp = winner["finetune_protocol"]
        print(
            f"    FT-specific protocol retained for forced final refit: pool={_ftp['slice_pool']} | "
            f"staged={_ftp['use_staged_unfreeze']} | backbone LR={_ftp['backbone_lr']:.2e} | "
            f"head_ep={_ftp['head_epochs']} | "
            f"ft_ep={_ftp['finetune_epochs']} | weights={_ftp['final_weight_strategy']} | "
            f"FT AUC={_ftp['finetune_auc']:.3f}"
        )
        winner["all_candidate_scores"] = [
            {k: x[k] for k in ("slice_pool", "use_staged_unfreeze", "backbone_lr", "run_finetune_phase", "selected_auc", "selected_logloss", "inner_youden_threshold", "final_weight_strategy")}
            for x in candidates
        ]
        return winner




    all_results = []   # one row per arm × mode × backbone × outcome in OPTIMIZATION mode

    # Fail-closed completeness check. In OPTIMIZATION mode, arm identity is part of
    # the experiment key; otherwise multiple optimization arms collapse to the same
    # mode/backbone/outcome key and a completed comparison without arm-specific keys is rejected.
    # Every run yields one result row per holdout test set, so the test set is part of the key.
    if OPTIMIZATION:
        EXPECTED_RUN_KEYS = {
            (str(_arm["arm_id"]), str(_mode), str(_backbone), str(_cfg["tag"]), str(_test_set))
            for _arm in OPTIMIZATION_ARMS
            for _mode in END2END_MODES
            for _backbone in RUN_BACKBONES
            for _cfg in ACTIVE_LABEL_CONFIGS
            for _test_set in TEST_SET_NAMES
        }
        _expected_formula_count = (
            len(OPTIMIZATION_ARMS)
            * len(END2END_MODES)
            * len(RUN_BACKBONES)
            * len(ACTIVE_LABEL_CONFIGS)
            * len(TEST_SET_NAMES)
        )
    else:
        EXPECTED_RUN_KEYS = {
            (str(_mode), str(_backbone), str(_cfg["tag"]), str(_test_set))
            for _mode in END2END_MODES
            for _backbone in RUN_BACKBONES
            for _cfg in ACTIVE_LABEL_CONFIGS
            for _test_set in TEST_SET_NAMES
        }
        _expected_formula_count = (
            len(END2END_MODES)
            * len(RUN_BACKBONES)
            * len(ACTIVE_LABEL_CONFIGS)
            * len(TEST_SET_NAMES)
        )

    EXPECTED_RUN_COUNT = len(EXPECTED_RUN_KEYS)
    if EXPECTED_RUN_COUNT != _expected_formula_count:
        raise RuntimeError(
            "[configuration][FATAL] Duplicate optimization-arm/mode/backbone/outcome "
            "configuration keys detected; cannot verify run completeness unambiguously."
        )

    # The combined-results CSV is appended row by row as each backbone × outcome
    # finishes, so a mid-run failure still leaves a complete table for what completed.
    INCREMENTAL_SUMMARY_CSV = Path(OUT_DIR) / "cnn_finetuning_results_gray3_all_supported_backbones_holdout.csv"
    if INCREMENTAL_SUMMARY_CSV.exists():
        INCREMENTAL_SUMMARY_CSV.unlink()   # start this run's summary fresh

    for _opt_arm, e2e_mode in OPTIMIZATION_ARM_MODE_PAIRS:
        apply_optimization_arm(_opt_arm)   # no-op when OPTIMIZATION is False
        print("\n" + "#" * 78)
        print(f"# END-TO-END MODE: {e2e_mode}")
        print("#" * 78)
        for backbone_name in RUN_BACKBONES:
            # Backbone loaded once; each fold still gets its own model instance.
            try:
                prime_backbone_initial_state(backbone_name)
            except Exception as _preflight_error:
                raise RuntimeError(
                    f"[backbone][FATAL] Required backbone '{backbone_name}' failed setup/preflight: "
                    f"{repr(_preflight_error)}"
                ) from _preflight_error

            for lcfg in ACTIVE_LABEL_CONFIGS:
                ltag = lcfg["tag"]
                label_df = all_labels_dfs.get(ltag)
                if label_df is None:
                    raise RuntimeError(
                        f"[labels][FATAL] Required label table '{ltag}' is unavailable."
                    )

                print(f"\n{'='*78}")
                print(f"Backbone: {backbone_name} | Mode: {e2e_mode} | Outcome: {ltag} | Policy: {CHANNEL_POLICY}")
                print("Split: DEVELOPMENT folder = development/CV; each TEST folder = one holdout test set")

                labelled_pairs = attach_labels(raw_pairs, label_df)
                if len(labelled_pairs) < 4:
                    raise RuntimeError(
                        f"[data][FATAL] Too few labelled cases for required run "
                        f"{backbone_name} × {ltag}: {len(labelled_pairs)}."
                    )

                try:
                    dev_pairs, all_test_pairs = split_dev_test_by_patient_id(
                        labelled_pairs,
                        split_key=patient_split_record_key(ltag),
                        context=f"Section 3 | outcome={ltag}",
                    )
                    # One entry per holdout test set (one per TEST_ROOT folder). Checked
                    # here, before any training, that every test set can be scored.
                    _holdout_sets = holdout_test_sets(
                        [p["canonical_id"] for p in all_test_pairs],
                        labels=[p["label"] for p in all_test_pairs],
                        context=f"Section 3 | outcome={ltag}",
                    )
                except Exception as e:
                    raise RuntimeError(
                        f"[split][FATAL] Could not create the required development/holdout split "
                        f"for {backbone_name} × {ltag}: {e}"
                    ) from e

                print(
                    f"  Total labelled={len(labelled_pairs)} | development={len(dev_pairs)} "
                    f"(pos={sum(p['label'] for p in dev_pairs)}) | holdout test={len(all_test_pairs)} "
                    f"(pos={sum(p['label'] for p in all_test_pairs)})"
                )
                if len(_holdout_sets) > 1:
                    for _test_set_name, _test_positions in _holdout_sets:
                        print(
                            f"    holdout test set '{_test_set_name}': {len(_test_positions)} "
                            f"(pos={sum(all_test_pairs[_i]['label'] for _i in _test_positions)})"
                        )
                print_patient_split(
                    section=f"Section 3 | backbone={backbone_name} | mode={e2e_mode} | outcome={ltag}",
                    dev_ids=[p["canonical_id"] for p in dev_pairs],
                    test_ids=[p["canonical_id"] for p in all_test_pairs],
                )


                # ── Nested outer CV inside development set only ─────────────────────
                indices = list(range(len(dev_pairs)))
                y_dev = np.array([p["label"] for p in dev_pairs])
                n_folds = min(N_FOLDS, int(y_dev.sum()), int((1 - y_dev).sum()))
                if n_folds < 2:
                    raise RuntimeError(
                        f"[CV][FATAL] Required run {backbone_name} × {ltag} has insufficient "
                        f"minority-class samples for CV (resolved n_folds={n_folds})."
                    )

                rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=N_REPEATS, random_state=GLOBAL_SEED)
                fold_metrics = []
                fold_training_info = []
                oof_prob_accum, oof_logit_accum, oof_label_by_pid = {}, {}, {}
                # One independent OOF score per patient per repeat, so the single-model
                # threshold is the median of repeat-wise thresholds rather than one fitted
                # to an averaged multi-model score.
                oof_prob_by_repeat = [dict() for _ in range(int(N_REPEATS))]
                oof_logit_by_repeat = [dict() for _ in range(int(N_REPEATS))]

                # Paired outer-OOF diagnostic: the same model is scored after phase 1 and
                # again after phase 2, averaged by patient across repeats before comparison.
                paired_head_prob_accum, paired_ft_prob_accum, paired_label_by_pid = {}, {}, {}
                paired_head_logit_accum, paired_ft_logit_accum = {}, {}
                # Family-specific OOF scores per repeat, so the final threshold is calibrated
                # on the same family (head-only or fine-tuned) as the final refit.
                paired_head_prob_by_repeat = [dict() for _ in range(int(N_REPEATS))]
                paired_ft_prob_by_repeat = [dict() for _ in range(int(N_REPEATS))]

                for fold_idx, (train_idx, val_idx) in enumerate(rskf.split(indices, y_dev)):
                    print(f"\n  OUTER CV fold {fold_idx + 1}/{n_folds * N_REPEATS}")
                    train_pairs_fold = [dev_pairs[i] for i in train_idx]
                    val_pairs_fold = [dev_pairs[i] for i in val_idx]
                    print(
                        f"    outer train={len(train_pairs_fold)} (pos={sum(p['label'] for p in train_pairs_fold)}) | "
                        f"outer val={len(val_pairs_fold)} (pos={sum(p['label'] for p in val_pairs_fold)})"
                    )

                    fold_seed = GLOBAL_SEED + fold_idx
                    repeat_idx = int(fold_idx // n_folds)
                    protocol = select_protocol_on_inner_split(train_pairs_fold, backbone_name, fold_seed)

                    # Phase 1 is scored on the untouched outer fold, then the same model
                    # continues through phase 2 and is scored again, giving a paired
                    # patient-level comparison without duplicating training.
                    seed_everything(fold_seed + 50_000)
                    model, feat_dim, norm_mode = build_classifier(
                        backbone_name, dropout=DROPOUT, slice_pool=protocol["slice_pool"]
                    )
                    model = model.to(device)

                    val_loader = make_dataloader(
                        val_pairs_fold, CNN_INPUT_SIZE, augment=False,
                        batch_size=BATCH_SIZE, norm=norm_mode,
                    )

                    _ft_proto = protocol["finetune_protocol"]
                    _outer_ft_epochs = (
                        int(protocol["finetune_epochs"])
                        if bool(protocol["run_finetune_phase"])
                        else int(_ft_proto["finetune_epochs"])
                    )
                    _outer_ft_staged = (
                        bool(protocol["use_staged_unfreeze"])
                        if bool(protocol["run_finetune_phase"])
                        else bool(_ft_proto["use_staged_unfreeze"])
                    )
                    _outer_ft_weights = (
                        str(protocol["final_weight_strategy"])
                        if bool(protocol["run_finetune_phase"])
                        else str(_ft_proto["final_weight_strategy"])
                    )
                    _outer_ft_lr = (
                        float(protocol["backbone_lr"])
                        if bool(protocol["run_finetune_phase"])
                        else float(_ft_proto["backbone_lr"])
                    )

                    _outer_history, head_val_scored, head_phase1_state, _outer_clean_train = train_classifier_fixed_budget(
                        model,
                        train_pairs_fold,
                        backbone_name=backbone_name,
                        input_size=CNN_INPUT_SIZE,
                        norm=norm_mode,
                        backbone_lr=_outer_ft_lr,
                        head_epochs=protocol["head_epochs"],
                        finetune_epochs=_outer_ft_epochs,
                        run_finetune_phase=True,  # always produce the paired FT arm
                        use_staged_unfreeze=_outer_ft_staged,
                        final_weight_strategy=_outer_ft_weights,
                        phase1_eval_loader=val_loader,
                    )
                    if head_val_scored is None:
                        raise RuntimeError("Paired OOF head-only predictions were not captured.")
                    ft_val_scored = evaluate_return_logits(model, val_loader)
                    if not np.array_equal(head_val_scored["labels"], ft_val_scored["labels"]):
                        raise RuntimeError("Head-only and fine-tuned outer-fold labels are misaligned.")

                    # The selected-model stream follows the nested protocol; the extra
                    # fine-tuning arm is diagnostic when head-only won.
                    val_scored = (
                        ft_val_scored if bool(protocol["run_finetune_phase"]) else head_val_scored
                    )
                    # Threshold comes from the inner split and is frozen before the outer
                    # fold is scored.
                    outer_fold_threshold = float(protocol["inner_youden_threshold"])
                    val_metrics = compute_binary_metrics(
                        val_scored["labels"], val_scored["probs"],
                        threshold=outer_fold_threshold,
                        threshold_source="nested inner-validation Youden threshold",
                    )
                    val_metrics.update({"probs": val_scored["probs"], "labels": val_scored["labels"]})
                    fold_metrics.append(val_metrics)

                    _paired_head_auc = float(roc_auc_score(
                        head_val_scored["labels"], head_val_scored["probs"]
                    ))
                    _paired_ft_auc = float(roc_auc_score(
                        ft_val_scored["labels"], ft_val_scored["probs"]
                    ))
                    print(
                        f"    PAIRED OUTER diagnostic (not used for inner selection): "
                        f"head AUC={_paired_head_auc:.3f} | FT AUC={_paired_ft_auc:.3f} | "
                        f"delta={_paired_ft_auc - _paired_head_auc:+.3f}"
                    )

                    info = dict(protocol["inner_info"])
                    info.update({
                        "selected_slice_pool": protocol["slice_pool"],
                        "selected_staged_unfreeze": bool(protocol["use_staged_unfreeze"]),
                        "selected_backbone_lr": float(protocol["backbone_lr"]),
                        "selected_run_finetune_phase": bool(protocol["run_finetune_phase"]),
                        "selected_weight_strategy": str(protocol["final_weight_strategy"]),
                        "selected_head_epochs": int(protocol["head_epochs"]),
                        "selected_finetune_epochs": int(protocol["finetune_epochs"]),
                        "inner_selected_auc": float(protocol["selected_auc"]),
                        "inner_youden_threshold": float(protocol["inner_youden_threshold"]),
                        "ft_specific_slice_pool": str(protocol["finetune_protocol"]["slice_pool"]),
                        "ft_specific_staged_unfreeze": bool(protocol["finetune_protocol"]["use_staged_unfreeze"]),
                        "ft_specific_backbone_lr": float(protocol["finetune_protocol"]["backbone_lr"]),
                        "ft_specific_head_epochs": int(protocol["finetune_protocol"]["head_epochs"]),
                        "ft_specific_finetune_epochs": int(protocol["finetune_protocol"]["finetune_epochs"]),
                        "ft_specific_weight_strategy": str(protocol["finetune_protocol"]["final_weight_strategy"]),
                        "ft_specific_auc": float(protocol["finetune_protocol"]["finetune_auc"]),
                        "ft_specific_backbone_rel_rms_move": float(protocol["finetune_protocol"]["backbone_rel_rms_move"]),
                        "ft_specific_last_l2sp_fraction_of_bce": float(protocol["finetune_protocol"]["last_l2sp_fraction_of_bce"]),
                        "ft_specific_max_l2sp_fraction_of_bce": float(protocol["finetune_protocol"]["max_l2sp_fraction_of_bce"]),
                        "ft_specific_delta_vs_head": float(protocol["finetune_protocol"]["delta_vs_head"]),
                        "outer_paired_head_auc": float(_paired_head_auc),
                        "outer_paired_finetune_auc": float(_paired_ft_auc),
                        "outer_paired_finetune_delta": float(_paired_ft_auc - _paired_head_auc),
                        "clean_train_auc_head": float(_outer_clean_train.get("head_auc", np.nan)),
                        "clean_train_auc_finetune": float(_outer_clean_train.get("finetune_auc", np.nan)),
                        "clean_train_auc_delta": float(_outer_clean_train.get("finetune_minus_head", np.nan)),
                        "head_train_outer_gap": float(_outer_clean_train.get("head_auc", np.nan) - _paired_head_auc),
                        "finetune_train_outer_gap": float(_outer_clean_train.get("finetune_auc", np.nan) - _paired_ft_auc),
                        "n_train_fold": int(len(train_pairs_fold)),
                    })
                    fold_training_info.append(info)

                    for _p, _hprob, _hlogit, _fprob, _flogit in zip(
                        val_pairs_fold,
                        head_val_scored["probs"].tolist(),
                        head_val_scored["logits"].tolist(),
                        ft_val_scored["probs"].tolist(),
                        ft_val_scored["logits"].tolist(),
                    ):
                        _pid = str(_p["canonical_id"])
                        paired_head_prob_accum.setdefault(_pid, []).append(float(_hprob))
                        paired_head_logit_accum.setdefault(_pid, []).append(float(_hlogit))
                        paired_ft_prob_accum.setdefault(_pid, []).append(float(_fprob))
                        paired_ft_logit_accum.setdefault(_pid, []).append(float(_flogit))
                        if _pid in paired_head_prob_by_repeat[repeat_idx] or _pid in paired_ft_prob_by_repeat[repeat_idx]:
                            raise RuntimeError(
                                f"Duplicate paired OOF prediction for patient {_pid} in repeat {repeat_idx}."
                            )
                        paired_head_prob_by_repeat[repeat_idx][_pid] = float(_hprob)
                        paired_ft_prob_by_repeat[repeat_idx][_pid] = float(_fprob)
                        paired_label_by_pid[_pid] = int(_p["label"])

                    for _p, _prob, _logit in zip(
                        val_pairs_fold,
                        val_scored["probs"].tolist(),
                        val_scored["logits"].tolist(),
                    ):
                        _pid = str(_p["canonical_id"])
                        oof_prob_accum.setdefault(_pid, []).append(float(_prob))
                        oof_logit_accum.setdefault(_pid, []).append(float(_logit))
                        if _pid in oof_prob_by_repeat[repeat_idx]:
                            raise RuntimeError(f"Duplicate OOF prediction for patient {_pid} in repeat {repeat_idx}.")
                        oof_prob_by_repeat[repeat_idx][_pid] = float(_prob)
                        oof_logit_by_repeat[repeat_idx][_pid] = float(_logit)
                        oof_label_by_pid[_pid] = int(_p["label"])

                    print_metrics("    OUTER validation (never used for selection)", val_metrics)

                    del model
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                cv_summary = {}
                for key in ["auc", "accuracy", "balacc", "f1", "precision", "sensitivity", "specificity", "mcc", "brier"]:
                    cv_summary[f"cv_mean_{key}"], cv_summary[f"cv_std_{key}"] = mean_std([m[key] for m in fold_metrics])
                cv_summary["cv_mean_inner_youden_threshold"], cv_summary["cv_std_inner_youden_threshold"] = mean_std(
                    [m["threshold"] for m in fold_metrics]
                )

                # Inner-selection diagnostics and hyper-parameter votes. Outer-validation
                # predictions never choose epochs, pooling or weight strategy.
                cv_summary["cv_mean_head_best_auc"], cv_summary["cv_std_head_best_auc"] = mean_std(
                    [info["head_best_auc"] for info in fold_training_info]
                )
                cv_summary["cv_mean_finetune_best_auc"], cv_summary["cv_std_finetune_best_auc"] = mean_std(
                    [info["finetune_best_auc"] for info in fold_training_info]
                )
                cv_summary["cv_mean_finetune_delta_vs_head"], cv_summary["cv_std_finetune_delta_vs_head"] = mean_std(
                    [info["finetune_delta_vs_head"] for info in fold_training_info]
                )
                cv_summary["cv_mean_finetune_best_backbone_rel_rms_move"], cv_summary["cv_std_finetune_best_backbone_rel_rms_move"] = mean_std(
                    [info.get("ft_specific_backbone_rel_rms_move", np.nan) for info in fold_training_info]
                )
                cv_summary["cv_mean_finetune_last_l2sp_fraction_of_bce"], cv_summary["cv_std_finetune_last_l2sp_fraction_of_bce"] = mean_std(
                    [info.get("ft_specific_last_l2sp_fraction_of_bce", np.nan) for info in fold_training_info]
                )
                cv_summary["cv_mean_finetune_grad_clip_fraction"], cv_summary["cv_std_finetune_grad_clip_fraction"] = mean_std(
                    [info.get("finetune_mean_grad_clip_fraction", np.nan) for info in fold_training_info]
                )

                # Paired patient-level comparison: one head-only and one fine-tuned score per
                # patient per repeat from the same trajectory, averaged across repeats before
                # the AUCs are compared.
                _paired_oof_pids = list(paired_label_by_pid.keys())
                if len(_paired_oof_pids) != len(dev_pairs):
                    raise RuntimeError(
                        f"Paired OOF diagnostic has {len(_paired_oof_pids)} patients; "
                        f"expected {len(dev_pairs)}."
                    )
                _paired_oof_labels = np.asarray(
                    [paired_label_by_pid[_pid] for _pid in _paired_oof_pids], dtype=int
                )
                _paired_head_probs = np.asarray(
                    [float(np.mean(paired_head_prob_accum[_pid])) for _pid in _paired_oof_pids],
                    dtype=float,
                )
                _paired_ft_probs = np.asarray(
                    [float(np.mean(paired_ft_prob_accum[_pid])) for _pid in _paired_oof_pids],
                    dtype=float,
                )
                _paired_oof = paired_bootstrap_auc_delta(
                    _paired_oof_labels, _paired_head_probs, _paired_ft_probs,
                    confidence=AUC_CI_CONFIDENCE,
                    n_bootstrap=AUC_BOOTSTRAP_REPEATS,
                    random_state=GLOBAL_SEED + 137,
                )
                cv_summary["paired_oof_head_auc"] = float(_paired_oof["head_auc"])
                cv_summary["paired_oof_finetune_auc"] = float(_paired_oof["finetune_auc"])
                cv_summary["paired_oof_finetune_delta"] = float(_paired_oof["delta"])
                cv_summary["paired_oof_delta_ci_lower"] = float(_paired_oof["ci_lower"])
                cv_summary["paired_oof_delta_ci_upper"] = float(_paired_oof["ci_upper"])
                cv_summary["paired_oof_bootstrap_repeats"] = int(_paired_oof["n_bootstrap"])
                cv_summary["paired_oof_p_finetune_gt_head"] = float(_paired_oof["p_finetune_gt_head"])

                def _median_epoch(values, upper):
                    median_value = float(np.median(np.asarray(values, dtype=float)))
                    return int(np.clip(np.floor(median_value + 0.5), 1, upper))

                # ── Study-level head-vs-fine-tuning decision ──────────────────────
                # Paired pooled outer-OOF AUC delta over all development patients, avoiding the
                # coarse grid of the inner split. The median inner delta stays secondary.
                _median_delta = float(np.median(np.asarray(
                    [info.get("finetune_delta_vs_head", 0.0) for info in fold_training_info], dtype=float
                )))
                _paired_oof_delta = float(cv_summary["paired_oof_finetune_delta"])
                _cv_supports_finetune_phase2 = not (
                    SKIP_FINETUNE_IF_HEAD_WINS
                    and _paired_oof_delta < float(SKIP_FINETUNE_MIN_POOLED_OOF_DELTA)
                )
                run_finetune_phase_final = bool(
                    FORCE_FINAL_REFIT_FINETUNE or _cv_supports_finetune_phase2
                )
                print(
                    f"  Study-level paired OOF diagnostic: head AUC={cv_summary['paired_oof_head_auc']:.3f} | "
                    f"FT AUC={cv_summary['paired_oof_finetune_auc']:.3f} | "
                    f"delta={_paired_oof_delta:+.4f} "
                    f"(95% paired bootstrap CI {cv_summary['paired_oof_delta_ci_lower']:+.4f} to "
                    f"{cv_summary['paired_oof_delta_ci_upper']:+.4f}) | "
                    f"P(FT>head)={cv_summary['paired_oof_p_finetune_gt_head']:.3f}"
                )
                print(
                    f"  Secondary inner-fold diagnostic: median quantized delta={_median_delta:+.4f}"
                )
                print(
                    f"  Phase-2 support threshold={float(SKIP_FINETUNE_MIN_POOLED_OOF_DELTA):+.4f} -> "
                    f"CV_support={_cv_supports_finetune_phase2}"
                )
                print(
                    f"  FINAL refit phase-2 policy: force={bool(FORCE_FINAL_REFIT_FINETUNE)} -> "
                    f"phase2={run_finetune_phase_final}"
                )

                # Vote on the remaining settings. With final fine-tuning forced, each fold
                # contributes its fine-tuning protocol regardless of its own gate; otherwise
                # only folds agreeing with the study-level decision vote. Voting on the full
                # 4-tuple would favour head-only structurally, since run_ft=False forms one
                # bucket while run_ft=True fragments across staged x {raw, ema, swa}.
                def _sub_protocol_tuple(info):
                    if run_finetune_phase_final:
                        return (
                            str(info.get("ft_specific_slice_pool", info["selected_slice_pool"])),
                            bool(info.get("ft_specific_staged_unfreeze", info["selected_staged_unfreeze"])),
                            float(info.get("ft_specific_backbone_lr", info["selected_backbone_lr"])),
                            str(info.get("ft_specific_weight_strategy", info.get("finetune_best_source", "raw"))),
                        )
                    return (
                        str(info["selected_slice_pool"]),
                        False,
                        float(info["selected_backbone_lr"]),
                        "raw",
                    )

                if bool(FORCE_FINAL_REFIT_FINETUNE) and run_finetune_phase_final:
                    # Every fold ran a real fine-tuning arm, so all of them vote on the forced
                    # fine-tuning protocol.
                    _voting_infos = list(fold_training_info)
                else:
                    _voting_infos = [
                        info for info in fold_training_info
                        if bool(info["selected_run_finetune_phase"]) == run_finetune_phase_final
                    ] or list(fold_training_info)

                _protocol_counts = {}
                for _info in _voting_infos:
                    _key = _sub_protocol_tuple(_info)
                    _protocol_counts[_key] = _protocol_counts.get(_key, 0) + 1
                _max_votes = max(_protocol_counts.values())
                _tied_protocols = [k for k, v in _protocol_counts.items() if v == _max_votes]

                def _protocol_complexity_key(k):
                    pool, staged, backbone_lr, weights = k
                    # Lower is simpler/more conservative when vote counts tie.
                    return (
                        int(staged),
                        int(pool != "mean"),
                        int(weights != "raw"),
                        float(backbone_lr),
                        str(k),
                    )

                _selected_sub_tuple = sorted(_tied_protocols, key=_protocol_complexity_key)[0]
                (
                    selected_slice_pool,
                    selected_staged_unfreeze,
                    selected_backbone_lr,
                    selected_weight_strategy,
                ) = _selected_sub_tuple
                _selected_protocol_tuple = (
                    selected_slice_pool, selected_staged_unfreeze,
                    selected_backbone_lr, run_finetune_phase_final, selected_weight_strategy,
                )
                _matching_infos = [
                    info for info in _voting_infos
                    if _sub_protocol_tuple(info) == _selected_sub_tuple
                ]
                if run_finetune_phase_final:
                    selected_head_epochs = _median_epoch(
                        [info.get("ft_specific_head_epochs", info["selected_head_epochs"]) for info in _matching_infos],
                        EPOCHS_HEAD_MAX,
                    )
                    selected_finetune_epochs = _median_epoch(
                        [info.get("ft_specific_finetune_epochs", info["selected_finetune_epochs"]) for info in _matching_infos],
                        EPOCHS_FINETUNE_MAX,
                    )
                else:
                    selected_head_epochs = _median_epoch(
                        [info["selected_head_epochs"] for info in _matching_infos], EPOCHS_HEAD_MAX
                    )
                    selected_finetune_epochs = 1
                # No n_dev/n_train_fold multiplication: larger refit sets already contain
                # more optimizer steps per epoch.
                _vote_basis = (
                    "all folds' FT-specific protocols"
                    if bool(FORCE_FINAL_REFIT_FINETUNE) and run_finetune_phase_final
                    else "folds agreeing with the study-level phase-2 decision"
                )
                print(
                    f"  Final protocol tuple vote: {_selected_protocol_tuple} | "
                    f"votes={_max_votes}/{len(_voting_infos)} | basis={_vote_basis} "
                    f"({len(_voting_infos)}/{len(fold_training_info)} folds)"
                )

                _oof_pids = list(oof_label_by_pid.keys())
                fold_labels_all = [oof_label_by_pid[_pid] for _pid in _oof_pids]
                fold_probs_all = [float(np.mean(oof_prob_accum[_pid])) for _pid in _oof_pids]
                fold_logits_all = [float(np.mean(oof_logit_accum[_pid])) for _pid in _oof_pids]

                # One Youden cut-off per repeat, then the median. Each patient is predicted by
                # exactly one model per repeat, which matches the single final refit better than
                # calibrating on averaged scores.
                repeat_youden_thresholds = []
                repeat_toprisk_thresholds = []
                for _rep_idx, _rep_probs_map in enumerate(oof_prob_by_repeat):
                    _rpids = list(_rep_probs_map.keys())
                    if len(_rpids) != len(dev_pairs):
                        raise RuntimeError(
                            f"Repeat {_rep_idx} has {len(_rpids)} OOF patients; expected {len(dev_pairs)}."
                        )
                    _rlabels = [oof_label_by_pid[_pid] for _pid in _rpids]
                    _rprobs = [_rep_probs_map[_pid] for _pid in _rpids]
                    repeat_youden_thresholds.append(optimal_threshold_youden(_rlabels, _rprobs))
                    if top_risk_mode_enabled():
                        repeat_toprisk_thresholds.append(
                            threshold_for_top_risk(_rprobs, percentage=TOP_RISK_PERCENT)
                        )
                oof_youden_threshold = float(np.median(np.asarray(repeat_youden_thresholds, dtype=float)))
                oof_metrics = compute_binary_metrics(
                    fold_labels_all, fold_probs_all,
                    threshold=oof_youden_threshold,
                    threshold_source="median of repeat-wise nested development OOF Youden thresholds",
                )

                # Pooled development OOF AUC interval; threshold-free, so it describes the
                # selected-model OOF stream as it stands.
                oof_auc_intervals = calculate_auc_intervals(
                    fold_labels_all,
                    fold_probs_all,
                )

                # The threshold is calibrated on the same family as the final refit: only
                # fine-tuned outer-OOF predictions when phase 2 is forced, only head-only ones
                # otherwise.
                if run_finetune_phase_final:
                    final_refit_oof_prob_by_repeat = paired_ft_prob_by_repeat
                    final_refit_oof_family = "fine-tuned"
                else:
                    final_refit_oof_prob_by_repeat = paired_head_prob_by_repeat
                    final_refit_oof_family = "head-only"

                final_refit_repeat_youden_thresholds = []
                final_refit_repeat_toprisk_thresholds = []
                for _rep_idx, _rep_probs_map in enumerate(final_refit_oof_prob_by_repeat):
                    _rpids = list(_rep_probs_map.keys())
                    if len(_rpids) != len(dev_pairs):
                        raise RuntimeError(
                            f"Final-refit {final_refit_oof_family} OOF repeat {_rep_idx} has "
                            f"{len(_rpids)} patients; expected {len(dev_pairs)}."
                        )
                    _rlabels = [paired_label_by_pid[_pid] for _pid in _rpids]
                    _rprobs = [_rep_probs_map[_pid] for _pid in _rpids]
                    final_refit_repeat_youden_thresholds.append(
                        optimal_threshold_youden(_rlabels, _rprobs)
                    )
                    if top_risk_mode_enabled():
                        final_refit_repeat_toprisk_thresholds.append(
                            threshold_for_top_risk(_rprobs, percentage=TOP_RISK_PERCENT)
                        )

                final_refit_oof_youden_threshold = float(
                    np.median(np.asarray(final_refit_repeat_youden_thresholds, dtype=float))
                )
                final_refit_oof_toprisk_threshold = float("nan")
                if top_risk_mode_enabled():
                    final_refit_oof_toprisk_threshold = float(
                        np.median(np.asarray(final_refit_repeat_toprisk_thresholds, dtype=float))
                    )
                final_refit_threshold_source = (
                    f"median repeat-wise development {final_refit_oof_family} outer-OOF "
                    "Youden threshold matched to final refit model family"
                )
                final_refit_toprisk_threshold_source = (
                    f"median repeat-wise development {final_refit_oof_family} outer-OOF "
                    f"top {float(TOP_RISK_PERCENT):g}% score cutoff matched to final refit model family"
                )

                # Development-derived top-risk cutoffs. Holdout cutoffs are never estimated
                # from the holdout score distribution.
                oof_toprisk_metrics = None
                oof_toprisk_threshold = float("nan")
                if top_risk_mode_enabled():
                    oof_toprisk_threshold = float(np.median(np.asarray(repeat_toprisk_thresholds, dtype=float)))
                    oof_toprisk_metrics = compute_binary_metrics(
                        fold_labels_all, fold_probs_all,
                        threshold=oof_toprisk_threshold,
                        threshold_source=f"median repeat-wise development OOF top {float(TOP_RISK_PERCENT):g}% score cutoff",
                    )

                print(f"\n  ── CV summary inside development set [{backbone_name} | {ltag}] ──")
                print(
                    f"  Mean over folds: AUC={cv_summary['cv_mean_auc']:.3f} ± {cv_summary['cv_std_auc']:.3f} | "
                    f"Acc={cv_summary['cv_mean_accuracy']:.3f} ± {cv_summary['cv_std_accuracy']:.3f} | "
                    f"BalAcc={cv_summary['cv_mean_balacc']:.3f} ± {cv_summary['cv_std_balacc']:.3f} | "
                    f"F1={cv_summary['cv_mean_f1']:.3f} ± {cv_summary['cv_std_f1']:.3f} | "
                    f"Sens={cv_summary['cv_mean_sensitivity']:.3f} ± {cv_summary['cv_std_sensitivity']:.3f} | "
                    f"Spec={cv_summary['cv_mean_specificity']:.3f} ± {cv_summary['cv_std_specificity']:.3f}"
                )
                print(
                    f"  Nested inner-selection diagnostic: head-best AUC="
                    f"{cv_summary['cv_mean_head_best_auc']:.3f} ± {cv_summary['cv_std_head_best_auc']:.3f} | "
                    f"fine-tuned-best AUC={cv_summary['cv_mean_finetune_best_auc']:.3f} ± "
                    f"{cv_summary['cv_std_finetune_best_auc']:.3f} | "
                    f"mean delta={cv_summary['cv_mean_finetune_delta_vs_head']:+.3f}"
                )
                print(
                    f"  Fine-tune movement diagnostic across inner selections: "
                    f"relative RMS={100*cv_summary['cv_mean_finetune_best_backbone_rel_rms_move']:.3f}% ± "
                    f"{100*cv_summary['cv_std_finetune_best_backbone_rel_rms_move']:.3f}% | "
                    f"last L2-SP/BCE={100*cv_summary['cv_mean_finetune_last_l2sp_fraction_of_bce']:.4f}% ± "
                    f"{100*cv_summary['cv_std_finetune_last_l2sp_fraction_of_bce']:.4f}% (scale only) | "
                    f"mean clipped batches={100*cv_summary['cv_mean_finetune_grad_clip_fraction']:.1f}%"
                )
                print(
                    f"  PAIRED pooled OOF head-vs-FT: head={cv_summary['paired_oof_head_auc']:.3f} | "
                    f"FT={cv_summary['paired_oof_finetune_auc']:.3f} | "
                    f"delta={cv_summary['paired_oof_finetune_delta']:+.4f} "
                    f"(95% CI {cv_summary['paired_oof_delta_ci_lower']:+.4f} to "
                    f"{cv_summary['paired_oof_delta_ci_upper']:+.4f})"
                )
                print(
                    f"  Final protocol from nested-selection votes: pool={selected_slice_pool} | "
                    f"staged={selected_staged_unfreeze} | backbone LR={selected_backbone_lr:.2e} | "
                    f"head={selected_head_epochs} ep | "
                    f"fine-tune={selected_finetune_epochs} ep | phase2={run_finetune_phase_final} | "
                    f"weights={selected_weight_strategy}"
                )
                print_metrics("  Pooled OOF @ OOF-Youden threshold", oof_metrics)
                print(f"  Pooled OOF DeLong AUC with 95% CI: {oof_auc_intervals['AUC_with_DeLong_95CI']}")
                print(f"  Pooled OOF Jackknife AUC with 95% CI: {oof_auc_intervals['AUC_with_Jackknife_95CI']}")
                print(f"  Pooled OOF Bootstrap AUC with 95% CI: {oof_auc_intervals['AUC_with_Bootstrap_95CI']}")
                print(
                    f"  Final-refit classification threshold source: {final_refit_oof_family} OOF | "
                    f"Youden={final_refit_oof_youden_threshold:.6g}"
                )
                if oof_toprisk_metrics is not None:
                    print_metrics(
                        f"  Pooled OOF @ top {float(TOP_RISK_PERCENT):g}% risk",
                        oof_toprisk_metrics,
                    )

                if FINAL_REFIT_AND_HOLDOUT_ACTIVE:
                    # ── Final retrain on all development patients, evaluate holdout test ─
                    print(f"\n  ── Final retrain on all development patients, then untouched holdout test [{backbone_name} | {ltag}] ──")
                    seed_everything(GLOBAL_SEED + 10_000)
                    final_model, feat_dim, norm_mode = build_classifier(
                        backbone_name, dropout=DROPOUT, slice_pool=selected_slice_pool
                    )
                    final_model = final_model.to(device)

                    final_history, _, _, final_clean_train = train_classifier_fixed_budget(
                        final_model,
                        dev_pairs,
                        backbone_name=backbone_name,
                        input_size=CNN_INPUT_SIZE,
                        norm=norm_mode,
                        backbone_lr=float(selected_backbone_lr),
                        head_epochs=selected_head_epochs,
                        finetune_epochs=selected_finetune_epochs,
                        run_finetune_phase=run_finetune_phase_final,
                        use_staged_unfreeze=selected_staged_unfreeze,
                        final_weight_strategy=selected_weight_strategy,
                    )

                    # The holdout evaluation follows below, once per test set.
                else:
                    # No final refit, no holdout scoring and no saved model. This is the
                    # default of an OPTIMIZATION run (OPTIMIZATION_EVALUATE_HOLDOUT is False)
                    # and the effect of setting RUN_FINAL_REFIT_AND_HOLDOUT to False. The loop
                    # below still writes one row per test set. Their holdout values stay empty.
                    _not_run_reason = (
                        HOLDOUT_NOT_EVALUATED_TEXT if not holdout_evaluation_enabled()
                        else "holdout not evaluated (RUN_FINAL_REFIT_AND_HOLDOUT is False)"
                    )
                    final_model = None
                    final_history = []
                    final_clean_train = {}
                    test_scored = None
                    test_metrics = HoldoutNotEvaluated(threshold_source=_not_run_reason)
                    test_toprisk_metrics = None
                    test_auc_intervals = {}
                    print(f"\n  Final refit and holdout test not run for this arm: {_not_run_reason}.")

                out_prefix = f"{safe_token(backbone_name)}_{safe_token(ltag)}_{CHANNEL_POLICY}_{safe_token(e2e_mode)}"
                # The representation is part of every filename: in OPTIMIZATION mode the
                # arm id already carries it, and in a production run the context tag keeps
                # a MIN_BBOX run from overwriting a ROI_ONLY one inside the same folder.
                if _opt_arm is not None:
                    out_prefix = f"{safe_token(_opt_arm['arm_id'])}__{out_prefix}"
                else:
                    out_prefix = f"{safe_token(tumor_context_tag())}__{out_prefix}"
                if final_history:
                    pd.DataFrame(final_history).to_csv(Path(OUT_DIR) / f"{out_prefix}_final_retrain_history.csv", index=False)

                # Per-fold CV metrics + best-epoch records (one row per outer fold × repeat).
                _fold_rows = []
                for _fi, (_fm, _fti) in enumerate(zip(fold_metrics, fold_training_info)):
                    _fold_rows.append({
                        "fold": _fi + 1,
                        "val_auc": _fm.get("auc"),
                        "val_accuracy": _fm.get("accuracy"),
                        "val_balacc": _fm.get("balacc"),
                        "val_f1": _fm.get("f1"),
                        "val_sensitivity": _fm.get("sensitivity"),
                        "val_specificity": _fm.get("specificity"),
                        "val_mcc": _fm.get("mcc"),
                        "val_brier": _fm.get("brier"),
                        "head_best_auc": _fti.get("head_best_auc"),
                        "head_best_epoch": _fti.get("head_best_epoch"),
                        "finetune_best_auc": _fti.get("finetune_best_auc"),
                        "finetune_best_epoch": _fti.get("finetune_best_epoch"),
                        "finetune_best_source": _fti.get("finetune_best_source"),
                        "swa_val_auc": _fti.get("swa_val_auc"),
                        "n_train_fold": _fti.get("n_train_fold"),
                        "finetune_delta_vs_head": _fti.get("finetune_delta_vs_head"),
                        "finetune_best_backbone_rel_rms_move": _fti.get("finetune_best_backbone_rel_rms_move"),
                        "finetune_last_l2sp_fraction_of_bce": _fti.get("finetune_last_l2sp_fraction_of_bce"),
                        "finetune_max_l2sp_fraction_of_bce": _fti.get("finetune_max_l2sp_fraction_of_bce"),
                        "ft_specific_slice_pool": _fti.get("ft_specific_slice_pool"),
                        "ft_specific_staged_unfreeze": _fti.get("ft_specific_staged_unfreeze"),
                        "ft_specific_backbone_lr": _fti.get("ft_specific_backbone_lr"),
                        "ft_specific_head_epochs": _fti.get("ft_specific_head_epochs"),
                        "ft_specific_finetune_epochs": _fti.get("ft_specific_finetune_epochs"),
                        "ft_specific_weight_strategy": _fti.get("ft_specific_weight_strategy"),
                        "ft_specific_auc": _fti.get("ft_specific_auc"),
                        "ft_specific_delta_vs_head": _fti.get("ft_specific_delta_vs_head"),
                        "ft_specific_backbone_rel_rms_move": _fti.get("ft_specific_backbone_rel_rms_move"),
                        "ft_specific_last_l2sp_fraction_of_bce": _fti.get("ft_specific_last_l2sp_fraction_of_bce"),
                        "ft_specific_max_l2sp_fraction_of_bce": _fti.get("ft_specific_max_l2sp_fraction_of_bce"),
                        "finetune_mean_grad_backbone_norm": _fti.get("finetune_mean_grad_backbone_norm"),
                        "finetune_mean_grad_head_norm": _fti.get("finetune_mean_grad_head_norm"),
                        "finetune_mean_grad_total_norm": _fti.get("finetune_mean_grad_total_norm"),
                        "finetune_mean_grad_clip_fraction": _fti.get("finetune_mean_grad_clip_fraction"),
                        "outer_paired_head_auc": _fti.get("outer_paired_head_auc"),
                        "outer_paired_finetune_auc": _fti.get("outer_paired_finetune_auc"),
                        "outer_paired_finetune_delta": _fti.get("outer_paired_finetune_delta"),
                        "clean_train_auc_head": _fti.get("clean_train_auc_head"),
                        "clean_train_auc_finetune": _fti.get("clean_train_auc_finetune"),
                        "clean_train_auc_delta": _fti.get("clean_train_auc_delta"),
                        "head_train_outer_gap": _fti.get("head_train_outer_gap"),
                        "finetune_train_outer_gap": _fti.get("finetune_train_outer_gap"),
                        "selected_slice_pool": _fti.get("selected_slice_pool"),
                        "selected_staged_unfreeze": _fti.get("selected_staged_unfreeze"),
                        "selected_backbone_lr": _fti.get("selected_backbone_lr"),
                        "selected_run_finetune_phase": _fti.get("selected_run_finetune_phase"),
                        "selected_weight_strategy": _fti.get("selected_weight_strategy"),
                        "selected_head_epochs": _fti.get("selected_head_epochs"),
                        "selected_finetune_epochs": _fti.get("selected_finetune_epochs"),
                        "inner_selected_auc": _fti.get("inner_selected_auc"),
                    })
                pd.DataFrame(_fold_rows).to_csv(
                    Path(OUT_DIR) / f"{out_prefix}_cv_fold_metrics.csv", index=False
                )

                # Per-patient pooled development OOF predictions with the frozen decisions,
                # so the OOF ROC and threshold can be re-audited.
                _oof_pred_export = {
                    "canonical_id": _oof_pids,
                    "true_label": fold_labels_all,
                    "oof_probability_mean": fold_probs_all,
                    "oof_mean_logit": fold_logits_all,
                    "paired_head_probability_mean": [
                        float(np.mean(paired_head_prob_accum[_pid])) for _pid in _oof_pids
                    ],
                    "paired_finetune_probability_mean": [
                        float(np.mean(paired_ft_prob_accum[_pid])) for _pid in _oof_pids
                    ],
                    "paired_probability_delta": [
                        float(np.mean(paired_ft_prob_accum[_pid]) - np.mean(paired_head_prob_accum[_pid]))
                        for _pid in _oof_pids
                    ],
                    "paired_oof_head_auc": float(cv_summary["paired_oof_head_auc"]),
                    "paired_oof_finetune_auc": float(cv_summary["paired_oof_finetune_auc"]),
                    "paired_oof_auc_delta": float(cv_summary["paired_oof_finetune_delta"]),
                    "paired_oof_delta_ci_lower": float(cv_summary["paired_oof_delta_ci_lower"]),
                    "paired_oof_delta_ci_upper": float(cv_summary["paired_oof_delta_ci_upper"]),
                    "n_oof_predictions": [len(oof_prob_accum[_pid]) for _pid in _oof_pids],
                    "predicted_label_oof_youden": [
                        int(_pp >= oof_youden_threshold) for _pp in fold_probs_all
                    ],
                    "oof_youden_threshold": float(oof_youden_threshold),
                    "final_refit_oof_model_family": final_refit_oof_family,
                    "final_refit_oof_youden_threshold": float(final_refit_oof_youden_threshold),
                    "final_refit_threshold_source": final_refit_threshold_source,
                }
                if oof_toprisk_metrics is not None:
                    _oof_pred_export["predicted_label_top_risk"] = [
                        int(_pp >= oof_toprisk_metrics["threshold"]) for _pp in fold_probs_all
                    ]
                    _oof_pred_export["top_risk_threshold"] = float(oof_toprisk_threshold)
                pd.DataFrame(_oof_pred_export).to_csv(
                    Path(OUT_DIR) / f"{out_prefix}_oof_dev_predictions.csv", index=False
                )
                if FINAL_REFIT_AND_HOLDOUT_ACTIVE:
                    torch.save(final_model.state_dict(), Path(OUT_DIR) / f"{out_prefix}_final_retrain_model.pt")

                _final_history_last = final_history[-1] if final_history else {}
                _final_refit_rel_move = float(_final_history_last.get("final_selected_backbone_rel_rms_move", np.nan))
                _final_refit_rms_delta = float(_final_history_last.get("final_selected_backbone_rms_delta", np.nan))

                # The model refitted above is scored on each holdout test set separately:
                # one pass, one prediction file and one result row per test set. With a
                # single test set this loop runs once.
                _test_results_by_set = {}
                _run_rows = []
                for _test_set_name, _test_positions in _holdout_sets:
                    test_pairs = [all_test_pairs[_i] for _i in _test_positions]
                    # Scoring an additional test set must not move the random-number generators,
                    # so later results are exactly those of a run with one test set.
                    _rng_state = rng_state_snapshot() if _test_set_name != _holdout_sets[0][0] else None
                    if FINAL_REFIT_AND_HOLDOUT_ACTIVE:
                        if len(_holdout_sets) > 1:
                            print(f"\n  ── Holdout test set '{_test_set_name}' ({len(test_pairs)} patients) ──")
                        test_loader = make_dataloader(test_pairs, CNN_INPUT_SIZE, augment=False, batch_size=BATCH_SIZE, norm=norm_mode)
                        test_scored = evaluate(final_model, test_loader)

                        # Holdout metrics use a development-only OOF threshold from the same model
                        # family as the final refit; holdout labels never inform it.
                        test_metrics = compute_binary_metrics(
                            test_scored["labels"],
                            test_scored["probs"],
                            threshold=final_refit_oof_youden_threshold,
                            threshold_source=final_refit_threshold_source + " applied unchanged to holdout",
                        )
                        test_metrics.update({
                            "labels": test_scored["labels"],
                            "probs": test_scored["probs"],
                        })

                        # Secondary cohort-relative top-risk view: rank the holdout predictions
                        # themselves and classify the highest-scoring TOP_RISK_PERCENT as positive.
                        # The percentage is prespecified; holdout outcomes do not determine the cutoff.
                        test_toprisk_metrics = None
                        if top_risk_mode_enabled():
                            test_toprisk_metrics = compute_top_risk_binary_metrics(
                                test_scored["labels"],
                                test_scored["probs"],
                                percentage=TOP_RISK_PERCENT,
                            )

                        test_auc_intervals = calculate_auc_intervals(
                            test_metrics["labels"],
                            test_metrics["probs"],
                        )
                        test_metrics.update(test_auc_intervals)

                        print_metrics("  HOLDOUT TEST @ development OOF-Youden threshold", test_metrics)
                        if test_toprisk_metrics is not None:
                            print_metrics(
                                f"  HOLDOUT TEST @ top {float(TOP_RISK_PERCENT):g}% risk",
                                test_toprisk_metrics,
                            )
                        print(f"  DeLong AUC with 95% CI: {test_metrics['AUC_with_DeLong_95CI']}")
                        print(f"  Jackknife AUC with 95% CI: {test_metrics['AUC_with_Jackknife_95CI']}")
                        print(f"  Bootstrap AUC with 95% CI: {test_metrics['AUC_with_Bootstrap_95CI']}")
                        print(
                            f"  Jackknife leave-one-out AUC range: "
                            f"{test_metrics['Jackknife_min_AUC']:.3f}–"
                            f"{test_metrics['Jackknife_max_AUC']:.3f}"
                        )

                        _test_pred_export = {
                            "canonical_id": [str(p["canonical_id"]) for p in test_pairs],
                            "true_label": test_metrics["labels"],
                            "probability": test_metrics["probs"],
                            "predicted_label_oof_youden": (
                                test_metrics["probs"] >= test_metrics["threshold"]
                            ).astype(int),
                            "classification_threshold_oof_youden": test_metrics["threshold"],
                            "threshold_source_oof_youden": test_metrics["threshold_source"],
                        }
                        if test_toprisk_metrics is not None:
                            _test_pred_export.update({
                                "predicted_label_top_risk": (
                                    test_metrics["probs"] >= test_toprisk_metrics["threshold"]
                                ).astype(int),
                                "classification_threshold_top_risk": test_toprisk_metrics["threshold"],
                                "threshold_source_top_risk": test_toprisk_metrics["threshold_source"],
                            })
                        pd.DataFrame(_test_pred_export).to_csv(
                            Path(OUT_DIR) / f"{out_prefix}_holdout_test_predictions{test_set_file_tag(_test_set_name)}.csv",
                            index=False,
                        )

                    row = {
                        "optimization": bool(OPTIMIZATION),
                        "opt_arm_id": (_opt_arm["arm_id"] if _opt_arm is not None else ""),
                        "opt_is_reference": bool(
                            optimization_arm_is_reference("finetune", _opt_arm)
                        ),
                        "opt_input_size": int(CNN_INPUT_SIZE),
                        "opt_n_slices": int(N_SLICES),
                        "opt_augmentation": str(TRAIN_AUGMENTATION_LEVEL).upper(),
                        "tumor_context_mode": str(TUMOR_CONTEXT_MODE).upper(),
                        "roi_only_background_value": (
                            float(ROI_ONLY_BACKGROUND_VALUE)
                            if str(TUMOR_CONTEXT_MODE).upper() == "ROI_ONLY" else np.nan
                        ),
                        "backbone": backbone_name,
                        "e2e_mode": e2e_mode,
                        "outcome": ltag,
                        "test_set": _test_set_name,
                        "channel_policy": CHANNEL_POLICY,
                        "n_total_cases": len(labelled_pairs),
                        "n_dev_cases": len(dev_pairs),
                        "n_test_cases": len(test_pairs),
                        "n_folds": n_folds,
                        "n_repeats": N_REPEATS,
                        "selected_head_epochs": selected_head_epochs,
                        "selected_finetune_epochs": selected_finetune_epochs,
                        "nested_outer_cv": True,
                        "inner_val_fraction": float(INNER_VAL_FRACTION),
                        "inner_complexity_min_auc_gain": float(small_data_finetune_profile(backbone_name)["complexity_min_gain"]),
                        "small_data_finetune_profile": str(small_data_finetune_profile(backbone_name)["profile"]),
                        "inner_backbone_lr_candidates": "|".join(
                            f"{x:.0e}" for x in inner_backbone_lr_candidates(backbone_name)
                        ),
                        "selected_backbone_lr": float(selected_backbone_lr),
                        "final_refit_ran_phase2": bool(run_finetune_phase_final),
                        "force_final_refit_finetune": bool(FORCE_FINAL_REFIT_FINETUNE),
                        "cv_supports_finetune_phase2": bool(_cv_supports_finetune_phase2),
                        "selected_weight_strategy": str(selected_weight_strategy),
                        "cv_median_finetune_delta_vs_head": _median_delta,
                        "skip_finetune_if_head_wins": bool(SKIP_FINETUNE_IF_HEAD_WINS),
                        "skip_finetune_min_median_delta": float(SKIP_FINETUNE_MIN_MEDIAN_DELTA),
                        "skip_finetune_min_pooled_oof_delta": float(SKIP_FINETUNE_MIN_POOLED_OOF_DELTA),
                        "slice_pool": selected_slice_pool,
                        "use_mixup": bool(USE_MIXUP),
                        "mixup_alpha": float(MIXUP_ALPHA),
                        "label_smoothing": float(LABEL_SMOOTHING),
                        "use_ema": bool(USE_EMA),
                        "ema_decay": float(EMA_DECAY),
                        "dropout": float(DROPOUT),
                        "drop_path_rate": float(DROP_PATH_RATE),
                        "training_augmentation": str(TRAIN_AUGMENTATION),
                        "use_staged_unfreeze": bool(selected_staged_unfreeze),
                        "stage1_unfreeze_epochs": int(STAGE1_UNFREEZE_EPOCHS),
                        "stage2_lr_factor": float(STAGE2_LR_FACTOR),
                        "use_swa": bool(USE_SWA),
                        "swa_window": int(SWA_WINDOW),
                        "use_smoothed_selection": bool(USE_SMOOTHED_SELECTION),
                        "smoothing_window": int(SMOOTHING_WINDOW),
                        "use_tta_rotations": bool(USE_TTA_ROTATIONS),
                        "lr_head_phase1": LR_HEAD,
                        "lr_finetune_backbone": float(selected_backbone_lr),
                        "lr_finetune_head": LR_FINETUNE_HEAD,
                        "finetune_warmup_epochs": FINETUNE_WARMUP_EPOCHS,
                        "finetune_min_selection_epoch": FINETUNE_MIN_SELECTION_EPOCH,
                        "val_logloss_tie_tol": VAL_LOGLOSS_TIE_TOL,
                        "grad_clip_norm": GRAD_CLIP_NORM,
                        "inner_staged_unfreeze_candidates": "|".join(str(bool(x)) for x in INNER_STAGED_UNFREEZE_CANDIDATES),
                        "inner_backbone_lr_candidates_configured": "|".join(
                            f"{x:.0e}" for x in inner_backbone_lr_candidates(backbone_name)
                        ),
                        "l2sp_lambda": L2SP_LAMBDA,
                        "final_refit_backbone_rel_rms_move": _final_refit_rel_move,
                        "final_refit_backbone_rms_delta": _final_refit_rms_delta,
                        "final_refit_clean_train_auc_head": float(final_clean_train.get("head_auc", np.nan)),
                        "final_refit_clean_train_auc_finetune": float(final_clean_train.get("finetune_auc", np.nan)),
                        "final_refit_clean_train_auc_final": float(final_clean_train.get("final_auc", np.nan)),
                        "final_refit_clean_train_auc_delta": float(final_clean_train.get("finetune_minus_head", np.nan)),
                        **cv_summary,

                        # Development OOF metrics using the median repeat-wise Youden threshold.
                        "oof_auc": oof_metrics["auc"],
                        "oof_DeLong_CI_lower": oof_auc_intervals["DeLong_CI_lower"],
                        "oof_DeLong_CI_upper": oof_auc_intervals["DeLong_CI_upper"],
                        "oof_AUC_with_DeLong_95CI": oof_auc_intervals["AUC_with_DeLong_95CI"],
                        "oof_Jackknife_CI_lower": oof_auc_intervals["Jackknife_CI_lower"],
                        "oof_Jackknife_CI_upper": oof_auc_intervals["Jackknife_CI_upper"],
                        "oof_AUC_with_Jackknife_95CI": oof_auc_intervals["AUC_with_Jackknife_95CI"],
                        "oof_Jackknife_min_AUC": oof_auc_intervals["Jackknife_min_AUC"],
                        "oof_Jackknife_max_AUC": oof_auc_intervals["Jackknife_max_AUC"],
                        "oof_Bootstrap_CI_lower": oof_auc_intervals["Bootstrap_CI_lower"],
                        "oof_Bootstrap_CI_upper": oof_auc_intervals["Bootstrap_CI_upper"],
                        "oof_AUC_with_Bootstrap_95CI": oof_auc_intervals["AUC_with_Bootstrap_95CI"],
                        "oof_Bootstrap_repeats": oof_auc_intervals["Bootstrap_repeats"],
                        "oof_accuracy": oof_metrics["accuracy"],
                        "oof_balacc": oof_metrics["balacc"],
                        "oof_f1": oof_metrics["f1"],
                        "oof_sensitivity": oof_metrics["sensitivity"],
                        "oof_specificity": oof_metrics["specificity"],
                        "oof_mcc": oof_metrics["mcc"],
                        "oof_brier": oof_metrics["brier"],
                        "oof_youden_threshold": float(oof_youden_threshold),
                        "oof_repeat_youden_thresholds": "|".join(f"{x:.8g}" for x in repeat_youden_thresholds),
                        "final_refit_oof_model_family": final_refit_oof_family,
                        "final_refit_oof_youden_threshold": float(final_refit_oof_youden_threshold),
                        "final_refit_oof_repeat_youden_thresholds": "|".join(
                            f"{x:.8g}" for x in final_refit_repeat_youden_thresholds
                        ),
                        "final_refit_oof_threshold_source": final_refit_threshold_source,
                        "final_refit_oof_toprisk_threshold": (
                            final_refit_oof_toprisk_threshold if top_risk_mode_enabled() else np.nan
                        ),
                        "final_refit_oof_toprisk_threshold_source": (
                            final_refit_toprisk_threshold_source if top_risk_mode_enabled() else ""
                        ),

                        # Development OOF, top-risk cut-off.
                        "oof_toprisk_percent": float(TOP_RISK_PERCENT) if oof_toprisk_metrics is not None else np.nan,
                        "oof_toprisk_threshold": (
                            oof_toprisk_threshold if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_accuracy": (
                            oof_toprisk_metrics["accuracy"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_balacc": (
                            oof_toprisk_metrics["balacc"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_f1": (
                            oof_toprisk_metrics["f1"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_sensitivity": (
                            oof_toprisk_metrics["sensitivity"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_specificity": (
                            oof_toprisk_metrics["specificity"] if oof_toprisk_metrics is not None else np.nan
                        ),
                        "oof_toprisk_mcc": (
                            oof_toprisk_metrics["mcc"] if oof_toprisk_metrics is not None else np.nan
                        ),

                        # Holdout AUC and intervals (threshold-free).
                        "final_test_auc": test_metrics["auc"],
                        "final_refit_clean_train_minus_test_auc_gap": float(
                            final_clean_train.get("final_auc", np.nan) - test_metrics["auc"]
                        ),
                        "final_refit_head_clean_train_minus_test_auc_gap": float(
                            final_clean_train.get("head_auc", np.nan) - test_metrics["auc"]
                        ),
                        "final_test_DeLong_CI_lower": test_metrics["DeLong_CI_lower"],
                        "final_test_DeLong_CI_upper": test_metrics["DeLong_CI_upper"],
                        "final_test_AUC_with_DeLong_95CI": test_metrics["AUC_with_DeLong_95CI"],
                        "final_test_Jackknife_CI_lower": test_metrics["Jackknife_CI_lower"],
                        "final_test_Jackknife_CI_upper": test_metrics["Jackknife_CI_upper"],
                        "final_test_AUC_with_Jackknife_95CI": test_metrics["AUC_with_Jackknife_95CI"],
                        "final_test_Jackknife_min_AUC": test_metrics["Jackknife_min_AUC"],
                        "final_test_Jackknife_max_AUC": test_metrics["Jackknife_max_AUC"],
                        "final_test_Bootstrap_CI_lower": test_metrics["Bootstrap_CI_lower"],
                        "final_test_Bootstrap_CI_upper": test_metrics["Bootstrap_CI_upper"],
                        "final_test_AUC_with_Bootstrap_95CI": test_metrics["AUC_with_Bootstrap_95CI"],
                        "final_test_Bootstrap_repeats": test_metrics["Bootstrap_repeats"],

                        # Holdout metrics at the OOF-derived Youden cut-off.
                        "final_test_accuracy": test_metrics["accuracy"],
                        "final_test_balacc": test_metrics["balacc"],
                        "final_test_f1": test_metrics["f1"],
                        "final_test_precision": test_metrics["precision"],
                        "final_test_sensitivity": test_metrics["sensitivity"],
                        "final_test_specificity": test_metrics["specificity"],
                        "final_test_mcc": test_metrics["mcc"],
                        "final_test_brier": test_metrics["brier"],
                        "final_test_threshold": test_metrics["threshold"],
                        "final_test_threshold_source": test_metrics["threshold_source"],

                        # Holdout metrics, top-risk cut-off.
                        "final_test_toprisk_percent": float(TOP_RISK_PERCENT) if test_toprisk_metrics is not None else np.nan,
                        "final_test_toprisk_threshold": (
                            test_toprisk_metrics["threshold"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_accuracy": (
                            test_toprisk_metrics["accuracy"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_balacc": (
                            test_toprisk_metrics["balacc"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_f1": (
                            test_toprisk_metrics["f1"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_precision": (
                            test_toprisk_metrics["precision"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_sensitivity": (
                            test_toprisk_metrics["sensitivity"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_specificity": (
                            test_toprisk_metrics["specificity"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_mcc": (
                            test_toprisk_metrics["mcc"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_brier": (
                            test_toprisk_metrics["brier"] if test_toprisk_metrics is not None else np.nan
                        ),
                        "final_test_toprisk_threshold_source": (
                            test_toprisk_metrics["threshold_source"] if test_toprisk_metrics is not None else ""
                        ),

                    }

                    _test_results_by_set[_test_set_name] = {
                        _k: _v for _k, _v in row.items()
                        if str(_k).startswith("final_test_")
                        or _k in (
                            "final_refit_clean_train_minus_test_auc_gap",
                            "final_refit_head_clean_train_minus_test_auc_gap",
                        )
                    }

                    if OPTIMIZATION and _opt_arm is not None:
                        # Per-patient pooled OOF probabilities, kept in memory so arms can be
                        # compared PAIRED on identical patients rather than as two independent
                        # AUCs. Labels are stored once and cross-checked per arm.
                        OPTIMIZATION_OOF_STORE[(_opt_arm["arm_id"], backbone_name, ltag)] = {
                            "probs": {str(_pid): float(_p) for _pid, _p in zip(_oof_pids, fold_probs_all)},
                            "labels": {str(_pid): int(_l) for _pid, _l in zip(_oof_pids, fold_labels_all)},
                            "oof_auc": float(oof_metrics["auc"]),
                            "paired_oof_finetune_delta": float(cv_summary["paired_oof_finetune_delta"]),
                            "is_reference": bool(
                                optimization_arm_is_reference("finetune", _opt_arm)
                            ),
                            "input_size": int(_opt_arm["input_size"]),
                            "n_slices": int(_opt_arm["n_slices"]),
                            "augmentation": str(_opt_arm["augmentation"]),
                            "context_mode": str(_opt_arm["context_mode"]),
                            "roi_background": float(_opt_arm["roi_background"]),
                            # Keep every holdout-test result beside the paired OOF record so
                            # the optimization summary can display OOF and test performance
                            # for the same arm without re-reading any files.
                            "test_results": dict(next(iter(_test_results_by_set.values()))),
                            "test_results_by_test_set": {_n: dict(_r) for _n, _r in _test_results_by_set.items()},
                        }

                    all_results.append(row)
                    _run_rows.append(row)

                    # Rewrite the combined summary now, sorted by holdout AUC, so completed runs
                    # survive a later failure.
                    pd.DataFrame(all_results).sort_values(
                        "final_test_auc", ascending=False, na_position="last"
                    ).to_csv(INCREMENTAL_SUMMARY_CSV, index=False)
                    rng_state_restore(_rng_state)

                print(f"\n  ── Completed [{backbone_name} | {ltag}] ──")

                # ── PRIMARY ENDPOINT ─────────────────────────────────────────────
                # The scientific question in this arm is whether fine-tuning the backbone
                # tail beats the frozen head on the SAME patients. That is the paired
                # development OOF delta with its bootstrap CI, on all development patients.
                # It is reported first, and it is reported whatever its sign. The holdout
                # AUC is a descriptive confirmation on a handful of positives and cannot
                # rank backbones; it is printed below under SECONDARY.
                _pd_lo = row['paired_oof_delta_ci_lower']
                _pd_hi = row['paired_oof_delta_ci_upper']
                _pd = row['paired_oof_finetune_delta']
                _pd_verdict = (
                    "fine-tuning helps"      if _pd_lo > 0.0 else
                    "fine-tuning hurts"      if _pd_hi < 0.0 else
                    "no detectable effect (CI includes 0)"
                )
                print("  " + "=" * 76)
                if OPTIMIZATION:
                    _roi_bg_txt = (
                        f"{row['roi_only_background_value']:.3f}"
                        if np.isfinite(float(row.get('roi_only_background_value', np.nan)))
                        else "n/a"
                    )
                    _ref_txt = " | REFERENCE" if bool(row.get("opt_is_reference", False)) else ""
                    print(
                        f"  OPTIMIZATION ARM: {row['opt_arm_id']} | "
                        f"input={int(row['opt_input_size'])} px | "
                        f"slices={int(row['opt_n_slices'])} | "
                        f"augmentation={row['opt_augmentation']} | "
                        f"context={row['tumor_context_mode']} | ROI_bg={_roi_bg_txt}{_ref_txt}"
                    )
                print(f"  PRIMARY ENDPOINT: paired development OOF, n={row['n_dev_cases']} patients, "
                      f"{row['n_folds']}x{row['n_repeats']} CV")
                print(f"    head-only OOF AUC   = {row['paired_oof_head_auc']:.3f}")
                print(f"    fine-tuned OOF AUC  = {row['paired_oof_finetune_auc']:.3f}")
                print(f"    paired delta        = {_pd:+.4f}  "
                      f"(95% CI {_pd_lo:+.4f} to {_pd_hi:+.4f})  ->  {_pd_verdict}")
                print(f"    P(FT > head)        = {row['paired_oof_p_finetune_gt_head']:.3f}")
                print(f"    pooled OOF AUC      = {row['oof_auc']:.3f} "
                      f"{row.get('oof_AUC_with_DeLong_95CI', '')}")
                print("  " + "-" * 76)
                if FINAL_REFIT_AND_HOLDOUT_ACTIVE:
                    print(f"  SECONDARY (descriptive only: small holdout, do not rank backbones on it)")
                    for _r in _run_rows:
                        print(
                            f"    CV mean AUC={_r['cv_mean_auc']:.3f} ± {_r['cv_std_auc']:.3f} | "
                            f"Clean-train final AUC={_r['final_refit_clean_train_auc_final']:.3f} | "
                            + (f"test set {_r['test_set']}: " if len(_run_rows) > 1 else "")
                            + f"holdout-test AUC={_r['final_test_auc']:.3f} (n={_r['n_test_cases']}) | "
                            f"train-test gap={_r['final_refit_clean_train_minus_test_auc_gap']:+.3f}"
                        )
                else:
                    print("  SECONDARY (final refit and holdout test not run)")
                    print(f"    CV mean AUC={row['cv_mean_auc']:.3f} ± {row['cv_std_auc']:.3f}")
                print("  " + "=" * 76)

                del final_model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # ── Per-backbone summary across outcome definitions ───────────────────
            _bb_rows = [r for r in all_results if r["backbone"] == backbone_name and r.get("e2e_mode") == e2e_mode]
            print_final_retrain_summary(
                _bb_rows,
                f"FINAL-RETRAIN SUMMARY (holdout test): backbone={backbone_name}, mode={e2e_mode}, all outcomes",
            )

    if OPTIMIZATION:
        restore_finetune_optimization_arm_settings()
        print("[OPTIMIZATION] Restored fine-tuning image/slice/augmentation settings after sweep.")

    # ── Fail-closed completeness check ──────────────────────────────────────────
    if OPTIMIZATION:
        _actual_run_keys = {
            (
                str(_row["opt_arm_id"]),
                str(_row["e2e_mode"]),
                str(_row["backbone"]),
                str(_row["outcome"]),
                str(_row["test_set"]),
            )
            for _row in all_results
        }
    else:
        _actual_run_keys = {
            (str(_row["e2e_mode"]), str(_row["backbone"]), str(_row["outcome"]), str(_row["test_set"]))
            for _row in all_results
        }

    _missing_run_keys = sorted(EXPECTED_RUN_KEYS - _actual_run_keys)
    _unexpected_run_keys = sorted(_actual_run_keys - EXPECTED_RUN_KEYS)
    if (
        len(all_results) != EXPECTED_RUN_COUNT
        or len(_actual_run_keys) != EXPECTED_RUN_COUNT
        or _missing_run_keys
        or _unexpected_run_keys
    ):
        raise RuntimeError(
            "[final-run][FATAL] Incomplete or inconsistent experiment matrix: "
            f"completed_rows={len(all_results)}, unique_completed={len(_actual_run_keys)}, "
            f"expected={EXPECTED_RUN_COUNT}, missing={_missing_run_keys}, "
            f"unexpected={_unexpected_run_keys}. "
            "The run is NOT complete."
        )

    # ── Overall summary ──────────────────────────────────────────────────────────
    print_final_retrain_summary(
        all_results,
        (
            "OVERALL OPTIMIZATION SUMMARY: all arms × CNNs × outcomes"
            if OPTIMIZATION
            else "OVERALL FINAL-RETRAIN SUMMARY (holdout test): all CNNs × all outcomes"
        ),
    )

    print("\n" + "="*78)
    _run_unit = ("arm × mode × backbone × outcome × test set" if OPTIMIZATION
                 else "mode × backbone × outcome × test set")
    print(
        f"All required {_run_unit} runs complete "
        f"({len(all_results)}/{EXPECTED_RUN_COUNT}); completeness assertion passed."
    )


In [ ]:
# Section 3, cell 17: Results export
if globals().get("RUN_CNN_FINETUNE_E2E", True):

    results_df = pd.DataFrame(all_results)
    results_df["segmentation_method"] = SEGMENTATION_ANALYSIS_METHOD

    # Format CV AUC as mean ± SD for each backbone and outcome.
    if len(results_df) > 0:
        results_df["cv_auc_mean_sd"] = [
            ("nan" if pd.isna(m) else f"{m:.3f} ± {(0.0 if pd.isna(s) else s):.3f}")
            for m, s in zip(results_df["cv_mean_auc"], results_df["cv_std_auc"])
        ]

    display_cols = [
        "backbone", "e2e_mode", "outcome", "test_set", "channel_policy",
        "selected_head_epochs", "selected_finetune_epochs",
        "small_data_finetune_profile", "inner_backbone_lr_candidates", "selected_backbone_lr",
        "lr_head_phase1", "lr_finetune_backbone", "lr_finetune_head",
        "finetune_warmup_epochs", "l2sp_lambda",
        "cv_mean_head_best_auc", "cv_std_head_best_auc",
        "cv_mean_finetune_best_auc", "cv_std_finetune_best_auc",
        "cv_mean_finetune_delta_vs_head", "cv_std_finetune_delta_vs_head",
        "paired_oof_head_auc", "paired_oof_finetune_auc", "paired_oof_finetune_delta",
        "paired_oof_delta_ci_lower", "paired_oof_delta_ci_upper",
        "paired_oof_p_finetune_gt_head", "cv_mean_finetune_grad_clip_fraction",
        "n_total_cases", "n_dev_cases", "n_test_cases", "n_folds", "n_repeats",
        "cv_auc_mean_sd",
        "cv_mean_auc", "cv_std_auc",
        "cv_mean_accuracy", "cv_mean_balacc", "cv_mean_f1",
        "cv_mean_sensitivity", "cv_mean_specificity",
        "oof_auc",
        "oof_DeLong_CI_lower", "oof_DeLong_CI_upper",
        "oof_AUC_with_DeLong_95CI",
        "oof_Jackknife_CI_lower", "oof_Jackknife_CI_upper",
        "oof_AUC_with_Jackknife_95CI",
        "oof_Bootstrap_CI_lower", "oof_Bootstrap_CI_upper",
        "oof_AUC_with_Bootstrap_95CI", "oof_Bootstrap_repeats",
        "final_refit_oof_model_family", "final_refit_oof_youden_threshold",
        "final_refit_oof_threshold_source",
        "final_test_auc",
        "final_test_DeLong_CI_lower", "final_test_DeLong_CI_upper",
        "final_test_AUC_with_DeLong_95CI",
        "final_test_Jackknife_CI_lower", "final_test_Jackknife_CI_upper",
        "final_test_AUC_with_Jackknife_95CI",
        "final_test_Jackknife_min_AUC", "final_test_Jackknife_max_AUC",
        "final_test_Bootstrap_CI_lower", "final_test_Bootstrap_CI_upper",
        "final_test_AUC_with_Bootstrap_95CI", "final_test_Bootstrap_repeats",
        "final_test_accuracy", "final_test_balacc", "final_test_f1", "final_test_precision",
        "final_test_sensitivity", "final_test_specificity", "final_test_mcc", "final_test_brier",
        "final_test_threshold", "final_test_threshold_source",
        "final_refit_clean_train_minus_test_auc_gap",
        "final_refit_head_clean_train_minus_test_auc_gap",
        "final_test_toprisk_percent", "final_test_toprisk_threshold",
        "final_test_toprisk_accuracy", "final_test_toprisk_balacc",
        "final_test_toprisk_sensitivity", "final_test_toprisk_specificity",
        "final_test_toprisk_f1", "final_test_toprisk_precision", "final_test_toprisk_mcc",
        "final_test_toprisk_brier", "final_test_toprisk_threshold_source",
    ]

    if "e2e_mode" in results_df.columns and "e2e_mode" not in display_cols:
        display_cols.insert(1, "e2e_mode")

    # In OPTIMIZATION mode the columns that define the arm come first. The final_test_*
    # columns are filled only when the run scored the holdout.
    if globals().get("OPTIMIZATION", False):
        for _c in reversed([
            "opt_arm_id", "opt_is_reference", "opt_input_size", "opt_n_slices",
            "opt_augmentation", "tumor_context_mode", "roi_only_background_value"
        ]):
            if _c in results_df.columns and _c not in display_cols:
                display_cols.insert(0, _c)
        _sort_col = "final_test_auc"
    else:
        _sort_col = "final_test_auc"

    # The pooled development OOF threshold metrics belong on screen, not only in the CSV:
    # on 150 patients they are the reliable threshold-dependent numbers, unlike the
    # holdout equivalents on 28.
    for _c in ["oof_accuracy", "oof_balacc", "oof_f1", "oof_sensitivity",
               "oof_specificity", "oof_mcc", "oof_brier"]:
        if _c in results_df.columns and _c not in display_cols:
            display_cols.insert(display_cols.index("oof_auc") + 1, _c)

    display_cols = [c for c in display_cols if c in results_df.columns]

    # Without a final refit there is no holdout AUC to sort by. The table is then sorted
    # by pooled development out-of-fold AUC and the empty holdout columns are not printed.
    _holdout_scored = bool(globals().get("FINAL_REFIT_AND_HOLDOUT_ACTIVE", True))
    if not _holdout_scored:
        _sort_col = "oof_auc"
        display_cols = [
            c for c in display_cols
            if not c.startswith("final_test_") and not c.endswith("_minus_test_auc_gap")
        ]

    print("\n── Final Results ──")
    if not _holdout_scored:
        print("[note] The final refit and the holdout test were not run. "
              "Rows are sorted by development out-of-fold AUC.")
    if len(results_df) == 0:
        print("No completed runs.")
    else:
        if _holdout_scored:
            # One block of rows per test set, best holdout AUC first within each.
            print(order_by_test_set(results_df, _sort_col)[display_cols].to_string(index=False))
        else:
            # An OPTIMIZATION run that keeps the holdout unseen shows every run once. The
            # CSV below keeps one row per test set.
            _dev_table = rows_for_display(order_by_test_set(results_df, _sort_col))
            if "test_set" not in _dev_table.columns:
                display_cols = [c for c in display_cols if c not in ("test_set", "n_test_cases")]
            print(_dev_table[display_cols].to_string(index=False))

    out_csv = Path(OUT_DIR) / (
        "optimization_sweep_results.csv" if globals().get("OPTIMIZATION", False)
        else "cnn_finetuning_results_gray3_all_supported_backbones_holdout.csv"
    )
    # Save the FULL results dataframe, not merely the on-screen subset. This preserves
    # every arm-defining field (ID/reference flag/input/slices/augmentation/context/background)
    # plus all protocol and metric columns in the CSV.
    out_df = results_df
    if len(results_df) > 0:
        out_df = order_by_test_set(results_df, _sort_col)
    out_df.to_csv(out_csv, index=False)
    print(f"\nResults CSV: {out_csv}")

    # Manifest of the output location and filenames.
    manifest_path = Path(OUT_DIR) / "OUTPUT_MANIFEST.txt"
    manifest_lines = [
        f"OUT_ROOT={OUT_ROOT}",
        f"OUT_DIR={OUT_DIR}",
        f"summary_csv={out_csv.name}",
        "Per completed run:",
        "  *_final_retrain_history.csv",
        "  *_cv_fold_metrics.csv",
        "  *_oof_dev_predictions.csv",
        "  *_holdout_test_predictions.csv   (with two test sets: one file per test set, *_holdout_test_predictions__<test set>.csv)",
        "  *_final_retrain_model.pt",
    ]
    if not _holdout_scored:
        manifest_lines.append(
            "Final refit and holdout test were not run: the *_final_retrain_history.csv, "
            "*_holdout_test_predictions.csv and *_final_retrain_model.pt files were not written."
        )
    manifest_path.write_text("\n".join(manifest_lines) + "\n", encoding="utf-8")
    print(f"Output manifest: {manifest_path}")

    # OUT_DIR is already on Google Drive; no temporary-runtime copy is used.
    print(f"Persistent Google Drive results folder: {OUT_DIR}")
    print(f"[SECTION 3] Completed {len(results_df)} backbone × outcome × test-set rows.")

    # Hand the result table to the final overview, as Sections 1 and 2 do.
    try:
        RESULTS_REGISTRY['segment4_finetune'] = results_df.copy()
        print('[SECTION 3] final overview contains', len(results_df), 'backbone x outcome x test-set rows.')
    except Exception as _e:
        print('[SECTION 3] result table is unavailable to the final overview:', repr(_e))


## Section 4, cell 1 - Optimization comparison summary for enabled workflows

Runs only when `OPTIMIZATION = True`. Every optimization arm/representation is compared through its **paired**
development OOF AUC delta against the reference arm on identical patients/folds. No holdout test
set is scored for any arm unless `OPTIMIZATION_EVALUATE_HOLDOUT` is set to `True` in
Section 0, cell 3. In that case every arm is also refitted on the complete development cohort and
evaluated on each holdout test set. The summary then prints those holdout-test metrics beside the comparison.


In [ ]:
# Section 4, cell 2: Build optimization comparison tables
if globals().get("OPTIMIZATION", False):
    from pathlib import Path
    import numpy as np
    import pandas as pd
    # Optimization comparison summary for all enabled workflows
    #
    # Development OOF probabilities are compared PAIRED, patient by patient, against
    # the reference representation within the same backbone/outcome/model signature.
    # The verdict comes from the development OOF results only. Holdout/test measures are
    # printed beside it only when the run scored the holdout
    # (OPTIMIZATION_EVALUATE_HOLDOUT set to True in Section 0, cell 3).

    def _summarize_optimization_store(store, workflow, group_key_fn, group_label_fn):
        if not store:
            print(f"[OPTIMIZATION][{workflow}] No stored arm results.")
            return pd.DataFrame()

        groups = {}
        for key, entry in store.items():
            groups.setdefault(group_key_fn(key, entry), []).append((key, entry))

        rows = []
        for group_key, members in sorted(groups.items(), key=lambda kv: str(kv[0])):
            _refs = [x for x in members if bool(x[1].get("is_reference"))]
            if len(_refs) != 1:
                _member_ids = [str(x[0][0]) for x in members]
                _ref_ids = [str(x[0][0]) for x in _refs]
                raise RuntimeError(
                    f"[OPTIMIZATION][{workflow}] Expected exactly one reference arm for "
                    f"group {group_key!r}, found {len(_refs)}. "
                    f"Reference IDs={_ref_ids}; completed member IDs={_member_ids}. "
                    "Refusing to substitute a candidate arm as the reference."
                )
            ref_key, ref = _refs[0]
            print("\n" + "=" * 110)
            print(f"OPTIMIZATION: {workflow} | {group_label_fn(group_key, ref)}")
            print(f"Reference arm: {ref_key[0]} | pooled development OOF AUC={float(ref['oof_auc']):.3f}")
            print("Paired deltas are candidate minus reference on the SAME development patients.")
            print("=" * 110)

            for key, entry in members:
                pids = sorted(set(entry["probs"]) & set(ref["probs"]), key=patient_id_sort_key)
                if not pids:
                    print(f"  [warning] {key[0]} has no patients in common with the reference; skipped.")
                    continue
                if any(entry["labels"][p] != ref["labels"][p] for p in pids):
                    raise RuntimeError(f"{workflow} | {key[0]}: labels disagree with reference arm")
                y = [entry["labels"][p] for p in pids]
                ref_scores = [ref["probs"][p] for p in pids]
                cand_scores = [entry["probs"][p] for p in pids]
                is_ref = (key == ref_key)
                if is_ref:
                    delta = lo = hi = 0.0
                    p_better = float("nan")
                else:
                    cmp = optimization_paired_bootstrap_auc_delta(y, ref_scores, cand_scores)
                    delta = float(cmp["delta"])
                    lo, hi = float(cmp["ci_lower"]), float(cmp["ci_upper"])
                    p_better = float(cmp["p_candidate_gt_reference"])

                if is_ref:
                    verdict = "reference"
                elif delta >= float(OPTIMIZATION_MIN_DELTA) and lo > 0.0:
                    verdict = "ADOPT"
                elif delta <= -float(OPTIMIZATION_MIN_DELTA) and hi < 0.0:
                    verdict = "reject (worse)"
                else:
                    verdict = "no change"

                row = {
                    "workflow": workflow,
                    "arm_id": key[0],
                    "is_reference": bool(entry.get("is_reference")),
                    "input_size": entry.get("input_size", np.nan),
                    "n_slices": entry.get("n_slices", np.nan),
                    "augmentation": entry.get("augmentation", ""),
                    "context_mode": entry.get("context_mode", ""),
                    "roi_background": (
                        entry.get("roi_background", np.nan)
                        if str(entry.get("context_mode", "")).upper() == "ROI_ONLY" else np.nan
                    ),
                    "oof_auc": entry.get("oof_auc", np.nan),
                    "paired_delta_vs_reference": delta,
                    "delta_ci_lower": lo,
                    "delta_ci_upper": hi,
                    "p_better_than_reference": p_better,
                    "verdict": verdict,
                    "n_patients": len(pids),
                }
                # Keep group identity columns explicit in the CSV.
                if workflow in ("frozen", "finetune"):
                    row.update({
                        "backbone": key[1],
                        "outcome": key[2],
                    })
                    if workflow == "finetune":
                        row["paired_oof_finetune_delta"] = entry.get("paired_oof_finetune_delta", np.nan)
                elif workflow == "efficient_ML":
                    row.update({
                        "backbone": key[1],
                        "outcome": key[2],
                        "channel_policy": key[3],
                        "model_id": key[4],
                        "classifier": entry.get("classifier", ""),
                    })
                # Holdout results: one row per test set. The development-side comparison
                # (paired delta and verdict) is the same on each of them. A run that did
                # not score the holdout has no holdout results, so each arm gets one row
                # without holdout columns.
                if holdout_evaluation_enabled():
                    _by_test_set = entry.get("test_results_by_test_set") or {
                        "": entry.get("test_results", {})
                    }
                else:
                    _by_test_set = {"": {}}
                for _test_set_name, _test_results in _by_test_set.items():
                    _row = dict(row)
                    if _test_set_name:
                        _row["test_set"] = _test_set_name
                    _row.update(_test_results)
                    rows.append(_row)

        return pd.DataFrame(rows)

    _opt_tables = []

    _frozen_df = _summarize_optimization_store(
        globals().get("OPTIMIZATION_FROZEN_OOF_STORE", {}),
        "frozen",
        lambda k, e: (k[1], k[2]),
        lambda g, e: f"backbone={g[0]} | outcome={g[1]}",
    )
    if len(_frozen_df):
        _opt_tables.append(_frozen_df)
        _out = Path(PART1_OUT_DIR) / "optimization_sweep_paired_comparison_frozen_e2e.csv"
        _frozen_df.to_csv(_out, index=False)
        RESULTS_REGISTRY["optimization_frozen_e2e"] = _frozen_df.copy()
        print(f"[OPTIMIZATION][frozen] comparison CSV: {_out}")

    _ml_df = _summarize_optimization_store(
        globals().get("OPTIMIZATION_ML_OOF_STORE", {}),
        "efficient_ML",
        lambda k, e: (k[1], k[2], k[3], k[4]),
        lambda g, e: (
            f"backbone={g[0]} | outcome={g[1]} | channel={g[2]} | model={g[3]}"
        ),
    )
    if len(_ml_df):
        _opt_tables.append(_ml_df)
        _out = Path(PART3_OUT_DIR) / "optimization_sweep_paired_comparison_cnn_ml.csv"
        _ml_df.to_csv(_out, index=False)
        RESULTS_REGISTRY["optimization_cnn_ml"] = _ml_df.copy()
        print(f"[OPTIMIZATION][efficient_ML] comparison CSV: {_out}")

    _ft_df = _summarize_optimization_store(
        globals().get("OPTIMIZATION_OOF_STORE", {}),
        "finetune",
        lambda k, e: (k[1], k[2]),
        lambda g, e: f"backbone={g[0]} | outcome={g[1]}",
    )
    if len(_ft_df):
        _opt_tables.append(_ft_df)
        _out = Path(FINETUNE_OUT_DIR) / "optimization_sweep_paired_comparison_finetuned_e2e.csv"
        _ft_df.to_csv(_out, index=False)
        RESULTS_REGISTRY["optimization_finetuned_e2e"] = _ft_df.copy()
        print(f"[OPTIMIZATION][finetune] comparison CSV: {_out}")

    if _opt_tables:
        _all_opt_df = pd.concat(_opt_tables, ignore_index=True, sort=False)
        RESULTS_REGISTRY["optimization_all_workflows"] = _all_opt_df.copy()
        _all_out = Path(COMBINED_OVERVIEW_DIR) / "optimization_sweep_paired_comparison_all_workflows.csv"
        _all_opt_df.to_csv(_all_out, index=False)

        print("\n" + "=" * 110)
        print("OPTIMIZATION: ALL WORKFLOWS, ALL ARMS")
        print("=" * 110)
        _base_cols = [
            c for c in [
                "workflow", "backbone", "outcome", "test_set", "channel_policy", "model_id",
                "classifier", "arm_id", "is_reference", "input_size", "n_slices", "augmentation",
                "context_mode", "roi_background",
                "oof_auc", "paired_delta_vs_reference", "delta_ci_lower", "delta_ci_upper",
                "p_better_than_reference", "verdict", "n_patients",
            ] if c in _all_opt_df.columns
        ]
        # Research-facing display: append EVERY holdout/test measure carried by any workflow.
        # The list is empty when the run did not score the holdout.
        _all_test_cols = [
            c for c in _all_opt_df.columns
            if (str(c).startswith("final_test_") or str(c).startswith("test_"))
            and c not in _base_cols
        ]
        _core = _base_cols + _all_test_cols
        print(_all_opt_df[_core].to_string(index=False))
        print(f"[OPTIMIZATION] Holdout/test columns shown: {len(_all_test_cols)}")
        print(f"\nCombined optimization CSV: {_all_out}")
        print(f"ADOPTION RULE: paired development-OOF delta >= {OPTIMIZATION_MIN_DELTA:+.3f} "
              "and bootstrap CI excludes zero.")
        print("efficient_ML note: augmentation is NOT_APPLICABLE because the CNN feature extractor is frozen; "
              "its optimization arms therefore de-duplicate augmentation and sweep size x slice count.")
        if "test_set" in _all_opt_df.columns and _all_opt_df["test_set"].nunique() > 1:
            print("TEST SETS: each arm has one row per holdout test set (column test_set). "
                  "The paired development-OOF columns and the verdict are identical on those rows.")
        if holdout_evaluation_enabled():
            print("TEST RESULTS: all available holdout metrics are shown for every arm. "
                  "The printed ADOPT/reject verdict itself is still computed from paired development OOF results.")
        else:
            print("TEST RESULTS: none. No holdout test set was scored for any arm "
                  "(OPTIMIZATION_EVALUATE_HOLDOUT is False), so the holdout stays unseen for the final run "
                  "with the chosen settings and OPTIMIZATION set to False.")
    else:
        print("[OPTIMIZATION] No completed optimization results were available from any workflow.")
else:
    print("[OPTIMIZATION] Off: no sweep summary. Set OPTIMIZATION = True in Section 0 to sweep.")


## Section 5, cell 1 - Final overview

Runs after the workflow sections and includes only the workflows enabled by the master switches in Section 0.

The `workflow` column tags each row as `frozen` (Section 1), `efficient_ML` (Section 2) or `finetune` (Section 3).
The `model` column reads `backbone|frozen` or `backbone|finetune` for the two end-to-end CNN workflows and
`backbone_efficient_ML|classifier` for Section 2, for example `radimagenet_resnet50|finetune` and
`radimagenet_resnet50_efficient_ML|NAIVE_BAYES`.

With two holdout test sets (two folders in `TEST_ROOT`), the final table is printed once per test set and each
table is saved to its own file, `combined_pipeline_overview__<test set>.csv`. The file
`combined_pipeline_overview.csv` then holds all rows, with a `test_set` column. With one test set there is one
table and no `test_set` column.

Section 5 first saves results for the current mask method. Run through Section 6 to
refresh the main `COMBINED_OVERVIEW_DIR/combined_pipeline_overview.csv` and its
per-test-set files with **original plus every enabled generated-mask analysis**.
The `segmentation_method` column distinguishes `original`, `sam2_box_per_slice`
(CAS2/SAM2) and `nninteractive_lasso`. The `ANALYZE_SAM2_MASKS` and
`ANALYZE_NNINTERACTIVE_MASKS` switches select which additional results are included.

The label file is not part of the names, because one run analyses one label file (`LABELS_CSV` in Section 0).
A `channel_policy` column is added only when a backbone was run with more than one channel policy.

After an optimization comparison that did not score the holdout (`OPTIMIZATION_EVALUATE_HOLDOUT` left at `False`),
the holdout columns of this table are empty.

In [ ]:
# Section 5, cell 2: Combined results table
# One row per model. AUC is threshold-free; 95% intervals are given by DeLong,
# delete-one jackknife and stratified patient-level bootstrap. Threshold-dependent
# metrics are reported at the development out-of-fold Youden cut-off (OOF_*), at the
# in-sample cut-off where available (INSAMPLE_*) and for the top-risk quartile.
import os
import re
import numpy as np
import pandas as pd

print("=" * 120)
print("COMBINED PIPELINE: FINAL PUBLICATION-METRICS OVERVIEW")
print(
    "Workflow switches: "
    f"frozen={globals().get('RUN_FROZEN_CNN_E2E', True)} | "
    f"efficient_ML={globals().get('RUN_CNN_LR', True)} | "
    f"finetune={globals().get('RUN_CNN_FINETUNE_E2E', True)}"
)
print("=" * 120)
if not holdout_evaluation_enabled():
    print(
        "[FINAL OVERVIEW] No holdout test set was scored in this run "
        "(OPTIMIZATION mode with OPTIMIZATION_EVALUATE_HOLDOUT set to False). "
        "The holdout columns below are empty. The development comparison is in Section 4."
    )


# Workflow names used in the combined table, one per workflow section.
WORKFLOW_FROZEN = "frozen"              # Section 1: end-to-end CNN, backbone frozen
WORKFLOW_EFFICIENT_ML = "efficient_ML"  # Section 2: frozen-CNN features + classical ML
WORKFLOW_FINETUNE = "finetune"          # Section 3: end-to-end CNN, backbone fine-tuned


def _text(value):
    """A table value as clean text; missing values become an empty string."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return "" if text.lower() in {"nan", "none"} else text


def _parts_from_feature_file(value):
    """Backbone, channel policy and outcome read from a Section 2 feature-file name.

    The name is cnn_features_patient_mean__<backbone>__<channel policy>__label_<outcome>.csv,
    with the optimization arm in front of the backbone in OPTIMIZATION runs. This is only a
    fallback for result rows that do not carry these values in their own columns.
    """
    stem = os.path.splitext(os.path.basename(_text(value)))[0]
    stem = re.sub(r"^cnn_features_patient_mean__", "", stem, flags=re.IGNORECASE)
    outcome = ""
    match = re.search(r"__label_(.+)$", stem, flags=re.IGNORECASE)
    if match:
        outcome = match.group(1)
        stem = stem[:match.start()]
    pieces = [p for p in stem.split("__") if p]
    policy = pieces.pop() if len(pieces) >= 2 else ""
    backbone = pieces[-1] if pieces else stem
    return backbone, policy, outcome


# Holdout test sets in the order given in TEST_ROOT (Section 0).
_TEST_SET_ORDER = list(globals().get("TEST_SET_NAMES", []))


def _by_test_set_then_auc(df, auc_col):
    """Rows grouped by test set in TEST_ROOT order, best AUC first within each test set."""
    out = df
    if auc_col in out.columns:
        out = out.sort_values(auc_col, ascending=False, na_position="last", kind="mergesort")
    if "test_set" in out.columns:
        _rank = {name: i for i, name in enumerate(_TEST_SET_ORDER)}
        out = out.sort_values(
            "test_set", key=lambda s: s.map(lambda v: _rank.get(v, len(_rank))), kind="mergesort"
        )
    return out


def _safe_ci_text(auc, lower, upper):
    try:
        return format_auc_with_ci(float(auc), float(lower), float(upper))
    except Exception:
        return "nan"


# ── End-to-end CNN results, Sections 1 and 3 ──────────────────────────────
# Both produce identically named result columns, so they share one column list and
# one display helper.
_CNN_OVERVIEW_COLS = [
    "backbone", "e2e_mode", "outcome", "test_set", "channel_policy",
    "optimization", "opt_arm_id", "opt_is_reference",
    "opt_input_size", "opt_n_slices", "opt_augmentation",
    "tumor_context_mode", "roi_only_background_value",
    "final_test_auc",
    "final_test_DeLong_CI_lower", "final_test_DeLong_CI_upper",
    "final_test_AUC_with_DeLong_95CI",
    "final_test_Jackknife_CI_lower", "final_test_Jackknife_CI_upper",
    "final_test_AUC_with_Jackknife_95CI",
    "final_test_Jackknife_min_AUC", "final_test_Jackknife_max_AUC",
    "final_test_Bootstrap_CI_lower", "final_test_Bootstrap_CI_upper",
    "final_test_AUC_with_Bootstrap_95CI", "final_test_Bootstrap_repeats",
    "final_test_accuracy", "final_test_balacc",
    "final_test_sensitivity", "final_test_specificity",
    "final_test_f1", "final_test_mcc", "final_test_brier",
    "final_test_threshold", "final_test_threshold_source",
    "final_test_toprisk_percent", "final_test_toprisk_threshold",
    "final_test_toprisk_accuracy", "final_test_toprisk_balacc",
    "final_test_toprisk_sensitivity", "final_test_toprisk_specificity",
    "final_test_toprisk_f1", "final_test_toprisk_mcc",
    "final_test_toprisk_threshold_source",
    "n_test_cases",
]


def _cnn_section_overview(df):
    """Trim a CNN result table to the overview columns: one block of rows per test set,
    best holdout AUC first within each."""
    _cols = [c for c in _CNN_OVERVIEW_COLS if c in df.columns]
    return _by_test_set_then_auc(df[_cols].copy(), "final_test_auc")


seg1 = RESULTS_REGISTRY.get("segment1_cnn")
seg4 = RESULTS_REGISTRY.get("segment4_finetune")
seg1_overview = pd.DataFrame()
seg4_overview = pd.DataFrame()

if seg1 is not None and len(seg1):
    seg1_overview = _cnn_section_overview(seg1)
    print("\n[SECTION 1 | frozen] END-TO-END CNN CLASSIFICATION, FROZEN BACKBONE")
    print(seg1_overview.to_string(index=False))
else:
    print("\n[SECTION 1 | frozen] END-TO-END CNN CLASSIFICATION, FROZEN BACKBONE: no results")

if seg4 is not None and len(seg4):
    seg4_overview = _cnn_section_overview(seg4)
    print("\n[SECTION 3 | finetune] END-TO-END CNN CLASSIFICATION, FINE-TUNED BACKBONE")
    print(seg4_overview.to_string(index=False))
else:
    print("\n[SECTION 3 | finetune] END-TO-END CNN CLASSIFICATION, FINE-TUNED BACKBONE: no results")


# ── Section 2 status: which feature files were produced ──────────────────
seg2 = RESULTS_REGISTRY.get("segment2_features")
print("\n[SECTION 2 | efficient_ML] FROZEN-CNN FEATURE EXTRACTION STATUS")
if seg2 is not None and len(seg2):
    _cols = [
        c for c in [
            "cnn_backbone", "cnn_channel_policy", "outcome",
            "optimization", "opt_arm_id", "opt_is_reference",
            "opt_input_size", "opt_n_slices", "opt_augmentation",
            "status", "n_patients", "cnn_feature_dim",
        ]
        if c in seg2.columns
    ]
    print(seg2[_cols].to_string(index=False))
else:
    print("  (no feature-extraction summary)")


# ── Section 2 results: machine-learning models on the extracted features ─
seg3 = RESULTS_REGISTRY.get("segment3_ml")
print("\n[SECTION 2 | efficient_ML] CLASSICAL ML ON FROZEN-CNN FEATURES: HOLDOUT PUBLICATION METRICS")
if seg3 is not None and len(seg3):
    _cols = [
        c for c in [
            "feature_file", "cnn_backbone", "cnn_channel_policy", "outcome", "test_set",
            "optimization", "opt_arm_id", "opt_is_reference",
            "opt_input_size", "opt_n_slices", "opt_augmentation",
            "classifier", "val_oof_AUC", "val_AUC_mean", "val_AUC_std",
            "test_AUC",
            "test_DeLong_CI_lower", "test_DeLong_CI_upper",
            "test_AUC_with_DeLong_95CI",
            "test_Jackknife_CI_lower", "test_Jackknife_CI_upper",
            "test_AUC_with_Jackknife_95CI",
            "test_Jackknife_min_AUC", "test_Jackknife_max_AUC",
            "test_Bootstrap_CI_lower", "test_Bootstrap_CI_upper",
            "test_AUC_with_Bootstrap_95CI", "test_Bootstrap_repeats",
            "test_Brier",
            "test_oof_threshold",
            "test_oof_accuracy", "test_oof_balanced_accuracy",
            "test_oof_sensitivity", "test_oof_specificity",
            "test_oof_F1", "test_oof_MCC", "test_oof_YOUDEN",
            "test_oof_threshold_source",
            "test_insample_threshold",
            "test_insample_accuracy", "test_insample_balanced_accuracy",
            "test_insample_sensitivity", "test_insample_specificity",
            "test_insample_F1", "test_insample_MCC", "test_insample_YOUDEN",
            "test_insample_threshold_source",
            "toprisk_percent", "toprisk_threshold",
            "test_toprisk_accuracy", "test_toprisk_balanced_accuracy",
            "test_toprisk_sensitivity", "test_toprisk_specificity",
            "test_toprisk_F1", "test_toprisk_MCC", "toprisk_threshold_source",
            "n_test_patients",
        ]
        if c in seg3.columns
    ]
    print(_by_test_set_then_auc(seg3, "test_AUC")[_cols].to_string(index=False))
else:
    print("  (no efficient_ML results)")

seg3_errors = RESULTS_REGISTRY.get("segment3_ml_errors")
if seg3_errors is not None and len(seg3_errors):
    print("\n[SECTION 2 | efficient_ML] FEATURE FILES WITH ERRORS")
    _cols = [
        c for c in ["feature_file", "status", "error"]
        if c in seg3_errors.columns
    ]
    print(seg3_errors[_cols].to_string(index=False))


# ── Build one combined CSV with consistent column names ──────────────────
publication_rows = []

# One loop turns both CNN tables into rows of the combined table. The model name is
# backbone|workflow. The outcome definition and the channel policy are kept in their
# own columns, not in the name.
for _cnn_key, _workflow in (("segment1_cnn", WORKFLOW_FROZEN),
                            ("segment4_finetune", WORKFLOW_FINETUNE)):
    _cnn_df = RESULTS_REGISTRY.get(_cnn_key)
    if _cnn_df is None or not len(_cnn_df) or "final_test_auc" not in _cnn_df.columns:
        continue
    for _, r in _cnn_df.iterrows():
        auc = r.get("final_test_auc", np.nan)
        delong_lower = r.get("final_test_DeLong_CI_lower", np.nan)
        delong_upper = r.get("final_test_DeLong_CI_upper", np.nan)
        jackknife_lower = r.get("final_test_Jackknife_CI_lower", np.nan)
        jackknife_upper = r.get("final_test_Jackknife_CI_upper", np.nan)
        _backbone = _text(r.get("backbone", ""))
        model_name = f"{_backbone}|{_workflow}"

        publication_rows.append({
            "segmentation_method": SEGMENTATION_ANALYSIS_METHOD,
            "workflow": _workflow,
            "model": model_name,
            "test_set": _text(r.get("test_set", "")),
            "outcome": _text(r.get("outcome", "")),
            "channel_policy": _text(r.get("channel_policy", "")),
            "_backbone": _backbone,
            "optimization": r.get("optimization", False),
            "opt_arm_id": r.get("opt_arm_id", ""),
            "opt_is_reference": r.get("opt_is_reference", False),
            "opt_input_size": r.get("opt_input_size", np.nan),
            "opt_n_slices": r.get("opt_n_slices", np.nan),
            "opt_augmentation": r.get("opt_augmentation", ""),
            "tumor_context_mode": r.get("tumor_context_mode", ""),
            "roi_only_background_value": r.get("roi_only_background_value", np.nan),
            "AUC": auc,
            "DeLong_CI_lower": delong_lower,
            "DeLong_CI_upper": delong_upper,
            "AUC_with_DeLong_95CI": r.get(
                "final_test_AUC_with_DeLong_95CI",
                _safe_ci_text(auc, delong_lower, delong_upper),
            ),
            "Jackknife_CI_lower": jackknife_lower,
            "Jackknife_CI_upper": jackknife_upper,
            "AUC_with_Jackknife_95CI": r.get(
                "final_test_AUC_with_Jackknife_95CI",
                _safe_ci_text(auc, jackknife_lower, jackknife_upper),
            ),
            "Jackknife_min_AUC": r.get("final_test_Jackknife_min_AUC", np.nan),
            "Jackknife_max_AUC": r.get("final_test_Jackknife_max_AUC", np.nan),
            "Bootstrap_CI_lower": r.get("final_test_Bootstrap_CI_lower", np.nan),
            "Bootstrap_CI_upper": r.get("final_test_Bootstrap_CI_upper", np.nan),
            "AUC_with_Bootstrap_95CI": r.get(
                "final_test_AUC_with_Bootstrap_95CI",
                _safe_ci_text(
                    auc,
                    r.get("final_test_Bootstrap_CI_lower", np.nan),
                    r.get("final_test_Bootstrap_CI_upper", np.nan),
                ),
            ),
            "Bootstrap_repeats": r.get("final_test_Bootstrap_repeats", np.nan),
            "Brier_score": r.get("final_test_brier", np.nan),
            "n_test_patients": r.get("n_test_cases", np.nan),

            # Both CNN sections report the development OOF cut-off; neither computes an
            # in-sample cut-off, so INSAMPLE_* stays blank.
            "OOF_threshold": r.get("final_test_threshold", np.nan),
            "OOF_accuracy": r.get("final_test_accuracy", np.nan),
            "OOF_balanced_accuracy": r.get("final_test_balacc", np.nan),
            "OOF_sensitivity": r.get("final_test_sensitivity", np.nan),
            "OOF_specificity": r.get("final_test_specificity", np.nan),
            "OOF_F1_score": r.get("final_test_f1", np.nan),
            "OOF_MCC": r.get("final_test_mcc", np.nan),
            "OOF_YOUDEN": np.nan,
            "OOF_threshold_source": r.get(
                "final_test_threshold_source",
                "development pooled OOF Youden threshold applied unchanged to holdout",
            ),
            "INSAMPLE_threshold": np.nan,
            "INSAMPLE_accuracy": np.nan,
            "INSAMPLE_balanced_accuracy": np.nan,
            "INSAMPLE_sensitivity": np.nan,
            "INSAMPLE_specificity": np.nan,
            "INSAMPLE_F1_score": np.nan,
            "INSAMPLE_MCC": np.nan,
            "INSAMPLE_YOUDEN": np.nan,
            "INSAMPLE_threshold_source": "",
            "toprisk_percent": r.get("final_test_toprisk_percent", np.nan),
            "toprisk_threshold": r.get("final_test_toprisk_threshold", np.nan),
            "toprisk_accuracy": r.get("final_test_toprisk_accuracy", np.nan),
            "toprisk_balanced_accuracy": r.get("final_test_toprisk_balacc", np.nan),
            "toprisk_sensitivity": r.get("final_test_toprisk_sensitivity", np.nan),
            "toprisk_specificity": r.get("final_test_toprisk_specificity", np.nan),
            "toprisk_F1_score": r.get("final_test_toprisk_f1", np.nan),
            "toprisk_MCC": r.get("final_test_toprisk_mcc", np.nan),
            "toprisk_threshold_source": r.get("final_test_toprisk_threshold_source", ""),
            "DeLong_CI_method": "single-model DeLong variance; normal 95% CI",
            "Jackknife_CI_method": "delete-one patient jackknife; normal 95% CI",
            "Bootstrap_CI_method": "stratified patient-level nonparametric bootstrap; percentile 95% CI",
        })

if seg3 is not None and len(seg3) and "test_AUC" in seg3.columns:
    for _, r in seg3.iterrows():
        auc = r.get("test_AUC", np.nan)
        delong_lower = r.get("test_DeLong_CI_lower", np.nan)
        delong_upper = r.get("test_DeLong_CI_upper", np.nan)
        jackknife_lower = r.get("test_Jackknife_CI_lower", np.nan)
        jackknife_upper = r.get("test_Jackknife_CI_upper", np.nan)
        # The model name is backbone_efficient_ML|classifier. Backbone, channel policy and
        # outcome come from the row itself; the feature-file name is only a fallback.
        _file_backbone, _file_policy, _file_outcome = _parts_from_feature_file(
            r.get("feature_file", "")
        )
        _backbone = _text(r.get("cnn_backbone", "")) or _file_backbone
        model_name = f"{_backbone}_{WORKFLOW_EFFICIENT_ML}|{_text(r.get('classifier', ''))}"

        publication_rows.append({
            "segmentation_method": SEGMENTATION_ANALYSIS_METHOD,
            "workflow": WORKFLOW_EFFICIENT_ML,
            "model": model_name,
            "test_set": _text(r.get("test_set", "")),
            "outcome": _text(r.get("outcome", "")) or _file_outcome,
            "channel_policy": _text(r.get("cnn_channel_policy", "")) or _file_policy,
            "_backbone": _backbone,
            "optimization": r.get("optimization", False),
            "opt_arm_id": r.get("opt_arm_id", ""),
            "opt_is_reference": r.get("opt_is_reference", False),
            "opt_input_size": r.get("opt_input_size", np.nan),
            "opt_n_slices": r.get("opt_n_slices", np.nan),
            "opt_augmentation": r.get("opt_augmentation", "NOT_APPLICABLE"),
            "tumor_context_mode": r.get("tumor_context_mode", ""),
            "roi_only_background_value": r.get("roi_only_background_value", np.nan),
            "AUC": auc,
            "DeLong_CI_lower": delong_lower,
            "DeLong_CI_upper": delong_upper,
            "AUC_with_DeLong_95CI": r.get(
                "test_AUC_with_DeLong_95CI",
                _safe_ci_text(auc, delong_lower, delong_upper),
            ),
            "Jackknife_CI_lower": jackknife_lower,
            "Jackknife_CI_upper": jackknife_upper,
            "AUC_with_Jackknife_95CI": r.get(
                "test_AUC_with_Jackknife_95CI",
                _safe_ci_text(auc, jackknife_lower, jackknife_upper),
            ),
            "Jackknife_min_AUC": r.get("test_Jackknife_min_AUC", np.nan),
            "Jackknife_max_AUC": r.get("test_Jackknife_max_AUC", np.nan),
            "Bootstrap_CI_lower": r.get("test_Bootstrap_CI_lower", np.nan),
            "Bootstrap_CI_upper": r.get("test_Bootstrap_CI_upper", np.nan),
            "AUC_with_Bootstrap_95CI": r.get(
                "test_AUC_with_Bootstrap_95CI",
                _safe_ci_text(
                    auc,
                    r.get("test_Bootstrap_CI_lower", np.nan),
                    r.get("test_Bootstrap_CI_upper", np.nan),
                ),
            ),
            "Bootstrap_repeats": r.get("test_Bootstrap_repeats", np.nan),
            # Brier score: how accurate the predicted probabilities are (needs no cut-off).
            "Brier_score": r.get("test_Brier", np.nan),
            "n_test_patients": r.get("n_test_patients", np.nan),
            # Cut-off metrics using the development-data ("OOF") cut-off.
            "OOF_threshold": r.get("test_oof_threshold", r.get("classification_threshold", np.nan)),
            "OOF_accuracy": r.get("test_oof_accuracy", r.get("test_accuracy", np.nan)),
            "OOF_balanced_accuracy": r.get("test_oof_balanced_accuracy", r.get("test_balanced_accuracy", np.nan)),
            "OOF_sensitivity": r.get("test_oof_sensitivity", r.get("test_sensitivity", np.nan)),
            "OOF_specificity": r.get("test_oof_specificity", r.get("test_specificity", np.nan)),
            "OOF_F1_score": r.get("test_oof_F1", r.get("test_F1", np.nan)),
            "OOF_MCC": r.get("test_oof_MCC", r.get("test_MCC_publication", np.nan)),
            "OOF_YOUDEN": r.get("test_oof_YOUDEN", np.nan),
            "OOF_threshold_source": r.get(
                "test_oof_threshold_source",
                r.get("threshold_source", "development pooled OOF Youden threshold applied unchanged to holdout"),
            ),

            # Cut-off metrics using the alternative cut-off re-chosen on the full development set ("in-sample").
            "INSAMPLE_threshold": r.get("test_insample_threshold", np.nan),
            "INSAMPLE_accuracy": r.get("test_insample_accuracy", np.nan),
            "INSAMPLE_balanced_accuracy": r.get("test_insample_balanced_accuracy", np.nan),
            "INSAMPLE_sensitivity": r.get("test_insample_sensitivity", np.nan),
            "INSAMPLE_specificity": r.get("test_insample_specificity", np.nan),
            "INSAMPLE_F1_score": r.get("test_insample_F1", np.nan),
            "INSAMPLE_MCC": r.get("test_insample_MCC", np.nan),
            "INSAMPLE_YOUDEN": r.get("test_insample_YOUDEN", np.nan),
            "INSAMPLE_threshold_source": r.get("test_insample_threshold_source", ""),
            "toprisk_percent": r.get("toprisk_percent", np.nan),
            "toprisk_threshold": r.get("toprisk_threshold", np.nan),
            "toprisk_accuracy": r.get("test_toprisk_accuracy", np.nan),
            "toprisk_balanced_accuracy": r.get("test_toprisk_balanced_accuracy", np.nan),
            "toprisk_sensitivity": r.get("test_toprisk_sensitivity", np.nan),
            "toprisk_specificity": r.get("test_toprisk_specificity", np.nan),
            "toprisk_F1_score": r.get("test_toprisk_F1", np.nan),
            "toprisk_MCC": r.get("test_toprisk_MCC", np.nan),
            "toprisk_threshold_source": r.get("toprisk_threshold_source", ""),
            "DeLong_CI_method": "single-model DeLong variance; normal 95% CI",
            "Jackknife_CI_method": "delete-one patient jackknife; normal 95% CI",
            "Bootstrap_CI_method": "stratified patient-level nonparametric bootstrap; percentile 95% CI",
        })

publication_df = pd.DataFrame(publication_rows)

# The outcome definition and the channel policy are shown only when the run compares more
# than one of them. With a single value the column would repeat that value on every row.
if len(publication_df):
    _named_outcomes = publication_df.loc[publication_df["outcome"] != "", "outcome"]
    if _named_outcomes.nunique() <= 1:
        publication_df = publication_df.drop(columns="outcome")
    _policies_per_backbone = (
        publication_df[publication_df["channel_policy"] != ""]
        .groupby(["workflow", "_backbone"])["channel_policy"].nunique()
    )
    if not bool((_policies_per_backbone > 1).any()):
        publication_df = publication_df.drop(columns="channel_policy")
    publication_df = publication_df.drop(columns="_backbone")

print("\n" + "=" * 120)
print("CONSOLIDATED PUBLICATION-METRICS TABLE")
print("=" * 120)
print(
    "Threshold reporting: AUC/CIs and Brier are threshold-independent. "
    "For the efficient_ML workflow, threshold-dependent holdout metrics are reported "
    "both at (1) the pooled development OOF-derived Youden threshold and "
    "(2) the final-refit full-development in-sample-derived Youden threshold. "
    "The two analyses are labelled separately and should not be interpreted as identical estimands."
)

if len(publication_df):
    # One block of rows per holdout test set, best AUC first within each.
    publication_df = _by_test_set_then_auc(publication_df, "AUC").reset_index(drop=True)
    _final_test_sets = [n for n in dict.fromkeys(publication_df["test_set"]) if n]

    # Completeness guard: every primary result row from Sections 1, 2 and 3
    # must appear exactly once in the consolidated table.
    _expected_primary_rows = sum(
        len(_df) for _df in (seg1, seg3, seg4)
        if _df is not None
    )
    if len(publication_df) != _expected_primary_rows:
        raise RuntimeError(
            "FINAL OVERVIEW row-count mismatch: "
            f"expected {_expected_primary_rows} primary result rows from the registry, "
            f"built {len(publication_df)}."
        )
    print(f"[FINAL OVERVIEW] completeness check passed: {len(publication_df)} primary result rows.")
    overview_csv = os.path.join(
        COMBINED_OVERVIEW_DIR,
        "combined_pipeline_overview.csv",
    )
    overview_csv_by_test_set = {}
    if len(_final_test_sets) > 1:
        # Two test sets: the final table is given separately for each test set, and each
        # table is also saved to its own CSV. The CSV of all rows keeps the test_set column.
        for _test_set_name in _final_test_sets:
            _part = (
                publication_df[publication_df["test_set"] == _test_set_name]
                .drop(columns="test_set")
                .reset_index(drop=True)
            )
            print("\n" + "=" * 120)
            print(f"FINAL TABLE, HOLDOUT TEST SET: {_test_set_name}  ({len(_part)} result rows)")
            print("=" * 120)
            print(_part.to_string(index=False))
            _part_csv = os.path.join(
                COMBINED_OVERVIEW_DIR,
                "combined_pipeline_overview__"
                + re.sub(r"[^A-Za-z0-9]+", "_", _test_set_name).strip("_") + ".csv",
            )
            _part.to_csv(_part_csv, index=False, encoding="utf-8-sig")
            overview_csv_by_test_set[_test_set_name] = _part_csv
    else:
        # One test set: a single table, without a column that would repeat its name.
        publication_df = publication_df.drop(columns="test_set")
        print(publication_df.to_string(index=False))

    publication_df.to_csv(
        overview_csv,
        index=False,
        encoding="utf-8-sig",
    )

    if not seg1_overview.empty:
        seg1_overview.to_csv(
            os.path.join(
                COMBINED_OVERVIEW_DIR,
                "part1_cnn_frozen_holdout.csv",
            ),
            index=False,
            encoding="utf-8-sig",
        )

    if not seg4_overview.empty:
        seg4_overview.to_csv(
            os.path.join(
                COMBINED_OVERVIEW_DIR,
                "part4_cnn_finetuned_holdout.csv",
            ),
            index=False,
            encoding="utf-8-sig",
        )

    print("\nOverview CSV:", overview_csv)
    if SEGMENTATION_ANALYSIS_METHOD == "original" and selected_segmentation_analysis_methods():
        print(
            "[FINAL OVERVIEW] Section 6 will refresh this CSV and its per-test-set files "
            "with original + all selected generated-mask results. Run through Section 6."
        )
    for _test_set_name, _part_csv in overview_csv_by_test_set.items():
        print(f"Overview CSV, test set {_test_set_name}:", _part_csv)
else:
    print("  (no holdout-test results available)")

print("\nDone. Every section wrote into its own sub-folder of the shared OUT_ROOT:", OUT_ROOT)



## Section 6, cell 1 - Run additional full segmentation analyses

Replays the same completed Sections 1–5 once per selected generated-mask method,
using isolated results/caches and the original patients and starting random state.
Only workflows enabled by the usual `RUN_*` switches run. The original analysis
remains included in the combined segmentation comparison.
After collecting these results, this section also replaces the main
`COMBINED_OVERVIEW_DIR/combined_pipeline_overview.csv` and its per-test-set files
with all available original, CAS2/SAM2 and nnInteractive results. Every row retains
its `segmentation_method` label. Rerunning this cell after a successful analysis
refreshes these exports without repeating model training.


The final `segmentation_analysis_status.csv` lists original and every selected mask
method, with completed result counts and errors. An incomplete method raises an
explicit error after saving available comparison rows. Existing Drive masks remain
reusable; rerunning the notebook does not regenerate cached segmentations.


In [ ]:
# Section 6, cell 2: Additional full analyses and cross-segmentation overview
run_additional_segmentation_analyses()

# ── Whole-notebook wall-clock duration ──────────────────────────────────────
# This is deliberately the final output, after every enabled workflow, summary,
# test table and save operation has completed.
if "_NOTEBOOK_RUN_T0_MONOTONIC" in globals():
    _NOTEBOOK_RUN_FINISHED_AT = _notebook_datetime.now().astimezone()
    _NOTEBOOK_RUN_ELAPSED_SECONDS = (
        _notebook_time.perf_counter() - _NOTEBOOK_RUN_T0_MONOTONIC
    )

    _elapsed_total = int(round(_NOTEBOOK_RUN_ELAPSED_SECONDS))
    _elapsed_hours, _elapsed_rem = divmod(_elapsed_total, 3600)
    _elapsed_minutes, _elapsed_seconds = divmod(_elapsed_rem, 60)
    _elapsed_hms = f"{_elapsed_hours:02d}:{_elapsed_minutes:02d}:{_elapsed_seconds:02d}"

    print("\n" + "=" * 120)
    print("TOTAL NOTEBOOK RUN TIME")
    print("=" * 120)
    print(
        "Started :",
        _NOTEBOOK_RUN_STARTED_AT.isoformat(timespec="seconds"),
    )
    print(
        "Finished:",
        _NOTEBOOK_RUN_FINISHED_AT.isoformat(timespec="seconds"),
    )
    print(
        f"Elapsed : {_elapsed_hms} "
        f"({_NOTEBOOK_RUN_ELAPSED_SECONDS / 3600.0:.2f} hours; "
        f"{_NOTEBOOK_RUN_ELAPSED_SECONDS / 60.0:.1f} minutes)"
    )
    print("=" * 120)
else:
    print(
        "\n[RUN TIMER] Start timestamp is unavailable. "
        "Run the notebook from the first code cell (or use Run all) "
        "to obtain complete start-to-finish duration."
    )
